# Subsetting & Format Conversion for Downstream Analysis

**Module 04: Cell Type Selection, Data Export & Subclustering**

**Method:** Subset to significant cell types from Module 02 (GLMM) and Module 03 (meta-analysis), convert h5ad to Seurat-compatible format, reprocess in Seurat, and subcluster key glial populations using published marker sets.

**Workflow:**
1. Cell type subsetting based on Module 02/03 significance (OPC, Astrocyte, Microglia)
2. Format conversion: h5ad → MTX + metadata for Seurat import
3. Seurat object creation & reprocessing (SCTransform, PCA, UMAP, clustering)
4. Microglia subclustering & annotation (Harari Lab integrated markers)
5. Astrocyte subclustering & annotation (Serrano-Pozo et al. 2024)
6. Export annotated objects (.qs) for Module 05+ downstream analysis

**Author:** Gerald Gaitos  
**Date:** February 2026

In [ ]:
print("="*80)
print("MODULE 04: SUBSETTING & FORMAT CONVERSION")
print("="*80)

# ═══════════════════════════════════════════════════════════════════════════════
# IMPORTS
# ═══════════════════════════════════════════════════════════════════════════════

import scanpy as sc
import pandas as pd
import numpy as np
import scipy.io
import gzip
import shutil
import os
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

print("\n✓ Packages imported successfully")
print(f"  scanpy: {sc.__version__}")
print(f"  pandas: {pd.__version__}")
print(f"  numpy: {np.__version__}")

In [ ]:
print("\n" + "="*80)
print("CONFIGURATION")
print("="*80)

# ════════════════════════════════════════════════════════════════════════════════
# DATASET SELECTION
# ════════════════════════════════════════════════════════════════════════════════

DATASET = 'psychad_aging'  # Options: 'psychad_aging', 'psychad_ad', 'psychencode', 'mathys'

# ════════════════════════════════════════════════════════════════════════════════
# PATHS
# ════════════════════════════════════════════════════════════════════════════════

BASE_DIR = Path('/fs/scratch/PAS2598/senescence_analysis')
INPUT_FILE = BASE_DIR / 'data' / 'processed' / f'{DATASET}_pearson_senescence_scored.h5ad'
OUTPUT_DIR = BASE_DIR / 'data' / '04_subsetting' / DATASET
RESULTS_DIR = BASE_DIR / 'results' / '04_subsetting' / DATASET
FIGURES_DIR = BASE_DIR / 'figures' / '04_subsetting' / DATASET

# ════════════════════════════════════════════════════════════════════════════════
# SUBSETTING PARAMETERS
# ════════════════════════════════════════════════════════════════════════════════

# Cell types to include (from Module 03 meta-analysis results)
# Set to None to include all cell types
SIGNIFICANT_CELLTYPES = ['Astrocyte', 'OPC'] #'Astrocyte', 'OPC', 

# Minimum cells per donor per cell type (for pseudobulk QC)
MIN_CELLS_PER_DONOR = 10

# ════════════════════════════════════════════════════════════════════════════════
# COLUMN NAMES (from Modules 00-01)
# ════════════════════════════════════════════════════════════════════════════════

# Senescence columns (created in Module 01)
SENESCENCE_SCORE_COL = 'senescence_score'    # Continuous score from SenePy
SENESCENCE_BOOL_COL = 'is_senescent'         # Boolean classification
SENESCENCE_LABEL_COL = 'senescence_label'    # Categorical: 'SnC' or 'Non-SnC'

STUDY_GROUP_COLUMN = 'Study_Group'

# ════════════════════════════════════════════════════════════════════════════════
# DATASET-SPECIFIC CONFIGURATIONS
# ════════════════════════════════════════════════════════════════════════════════

DATASET_CONFIG = {
    'psychad_aging': {
        'type': 'aging',
        'reference_group': 'Age_20_29',
        'target_group': 'Age_80_100',
        'donor_column': 'Sample',
        'cell_type_column': 'subclass',
    },
    'psychad_ad': {
        'type': 'disease',
        'reference_group': 'Control',
        'target_group': 'AD',
        'donor_column': 'Sample',
        'cell_type_column': 'subclass',
    },
    'psychencode': {
        'type': 'aging',
        'reference_group': 'Age_30_39',
        'target_group': 'Age_80_100',
        'donor_column': 'sample_id',
        'cell_type_column': 'major_celltype',
    },
    'mathys': {
        'type': 'disease',
        'reference_group': 'NCI',
        'target_group': 'AD',
        'donor_column': 'Subject',
        'cell_type_column': 'broad.cell.type',
    },
}

config = DATASET_CONFIG[DATASET]
STUDY_TYPE = config['type']
REFERENCE_GROUP = config['reference_group']
TARGET_GROUP = config['target_group']
DONOR_COLUMN = config['donor_column']
CELL_TYPE_COLUMN = config['cell_type_column']

print("="*80)
print(f"Dataset: {DATASET}")
print(f"Study type: {STUDY_TYPE}")
print(f"Comparison: {TARGET_GROUP} vs {REFERENCE_GROUP} (reference)")
print(f"Cell types to subset: {SIGNIFICANT_CELLTYPES}")
print(f"Min cells per donor: {MIN_CELLS_PER_DONOR}")
print(f"Input: {INPUT_FILE.name}")
print(f"Output: {OUTPUT_DIR}")
print("="*80)

In [ ]:
print("\n" + "="*80)
print("DIRECTORY SETUP")
print("="*80)

# ════════════════════════════════════════════════════════════════════════════════
# CREATE OUTPUT DIRECTORIES
# ════════════════════════════════════════════════════════════════════════════════

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# Subdirectories for organized output
MTX_DIR = OUTPUT_DIR / 'mtx_export'
MTX_DIR.mkdir(parents=True, exist_ok=True)

print(f"\n✓ Directories created:")
print(f"  Output: {OUTPUT_DIR}")
print(f"  MTX export: {MTX_DIR}")
print(f"  Results: {RESULTS_DIR}")
print(f"  Figures: {FIGURES_DIR}")

In [ ]:
print("\n" + "="*80)
print("LOADING DATA")
print("="*80)

# ════════════════════════════════════════════════════════════════════════════════
# LOAD SCORED H5AD FROM MODULE 01
# ════════════════════════════════════════════════════════════════════════════════

adata = sc.read_h5ad(INPUT_FILE)

print(f"\n✓ Loaded: {INPUT_FILE.name}")
print(f"  Cells: {adata.n_obs:,}")
print(f"  Genes: {adata.n_vars:,}")

# Verify required columns exist
required_cols = [CELL_TYPE_COLUMN, DONOR_COLUMN, STUDY_GROUP_COLUMN, 
                 SENESCENCE_SCORE_COL, SENESCENCE_BOOL_COL, SENESCENCE_LABEL_COL]

missing_cols = [col for col in required_cols if col not in adata.obs.columns]
if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")

print(f"\n✓ Required columns verified:")
for col in required_cols:
    n_unique = adata.obs[col].nunique()
    print(f"  {col}: {n_unique} unique values")

In [ ]:
print("\n" + "="*80)
print("DATA OVERVIEW")
print("="*80)

# ════════════════════════════════════════════════════════════════════════════════
# CELL TYPE DISTRIBUTION
# ════════════════════════════════════════════════════════════════════════════════

print(f"\nCell types in dataset:")
ct_counts = adata.obs[CELL_TYPE_COLUMN].value_counts()
for ct, count in ct_counts.items():
    marker = "→" if ct in SIGNIFICANT_CELLTYPES else " "
    print(f"  {marker} {ct}: {count:,} ({count/adata.n_obs*100:.1f}%)")

# ════════════════════════════════════════════════════════════════════════════════
# STUDY GROUP DISTRIBUTION
# ════════════════════════════════════════════════════════════════════════════════

print(f"\nStudy groups in dataset:")
sg_counts = adata.obs[STUDY_GROUP_COLUMN].value_counts()
for sg, count in sg_counts.items():
    marker = "→" if sg in [REFERENCE_GROUP, TARGET_GROUP] else " "
    print(f"  {marker} {sg}: {count:,} ({count/adata.n_obs*100:.1f}%)")

# ════════════════════════════════════════════════════════════════════════════════
# SENESCENCE DISTRIBUTION
# ════════════════════════════════════════════════════════════════════════════════

print(f"\nSenescence status:")
snc_counts = adata.obs[SENESCENCE_LABEL_COL].value_counts()
for label, count in snc_counts.items():
    print(f"  {label}: {count:,} ({count/adata.n_obs*100:.1f}%)")

# ════════════════════════════════════════════════════════════════════════════════
# DONOR COUNTS
# ════════════════════════════════════════════════════════════════════════════════

n_donors = adata.obs[DONOR_COLUMN].nunique()
print(f"\nDonors: {n_donors}")

# Donors per study group
print(f"\nDonors per study group:")
for sg in [REFERENCE_GROUP, TARGET_GROUP]:
    if sg in adata.obs[STUDY_GROUP_COLUMN].values:
        n = adata.obs[adata.obs[STUDY_GROUP_COLUMN] == sg][DONOR_COLUMN].nunique()
        print(f"  {sg}: {n} donors")

In [ ]:
print("\n" + "="*80)
print("SUBSET TO SIGNIFICANT CELL TYPES")
print("="*80)

# ════════════════════════════════════════════════════════════════════════════════
# FILTER TO CELL TYPES FROM META-ANALYSIS
# ════════════════════════════════════════════════════════════════════════════════

if SIGNIFICANT_CELLTYPES is not None:
    # Verify all specified cell types exist
    available_cts = adata.obs[CELL_TYPE_COLUMN].unique().tolist()
    missing_cts = [ct for ct in SIGNIFICANT_CELLTYPES if ct not in available_cts]
    if missing_cts:
        print(f"⚠ Warning: Cell types not found: {missing_cts}")
        SIGNIFICANT_CELLTYPES = [ct for ct in SIGNIFICANT_CELLTYPES if ct in available_cts]
    
    # Subset
    adata_subset = adata[adata.obs[CELL_TYPE_COLUMN].isin(SIGNIFICANT_CELLTYPES)].copy()
    
    print(f"\n✓ Subsetted to {len(SIGNIFICANT_CELLTYPES)} cell types:")
    print(f"  Before: {adata.n_obs:,} cells")
    print(f"  After: {adata_subset.n_obs:,} cells")
    print(f"  Retained: {adata_subset.n_obs/adata.n_obs*100:.1f}%")
else:
    adata_subset = adata.copy()
    print("\n✓ Keeping all cell types (SIGNIFICANT_CELLTYPES = None)")

# Show breakdown
print(f"\nCell type breakdown after subsetting:")
for ct in adata_subset.obs[CELL_TYPE_COLUMN].unique():
    n = (adata_subset.obs[CELL_TYPE_COLUMN] == ct).sum()
    n_snc = ((adata_subset.obs[CELL_TYPE_COLUMN] == ct) & (adata_subset.obs[SENESCENCE_BOOL_COL])).sum()
    pct_snc = n_snc / n * 100 if n > 0 else 0
    print(f"  {ct}: {n:,} cells ({n_snc:,} SnC, {pct_snc:.1f}%)")

In [ ]:
print("\n" + "="*80)
print("PSEUDOBULK QC CHECK")
print("="*80)

# ════════════════════════════════════════════════════════════════════════════════
# CHECK CELLS PER DONOR PER CELL TYPE
# Ensures sufficient cells for downstream pseudobulk aggregation
# ════════════════════════════════════════════════════════════════════════════════

print(f"\nMinimum cells per donor threshold: {MIN_CELLS_PER_DONOR}")

# Calculate cells per donor per cell type
donor_ct_counts = adata_subset.obs.groupby([DONOR_COLUMN, CELL_TYPE_COLUMN]).size().reset_index(name='n_cells')

# Flag donors below threshold
below_threshold = donor_ct_counts[donor_ct_counts['n_cells'] < MIN_CELLS_PER_DONOR]

print(f"\nDonor × Cell Type combinations:")
print(f"  Total: {len(donor_ct_counts)}")
print(f"  Below threshold (<{MIN_CELLS_PER_DONOR} cells): {len(below_threshold)}")
print(f"  Pass threshold: {len(donor_ct_counts) - len(below_threshold)}")

# Summary per cell type
print(f"\nSummary per cell type:")
for ct in sorted(adata_subset.obs[CELL_TYPE_COLUMN].unique()):
    ct_data = donor_ct_counts[donor_ct_counts[CELL_TYPE_COLUMN] == ct]
    n_donors = len(ct_data)
    n_pass = (ct_data['n_cells'] >= MIN_CELLS_PER_DONOR).sum()
    median_cells = ct_data['n_cells'].median()
    min_cells = ct_data['n_cells'].min()
    max_cells = ct_data['n_cells'].max()
    print(f"  {ct}: {n_pass}/{n_donors} donors pass, median={median_cells:.0f} cells (range: {min_cells}-{max_cells})")

# Save QC table as CSV
qc_output = RESULTS_DIR / f'{DATASET}_pseudobulk_qc.csv'
donor_ct_counts.to_csv(qc_output, index=False)
print(f"\n✓ QC table saved: {qc_output}")

In [ ]:
print("\n" + "="*80)
print("SAVE SUBSETTED H5AD")
print("="*80)

# ════════════════════════════════════════════════════════════════════════════════
# SAVE CELL TYPE SUBSET FOR DOWNSTREAM ANALYSIS
# ════════════════════════════════════════════════════════════════════════════════

output_h5ad = OUTPUT_DIR / f'{DATASET}_celltypes_subset.h5ad'
adata_subset.write_h5ad(output_h5ad)

print(f"\n✓ Saved: {output_h5ad}")
print(f"  Cells: {adata_subset.n_obs:,}")
print(f"  Genes: {adata_subset.n_vars:,}")
print(f"  Cell types: {', '.join(adata_subset.obs[CELL_TYPE_COLUMN].unique())}")

# Summary of what's included
print(f"\nDataset contains:")
print(f"  Study groups: {adata_subset.obs[STUDY_GROUP_COLUMN].nunique()} groups")
print(f"  Donors: {adata_subset.obs[DONOR_COLUMN].nunique()}")
print(f"  SnC cells: {adata_subset.obs[SENESCENCE_BOOL_COL].sum():,} ({adata_subset.obs[SENESCENCE_BOOL_COL].mean()*100:.1f}%)")

---

# 04B: Export to MTX Format (Python)

Convert h5ad to 10X-style MTX format for Seurat import.

**Exports:**
- `matrix.mtx.gz` - Raw count matrix (cells × genes)
- `barcodes.tsv.gz` - Cell barcodes
- `features.tsv.gz` - Gene names
- `obs_metadata.csv` - Cell metadata (senescence labels, study groups, etc.)

In [ ]:
print("\n" + "="*80)
print("VERIFY RAW COUNTS")
print("="*80)

# ════════════════════════════════════════════════════════════════════════════════
# CHECK COUNTS LAYER FOR SEURAT EXPORT
# Seurat requires raw integer counts, not normalized data
# ════════════════════════════════════════════════════════════════════════════════

# Check available layers
print(f"\nAvailable layers: {list(adata_subset.layers.keys())}")

# Check if counts layer exists
if 'counts' in adata_subset.layers:
    counts = adata_subset.layers['counts']
    print(f"\n✓ 'counts' layer found")
else:
    counts = adata_subset.X
    print(f"\n⚠ No 'counts' layer - using adata.X")

# Convert sparse to dense for inspection
if hasattr(counts, 'toarray'):
    sample_vals = counts[:5, :5].toarray()
elif hasattr(counts, 'todense'):
    sample_vals = np.array(counts[:5, :5].todense())
else:
    sample_vals = np.array(counts[:5, :5])

# Validate counts
min_val = counts.min()
max_val = counts.max()

# Check if integers
if hasattr(counts, 'data'):
    all_integers = np.all(np.equal(np.mod(counts.data, 1), 0))
else:
    all_integers = np.all(np.equal(np.mod(counts, 1), 0))

print(f"\nCounts matrix:")
print(f"  Shape: {counts.shape}")
print(f"  Min value: {min_val}")
print(f"  Max value: {max_val}")
print(f"  All integers: {all_integers}")
print(f"\nSample values (5x5):\n{sample_vals}")

if not all_integers:
    print("\n⚠ WARNING: Values are not integers - may be normalized data!")
    print("  Seurat requires raw counts for DESeq2/pseudobulk analysis")

In [ ]:
print("\n" + "="*80)
print("EXPORT MTX FORMAT")
print("="*80)

# ════════════════════════════════════════════════════════════════════════════════
# EXPORT TO 10X-STYLE MTX FORMAT FOR SEURAT
# ════════════════════════════════════════════════════════════════════════════════

# Set adata.X to raw counts for export
if 'counts' in adata_subset.layers:
    adata_subset.X = adata_subset.layers['counts'].copy()
    print("✓ Set adata.X to counts layer")

# Create MTX output directory
mtx_output = MTX_DIR / DATASET
mtx_output.mkdir(parents=True, exist_ok=True)

# --- Export matrix.mtx.gz ---
print(f"\nExporting matrix.mtx.gz...")
mtx_file = mtx_output / 'matrix.mtx'
scipy.io.mmwrite(mtx_file, adata_subset.X.T)  # Transpose: genes × cells for 10X format

# Compress
with open(mtx_file, 'rb') as f_in:
    with gzip.open(f'{mtx_file}.gz', 'wb') as f_out:
        shutil.copyfileobj(f_in, f_out)
os.remove(mtx_file)
print(f"  ✓ matrix.mtx.gz")

# --- Export barcodes.tsv.gz ---
print(f"Exporting barcodes.tsv.gz...")
barcodes_file = mtx_output / 'barcodes.tsv.gz'
adata_subset.obs_names.to_series().to_csv(
    barcodes_file, index=False, header=False, sep='\t', compression='gzip'
)
print(f"  ✓ barcodes.tsv.gz")

# --- Export features.tsv.gz ---
print(f"Exporting features.tsv.gz...")
features_file = mtx_output / 'features.tsv.gz'
features = pd.DataFrame({
    'gene_id': adata_subset.var_names,
    'gene_name': adata_subset.var_names,
    'feature_type': 'Gene Expression'
})
features.to_csv(
    features_file, index=False, header=False, sep='\t', compression='gzip'
)
print(f"  ✓ features.tsv.gz")

# --- Export metadata ---
print(f"Exporting metadata...")
obs_file = mtx_output / 'obs_metadata.csv'
adata_subset.obs.to_csv(obs_file)
print(f"  ✓ obs_metadata.csv")

var_file = mtx_output / 'var_metadata.csv'
adata_subset.var.to_csv(var_file)
print(f"  ✓ var_metadata.csv")

print(f"\n✓ MTX export complete: {mtx_output}")
print(f"  Files: matrix.mtx.gz, barcodes.tsv.gz, features.tsv.gz, obs_metadata.csv, var_metadata.csv")

---

# 04C: Seurat Object Creation & Reprocessing (R)

Load MTX files into Seurat, add metadata, and reprocess the subset.

**Steps:**
1. Load MTX files (matrix, barcodes, features)
2. Create Seurat object with raw counts
3. Add metadata (senescence labels, study groups, donor info)
4. Reprocess: Normalize → FindVariableFeatures → Scale → PCA → UMAP → Cluster
5. QC visualization
6. Save as .qs file

**Note:** Switch kernel to R for the following cells.

In [ ]:
cat("================================================================================\n")
cat("MODULE 04C: SEURAT OBJECT CREATION & REPROCESSING\n")
cat("================================================================================\n")

# ════════════════════════════════════════════════════════════════════════════════
# LOAD LIBRARIES
# ════════════════════════════════════════════════════════════════════════════════

suppressPackageStartupMessages({
    library(Seurat)
    library(Matrix)
    library(dplyr)
    library(ggplot2)
    library(qs)
})

cat("\n✓ Libraries loaded\n")
cat("  Seurat:", as.character(packageVersion("Seurat")), "\n")
cat("  Matrix:", as.character(packageVersion("Matrix")), "\n")
cat("  qs:", as.character(packageVersion("qs")), "\n")

In [ ]:
cat("\n================================================================================\n")
cat("04C: R CONFIGURATION\n")
cat("================================================================================\n")

# ════════════════════════════════════════════════════════════════════════════════
# DATASET CONFIGURATION
# ════════════════════════════════════════════════════════════════════════════════

DATASET <- "psychad_aging"

# ════════════════════════════════════════════════════════════════════════════════
# COLUMN NAMES (must match Python config)
# ════════════════════════════════════════════════════════════════════════════════

CELL_TYPE_COL <- "subclass"
SENESCENCE_SCORE_COL <- "senescence_score"
SENESCENCE_BOOL_COL <- "is_senescent"
SENESCENCE_LABEL_COL <- "senescence_label"
STUDY_GROUP_COL <- "Study_Group"
DONOR_COL <- "Sample"

# ════════════════════════════════════════════════════════════════════════════════
# PATHS
# ════════════════════════════════════════════════════════════════════════════════

BASE_DIR <- "/fs/scratch/PAS2598/senescence_analysis"
OUTPUT_DIR <- file.path(BASE_DIR, "data", "04_subsetting", DATASET)
MTX_DIR <- file.path(OUTPUT_DIR, "mtx_export", DATASET)
RESULTS_DIR <- file.path(OUTPUT_DIR, "results")
FIGURES_DIR <- file.path(OUTPUT_DIR, "figures")

# Create directories
dir.create(RESULTS_DIR, recursive = TRUE, showWarnings = FALSE)
dir.create(FIGURES_DIR, recursive = TRUE, showWarnings = FALSE)

# ════════════════════════════════════════════════════════════════════════════════
# PROCESSING PARAMETERS
# ════════════════════════════════════════════════════════════════════════════════

N_VARIABLE_FEATURES <- 2000
N_PCS <- 50
N_DIMS_USE <- 30
CLUSTERING_RESOLUTION <- 0.5

# ════════════════════════════════════════════════════════════════════════════════
# PRINT CONFIGURATION
# ════════════════════════════════════════════════════════════════════════════════

cat("\nDataset:", DATASET, "\n")
cat("\nColumn names:\n")
cat("  Cell type:", CELL_TYPE_COL, "\n")
cat("  Senescence label:", SENESCENCE_LABEL_COL, "\n")
cat("  Study group:", STUDY_GROUP_COL, "\n")
cat("  Donor:", DONOR_COL, "\n")
cat("\nPaths:\n")
cat("  MTX input:", MTX_DIR, "\n")
cat("  Output:", OUTPUT_DIR, "\n")
cat("  Results:", RESULTS_DIR, "\n")
cat("  Figures:", FIGURES_DIR, "\n")
cat("\nProcessing parameters:\n")
cat("  Variable features:", N_VARIABLE_FEATURES, "\n")
cat("  PCs:", N_PCS, "\n")
cat("  Dims for UMAP/clustering:", N_DIMS_USE, "\n")
cat("  Clustering resolution:", CLUSTERING_RESOLUTION, "\n")

In [ ]:
cat("\n================================================================================\n")
cat("LOAD MTX AND CREATE SEURAT OBJECT\n")
cat("================================================================================\n")

# ════════════════════════════════════════════════════════════════════════════════
# LOAD 10X-STYLE MTX FILES
# ════════════════════════════════════════════════════════════════════════════════

cat("\nLoading MTX files...\n")

counts <- Read10X(data.dir = MTX_DIR)

cat("✓ Matrix loaded:", nrow(counts), "genes x", ncol(counts), "cells\n")

# ════════════════════════════════════════════════════════════════════════════════
# CREATE SEURAT OBJECT
# ════════════════════════════════════════════════════════════════════════════════

cat("\nCreating Seurat object...\n")

data <- CreateSeuratObject(
    counts = counts,
    project = DATASET,
    min.cells = 0,
    min.features = 0
)

cat("✓ Seurat object created:", ncol(data), "cells,", nrow(data), "genes\n")

# ════════════════════════════════════════════════════════════════════════════════
# ADD METADATA
# ════════════════════════════════════════════════════════════════════════════════

cat("\nAdding metadata...\n")

metadata <- read.csv(file.path(MTX_DIR, 'obs_metadata.csv'), row.names = 1)

if (!all(rownames(metadata) == colnames(data))) {
    metadata <- metadata[colnames(data), ]
}

data <- AddMetaData(data, metadata)

cat("✓ Metadata added:", ncol(metadata), "columns\n")
cat("  Columns:", paste(colnames(metadata)[1:min(10, ncol(metadata))], collapse = ", "), "...\n")

In [ ]:
cat("\n================================================================================\n")
cat("REPROCESSING: NORMALIZATION & VARIABLE FEATURES\n")
cat("================================================================================\n")

# ════════════════════════════════════════════════════════════════════════════════
# NORMALIZATION
# ════════════════════════════════════════════════════════════════════════════════

cat("\nNormalizing data...\n")

data <- NormalizeData(
    data,
    normalization.method = "LogNormalize",
    scale.factor = 10000,
    verbose = TRUE
)

cat("✓ LogNormalize complete (scale factor = 10,000)\n")

# ════════════════════════════════════════════════════════════════════════════════
# FIND VARIABLE FEATURES
# ════════════════════════════════════════════════════════════════════════════════

cat("\nFinding top", N_VARIABLE_FEATURES, "variable features...\n")

data <- FindVariableFeatures(
    data,
    selection.method = "vst",
    nfeatures = N_VARIABLE_FEATURES,
    verbose = TRUE
)

cat("✓", length(VariableFeatures(data)), "variable features identified\n")

top10 <- head(VariableFeatures(data), 10)
cat("  Top 10:", paste(top10, collapse = ", "), "\n")

In [ ]:
cat("\n================================================================================\n")
cat("REPROCESSING: SCALING & PCA\n")
cat("================================================================================\n")

# ════════════════════════════════════════════════════════════════════════════════
# SCALE DATA
# ════════════════════════════════════════════════════════════════════════════════

cat("\nScaling data...\n")

data <- ScaleData(
    data,
    features = rownames(data),
    verbose = TRUE
)

cat("✓ ScaleData complete\n")

# ════════════════════════════════════════════════════════════════════════════════
# RUN PCA
# ════════════════════════════════════════════════════════════════════════════════

cat("\nRunning PCA (", N_PCS, " PCs)...\n", sep = "")

data <- RunPCA(
    data,
    features = VariableFeatures(data),
    npcs = N_PCS,
    verbose = TRUE
)

cat("✓ PCA complete:", N_PCS, "PCs computed\n")

pct_var <- data[["pca"]]@stdev^2 / sum(data[["pca"]]@stdev^2) * 100
cat("  Variance explained (PC1-5):", paste(round(pct_var[1:5], 1), "%", sep = "", collapse = ", "), "\n")

In [ ]:
cat("\n================================================================================\n")
cat("REPROCESSING: UMAP & CLUSTERING\n")
cat("================================================================================\n")

# ════════════════════════════════════════════════════════════════════════════════
# FIND NEIGHBORS
# ════════════════════════════════════════════════════════════════════════════════

cat("\nFinding neighbors (dims = 1:", N_DIMS_USE, ")...\n", sep = "")

data <- FindNeighbors(
    data,
    dims = 1:N_DIMS_USE,
    verbose = TRUE
)

cat("✓ FindNeighbors complete\n")

# ════════════════════════════════════════════════════════════════════════════════
# RUN UMAP
# ════════════════════════════════════════════════════════════════════════════════

cat("\nRunning UMAP (dims = 1:", N_DIMS_USE, ")...\n", sep = "")

data <- RunUMAP(
    data,
    dims = 1:N_DIMS_USE,
    verbose = TRUE
)

cat("✓ UMAP complete\n")

# ════════════════════════════════════════════════════════════════════════════════
# CLUSTERING
# ════════════════════════════════════════════════════════════════════════════════

cat("\nClustering (resolution =", CLUSTERING_RESOLUTION, ")...\n")

data <- FindClusters(
    data,
    resolution = CLUSTERING_RESOLUTION,
    verbose = TRUE
)

n_clusters <- length(unique(Idents(data)))
cat("✓ Clustering complete:", n_clusters, "clusters identified\n")

In [ ]:
cat("\n================================================================================\n")
cat("QC VISUALIZATION\n")
cat("================================================================================\n")

# ════════════════════════════════════════════════════════════════════════════════
# DIMPLOTS
# ════════════════════════════════════════════════════════════════════════════════

celltype_col <- 'subclass'
senescence_col <- 'senescence_label'
studygroup_col <- 'Study_Group'

cat("\nGenerating UMAP plots...\n")

p1 <- DimPlot(data, reduction = "umap", group.by = celltype_col, label = TRUE, repel = TRUE) +
    ggtitle("Cell Types") +
    theme(legend.position = "right")

p2 <- DimPlot(data, reduction = "umap", group.by = senescence_col) +
    ggtitle("Senescence Status") +
    theme(legend.position = "right")

p3 <- DimPlot(data, reduction = "umap", group.by = studygroup_col) +
    ggtitle("Study Group") +
    theme(legend.position = "right")

p4 <- DimPlot(data, reduction = "umap", group.by = "seurat_clusters", label = TRUE) +
    ggtitle("Clusters (Reprocessed)") +
    theme(legend.position = "right")

cat("✓ Plots generated\n")

In [ ]:
cat("\n================================================================================\n")
cat("DISPLAY PLOTS\n")
cat("================================================================================\n")

# ════════════════════════════════════════════════════════════════════════════════
# DISPLAY PLOTS
# ════════════════════════════════════════════════════════════════════════════════

options(repr.plot.width = 16, repr.plot.height = 12)

combined_plot <- (p1 | p2) / (p3 | p4)
print(combined_plot)

# ════════════════════════════════════════════════════════════════════════════════
# SAVE PLOTS
# ════════════════════════════════════════════════════════════════════════════════

cat("\nSaving plots...\n")

pdf(file.path(FIGURES_DIR, paste0(DATASET, "_umap_qc.svg")), width = 16, height = 12)
print(combined_plot)
dev.off()

cat("✓ Saved:", file.path(FIGURES_DIR, paste0(DATASET, "_umap_qc.svg")), "\n")

In [ ]:
cat("\n================================================================================\n")
cat("SUMMARY & SAVE\n")
cat("================================================================================\n")

# ════════════════════════════════════════════════════════════════════════════════
# OBJECT SUMMARY
# ════════════════════════════════════════════════════════════════════════════════

cat("\nSeurat object summary:\n")
cat("  Cells:", ncol(data), "\n")
cat("  Genes:", nrow(data), "\n")
cat("  Assays:", paste(Assays(data), collapse = ", "), "\n")
cat("  Reductions:", paste(Reductions(data), collapse = ", "), "\n")
cat("  Clusters:", length(unique(Idents(data))), "\n")

cat("\nCell type distribution:\n")
print(table(data[[celltype_col]]))

cat("\nSenescence distribution:\n")
print(table(data[[senescence_col]]))

# ════════════════════════════════════════════════════════════════════════════════
# SAVE SEURAT OBJECT
# ════════════════════════════════════════════════════════════════════════════════

cat("\nSaving Seurat object...\n")

output_file <- file.path(OUTPUT_DIR, paste0(DATASET, "_seurat.qs"))
qsave(data, output_file)

cat("✓ Saved:", output_file, "\n")

cat("\n================================================================================\n")
cat("MODULE 04 COMPLETE\n")
cat("================================================================================\n")
cat("\nOutput files:\n")
cat("  H5AD:", file.path(OUTPUT_DIR, paste0(DATASET, "_celltypes_subset.h5ad")), "\n")
cat("  MTX:", MTX_DIR, "\n")
cat("  Seurat:", output_file, "\n")
cat("\nNext: Module 05 (DEG Analysis)\n")
cat("================================================================================\n")

---

# 04D: Microglia Subclustering & Annotation (R)

**Purpose:** Subset microglia, reprocess, annotate functional states, and analyze senescence patterns.

**Method:**
1. Subset to Microglia cells
2. Reprocess: Normalize → HVG → Scale → PCA → UMAP → Clustering
3. Load Harari Lab integrated marker gene sets
4. AddModuleScore for each signature
5. Assign top signature per cluster
6. Validation: Heatmap, Dotplot
7. Post-subclustering analysis: Composition, %SnC by state

**Reference:** Harari Lab integrated markers

In [ ]:
cat("\n================================================================================\n")
cat("04D: MICROGLIA SUBCLUSTERING & ANNOTATION\n")
cat("================================================================================\n")

# ════════════════════════════════════════════════════════════════════════════════
# LOAD LIBRARIES
# ════════════════════════════════════════════════════════════════════════════════

suppressPackageStartupMessages({
    library(Seurat)
    library(Matrix)
    library(dplyr)
    library(tidyr)
    library(ggplot2)
    library(qs)
    library(pheatmap)
    library(patchwork)
    library(RColorBrewer)
    library(lme4)
    library(lmerTest)
})

cat("\n Libraries loaded\n")

# ════════════════════════════════════════════════════════════════════════════════
# DATASET SELECTION
# ════════════════════════════════════════════════════════════════════════════════

DATASET <- "psychad_aging"  # Options: 'psychad_aging', 'psychad_ad', 'psychencode', 'mathys'

# ════════════════════════════════════════════════════════════════════════════════
# DATASET-SPECIFIC CONFIGURATION
# ════════════════════════════════════════════════════════════════════════════════

DATASET_CONFIG <- list(
    'psychad_aging' = list(
        # Column mappings
        cell_type_col = 'subclass',
        donor_col = 'Sample',
        study_group_col = 'Study_Group',
        sex_col = 'Sex',
        cohort_col = 'Cohort',
        # Study design
        study_type = 'aging',
        primary_var = 'Age',
        primary_var_type = 'continuous',
        covariates = c('Sex', 'Cohort'),
        group_order = c('Age_20_29', 'Age_30_39', 'Age_40_49', 'Age_50_59', 
                        'Age_60_69', 'Age_70_79', 'Age_80_100')
    ),
    'psychad_ad' = list(
        cell_type_col = 'subclass',
        donor_col = 'Sample',
        study_group_col = 'Study_Group',
        sex_col = 'Sex',
        cohort_col = 'Cohort',
        study_type = 'disease',
        primary_var = 'Study_Group',
        primary_var_type = 'categorical',
        reference_group = 'Control',
        covariates = c('Age', 'Sex', 'Cohort'),
        group_order = c('Control', 'MCI', 'AD')
    ),
    'psychencode' = list(
        cell_type_col = 'major_celltype',
        donor_col = 'sample_id',
        study_group_col = 'Study_Group',
        sex_col = 'Sex',
        cohort_col = 'Batch',
        study_type = 'aging',
        primary_var = 'Age_death',
        primary_var_type = 'continuous',
        covariates = c('Sex', 'Batch'),
        group_order = c('Age_20_29', 'Age_30_39', 'Age_40_49', 'Age_50_59', 
                        'Age_60_69', 'Age_70_79', 'Age_80_100')
    ),
    'mathys' = list(
        cell_type_col = 'broad.cell.type',
        donor_col = 'Subject',
        study_group_col = 'Study_Group',
        sex_col = 'sex',
        cohort_col = 'batch',
        study_type = 'disease',
        primary_var = 'Study_Group',
        primary_var_type = 'categorical',
        reference_group = 'Control',
        covariates = c('Age', 'sex', 'batch'),
        group_order = c('Control', 'AD')
    )
)

config <- DATASET_CONFIG[[DATASET]]

# ─────────────────────────────────────────────────────────────────────────────
# COLUMN NAMES
# ─────────────────────────────────────────────────────────────────────────────

CELL_TYPE_COL <- config$cell_type_col
DONOR_COL <- config$donor_col
STUDY_GROUP_COL <- config$study_group_col
SEX_COL <- config$sex_col
COHORT_COL <- config$cohort_col
SENESCENCE_LABEL_COL <- "senescence_label"
SUBCLUSTER_COL <- "microglia_state"  # For microglia state analysis

# ─────────────────────────────────────────────────────────────────────────────
# STUDY DESIGN PARAMETERS
# ─────────────────────────────────────────────────────────────────────────────

STUDY_TYPE <- config$study_type
PRIMARY_VAR <- config$primary_var
PRIMARY_VAR_TYPE <- config$primary_var_type
COVARIATES <- config$covariates
GROUP_ORDER <- config$group_order

# ─────────────────────────────────────────────────────────────────────────────
# PROCESSING PARAMETERS
# ─────────────────────────────────────────────────────────────────────────────

N_VARIABLE_FEATURES <- 2000
N_PCS <- 50
N_DIMS_USE <- 30
CLUSTERING_RESOLUTION <- 0.7

# ════════════════════════════════════════════════════════════════════════════════
# COLOR PALETTES (project-wide consistency)
# ════════════════════════════════════════════════════════════════════════════════

# Study Group colors - sequential cool -> warm for age progression
STUDY_GROUP_COLORS <- c(
    "Age_20_29"  = "#2E86AB",
    "Age_30_39"  = "#4A90E2",
    "Age_40_49"  = "#50C878",
    "Age_50_59"  = "#FFB347",
    "Age_60_69"  = "#FF8C00",
    "Age_70_79"  = "#E24A4A",
    "Age_80_100" = "#8B0000",
    # Disease cohorts
    "Control"    = "#4E79A7",
    "NCI"        = "#4E79A7",
    "MCI"        = "#F28E2B",
    "AD"         = "#E15759"
)

# Senescence status
SENESCENCE_COLORS <- c(
    "Non-SnC" = "#B0B0B0",
    "SnC"     = "#E41A1C"
)

# Microglia state colors - biologically meaningful, publication-quality
MICROGLIA_STATE_COLORS <- c(
    # Homeostatic / surveillance states
    "Homeostatic"            = "#4E79A7",
    "Neuronal_Surveillance"  = "#59A14F",
    # Interferon-responsive states
    "IFN-I"                  = "#E15759",
    "IFN-II"                 = "#F28E2B",
    "IFN-III"                = "#EDC948",
    # Antigen presentation / immune
    "MCHII"                  = "#76B7B2",
    "Cytokine"               = "#FFBE7D",
    "ILB"                    = "#BAB0AC",
    # Stress / activation states
    "Stress"                 = "#B07AA1",
    "Activated"              = "#FF9DA7",
    # Functional states
    "Phagocytic"             = "#A0CBE8",
    "Lipid_Processing"       = "#D37295",
    # Proliferation / senescence
    "Cycling"                = "#9C755F",
    "Senescent"              = "#8B0000"
)

# Astrocyte state colors (Serrano-Pozo et al. 2024)
ASTROCYTE_STATE_COLORS <- c(
    "Homeostatic"   = "#4E79A7",
    "Intermediate"  = "#76B7B2",
    "Trophic"       = "#59A14F",
    "Metabolic"     = "#EDC948",
    "Reactive_R0"   = "#F28E2B",
    "Reactive_R1"   = "#E15759",
    "Reactive_R2"   = "#8B0000"
)

# Sex colors
SEX_COLORS <- c(
    "Male"   = "#5D6D7E",
    "Female" = "#A569BD",
    "M"      = "#5D6D7E",
    "F"      = "#A569BD"
)

# Vibrant colors for clusters
VIBRANT_COLORS <- c(
    "#E41A1C", "#377EB8", "#4DAF4A", "#984EA3", "#FF7F00",
    "#FFFF33", "#A65628", "#F781BF", "#1B9E77", "#D95F02",
    "#7570B3", "#E7298A", "#66A61E", "#E6AB02"
)

# ─────────────────────────────────────────────────────────────────────────────
# HELPER: Get color with fallback
# ─────────────────────────────────────────────────────────────────────────────

get_state_colors <- function(states, palette, fallback = "#808080") {
    colors <- sapply(states, function(s) {
        if (s %in% names(palette)) palette[[s]] else fallback
    })
    names(colors) <- states
    
    unmapped <- states[!states %in% names(palette)]
    if (length(unmapped) > 0) {
        warning("States not in palette (using fallback): ", paste(unmapped, collapse = ", "))
    }
    
    return(colors)
}

# ════════════════════════════════════════════════════════════════════════════════
# PATHS
# ════════════════════════════════════════════════════════════════════════════════

BASE_DIR <- "/fs/scratch/PAS2598/senescence_analysis"

INPUT_FILE <- file.path(BASE_DIR, "data", "04_microglia", DATASET, paste0(DATASET, "_microglia_annotated.qs"))
OUTPUT_DIR <- file.path(BASE_DIR, "data", "04_microglia", DATASET)
RESULTS_DIR <- file.path(BASE_DIR, "results", "04_microglia", DATASET)
FIGURES_DIR <- file.path(BASE_DIR, "figures", "04_microglia", DATASET)
MARKER_DIR <- file.path(BASE_DIR, "markers", "microglia")

dir.create(OUTPUT_DIR, recursive = TRUE, showWarnings = FALSE)
dir.create(RESULTS_DIR, recursive = TRUE, showWarnings = FALSE)
dir.create(FIGURES_DIR, recursive = TRUE, showWarnings = FALSE)

# ════════════════════════════════════════════════════════════════════════════════
# PRINT CONFIGURATION
# ════════════════════════════════════════════════════════════════════════════════

cat(paste(rep("=", 80), collapse = ""), "\n")
cat("CONFIGURATION\n")
cat(paste(rep("=", 80), collapse = ""), "\n")

cat("\nDataset:", DATASET, "\n")
cat("Study type:", STUDY_TYPE, "\n")
cat("Primary variable:", PRIMARY_VAR, "(", PRIMARY_VAR_TYPE, ")\n")
cat("Covariates:", paste(COVARIATES, collapse = ", "), "\n")
cat("Group order:", paste(GROUP_ORDER, collapse = ", "), "\n")

cat("\nColumn mappings:\n")
cat("  Cell type:", CELL_TYPE_COL, "\n")
cat("  Donor:", DONOR_COL, "\n")
cat("  Study group:", STUDY_GROUP_COL, "\n")
cat("  Sex:", SEX_COL, "\n")
cat("  Cohort:", COHORT_COL, "\n")
cat("  Subcluster:", SUBCLUSTER_COL, "\n")
cat("  Senescence label:", SENESCENCE_LABEL_COL, "\n")

cat("\nProcessing parameters:\n")
cat("  Variable features:", N_VARIABLE_FEATURES, "\n")
cat("  PCs:", N_PCS, "\n")
cat("  Dims for UMAP:", N_DIMS_USE, "\n")
cat("  Clustering resolution:", CLUSTERING_RESOLUTION, "\n")

cat("\nPaths:\n")
cat("  Input:", INPUT_FILE, "\n")
cat("  Output:", OUTPUT_DIR, "\n")
cat("  Results:", RESULTS_DIR, "\n")
cat("  Figures:", FIGURES_DIR, "\n")
cat("  Markers:", MARKER_DIR, "\n")

cat(paste(rep("=", 80), collapse = ""), "\n")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# COLOR PALETTES (project-wide consistency)
# ════════════════════════════════════════════════════════════════════════════════

# Study Group colors - sequential cool → warm for age progression
STUDY_GROUP_COLORS <- c(
    "Age_20_29"  = "#2E86AB",
    "Age_30_39"  = "#4A90E2",
    "Age_40_49"  = "#50C878",
    "Age_50_59"  = "#FFB347",
    "Age_60_69"  = "#FF8C00",
    "Age_70_79"  = "#E24A4A",
    "Age_80_100" = "#8B0000",
    # Disease cohorts
    "Control"    = "#4E79A7",
    "NCI"        = "#4E79A7",
    "MCI"        = "#F28E2B",
    "AD"         = "#E15759"
)

# Senescence status
SENESCENCE_COLORS <- c(
    "Non-SnC" = "#B0B0B0",
    "SnC"     = "#E41A1C"
)

# Microglia state colors - biologically meaningful, publication-quality
MICROGLIA_STATE_COLORS <- c(
    # Homeostatic / surveillance states
    "Homeostatic"            = "#4E79A7",
    "Neuronal_Surveillance"  = "#59A14F",
    # Interferon-responsive states
    "IFN-I"                  = "#E15759",
    "IFN-II"                 = "#F28E2B",
    "IFN-III"                = "#EDC948",
    # Antigen presentation / immune
    "MCHII"                  = "#76B7B2",
    "Cytokine"               = "#FFBE7D",
    "ILB"                    = "#BAB0AC",
    # Stress / activation states
    "Stress"                 = "#B07AA1",
    "Activated"              = "#FF9DA7",
    # Functional states
    "Phagocytic"             = "#A0CBE8",
    "Lipid_Processing"       = "#D37295",
    # Proliferation / senescence
    "Cycling"                = "#9C755F",
    "Senescent"              = "#8B0000"
)

# Astrocyte state colors (Serrano-Pozo et al. 2024)
ASTROCYTE_STATE_COLORS <- c(
    "Homeostatic"   = "#4E79A7",
    "Intermediate"  = "#76B7B2",
    "Trophic"       = "#59A14F",
    "Metabolic"     = "#EDC948",
    "Reactive_R0"   = "#F28E2B",
    "Reactive_R1"   = "#E15759",
    "Reactive_R2"   = "#8B0000"
)

# Sex colors
SEX_COLORS <- c(
    "Male"   = "#5D6D7E",
    "Female" = "#A569BD",
    "M"      = "#5D6D7E",
    "F"      = "#A569BD"
)

# ─────────────────────────────────────────────────────────────────────────────
# HELPER: Get color with fallback
# ─────────────────────────────────────────────────────────────────────────────

get_state_colors <- function(states, palette, fallback = "#808080") {
    # Map states to colors, using fallback for unknown states
    colors <- sapply(states, function(s) {
        if (s %in% names(palette)) palette[[s]] else fallback
    })
    names(colors) <- states
    
    # Report unmapped states
    unmapped <- states[!states %in% names(palette)]
    if (length(unmapped) > 0) {
        warning("States not in palette (using fallback): ", paste(unmapped, collapse = ", "))
    }
    
    return(colors)
}

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# LOAD DATA
# ════════════════════════════════════════════════════════════════════════════════

cat("\n================================================================================\n")
cat("LOADING DATA\n")
cat("================================================================================\n")

data <- qread(INPUT_FILE)

cat("\n✓ Loaded:", ncol(data), "cells,", nrow(data), "genes\n")
cat("  Cell types:", paste(unique(data[[CELL_TYPE_COL, drop = TRUE]]), collapse = ", "), "\n")
cat("  Donors:", length(unique(data[[DONOR_COL, drop = TRUE]])), "\n")

# ════════════════════════════════════════════════════════════════════════════════
# SUBSET TO MICROGLIA
# ════════════════════════════════════════════════════════════════════════════════

cat("\n================================================================================\n")
cat("SUBSETTING TO MICROGLIA\n")
cat("================================================================================\n")

microglia <- subset(data, subset = !!sym(CELL_TYPE_COL) == "Microglia")

cat("\n✓ Microglia subset:", ncol(microglia), "cells\n")
cat("  Donors:", length(unique(microglia[[DONOR_COL, drop = TRUE]])), "\n")
cat("  Study groups:", paste(sort(unique(microglia[[STUDY_GROUP_COL, drop = TRUE]])), collapse = ", "), "\n")
cat("  Senescence:\n")
print(table(microglia[[SENESCENCE_LABEL_COL]]))

# Get study colors for detected groups
study_groups <- sort(unique(microglia[[STUDY_GROUP_COL, drop = TRUE]]))
study_colors <- STUDY_GROUP_COLORS[names(STUDY_GROUP_COLORS) %in% study_groups]

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# REPROCESS MICROGLIA
# ════════════════════════════════════════════════════════════════════════════════

cat("\n================================================================================\n")
cat("REPROCESSING MICROGLIA\n")
cat("================================================================================\n")

# Normalize
cat("\nNormalizing...\n")
microglia <- NormalizeData(microglia, verbose = TRUE)

# Find variable features
cat("Finding variable features...\n")
microglia <- FindVariableFeatures(microglia, nfeatures = N_VARIABLE_FEATURES, verbose = TRUE)

# Scale
cat("Scaling...\n")
microglia <- ScaleData(microglia, verbose = TRUE)

# PCA
cat("Running PCA...\n")
microglia <- RunPCA(microglia, npcs = N_PCS, verbose = TRUE)

# UMAP
cat("Running UMAP...\n")
microglia <- RunUMAP(microglia, dims = 1:N_DIMS_USE, verbose = TRUE)

# Find neighbors
cat("Finding neighbors...\n")
microglia <- FindNeighbors(microglia, dims = 1:N_DIMS_USE, verbose = TRUE)

# Cluster
cat("Clustering (resolution =", CLUSTERING_RESOLUTION, ")...\n")
microglia <- FindClusters(microglia, resolution = CLUSTERING_RESOLUTION, verbose = TRUE)

n_clusters <- length(unique(Idents(microglia)))
cat("\n✓ Reprocessing complete:", n_clusters, "clusters\n")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# LOAD MICROGLIA MARKER SIGNATURES
# ════════════════════════════════════════════════════════════════════════════════

cat("\n================================================================================\n")
cat("LOADING MARKER GENE SETS\n")
cat("================================================================================\n")

# Source location (original marker files)
SOURCE_MARKER_DIR <- "/users/PAS2598/ggaitos/2025/scripts/single-cell/deepsas/psychad_samples/microglia_integrated_markers"

# Check if files already copied
existing_files <- list.files(MARKER_DIR, pattern = "^Integrated_.*\\.txt$")

if (length(existing_files) == 0) {
    cat("Copying marker files from source...\n")
    
    source_files <- list.files(SOURCE_MARKER_DIR, pattern = "^Integrated_.*\\.txt$", full.names = TRUE)
    
    if (length(source_files) == 0) {
        stop("No marker files found in source: ", SOURCE_MARKER_DIR)
    }
    
    for (f in source_files) {
        file.copy(f, MARKER_DIR, overwrite = FALSE)
        cat("  ✓ Copied:", basename(f), "\n")
    }
    
    cat("\n✓ Copied", length(source_files), "marker files\n")
} else {
    cat("✓ Marker files already present:", length(existing_files), "files\n")
}

# ════════════════════════════════════════════════════════════════════════════════
# LOAD GENE SIGNATURES
# ════════════════════════════════════════════════════════════════════════════════

cat("\nLoading gene signatures...\n")

gene_set_files <- list.files(MARKER_DIR, pattern = "^Integrated_.*\\.txt$", full.names = TRUE)

if (length(gene_set_files) == 0) {
    stop("No marker files found in: ", MARKER_DIR)
}

gene_sets <- list()

for (f in gene_set_files) {
    sig_name <- gsub("Integrated_", "", gsub(".txt", "", basename(f)))
    genes <- read.table(f, header = FALSE, stringsAsFactors = FALSE)[, 1]
    gene_sets[[sig_name]] <- genes
    cat(sprintf("  %s: %d genes\n", sig_name, length(genes)))
}

cat("\n✓ Loaded", length(gene_sets), "signatures\n")

# ════════════════════════════════════════════════════════════════════════════════
# ADD MODULE SCORES
# ════════════════════════════════════════════════════════════════════════════════

cat("\n================================================================================\n")
cat("CALCULATING MODULE SCORES\n")
cat("================================================================================\n")

DefaultAssay(microglia) <- "RNA"

for (sig_name in names(gene_sets)) {
    genes <- gene_sets[[sig_name]]
    genes_present <- intersect(genes, rownames(microglia))
    
    cat(sprintf("  %s: %d/%d genes present\n", sig_name, length(genes_present), length(genes)))
    
    if (length(genes_present) > 0) {
        microglia <- AddModuleScore(
            object = microglia,
            features = list(genes_present),
            name = sig_name,
            seed = 42
        )
    }
}

cat("\n✓ Module scores calculated\n")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# CALCULATE MEAN SCORES PER CLUSTER
# ════════════════════════════════════════════════════════════════════════════════

cat("\n================================================================================\n")
cat("ANNOTATING CLUSTERS\n")
cat("================================================================================\n")

# Get score columns
all_cols <- colnames(microglia@meta.data)
score_cols <- c()

for (sig_name in names(gene_sets)) {
    pattern <- paste0("^", sig_name, "[0-9]+$")
    matching <- grep(pattern, all_cols, value = TRUE)
    if (length(matching) > 0) {
        score_cols <- c(score_cols, matching[1])
    }
}

cat("Score columns found:", length(score_cols), "\n")

# Calculate mean per cluster
cluster_scores <- data.frame(cluster = unique(microglia$seurat_clusters))
rownames(cluster_scores) <- cluster_scores$cluster

for (col in score_cols) {
    means <- tapply(microglia@meta.data[[col]], microglia$seurat_clusters, mean)
    cluster_scores[[col]] <- means[as.character(cluster_scores$cluster)]
}

# Clean up column names
cluster_scores <- cluster_scores[, -1]
colnames(cluster_scores) <- gsub("[0-9]+$", "", colnames(cluster_scores))

cat("\nMean signature scores per cluster:\n")
print(round(cluster_scores, 3))

# Save scores
scores_file <- file.path(RESULTS_DIR, paste0(DATASET, "_microglia_cluster_scores.csv"))
write.csv(cluster_scores, scores_file)
cat("\n✓ Saved:", scores_file, "\n")

# ════════════════════════════════════════════════════════════════════════════════
# ASSIGN TOP SIGNATURE PER CLUSTER
# ════════════════════════════════════════════════════════════════════════════════

top_signatures <- data.frame(
    cluster = rownames(cluster_scores),
    top_signature = colnames(cluster_scores)[apply(cluster_scores, 1, which.max)],
    top_score = apply(cluster_scores, 1, max),
    row.names = NULL
)

cat("\nTop signature per cluster:\n")
print(top_signatures)

# Save
top_sig_file <- file.path(RESULTS_DIR, paste0(DATASET, "_microglia_top_signatures.csv"))
write.csv(top_signatures, top_sig_file, row.names = FALSE)
cat("\n✓ Saved:", top_sig_file, "\n")

# ════════════════════════════════════════════════════════════════════════════════
# ADD ANNOTATIONS TO METADATA
# ════════════════════════════════════════════════════════════════════════════════

annotation_lookup <- setNames(top_signatures$top_signature, top_signatures$cluster)
microglia$microglia_state <- as.character(annotation_lookup[as.character(microglia$seurat_clusters)])

cat("\nState distribution:\n")
print(table(microglia$microglia_state))

# Get state colors
states <- sort(unique(microglia$microglia_state))
state_colors <- setNames(VIBRANT_COLORS[1:length(states)], states)

# ════════════════════════════════════════════════════════════════════════════════
# SAVE ANNOTATED OBJECT
# ════════════════════════════════════════════════════════════════════════════════

microglia_file <- file.path(OUTPUT_DIR, paste0(DATASET, "_microglia_annotated.qs"))
qsave(microglia, microglia_file)

cat("\n✓ Microglia object saved:", microglia_file, "\n")
cat("  Cells:", ncol(microglia), "\n")
cat("  States:", length(unique(microglia$microglia_state)), "\n")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# VALIDATION: SIGNATURE HEATMAP
# ════════════════════════════════════════════════════════════════════════════════

cat("\n================================================================================\n")
cat("VALIDATION: SIGNATURE HEATMAP\n")
cat("================================================================================\n")

# Z-scale the scores for visualization
cluster_scores_scaled <- t(scale(t(as.matrix(cluster_scores))))

# Order clusters numerically
cluster_order <- as.character(sort(as.numeric(rownames(cluster_scores_scaled))))
cluster_scores_scaled <- cluster_scores_scaled[cluster_order, ]

options(repr.plot.width = 10, repr.plot.height = 8)

p_heatmap <- pheatmap(
    cluster_scores_scaled,
    cluster_rows = FALSE,
    cluster_cols = TRUE,
    color = colorRampPalette(c("#313695", "#4575B4", "#74ADD1", "#FFFFBF", "#FDAE61", "#F46D43", "#A50026"))(100),
    breaks = seq(-2.5, 2.5, length.out = 101),
    fontsize = 10,
    fontsize_row = 9,
    fontsize_col = 9,
    angle_col = 45,
    border_color = "white",
    cellwidth = 30,
    cellheight = 18,
    main = paste0("Microglia Signature Scores (Z-scaled) - ", DATASET),
    silent = TRUE
)

print(p_heatmap)

heatmap_file <- file.path(FIGURES_DIR, paste0(DATASET, "_microglia_signature_heatmap.svg"))
pdf(heatmap_file, width = 10, height = 8)
print(p_heatmap)
dev.off()

cat("✓ Saved:", heatmap_file, "\n")

# ════════════════════════════════════════════════════════════════════════════════
# VALIDATION: CANONICAL MARKERS DOTPLOT
# ════════════════════════════════════════════════════════════════════════════════

cat("\n================================================================================\n")
cat("VALIDATION: CANONICAL MARKERS DOTPLOT\n")
cat("================================================================================\n")

# Define canonical markers (top 2-3 per state)
canonical_markers <- list(
    'Homeostatic' = c('TMEM119', 'P2RY12', 'CX3CR1'),
    'Activated' = c('CD68', 'APOE', 'TREM2'),
    'IFN-I' = c('IFITM3', 'IFIT3', 'ISG15'),
    'IFN-II' = c('GBP2', 'STAT1'),
    'IFN-III' = c('IFI44L', 'IRF7'),
    'MHCII' = c('HLA-DRA', 'HLA-DRB1', 'CD74'),
    'Stress' = c('HSP90AA1', 'HSPA1A', 'FOS'),
    'Neuronal_Surveillance' = c('MT-ND3', 'MT-CYB', 'MT-CO2'),
    'Lipid_Processing' = c('APOC1', 'LPL', 'ABCA1')
)

# Filter to available genes
canonical_markers_filtered <- lapply(canonical_markers, function(genes) {
    genes[genes %in% rownames(microglia)]
})
canonical_markers_filtered <- canonical_markers_filtered[sapply(canonical_markers_filtered, length) > 0]

cat("\nMarkers available:\n")
for (ct in names(canonical_markers_filtered)) {
    cat(sprintf("  %s: %s\n", ct, paste(canonical_markers_filtered[[ct]], collapse = ", ")))
}

# Get all markers
all_markers <- unlist(canonical_markers_filtered)

# Set identity and get cluster order
Idents(microglia) <- "seurat_clusters"
cluster_order <- as.character(sort(as.numeric(levels(Idents(microglia)))))

# Calculate expression data
cat("\nCalculating expression statistics...\n")

# Initialize empty list to collect data
dot_list <- list()
idx <- 1

for (cluster in cluster_order) {
    cells <- WhichCells(microglia, idents = cluster)
    if (length(cells) == 0) next
    
    for (category in names(canonical_markers_filtered)) {
        for (gene in canonical_markers_filtered[[category]]) {
            expr <- GetAssayData(microglia, slot = "data")[gene, cells]
            
            dot_list[[idx]] <- data.frame(
                Cluster = cluster,
                Gene = gene,
                Category = category,
                Pct_Exp = sum(expr > 0) / length(expr) * 100,
                Avg_Exp = mean(expr),
                stringsAsFactors = FALSE
            )
            idx <- idx + 1
        }
    }
}

# Combine into data frame
dot_data <- do.call(rbind, dot_list)

cat("✓ Collected", nrow(dot_data), "data points\n")

# Scale expression per gene
dot_data <- dot_data %>%
    group_by(Gene) %>%
    mutate(Scaled_Exp = as.numeric(scale(Avg_Exp))) %>%
    ungroup()

# Cap values
dot_data$Scaled_Exp <- pmax(pmin(dot_data$Scaled_Exp, 2.5), -2.5)

# Set factor levels
dot_data$Cluster <- factor(dot_data$Cluster, levels = rev(cluster_order))
dot_data$Gene <- factor(dot_data$Gene, levels = all_markers)
dot_data$Category <- factor(dot_data$Category, levels = names(canonical_markers_filtered))

cat("✓ Expression data ready for", length(unique(dot_data$Gene)), 
    "genes across", length(unique(dot_data$Cluster)), "clusters\n")

# Create dotplot
options(repr.plot.width = 12, repr.plot.height = 10)

p_dotplot <- ggplot(dot_data, aes(x = Gene, y = Cluster)) +
    geom_point(aes(size = Pct_Exp, fill = Scaled_Exp), shape = 21, color = "black", stroke = 0.3) +
    scale_size_continuous(
        range = c(1, 6),
        limits = c(0, 100),
        breaks = c(0, 25, 50, 75, 100),
        name = "% Expr"
    ) +
    scale_fill_gradientn(
        colors = c("#313695", "#4575B4", "#74ADD1", "#FFFFBF", "#FDAE61", "#F46D43", "#A50026"),
        limits = c(-2.5, 2.5),
        name = "Scaled\nExpr"
    ) +
    facet_grid(cols = vars(Category), scales = "free_x", space = "free_x") +
    labs(x = NULL, y = "Cluster") +
    theme_bw(base_size = 10) +
    theme(
        axis.text.x = element_text(angle = 45, hjust = 1, size = 8, face = "italic"),
        axis.text.y = element_text(size = 9),
        axis.title.y = element_text(size = 10),
        strip.text = element_text(size = 8, face = "bold"),
        strip.background = element_rect(fill = "gray95"),
        panel.grid = element_blank(),
        panel.spacing = unit(0.2, "lines"),
        legend.position = "right",
        legend.title = element_text(size = 8),
        legend.text = element_text(size = 7),
        legend.key.size = unit(0.4, "cm"),
        plot.margin = margin(10, 10, 10, 10)
    )

print(p_dotplot)

dotplot_file <- file.path(FIGURES_DIR, paste0(DATASET, "_microglia_dotplot_canonical.svg"))
ggsave(dotplot_file, p_dotplot, width = 12, height = 10)

cat("✓ Saved:", dotplot_file, "\n")

In [ ]:
ls()

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# SENESCENCE RE-THRESHOLDING AT CELL STATE LEVEL
# ════════════════════════════════════════════════════════════════════════════════
#
# Purpose: 
#   Module 01 applied thresholds at subclass level. After subclustering,
#   re-threshold per cell state to see if state-specific thresholds reveal
#   different senescence patterns.
#
# New columns created:
#   - is_senescent_state     (logical)
#   - senescence_label_state (SnC / Non-SnC)
#
# Uses existing variables from notebook:
#   - microglia (Seurat object — rename for other cell types)
#   - SUBCLUSTER_COL, STUDY_GROUP_COL, SENESCENCE_LABEL_COL
#   - STUDY_TYPE, GROUP_ORDER, config
#
# ════════════════════════════════════════════════════════════════════════════════

cat("\n", paste(rep("=", 80), collapse = ""), "\n")
cat("SENESCENCE RE-THRESHOLDING AT CELL STATE LEVEL\n")
cat(paste(rep("=", 80), collapse = ""), "\n")

# ─────────────────────────────────────────────────────────────────────────────
# CONFIGURATION
# ─────────────────────────────────────────────────────────────────────────────

# ── Seurat object to rethreshold ──
# Change this when running for other cell types:
#   seurat_obj <- astrocyte
#   seurat_obj <- opc
seurat_obj <- microglia

SD_THRESHOLD <- 2
SCORE_COL <- "senescence_score"

# Reference group: youngest for aging, control for disease
if (STUDY_TYPE == "aging") {
    REFERENCE_GROUP <- GROUP_ORDER[1]
} else {
    REFERENCE_GROUP <- config$reference_group
}

cat(sprintf("\n  Seurat object: %s cells\n", format(ncol(seurat_obj), big.mark = ",")))
cat(sprintf("  Score column: %s\n", SCORE_COL))
cat(sprintf("  State column: %s\n", SUBCLUSTER_COL))
cat(sprintf("  Study type: %s\n", STUDY_TYPE))
cat(sprintf("  Reference group: %s\n", REFERENCE_GROUP))
cat(sprintf("  Threshold: Mean + %s SD per cell state\n", SD_THRESHOLD))

# ─────────────────────────────────────────────────────────────────────────────
# VALIDATE
# ─────────────────────────────────────────────────────────────────────────────

# Check required columns
required_cols <- c(SCORE_COL, SENESCENCE_LABEL_COL, STUDY_GROUP_COL, SUBCLUSTER_COL)
missing <- required_cols[!required_cols %in% colnames(seurat_obj@meta.data)]
if (length(missing) > 0) {
    stop(sprintf("Missing columns: %s", paste(missing, collapse = ", ")))
}

# Check reference group
ref_mask <- seurat_obj@meta.data[[STUDY_GROUP_COL]] == REFERENCE_GROUP
n_ref <- sum(ref_mask)
if (n_ref == 0) {
    stop(sprintf("Reference group '%s' not found! Available: %s",
                 REFERENCE_GROUP,
                 paste(unique(seurat_obj@meta.data[[STUDY_GROUP_COL]]), collapse = ", ")))
}
cat(sprintf("  Reference cells: %s\n", format(n_ref, big.mark = ",")))

# ─────────────────────────────────────────────────────────────────────────────
# ORIGINAL (SUBCLASS-LEVEL) SUMMARY
# ─────────────────────────────────────────────────────────────────────────────

orig_snc <- sum(seurat_obj@meta.data[[SENESCENCE_LABEL_COL]] == "SnC")
n_total <- nrow(seurat_obj@meta.data)

cat(sprintf("\n  Original (subclass-level): %s SnC / %s (%.1f%%)\n",
            format(orig_snc, big.mark = ","),
            format(n_total, big.mark = ","),
            orig_snc / n_total * 100))

# ─────────────────────────────────────────────────────────────────────────────
# CALCULATE STATE-LEVEL THRESHOLDS
# ─────────────────────────────────────────────────────────────────────────────

cat("\n  Calculating thresholds per cell state...\n")

states <- unique(seurat_obj@meta.data[[SUBCLUSTER_COL]])
thresholds <- list()

for (state in states) {
    state_ref_mask <- ref_mask & (seurat_obj@meta.data[[SUBCLUSTER_COL]] == state)
    ref_scores <- seurat_obj@meta.data[state_ref_mask, SCORE_COL]
    
    if (length(ref_scores) >= 10) {
        # Enough reference cells — use state-specific threshold
        state_mean <- mean(ref_scores)
        state_sd <- sd(ref_scores)
        thresholds[[state]] <- state_mean + (SD_THRESHOLD * state_sd)
        
        cat(sprintf("    %s:\n", state))
        cat(sprintf("      n_ref=%s, mean=%.4f, sd=%.4f, threshold=%.4f\n",
                    format(length(ref_scores), big.mark = ","),
                    state_mean, state_sd, thresholds[[state]]))
    } else {
        # Fallback: use ALL cells of this state (not just reference)
        all_state_scores <- seurat_obj@meta.data[
            seurat_obj@meta.data[[SUBCLUSTER_COL]] == state, SCORE_COL
        ]
        state_mean <- mean(all_state_scores)
        state_sd <- sd(all_state_scores)
        thresholds[[state]] <- state_mean + (SD_THRESHOLD * state_sd)
        
        cat(sprintf("    %s: (fallback — only %d ref cells, using all %s cells)\n",
                    state, length(ref_scores),
                    format(length(all_state_scores), big.mark = ",")))
        cat(sprintf("      mean=%.4f, sd=%.4f, threshold=%.4f\n",
                    state_mean, state_sd, thresholds[[state]]))
    }
}

# ─────────────────────────────────────────────────────────────────────────────
# APPLY STATE-LEVEL THRESHOLDS
# ─────────────────────────────────────────────────────────────────────────────

cat("\n  Applying state-level thresholds...\n")

seurat_obj@meta.data$is_senescent_state <- FALSE

for (state in names(thresholds)) {
    mask <- (seurat_obj@meta.data[[SUBCLUSTER_COL]] == state) &
            (seurat_obj@meta.data[[SCORE_COL]] >= thresholds[[state]])
    seurat_obj@meta.data$is_senescent_state[mask] <- TRUE
}

seurat_obj@meta.data$senescence_label_state <- ifelse(
    seurat_obj@meta.data$is_senescent_state, "SnC", "Non-SnC"
)

# ─────────────────────────────────────────────────────────────────────────────
# COMPARISON: ORIGINAL VS STATE-LEVEL
# ─────────────────────────────────────────────────────────────────────────────

new_snc <- sum(seurat_obj@meta.data$is_senescent_state)

cat("\n  ✓ Re-thresholding complete\n")
cat(sprintf("\n  %-25s %12s %12s\n", "", "Subclass", "State-level"))
cat(paste(rep("-", 55), collapse = ""), "\n")
cat(sprintf("  %-25s %12s %12s\n",
            "Total SnC",
            format(orig_snc, big.mark = ","),
            format(new_snc, big.mark = ",")))
cat(sprintf("  %-25s %11.1f%% %11.1f%%\n",
            "%SnC overall",
            orig_snc / n_total * 100,
            new_snc / n_total * 100))

# Per-state breakdown
cat("\n  Per state:\n")
cat(sprintf("  %-25s %6s %8s %8s %8s %8s %8s\n",
            "State", "Total", "Orig_n", "Orig_%", "New_n", "New_%", "Change"))
cat(paste(rep("-", 80), collapse = ""), "\n")

for (state in sort(states)) {
    state_mask <- seurat_obj@meta.data[[SUBCLUSTER_COL]] == state
    state_total <- sum(state_mask)
    
    orig_n <- sum(state_mask & (seurat_obj@meta.data[[SENESCENCE_LABEL_COL]] == "SnC"))
    new_n <- sum(state_mask & seurat_obj@meta.data$is_senescent_state)
    
    orig_pct <- orig_n / state_total * 100
    new_pct <- new_n / state_total * 100
    change <- new_pct - orig_pct
    
    flag <- if (abs(change) > 2) sprintf("%+.1f ←", change) else sprintf("%+.1f", change)
    
    cat(sprintf("  %-25s %6s %8s %7.1f%% %8s %7.1f%% %8s\n",
                state,
                format(state_total, big.mark = ","),
                format(orig_n, big.mark = ","), orig_pct,
                format(new_n, big.mark = ","), new_pct,
                flag))
}

# ─────────────────────────────────────────────────────────────────────────────
# WRITE BACK TO ORIGINAL OBJECT
# ─────────────────────────────────────────────────────────────────────────────

# Write back to the named object (microglia/astrocyte/opc)
microglia <- seurat_obj

cat("\n  ✓ Columns added: is_senescent_state, senescence_label_state\n")
cat(paste(rep("=", 80), collapse = ""), "\n")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# ACTIVATE STATE-LEVEL LABELS FOR DOWNSTREAM PLOTS
# ─────────────────────────────────────────────────────────────────────────────

SENESCENCE_LABEL_COL <- "senescence_label_state"

# Write back from seurat_obj to the named object
microglia <- seurat_obj  # Change to: astrocyte <- seurat_obj / opc <- seurat_obj

cat(sprintf("\n  Active senescence column: %s\n", SENESCENCE_LABEL_COL))
cat(sprintf("  SnC: %s\n", format(sum(seurat_obj@meta.data[[SENESCENCE_LABEL_COL]] == "SnC"), big.mark = ",")))
cat(sprintf("  Non-SnC: %s\n", format(sum(seurat_obj@meta.data[[SENESCENCE_LABEL_COL]] == "Non-SnC"), big.mark = ",")))
cat("\n  All downstream plots will use state-level labels.\n")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# VALIDATION: UMAP PANEL
# ════════════════════════════════════════════════════════════════════════════════

cat("\n================================================================================\n")
cat("VALIDATION: UMAP PANEL\n")
cat("================================================================================\n")

# Helper function: add corner arrows (scanpy style)
add_corner_arrows <- function(p, label_size = 2.5) {
    build <- ggplot_build(p)
    x_range <- build$layout$panel_params[[1]]$x.range
    y_range <- build$layout$panel_params[[1]]$y.range
    
    x_start <- x_range[1] + diff(x_range) * 0.02
    y_start <- y_range[1] + diff(y_range) * 0.02
    x_arrow <- diff(x_range) * 0.10
    y_arrow <- diff(y_range) * 0.10
    
    p + 
        annotate("segment", x = x_start, xend = x_start + x_arrow, y = y_start, yend = y_start,
                 arrow = arrow(length = unit(0.12, "cm"), type = "closed"), linewidth = 0.4) +
        annotate("text", x = x_start + x_arrow/2, y = y_start - diff(y_range) * 0.025,
                 label = "UMAP1", size = label_size, hjust = 0.5, vjust = 1) +
        annotate("segment", x = x_start, xend = x_start, y = y_start, yend = y_start + y_arrow,
                 arrow = arrow(length = unit(0.12, "cm"), type = "closed"), linewidth = 0.4) +
        annotate("text", x = x_start - diff(x_range) * 0.025, y = y_start + y_arrow/2,
                 label = "UMAP2", size = label_size, hjust = 1, vjust = 0.5, angle = 90) +
        coord_cartesian(clip = "off")
}

options(repr.plot.width = 14, repr.plot.height = 12)

umap_theme <- theme_void(base_size = 10) +
    theme(
        plot.title = element_text(size = 11, face = "bold", hjust = 0.5),
        legend.position = "right",
        legend.title = element_blank(),
        legend.text = element_text(size = 8),
        legend.key.size = unit(0.3, "cm"),
        plot.margin = margin(10, 10, 15, 15)
    )

# Clusters
p1 <- DimPlot(microglia, reduction = "umap", group.by = "seurat_clusters", 
              label = TRUE, label.size = 3, pt.size = 0.3) +
    ggtitle("Clusters") + umap_theme + theme(legend.position = "none")
p1 <- add_corner_arrows(p1)

# States
p2 <- DimPlot(microglia, reduction = "umap", group.by = "microglia_state", 
              label = TRUE, label.size = 2.5, repel = TRUE, pt.size = 0.3) +
    ggtitle("Microglia States") + umap_theme
p2 <- add_corner_arrows(p2)

# Senescence
p3 <- DimPlot(microglia, reduction = "umap", group.by = SENESCENCE_LABEL_COL, 
              pt.size = 0.3, order = c("SnC", "Non-SnC")) +
    scale_color_manual(values = SENESCENCE_COLORS) +
    ggtitle("Senescence") + umap_theme
p3 <- add_corner_arrows(p3)

# Study Group
p4 <- DimPlot(microglia, reduction = "umap", group.by = STUDY_GROUP_COL, pt.size = 0.3) +
    scale_color_manual(values = study_colors) +
    ggtitle("Study Group") + umap_theme
p4 <- add_corner_arrows(p4)

# Combine
umap_panel <- (p1 | p2) / (p3 | p4) +
    plot_annotation(
        title = paste0("Microglia Subclustering - ", DATASET),
        theme = theme(plot.title = element_text(size = 14, face = "bold", hjust = 0.5))
    )

print(umap_panel)

umap_file <- file.path(FIGURES_DIR, paste0(DATASET, "_microglia_umap_panel.svg"))
ggsave(umap_file, umap_panel, width = 14, height = 12)

cat("✓ Saved:", umap_file, "\n")

cat("\n================================================================================\n")
cat("VALIDATION COMPLETE\n")
cat("================================================================================\n")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# 04D5: POST-SUBCLUSTERING ANALYSIS
# ════════════════════════════════════════════════════════════════════════════════

cat("\n================================================================================\n")
cat("04D5: POST-SUBCLUSTERING ANALYSIS\n")
cat("================================================================================\n")

# ─────────────────────────────────────────────────────────────────────────────
# SETUP
# ─────────────────────────────────────────────────────────────────────────────

SUBCLUSTER_COL <- "microglia_state"

# Get states ordered by frequency
state_order <- microglia@meta.data %>%
    count(.data[[SUBCLUSTER_COL]]) %>%
    arrange(desc(n)) %>%
    pull(.data[[SUBCLUSTER_COL]])

n_states <- length(state_order)

# Map colors using the standardized palette
state_colors <- get_state_colors(state_order, MICROGLIA_STATE_COLORS)

cat("\nStates detected:", n_states, "\n")
cat("  ", paste(state_order, collapse = ", "), "\n")
cat("Colors mapped:", sum(state_order %in% names(MICROGLIA_STATE_COLORS)), "/", n_states, "\n")

# ════════════════════════════════════════════════════════════════════════════════
# PLOT 1: COMPOSITION BY STUDY GROUP
# ════════════════════════════════════════════════════════════════════════════════

cat("\n--- Plot 1: Composition by Study Group ---\n")

# Calculate composition
comp_studygroup <- microglia@meta.data %>%
    group_by(.data[[STUDY_GROUP_COL]], .data[[SUBCLUSTER_COL]]) %>%
    summarise(n = n(), .groups = "drop") %>%
    group_by(.data[[STUDY_GROUP_COL]]) %>%
    mutate(pct = n / sum(n) * 100) %>%
    ungroup()

colnames(comp_studygroup)[1:2] <- c("Study_Group", "State")

# ─────────────────────────────────────────────────────────────────────────────
# TABLE OUTPUT
# ─────────────────────────────────────────────────────────────────────────────

comp_wide <- comp_studygroup %>%
    select(Study_Group, State, pct) %>%
    pivot_wider(names_from = Study_Group, values_from = pct, values_fill = 0) %>%
    mutate(across(where(is.numeric), ~ round(.x, 1)))

cat("\nState Composition by Study Group (%):\n")
print(as.data.frame(comp_wide))

write.csv(comp_wide, 
          file.path(RESULTS_DIR, paste0(DATASET, "_microglia_composition_by_studygroup.csv")),
          row.names = FALSE)

# ─────────────────────────────────────────────────────────────────────────────
# PREPARE FACTORS
# ─────────────────────────────────────────────────────────────────────────────

# Detect cohort type
sg_detected <- unique(comp_studygroup$Study_Group)
is_aging <- any(grepl("^Age_", sg_detected))

# Order study groups using standardized palette order
sg_order <- names(STUDY_GROUP_COLORS)[names(STUDY_GROUP_COLORS) %in% sg_detected]
comp_studygroup$Study_Group <- factor(comp_studygroup$Study_Group, levels = sg_order)
comp_studygroup$State <- factor(comp_studygroup$State, levels = rev(state_order))

# Create readable x-axis labels based on cohort type
if (is_aging) {
    # Age_20_29 → "20-29" (using regular hyphen for compatibility)
    x_labels <- setNames(
        gsub("Age_", "", gsub("_", "-", sg_order)),
        sg_order
    )
    x_title <- "Age Group (years)"
} else {
    # Disease labels - keep as-is or clean up underscores
    x_labels <- setNames(
        gsub("_", " ", sg_order),
        sg_order
    )
    x_title <- "Disease Status"
}

# ─────────────────────────────────────────────────────────────────────────────
# PLOT
# ─────────────────────────────────────────────────────────────────────────────

options(repr.plot.width = 8, repr.plot.height = 6)

p_comp_sg <- ggplot(comp_studygroup, aes(x = Study_Group, y = pct, fill = State)) +
    geom_col(width = 0.75, color = "white", linewidth = 0.3) +
    scale_fill_manual(values = state_colors) +
    scale_x_discrete(labels = x_labels) +
    scale_y_continuous(expand = expansion(mult = c(0, 0.02))) +
    labs(
        x = x_title, 
        y = "Proportion (%)",
        fill = "Microglia State"
    ) +
    theme_minimal(base_size = 14) +
    theme(
        # Axis text - larger fonts
        axis.text.x = element_text(size = 13, color = "black", face = "bold",
                                   angle = 45, hjust = 1, vjust = 1),
        axis.text.y = element_text(size = 12, color = "black"),
        # Axis titles - larger and bold
        axis.title.x = element_text(size = 15, face = "bold", margin = margin(t = 12)),
        axis.title.y = element_text(size = 15, face = "bold", margin = margin(r = 12)),
        # Axis lines
        axis.line = element_line(color = "black", linewidth = 0.5),
        axis.ticks = element_line(color = "black", linewidth = 0.3),
        axis.ticks.length = unit(0.15, "cm"),
        # Legend
        legend.position = "right",
        legend.title = element_text(size = 12, face = "bold"),
        legend.text = element_text(size = 10),
        legend.key.size = unit(0.5, "cm"),
        # Background
        panel.grid = element_blank(),
        panel.background = element_blank(),
        plot.margin = margin(15, 15, 15, 15)
    ) +
    guides(fill = guide_legend(ncol = 1, reverse = TRUE))

print(p_comp_sg)

ggsave(file.path(FIGURES_DIR, paste0(DATASET, "_microglia_composition_by_studygroup.svg")), 
       p_comp_sg, width = 8, height = 6, dpi = 300)

cat("\n✓ Saved:", paste0(DATASET, "_microglia_composition_by_studygroup.svg"), "\n")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# PLOT: SnC COMPOSITION BY STUDY GROUP (MICROGLIA)
# ════════════════════════════════════════════════════════════════════════════════

cat("\n--- Plot: SnC Composition by Study Group (Microglia) ---\n")

# Calculate composition of SnC cells only
snc_comp_sg <- microglia@meta.data %>%
    filter(.data[[SENESCENCE_LABEL_COL]] == "SnC") %>%
    group_by(.data[[STUDY_GROUP_COL]], .data[[SUBCLUSTER_COL]]) %>%
    summarise(n = n(), .groups = "drop") %>%
    group_by(.data[[STUDY_GROUP_COL]]) %>%
    mutate(pct = n / sum(n) * 100) %>%
    ungroup()

colnames(snc_comp_sg)[1:2] <- c("Study_Group", "State")

# ─────────────────────────────────────────────────────────────────────────────
# PREPARE FACTORS
# ─────────────────────────────────────────────────────────────────────────────

# Detect cohort type
sg_detected <- unique(snc_comp_sg$Study_Group)
is_aging <- any(grepl("^Age_", sg_detected))

# Order study groups using standardized palette order
sg_order <- names(STUDY_GROUP_COLORS)[names(STUDY_GROUP_COLORS) %in% sg_detected]
snc_comp_sg$Study_Group <- factor(snc_comp_sg$Study_Group, levels = sg_order)
snc_comp_sg$State <- factor(snc_comp_sg$State, levels = rev(state_order))

# Create readable x-axis labels based on cohort type
if (is_aging) {
    # Age_20_29 → "20-29" (using regular hyphen for compatibility)
    x_labels <- setNames(
        gsub("Age_", "", gsub("_", "-", sg_order)),
        sg_order
    )
    x_title <- "Age Group (years)"
} else {
    # Disease labels - keep as-is or clean up underscores
    x_labels <- setNames(
        gsub("_", " ", sg_order),
        sg_order
    )
    x_title <- "Disease Status"
}

# ─────────────────────────────────────────────────────────────────────────────
# TABLE OUTPUT
# ─────────────────────────────────────────────────────────────────────────────

snc_comp_sg_wide <- snc_comp_sg %>%
    select(Study_Group, State, pct) %>%
    pivot_wider(names_from = Study_Group, values_from = pct, values_fill = 0) %>%
    mutate(across(where(is.numeric), ~ round(.x, 1)))

cat("\nSnC Composition by Study Group (%):\n")
print(as.data.frame(snc_comp_sg_wide))

write.csv(snc_comp_sg_wide, 
          file.path(RESULTS_DIR, paste0(DATASET, "_microglia_snc_composition_by_studygroup.csv")),
          row.names = FALSE)

# Get total SnC per study group
snc_totals <- microglia@meta.data %>%
    filter(.data[[SENESCENCE_LABEL_COL]] == "SnC") %>%
    group_by(.data[[STUDY_GROUP_COL]]) %>%
    summarise(n = n(), .groups = "drop")

colnames(snc_totals)[1] <- "Study_Group"

cat("\nTotal SnC cells per study group:\n")
print(as.data.frame(snc_totals))

# ─────────────────────────────────────────────────────────────────────────────
# PLOT
# ─────────────────────────────────────────────────────────────────────────────

options(repr.plot.width = 8, repr.plot.height = 6)

p_snc_comp_sg <- ggplot(snc_comp_sg, aes(x = Study_Group, y = pct, fill = State)) +
    geom_col(width = 0.75, color = "white", linewidth = 0.3) +
    scale_fill_manual(values = state_colors) +
    scale_x_discrete(labels = x_labels) +
    scale_y_continuous(limits = c(0, 100), expand = c(0, 0)) +
    labs(
        x = x_title, 
        y = "Proportion of SnC Microglia (%)",
        fill = "Microglia State"
    ) +
    theme_minimal(base_size = 14) +
    theme(
        # Axis text - larger fonts
        axis.text.x = element_text(size = 13, color = "black", face = "bold",
                                   angle = 45, hjust = 1, vjust = 1),
        axis.text.y = element_text(size = 12, color = "black"),
        # Axis titles - larger and bold
        axis.title.x = element_text(size = 15, face = "bold", margin = margin(t = 12)),
        axis.title.y = element_text(size = 15, face = "bold", margin = margin(r = 12)),
        # Axis lines
        axis.line = element_line(color = "black", linewidth = 0.5),
        axis.ticks = element_line(color = "black", linewidth = 0.3),
        axis.ticks.length = unit(0.15, "cm"),
        # Legend
        legend.position = "right",
        legend.title = element_text(size = 12, face = "bold"),
        legend.text = element_text(size = 10),
        legend.key.size = unit(0.5, "cm"),
        # Background
        panel.grid = element_blank(),
        panel.background = element_blank(),
        plot.margin = margin(15, 15, 15, 15)
    ) +
    guides(fill = guide_legend(ncol = 1, reverse = TRUE))

print(p_snc_comp_sg)

ggsave(file.path(FIGURES_DIR, paste0(DATASET, "_microglia_snc_composition_by_studygroup.svg")), 
       p_snc_comp_sg, width = 8, height = 6, dpi = 300)

cat("\n✓ Saved:", paste0(DATASET, "_microglia_snc_composition_by_studygroup.svg"), "\n")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# PLOT 2: COMPOSITION BY SAMPLE (SPLIT BY STUDY GROUP)
# ════════════════════════════════════════════════════════════════════════════════

cat("\n--- Plot 2: Composition by Sample (Split by Study Group) ---\n")

suppressWarnings(suppressMessages({

# ─────────────────────────────────────────────────────────────────────────────
# CALCULATE COMPOSITION
# ─────────────────────────────────────────────────────────────────────────────

comp_sample <- microglia@meta.data %>%
    group_by(.data[[DONOR_COL]], .data[[STUDY_GROUP_COL]], .data[[SUBCLUSTER_COL]]) %>%
    summarise(n = n(), .groups = "drop") %>%
    group_by(.data[[DONOR_COL]]) %>%
    mutate(pct = n / sum(n) * 100) %>%
    ungroup()

colnames(comp_sample)[1:3] <- c("Donor", "Study_Group", "State")

}))

# ─────────────────────────────────────────────────────────────────────────────
# TABLE: COMPOSITION SUMMARY
# ─────────────────────────────────────────────────────────────────────────────

comp_summary <- comp_sample %>%
    group_by(Study_Group, State) %>%
    summarise(
        mean_pct = mean(pct),
        sd_pct = sd(pct),
        .groups = "drop"
    ) %>%
    mutate(display = sprintf("%.1f ± %.1f", mean_pct, sd_pct)) %>%
    select(Study_Group, State, display) %>%
    pivot_wider(names_from = Study_Group, values_from = display)

cat("\nComposition Summary (Mean ± SD):\n")
print(as.data.frame(comp_summary))

# ─────────────────────────────────────────────────────────────────────────────
# STATISTICAL TEST: CUBE-ROOT TRANSFORMATION WITH COVARIATES
# ─────────────────────────────────────────────────────────────────────────────

cat("\n--- Statistical Test ---\n")

# Check if covariate columns exist
has_sex <- SEX_COL %in% colnames(microglia@meta.data)
has_cohort <- COHORT_COL %in% colnames(microglia@meta.data)

cat("\nCovariates:\n")
cat("  Sex column (", SEX_COL, "):", ifelse(has_sex, "FOUND", "NOT FOUND"), "\n")
cat("  Cohort column (", COHORT_COL, "):", ifelse(has_cohort, "FOUND", "NOT FOUND"), "\n")

# Build formula text
if (has_sex && has_cohort) {
    formula_text <- "CubeRoot(State_%) ~ Predictor + Sex + Cohort"
} else if (has_sex) {
    formula_text <- "CubeRoot(State_%) ~ Predictor + Sex"
} else if (has_cohort) {
    formula_text <- "CubeRoot(State_%) ~ Predictor + Cohort"
} else {
    formula_text <- "CubeRoot(State_%) ~ Predictor"
}

cat("\nModel:", formula_text, "\n")
cat("Transformation: (proportion / 100)^(1/3)\n\n")

# Get donor-level metadata for covariates
covariate_cols <- c(DONOR_COL)
if (has_sex) covariate_cols <- c(covariate_cols, SEX_COL)
if (has_cohort) covariate_cols <- c(covariate_cols, COHORT_COL)

donor_meta <- microglia@meta.data %>%
    select(all_of(covariate_cols)) %>%
    distinct()

colnames(donor_meta)[1] <- "Donor"
if (has_sex) colnames(donor_meta)[colnames(donor_meta) == SEX_COL] <- "Sex"
if (has_cohort) colnames(donor_meta)[colnames(donor_meta) == COHORT_COL] <- "Cohort"

# Prepare data for stats
comp_stats_data <- comp_sample %>%
    mutate(pct_cuberoot = (pct / 100)^(1/3)) %>%
    left_join(donor_meta, by = "Donor")

# Detect cohort type
sg_detected <- unique(comp_stats_data$Study_Group)
is_aging <- any(grepl("^Age_", sg_detected))

if (is_aging) {
    comp_stats_data <- comp_stats_data %>%
        mutate(Predictor = as.numeric(gsub("Age_([0-9]+)_.*", "\\1", Study_Group)))
    cat("Cohort type: AGING (testing linear trend with age)\n\n")
} else {
    disease_order <- c("Control", "NCI", "MCI", "AD")
    disease_order <- disease_order[disease_order %in% sg_detected]
    comp_stats_data <- comp_stats_data %>%
        mutate(Predictor = as.numeric(factor(Study_Group, levels = disease_order)) - 1)
    cat("Cohort type: DISEASE (testing trend across disease stages)\n\n")
}

# Convert covariates to factors
if (has_sex) comp_stats_data$Sex <- as.factor(comp_stats_data$Sex)
if (has_cohort) comp_stats_data$Cohort <- as.factor(comp_stats_data$Cohort)

# Test each state
results_list <- list()

for (state in unique(comp_stats_data$State)) {
    df_state <- comp_stats_data %>% filter(State == state)
    
    # Build formula dynamically
    if (has_sex && has_cohort) {
        model <- lm(pct_cuberoot ~ Predictor + Sex + Cohort, data = df_state)
    } else if (has_sex) {
        model <- lm(pct_cuberoot ~ Predictor + Sex, data = df_state)
    } else if (has_cohort) {
        model <- lm(pct_cuberoot ~ Predictor + Cohort, data = df_state)
    } else {
        model <- lm(pct_cuberoot ~ Predictor, data = df_state)
    }
    
    coef_summary <- summary(model)$coefficients
    
    results_list[[state]] <- data.frame(
        State = state,
        estimate = coef_summary["Predictor", "Estimate"],
        std_error = coef_summary["Predictor", "Std. Error"],
        p_value = coef_summary["Predictor", "Pr(>|t|)"]
    )
}

comp_stats <- do.call(rbind, results_list) %>%
    mutate(
        p_adj = p.adjust(p_value, method = "BH"),
        direction = case_when(
            estimate > 0 ~ "Up",
            estimate < 0 ~ "Down",
            TRUE ~ ""
        ),
        sig = case_when(
            p_adj < 0.001 ~ "***",
            p_adj < 0.01 ~ "**",
            p_adj < 0.05 ~ "*",
            TRUE ~ ""
        )
    ) %>%
    arrange(p_adj)

cat("Results (BH-adjusted):\n")
print(comp_stats %>% select(State, estimate, p_value, p_adj, direction, sig))

# Save stats
write.csv(comp_stats, 
          file.path(RESULTS_DIR, paste0(DATASET, "_microglia_composition_stats.csv")),
          row.names = FALSE)

# ─────────────────────────────────────────────────────────────────────────────
# PREPARE PLOT DATA
# ─────────────────────────────────────────────────────────────────────────────

# Order study groups using standardized palette order
sg_order <- names(STUDY_GROUP_COLORS)[names(STUDY_GROUP_COLORS) %in% sg_detected]

# Order samples within each study group
sample_order <- microglia@meta.data %>%
    select(all_of(c(DONOR_COL, STUDY_GROUP_COL))) %>%
    distinct() %>%
    mutate(sg_factor = factor(.data[[STUDY_GROUP_COL]], levels = sg_order)) %>%
    arrange(sg_factor, .data[[DONOR_COL]]) %>%
    pull(.data[[DONOR_COL]])

comp_sample$Donor <- factor(comp_sample$Donor, levels = sample_order)
comp_sample$Study_Group <- factor(comp_sample$Study_Group, levels = sg_order)
comp_sample$State <- factor(comp_sample$State, levels = rev(state_order))

# Count samples per study group
samples_per_group <- comp_sample %>%
    select(Donor, Study_Group) %>%
    distinct() %>%
    count(Study_Group)

cat("\nSamples per study group:\n")
print(as.data.frame(samples_per_group))

# Create readable facet labels based on cohort type
if (is_aging) {
    # Age_20_29 → "20-29"
    facet_labels <- setNames(
        gsub("Age_", "", gsub("_", "-", sg_order)),
        sg_order
    )
} else {
    # Disease labels - keep as-is or clean up underscores
    facet_labels <- setNames(
        gsub("_", " ", sg_order),
        sg_order
    )
}

# ─────────────────────────────────────────────────────────────────────────────
# CREATE LEGEND LABELS WITH SIGNIFICANCE MARKERS
# ─────────────────────────────────────────────────────────────────────────────

sig_lookup <- setNames(comp_stats$sig, comp_stats$State)
dir_lookup <- setNames(comp_stats$direction, comp_stats$State)

state_labels <- sapply(state_order, function(s) {
    sig <- sig_lookup[s]
    dir <- dir_lookup[s]
    if (!is.na(sig) && sig != "") {
        dir_symbol <- ifelse(dir == "Up", "(+)", "(-)")
        paste0(s, " ", dir_symbol, sig)
    } else {
        s
    }
})

# ─────────────────────────────────────────────────────────────────────────────
# PLOT
# ─────────────────────────────────────────────────────────────────────────────

options(repr.plot.width = 14, repr.plot.height = 6)

p_comp_sample <- ggplot(comp_sample, aes(x = Donor, y = pct, fill = State)) +
    geom_col(width = 0.85, color = "white", linewidth = 0.2) +
    scale_fill_manual(values = state_colors, labels = state_labels) +
    scale_y_continuous(limits = c(0, 100), expand = c(0, 0)) +
    facet_grid(cols = vars(Study_Group), scales = "free_x", space = "free_x",
               labeller = labeller(Study_Group = facet_labels)) +
    labs(
        x = "Samples", 
        y = "Proportion (%)",
        fill = "Microglia State"
    ) +
    theme_minimal(base_size = 14) +
    theme(
        # Axis text
        axis.text.x = element_blank(),
        axis.ticks.x = element_blank(),
        axis.text.y = element_text(size = 12, color = "black"),
        # Axis titles - larger and bold
        axis.title.x = element_text(size = 15, face = "bold", margin = margin(t = 10)),
        axis.title.y = element_text(size = 15, face = "bold", margin = margin(r = 10)),
        # Axis lines
        axis.line = element_line(color = "black", linewidth = 0.5),
        axis.ticks.y = element_line(color = "black", linewidth = 0.3),
        axis.ticks.length = unit(0.15, "cm"),
        # Facet strips - larger text
        strip.text = element_text(size = 13, face = "bold", color = "black"),
        strip.background = element_rect(fill = "gray80", color = "black", linewidth = 0.5),
        panel.spacing = unit(0.5, "lines"),
        panel.border = element_rect(color = "black", fill = NA, linewidth = 0.5),
        # Legend
        legend.position = "right",
        legend.title = element_text(size = 12, face = "bold"),
        legend.text = element_text(size = 10),
        legend.key.size = unit(0.45, "cm"),
        # Background
        panel.grid = element_blank(),
        plot.margin = margin(15, 15, 15, 15)
    ) +
    guides(fill = guide_legend(ncol = 1, reverse = TRUE))

print(p_comp_sample)

ggsave(file.path(FIGURES_DIR, paste0(DATASET, "_microglia_composition_by_sample.svg")), 
       p_comp_sample, width = 14, height = 6, dpi = 300)

# Save table
write.csv(comp_sample, 
          file.path(RESULTS_DIR, paste0(DATASET, "_microglia_composition_by_sample.csv")),
          row.names = FALSE)

cat("\n✓ Saved:", paste0(DATASET, "_microglia_composition_by_sample.svg"), "\n")
cat("✓ Saved:", paste0(DATASET, "_microglia_composition_stats.csv"), "\n")

# ─────────────────────────────────────────────────────────────────────────────
# SUMMARY
# ─────────────────────────────────────────────────────────────────────────────

sig_states <- comp_stats %>% filter(p_adj < 0.05)

cat("\n================================================================================\n")
cat("SIGNIFICANT CHANGES (p_adj < 0.05):\n")
cat("================================================================================\n")

if (nrow(sig_states) > 0) {
    direction_text <- if (is_aging) {
        c("Up" = "increases with age", "Down" = "decreases with age")
    } else {
        c("Up" = "increases with disease", "Down" = "decreases with disease")
    }
    
    for (i in 1:nrow(sig_states)) {
        dir_desc <- direction_text[sig_states$direction[i]]
        cat(sprintf("  - %s %s (p_adj = %.2e)\n", 
                    sig_states$State[i], 
                    dir_desc,
                    sig_states$p_adj[i]))
    }
} else {
    cat("  No significant changes detected\n")
}

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# PLOT 2: COMPOSITION BY SAMPLE (SPLIT BY STUDY GROUP)
# ════════════════════════════════════════════════════════════════════════════════

cat("\n--- Plot 2: Composition by Sample (Split by Study Group) ---\n")

# ─────────────────────────────────────────────────────────────────────────────
# STEP 1: CALCULATE COMPOSITION (keep as character, no factors yet)
# ─────────────────────────────────────────────────────────────────────────────

comp_sample <- microglia@meta.data %>%
    group_by(.data[[DONOR_COL]], .data[[STUDY_GROUP_COL]], .data[[SUBCLUSTER_COL]]) %>%
    summarise(n = n(), .groups = "drop") %>%
    group_by(.data[[DONOR_COL]]) %>%
    mutate(pct = n / sum(n) * 100) %>%
    ungroup()

colnames(comp_sample)[1:3] <- c("Donor", "Study_Group", "State")

# Ensure all columns are character (not factors) for clean joining
comp_sample$Donor <- as.character(comp_sample$Donor)
comp_sample$Study_Group <- as.character(comp_sample$Study_Group)
comp_sample$State <- as.character(comp_sample$State)

cat("\nStep 1 - Initial calculation:\n")
cat("  Rows:", nrow(comp_sample), "\n")
cat("  Unique donors:", n_distinct(comp_sample$Donor), "\n")
cat("  Unique states:", n_distinct(comp_sample$State), "\n")

# ─────────────────────────────────────────────────────────────────────────────
# STEP 2: COMPLETE MISSING COMBINATIONS (fill with 0%) - BEFORE STATS!
# ─────────────────────────────────────────────────────────────────────────────

# Get all unique donors with their study groups
donor_sg <- comp_sample %>%
    select(Donor, Study_Group) %>%
    distinct()

# Get all unique states
all_states <- unique(comp_sample$State)

# Create complete grid: every donor × every state
complete_grid <- donor_sg %>%
    crossing(State = all_states)

# Join with actual data, fill missing with 0
comp_sample <- complete_grid %>%
    left_join(comp_sample %>% select(Donor, State, n, pct), 
              by = c("Donor", "State")) %>%
    mutate(
        n = replace_na(n, 0),
        pct = replace_na(pct, 0)
    )

cat("\nStep 2 - After grid completion:\n")
cat("  Rows:", nrow(comp_sample), "\n")
cat("  Expected:", n_distinct(donor_sg$Donor), "x", length(all_states), "=", 
    n_distinct(donor_sg$Donor) * length(all_states), "\n")

# Verify all samples sum to 100%
sample_totals <- comp_sample %>%
    group_by(Donor) %>%
    summarise(total_pct = sum(pct), .groups = "drop")

cat("  Sample totals - Min:", round(min(sample_totals$total_pct), 1), 
    "Max:", round(max(sample_totals$total_pct), 1), "\n")

# ─────────────────────────────────────────────────────────────────────────────
# STEP 3: DETECT COHORT TYPE AND SET ORDERS
# ─────────────────────────────────────────────────────────────────────────────

sg_detected <- unique(comp_sample$Study_Group)
is_aging <- any(grepl("^Age_", sg_detected))

# Order study groups using standardized palette order
sg_order <- names(STUDY_GROUP_COLORS)[names(STUDY_GROUP_COLORS) %in% sg_detected]

# Order states by total frequency
state_order <- comp_sample %>%
    group_by(State) %>%
    summarise(total_n = sum(n), .groups = "drop") %>%
    arrange(desc(total_n)) %>%
    pull(State)

# Order samples within each study group
sample_order <- microglia@meta.data %>%
    select(all_of(c(DONOR_COL, STUDY_GROUP_COL))) %>%
    distinct() %>%
    mutate(sg_factor = factor(.data[[STUDY_GROUP_COL]], levels = sg_order)) %>%
    arrange(sg_factor, .data[[DONOR_COL]]) %>%
    pull(.data[[DONOR_COL]])

cat("\nStep 3 - Orders defined:\n")
cat("  Study groups:", paste(sg_order, collapse = ", "), "\n")
cat("  States (by freq):", paste(state_order, collapse = ", "), "\n")
cat("  Samples:", length(sample_order), "\n")

# ─────────────────────────────────────────────────────────────────────────────
# STEP 4: TABLE - COMPOSITION SUMMARY (MEAN ± SD)
# ─────────────────────────────────────────────────────────────────────────────

comp_summary <- comp_sample %>%
    group_by(Study_Group, State) %>%
    summarise(
        mean_pct = mean(pct),
        sd_pct = sd(pct),
        .groups = "drop"
    ) %>%
    mutate(display = sprintf("%.1f ± %.1f", mean_pct, sd_pct)) %>%
    select(Study_Group, State, display) %>%
    pivot_wider(names_from = Study_Group, values_from = display)

cat("\nComposition Summary (Mean ± SD):\n")
print(as.data.frame(comp_summary))

# ─────────────────────────────────────────────────────────────────────────────
# STEP 5: STATISTICAL TEST (uses complete data with zeros)
# ─────────────────────────────────────────────────────────────────────────────

cat("\n--- Statistical Test ---\n")

# Check if covariate columns exist
has_sex <- SEX_COL %in% colnames(microglia@meta.data)
has_cohort <- COHORT_COL %in% colnames(microglia@meta.data)

cat("\nCovariates:\n")
cat("  Sex column (", SEX_COL, "):", ifelse(has_sex, "FOUND", "NOT FOUND"), "\n")
cat("  Cohort column (", COHORT_COL, "):", ifelse(has_cohort, "FOUND", "NOT FOUND"), "\n")

# Build formula text
if (has_sex && has_cohort) {
    formula_text <- "CubeRoot(State_%) ~ Predictor + Sex + Cohort"
} else if (has_sex) {
    formula_text <- "CubeRoot(State_%) ~ Predictor + Sex"
} else if (has_cohort) {
    formula_text <- "CubeRoot(State_%) ~ Predictor + Cohort"
} else {
    formula_text <- "CubeRoot(State_%) ~ Predictor"
}

cat("\nModel:", formula_text, "\n")
cat("Transformation: (proportion / 100)^(1/3)\n")

# Get donor-level metadata for covariates
covariate_cols <- c(DONOR_COL)
if (has_sex) covariate_cols <- c(covariate_cols, SEX_COL)
if (has_cohort) covariate_cols <- c(covariate_cols, COHORT_COL)

donor_meta <- microglia@meta.data %>%
    select(all_of(covariate_cols)) %>%
    distinct()

colnames(donor_meta)[1] <- "Donor"
if (has_sex) colnames(donor_meta)[colnames(donor_meta) == SEX_COL] <- "Sex"
if (has_cohort) colnames(donor_meta)[colnames(donor_meta) == COHORT_COL] <- "Cohort"

# Prepare data for stats
comp_stats_data <- comp_sample %>%
    mutate(pct_cuberoot = (pct / 100)^(1/3)) %>%
    left_join(donor_meta, by = "Donor")

if (is_aging) {
    comp_stats_data <- comp_stats_data %>%
        mutate(Predictor = as.numeric(gsub("Age_([0-9]+)_.*", "\\1", Study_Group)))
    cat("Cohort type: AGING (testing linear trend with age)\n\n")
} else {
    disease_order <- c("Control", "NCI", "MCI", "AD")
    disease_order <- disease_order[disease_order %in% sg_detected]
    comp_stats_data <- comp_stats_data %>%
        mutate(Predictor = as.numeric(factor(Study_Group, levels = disease_order)) - 1)
    cat("Cohort type: DISEASE (testing trend across disease stages)\n\n")
}

# Convert covariates to factors
if (has_sex) comp_stats_data$Sex <- as.factor(comp_stats_data$Sex)
if (has_cohort) comp_stats_data$Cohort <- as.factor(comp_stats_data$Cohort)

# Test each state
results_list <- list()

for (state in state_order) {
    df_state <- comp_stats_data %>% filter(State == state)
    
    if (has_sex && has_cohort) {
        model <- lm(pct_cuberoot ~ Predictor + Sex + Cohort, data = df_state)
    } else if (has_sex) {
        model <- lm(pct_cuberoot ~ Predictor + Sex, data = df_state)
    } else if (has_cohort) {
        model <- lm(pct_cuberoot ~ Predictor + Cohort, data = df_state)
    } else {
        model <- lm(pct_cuberoot ~ Predictor, data = df_state)
    }
    
    coef_summary <- summary(model)$coefficients
    
    results_list[[state]] <- data.frame(
        State = state,
        estimate = coef_summary["Predictor", "Estimate"],
        std_error = coef_summary["Predictor", "Std. Error"],
        p_value = coef_summary["Predictor", "Pr(>|t|)"]
    )
}

comp_stats <- do.call(rbind, results_list) %>%
    mutate(
        p_adj = p.adjust(p_value, method = "BH"),
        direction = case_when(
            estimate > 0 ~ "Up",
            estimate < 0 ~ "Down",
            TRUE ~ ""
        ),
        sig = case_when(
            p_adj < 0.001 ~ "***",
            p_adj < 0.01 ~ "**",
            p_adj < 0.05 ~ "*",
            TRUE ~ ""
        )
    ) %>%
    arrange(p_adj)

cat("Results (BH-adjusted):\n")
print(comp_stats %>% select(State, estimate, p_value, p_adj, direction, sig))

# Save stats
write.csv(comp_stats, 
          file.path(RESULTS_DIR, paste0(DATASET, "_microglia_composition_stats.csv")),
          row.names = FALSE)

# ─────────────────────────────────────────────────────────────────────────────
# STEP 6: CONVERT TO FACTORS (after all data manipulation is done)
# ─────────────────────────────────────────────────────────────────────────────

comp_sample$Donor <- factor(comp_sample$Donor, levels = sample_order)
comp_sample$Study_Group <- factor(comp_sample$Study_Group, levels = sg_order)
comp_sample$State <- factor(comp_sample$State, levels = rev(state_order))

cat("\nStep 6 - Factor check (should all be 0):\n")
cat("  NA in Donor:", sum(is.na(comp_sample$Donor)), "\n")
cat("  NA in Study_Group:", sum(is.na(comp_sample$Study_Group)), "\n")
cat("  NA in State:", sum(is.na(comp_sample$State)), "\n")

# Count samples per study group
samples_per_group <- comp_sample %>%
    select(Donor, Study_Group) %>%
    distinct() %>%
    count(Study_Group)

cat("\nSamples per study group:\n")
print(as.data.frame(samples_per_group))

# ─────────────────────────────────────────────────────────────────────────────
# STEP 7: PREPARE COLORS AND LABELS
# ─────────────────────────────────────────────────────────────────────────────

# Get colors for states
state_colors <- get_state_colors(state_order, MICROGLIA_STATE_COLORS)

# Create readable facet labels
if (is_aging) {
    facet_labels <- setNames(
        gsub("Age_", "", gsub("_", "-", sg_order)),
        sg_order
    )
} else {
    facet_labels <- setNames(
        gsub("_", " ", sg_order),
        sg_order
    )
}

# Create legend labels with significance markers
sig_lookup <- setNames(comp_stats$sig, comp_stats$State)
dir_lookup <- setNames(comp_stats$direction, comp_stats$State)

state_labels <- sapply(state_order, function(s) {
    sig <- sig_lookup[s]
    dir <- dir_lookup[s]
    if (!is.na(sig) && sig != "") {
        dir_symbol <- ifelse(dir == "Up", "(+)", "(-)")
        paste0(s, " ", dir_symbol, sig)
    } else {
        s
    }
})

# ─────────────────────────────────────────────────────────────────────────────
# STEP 8: PLOT
# ─────────────────────────────────────────────────────────────────────────────

options(repr.plot.width = 14, repr.plot.height = 6)

p_comp_sample <- ggplot(comp_sample, aes(x = Donor, y = pct, fill = State)) +
    geom_col(width = 0.85, color = "white", linewidth = 0.2) +
    scale_fill_manual(values = state_colors, labels = state_labels) +
    scale_y_continuous(expand = c(0, 0)) +
    coord_cartesian(ylim = c(0, 100)) +
    facet_grid(cols = vars(Study_Group), scales = "free_x", space = "free_x",
               labeller = labeller(Study_Group = facet_labels)) +
    labs(
        x = "Samples", 
        y = "Proportion (%)",
        fill = "Microglia State"
    ) +
    theme_minimal(base_size = 14) +
    theme(
        # Axis text
        axis.text.x = element_blank(),
        axis.ticks.x = element_blank(),
        axis.text.y = element_text(size = 12, color = "black"),
        # Axis titles
        axis.title.x = element_text(size = 15, face = "bold", margin = margin(t = 10)),
        axis.title.y = element_text(size = 15, face = "bold", margin = margin(r = 10)),
        # Axis lines
        axis.line = element_line(color = "black", linewidth = 0.5),
        axis.ticks.y = element_line(color = "black", linewidth = 0.3),
        axis.ticks.length = unit(0.15, "cm"),
        # Facet strips
        strip.text = element_text(size = 13, face = "bold", color = "black"),
        strip.background = element_rect(fill = "gray80", color = "black", linewidth = 0.5),
        panel.spacing = unit(0.5, "lines"),
        panel.border = element_rect(color = "black", fill = NA, linewidth = 0.5),
        # Legend
        legend.position = "right",
        legend.title = element_text(size = 12, face = "bold"),
        legend.text = element_text(size = 10),
        legend.key.size = unit(0.45, "cm"),
        # Background
        panel.grid = element_blank(),
        plot.margin = margin(15, 15, 15, 15)
    ) +
    guides(fill = guide_legend(ncol = 1, reverse = TRUE))

print(p_comp_sample)

ggsave(file.path(FIGURES_DIR, paste0(DATASET, "_microglia_composition_by_sample.svg")), 
       p_comp_sample, width = 14, height = 6, dpi = 300)

# Save table
write.csv(comp_sample, 
          file.path(RESULTS_DIR, paste0(DATASET, "_microglia_composition_by_sample.csv")),
          row.names = FALSE)

cat("\n✓ Saved:", paste0(DATASET, "_microglia_composition_by_sample.svg"), "\n")
cat("✓ Saved:", paste0(DATASET, "_microglia_composition_stats.csv"), "\n")

# ─────────────────────────────────────────────────────────────────────────────
# SUMMARY
# ─────────────────────────────────────────────────────────────────────────────

sig_states <- comp_stats %>% filter(p_adj < 0.05)

cat("\n================================================================================\n")
cat("SIGNIFICANT CHANGES (p_adj < 0.05):\n")
cat("================================================================================\n")

if (nrow(sig_states) > 0) {
    direction_text <- if (is_aging) {
        c("Up" = "increases with age", "Down" = "decreases with age")
    } else {
        c("Up" = "increases with disease", "Down" = "decreases with disease")
    }
    
    for (i in 1:nrow(sig_states)) {
        dir_desc <- direction_text[sig_states$direction[i]]
        cat(sprintf("  - %s %s (p_adj = %.2e)\n", 
                    sig_states$State[i], 
                    dir_desc,
                    sig_states$p_adj[i]))
    }
} else {
    cat("  No significant changes detected\n")
}

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# PLOT 3: UMAP PER STUDY GROUP
# ════════════════════════════════════════════════════════════════════════════════

cat("\n--- Plot 3: UMAP per Study Group ---\n")

# ─────────────────────────────────────────────────────────────────────────────
# SETUP
# ─────────────────────────────────────────────────────────────────────────────

# Get study groups in order using standardized palette
sg_detected <- unique(microglia[[STUDY_GROUP_COL, drop = TRUE]])
sg_order <- names(STUDY_GROUP_COLORS)[names(STUDY_GROUP_COLORS) %in% sg_detected]

# Detect cohort type for labels
is_aging <- any(grepl("^Age_", sg_detected))

# Create readable facet labels
if (is_aging) {
    sg_labels <- setNames(
        gsub("Age_", "", gsub("_", "-", sg_order)),
        sg_order
    )
} else {
    sg_labels <- setNames(
        gsub("_", " ", sg_order),
        sg_order
    )
}

cat("Study groups:", paste(sg_order, collapse = ", "), "\n")

# Get state colors using standardized palette
state_order <- microglia@meta.data %>%
    count(microglia_state) %>%
    arrange(desc(n)) %>%
    pull(microglia_state)

state_colors <- get_state_colors(state_order, MICROGLIA_STATE_COLORS)

cat("States:", paste(state_order, collapse = ", "), "\n")

# ─────────────────────────────────────────────────────────────────────────────
# HELPER FUNCTION: ADD CORNER ARROWS
# ─────────────────────────────────────────────────────────────────────────────

add_corner_arrows <- function(p, label_size = 3) {
    build <- ggplot_build(p)
    x_range <- build$layout$panel_params[[1]]$x.range
    y_range <- build$layout$panel_params[[1]]$y.range
    
    x_start <- x_range[1] + diff(x_range) * 0.02
    y_start <- y_range[1] + diff(y_range) * 0.02
    x_arrow <- diff(x_range) * 0.12
    y_arrow <- diff(y_range) * 0.12
    
    p + 
        annotate("segment", x = x_start, xend = x_start + x_arrow, y = y_start, yend = y_start,
                 arrow = arrow(length = unit(0.12, "cm"), type = "closed"), linewidth = 0.4) +
        annotate("text", x = x_start + x_arrow/2, y = y_start - diff(y_range) * 0.04,
                 label = "UMAP1", size = label_size, hjust = 0.5, vjust = 1, fontface = "bold") +
        annotate("segment", x = x_start, xend = x_start, y = y_start, yend = y_start + y_arrow,
                 arrow = arrow(length = unit(0.12, "cm"), type = "closed"), linewidth = 0.4) +
        annotate("text", x = x_start - diff(x_range) * 0.04, y = y_start + y_arrow/2,
                 label = "UMAP2", size = label_size, hjust = 1, vjust = 0.5, angle = 90, fontface = "bold") +
        coord_cartesian(clip = "off")
}

# ─────────────────────────────────────────────────────────────────────────────
# UMAP THEME
# ─────────────────────────────────────────────────────────────────────────────

umap_theme <- theme_void(base_size = 12) +
    theme(
        plot.title = element_text(size = 13, face = "bold", hjust = 0.5),
        legend.position = "none",
        plot.margin = margin(15, 15, 20, 20)
    )

# ─────────────────────────────────────────────────────────────────────────────
# GENERATE UMAPS PER STUDY GROUP
# ─────────────────────────────────────────────────────────────────────────────

umap_list <- list()

for (sg in sg_order) {
    cells_sg <- colnames(microglia)[microglia[[STUDY_GROUP_COL]] == sg]
    n_cells <- length(cells_sg)
    
    # Use readable label for title
    sg_label <- sg_labels[sg]
    
    p <- DimPlot(microglia, reduction = "umap", cells = cells_sg,
                 group.by = "microglia_state", pt.size = 0.3) +
        scale_color_manual(values = state_colors) +
        ggtitle(paste0(sg_label, "\n(n=", format(n_cells, big.mark = ","), ")")) +
        umap_theme
    
    p <- add_corner_arrows(p, label_size = 2.5)
    
    umap_list[[sg]] <- p
}

# ─────────────────────────────────────────────────────────────────────────────
# CREATE SHARED LEGEND
# ─────────────────────────────────────────────────────────────────────────────

p_legend <- DimPlot(microglia, reduction = "umap", group.by = "microglia_state", pt.size = 0.5) +
    scale_color_manual(values = state_colors) +
    theme_void() +
    theme(
        legend.position = "right",
        legend.title = element_text(size = 12, face = "bold"),
        legend.text = element_text(size = 10),
        legend.key.size = unit(0.5, "cm")
    ) +
    labs(color = "Microglia State") +
    guides(color = guide_legend(ncol = 1, override.aes = list(size = 4)))

legend <- cowplot::get_legend(p_legend)

# ─────────────────────────────────────────────────────────────────────────────
# LAYOUT AND COMBINE
# ─────────────────────────────────────────────────────────────────────────────

# Layout: 2 rows x 4 cols for 7 groups
n_groups <- length(sg_order)
n_cols <- 4
n_rows <- ceiling(n_groups / n_cols)

# Add empty plots if needed to fill grid
while (length(umap_list) < n_rows * n_cols) {
    umap_list[[length(umap_list) + 1]] <- plot_spacer()
}

options(repr.plot.width = 16, repr.plot.height = 4 * n_rows + 1)

# Arrange plots
umap_grid <- wrap_plots(umap_list, ncol = n_cols)

# Combine with legend
umap_panel_sg <- (umap_grid | legend) + 
    plot_layout(widths = c(1, 0.15)) +
    plot_annotation(
        title = paste0("Microglia States by Age Group"),
        theme = theme(plot.title = element_text(size = 16, face = "bold", hjust = 0.5))
    )

print(umap_panel_sg)

ggsave(file.path(FIGURES_DIR, paste0(DATASET, "_microglia_umap_by_studygroup.svg")), 
       umap_panel_sg, width = 16, height = 4 * n_rows + 1, dpi = 300)

cat("\n✓ Saved:", paste0(DATASET, "_microglia_umap_by_studygroup.svg"), "\n")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# SECTION 1: DATA PREPARATION
# ════════════════════════════════════════════════════════════════════════════════

cat("\n", paste(rep("=", 80), collapse = ""), "\n")
cat("SECTION 1: DATA PREPARATION\n")
cat(paste(rep("=", 80), collapse = ""), "\n")

# ─────────────────────────────────────────────────────────────────────────────
# STEP 1.1: CALCULATE %SnC PER STATE PER DONOR
# ─────────────────────────────────────────────────────────────────────────────

cat("\n[Step 1.1] Calculating %SnC per state per donor...\n")

snc_by_state_donor <- seurat_obj@meta.data %>%
    group_by(.data[[DONOR_COL]], .data[[STUDY_GROUP_COL]], .data[[SUBCLUSTER_COL]]) %>%
    summarise(
        n_total = n(),
        n_snc = sum(.data[[SENESCENCE_LABEL_COL]] == "SnC"),
        pct_snc = n_snc / n_total * 100,
        mean_log_umi = mean(log10(total_counts + 1)),
        .groups = "drop"
    )

colnames(snc_by_state_donor)[1:3] <- c("Donor", "Study_Group", "State")

snc_by_state_donor$Donor <- as.character(snc_by_state_donor$Donor)
snc_by_state_donor$Study_Group <- as.character(snc_by_state_donor$Study_Group)
snc_by_state_donor$State <- as.character(snc_by_state_donor$State)

cat("  Rows:", nrow(snc_by_state_donor), "\n")
cat("  Unique donors:", n_distinct(snc_by_state_donor$Donor), "\n")
cat("  Unique states:", n_distinct(snc_by_state_donor$State), "\n")

# ─────────────────────────────────────────────────────────────────────────────
# STEP 1.2: COMPLETE MISSING COMBINATIONS (fill with 0% SnC)
# ─────────────────────────────────────────────────────────────────────────────

cat("\n[Step 1.2] Completing missing donor-state combinations...\n")

donor_sg <- snc_by_state_donor %>%
    select(Donor, Study_Group) %>%
    distinct()

all_states <- unique(snc_by_state_donor$State)

complete_grid <- donor_sg %>%
    crossing(State = all_states)

snc_by_state_donor <- complete_grid %>%
    left_join(snc_by_state_donor %>% select(Donor, State, n_total, n_snc, pct_snc, mean_log_umi), 
              by = c("Donor", "State")) %>%
    mutate(
        n_total = replace_na(n_total, 0),
        n_snc = replace_na(n_snc, 0),
        pct_snc = replace_na(pct_snc, 0)
        # mean_log_umi stays NA for missing combinations
    )

cat("  Rows after completion:", nrow(snc_by_state_donor), "\n")
cat("  Expected:", n_distinct(donor_sg$Donor), "x", length(all_states), "=", 
    n_distinct(donor_sg$Donor) * length(all_states), "\n")

# ─────────────────────────────────────────────────────────────────────────────
# STEP 1.3: MERGE DONOR-LEVEL METADATA (Age, Sex, Cohort)
# ─────────────────────────────────────────────────────────────────────────────

cat("\n[Step 1.3] Merging donor-level metadata...\n")

available_cols <- colnames(seurat_obj@meta.data)

cols_to_extract <- c(DONOR_COL)

has_primary_var <- PRIMARY_VAR %in% available_cols
if (has_primary_var) {
    cols_to_extract <- c(cols_to_extract, PRIMARY_VAR)
    cat("  Found PRIMARY_VAR:", PRIMARY_VAR, "\n")
} else {
    cat("  WARNING: PRIMARY_VAR", PRIMARY_VAR, "not found in metadata\n")
}

has_sex <- SEX_COL %in% available_cols
if (has_sex) {
    cols_to_extract <- c(cols_to_extract, SEX_COL)
    cat("  Found SEX_COL:", SEX_COL, "\n")
}

has_cohort <- COHORT_COL %in% available_cols
if (has_cohort) {
    cols_to_extract <- c(cols_to_extract, COHORT_COL)
    cat("  Found COHORT_COL:", COHORT_COL, "\n")
}

donor_meta <- seurat_obj@meta.data %>%
    select(all_of(cols_to_extract)) %>%
    distinct()

colnames(donor_meta)[1] <- "Donor"
if (has_primary_var) colnames(donor_meta)[colnames(donor_meta) == PRIMARY_VAR] <- "PrimaryVar"
if (has_sex) colnames(donor_meta)[colnames(donor_meta) == SEX_COL] <- "Sex"
if (has_cohort) colnames(donor_meta)[colnames(donor_meta) == COHORT_COL] <- "Cohort"

donor_meta$Donor <- as.character(donor_meta$Donor)

cat("  Donor metadata rows:", nrow(donor_meta), "\n")
cat("  Donor metadata columns:", paste(colnames(donor_meta), collapse = ", "), "\n")

snc_by_state_donor <- snc_by_state_donor %>%
    left_join(donor_meta, by = "Donor")

cat("  Final columns:", paste(colnames(snc_by_state_donor), collapse = ", "), "\n")

# ─────────────────────────────────────────────────────────────────────────────
# STEP 1.4: CREATE NUMERIC PREDICTOR
# ─────────────────────────────────────────────────────────────────────────────

cat("\n[Step 1.4] Creating numeric predictor...\n")

cat("  Study type:", STUDY_TYPE, "\n")
cat("  Primary variable:", PRIMARY_VAR, "(", PRIMARY_VAR_TYPE, ")\n")

is_aging <- STUDY_TYPE == "aging"

if (PRIMARY_VAR_TYPE == "continuous" && has_primary_var) {
    snc_by_state_donor$Predictor <- as.numeric(snc_by_state_donor$PrimaryVar)
    predictor_desc <- paste0(PRIMARY_VAR, " (continuous)")
    cat("  Using continuous predictor:", PRIMARY_VAR, "\n")
    cat("  Range:", min(snc_by_state_donor$Predictor, na.rm = TRUE), "-", 
        max(snc_by_state_donor$Predictor, na.rm = TRUE), "\n")
} else {
    sg_order <- names(STUDY_GROUP_COLORS)[names(STUDY_GROUP_COLORS) %in% unique(snc_by_state_donor$Study_Group)]
    snc_by_state_donor$Study_Group <- factor(snc_by_state_donor$Study_Group, levels = sg_order)
    snc_by_state_donor$Predictor <- as.numeric(snc_by_state_donor$Study_Group)
    predictor_desc <- paste0("Study_Group (ordinal)")
    cat("  Using ordinal predictor from Study_Group\n")
    cat("  Levels:", paste(sg_order, collapse = ", "), "\n")
}

# ─────────────────────────────────────────────────────────────────────────────
# STEP 1.5: SETUP ORDERS AND LABELS
# ─────────────────────────────────────────────────────────────────────────────

cat("\n[Step 1.5] Setting up orders and labels...\n")

sg_detected <- unique(snc_by_state_donor$Study_Group)
sg_order <- names(STUDY_GROUP_COLORS)[names(STUDY_GROUP_COLORS) %in% sg_detected]

state_order <- snc_by_state_donor %>%
    group_by(State) %>%
    summarise(median_snc = median(pct_snc), .groups = "drop") %>%
    arrange(desc(median_snc)) %>%
    pull(State)

if (is_aging) {
    x_labels <- setNames(gsub("Age_", "", gsub("_", "-", sg_order)), sg_order)
    x_title <- "Age Group (years)"
} else {
    x_labels <- setNames(gsub("_", " ", sg_order), sg_order)
    x_title <- "Disease Status"
}

snc_by_state_donor$Study_Group <- factor(snc_by_state_donor$Study_Group, levels = sg_order)
snc_by_state_donor$State <- factor(snc_by_state_donor$State, levels = state_order)

cat("  Study groups:", paste(sg_order, collapse = ", "), "\n")
cat("  States (by median %SnC):", paste(state_order, collapse = ", "), "\n")

cat("\n[Section 1 Complete]\n")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# SECTION 2: STATISTICAL MODELING
# ════════════════════════════════════════════════════════════════════════════════

cat("\n", paste(rep("=", 80), collapse = ""), "\n")
cat("SECTION 2: STATISTICAL MODELING\n")
cat(paste(rep("=", 80), collapse = ""), "\n")

# Cell type label for filenames
cell_type_label <- gsub("_state$", "", SUBCLUSTER_COL)

# ─────────────────────────────────────────────────────────────────────────────
# STEP 2.1: CHECK COVARIATES
# ─────────────────────────────────────────────────────────────────────────────

cat("\n[Step 2.1] Checking covariates...\n")

has_sex <- "Sex" %in% colnames(snc_by_state_donor)
has_cohort <- "Cohort" %in% colnames(snc_by_state_donor)
has_umi <- "mean_log_umi" %in% colnames(snc_by_state_donor)

cat("  Sex:", ifelse(has_sex, "FOUND", "NOT FOUND"), "\n")
cat("  Cohort:", ifelse(has_cohort, "FOUND", "NOT FOUND"), "\n")
cat("  Log UMI:", ifelse(has_umi, "FOUND", "NOT FOUND"), "\n")

# ─────────────────────────────────────────────────────────────────────────────
# STEP 2.2: BUILD FORMULA AND DESCRIPTOR STRINGS
# ─────────────────────────────────────────────────────────────────────────────

cat("\n[Step 2.2] Building model formula...\n")

covariates_used <- c()
if (has_sex) covariates_used <- c(covariates_used, "Sex")
if (has_cohort) covariates_used <- c(covariates_used, "Cohort")
if (has_umi) covariates_used <- c(covariates_used, "mean_log_umi")

if (length(covariates_used) == 0) {
    cov_part <- ""
    covariates_display <- "none"
} else {
    cov_part <- paste0(" + ", paste(covariates_used, collapse = " + "))
    covariates_display <- paste(covariates_used, collapse = ", ")
}

formula_lm <- paste0("pct_cuberoot ~ Predictor", cov_part)
model_formula <- as.formula(formula_lm)

# Predictor description
if (is_aging) {
    predictor_desc <- "Age (continuous)"
    PRIMARY_VAR <- "Age"
    x_title <- "Age Group"
} else {
    predictor_desc <- "Disease stage (ordinal)"
    PRIMARY_VAR <- "Disease"
    x_title <- "Disease Stage"
}

cat("  Formula:", formula_lm, "\n")
cat("  Predictor:", predictor_desc, "\n")
cat("  Covariates:", covariates_display, "\n")
cat("  PRIMARY_VAR:", PRIMARY_VAR, "\n")

# ─────────────────────────────────────────────────────────────────────────────
# STEP 2.3: SET UP COLORS AND LABELS FOR VISUALIZATION
# ─────────────────────────────────────────────────────────────────────────────

cat("\n[Step 2.3] Setting up colors and labels...\n")

if (is_aging) {
    x_labels <- setNames(
        gsub("Age_", "", gsub("_", "-", sg_order)),
        sg_order
    )
} else {
    x_labels <- setNames(
        gsub("_", " ", sg_order),
        sg_order
    )
}

cat("  X-axis labels:", paste(x_labels, collapse = ", "), "\n")

if (!exists("STUDY_GROUP_COLORS")) {
    cat("  WARNING: STUDY_GROUP_COLORS not found, creating default palette\n")
    n_groups <- length(sg_order)
    STUDY_GROUP_COLORS <- setNames(
        scales::hue_pal()(n_groups),
        sg_order
    )
}

cat("  Study group colors:", length(STUDY_GROUP_COLORS), "defined\n")

# ─────────────────────────────────────────────────────────────────────────────
# STEP 2.4: PREPARE DATA FOR MODELING
# ─────────────────────────────────────────────────────────────────────────────

cat("\n[Step 2.4] Preparing data for modeling...\n")

snc_model_data <- snc_by_state_donor %>%
    mutate(pct_cuberoot = (pct_snc / 100)^(1/3))

if (!"Predictor" %in% colnames(snc_model_data)) {
    cat("  Adding Predictor column...\n")
    if (is_aging) {
        snc_model_data <- snc_model_data %>%
            mutate(Predictor = PrimaryVar)
    } else {
        disease_order <- sg_order
        snc_model_data <- snc_model_data %>%
            mutate(Predictor = as.numeric(factor(Study_Group, levels = disease_order)) - 1)
    }
}

if (has_sex) snc_model_data$Sex <- as.factor(snc_model_data$Sex)
if (has_cohort) snc_model_data$Cohort <- as.factor(snc_model_data$Cohort)

cat("  Rows:", nrow(snc_model_data), "\n")
cat("  Predictor range:", min(snc_model_data$Predictor, na.rm = TRUE), "-", 
    max(snc_model_data$Predictor, na.rm = TRUE), "\n")
if (has_umi) {
    cat("  Log UMI range:", 
        sprintf("%.2f - %.2f", 
                min(snc_model_data$mean_log_umi, na.rm = TRUE),
                max(snc_model_data$mean_log_umi, na.rm = TRUE)), "\n")
}

# ─────────────────────────────────────────────────────────────────────────────
# STEP 2.5: RUN LINEAR MODEL PER STATE
# ─────────────────────────────────────────────────────────────────────────────

cat("\n[Step 2.5] Running linear models per state...\n")
cat("  Formula:", formula_lm, "\n\n")

results_list <- list()

for (state in state_order) {
    df_state <- snc_model_data %>% filter(State == state)
    n_donors <- n_distinct(df_state$Donor)
    
    model <- tryCatch(
        lm(model_formula, data = df_state),
        error = function(e) {
            cat(sprintf("    %s: MODEL FAILED — %s\n", state, e$message))
            return(NULL)
        }
    )
    
    if (is.null(model)) next
    
    coef_summary <- summary(model)$coefficients
    
    results_list[[state]] <- data.frame(
        State = state,
        n_donors = n_donors,
        estimate = coef_summary["Predictor", "Estimate"],
        std_error = coef_summary["Predictor", "Std. Error"],
        p_value = coef_summary["Predictor", "Pr(>|t|)"]
    )
    
    cat(sprintf("    %s (n=%d): b=%.4f, p=%.2e\n", 
                state, n_donors, 
                coef_summary["Predictor", "Estimate"],
                coef_summary["Predictor", "Pr(>|t|)"]))
}

# ─────────────────────────────────────────────────────────────────────────────
# STEP 2.6: MULTIPLE TESTING CORRECTION
# ─────────────────────────────────────────────────────────────────────────────

cat("\n[Step 2.6] Applying BH (FDR) correction...\n")

snc_state_stats <- do.call(rbind, results_list) %>%
    mutate(
        p_adj = p.adjust(p_value, method = "BH"),
        direction = case_when(
            estimate > 0 ~ "Up",
            estimate < 0 ~ "Down",
            TRUE ~ ""
        ),
        sig = case_when(
            p_adj < 0.001 ~ "***",
            p_adj < 0.01 ~ "**",
            p_adj < 0.05 ~ "*",
            TRUE ~ ""
        ),
        Significant = p_adj < 0.05
    ) %>%
    arrange(p_adj)

rownames(snc_state_stats) <- NULL

cat("\n  Results (BH-adjusted):\n")
print(snc_state_stats %>% select(State, n_donors, estimate, std_error, p_value, p_adj, direction, sig))

sig_results <- snc_state_stats %>% filter(Significant)

cat("\n  Significant states (FDR < 0.05):", nrow(sig_results), "of", nrow(snc_state_stats), "\n")

if (nrow(sig_results) > 0) {
    for (i in 1:nrow(sig_results)) {
        dir_text <- ifelse(sig_results$direction[i] == "Up",
                           paste0("increases with ", PRIMARY_VAR),
                           paste0("decreases with ", PRIMARY_VAR))
        cat(sprintf("    • %s: SnC %s (b=%.4f, p_adj=%.3f)\n",
                    sig_results$State[i], dir_text,
                    sig_results$estimate[i], sig_results$p_adj[i]))
    }
}

# ─────────────────────────────────────────────────────────────────────────────
# STEP 2.7: SAVE RESULTS
# ─────────────────────────────────────────────────────────────────────────────

cat("\n[Step 2.7] Saving results...\n")

write.csv(snc_state_stats, 
          file.path(RESULTS_DIR, paste0(DATASET, "_", cell_type_label, "_snc_state_stats.csv")),
          row.names = FALSE)

cat("  Saved:", paste0(DATASET, "_", cell_type_label, "_snc_state_stats.csv"), "\n")

write.csv(snc_by_state_donor %>%
              group_by(State, Study_Group) %>%
              summarise(
                  mean_pct = mean(pct_snc),
                  sd_pct = sd(pct_snc),
                  n = n(),
                  .groups = "drop"
              ),
          file.path(RESULTS_DIR, paste0(DATASET, "_", cell_type_label, "_snc_by_state_studygroup.csv")),
          row.names = FALSE)

cat("  Saved:", paste0(DATASET, "_", cell_type_label, "_snc_by_state_studygroup.csv"), "\n")

cat("\n[Section 2 Complete]\n")
cat("  Formula:", formula_lm, "\n")
cat("  Covariates:", covariates_display, "\n")
cat("  Variables created: formula_lm, model_formula, predictor_desc, covariates_used,\n")
cat("    snc_state_stats, sig_results, PRIMARY_VAR, x_title, x_labels\n")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# SECTION 3: VISUALIZATION
# ════════════════════════════════════════════════════════════════════════════════

cat("\n", paste(rep("=", 80), collapse = ""), "\n")
cat("SECTION 3: VISUALIZATION\n")
cat(paste(rep("=", 80), collapse = ""), "\n")

# Derive cell type label (reuse from Section 2, or create here)
cell_type_label <- gsub("_state$", "", SUBCLUSTER_COL)

# ─────────────────────────────────────────────────────────────────────────────
# STEP 3.1: DISPLAY SUMMARY TABLE
# ─────────────────────────────────────────────────────────────────────────────

cat("\n[Step 3.1] Summary table: Mean %SnC by State x Study Group\n\n")

snc_summary_display <- snc_by_state_donor %>%
    group_by(State, Study_Group) %>%
    summarise(
        mean_pct = mean(pct_snc),
        sd_pct = sd(pct_snc),
        n = n(),
        .groups = "drop"
    ) %>%
    mutate(display = sprintf("%.1f ± %.1f", mean_pct, sd_pct)) %>%
    select(State, Study_Group, display) %>%
    pivot_wider(names_from = Study_Group, values_from = display)

print(as.data.frame(snc_summary_display))

# ─────────────────────────────────────────────────────────────────────────────
# STEP 3.2: DISPLAY MODEL FORMULA AND RESULTS
# ─────────────────────────────────────────────────────────────────────────────

cat("\n[Step 3.2] Model specification\n")

cat("\n", paste(rep("-", 60), collapse = ""), "\n")
cat("MODEL FORMULA\n")
cat(paste(rep("-", 60), collapse = ""), "\n")

cat("\n  Response: pct_cuberoot = (pct_snc / 100)^(1/3)\n")
cat("  Formula:", formula_lm, "\n")
cat("  Predictor:", predictor_desc, "\n")
cat("  Covariates:", paste(covariates_used, collapse = ", "), "\n")
cat("  Multiple testing: BH (FDR) correction\n")

cat("\n", paste(rep("-", 60), collapse = ""), "\n")
cat("RESULTS\n")
cat(paste(rep("-", 60), collapse = ""), "\n\n")

results_display <- snc_state_stats %>%
    select(State, n_donors, estimate, std_error, p_value, p_adj, direction, sig) %>%
    mutate(
        estimate = sprintf("%.4f", estimate),
        std_error = sprintf("%.4f", std_error),
        p_value = sprintf("%.2e", p_value),
        p_adj = sprintf("%.3f", p_adj)
    )

print(as.data.frame(results_display))

if (nrow(sig_results) > 0) {
    cat("\nSignificant associations (FDR < 0.05):\n")
    for (i in 1:nrow(sig_results)) {
        dir_text <- ifelse(sig_results$direction[i] == "Up", 
                           paste0("increases with ", PRIMARY_VAR),
                           paste0("decreases with ", PRIMARY_VAR))
        cat(sprintf("  • %s: SnC %s (b=%s, p_adj=%s)\n", 
                    sig_results$State[i], 
                    dir_text,
                    sprintf("%.4f", sig_results$estimate[i]),
                    sprintf("%.3f", sig_results$p_adj[i])))
    }
} else {
    cat("\nNo significant associations (FDR < 0.05)\n")
}

# ─────────────────────────────────────────────────────────────────────────────
# STEP 3.3: CHECK MAX %SNC PER STATE (DONOR-LEVEL)
# ─────────────────────────────────────────────────────────────────────────────

cat("\n[Step 3.3] Checking max %SnC per state (donor-level)...\n\n")

state_y_stats <- snc_by_state_donor %>%
    group_by(State) %>%
    summarise(
        n_donors = n(),
        min_pct = min(pct_snc, na.rm = TRUE),
        max_pct = max(pct_snc, na.rm = TRUE),
        mean_pct = mean(pct_snc, na.rm = TRUE),
        median_pct = median(pct_snc, na.rm = TRUE),
        .groups = "drop"
    ) %>%
    arrange(desc(max_pct))

cat("  Max %SnC per state (donor-level):\n")
cat("  ", paste(rep("-", 60), collapse = ""), "\n")
cat(sprintf("  %-25s %8s %8s %8s\n", "State", "Min", "Max", "Mean"))
cat("  ", paste(rep("-", 60), collapse = ""), "\n")

for (i in 1:nrow(state_y_stats)) {
    cat(sprintf("  %-25s %7.1f%% %7.1f%% %7.1f%%\n", 
                state_y_stats$State[i],
                state_y_stats$min_pct[i],
                state_y_stats$max_pct[i],
                state_y_stats$mean_pct[i]))
}
cat("  ", paste(rep("-", 60), collapse = ""), "\n")

y_max_lookup <- state_y_stats %>%
    mutate(y_limit = max_pct * 1.15) %>%
    select(State, max_pct, y_limit)

cat("\n  Y-axis limits to use:\n")
for (i in 1:nrow(y_max_lookup)) {
    cat(sprintf("    %s: 0 - %.1f%% (max=%.1f%%)\n", 
                y_max_lookup$State[i],
                y_max_lookup$y_limit[i],
                y_max_lookup$max_pct[i]))
}

# ─────────────────────────────────────────────────────────────────────────────
# STEP 3.4: PREPARE PLOT DATA
# ─────────────────────────────────────────────────────────────────────────────

cat("\n[Step 3.4] Preparing plot data...\n")

sig_lookup <- setNames(snc_state_stats$sig, snc_state_stats$State)
dir_lookup <- setNames(snc_state_stats$direction, snc_state_stats$State)

facet_labels <- sapply(state_order, function(s) {
    sig <- sig_lookup[s]
    dir <- dir_lookup[s]
    if (!is.na(sig) && sig != "") {
        dir_symbol <- ifelse(dir == "Up", "(+)", "(-)")
        paste0(s, " ", dir_symbol, sig)
    } else {
        s
    }
})

label_map <- setNames(facet_labels, state_order)

# ─────────────────────────────────────────────────────────────────────────────
# STEP 3.5: CREATE INDIVIDUAL PLOTS PER STATE
# ─────────────────────────────────────────────────────────────────────────────

cat("\n[Step 3.5] Creating individual plots per state...\n")

suppressPackageStartupMessages({
    if (!require(cowplot, quietly = TRUE)) {
        install.packages("cowplot", quiet = TRUE)
        library(cowplot)
    }
})

plot_list <- list()

for (s in state_order) {
    
    state_data <- snc_by_state_donor %>% filter(State == s)
    state_stats <- snc_state_stats %>% filter(State == s)
    y_limit <- y_max_lookup %>% filter(State == s) %>% pull(y_limit)
    
    p_fmt <- if (state_stats$p_value < 0.001) {
        sprintf("%.1e", state_stats$p_value)
    } else if (state_stats$p_value < 0.01) {
        sprintf("%.3f", state_stats$p_value)
    } else {
        sprintf("%.2f", state_stats$p_value)
    }
    
    label_text <- paste0("b = ", sprintf("%.4f", state_stats$estimate), "\np = ", p_fmt)
    label_color <- ifelse(state_stats$Significant, "#333333", "#888888")
    label_face <- ifelse(state_stats$Significant, "bold", "plain")
    facet_title <- label_map[s]
    
    cat(sprintf("    %s: y_limit = %.1f%%\n", s, y_limit))
    
    p <- ggplot(state_data, aes(x = Study_Group, y = pct_snc)) +
        geom_boxplot(aes(fill = Study_Group), 
                     alpha = 0.7, outlier.shape = NA, width = 0.6,
                     linewidth = 0.5, color = "black") +
        geom_jitter(aes(color = Study_Group), 
                    width = 0.15, size = 1.2, alpha = 0.6) +
        geom_smooth(aes(x = as.numeric(Study_Group), y = pct_snc),
                    method = "lm", se = FALSE,
                    color = "black", linetype = "dashed", linewidth = 0.8) +
        annotate("text", x = 1, y = y_limit * 0.92, 
                 label = label_text, hjust = 0, vjust = 1,
                 size = 2.5, color = label_color, fontface = label_face) +
        coord_cartesian(ylim = c(0, y_limit)) +
        scale_fill_manual(values = STUDY_GROUP_COLORS) +
        scale_color_manual(values = STUDY_GROUP_COLORS) +
        scale_x_discrete(labels = x_labels) +
        labs(title = facet_title, x = NULL, y = NULL) +
        theme_minimal(base_size = 10) +
        theme(
            plot.title = element_text(size = 10, face = "bold", hjust = 0.5),
            axis.text.x = element_text(angle = 45, hjust = 1, vjust = 1, size = 8, 
                                       color = "black", face = "bold"),
            axis.text.y = element_text(size = 8, color = "black"),
            axis.line = element_line(color = "black", linewidth = 0.5),
            axis.ticks = element_line(color = "black", linewidth = 0.3),
            panel.grid = element_blank(),
            panel.border = element_rect(color = "black", fill = NA, linewidth = 0.5),
            legend.position = "none",
            plot.margin = margin(5, 5, 5, 5)
        )
    
    plot_list[[s]] <- p
}

# ─────────────────────────────────────────────────────────────────────────────
# STEP 3.6: COMBINE PLOTS
# ─────────────────────────────────────────────────────────────────────────────

cat("\n[Step 3.6] Combining plots...\n")

n_states <- length(state_order)
n_cols <- min(4, n_states)
n_rows <- ceiling(n_states / n_cols)

fig_width <- 3 * n_cols
fig_height <- 3 * n_rows

options(repr.plot.width = fig_width, repr.plot.height = fig_height)

p_combined <- plot_grid(
    plotlist = plot_list,
    ncol = n_cols,
    align = "hv"
)

p_final <- ggdraw() +
    draw_plot(p_combined, x = 0.03, y = 0.03, width = 0.95, height = 0.88) +
    draw_label(x_title, x = 0.5, y = 0.01, size = 12, fontface = "bold") +
    draw_label("Senescent Cells (%)", x = 0.01, y = 0.5, size = 12, fontface = "bold", angle = 90) +
    draw_label(paste0("Senescent Cell Proportion by ", tools::toTitleCase(cell_type_label), " State — ", DATASET), 
               x = 0.5, y = 0.97, size = 14, fontface = "bold") +
    draw_label(paste0("Model: ", predictor_desc, " + ", paste(covariates_used, collapse = " + ")),
               x = 0.5, y = 0.93, size = 10, color = "gray40")

print(p_final)

# ─────────────────────────────────────────────────────────────────────────────
# STEP 3.7: SAVE FIGURE
# ─────────────────────────────────────────────────────────────────────────────

cat("\n[Step 3.7] Saving figure...\n")

ggsave(file.path(FIGURES_DIR, paste0(DATASET, "_", cell_type_label, "_snc_by_state_boxplot.svg")), 
       p_final, width = fig_width, height = fig_height, dpi = 300)

cat("  Saved:", paste0(DATASET, "_", cell_type_label, "_snc_by_state_boxplot.svg"), "\n")

# ════════════════════════════════════════════════════════════════════════════════
# SUMMARY
# ════════════════════════════════════════════════════════════════════════════════

cat("\n", paste(rep("=", 80), collapse = ""), "\n")
cat("PLOT 4 COMPLETE\n")
cat(paste(rep("=", 80), collapse = ""), "\n")

cat("\nOutputs:\n")
cat("  ", file.path(RESULTS_DIR, paste0(DATASET, "_", cell_type_label, "_snc_state_stats.csv")), "\n")
cat("  ", file.path(RESULTS_DIR, paste0(DATASET, "_", cell_type_label, "_snc_by_state_studygroup.csv")), "\n")
cat("  ", file.path(FIGURES_DIR, paste0(DATASET, "_", cell_type_label, "_snc_by_state_boxplot.svg")), "\n")

cat("\n", paste(rep("=", 80), collapse = ""), "\n")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# SAVE ANNOTATED OBJECT
# ════════════════════════════════════════════════════════════════════════════════

cat("\n", paste(rep("=", 80), collapse = ""), "\n")
cat(sprintf("SAVING ANNOTATED %s OBJECT\n", toupper(cell_type_label)))
cat(paste(rep("=", 80), collapse = ""), "\n")

# Save object
save_file <- file.path(OUTPUT_DIR, paste0(DATASET, "_", cell_type_label, "_annotated.qs"))
qsave(seurat_obj, save_file)

cat("\n✓ Saved:", save_file, "\n")
cat("  Cells:", ncol(seurat_obj), "\n")
cat("  States:", length(unique(seurat_obj@meta.data[[SUBCLUSTER_COL]])), "\n")
cat("  Clusters:", length(unique(seurat_obj$seurat_clusters)), "\n")

# ════════════════════════════════════════════════════════════════════════════════
# MODULE SUMMARY
# ════════════════════════════════════════════════════════════════════════════════

cat("\n", paste(rep("=", 80), collapse = ""), "\n")
cat(sprintf("SUMMARY: %s SUBCLUSTERING & ANNOTATION\n", toupper(cell_type_label)))
cat(paste(rep("=", 80), collapse = ""), "\n")

cat("\n--- Data Summary ---\n")
cat("  Total cells:", ncol(seurat_obj), "\n")
cat("  Donors:", length(unique(seurat_obj@meta.data[[DONOR_COL]])), "\n")
cat("  Study groups:", paste(sg_order, collapse = ", "), "\n")

cat(sprintf("\n--- %s States ---\n", tools::toTitleCase(cell_type_label)))
state_counts <- table(seurat_obj@meta.data[[SUBCLUSTER_COL]])
for (s in names(sort(state_counts, decreasing = TRUE))) {
    cat(sprintf("  %s: %d (%.1f%%)\n", s, state_counts[s], state_counts[s] / sum(state_counts) * 100))
}

cat("\n--- Senescence ---\n")
cat(sprintf("  Label column: %s\n", SENESCENCE_LABEL_COL))
snc_counts <- table(seurat_obj@meta.data[[SENESCENCE_LABEL_COL]])
cat(sprintf("  SnC: %d (%.1f%%)\n", snc_counts["SnC"], snc_counts["SnC"] / sum(snc_counts) * 100))
cat(sprintf("  Non-SnC: %d (%.1f%%)\n", snc_counts["Non-SnC"], snc_counts["Non-SnC"] / sum(snc_counts) * 100))

# Show both original and state-level if rethresholding was done
if ("senescence_label_state" %in% colnames(seurat_obj@meta.data) && 
    SENESCENCE_LABEL_COL == "senescence_label_state") {
    cat("\n  (Original labels for comparison:)\n")
    orig_counts <- table(seurat_obj@meta.data$senescence_label)
    cat(sprintf("  SnC: %d (%.1f%%)\n", orig_counts["SnC"], orig_counts["SnC"] / sum(orig_counts) * 100))
}

cat("\n--- Output Files ---\n")
cat("  Data:\n")
cat("    •", save_file, "\n")

cat("\n  Figures:\n")
fig_files <- list.files(FIGURES_DIR, pattern = paste0("^", DATASET, "_", cell_type_label), full.names = FALSE)
for (f in fig_files) cat("    •", f, "\n")

cat("\n  Results:\n")
res_files <- list.files(RESULTS_DIR, pattern = paste0("^", DATASET, "_", cell_type_label), full.names = FALSE)
for (f in res_files) cat("    •", f, "\n")

cat("\n", paste(rep("=", 80), collapse = ""), "\n")
cat(sprintf("%s COMPLETE\n", toupper(cell_type_label)))
cat(paste(rep("=", 80), collapse = ""), "\n")

---

# 04E: Astrocyte Subclustering & Annotation (R)

**Purpose:** Subset astrocytes, reprocess, annotate functional states, and analyze senescence patterns.

**Reference:** Serrano-Pozo et al. 2024, Nature Neuroscience  

## Astrocyte Subclustering Notes

### Annotation: Cluster-Level, Z-Scored

Each cluster scored via AddModuleScore, then z-scored across clusters to normalize baseline expression differences. Assigned to highest z-scored state. Z-scoring needed because homeostatic genes dominate raw scores in all astrocytes.

### 5 States (Serrano-Pozo et al. 2024)

Reactive R0/R1/R2 merged due to marker overlap.

- Homeostatic: GLUL, SLC1A2, GRM3, GJA1, KCNJ10
- Intermediate: ETNPPL, GLUD1, NTRK2, GABRB1
- Metabolic: ENO1, LDHA, MT1X, HSP90AA1
- Trophic: FGF1, FGF2, CHI3L1, SOCS3
- Reactive: GFAP, C3, CD44, COL21A1, SPARC

### Workflow

Subset → QC (doublet removal) → Harmony (Cohort) → t-SNE/UMAP → Cluster → AddModuleScore → Z-score per cluster → Assign states → Validate → GLMM

In [ ]:
cat("\n================================================================================\n")
cat("04E: ASTROCYTE SUBCLUSTERING & ANNOTATION\n")
cat("================================================================================\n")

# ════════════════════════════════════════════════════════════════════════════════
# LOAD LIBRARIES
# ════════════════════════════════════════════════════════════════════════════════

suppressPackageStartupMessages({
    library(Seurat)
    library(Matrix)
    library(dplyr)
    library(tidyr)
    library(ggplot2)
    library(qs)
    library(pheatmap)
    library(patchwork)
    library(RColorBrewer)
    library(harmony)
})

cat("\n✓ Libraries loaded\n")

# ════════════════════════════════════════════════════════════════════════════════
# DATASET SELECTION
# ════════════════════════════════════════════════════════════════════════════════

DATASET <- "psychad_aging"  # Options: 'psychad_aging', 'psychad_ad', 'psychencode', 'mathys'

# ════════════════════════════════════════════════════════════════════════════════
# DATASET-SPECIFIC CONFIGURATION
# ════════════════════════════════════════════════════════════════════════════════

DATASET_CONFIG <- list(
    'psychad_aging' = list(
        cell_type_col = 'subclass',
        donor_col = 'Sample',
        study_group_col = 'Study_Group',
        sex_col = 'Sex',
        cohort_col = 'Cohort',
        study_type = 'aging',
        primary_var = 'Age',
        primary_var_type = 'continuous',
        covariates = c('Sex', 'Cohort'),
        group_order = c('Age_20_29', 'Age_30_39', 'Age_40_49', 'Age_50_59', 
                        'Age_60_69', 'Age_70_79', 'Age_80_100')
    ),
    'psychad_ad' = list(
        cell_type_col = 'subclass',
        donor_col = 'Sample',
        study_group_col = 'Study_Group',
        sex_col = 'Sex',
        cohort_col = 'Cohort',
        study_type = 'disease',
        primary_var = 'Study_Group',
        primary_var_type = 'categorical',
        reference_group = 'Control',
        covariates = c('Age', 'Sex', 'Cohort'),
        group_order = c('Control', 'MCI', 'AD')
    ),
    'psychencode' = list(
        cell_type_col = 'cell_type',
        donor_col = 'sample_id',
        study_group_col = 'Study_Group',
        sex_col = 'Biological_Sex',
        cohort_col = 'Batch',
        study_type = 'aging',
        primary_var = 'Age_death',
        primary_var_type = 'continuous',
        covariates = c('Sex', 'Batch'),
        group_order = c('Age_20_29', 'Age_30_39', 'Age_40_49', 'Age_50_59', 
                        'Age_60_69', 'Age_70_79', 'Age_80_100')
    ),
    'mathys' = list(
        cell_type_col = 'broad.cell.type',
        donor_col = 'Subject',
        study_group_col = 'Study_Group',
        sex_col = 'sex',
        cohort_col = 'batch',
        study_type = 'disease',
        primary_var = 'Study_Group',
        primary_var_type = 'categorical',
        reference_group = 'Control',
        covariates = c('Age', 'sex', 'batch'),
        group_order = c('Control', 'AD')
    )
)

config <- DATASET_CONFIG[[DATASET]]

# ─────────────────────────────────────────────────────────────────────────────
# COLUMN NAMES
# ─────────────────────────────────────────────────────────────────────────────

CELL_TYPE_COL <- config$cell_type_col
DONOR_COL <- config$donor_col
STUDY_GROUP_COL <- config$study_group_col
SEX_COL <- config$sex_col
COHORT_COL <- config$cohort_col
SENESCENCE_LABEL_COL <- "senescence_label"

# ─────────────────────────────────────────────────────────────────────────────
# STUDY DESIGN PARAMETERS
# ─────────────────────────────────────────────────────────────────────────────

STUDY_TYPE <- config$study_type
PRIMARY_VAR <- config$primary_var
PRIMARY_VAR_TYPE <- config$primary_var_type
COVARIATES <- config$covariates
GROUP_ORDER <- config$group_order

# Processing parameters
N_VARIABLE_FEATURES <- 2000
N_PCS <- 50
N_DIMS_USE <- 30
CLUSTERING_RESOLUTION <- 0.7

# ════════════════════════════════════════════════════════════════════════════════
# COLOR PALETTES
# ════════════════════════════════════════════════════════════════════════════════

STUDY_GROUP_COLORS <- c(
    'Age_20_29' = '#2E86AB',
    'Age_30_39' = '#4A90E2',
    'Age_40_49' = '#50C878',
    'Age_50_59' = '#FFB347',
    'Age_60_69' = '#FF8C00',
    'Age_70_79' = '#E24A4A',
    'Age_80_100' = '#8B0000',
    'Control' = '#4E79A7',
    'MCI' = '#F28E2B',
    'AD' = '#E15759',
    'NCI' = '#4E79A7'
)

study_colors <- STUDY_GROUP_COLORS[names(STUDY_GROUP_COLORS) %in% GROUP_ORDER]

SENESCENCE_COLORS <- c('Non-SnC' = '#D3D3D3', 'SnC' = '#C44E52')

SEX_COLORS <- c('Male' = '#4878CF', 'Female' = '#E97B8A')

# Base state palette (same colors for all cell types, assigned by position)
BASE_STATE_PALETTE <- c("#4E79A7", "#59A14F", "#E15759", "#F28E2B", "#EDC948",
                        "#76B7B2", "#FFBE7D", "#BAB0AC", "#B07AA1", "#FF9DA7",
                        "#A0CBE8", "#D37295", "#9C755F", "#8B0000")

get_state_colors <- function(states) {
    setNames(BASE_STATE_PALETTE[1:length(states)], states)
}

# ════════════════════════════════════════════════════════════════════════════════
# PATHS
# ════════════════════════════════════════════════════════════════════════════════

BASE_DIR <- "/fs/scratch/PAS2598/senescence_analysis"

INPUT_FILE <- file.path(BASE_DIR, "data", "04_subsetting", DATASET, paste0(DATASET, "_seurat.qs"))
OUTPUT_DIR <- file.path(BASE_DIR, "data", "04_astrocyte", DATASET)
RESULTS_DIR <- file.path(BASE_DIR, "results", "04_astrocyte", DATASET)
FIGURES_DIR <- file.path(BASE_DIR, "figures", "04_astrocyte", DATASET)

dir.create(OUTPUT_DIR, recursive = TRUE, showWarnings = FALSE)
dir.create(RESULTS_DIR, recursive = TRUE, showWarnings = FALSE)
dir.create(FIGURES_DIR, recursive = TRUE, showWarnings = FALSE)

# ════════════════════════════════════════════════════════════════════════════════
# PRINT CONFIGURATION
# ════════════════════════════════════════════════════════════════════════════════

cat("\nConfiguration:\n")
cat("  Dataset:", DATASET, "\n")
cat("  Cell type column:", CELL_TYPE_COL, "\n")
cat("  Donor column:", DONOR_COL, "\n")
cat("  Study group column:", STUDY_GROUP_COL, "\n")
cat("  Sex column:", SEX_COL, "\n")
cat("  Cohort column:", COHORT_COL, "\n")
cat("  Study type:", STUDY_TYPE, "\n")
cat("  Primary variable:", PRIMARY_VAR, "(", PRIMARY_VAR_TYPE, ")\n")
cat("  Covariates:", paste(COVARIATES, collapse = ", "), "\n")
cat("  Group order:", paste(GROUP_ORDER, collapse = ", "), "\n")
cat("  Clustering resolution:", CLUSTERING_RESOLUTION, "\n")

cat("\nPaths:\n")
cat("  Input:", INPUT_FILE, "\n")
cat("  Output:", OUTPUT_DIR, "\n")
cat("  Results:", RESULTS_DIR, "\n")
cat("  Figures:", FIGURES_DIR, "\n")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# LOAD DATA
# ════════════════════════════════════════════════════════════════════════════════

cat("\n================================================================================\n")
cat("LOADING DATA\n")
cat("================================================================================\n")

data <- qread(INPUT_FILE)

cat("\n✓ Loaded:", ncol(data), "cells,", nrow(data), "genes\n")
cat("  Cell types:", paste(unique(data[[CELL_TYPE_COL, drop = TRUE]]), collapse = ", "), "\n")
cat("  Donors:", length(unique(data[[DONOR_COL, drop = TRUE]])), "\n")

# ════════════════════════════════════════════════════════════════════════════════
# SUBSET TO ASTROCYTES
# ════════════════════════════════════════════════════════════════════════════════

cat("\n================================================================================\n")
cat("SUBSETTING TO ASTROCYTES\n")
cat("================================================================================\n")

astrocyte <- subset(data, subset = !!sym(CELL_TYPE_COL) == "Astrocyte")

cat("\n✓ Astrocyte subset:", ncol(astrocyte), "cells\n")
cat("  Donors:", length(unique(astrocyte[[DONOR_COL, drop = TRUE]])), "\n")
cat("  Study groups:", paste(sort(unique(astrocyte[[STUDY_GROUP_COL, drop = TRUE]])), collapse = ", "), "\n")
cat("  Senescence:\n")
print(table(astrocyte[[SENESCENCE_LABEL_COL]]))

# Get study colors for detected groups
study_groups <- sort(unique(astrocyte[[STUDY_GROUP_COL, drop = TRUE]]))
study_colors <- STUDY_GROUP_COLORS[names(STUDY_GROUP_COLORS) %in% study_groups]

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# PARAMETER SWEEP — FIND BEST CLUSTERING
# ════════════════════════════════════════════════════════════════════════════════


cat("\n================================================================================\n")
cat("PARAMETER SWEEP\n")
cat("================================================================================\n")

# Save clean PCA object (reuse for each combo)
astro_base <- astrocyte
BATCH_COL <- COHORT_COL

# Parameter grid
param_grid <- list(
    list(dims = 15, theta = 2, res = 0.6, label = "A: dims15_th2_r0.6"),
    list(dims = 15, theta = 4, res = 0.8, label = "B: dims15_th4_r0.8"),
    list(dims = 20, theta = 2, res = 0.8, label = "C: dims20_th2_r0.8"),
    list(dims = 20, theta = 4, res = 0.8, label = "D: dims20_th4_r0.8"),
    list(dims = 20, theta = 4, res = 1.0, label = "E: dims20_th4_r1.0"),
    list(dims = 30, theta = 2, res = 0.8, label = "F: dims30_th2_r0.8")
)

plot_list <- list()

for (p in param_grid) {
    cat(sprintf("\n▸ %s\n", p$label))

    obj <- astro_base

    # Harmony
    obj <- RunHarmony(obj, group.by.vars = BATCH_COL,
                      theta = p$theta, max_iter = 30, verbose = FALSE)

    # UMAP + clustering
    obj <- RunUMAP(obj, reduction = "harmony", dims = 1:p$dims, verbose = FALSE)
    obj <- FindNeighbors(obj, reduction = "harmony", dims = 1:p$dims, verbose = FALSE)
    obj <- FindClusters(obj, resolution = p$res, verbose = FALSE)

    n_cl <- length(unique(Idents(obj)))
    cat(sprintf("  %d clusters\n", n_cl))

    plot_list[[p$label]] <- DimPlot(obj, reduction = "umap", label = TRUE, repel = TRUE) +
        NoLegend() +
        ggtitle(paste0(p$label, " (", n_cl, " cl)")) +
        theme(plot.title = element_text(size = 8, face = "bold"))

    rm(obj); gc(verbose = FALSE)
}

# Display grid
options(repr.plot.width = 16, repr.plot.height = 8)

print(
    (plot_list[[1]] + plot_list[[2]] + plot_list[[3]]) /
    (plot_list[[4]] + plot_list[[5]] + plot_list[[6]])
)

cat("\n================================================================================\n")
cat("Pick the best panel, then set the parameters and re-run the full pipeline\n")
cat("================================================================================\n")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# ELBOW PLOT + VARIANCE TABLE
# ════════════════════════════════════════════════════════════════════════════════

# Elbow plot
options(repr.plot.width = 6, repr.plot.height = 4)
ElbowPlot(astrocyte, ndims = 50) + 
    geom_vline(xintercept = c(15, 20, 30), linetype = "dashed", color = c("blue", "red", "gray")) +
    ggtitle("PCA Elbow Plot — Astrocytes") +
    theme_minimal()

# Variance table
stdev <- astrocyte[["pca"]]@stdev
var_explained <- stdev^2 / sum(stdev^2) * 100
cumvar <- cumsum(var_explained)

pc_table <- data.frame(
    PC = 1:50,
    StDev = round(stdev, 3),
    Var_Pct = round(var_explained, 2),
    Cumulative_Pct = round(cumvar, 2)
)

cat("\nPC Variance (key PCs):\n")
print(pc_table[c(1:5, 10, 15, 20, 25, 30, 40, 50), ])

cat(sprintf("\nCumulative variance at key dims:\n"))
cat(sprintf("  PC15: %.1f%%\n", cumvar[15]))
cat(sprintf("  PC20: %.1f%%\n", cumvar[20]))
cat(sprintf("  PC30: %.1f%%\n", cumvar[30]))

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# PROCESSING PARAMETERS
# ════════════════════════════════════════════════════════════════════════════════

N_VARIABLE_FEATURES <- 3000
N_PCS <- 50
N_DIMS_USE <- 15
CLUSTERING_RESOLUTION <- 0.6   # enough clusters to capture all 5 states
BATCH_COL <- COHORT_COL

library(harmony)
library(Seurat)

# ════════════════════════════════════════════════════════════════════════════════
# STEP 1: QC FILTERING (POST-SUBSET)
# ════════════════════════════════════════════════════════════════════════════════

cat("\n================================================================================\n")
cat("STEP 1: QC FILTERING — ASTROCYTE SUBSET\n")
cat("================================================================================\n")

cat("\nBefore filtering:", ncol(astrocyte), "cells\n")

# ── Remove low-quality cells ──
astrocyte <- subset(astrocyte,
                    nFeature_RNA > 200 &
                    nFeature_RNA < quantile(astrocyte$nFeature_RNA, 0.99) &
                    nCount_RNA > 500 &
                    nCount_RNA < quantile(astrocyte$nCount_RNA, 0.99))
cat("After quality filter:", ncol(astrocyte), "cells\n")

# ── Remove high mito ──
if ("percent.mt" %in% colnames(astrocyte@meta.data)) {
    astrocyte <- subset(astrocyte, percent.mt < 10)
    cat("After mito filter:", ncol(astrocyte), "cells\n")
}

# ── Remove doublets (cells expressing non-astrocyte markers) ──
cat("\nScoring contamination from other cell types...\n")

contam_signatures <- list(
    neuron = c("RBFOX3", "SYT1", "SNAP25", "STMN2"),
    oligo  = c("MBP", "MOG", "PLP1", "MAG"),
    micro  = c("CSF1R", "P2RY12", "CX3CR1", "TMEM119"),
    endo   = c("CLDN5", "FLT1", "PECAM1", "VWF")
)

for (ct in names(contam_signatures)) {
    genes <- contam_signatures[[ct]][contam_signatures[[ct]] %in% rownames(astrocyte)]
    if (length(genes) >= 2) {
        astrocyte <- AddModuleScore(astrocyte, features = list(genes), name = paste0(ct, "_contam"))
        cat(sprintf("  %s: %d/%d markers found\n", ct, length(genes), length(contam_signatures[[ct]])))
    }
}

contam_cols <- grep("_contam1$", colnames(astrocyte@meta.data), value = TRUE)
if (length(contam_cols) > 0) {
    keep <- rep(TRUE, ncol(astrocyte))
    for (col in contam_cols) {
        threshold <- quantile(astrocyte@meta.data[[col]], 0.95)
        keep <- keep & (astrocyte@meta.data[[col]] < threshold)
        cat(sprintf("  Removing top 5%% %s (threshold = %.3f)\n", col, threshold))
    }
    astrocyte <- astrocyte[, keep]
}

cat("\n✓ After all QC:", ncol(astrocyte), "cells\n")

# ════════════════════════════════════════════════════════════════════════════════
# STEP 2: REPROCESS
# ════════════════════════════════════════════════════════════════════════════════

cat("\n================================================================================\n")
cat("STEP 2: REPROCESSING ASTROCYTES\n")
cat("================================================================================\n")

cat("\n1. Normalizing...\n")
astrocyte <- NormalizeData(astrocyte, verbose = FALSE)

cat("2. Finding variable features...\n")
astrocyte <- FindVariableFeatures(astrocyte, nfeatures = N_VARIABLE_FEATURES, verbose = FALSE)

cat("3. Scaling (regressing nCount_RNA)...\n")
astrocyte <- ScaleData(astrocyte,
                       features = VariableFeatures(astrocyte),
                       vars.to.regress = "nCount_RNA",
                       verbose = FALSE)

cat("4. Running PCA...\n")
astrocyte <- RunPCA(astrocyte, npcs = N_PCS, verbose = FALSE)

# ════════════════════════════════════════════════════════════════════════════════
# STEP 3: HARMONY BATCH CORRECTION
# ════════════════════════════════════════════════════════════════════════════════

cat("5. Running Harmony (", BATCH_COL, ", theta=2)...\n")
astrocyte <- RunHarmony(
    astrocyte,
    group.by.vars = BATCH_COL,
    theta = c(4),
    max_iter = 30,
    verbose = TRUE
)

# ════════════════════════════════════════════════════════════════════════════════
# STEP 4: EMBEDDING + CLUSTERING
# ════════════════════════════════════════════════════════════════════════════════

cat("6. Running t-SNE...\n")
astrocyte <- RunTSNE(astrocyte, reduction = "harmony", dims = 1:N_DIMS_USE,
                     perplexity = 20, verbose = FALSE)

cat("7. Running UMAP...\n")
astrocyte <- RunUMAP(astrocyte, reduction = "harmony", dims = 1:N_DIMS_USE,
                     verbose = FALSE)

cat("8. Finding neighbors...\n")
astrocyte <- FindNeighbors(astrocyte, reduction = "harmony", dims = 1:N_DIMS_USE,
                           verbose = FALSE)

cat("9. Clustering (resolution =", CLUSTERING_RESOLUTION, ")...\n")
astrocyte <- FindClusters(astrocyte, resolution = CLUSTERING_RESOLUTION, verbose = FALSE)

n_clusters <- length(unique(Idents(astrocyte)))
cat("\n✓ Reprocessing complete:", ncol(astrocyte), "cells,", n_clusters, "clusters\n")

cat("\nCells per cluster:\n")
print(table(Idents(astrocyte)))

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# DEFINE ASTROCYTE MARKER SIGNATURES (Serrano-Pozo et al. 2024)
# ════════════════════════════════════════════════════════════════════════════════

cat("\n================================================================================\n")
cat("DEFINING ASTROCYTE MARKER SIGNATURES\n")
cat("================================================================================\n")

# Reference: Serrano-Pozo et al. 2024, Nature Neuroscience
# "Astrocyte transcriptomic changes along the spatiotemporal progression of Alzheimer's disease"
# 7-state classification: astH0, astIM, astMet, astTinf, astR0, astR1, astR2

ASTROCYTE_SIGNATURES <- list(
    
    # Homeostatic (astH0)
    'Homeostatic' = c(
        'GLUL', 'GRIA2', 'GRM3', 'SLC1A2', 'SLC6A1', 'SLC6A11',
        'KCNJ10', 'KCNJ16', 'GJA1', 'ERBB4', 'NRXN1'
    ),
    
    # Intermediate (astIM)
    'Intermediate' = c(
        'DNAH7', 'ETNPPL', 'GABRB1', 'GLUD1', 'NTRK2'
    ),
    
    # Metabolic/Stress (astMet)
    'Metabolic' = c(
        'ENO1', 'ENO2', 'GAPDH', 'GYS1', 'HK1', 'HK2', 'LDHA',
        'PDK1', 'PDK3', 'PGK1', 'PFKP', 'PKM',
        'MT1E', 'MT1F', 'MT1G', 'MT1M', 'MT1X', 'MT2A', 'MT3',
        'BAG3', 'HSP90AA1', 'HSPA1A', 'HSPB1', 'HSPH1'
    ),
    
    # Trophic (astTinf)
    'Trophic' = c(
        'FGF1', 'FGF2', 'FGFR2', 'HGF', 'OSMR',
        'CHI3L1', 'IL1R1', 'IL6R', 'ITGB1', 'MAPK4', 'SOCS3'
    ),
    
    # Reactive (merged R0 + R1 + R2) — shared core + unique discriminators
    'Reactive' = c(
        # Shared reactive core
        'GFAP', 'S100B', 'CRYAB', 'HSPA1B', 'HSPB8',
        'CD44', 'GPC6', 'LAMA4', 'SERPINA3', 'TNC', 'VCAN',
        # R0-specific (ECM/collagen)
        'COL21A1', 'COL24A1', 'COLGALT1', 'LAMA2',
        # R1-specific (complement/inflammatory)
        'ADAMTSL3', 'AQP1', 'C3', 'MAP1B', 'MAPT', 'SOD2',
        # R2-specific
        'GRIA1', 'SPARC'
    )
)

# Filter signatures to available genes
available_genes <- rownames(astrocyte)

signatures_filtered <- lapply(ASTROCYTE_SIGNATURES, function(genes) {
    genes[genes %in% available_genes]
})
signatures_filtered <- signatures_filtered[sapply(signatures_filtered, length) >= 2]

cat("\nSignatures defined (Serrano-Pozo et al. 2024, 7-state):\n")
for (sig in names(signatures_filtered)) {
    n_total <- length(ASTROCYTE_SIGNATURES[[sig]])
    n_avail <- length(signatures_filtered[[sig]])
    cat(sprintf("  %s: %d/%d genes available\n", sig, n_avail, n_total))
}

cat("\n✓", length(signatures_filtered), "signatures ready for scoring\n")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# CALCULATE MODULE SCORES
# ════════════════════════════════════════════════════════════════════════════════

cat("\n================================================================================\n")
cat("CALCULATING MODULE SCORES\n")
cat("================================================================================\n")

DefaultAssay(astrocyte) <- "RNA"

for (sig_name in names(signatures_filtered)) {
    genes <- signatures_filtered[[sig_name]]
    
    cat(sprintf("  %s: %d genes\n", sig_name, length(genes)))
    
    astrocyte <- AddModuleScore(
        object = astrocyte,
        features = list(genes),
        name = paste0(sig_name, "_"),
        seed = 42
    )
}

cat("\n✓ Module scores calculated\n")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# ANNOTATE ASTROCYTES (CLUSTER-LEVEL, Z-SCORED)
# ════════════════════════════════════════════════════════════════════════════════

cat("\n================================================================================\n")
cat("ANNOTATING ASTROCYTES (Cluster-Level, Z-Scored Assignment)\n")
cat("================================================================================\n")

# ════════════════════════════════════════════════════════════════════════════════
# STEP 1: GET SCORE COLUMNS
# ════════════════════════════════════════════════════════════════════════════════

score_cols <- paste0(names(signatures_filtered), "_1")
score_cols <- score_cols[score_cols %in% colnames(astrocyte@meta.data)]
sig_names <- gsub("_1$", "", score_cols)

cat("Using", length(score_cols), "signatures:", paste(sig_names, collapse = ", "), "\n")

# ════════════════════════════════════════════════════════════════════════════════
# STEP 2: COMPUTE MEAN SCORES PER CLUSTER
# ════════════════════════════════════════════════════════════════════════════════

cat("\n--- Computing mean scores per cluster ---\n")

cluster_ids <- sort(unique(astrocyte$seurat_clusters))

cluster_scores <- data.frame(cluster = cluster_ids)
rownames(cluster_scores) <- cluster_ids

for (col in score_cols) {
    means <- tapply(astrocyte@meta.data[[col]], astrocyte$seurat_clusters, mean)
    cluster_scores[[col]] <- means[as.character(cluster_ids)]
}

# ════════════════════════════════════════════════════════════════════════════════
# STEP 3: Z-SCORE ACROSS CLUSTERS → ASSIGN
# ════════════════════════════════════════════════════════════════════════════════

# Z-score normalizes signature strength so weaker signatures (e.g. Trophic)
# can compete with dominant ones (e.g. Homeostatic)
score_matrix <- as.matrix(cluster_scores[, score_cols])
z_cluster_scores <- scale(score_matrix)
colnames(z_cluster_scores) <- sig_names

# Assign each cluster to its highest z-scored state
cluster_scores$assigned_state <- sig_names[apply(z_cluster_scores, 1, which.max)]
cluster_scores$top_zscore <- apply(z_cluster_scores, 1, max)
cluster_scores$margin <- apply(z_cluster_scores, 1, function(x) {
    sorted <- sort(x, decreasing = TRUE)
    sorted[1] - sorted[2]
})

cat("\nCluster assignments (z-scored):\n")
print(cluster_scores[, c("cluster", "assigned_state", "top_zscore", "margin")])

# Flag low-confidence clusters
low_conf <- cluster_scores$margin < 0.3
if (any(low_conf)) {
    cat("\n⚠ Low-confidence clusters (margin < 0.3):\n")
    print(cluster_scores[low_conf, c("cluster", "assigned_state", "margin")])
}

# ════════════════════════════════════════════════════════════════════════════════
# STEP 4: MAP STATE TO CELLS
# ════════════════════════════════════════════════════════════════════════════════

state_map <- setNames(cluster_scores$assigned_state, as.character(cluster_scores$cluster))
astrocyte$astrocyte_state <- unname(state_map[as.character(astrocyte$seurat_clusters)])

SUBCLUSTER_COL <- "astrocyte_state"

# ════════════════════════════════════════════════════════════════════════════════
# STATE DISTRIBUTION
# ════════════════════════════════════════════════════════════════════════════════

cat("\nState distribution:\n")
print(table(astrocyte@meta.data[[SUBCLUSTER_COL]]))

cat("\nPercentages:\n")
print(round(prop.table(table(astrocyte@meta.data[[SUBCLUSTER_COL]])) * 100, 1))

# Check coverage
states_found <- unique(cluster_scores$assigned_state)
states_missing <- setdiff(sig_names, states_found)
cat("\n✓ States found:", paste(states_found, collapse = ", "), "\n")
if (length(states_missing) > 0) {
    cat("⚠ States not detected:", paste(states_missing, collapse = ", "), "\n")
    cat("  (No cluster scored highest for these — may not be present in data)\n")
}

# ════════════════════════════════════════════════════════════════════════════════
# SAVE CLUSTER SCORES
# ════════════════════════════════════════════════════════════════════════════════

cell_type_label <- gsub("_state$", "", SUBCLUSTER_COL)

scores_file <- file.path(RESULTS_DIR, paste0(DATASET, "_", cell_type_label, "_cluster_scores.csv"))
write.csv(cluster_scores, scores_file, row.names = FALSE)
cat("✓ Saved:", scores_file, "\n")

cluster_summary <- data.frame(
    cluster = names(state_map),
    assigned_state = unname(state_map),
    n_cells = as.numeric(table(astrocyte$seurat_clusters)[names(state_map)]),
    margin = cluster_scores$margin,
    row.names = NULL
)

cluster_summary_file <- file.path(RESULTS_DIR, paste0(DATASET, "_", cell_type_label, "_cluster_assignments.csv"))
write.csv(cluster_summary, cluster_summary_file, row.names = FALSE)
cat("✓ Saved:", cluster_summary_file, "\n")

# ════════════════════════════════════════════════════════════════════════════════
# STATE COLORS & ORDER
# ════════════════════════════════════════════════════════════════════════════════

state_order <- names(sort(table(astrocyte@meta.data[[SUBCLUSTER_COL]]), decreasing = TRUE))
state_colors <- get_state_colors(state_order)

cat("\nState order (by frequency):\n")
cat("  ", paste(state_order, collapse = " > "), "\n")

# ════════════════════════════════════════════════════════════════════════════════
# SET seurat_obj FOR DOWNSTREAM
# ════════════════════════════════════════════════════════════════════════════════

seurat_obj <- astrocyte

save_file <- file.path(OUTPUT_DIR, paste0(DATASET, "_", cell_type_label, "_annotated.qs"))
qsave(seurat_obj, save_file)

cat("\n✓ Saved:", save_file, "\n")
cat("  Cells:", ncol(seurat_obj), "\n")
cat("  Clusters:", length(cluster_ids), "\n")
cat("  States:", length(states_found), "\n")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# VALIDATION: CANONICAL MARKERS DOTPLOT
# ════════════════════════════════════════════════════════════════════════════════

cat("\n================================================================================\n")
cat("VALIDATION: CANONICAL MARKERS DOTPLOT\n")
cat("================================================================================\n")

cell_type_label <- gsub("_state$", "", SUBCLUSTER_COL)

# Define canonical markers (top 3 per state)
canonical_markers <- list(
    'Homeostatic'  = c('GLUL', 'SLC1A2', 'GRM3'),
    'Intermediate' = c('ETNPPL', 'GLUD1', 'NTRK2'),
    'Metabolic'    = c('LDHA', 'MT1X', 'HSP90AA1'),
    'Trophic'      = c('FGF1', 'CHI3L1', 'SOCS3'),
    'Reactive'     = c('GFAP', 'C3', 'CD44')
)

# Filter to available genes
canonical_markers_filtered <- lapply(canonical_markers, function(genes) {
    genes[genes %in% rownames(seurat_obj)]
})
canonical_markers_filtered <- canonical_markers_filtered[sapply(canonical_markers_filtered, length) > 0]

cat("\nMarkers available:\n")
for (ct in names(canonical_markers_filtered)) {
    cat(sprintf("  %s: %s\n", ct, paste(canonical_markers_filtered[[ct]], collapse = ", ")))
}

all_markers <- unlist(canonical_markers_filtered)

# Set identity and get cluster order
Idents(seurat_obj) <- "seurat_clusters"
cluster_order <- as.character(sort(as.numeric(levels(Idents(seurat_obj)))))

# Calculate expression data
cat("\nCalculating expression statistics...\n")

dot_list <- list()
idx <- 1

for (cluster in cluster_order) {
    cells <- WhichCells(seurat_obj, idents = cluster)
    if (length(cells) == 0) next
    
    for (category in names(canonical_markers_filtered)) {
        for (gene in canonical_markers_filtered[[category]]) {
            expr <- GetAssayData(seurat_obj, slot = "data")[gene, cells]
            
            dot_list[[idx]] <- data.frame(
                Cluster = cluster,
                Gene = gene,
                Category = category,
                Pct_Exp = sum(expr > 0) / length(expr) * 100,
                Avg_Exp = mean(expr),
                stringsAsFactors = FALSE
            )
            idx <- idx + 1
        }
    }
}

dot_data <- do.call(rbind, dot_list)

cat("✓ Collected", nrow(dot_data), "data points\n")

dot_data <- dot_data %>%
    group_by(Gene) %>%
    mutate(Scaled_Exp = as.numeric(scale(Avg_Exp))) %>%
    ungroup()

dot_data$Scaled_Exp <- pmax(pmin(dot_data$Scaled_Exp, 2.5), -2.5)

dot_data$Cluster <- factor(dot_data$Cluster, levels = rev(cluster_order))
dot_data$Gene <- factor(dot_data$Gene, levels = all_markers)
dot_data$Category <- factor(dot_data$Category, levels = names(canonical_markers_filtered))

cat("✓ Expression data ready for", length(unique(dot_data$Gene)), 
    "genes across", length(unique(dot_data$Cluster)), "clusters\n")

options(repr.plot.width = 12, repr.plot.height = 10)

p_dotplot <- ggplot(dot_data, aes(x = Gene, y = Cluster)) +
    geom_point(aes(size = Pct_Exp, fill = Scaled_Exp), shape = 21, color = "black", stroke = 0.3) +
    scale_size_continuous(
        range = c(1, 6),
        limits = c(0, 100),
        breaks = c(0, 25, 50, 75, 100),
        name = "% Expr"
    ) +
    scale_fill_gradientn(
        colors = c("#313695", "#4575B4", "#74ADD1", "#FFFFBF", "#FDAE61", "#F46D43", "#A50026"),
        limits = c(-2.5, 2.5),
        name = "Scaled\nExpr"
    ) +
    facet_grid(cols = vars(Category), scales = "free_x", space = "free_x") +
    labs(x = NULL, y = "Cluster") +
    theme_bw(base_size = 10) +
    theme(
        axis.text.x = element_text(angle = 45, hjust = 1, size = 8, face = "italic"),
        axis.text.y = element_text(size = 9),
        axis.title.y = element_text(size = 10, face = "bold"),
        strip.text = element_text(size = 8, face = "bold"),
        strip.background = element_rect(fill = "gray70", color = "black", linewidth = 0.5),
        panel.grid = element_blank(),
        panel.spacing = unit(0.2, "lines"),
        panel.border = element_rect(color = "black", linewidth = 0.5),
        legend.position = "right",
        legend.title = element_text(size = 8),
        legend.text = element_text(size = 7),
        legend.key.size = unit(0.4, "cm"),
        plot.margin = margin(10, 10, 10, 10)
    )

print(p_dotplot)

dotplot_file <- file.path(FIGURES_DIR, paste0(DATASET, "_", cell_type_label, "_dotplot_canonical.svg"))
ggsave(dotplot_file, p_dotplot, width = 12, height = 10)

cat("✓ Saved:", dotplot_file, "\n")

In [ ]:
ls()

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# SENESCENCE RE-THRESHOLDING AT CELL STATE LEVEL
# ════════════════════════════════════════════════════════════════════════════════

cat("\n", paste(rep("=", 80), collapse = ""), "\n")
cat("SENESCENCE RE-THRESHOLDING AT CELL STATE LEVEL\n")
cat(paste(rep("=", 80), collapse = ""), "\n")

# ─────────────────────────────────────────────────────────────────────────────
# CONFIGURATION
# ─────────────────────────────────────────────────────────────────────────────

SD_THRESHOLD <- 2
SCORE_COL <- "senescence_score"

if (STUDY_TYPE == "aging") {
    REFERENCE_GROUP <- GROUP_ORDER[1]
} else {
    REFERENCE_GROUP <- config$reference_group
}

cat(sprintf("\n  Seurat object: %s cells\n", format(ncol(seurat_obj), big.mark = ",")))
cat(sprintf("  Score column: %s\n", SCORE_COL))
cat(sprintf("  State column: %s\n", SUBCLUSTER_COL))
cat(sprintf("  Study type: %s\n", STUDY_TYPE))
cat(sprintf("  Reference group: %s\n", REFERENCE_GROUP))
cat(sprintf("  Threshold: Mean + %s SD per cell state\n", SD_THRESHOLD))

# ─────────────────────────────────────────────────────────────────────────────
# VALIDATE
# ─────────────────────────────────────────────────────────────────────────────

required_cols <- c(SCORE_COL, SENESCENCE_LABEL_COL, STUDY_GROUP_COL, SUBCLUSTER_COL)
missing <- required_cols[!required_cols %in% colnames(seurat_obj@meta.data)]
if (length(missing) > 0) {
    stop(sprintf("Missing columns: %s", paste(missing, collapse = ", ")))
}

ref_mask <- seurat_obj@meta.data[[STUDY_GROUP_COL]] == REFERENCE_GROUP
n_ref <- sum(ref_mask)
if (n_ref == 0) {
    stop(sprintf("Reference group '%s' not found! Available: %s",
                 REFERENCE_GROUP,
                 paste(unique(seurat_obj@meta.data[[STUDY_GROUP_COL]]), collapse = ", ")))
}
cat(sprintf("  Reference cells: %s\n", format(n_ref, big.mark = ",")))

# ─────────────────────────────────────────────────────────────────────────────
# ORIGINAL (SUBCLASS-LEVEL) SUMMARY
# ─────────────────────────────────────────────────────────────────────────────

orig_snc <- sum(seurat_obj@meta.data[[SENESCENCE_LABEL_COL]] == "SnC")
n_total <- nrow(seurat_obj@meta.data)

cat(sprintf("\n  Original (subclass-level): %s SnC / %s (%.1f%%)\n",
            format(orig_snc, big.mark = ","),
            format(n_total, big.mark = ","),
            orig_snc / n_total * 100))

# ─────────────────────────────────────────────────────────────────────────────
# CALCULATE STATE-LEVEL THRESHOLDS
# ─────────────────────────────────────────────────────────────────────────────

cat("\n  Calculating thresholds per cell state...\n")

states <- unique(seurat_obj@meta.data[[SUBCLUSTER_COL]])
thresholds <- list()

for (state in states) {
    state_ref_mask <- ref_mask & (seurat_obj@meta.data[[SUBCLUSTER_COL]] == state)
    ref_scores <- seurat_obj@meta.data[state_ref_mask, SCORE_COL]
    
    if (length(ref_scores) >= 10) {
        state_mean <- mean(ref_scores)
        state_sd <- sd(ref_scores)
        thresholds[[state]] <- state_mean + (SD_THRESHOLD * state_sd)
        
        cat(sprintf("    %s:\n", state))
        cat(sprintf("      n_ref=%s, mean=%.4f, sd=%.4f, threshold=%.4f\n",
                    format(length(ref_scores), big.mark = ","),
                    state_mean, state_sd, thresholds[[state]]))
    } else {
        all_state_scores <- seurat_obj@meta.data[
            seurat_obj@meta.data[[SUBCLUSTER_COL]] == state, SCORE_COL
        ]
        state_mean <- mean(all_state_scores)
        state_sd <- sd(all_state_scores)
        thresholds[[state]] <- state_mean + (SD_THRESHOLD * state_sd)
        
        cat(sprintf("    %s: (fallback — only %d ref cells, using all %s cells)\n",
                    state, length(ref_scores),
                    format(length(all_state_scores), big.mark = ",")))
        cat(sprintf("      mean=%.4f, sd=%.4f, threshold=%.4f\n",
                    state_mean, state_sd, thresholds[[state]]))
    }
}

# ─────────────────────────────────────────────────────────────────────────────
# APPLY STATE-LEVEL THRESHOLDS
# ─────────────────────────────────────────────────────────────────────────────

cat("\n  Applying state-level thresholds...\n")

seurat_obj@meta.data$is_senescent_state <- FALSE

for (state in names(thresholds)) {
    mask <- (seurat_obj@meta.data[[SUBCLUSTER_COL]] == state) &
            (seurat_obj@meta.data[[SCORE_COL]] >= thresholds[[state]])
    seurat_obj@meta.data$is_senescent_state[mask] <- TRUE
}

seurat_obj@meta.data$senescence_label_state <- ifelse(
    seurat_obj@meta.data$is_senescent_state, "SnC", "Non-SnC"
)

# ─────────────────────────────────────────────────────────────────────────────
# COMPARISON: ORIGINAL VS STATE-LEVEL
# ─────────────────────────────────────────────────────────────────────────────

new_snc <- sum(seurat_obj@meta.data$is_senescent_state)

cat("\n  ✓ Re-thresholding complete\n")
cat(sprintf("\n  %-25s %12s %12s\n", "", "Subclass", "State-level"))
cat(paste(rep("-", 55), collapse = ""), "\n")
cat(sprintf("  %-25s %12s %12s\n",
            "Total SnC",
            format(orig_snc, big.mark = ","),
            format(new_snc, big.mark = ",")))
cat(sprintf("  %-25s %11.1f%% %11.1f%%\n",
            "%SnC overall",
            orig_snc / n_total * 100,
            new_snc / n_total * 100))

cat("\n  Per state:\n")
cat(sprintf("  %-25s %6s %8s %8s %8s %8s %8s\n",
            "State", "Total", "Orig_n", "Orig_%", "New_n", "New_%", "Change"))
cat(paste(rep("-", 80), collapse = ""), "\n")

for (state in sort(states)) {
    state_mask <- seurat_obj@meta.data[[SUBCLUSTER_COL]] == state
    state_total <- sum(state_mask)
    
    orig_n <- sum(state_mask & (seurat_obj@meta.data[[SENESCENCE_LABEL_COL]] == "SnC"))
    new_n <- sum(state_mask & seurat_obj@meta.data$is_senescent_state)
    
    orig_pct <- orig_n / state_total * 100
    new_pct <- new_n / state_total * 100
    change <- new_pct - orig_pct
    
    flag <- if (abs(change) > 2) sprintf("%+.1f ←", change) else sprintf("%+.1f", change)
    
    cat(sprintf("  %-25s %6s %8s %7.1f%% %8s %7.1f%% %8s\n",
                state,
                format(state_total, big.mark = ","),
                format(orig_n, big.mark = ","), orig_pct,
                format(new_n, big.mark = ","), new_pct,
                flag))
}

cat("\n", paste(rep("=", 80), collapse = ""), "\n")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# RELABEL SUBCLUSTER 5 → Homeostatic
# ─────────────────────────────────────────────────────────────────────────────
#seurat_obj$astrocyte_state[seurat_obj$seurat_clusters == "5"] <- "Homeostatic"
#cat("  ✓ Subcluster 5 relabeled to 'Homeostatic'\n")
#cat("  Subcluster distribution:\n")
#print(table(seurat_obj$astrocyte_state))

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# ACTIVATE STATE-LEVEL LABELS FOR DOWNSTREAM PLOTS
# ─────────────────────────────────────────────────────────────────────────────

SENESCENCE_LABEL_COL <- "senescence_label_state"
astrocyte <- seurat_obj

cat(sprintf("\n  Active senescence column: %s\n", SENESCENCE_LABEL_COL))
cat(sprintf("  SnC: %s\n", format(sum(seurat_obj@meta.data[[SENESCENCE_LABEL_COL]] == "SnC"), big.mark = ",")))
cat(sprintf("  Non-SnC: %s\n", format(sum(seurat_obj@meta.data[[SENESCENCE_LABEL_COL]] == "Non-SnC"), big.mark = ",")))
cat("\n  All downstream plots will use state-level labels.\n")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# VALIDATION: UMAP PANEL
# ════════════════════════════════════════════════════════════════════════════════

cat("\n", paste(rep("=", 80), collapse = ""), "\n")
cat("VALIDATION: UMAP PANEL\n")
cat(paste(rep("=", 80), collapse = ""), "\n")

cell_type_label <- gsub("_state$", "", SUBCLUSTER_COL)
state_colors <- get_state_colors(unique(seurat_obj@meta.data[[SUBCLUSTER_COL]]))

add_corner_arrows <- function(p, label_size = 2.5) {
    build <- ggplot_build(p)
    x_range <- build$layout$panel_params[[1]]$x.range
    y_range <- build$layout$panel_params[[1]]$y.range
    
    x_start <- x_range[1] + diff(x_range) * 0.02
    y_start <- y_range[1] + diff(y_range) * 0.02
    x_arrow <- diff(x_range) * 0.12
    y_arrow <- diff(y_range) * 0.12
    
    p + 
        annotate("segment", x = x_start, xend = x_start + x_arrow, y = y_start, yend = y_start,
                 arrow = arrow(length = unit(0.1, "cm"), type = "closed"), linewidth = 0.3) +
        annotate("text", x = x_start + x_arrow/2, y = y_start - diff(y_range) * 0.04,
                 label = "UMAP1", size = label_size, hjust = 0.5, vjust = 1) +
        annotate("segment", x = x_start, xend = x_start, y = y_start, yend = y_start + y_arrow,
                 arrow = arrow(length = unit(0.1, "cm"), type = "closed"), linewidth = 0.3) +
        annotate("text", x = x_start - diff(x_range) * 0.04, y = y_start + y_arrow/2,
                 label = "UMAP2", size = label_size, hjust = 1, vjust = 0.5, angle = 90) +
        coord_cartesian(clip = "off")
}

options(repr.plot.width = 14, repr.plot.height = 12)

umap_theme <- theme_void(base_size = 10) +
    theme(
        plot.title = element_text(size = 11, face = "bold", hjust = 0.5),
        legend.position = "right",
        legend.title = element_blank(),
        legend.text = element_text(size = 8),
        legend.key.size = unit(0.3, "cm"),
        plot.margin = margin(15, 15, 20, 20)
    )

p1 <- DimPlot(seurat_obj, reduction = "umap", group.by = "seurat_clusters", 
              label = TRUE, label.size = 3, pt.size = 0.3) +
    ggtitle("Clusters") + umap_theme + theme(legend.position = "none")
p1 <- add_corner_arrows(p1)

p2 <- DimPlot(seurat_obj, reduction = "umap", group.by = SUBCLUSTER_COL, 
              pt.size = 0.3) +
    scale_color_manual(values = state_colors) +
    ggtitle("Cell States") + umap_theme
p2 <- add_corner_arrows(p2)

p3 <- DimPlot(seurat_obj, reduction = "umap", group.by = SENESCENCE_LABEL_COL, 
              pt.size = 0.3, order = c("SnC", "Non-SnC")) +
    scale_color_manual(values = SENESCENCE_COLORS) +
    ggtitle("Senescence") + umap_theme
p3 <- add_corner_arrows(p3)

p4 <- DimPlot(seurat_obj, reduction = "umap", group.by = STUDY_GROUP_COL, pt.size = 0.3) +
    scale_color_manual(values = study_colors) +
    ggtitle("Study Group") + umap_theme
p4 <- add_corner_arrows(p4)

umap_panel <- (p1 | p2) / (p3 | p4) +
    plot_annotation(
        title = paste0(tools::toTitleCase(cell_type_label), " Subclustering — ", DATASET),
        theme = theme(plot.title = element_text(size = 14, face = "bold", hjust = 0.5))
    )

print(umap_panel)

umap_file <- file.path(FIGURES_DIR, paste0(DATASET, "_", cell_type_label, "_umap_panel.svg"))
ggsave(umap_file, umap_panel, width = 14, height = 12)
cat("✓ Saved:", umap_file, "\n")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# POST-SUBCLUSTERING ANALYSIS: COMPOSITION BY STUDY GROUP
# ════════════════════════════════════════════════════════════════════════════════

cat("\n================================================================================\n")
cat("POST-SUBCLUSTERING ANALYSIS\n")
cat("================================================================================\n")

# ─────────────────────────────────────────────────────────────────────────────
# SETUP
# ─────────────────────────────────────────────────────────────────────────────

cell_type_label <- gsub("_state$", "", SUBCLUSTER_COL)

state_order <- seurat_obj@meta.data %>%
    count(.data[[SUBCLUSTER_COL]]) %>%
    arrange(desc(n)) %>%
    pull(.data[[SUBCLUSTER_COL]])

n_states <- length(state_order)
state_colors <- get_state_colors(state_order)

cat("\nCell type:", cell_type_label, "\n")
cat("States detected:", n_states, "\n")
cat("  ", paste(state_order, collapse = ", "), "\n")
cat("Colors mapped:", length(state_colors), "\n")

# ════════════════════════════════════════════════════════════════════════════════
# PLOT 1: COMPOSITION BY STUDY GROUP
# ════════════════════════════════════════════════════════════════════════════════

cat("\n--- Plot 1: Composition by Study Group ---\n")

comp_studygroup <- seurat_obj@meta.data %>%
    group_by(.data[[STUDY_GROUP_COL]], .data[[SUBCLUSTER_COL]]) %>%
    summarise(n = n(), .groups = "drop") %>%
    group_by(.data[[STUDY_GROUP_COL]]) %>%
    mutate(pct = n / sum(n) * 100) %>%
    ungroup()

colnames(comp_studygroup)[1:2] <- c("Study_Group", "State")

# ─────────────────────────────────────────────────────────────────────────────
# TABLE OUTPUT
# ─────────────────────────────────────────────────────────────────────────────

comp_wide <- comp_studygroup %>%
    select(Study_Group, State, pct) %>%
    pivot_wider(names_from = Study_Group, values_from = pct, values_fill = 0) %>%
    mutate(across(where(is.numeric), ~ round(.x, 1)))

cat("\nState Composition by Study Group (%):\n")
print(as.data.frame(comp_wide))

write.csv(comp_wide, 
          file.path(RESULTS_DIR, paste0(DATASET, "_", cell_type_label, "_composition_by_studygroup.csv")),
          row.names = FALSE)

# ─────────────────────────────────────────────────────────────────────────────
# PREPARE FACTORS
# ─────────────────────────────────────────────────────────────────────────────

sg_detected <- unique(comp_studygroup$Study_Group)
is_aging <- STUDY_TYPE == "aging"

sg_order <- names(STUDY_GROUP_COLORS)[names(STUDY_GROUP_COLORS) %in% sg_detected]
comp_studygroup$Study_Group <- factor(comp_studygroup$Study_Group, levels = sg_order)
comp_studygroup$State <- factor(comp_studygroup$State, levels = rev(state_order))

if (is_aging) {
    x_labels <- setNames(gsub("Age_", "", gsub("_", "-", sg_order)), sg_order)
    x_title <- "Age Group (years)"
} else {
    x_labels <- setNames(gsub("_", " ", sg_order), sg_order)
    x_title <- "Disease Status"
}

# ─────────────────────────────────────────────────────────────────────────────
# PLOT
# ─────────────────────────────────────────────────────────────────────────────

options(repr.plot.width = 8, repr.plot.height = 6)

p_comp_sg <- ggplot(comp_studygroup, aes(x = Study_Group, y = pct, fill = State)) +
    geom_col(width = 0.75, color = "white", linewidth = 0.3) +
    scale_fill_manual(values = state_colors) +
    scale_x_discrete(labels = x_labels) +
    scale_y_continuous(expand = expansion(mult = c(0, 0.02))) +
    labs(
        x = x_title, 
        y = "Proportion (%)",
        fill = paste0(tools::toTitleCase(cell_type_label), " State")
    ) +
    theme_minimal(base_size = 14) +
    theme(
        axis.text.x = element_text(size = 13, color = "black", face = "bold",
                                   angle = 45, hjust = 1, vjust = 1),
        axis.text.y = element_text(size = 12, color = "black"),
        axis.title.x = element_text(size = 15, face = "bold", margin = margin(t = 12)),
        axis.title.y = element_text(size = 15, face = "bold", margin = margin(r = 12)),
        axis.line = element_line(color = "black", linewidth = 0.5),
        axis.ticks = element_line(color = "black", linewidth = 0.3),
        axis.ticks.length = unit(0.15, "cm"),
        legend.position = "right",
        legend.title = element_text(size = 12, face = "bold"),
        legend.text = element_text(size = 10),
        legend.key.size = unit(0.5, "cm"),
        panel.grid = element_blank(),
        panel.background = element_blank(),
        plot.margin = margin(15, 15, 15, 15)
    ) +
    guides(fill = guide_legend(ncol = 1, reverse = TRUE))

print(p_comp_sg)

ggsave(file.path(FIGURES_DIR, paste0(DATASET, "_", cell_type_label, "_composition_by_studygroup.svg")), 
       p_comp_sg, width = 8, height = 6, dpi = 300)

cat("\n✓ Saved:", paste0(DATASET, "_", cell_type_label, "_composition_by_studygroup.svg"), "\n")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# PLOT: SnC COMPOSITION BY STUDY GROUP
# ════════════════════════════════════════════════════════════════════════════════

cat(sprintf("\n--- Plot: SnC Composition by Study Group (%s) ---\n", tools::toTitleCase(cell_type_label)))

snc_comp_sg <- seurat_obj@meta.data %>%
    filter(.data[[SENESCENCE_LABEL_COL]] == "SnC") %>%
    group_by(.data[[STUDY_GROUP_COL]], .data[[SUBCLUSTER_COL]]) %>%
    summarise(n = n(), .groups = "drop") %>%
    # ── FIX: Complete all combinations, fill missing with 0 ──
    complete(.data[[STUDY_GROUP_COL]], .data[[SUBCLUSTER_COL]], fill = list(n = 0)) %>%
    group_by(.data[[STUDY_GROUP_COL]]) %>%
    mutate(pct = n / sum(n) * 100) %>%
    ungroup()

colnames(snc_comp_sg)[1:2] <- c("Study_Group", "State")

# Replace any NaN from 0/0 division
snc_comp_sg$pct[is.nan(snc_comp_sg$pct)] <- 0

# ─────────────────────────────────────────────────────────────────────────────
# PREPARE FACTORS
# ─────────────────────────────────────────────────────────────────────────────

sg_detected <- unique(snc_comp_sg$Study_Group)
is_aging <- STUDY_TYPE == "aging"

sg_order <- names(STUDY_GROUP_COLORS)[names(STUDY_GROUP_COLORS) %in% sg_detected]
snc_comp_sg$Study_Group <- factor(snc_comp_sg$Study_Group, levels = sg_order)
snc_comp_sg$State <- factor(snc_comp_sg$State, levels = rev(state_order))

if (is_aging) {
    x_labels <- setNames(gsub("Age_", "", gsub("_", "-", sg_order)), sg_order)
    x_title <- "Age Group (years)"
} else {
    x_labels <- setNames(gsub("_", " ", sg_order), sg_order)
    x_title <- "Disease Status"
}

# ─────────────────────────────────────────────────────────────────────────────
# TABLE OUTPUT
# ─────────────────────────────────────────────────────────────────────────────

snc_comp_sg_wide <- snc_comp_sg %>%
    select(Study_Group, State, pct) %>%
    pivot_wider(names_from = Study_Group, values_from = pct, values_fill = 0) %>%
    mutate(across(where(is.numeric), ~ round(.x, 1)))

cat("\nSnC Composition by Study Group (%):\n")
print(as.data.frame(snc_comp_sg_wide))

write.csv(snc_comp_sg_wide, 
          file.path(RESULTS_DIR, paste0(DATASET, "_", cell_type_label, "_snc_composition_by_studygroup.csv")),
          row.names = FALSE)

snc_totals <- seurat_obj@meta.data %>%
    filter(.data[[SENESCENCE_LABEL_COL]] == "SnC") %>%
    group_by(.data[[STUDY_GROUP_COL]]) %>%
    summarise(n = n(), .groups = "drop")

colnames(snc_totals)[1] <- "Study_Group"

cat("\nTotal SnC cells per study group:\n")
print(as.data.frame(snc_totals))

# ─────────────────────────────────────────────────────────────────────────────
# PLOT
# ─────────────────────────────────────────────────────────────────────────────

options(repr.plot.width = 8, repr.plot.height = 6)

p_snc_comp_sg <- ggplot(snc_comp_sg, aes(x = Study_Group, y = pct, fill = State)) +
    geom_col(width = 0.75, color = "white", linewidth = 0.3) +
    scale_fill_manual(values = state_colors) +
    scale_x_discrete(labels = x_labels) +
    scale_y_continuous(limits = c(0, 100.1), expand = c(0, 0)) +
    labs(
        x = x_title, 
        y = paste0("Proportion of SnC ", tools::toTitleCase(cell_type_label), " (%)"),
        fill = paste0(tools::toTitleCase(cell_type_label), " State")
    ) +
    theme_minimal(base_size = 14) +
    theme(
        axis.text.x = element_text(size = 13, color = "black", face = "bold",
                                   angle = 45, hjust = 1, vjust = 1),
        axis.text.y = element_text(size = 12, color = "black"),
        axis.title.x = element_text(size = 15, face = "bold", margin = margin(t = 12)),
        axis.title.y = element_text(size = 15, face = "bold", margin = margin(r = 12)),
        axis.line = element_line(color = "black", linewidth = 0.5),
        axis.ticks = element_line(color = "black", linewidth = 0.3),
        axis.ticks.length = unit(0.15, "cm"),
        legend.position = "right",
        legend.title = element_text(size = 12, face = "bold"),
        legend.text = element_text(size = 10),
        legend.key.size = unit(0.5, "cm"),
        panel.grid = element_blank(),
        panel.background = element_blank(),
        plot.margin = margin(15, 15, 15, 15)
    ) +
    guides(fill = guide_legend(ncol = 1, reverse = TRUE))

print(p_snc_comp_sg)

ggsave(file.path(FIGURES_DIR, paste0(DATASET, "_", cell_type_label, "_snc_composition_by_studygroup.svg")), 
       p_snc_comp_sg, width = 8, height = 6, dpi = 300)

cat("\n✓ Saved:", paste0(DATASET, "_", cell_type_label, "_snc_composition_by_studygroup.svg"), "\n")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# PLOT 2: COMPOSITION BY SAMPLE (SPLIT BY STUDY GROUP)
# ════════════════════════════════════════════════════════════════════════════════

cat("\n--- Plot 2: Composition by Sample (Split by Study Group) ---\n")

cell_type_label <- gsub("_state$", "", SUBCLUSTER_COL)
is_aging <- STUDY_TYPE == "aging"

# ─────────────────────────────────────────────────────────────────────────────
# STEP 1: CALCULATE COMPOSITION
# ─────────────────────────────────────────────────────────────────────────────

comp_sample <- seurat_obj@meta.data %>%
    group_by(.data[[DONOR_COL]], .data[[STUDY_GROUP_COL]], .data[[SUBCLUSTER_COL]]) %>%
    summarise(n = n(), .groups = "drop") %>%
    group_by(.data[[DONOR_COL]]) %>%
    mutate(pct = n / sum(n) * 100) %>%
    ungroup()

colnames(comp_sample)[1:3] <- c("Donor", "Study_Group", "State")

comp_sample$Donor <- as.character(comp_sample$Donor)
comp_sample$Study_Group <- as.character(comp_sample$Study_Group)
comp_sample$State <- as.character(comp_sample$State)

cat("\nStep 1 - Initial calculation:\n")
cat("  Rows:", nrow(comp_sample), "\n")
cat("  Unique donors:", n_distinct(comp_sample$Donor), "\n")
cat("  Unique states:", n_distinct(comp_sample$State), "\n")

# ─────────────────────────────────────────────────────────────────────────────
# STEP 2: COMPLETE MISSING COMBINATIONS
# ─────────────────────────────────────────────────────────────────────────────

donor_sg <- comp_sample %>% select(Donor, Study_Group) %>% distinct()
all_states <- unique(comp_sample$State)
complete_grid <- donor_sg %>% crossing(State = all_states)

comp_sample <- complete_grid %>%
    left_join(comp_sample %>% select(Donor, State, n, pct), 
              by = c("Donor", "State")) %>%
    mutate(n = replace_na(n, 0), pct = replace_na(pct, 0))

cat("\nStep 2 - After grid completion:\n")
cat("  Rows:", nrow(comp_sample), "\n")
cat("  Expected:", n_distinct(donor_sg$Donor), "x", length(all_states), "=", 
    n_distinct(donor_sg$Donor) * length(all_states), "\n")

sample_totals <- comp_sample %>%
    group_by(Donor) %>%
    summarise(total_pct = sum(pct), .groups = "drop")

cat("  Sample totals - Min:", round(min(sample_totals$total_pct), 1), 
    "Max:", round(max(sample_totals$total_pct), 1), "\n")

# ─────────────────────────────────────────────────────────────────────────────
# STEP 3: DETECT COHORT TYPE AND SET ORDERS
# ─────────────────────────────────────────────────────────────────────────────

sg_detected <- unique(comp_sample$Study_Group)
sg_order <- names(STUDY_GROUP_COLORS)[names(STUDY_GROUP_COLORS) %in% sg_detected]

state_order <- comp_sample %>%
    group_by(State) %>%
    summarise(total_n = sum(n), .groups = "drop") %>%
    arrange(desc(total_n)) %>%
    pull(State)

sample_order <- seurat_obj@meta.data %>%
    select(all_of(c(DONOR_COL, STUDY_GROUP_COL))) %>%
    distinct() %>%
    mutate(sg_factor = factor(.data[[STUDY_GROUP_COL]], levels = sg_order)) %>%
    arrange(sg_factor, .data[[DONOR_COL]]) %>%
    pull(.data[[DONOR_COL]])

cat("\nStep 3 - Orders defined:\n")
cat("  Study groups:", paste(sg_order, collapse = ", "), "\n")
cat("  States (by freq):", paste(state_order, collapse = ", "), "\n")
cat("  Samples:", length(sample_order), "\n")

# ─────────────────────────────────────────────────────────────────────────────
# STEP 4: TABLE
# ─────────────────────────────────────────────────────────────────────────────

comp_summary <- comp_sample %>%
    group_by(Study_Group, State) %>%
    summarise(mean_pct = mean(pct), sd_pct = sd(pct), .groups = "drop") %>%
    mutate(display = sprintf("%.1f ± %.1f", mean_pct, sd_pct)) %>%
    select(Study_Group, State, display) %>%
    pivot_wider(names_from = Study_Group, values_from = display)

cat("\nComposition Summary (Mean ± SD):\n")
print(as.data.frame(comp_summary))

# ─────────────────────────────────────────────────────────────────────────────
# STEP 5: STATISTICAL TEST
# ─────────────────────────────────────────────────────────────────────────────

cat("\n--- Statistical Test ---\n")

has_sex <- SEX_COL %in% colnames(seurat_obj@meta.data)
has_cohort <- COHORT_COL %in% colnames(seurat_obj@meta.data)

cat("\nCovariates:\n")
cat("  Sex column (", SEX_COL, "):", ifelse(has_sex, "FOUND", "NOT FOUND"), "\n")
cat("  Cohort column (", COHORT_COL, "):", ifelse(has_cohort, "FOUND", "NOT FOUND"), "\n")

# Build dynamic formula
rhs <- "Predictor"
if (has_sex) rhs <- paste0(rhs, " + Sex")
if (has_cohort) rhs <- paste0(rhs, " + Cohort")
formula_text <- paste0("CubeRoot(State_%) ~ ", rhs)

cat("\nModel:", formula_text, "\n")
cat("Transformation: (proportion / 100)^(1/3)\n")

# Get donor metadata
covariate_cols <- c(DONOR_COL)
if (has_sex) covariate_cols <- c(covariate_cols, SEX_COL)
if (has_cohort) covariate_cols <- c(covariate_cols, COHORT_COL)

donor_meta <- seurat_obj@meta.data %>%
    select(all_of(covariate_cols)) %>%
    distinct()

colnames(donor_meta)[1] <- "Donor"
if (has_sex) colnames(donor_meta)[colnames(donor_meta) == SEX_COL] <- "Sex"
if (has_cohort) colnames(donor_meta)[colnames(donor_meta) == COHORT_COL] <- "Cohort"

comp_stats_data <- comp_sample %>%
    mutate(pct_cuberoot = (pct / 100)^(1/3)) %>%
    left_join(donor_meta, by = "Donor")

if (is_aging) {
    comp_stats_data <- comp_stats_data %>%
        mutate(Predictor = as.numeric(gsub("Age_([0-9]+)_.*", "\\1", Study_Group)))
    cat("Cohort type: AGING (testing linear trend with age)\n\n")
} else {
    disease_order <- sg_order
    comp_stats_data <- comp_stats_data %>%
        mutate(Predictor = as.numeric(factor(Study_Group, levels = disease_order)) - 1)
    cat("Cohort type: DISEASE (testing trend across disease stages)\n\n")
}

if (has_sex) comp_stats_data$Sex <- as.factor(comp_stats_data$Sex)
if (has_cohort) comp_stats_data$Cohort <- as.factor(comp_stats_data$Cohort)

model_formula_comp <- as.formula(paste0("pct_cuberoot ~ ", rhs))

results_list <- list()
for (state in state_order) {
    df_state <- comp_stats_data %>% filter(State == state)
    model <- lm(model_formula_comp, data = df_state)
    coef_summary <- summary(model)$coefficients
    results_list[[state]] <- data.frame(
        State = state,
        estimate = coef_summary["Predictor", "Estimate"],
        std_error = coef_summary["Predictor", "Std. Error"],
        p_value = coef_summary["Predictor", "Pr(>|t|)"]
    )
}

comp_stats <- do.call(rbind, results_list) %>%
    mutate(
        p_adj = p.adjust(p_value, method = "BH"),
        direction = case_when(estimate > 0 ~ "Up", estimate < 0 ~ "Down", TRUE ~ ""),
        sig = case_when(p_adj < 0.001 ~ "***", p_adj < 0.01 ~ "**", p_adj < 0.05 ~ "*", TRUE ~ "")
    ) %>%
    arrange(p_adj)

cat("Results (BH-adjusted):\n")
print(comp_stats %>% select(State, estimate, p_value, p_adj, direction, sig))

write.csv(comp_stats, 
          file.path(RESULTS_DIR, paste0(DATASET, "_", cell_type_label, "_composition_stats.csv")),
          row.names = FALSE)

# ─────────────────────────────────────────────────────────────────────────────
# STEP 6: CONVERT TO FACTORS
# ─────────────────────────────────────────────────────────────────────────────

comp_sample$Donor <- factor(comp_sample$Donor, levels = sample_order)
comp_sample$Study_Group <- factor(comp_sample$Study_Group, levels = sg_order)
comp_sample$State <- factor(comp_sample$State, levels = rev(state_order))

cat("\nStep 6 - Factor check (should all be 0):\n")
cat("  NA in Donor:", sum(is.na(comp_sample$Donor)), "\n")
cat("  NA in Study_Group:", sum(is.na(comp_sample$Study_Group)), "\n")
cat("  NA in State:", sum(is.na(comp_sample$State)), "\n")

samples_per_group <- comp_sample %>%
    select(Donor, Study_Group) %>%
    distinct() %>%
    count(Study_Group)

cat("\nSamples per study group:\n")
print(as.data.frame(samples_per_group))

# ─────────────────────────────────────────────────────────────────────────────
# STEP 7: COLORS AND LABELS
# ─────────────────────────────────────────────────────────────────────────────

state_colors <- get_state_colors(state_order)

if (is_aging) {
    facet_labels <- setNames(gsub("Age_", "", gsub("_", "-", sg_order)), sg_order)
} else {
    facet_labels <- setNames(gsub("_", " ", sg_order), sg_order)
}

sig_lookup <- setNames(comp_stats$sig, comp_stats$State)
dir_lookup <- setNames(comp_stats$direction, comp_stats$State)

state_labels <- sapply(state_order, function(s) {
    sig <- sig_lookup[s]
    dir <- dir_lookup[s]
    if (!is.na(sig) && sig != "") {
        dir_symbol <- ifelse(dir == "Up", "(+)", "(-)")
        paste0(s, " ", dir_symbol, sig)
    } else { s }
})

# ─────────────────────────────────────────────────────────────────────────────
# STEP 8: PLOT
# ─────────────────────────────────────────────────────────────────────────────

options(repr.plot.width = 14, repr.plot.height = 6)

p_comp_sample <- ggplot(comp_sample, aes(x = Donor, y = pct, fill = State)) +
    geom_col(width = 0.85, color = "white", linewidth = 0.2) +
    scale_fill_manual(values = state_colors, labels = state_labels) +
    scale_y_continuous(expand = c(0, 0)) +
    coord_cartesian(ylim = c(0, 100)) +
    facet_grid(cols = vars(Study_Group), scales = "free_x", space = "free_x",
               labeller = labeller(Study_Group = facet_labels)) +
    labs(
        x = "Samples", 
        y = "Proportion (%)",
        fill = paste0(tools::toTitleCase(cell_type_label), " State")
    ) +
    theme_minimal(base_size = 14) +
    theme(
        axis.text.x = element_blank(),
        axis.ticks.x = element_blank(),
        axis.text.y = element_text(size = 12, color = "black"),
        axis.title.x = element_text(size = 15, face = "bold", margin = margin(t = 10)),
        axis.title.y = element_text(size = 15, face = "bold", margin = margin(r = 10)),
        axis.line = element_line(color = "black", linewidth = 0.5),
        axis.ticks.y = element_line(color = "black", linewidth = 0.3),
        axis.ticks.length = unit(0.15, "cm"),
        strip.text = element_text(size = 13, face = "bold", color = "black"),
        strip.background = element_rect(fill = "gray80", color = "black", linewidth = 0.5),
        panel.spacing = unit(0.5, "lines"),
        panel.border = element_rect(color = "black", fill = NA, linewidth = 0.5),
        legend.position = "right",
        legend.title = element_text(size = 12, face = "bold"),
        legend.text = element_text(size = 10),
        legend.key.size = unit(0.45, "cm"),
        panel.grid = element_blank(),
        plot.margin = margin(15, 15, 15, 15)
    ) +
    guides(fill = guide_legend(ncol = 1, reverse = TRUE))

print(p_comp_sample)

ggsave(file.path(FIGURES_DIR, paste0(DATASET, "_", cell_type_label, "_composition_by_sample.svg")), 
       p_comp_sample, width = 14, height = 6, dpi = 300)

write.csv(comp_sample, 
          file.path(RESULTS_DIR, paste0(DATASET, "_", cell_type_label, "_composition_by_sample.csv")),
          row.names = FALSE)

cat("\n✓ Saved:", paste0(DATASET, "_", cell_type_label, "_composition_by_sample.svg"), "\n")
cat("✓ Saved:", paste0(DATASET, "_", cell_type_label, "_composition_stats.csv"), "\n")

# ─────────────────────────────────────────────────────────────────────────────
# SUMMARY
# ─────────────────────────────────────────────────────────────────────────────

sig_states <- comp_stats %>% filter(p_adj < 0.05)

cat("\n================================================================================\n")
cat("SIGNIFICANT CHANGES (p_adj < 0.05):\n")
cat("================================================================================\n")

if (nrow(sig_states) > 0) {
    direction_text <- if (is_aging) {
        c("Up" = "increases with age", "Down" = "decreases with age")
    } else {
        c("Up" = "increases with disease", "Down" = "decreases with disease")
    }
    for (i in 1:nrow(sig_states)) {
        dir_desc <- direction_text[sig_states$direction[i]]
        cat(sprintf("  - %s %s (p_adj = %.2e)\n", sig_states$State[i], dir_desc, sig_states$p_adj[i]))
    }
} else {
    cat("  No significant changes detected\n")
}

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# PLOT 3: UMAP PER STUDY GROUP
# ════════════════════════════════════════════════════════════════════════════════

cat("\n--- Plot 3: UMAP per Study Group ---\n")

cell_type_label <- gsub("_state$", "", SUBCLUSTER_COL)
is_aging <- STUDY_TYPE == "aging"

# ─────────────────────────────────────────────────────────────────────────────
# SETUP
# ─────────────────────────────────────────────────────────────────────────────

sg_detected <- unique(seurat_obj@meta.data[[STUDY_GROUP_COL]])
sg_order <- names(STUDY_GROUP_COLORS)[names(STUDY_GROUP_COLORS) %in% sg_detected]

if (is_aging) {
    sg_labels <- setNames(gsub("Age_", "", gsub("_", "-", sg_order)), sg_order)
} else {
    sg_labels <- setNames(gsub("_", " ", sg_order), sg_order)
}

cat("Study groups:", paste(sg_order, collapse = ", "), "\n")

state_order <- seurat_obj@meta.data %>%
    count(.data[[SUBCLUSTER_COL]]) %>%
    arrange(desc(n)) %>%
    pull(.data[[SUBCLUSTER_COL]])

state_colors <- get_state_colors(state_order)

cat("States:", paste(state_order, collapse = ", "), "\n")

# ─────────────────────────────────────────────────────────────────────────────
# HELPER FUNCTIONS
# ─────────────────────────────────────────────────────────────────────────────

add_corner_arrows <- function(p, label_size = 3) {
    build <- ggplot_build(p)
    x_range <- build$layout$panel_params[[1]]$x.range
    y_range <- build$layout$panel_params[[1]]$y.range
    
    x_start <- x_range[1] + diff(x_range) * 0.02
    y_start <- y_range[1] + diff(y_range) * 0.02
    x_arrow <- diff(x_range) * 0.12
    y_arrow <- diff(y_range) * 0.12
    
    p + 
        annotate("segment", x = x_start, xend = x_start + x_arrow, y = y_start, yend = y_start,
                 arrow = arrow(length = unit(0.12, "cm"), type = "closed"), linewidth = 0.4) +
        annotate("text", x = x_start + x_arrow/2, y = y_start - diff(y_range) * 0.04,
                 label = "UMAP1", size = label_size, hjust = 0.5, vjust = 1, fontface = "bold") +
        annotate("segment", x = x_start, xend = x_start, y = y_start, yend = y_start + y_arrow,
                 arrow = arrow(length = unit(0.12, "cm"), type = "closed"), linewidth = 0.4) +
        annotate("text", x = x_start - diff(x_range) * 0.04, y = y_start + y_arrow/2,
                 label = "UMAP2", size = label_size, hjust = 1, vjust = 0.5, angle = 90, fontface = "bold") +
        coord_cartesian(clip = "off")
}

umap_theme <- theme_void(base_size = 12) +
    theme(
        plot.title = element_text(size = 13, face = "bold", hjust = 0.5),
        legend.position = "none",
        plot.margin = margin(15, 15, 20, 20)
    )

# ─────────────────────────────────────────────────────────────────────────────
# GENERATE UMAPS
# ─────────────────────────────────────────────────────────────────────────────

umap_list <- list()

for (sg in sg_order) {
    cells_sg <- colnames(seurat_obj)[seurat_obj@meta.data[[STUDY_GROUP_COL]] == sg]
    n_cells <- length(cells_sg)
    sg_label <- sg_labels[sg]
    
    p <- DimPlot(seurat_obj, reduction = "umap", cells = cells_sg,
                 group.by = SUBCLUSTER_COL, pt.size = 0.3) +
        scale_color_manual(values = state_colors) +
        ggtitle(paste0(sg_label, "\n(n=", format(n_cells, big.mark = ","), ")")) +
        umap_theme
    
    p <- add_corner_arrows(p, label_size = 2.5)
    umap_list[[sg]] <- p
}

# ─────────────────────────────────────────────────────────────────────────────
# SHARED LEGEND
# ─────────────────────────────────────────────────────────────────────────────

p_legend <- DimPlot(seurat_obj, reduction = "umap", group.by = SUBCLUSTER_COL, pt.size = 0.5) +
    scale_color_manual(values = state_colors) +
    theme_void() +
    theme(
        legend.position = "right",
        legend.title = element_text(size = 12, face = "bold"),
        legend.text = element_text(size = 10),
        legend.key.size = unit(0.5, "cm")
    ) +
    labs(color = paste0(tools::toTitleCase(cell_type_label), " State")) +
    guides(color = guide_legend(ncol = 1, override.aes = list(size = 4)))

legend <- cowplot::get_legend(p_legend)

# ─────────────────────────────────────────────────────────────────────────────
# COMBINE
# ─────────────────────────────────────────────────────────────────────────────

n_groups <- length(sg_order)
n_cols <- 4
n_rows <- ceiling(n_groups / n_cols)

while (length(umap_list) < n_rows * n_cols) {
    umap_list[[length(umap_list) + 1]] <- plot_spacer()
}

options(repr.plot.width = 16, repr.plot.height = 4 * n_rows + 1)

umap_grid <- wrap_plots(umap_list, ncol = n_cols)

title_text <- if (is_aging) {
    paste0(tools::toTitleCase(cell_type_label), " States by Age Group")
} else {
    paste0(tools::toTitleCase(cell_type_label), " States by Disease Group")
}

umap_panel_sg <- (umap_grid | legend) + 
    plot_layout(widths = c(1, 0.15)) +
    plot_annotation(
        title = title_text,
        theme = theme(plot.title = element_text(size = 16, face = "bold", hjust = 0.5))
    )

print(umap_panel_sg)

ggsave(file.path(FIGURES_DIR, paste0(DATASET, "_", cell_type_label, "_umap_by_studygroup.svg")), 
       umap_panel_sg, width = 16, height = 4 * n_rows + 1, dpi = 300)

cat("\n✓ Saved:", paste0(DATASET, "_", cell_type_label, "_umap_by_studygroup.svg"), "\n")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# PLOT: UMAP PER STUDY GROUP (SnC HIGHLIGHT)
# ════════════════════════════════════════════════════════════════════════════════

cat("\n--- Plot: UMAP per Study Group (SnC Highlight) ---\n")

cell_type_label <- gsub("_state$", "", SUBCLUSTER_COL)
is_aging <- STUDY_TYPE == "aging"

# ─────────────────────────────────────────────────────────────────────────────
# SETUP
# ─────────────────────────────────────────────────────────────────────────────

sg_detected <- unique(seurat_obj@meta.data[[STUDY_GROUP_COL]])
sg_order <- names(STUDY_GROUP_COLORS)[names(STUDY_GROUP_COLORS) %in% sg_detected]

if (is_aging) {
    sg_labels <- setNames(gsub("Age_", "", gsub("_", "-", sg_order)), sg_order)
} else {
    sg_labels <- setNames(gsub("_", " ", sg_order), sg_order)
}

cat("Study groups:", paste(sg_order, collapse = ", "), "\n")
cat("Senescence column:", SENESCENCE_LABEL_COL, "\n")

# ─────────────────────────────────────────────────────────────────────────────
# HELPER FUNCTIONS
# ─────────────────────────────────────────────────────────────────────────────

add_corner_arrows <- function(p, label_size = 3) {
    build <- ggplot_build(p)
    x_range <- build$layout$panel_params[[1]]$x.range
    y_range <- build$layout$panel_params[[1]]$y.range
    
    x_start <- x_range[1] + diff(x_range) * 0.02
    y_start <- y_range[1] + diff(y_range) * 0.02
    x_arrow <- diff(x_range) * 0.12
    y_arrow <- diff(y_range) * 0.12
    
    p + 
        annotate("segment", x = x_start, xend = x_start + x_arrow, y = y_start, yend = y_start,
                 arrow = arrow(length = unit(0.12, "cm"), type = "closed"), linewidth = 0.4) +
        annotate("text", x = x_start + x_arrow/2, y = y_start - diff(y_range) * 0.04,
                 label = "UMAP1", size = label_size, hjust = 0.5, vjust = 1, fontface = "bold") +
        annotate("segment", x = x_start, xend = x_start, y = y_start, yend = y_start + y_arrow,
                 arrow = arrow(length = unit(0.12, "cm"), type = "closed"), linewidth = 0.4) +
        annotate("text", x = x_start - diff(x_range) * 0.04, y = y_start + y_arrow/2,
                 label = "UMAP2", size = label_size, hjust = 1, vjust = 0.5, angle = 90, fontface = "bold") +
        coord_cartesian(clip = "off")
}

umap_theme <- theme_void(base_size = 12) +
    theme(
        plot.title = element_text(size = 13, face = "bold", hjust = 0.5),
        legend.position = "none",
        plot.margin = margin(15, 15, 20, 20)
    )

# ─────────────────────────────────────────────────────────────────────────────
# GENERATE UMAPS (SnC highlighted, plotted on top)
# ─────────────────────────────────────────────────────────────────────────────

umap_list <- list()

for (sg in sg_order) {
    cells_sg <- colnames(seurat_obj)[seurat_obj@meta.data[[STUDY_GROUP_COL]] == sg]
    n_cells <- length(cells_sg)
    n_snc <- sum(seurat_obj@meta.data[cells_sg, SENESCENCE_LABEL_COL] == "SnC")
    sg_label <- sg_labels[sg]
    
    p <- DimPlot(seurat_obj, reduction = "umap", cells = cells_sg,
                 group.by = SENESCENCE_LABEL_COL, pt.size = 0.3,
                 order = c("SnC", "Non-SnC")) +
        scale_color_manual(values = SENESCENCE_COLORS) +
        ggtitle(paste0(sg_label, "\n(n=", format(n_cells, big.mark = ","), 
                       ", SnC=", format(n_snc, big.mark = ","), ")")) +
        umap_theme
    
    p <- add_corner_arrows(p, label_size = 2.5)
    umap_list[[sg]] <- p
}

# ─────────────────────────────────────────────────────────────────────────────
# SHARED LEGEND
# ─────────────────────────────────────────────────────────────────────────────

p_legend <- DimPlot(seurat_obj, reduction = "umap", group.by = SENESCENCE_LABEL_COL, 
                    pt.size = 0.5, order = c("SnC", "Non-SnC")) +
    scale_color_manual(values = SENESCENCE_COLORS) +
    theme_void() +
    theme(
        legend.position = "right",
        legend.title = element_text(size = 12, face = "bold"),
        legend.text = element_text(size = 10),
        legend.key.size = unit(0.5, "cm")
    ) +
    labs(color = "Senescence") +
    guides(color = guide_legend(ncol = 1, override.aes = list(size = 4)))

legend <- cowplot::get_legend(p_legend)

# ─────────────────────────────────────────────────────────────────────────────
# COMBINE
# ─────────────────────────────────────────────────────────────────────────────

n_groups <- length(sg_order)
n_cols <- 4
n_rows <- ceiling(n_groups / n_cols)

while (length(umap_list) < n_rows * n_cols) {
    umap_list[[length(umap_list) + 1]] <- plot_spacer()
}

options(repr.plot.width = 16, repr.plot.height = 4 * n_rows + 1)

umap_grid <- wrap_plots(umap_list, ncol = n_cols)

title_text <- if (is_aging) {
    paste0(tools::toTitleCase(cell_type_label), " Senescence by Age Group")
} else {
    paste0(tools::toTitleCase(cell_type_label), " Senescence by Disease Group")
}

umap_panel_snc <- (umap_grid | legend) + 
    plot_layout(widths = c(1, 0.15)) +
    plot_annotation(
        title = title_text,
        theme = theme(plot.title = element_text(size = 16, face = "bold", hjust = 0.5))
    )

print(umap_panel_snc)

ggsave(file.path(FIGURES_DIR, paste0(DATASET, "_", cell_type_label, "_umap_snc_by_studygroup.svg")), 
       umap_panel_snc, width = 16, height = 4 * n_rows + 1, dpi = 300)

cat("\n✓ Saved:", paste0(DATASET, "_", cell_type_label, "_umap_snc_by_studygroup.svg"), "\n")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# SECTION 1: DATA PREPARATION
# ════════════════════════════════════════════════════════════════════════════════

cat("\n", paste(rep("=", 80), collapse = ""), "\n")
cat("SECTION 1: DATA PREPARATION\n")
cat(paste(rep("=", 80), collapse = ""), "\n")

# ─────────────────────────────────────────────────────────────────────────────
# STEP 1.1: CALCULATE %SnC PER STATE PER DONOR
# ─────────────────────────────────────────────────────────────────────────────

cat("\n[Step 1.1] Calculating %SnC per state per donor...\n")

snc_by_state_donor <- seurat_obj@meta.data %>%
    group_by(.data[[DONOR_COL]], .data[[STUDY_GROUP_COL]], .data[[SUBCLUSTER_COL]]) %>%
    summarise(
        n_total = n(),
        n_snc = sum(.data[[SENESCENCE_LABEL_COL]] == "SnC"),
        pct_snc = n_snc / n_total * 100,
        mean_log_umi = mean(log10(total_counts + 1)),
        .groups = "drop"
    )

colnames(snc_by_state_donor)[1:3] <- c("Donor", "Study_Group", "State")

snc_by_state_donor$Donor <- as.character(snc_by_state_donor$Donor)
snc_by_state_donor$Study_Group <- as.character(snc_by_state_donor$Study_Group)
snc_by_state_donor$State <- as.character(snc_by_state_donor$State)

cat("  Rows:", nrow(snc_by_state_donor), "\n")
cat("  Unique donors:", n_distinct(snc_by_state_donor$Donor), "\n")
cat("  Unique states:", n_distinct(snc_by_state_donor$State), "\n")

# ─────────────────────────────────────────────────────────────────────────────
# STEP 1.2: COMPLETE MISSING COMBINATIONS (fill with 0% SnC)
# ─────────────────────────────────────────────────────────────────────────────

cat("\n[Step 1.2] Completing missing donor-state combinations...\n")

donor_sg <- snc_by_state_donor %>%
    select(Donor, Study_Group) %>%
    distinct()

all_states <- unique(snc_by_state_donor$State)

complete_grid <- donor_sg %>%
    crossing(State = all_states)

snc_by_state_donor <- complete_grid %>%
    left_join(snc_by_state_donor %>% select(Donor, State, n_total, n_snc, pct_snc, mean_log_umi), 
              by = c("Donor", "State")) %>%
    mutate(
        n_total = replace_na(n_total, 0),
        n_snc = replace_na(n_snc, 0),
        pct_snc = replace_na(pct_snc, 0)
        # mean_log_umi stays NA for missing combinations
    )

cat("  Rows after completion:", nrow(snc_by_state_donor), "\n")
cat("  Expected:", n_distinct(donor_sg$Donor), "x", length(all_states), "=", 
    n_distinct(donor_sg$Donor) * length(all_states), "\n")

# ─────────────────────────────────────────────────────────────────────────────
# STEP 1.3: MERGE DONOR-LEVEL METADATA (Age, Sex, Cohort)
# ─────────────────────────────────────────────────────────────────────────────

cat("\n[Step 1.3] Merging donor-level metadata...\n")

available_cols <- colnames(seurat_obj@meta.data)

cols_to_extract <- c(DONOR_COL)

has_primary_var <- PRIMARY_VAR %in% available_cols
if (has_primary_var) {
    cols_to_extract <- c(cols_to_extract, PRIMARY_VAR)
    cat("  Found PRIMARY_VAR:", PRIMARY_VAR, "\n")
} else {
    cat("  WARNING: PRIMARY_VAR", PRIMARY_VAR, "not found in metadata\n")
}

has_sex <- SEX_COL %in% available_cols
if (has_sex) {
    cols_to_extract <- c(cols_to_extract, SEX_COL)
    cat("  Found SEX_COL:", SEX_COL, "\n")
}

has_cohort <- COHORT_COL %in% available_cols
if (has_cohort) {
    cols_to_extract <- c(cols_to_extract, COHORT_COL)
    cat("  Found COHORT_COL:", COHORT_COL, "\n")
}

donor_meta <- seurat_obj@meta.data %>%
    select(all_of(cols_to_extract)) %>%
    distinct()

colnames(donor_meta)[1] <- "Donor"
if (has_primary_var) colnames(donor_meta)[colnames(donor_meta) == PRIMARY_VAR] <- "PrimaryVar"
if (has_sex) colnames(donor_meta)[colnames(donor_meta) == SEX_COL] <- "Sex"
if (has_cohort) colnames(donor_meta)[colnames(donor_meta) == COHORT_COL] <- "Cohort"

donor_meta$Donor <- as.character(donor_meta$Donor)

cat("  Donor metadata rows:", nrow(donor_meta), "\n")
cat("  Donor metadata columns:", paste(colnames(donor_meta), collapse = ", "), "\n")

snc_by_state_donor <- snc_by_state_donor %>%
    left_join(donor_meta, by = "Donor")

cat("  Final columns:", paste(colnames(snc_by_state_donor), collapse = ", "), "\n")

# ─────────────────────────────────────────────────────────────────────────────
# STEP 1.4: CREATE NUMERIC PREDICTOR
# ─────────────────────────────────────────────────────────────────────────────

cat("\n[Step 1.4] Creating numeric predictor...\n")

cat("  Study type:", STUDY_TYPE, "\n")
cat("  Primary variable:", PRIMARY_VAR, "(", PRIMARY_VAR_TYPE, ")\n")

is_aging <- STUDY_TYPE == "aging"

if (PRIMARY_VAR_TYPE == "continuous" && has_primary_var) {
    snc_by_state_donor$Predictor <- as.numeric(snc_by_state_donor$PrimaryVar)
    predictor_desc <- paste0(PRIMARY_VAR, " (continuous)")
    cat("  Using continuous predictor:", PRIMARY_VAR, "\n")
    cat("  Range:", min(snc_by_state_donor$Predictor, na.rm = TRUE), "-", 
        max(snc_by_state_donor$Predictor, na.rm = TRUE), "\n")
} else {
    sg_order <- names(STUDY_GROUP_COLORS)[names(STUDY_GROUP_COLORS) %in% unique(snc_by_state_donor$Study_Group)]
    snc_by_state_donor$Study_Group <- factor(snc_by_state_donor$Study_Group, levels = sg_order)
    snc_by_state_donor$Predictor <- as.numeric(snc_by_state_donor$Study_Group)
    predictor_desc <- paste0("Study_Group (ordinal)")
    cat("  Using ordinal predictor from Study_Group\n")
    cat("  Levels:", paste(sg_order, collapse = ", "), "\n")
}

# ─────────────────────────────────────────────────────────────────────────────
# STEP 1.5: SETUP ORDERS AND LABELS
# ─────────────────────────────────────────────────────────────────────────────

cat("\n[Step 1.5] Setting up orders and labels...\n")

sg_detected <- unique(snc_by_state_donor$Study_Group)
sg_order <- names(STUDY_GROUP_COLORS)[names(STUDY_GROUP_COLORS) %in% sg_detected]

state_order <- snc_by_state_donor %>%
    group_by(State) %>%
    summarise(median_snc = median(pct_snc), .groups = "drop") %>%
    arrange(desc(median_snc)) %>%
    pull(State)

if (is_aging) {
    x_labels <- setNames(gsub("Age_", "", gsub("_", "-", sg_order)), sg_order)
    x_title <- "Age Group (years)"
} else {
    x_labels <- setNames(gsub("_", " ", sg_order), sg_order)
    x_title <- "Disease Status"
}

snc_by_state_donor$Study_Group <- factor(snc_by_state_donor$Study_Group, levels = sg_order)
snc_by_state_donor$State <- factor(snc_by_state_donor$State, levels = state_order)

cat("  Study groups:", paste(sg_order, collapse = ", "), "\n")
cat("  States (by median %SnC):", paste(state_order, collapse = ", "), "\n")

cat("\n[Section 1 Complete]\n")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# SECTION 2: STATISTICAL MODELING
# ════════════════════════════════════════════════════════════════════════════════

cat("\n", paste(rep("=", 80), collapse = ""), "\n")
cat("SECTION 2: STATISTICAL MODELING\n")
cat(paste(rep("=", 80), collapse = ""), "\n")

# Cell type label for filenames
cell_type_label <- gsub("_state$", "", SUBCLUSTER_COL)

# ─────────────────────────────────────────────────────────────────────────────
# STEP 2.1: CHECK COVARIATES
# ─────────────────────────────────────────────────────────────────────────────

cat("\n[Step 2.1] Checking covariates...\n")

has_sex <- "Sex" %in% colnames(snc_by_state_donor)
has_cohort <- "Cohort" %in% colnames(snc_by_state_donor)
has_umi <- "mean_log_umi" %in% colnames(snc_by_state_donor)

cat("  Sex:", ifelse(has_sex, "FOUND", "NOT FOUND"), "\n")
cat("  Cohort:", ifelse(has_cohort, "FOUND", "NOT FOUND"), "\n")
cat("  Log UMI:", ifelse(has_umi, "FOUND", "NOT FOUND"), "\n")

# ─────────────────────────────────────────────────────────────────────────────
# STEP 2.2: BUILD FORMULA AND DESCRIPTOR STRINGS
# ─────────────────────────────────────────────────────────────────────────────

cat("\n[Step 2.2] Building model formula...\n")

covariates_used <- c()
if (has_sex) covariates_used <- c(covariates_used, "Sex")
if (has_cohort) covariates_used <- c(covariates_used, "Cohort")
if (has_umi) covariates_used <- c(covariates_used, "mean_log_umi")

if (length(covariates_used) == 0) {
    cov_part <- ""
    covariates_display <- "none"
} else {
    cov_part <- paste0(" + ", paste(covariates_used, collapse = " + "))
    covariates_display <- paste(covariates_used, collapse = ", ")
}

formula_lm <- paste0("pct_cuberoot ~ Predictor", cov_part)
model_formula <- as.formula(formula_lm)

# Predictor description
if (is_aging) {
    predictor_desc <- "Age (continuous)"
    PRIMARY_VAR <- "Age"
    x_title <- "Age Group"
} else {
    predictor_desc <- "Disease stage (ordinal)"
    PRIMARY_VAR <- "Disease"
    x_title <- "Disease Stage"
}

cat("  Formula:", formula_lm, "\n")
cat("  Predictor:", predictor_desc, "\n")
cat("  Covariates:", covariates_display, "\n")
cat("  PRIMARY_VAR:", PRIMARY_VAR, "\n")

# ─────────────────────────────────────────────────────────────────────────────
# STEP 2.3: SET UP COLORS AND LABELS FOR VISUALIZATION
# ─────────────────────────────────────────────────────────────────────────────

cat("\n[Step 2.3] Setting up colors and labels...\n")

if (is_aging) {
    x_labels <- setNames(
        gsub("Age_", "", gsub("_", "-", sg_order)),
        sg_order
    )
} else {
    x_labels <- setNames(
        gsub("_", " ", sg_order),
        sg_order
    )
}

cat("  X-axis labels:", paste(x_labels, collapse = ", "), "\n")

if (!exists("STUDY_GROUP_COLORS")) {
    cat("  WARNING: STUDY_GROUP_COLORS not found, creating default palette\n")
    n_groups <- length(sg_order)
    STUDY_GROUP_COLORS <- setNames(
        scales::hue_pal()(n_groups),
        sg_order
    )
}

cat("  Study group colors:", length(STUDY_GROUP_COLORS), "defined\n")

# ─────────────────────────────────────────────────────────────────────────────
# STEP 2.4: PREPARE DATA FOR MODELING
# ─────────────────────────────────────────────────────────────────────────────

cat("\n[Step 2.4] Preparing data for modeling...\n")

snc_model_data <- snc_by_state_donor %>%
    mutate(pct_cuberoot = (pct_snc / 100)^(1/3))

if (!"Predictor" %in% colnames(snc_model_data)) {
    cat("  Adding Predictor column...\n")
    if (is_aging) {
        snc_model_data <- snc_model_data %>%
            mutate(Predictor = PrimaryVar)
    } else {
        disease_order <- sg_order
        snc_model_data <- snc_model_data %>%
            mutate(Predictor = as.numeric(factor(Study_Group, levels = disease_order)) - 1)
    }
}

if (has_sex) snc_model_data$Sex <- as.factor(snc_model_data$Sex)
if (has_cohort) snc_model_data$Cohort <- as.factor(snc_model_data$Cohort)

cat("  Rows:", nrow(snc_model_data), "\n")
cat("  Predictor range:", min(snc_model_data$Predictor, na.rm = TRUE), "-", 
    max(snc_model_data$Predictor, na.rm = TRUE), "\n")
if (has_umi) {
    cat("  Log UMI range:", 
        sprintf("%.2f - %.2f", 
                min(snc_model_data$mean_log_umi, na.rm = TRUE),
                max(snc_model_data$mean_log_umi, na.rm = TRUE)), "\n")
}

# ─────────────────────────────────────────────────────────────────────────────
# STEP 2.5: RUN LINEAR MODEL PER STATE
# ─────────────────────────────────────────────────────────────────────────────

cat("\n[Step 2.5] Running linear models per state...\n")
cat("  Formula:", formula_lm, "\n\n")

results_list <- list()

for (state in state_order) {
    df_state <- snc_model_data %>% filter(State == state)
    n_donors <- n_distinct(df_state$Donor)
    
    model <- tryCatch(
        lm(model_formula, data = df_state),
        error = function(e) {
            cat(sprintf("    %s: MODEL FAILED — %s\n", state, e$message))
            return(NULL)
        }
    )
    
    if (is.null(model)) next
    
    coef_summary <- summary(model)$coefficients
    
    results_list[[state]] <- data.frame(
        State = state,
        n_donors = n_donors,
        estimate = coef_summary["Predictor", "Estimate"],
        std_error = coef_summary["Predictor", "Std. Error"],
        p_value = coef_summary["Predictor", "Pr(>|t|)"]
    )
    
    cat(sprintf("    %s (n=%d): b=%.4f, p=%.2e\n", 
                state, n_donors, 
                coef_summary["Predictor", "Estimate"],
                coef_summary["Predictor", "Pr(>|t|)"]))
}

# ─────────────────────────────────────────────────────────────────────────────
# STEP 2.6: MULTIPLE TESTING CORRECTION
# ─────────────────────────────────────────────────────────────────────────────

cat("\n[Step 2.6] Applying BH (FDR) correction...\n")

snc_state_stats <- do.call(rbind, results_list) %>%
    mutate(
        p_adj = p.adjust(p_value, method = "BH"),
        direction = case_when(
            estimate > 0 ~ "Up",
            estimate < 0 ~ "Down",
            TRUE ~ ""
        ),
        sig = case_when(
            p_adj < 0.001 ~ "***",
            p_adj < 0.01 ~ "**",
            p_adj < 0.05 ~ "*",
            TRUE ~ ""
        ),
        Significant = p_adj < 0.05
    ) %>%
    arrange(p_adj)

rownames(snc_state_stats) <- NULL

cat("\n  Results (BH-adjusted):\n")
print(snc_state_stats %>% select(State, n_donors, estimate, std_error, p_value, p_adj, direction, sig))

sig_results <- snc_state_stats %>% filter(Significant)

cat("\n  Significant states (FDR < 0.05):", nrow(sig_results), "of", nrow(snc_state_stats), "\n")

if (nrow(sig_results) > 0) {
    for (i in 1:nrow(sig_results)) {
        dir_text <- ifelse(sig_results$direction[i] == "Up",
                           paste0("increases with ", PRIMARY_VAR),
                           paste0("decreases with ", PRIMARY_VAR))
        cat(sprintf("    • %s: SnC %s (b=%.4f, p_adj=%.3f)\n",
                    sig_results$State[i], dir_text,
                    sig_results$estimate[i], sig_results$p_adj[i]))
    }
}

# ─────────────────────────────────────────────────────────────────────────────
# STEP 2.7: SAVE RESULTS
# ─────────────────────────────────────────────────────────────────────────────

cat("\n[Step 2.7] Saving results...\n")

write.csv(snc_state_stats, 
          file.path(RESULTS_DIR, paste0(DATASET, "_", cell_type_label, "_snc_state_stats.csv")),
          row.names = FALSE)

cat("  Saved:", paste0(DATASET, "_", cell_type_label, "_snc_state_stats.csv"), "\n")

write.csv(snc_by_state_donor %>%
              group_by(State, Study_Group) %>%
              summarise(
                  mean_pct = mean(pct_snc),
                  sd_pct = sd(pct_snc),
                  n = n(),
                  .groups = "drop"
              ),
          file.path(RESULTS_DIR, paste0(DATASET, "_", cell_type_label, "_snc_by_state_studygroup.csv")),
          row.names = FALSE)

cat("  Saved:", paste0(DATASET, "_", cell_type_label, "_snc_by_state_studygroup.csv"), "\n")

cat("\n[Section 2 Complete]\n")
cat("  Formula:", formula_lm, "\n")
cat("  Covariates:", covariates_display, "\n")
cat("  Variables created: formula_lm, model_formula, predictor_desc, covariates_used,\n")
cat("    snc_state_stats, sig_results, PRIMARY_VAR, x_title, x_labels\n")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# SECTION 3: VISUALIZATION
# ════════════════════════════════════════════════════════════════════════════════

cat("\n", paste(rep("=", 80), collapse = ""), "\n")
cat("SECTION 3: VISUALIZATION\n")
cat(paste(rep("=", 80), collapse = ""), "\n")

# Derive cell type label (reuse from Section 2, or create here)
cell_type_label <- gsub("_state$", "", SUBCLUSTER_COL)

# ─────────────────────────────────────────────────────────────────────────────
# STEP 3.1: DISPLAY SUMMARY TABLE
# ─────────────────────────────────────────────────────────────────────────────

cat("\n[Step 3.1] Summary table: Mean %SnC by State x Study Group\n\n")

snc_summary_display <- snc_by_state_donor %>%
    group_by(State, Study_Group) %>%
    summarise(
        mean_pct = mean(pct_snc),
        sd_pct = sd(pct_snc),
        n = n(),
        .groups = "drop"
    ) %>%
    mutate(display = sprintf("%.1f ± %.1f", mean_pct, sd_pct)) %>%
    select(State, Study_Group, display) %>%
    pivot_wider(names_from = Study_Group, values_from = display)

print(as.data.frame(snc_summary_display))

# ─────────────────────────────────────────────────────────────────────────────
# STEP 3.2: DISPLAY MODEL FORMULA AND RESULTS
# ─────────────────────────────────────────────────────────────────────────────

cat("\n[Step 3.2] Model specification\n")

cat("\n", paste(rep("-", 60), collapse = ""), "\n")
cat("MODEL FORMULA\n")
cat(paste(rep("-", 60), collapse = ""), "\n")

cat("\n  Response: pct_cuberoot = (pct_snc / 100)^(1/3)\n")
cat("  Formula:", formula_lm, "\n")
cat("  Predictor:", predictor_desc, "\n")
cat("  Covariates:", paste(covariates_used, collapse = ", "), "\n")
cat("  Multiple testing: BH (FDR) correction\n")

cat("\n", paste(rep("-", 60), collapse = ""), "\n")
cat("RESULTS\n")
cat(paste(rep("-", 60), collapse = ""), "\n\n")

results_display <- snc_state_stats %>%
    select(State, n_donors, estimate, std_error, p_value, p_adj, direction, sig) %>%
    mutate(
        estimate = sprintf("%.4f", estimate),
        std_error = sprintf("%.4f", std_error),
        p_value = sprintf("%.2e", p_value),
        p_adj = sprintf("%.3f", p_adj)
    )

print(as.data.frame(results_display))

if (nrow(sig_results) > 0) {
    cat("\nSignificant associations (FDR < 0.05):\n")
    for (i in 1:nrow(sig_results)) {
        dir_text <- ifelse(sig_results$direction[i] == "Up", 
                           paste0("increases with ", PRIMARY_VAR),
                           paste0("decreases with ", PRIMARY_VAR))
        cat(sprintf("  • %s: SnC %s (b=%s, p_adj=%s)\n", 
                    sig_results$State[i], 
                    dir_text,
                    sprintf("%.4f", sig_results$estimate[i]),
                    sprintf("%.3f", sig_results$p_adj[i])))
    }
} else {
    cat("\nNo significant associations (FDR < 0.05)\n")
}

# ─────────────────────────────────────────────────────────────────────────────
# STEP 3.3: CHECK MAX %SNC PER STATE (DONOR-LEVEL)
# ─────────────────────────────────────────────────────────────────────────────

cat("\n[Step 3.3] Checking max %SnC per state (donor-level)...\n\n")

state_y_stats <- snc_by_state_donor %>%
    group_by(State) %>%
    summarise(
        n_donors = n(),
        min_pct = min(pct_snc, na.rm = TRUE),
        max_pct = max(pct_snc, na.rm = TRUE),
        mean_pct = mean(pct_snc, na.rm = TRUE),
        median_pct = median(pct_snc, na.rm = TRUE),
        .groups = "drop"
    ) %>%
    arrange(desc(max_pct))

cat("  Max %SnC per state (donor-level):\n")
cat("  ", paste(rep("-", 60), collapse = ""), "\n")
cat(sprintf("  %-25s %8s %8s %8s\n", "State", "Min", "Max", "Mean"))
cat("  ", paste(rep("-", 60), collapse = ""), "\n")

for (i in 1:nrow(state_y_stats)) {
    cat(sprintf("  %-25s %7.1f%% %7.1f%% %7.1f%%\n", 
                state_y_stats$State[i],
                state_y_stats$min_pct[i],
                state_y_stats$max_pct[i],
                state_y_stats$mean_pct[i]))
}
cat("  ", paste(rep("-", 60), collapse = ""), "\n")

y_max_lookup <- state_y_stats %>%
    mutate(y_limit = max_pct * 1.15) %>%
    select(State, max_pct, y_limit)

cat("\n  Y-axis limits to use:\n")
for (i in 1:nrow(y_max_lookup)) {
    cat(sprintf("    %s: 0 - %.1f%% (max=%.1f%%)\n", 
                y_max_lookup$State[i],
                y_max_lookup$y_limit[i],
                y_max_lookup$max_pct[i]))
}

# ─────────────────────────────────────────────────────────────────────────────
# STEP 3.4: PREPARE PLOT DATA
# ─────────────────────────────────────────────────────────────────────────────

cat("\n[Step 3.4] Preparing plot data...\n")

sig_lookup <- setNames(snc_state_stats$sig, snc_state_stats$State)
dir_lookup <- setNames(snc_state_stats$direction, snc_state_stats$State)

facet_labels <- sapply(state_order, function(s) {
    sig <- sig_lookup[s]
    dir <- dir_lookup[s]
    if (!is.na(sig) && sig != "") {
        dir_symbol <- ifelse(dir == "Up", "(+)", "(-)")
        paste0(s, " ", dir_symbol, sig)
    } else {
        s
    }
})

label_map <- setNames(facet_labels, state_order)

# ─────────────────────────────────────────────────────────────────────────────
# STEP 3.5: CREATE INDIVIDUAL PLOTS PER STATE
# ─────────────────────────────────────────────────────────────────────────────

cat("\n[Step 3.5] Creating individual plots per state...\n")

suppressPackageStartupMessages({
    if (!require(cowplot, quietly = TRUE)) {
        install.packages("cowplot", quiet = TRUE)
        library(cowplot)
    }
})

plot_list <- list()

for (s in state_order) {
    
    state_data <- snc_by_state_donor %>% filter(State == s)
    state_stats <- snc_state_stats %>% filter(State == s)
    y_limit <- y_max_lookup %>% filter(State == s) %>% pull(y_limit)
    
    p_fmt <- if (state_stats$p_value < 0.001) {
        sprintf("%.1e", state_stats$p_value)
    } else if (state_stats$p_value < 0.01) {
        sprintf("%.3f", state_stats$p_value)
    } else {
        sprintf("%.2f", state_stats$p_value)
    }
    
    label_text <- paste0("b = ", sprintf("%.4f", state_stats$estimate), "\np = ", p_fmt)
    label_color <- ifelse(state_stats$Significant, "#333333", "#888888")
    label_face <- ifelse(state_stats$Significant, "bold", "plain")
    facet_title <- label_map[s]
    
    cat(sprintf("    %s: y_limit = %.1f%%\n", s, y_limit))
    
    p <- ggplot(state_data, aes(x = Study_Group, y = pct_snc)) +
        geom_boxplot(aes(fill = Study_Group), 
                     alpha = 0.7, outlier.shape = NA, width = 0.6,
                     linewidth = 0.5, color = "black") +
        geom_jitter(aes(color = Study_Group), 
                    width = 0.15, size = 1.2, alpha = 0.6) +
        geom_smooth(aes(x = as.numeric(Study_Group), y = pct_snc),
                    method = "lm", se = FALSE,
                    color = "black", linetype = "dashed", linewidth = 0.8) +
        annotate("text", x = 1, y = y_limit * 0.92, 
                 label = label_text, hjust = 0, vjust = 1,
                 size = 2.5, color = label_color, fontface = label_face) +
        coord_cartesian(ylim = c(0, y_limit)) +
        scale_fill_manual(values = STUDY_GROUP_COLORS) +
        scale_color_manual(values = STUDY_GROUP_COLORS) +
        scale_x_discrete(labels = x_labels) +
        labs(title = facet_title, x = NULL, y = NULL) +
        theme_minimal(base_size = 10) +
        theme(
            plot.title = element_text(size = 10, face = "bold", hjust = 0.5),
            axis.text.x = element_text(angle = 45, hjust = 1, vjust = 1, size = 8, 
                                       color = "black", face = "bold"),
            axis.text.y = element_text(size = 8, color = "black"),
            axis.line = element_line(color = "black", linewidth = 0.5),
            axis.ticks = element_line(color = "black", linewidth = 0.3),
            panel.grid = element_blank(),
            panel.border = element_rect(color = "black", fill = NA, linewidth = 0.5),
            legend.position = "none",
            plot.margin = margin(5, 5, 5, 5)
        )
    
    plot_list[[s]] <- p
}

# ─────────────────────────────────────────────────────────────────────────────
# STEP 3.6: COMBINE PLOTS
# ─────────────────────────────────────────────────────────────────────────────

cat("\n[Step 3.6] Combining plots...\n")

n_states <- length(state_order)
n_cols <- min(4, n_states)
n_rows <- ceiling(n_states / n_cols)

fig_width <- 3 * n_cols
fig_height <- 3 * n_rows

options(repr.plot.width = fig_width, repr.plot.height = fig_height)

p_combined <- plot_grid(
    plotlist = plot_list,
    ncol = n_cols,
    align = "hv"
)

p_final <- ggdraw() +
    draw_plot(p_combined, x = 0.03, y = 0.03, width = 0.95, height = 0.88) +
    draw_label(x_title, x = 0.5, y = 0.01, size = 12, fontface = "bold") +
    draw_label("Senescent Cells (%)", x = 0.01, y = 0.5, size = 12, fontface = "bold", angle = 90) +
    draw_label(paste0("Senescent Cell Proportion by ", tools::toTitleCase(cell_type_label), " State — ", DATASET), 
               x = 0.5, y = 0.97, size = 14, fontface = "bold") +
    draw_label(paste0("Model: ", predictor_desc, " + ", paste(covariates_used, collapse = " + ")),
               x = 0.5, y = 0.93, size = 10, color = "gray40")

print(p_final)

# ─────────────────────────────────────────────────────────────────────────────
# STEP 3.7: SAVE FIGURE
# ─────────────────────────────────────────────────────────────────────────────

cat("\n[Step 3.7] Saving figure...\n")

ggsave(file.path(FIGURES_DIR, paste0(DATASET, "_", cell_type_label, "_snc_by_state_boxplot.svg")), 
       p_final, width = fig_width, height = fig_height, dpi = 300)

cat("  Saved:", paste0(DATASET, "_", cell_type_label, "_snc_by_state_boxplot.svg"), "\n")

# ════════════════════════════════════════════════════════════════════════════════
# SUMMARY
# ════════════════════════════════════════════════════════════════════════════════

cat("\n", paste(rep("=", 80), collapse = ""), "\n")
cat("PLOT 4 COMPLETE\n")
cat(paste(rep("=", 80), collapse = ""), "\n")

cat("\nOutputs:\n")
cat("  ", file.path(RESULTS_DIR, paste0(DATASET, "_", cell_type_label, "_snc_state_stats.csv")), "\n")
cat("  ", file.path(RESULTS_DIR, paste0(DATASET, "_", cell_type_label, "_snc_by_state_studygroup.csv")), "\n")
cat("  ", file.path(FIGURES_DIR, paste0(DATASET, "_", cell_type_label, "_snc_by_state_boxplot.svg")), "\n")

cat("\n", paste(rep("=", 80), collapse = ""), "\n")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# STEP 1: DATA PREPARATION — SUBTYPE-LEVEL LOGISTIC GLMM
# ════════════════════════════════════════════════════════════════════════════════

cat("════════════════════════════════════════════════════════════════════════\n")
cat("STEP 1: DATA PREPARATION FOR SUBTYPE GLMM\n")
cat("════════════════════════════════════════════════════════════════════════\n\n")

library(lme4)

# ─────────────────────────────────────────────────────────────────────────────
# Column names (from config)
# ─────────────────────────────────────────────────────────────────────────────

COL_DONOR      <- DONOR_COL
COL_SUBTYPE    <- SUBCLUSTER_COL
COL_SEX        <- SEX_COL
COL_COHORT     <- COHORT_COL
COL_SENESCENT  <- "is_senescent"
COL_STUDYGROUP <- STUDY_GROUP_COL

# ─────────────────────────────────────────────────────────────────────────────
# Extract metadata
# ─────────────────────────────────────────────────────────────────────────────

cat("▸ Extracting metadata from seurat_obj...\n")
cat(sprintf("  Study type: %s | Primary variable: %s (%s)\n",
            STUDY_TYPE, PRIMARY_VAR, PRIMARY_VAR_TYPE))

df_glmm <- seurat_obj@meta.data

# Handle senescence column
if (!COL_SENESCENT %in% colnames(df_glmm)) {
    if (SENESCENCE_LABEL_COL %in% colnames(df_glmm)) {
        df_glmm$is_senescent <- as.integer(df_glmm[[SENESCENCE_LABEL_COL]] == "SnC")
        cat("  ✓ Created is_senescent from senescence_label\n")
    } else {
        stop("No senescence column found.")
    }
}

# Verify required columns
required_cols <- c(COL_DONOR, COL_SUBTYPE, COL_SEX, COL_COHORT, COL_SENESCENT)
if (PRIMARY_VAR_TYPE == "continuous") {
    required_cols <- c(required_cols, PRIMARY_VAR)
} else {
    required_cols <- c(required_cols, COL_STUDYGROUP)
    if (PRIMARY_VAR %in% colnames(df_glmm)) {
        required_cols <- c(required_cols, PRIMARY_VAR)  # Age as covariate in disease
    }
}

missing <- required_cols[!required_cols %in% colnames(df_glmm)]
if (length(missing) > 0) {
    stop(paste("Missing columns:", paste(missing, collapse = ", ")))
}

# ─────────────────────────────────────────────────────────────────────────────
# Clean and type columns
# ─────────────────────────────────────────────────────────────────────────────

cat("▸ Cleaning data...\n")

# Senescence — handle "True"/"False", TRUE/FALSE, "SnC"/"Non-SnC"
sen_vals <- df_glmm[[COL_SENESCENT]]
if (is.character(sen_vals)) {
    df_glmm[[COL_SENESCENT]] <- as.integer(sen_vals %in% c("True", "TRUE", "SnC"))
    cat("  ✓ Converted senescence from character to 0/1\n")
} else if (is.logical(sen_vals)) {
    df_glmm[[COL_SENESCENT]] <- as.integer(sen_vals)
    cat("  ✓ Converted senescence from logical to 0/1\n")
} else {
    df_glmm[[COL_SENESCENT]] <- as.integer(sen_vals)
}

df_glmm[[COL_DONOR]]   <- as.factor(df_glmm[[COL_DONOR]])
df_glmm[[COL_SUBTYPE]] <- as.factor(df_glmm[[COL_SUBTYPE]])
df_glmm[[COL_SEX]]     <- as.factor(df_glmm[[COL_SEX]])
df_glmm[[COL_COHORT]]  <- as.factor(df_glmm[[COL_COHORT]])

# Age — scale by decade (used as primary or covariate)
if (PRIMARY_VAR %in% colnames(df_glmm) || "Age" %in% colnames(df_glmm)) {
    age_col <- ifelse(PRIMARY_VAR %in% colnames(df_glmm) & PRIMARY_VAR_TYPE == "continuous",
                      PRIMARY_VAR, "Age")
    if (age_col %in% colnames(df_glmm)) {
        df_glmm[[age_col]] <- as.numeric(df_glmm[[age_col]])
        df_glmm$Age_scaled <- df_glmm[[age_col]] / 10
        cat(sprintf("  ✓ Age_scaled computed from %s\n", age_col))
    }
}

# Study group — set reference level for disease cohorts
if (PRIMARY_VAR_TYPE == "categorical") {
    df_glmm[[COL_STUDYGROUP]] <- as.factor(df_glmm[[COL_STUDYGROUP]])
    if (!is.null(config$reference_group)) {
        df_glmm[[COL_STUDYGROUP]] <- relevel(df_glmm[[COL_STUDYGROUP]], ref = config$reference_group)
        cat(sprintf("  ✓ Reference group set to: %s\n", config$reference_group))
    }
}

# Log10 total counts
has_counts <- "nCount_RNA" %in% colnames(df_glmm)
if (has_counts) {
    df_glmm$log10_total_counts <- log10(df_glmm$nCount_RNA + 1)
    cat("  ✓ log10_total_counts computed from nCount_RNA\n")
}

# Drop NAs
check_cols <- c(COL_DONOR, COL_SUBTYPE, COL_SEX, COL_COHORT, COL_SENESCENT)
if ("Age_scaled" %in% colnames(df_glmm)) check_cols <- c(check_cols, "Age_scaled")
if (PRIMARY_VAR_TYPE == "categorical") check_cols <- c(check_cols, COL_STUDYGROUP)

n_before <- nrow(df_glmm)
df_glmm <- df_glmm[complete.cases(df_glmm[, check_cols]), ]
n_after <- nrow(df_glmm)
if (n_before != n_after) {
    cat(sprintf("  ⚠ Dropped %d rows with missing values\n", n_before - n_after))
} else {
    cat(sprintf("  ✓ No missing values — all %s cells retained\n", format(n_after, big.mark = ",")))
}

# ─────────────────────────────────────────────────────────────────────────────
# Build formula
# ─────────────────────────────────────────────────────────────────────────────

if (PRIMARY_VAR_TYPE == "continuous") {
    # Aging: Age is primary, Sex/Cohort are covariates
    fixed_parts <- "Age_scaled"
} else {
    # Disease: Study_Group is primary, Age/Sex/Cohort are covariates
    fixed_parts <- COL_STUDYGROUP
    if ("Age_scaled" %in% colnames(df_glmm)) {
        fixed_parts <- c(fixed_parts, "Age_scaled")
    }
}

if (length(levels(df_glmm[[COL_SEX]])) > 1) {
    fixed_parts <- c(fixed_parts, COL_SEX)
}
if (length(levels(df_glmm[[COL_COHORT]])) > 1) {
    fixed_parts <- c(fixed_parts, COL_COHORT)
}
if (has_counts) {
    fixed_parts <- c(fixed_parts, "log10_total_counts")
}

formula_str <- paste0(COL_SENESCENT, " ~ ", paste(fixed_parts, collapse = " + "),
                      " + (1|", COL_DONOR, ")")
formula_glmm <- as.formula(formula_str)

# ─────────────────────────────────────────────────────────────────────────────
# Summary
# ─────────────────────────────────────────────────────────────────────────────

cat(sprintf("\n▸ Data ready: %s cells, %d donors\n",
            format(nrow(df_glmm), big.mark = ","),
            length(unique(df_glmm[[COL_DONOR]]))))
cat(sprintf("  SnC rate: %.2f%%\n", mean(df_glmm[[COL_SENESCENT]]) * 100))
cat(sprintf("  SnC cells: %s / %s\n",
            format(sum(df_glmm[[COL_SENESCENT]]), big.mark = ","),
            format(nrow(df_glmm), big.mark = ",")))
cat(sprintf("  Formula: %s\n", formula_str))

if (PRIMARY_VAR_TYPE == "categorical") {
    cat(sprintf("  Study groups: %s\n",
                paste(levels(df_glmm[[COL_STUDYGROUP]]), collapse = " vs ")))
}

cat(sprintf("\n▸ Subtypes (%d):\n", length(levels(df_glmm[[COL_SUBTYPE]]))))

for (st in levels(df_glmm[[COL_SUBTYPE]])) {
    sub <- df_glmm[df_glmm[[COL_SUBTYPE]] == st, ]
    n <- nrow(sub)
    nd <- length(unique(sub[[COL_DONOR]]))
    sr <- mean(sub[[COL_SENESCENT]]) * 100
    flag <- if (n < 100 | nd < 15) " ⚠ (may be skipped)" else ""
    cat(sprintf("  %-25s n=%6s  donors=%3d  SnC=%.1f%%%s\n",
                st, format(n, big.mark = ","), nd, sr, flag))
}

cat("\n════════════════════════════════════════════════════════════════════════\n")
cat("✓ STEP 1 COMPLETE — df_glmm and formula_glmm ready\n")
cat("════════════════════════════════════════════════════════════════════════\n")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# STEP 2: SUBTYPE-LEVEL LOGISTIC GLMM — MODEL FITTING
# ════════════════════════════════════════════════════════════════════════════════

cat("════════════════════════════════════════════════════════════════════════\n")
cat("STEP 2: SUBTYPE-LEVEL LOGISTIC GLMM\n")
cat("════════════════════════════════════════════════════════════════════════\n\n")

MIN_CELLS  <- 100
MIN_DONORS <- 15

# ─────────────────────────────────────────────────────────────────────────────
# Helper: extract primary variable effect from model coefficients
# ─────────────────────────────────────────────────────────────────────────────

extract_primary_effect <- function(coefs) {
    if (PRIMARY_VAR_TYPE == "continuous") {
        # Aging: extract Age_scaled
        if ("Age_scaled" %in% rownames(coefs)) {
            return(list(
                term = "Age_scaled",
                beta = coefs["Age_scaled", "Estimate"],
                se   = coefs["Age_scaled", "Std. Error"],
                z    = coefs["Age_scaled", "z value"],
                p    = coefs["Age_scaled", "Pr(>|z|)"]
            ))
        }
    } else {
        # Disease: extract Study_Group terms (there may be multiple: MCI, AD)
        sg_rows <- grep(COL_STUDYGROUP, rownames(coefs))
        if (length(sg_rows) > 0) {
            # Return the most significant study group term
            p_vals <- coefs[sg_rows, "Pr(>|z|)"]
            best <- sg_rows[which.min(p_vals)]
            return(list(
                term = rownames(coefs)[best],
                beta = coefs[best, "Estimate"],
                se   = coefs[best, "Std. Error"],
                z    = coefs[best, "z value"],
                p    = coefs[best, "Pr(>|z|)"]
            ))
        }
    }
    return(NULL)
}

# ═══════════════════════════════════════════════════════════════════════════════
# SECTION 1: OVERALL GLMM (ALL CELLS)
# ═══════════════════════════════════════════════════════════════════════════════

cat("─────────────────────────────────────────────────────────────────────\n")
cat("1. OVERALL GLMM (ALL CELLS)\n")
cat("─────────────────────────────────────────────────────────────────────\n")

cat(sprintf("\n▸ Formula: %s\n", deparse(formula_glmm)))
cat("▸ Fitting overall model...\n\n")

overall_model <- tryCatch({
    glmer(
        formula_glmm,
        data = df_glmm,
        family = binomial(link = "logit"),
        control = glmerControl(optimizer = "bobyqa", optCtrl = list(maxfun = 100000)),
        nAGQ = 1
    )
}, error = function(e) {
    cat(sprintf("✗ Overall model failed: %s\n", e$message))
    NULL
})

if (!is.null(overall_model)) {
    overall_coefs <- summary(overall_model)$coefficients
    re_var <- as.numeric(VarCorr(overall_model)[[COL_DONOR]])

    cat("✓ Model converged\n")
    cat(sprintf("  Observations: %s  |  Donors: %d  |  AIC: %.1f\n",
                format(nobs(overall_model), big.mark = ","),
                ngrps(overall_model),
                AIC(overall_model)))

    cat(sprintf("\n  %-30s %10s %10s %10s %12s %8s\n",
                "Term", "β", "SE", "z", "p-value", "OR"))
    cat(paste0("  ", paste(rep("─", 85), collapse = ""), "\n"))

    for (i in seq_len(nrow(overall_coefs))) {
        term <- rownames(overall_coefs)[i]
        beta <- overall_coefs[i, "Estimate"]
        se   <- overall_coefs[i, "Std. Error"]
        z    <- overall_coefs[i, "z value"]
        p    <- overall_coefs[i, "Pr(>|z|)"]
        or   <- exp(beta)
        sig  <- ifelse(p < 0.05, "*", "")
        cat(sprintf("  %-30s %10.4f %10.4f %10.3f %12.2e %8.3f %s\n",
                    term, beta, se, z, p, or, sig))
    }

    cat(sprintf("\n  Random Effects: Donor variance = %.4f, SD = %.4f\n",
                re_var, sqrt(re_var)))

    # Primary variable summary
    primary_eff <- extract_primary_effect(overall_coefs)
    if (!is.null(primary_eff)) {
        or_val <- exp(primary_eff$beta)
        or_lo  <- exp(primary_eff$beta - 1.96 * primary_eff$se)
        or_hi  <- exp(primary_eff$beta + 1.96 * primary_eff$se)

        if (PRIMARY_VAR_TYPE == "continuous") {
            cat(sprintf("\n▸ Age Effect (per decade):\n"))
        } else {
            cat(sprintf("\n▸ Disease Effect (%s):\n", primary_eff$term))
        }
        cat(sprintf("  β = %.4f (SE = %.4f)\n", primary_eff$beta, primary_eff$se))
        cat(sprintf("  OR = %.3f (95%% CI: %.3f – %.3f)\n", or_val, or_lo, or_hi))
        cat(sprintf("  p = %.2e\n", primary_eff$p))

        if (primary_eff$p < 0.05) {
            direction <- ifelse(primary_eff$beta > 0, "INCREASES", "DECREASES")
            pct <- abs((or_val - 1) * 100)
            cat(sprintf("  ✓ Senescence probability %s (+%.1f%% odds)\n", direction, pct))
        }
    }

    # For disease cohort: show ALL study group terms
    if (PRIMARY_VAR_TYPE == "categorical") {
        sg_rows <- grep(COL_STUDYGROUP, rownames(overall_coefs))
        if (length(sg_rows) > 1) {
            cat(sprintf("\n▸ All disease group effects (vs %s):\n", config$reference_group))
            for (i in sg_rows) {
                term <- gsub(COL_STUDYGROUP, "", rownames(overall_coefs)[i])
                beta <- overall_coefs[i, "Estimate"]
                or   <- exp(beta)
                p    <- overall_coefs[i, "Pr(>|z|)"]
                sig  <- ifelse(p < 0.05, "*", "")
                cat(sprintf("  %s: β=%.4f, OR=%.3f, p=%.2e %s\n", term, beta, or, p, sig))
            }
        }
    }
}

# ═══════════════════════════════════════════════════════════════════════════════
# SECTION 2: SUBTYPE-SPECIFIC GLMM
# ═══════════════════════════════════════════════════════════════════════════════

cat("\n─────────────────────────────────────────────────────────────────────\n")
cat("2. SUBTYPE-SPECIFIC GLMM\n")
cat("─────────────────────────────────────────────────────────────────────\n")

subtypes <- levels(df_glmm[[COL_SUBTYPE]])
cat(sprintf("\n▸ Running GLMM for %d subtypes...\n\n", length(subtypes)))

results_list <- list()

for (subtype in subtypes) {

    sub_data <- df_glmm[df_glmm[[COL_SUBTYPE]] == subtype, ]

    n_cells  <- nrow(sub_data)
    n_donors <- length(unique(sub_data[[COL_DONOR]]))
    snc_rate <- mean(sub_data[[COL_SENESCENT]])

    if (n_cells < MIN_CELLS) {
        cat(sprintf("  %-25s SKIPPED (n=%d < %d)\n", subtype, n_cells, MIN_CELLS))
        next
    }
    if (n_donors < MIN_DONORS) {
        cat(sprintf("  %-25s SKIPPED (donors=%d < %d)\n", subtype, n_donors, MIN_DONORS))
        next
    }
    if (snc_rate == 0 | snc_rate == 1) {
        cat(sprintf("  %-25s SKIPPED (no variance)\n", subtype))
        next
    }

    model <- tryCatch({
        glmer(
            formula_glmm,
            data = sub_data,
            family = binomial(link = "logit"),
            control = glmerControl(optimizer = "bobyqa", optCtrl = list(maxfun = 50000)),
            nAGQ = 1
        )
    }, error = function(e) {
        cat(sprintf("  %-25s FAILED — %s\n", subtype, substr(e$message, 1, 60)))
        NULL
    })

    if (!is.null(model)) {
        coefs <- summary(model)$coefficients
        re_var <- as.numeric(VarCorr(model)[[COL_DONOR]])

        # For disease cohorts: store results for EACH study group term
        if (PRIMARY_VAR_TYPE == "categorical") {
            sg_rows <- grep(COL_STUDYGROUP, rownames(coefs))
            for (idx in sg_rows) {
                term_name <- gsub(COL_STUDYGROUP, "", rownames(coefs)[idx])
                beta <- coefs[idx, "Estimate"]
                se   <- coefs[idx, "Std. Error"]
                z    <- coefs[idx, "z value"]
                p    <- coefs[idx, "Pr(>|z|)"]

                results_list[[paste0(subtype, "_", term_name)]] <- data.frame(
                    Subtype        = subtype,
                    Comparison     = paste0(term_name, " vs ", config$reference_group),
                    N_cells        = n_cells,
                    N_donors       = n_donors,
                    SnC_rate       = snc_rate,
                    Beta           = beta,
                    SE             = se,
                    Beta_CI_lower  = beta - 1.96 * se,
                    Beta_CI_upper  = beta + 1.96 * se,
                    OR             = exp(beta),
                    OR_CI_lower    = exp(beta - 1.96 * se),
                    OR_CI_upper    = exp(beta + 1.96 * se),
                    z_value        = z,
                    P_value        = p,
                    Donor_variance = re_var,
                    AIC            = AIC(model),
                    stringsAsFactors = FALSE
                )
            }
            # Print best effect
            best_idx <- sg_rows[which.min(coefs[sg_rows, "Pr(>|z|)"])]
            beta <- coefs[best_idx, "Estimate"]
            p    <- coefs[best_idx, "Pr(>|z|)"]
            term <- gsub(COL_STUDYGROUP, "", rownames(coefs)[best_idx])
            sig  <- ifelse(p < 0.05, "*", "")
            cat(sprintf("  %-25s %s: β=%.4f  OR=%.3f  p=%.2e %s\n",
                        subtype, term, beta, exp(beta), p, sig))

        } else {
            # Aging cohort: extract Age_scaled
            primary_eff <- extract_primary_effect(coefs)
            if (!is.null(primary_eff)) {
                beta <- primary_eff$beta
                se   <- primary_eff$se
                z    <- primary_eff$z
                p    <- primary_eff$p
            } else {
                beta <- NA; se <- NA; z <- NA; p <- NA
            }

            results_list[[subtype]] <- data.frame(
                Subtype        = subtype,
                Comparison     = "per decade",
                N_cells        = n_cells,
                N_donors       = n_donors,
                SnC_rate       = snc_rate,
                Beta           = beta,
                SE             = se,
                Beta_CI_lower  = beta - 1.96 * se,
                Beta_CI_upper  = beta + 1.96 * se,
                OR             = exp(beta),
                OR_CI_lower    = exp(beta - 1.96 * se),
                OR_CI_upper    = exp(beta + 1.96 * se),
                z_value        = z,
                P_value        = p,
                Donor_variance = re_var,
                AIC            = AIC(model),
                stringsAsFactors = FALSE
            )

            sig <- ifelse(p < 0.05, "*", "")
            cat(sprintf("  %-25s β=%.4f  OR=%.3f  p=%.2e %s\n",
                        subtype, beta, exp(beta), p, sig))
        }
    }
}

# ─────────────────────────────────────────────────────────────────────────────
# FDR correction
# ─────────────────────────────────────────────────────────────────────────────

df_glmm_results <- do.call(rbind, results_list)
rownames(df_glmm_results) <- NULL

if (nrow(df_glmm_results) > 0) {

    df_glmm_results$P_adj <- p.adjust(df_glmm_results$P_value, method = "BH")
    df_glmm_results$Significant <- df_glmm_results$P_adj < 0.05
    df_glmm_results <- df_glmm_results[order(df_glmm_results$P_value), ]

    n_sig <- sum(df_glmm_results$Significant)
    cat(sprintf("\n▸ Results: %d / %d significant (FDR < 0.05)\n",
                n_sig, nrow(df_glmm_results)))

    cat(sprintf("\n%-25s %-20s %7s %6s %8s %20s %6s %10s %10s %4s\n",
                "Subtype", "Comparison", "N", "%SnC", "β", "95% CI (β)", "OR",
                "P", "FDR", "Sig"))
    cat(paste(rep("─", 130), collapse = ""), "\n")

    for (i in seq_len(nrow(df_glmm_results))) {
        row <- df_glmm_results[i, ]
        sig_mark <- ifelse(row$Significant, "✓", "")
        ci_str <- sprintf("[%.3f, %.3f]", row$Beta_CI_lower, row$Beta_CI_upper)
        cat(sprintf("%-25s %-20s %7s %5.1f%% %8.4f %20s %6.2f %10.2e %10.2e %4s\n",
                    row$Subtype,
                    row$Comparison,
                    format(row$N_cells, big.mark = ","),
                    row$SnC_rate * 100,
                    row$Beta,
                    ci_str,
                    row$OR,
                    row$P_value,
                    row$P_adj,
                    sig_mark))
    }

    results_file <- file.path(RESULTS_DIR, paste0(DATASET, "_", cell_type_label, "_subtype_glmm_results.csv"))
    write.csv(df_glmm_results, results_file, row.names = FALSE)
    cat(sprintf("\n✓ Saved: %s\n", results_file))

    # ─────────────────────────────────────────────────────────────────────────
    # Interpretation
    # ─────────────────────────────────────────────────────────────────────────

    if (PRIMARY_VAR_TYPE == "continuous") {
        var_label <- "Age"
        unit_label <- "per decade"
    } else {
        var_label <- "Disease"
        unit_label <- paste0("vs ", config$reference_group)
    }

    sig_up   <- df_glmm_results[df_glmm_results$Significant & df_glmm_results$Beta > 0, ]
    sig_down <- df_glmm_results[df_glmm_results$Significant & df_glmm_results$Beta < 0, ]

    if (nrow(sig_up) > 0) {
        cat(sprintf("\n▸ INCREASING senescence (%s):\n", var_label))
        for (i in seq_len(nrow(sig_up))) {
            row <- sig_up[i, ]
            pct <- (row$OR - 1) * 100
            cat(sprintf("   • %s [%s]: OR=%.2f (+%.1f%% %s, FDR=%.2e)\n",
                        row$Subtype, row$Comparison, row$OR, pct, unit_label, row$P_adj))
        }
    }

    if (nrow(sig_down) > 0) {
        cat(sprintf("\n▸ DECREASING senescence (%s):\n", var_label))
        for (i in seq_len(nrow(sig_down))) {
            row <- sig_down[i, ]
            pct <- (1 - row$OR) * 100
            cat(sprintf("   • %s [%s]: OR=%.2f (-%.1f%% %s, FDR=%.2e)\n",
                        row$Subtype, row$Comparison, row$OR, pct, unit_label, row$P_adj))
        }
    }

    if (nrow(sig_up) == 0 & nrow(sig_down) == 0) {
        cat(sprintf("\n▸ No significant associations at FDR < 0.05\n"))
    }

} else {
    cat("\n⚠ No subtypes passed filters\n")
}

cat("\n════════════════════════════════════════════════════════════════════════\n")
cat("✓ STEP 2 COMPLETE — df_glmm_results ready for plotting\n")
cat("════════════════════════════════════════════════════════════════════════\n")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# STEP 3: FOREST PLOT (TABLE STYLE) — SUBTYPE GLMM
# ════════════════════════════════════════════════════════════════════════════════

cat("════════════════════════════════════════════════════════════════════════\n")
cat("STEP 3: FOREST PLOT — SUBTYPE GLMM\n")
cat("════════════════════════════════════════════════════════════════════════\n\n")

if (nrow(df_glmm_results) == 0) {
    cat("No results to plot.\n")
} else {

    df_plot <- df_glmm_results[order(df_glmm_results$OR, decreasing = TRUE), ]
    n_rows <- nrow(df_plot)

    # ─────────────────────────────────────────────────────────────────────────
    # Compact layout
    # ─────────────────────────────────────────────────────────────────────────

    fig_width  <- 7
    fig_height <- max(3, n_rows * 0.5 + 2)

    x_sub  <- 0.01    # subtype label
    x_fl   <- 0.19    # forest left
    x_fr   <- 0.58    # forest right
    x_or   <- 0.72    # OR [CI]
    x_fdr  <- 0.93    # FDR

    # ─────────────────────────────────────────────────────────────────────────
    # Display range — tight around point estimates, clip outlier CIs
    # ─────────────────────────────────────────────────────────────────────────

    betas <- df_plot$Beta
    iqr <- IQR(betas)
    center <- median(betas)
    display_min <- min(betas) - max(iqr * 1.5, 0.15)
    display_max <- max(betas) + max(iqr * 1.5, 0.15)

    beta_to_x <- function(val) {
        v <- pmax(pmin(val, display_max), display_min)
        x_fl + (v - display_min) / (display_max - display_min) * (x_fr - x_fl)
    }

    x_null <- beta_to_x(0)

    # Smart ticks
    cands <- c(0.5, 0.6, 0.7, 0.75, 0.8, 0.85, 0.9, 0.95,
               1.0, 1.05, 1.1, 1.15, 1.2, 1.3, 1.5, 2.0)
    valid <- cands[cands >= exp(display_min) & cands <= exp(display_max)]

    # Enforce minimum spacing
    min_sp <- (x_fr - x_fl) * 0.08
    ticks <- valid[1]
    for (t in valid[-1]) {
        if (abs(beta_to_x(log(t)) - beta_to_x(log(ticks[length(ticks)]))) >= min_sp) {
            ticks <- c(ticks, t)
        }
    }

    # ─────────────────────────────────────────────────────────────────────────
    # Draw
    # ─────────────────────────────────────────────────────────────────────────

    draw_forest <- function() {

        par(mar = c(1.8, 0.3, 1.8, 0.3), family = "sans", xpd = FALSE)
        plot.new()
        plot.window(xlim = c(0, 1), ylim = c(-1, n_rows + 0.8))

        # Title
        text(0.5, n_rows + 0.5,
             paste0(tools::toTitleCase(cell_type_label),
                    " Subtype — Age vs Senescence (", DATASET, ")"),
             cex = 0.8, font = 2, adj = 0.5)

        # Headers
        yh <- n_rows + 0.05
        text(x_sub, yh, "Subtype", cex = 0.65, font = 2, adj = 0)
        text((x_fl + x_fr) / 2, yh, "log-OR", cex = 0.65, font = 2, adj = 0.5)
        text(x_or, yh, "OR [95% CI]", cex = 0.65, font = 2, adj = 0.5)
        text(x_fdr, yh, "FDR", cex = 0.65, font = 2, adj = 0.5)
        segments(0, yh - 0.15, 0.99, yh - 0.15, lwd = 0.6)

        # Null line
        segments(x_null, -0.3, x_null, n_rows - 0.5,
                 lty = 2, col = "#aaaaaa", lwd = 0.5)

        # Rows
        for (i in seq_len(n_rows)) {
            row <- df_plot[i, ]
            y <- n_rows - i

            sig_fdr <- row$Significant
            sig_p   <- row$P_value < 0.05

            # Subtle alternating stripe
            if (i %% 2 == 0) {
                rect(0, y - 0.28, 0.99, y + 0.28,
                     col = "#fafafa", border = NA)
            }

            # Label
            lbl <- as.character(row$Subtype)
            if (sig_fdr) lbl <- paste0("* ", lbl)
            text(x_sub, y, lbl,
                 cex = 0.62, font = ifelse(sig_p, 2, 1), adj = 0)

            # Color
            cc <- state_colors[row$Subtype]
            if (is.na(cc)) cc <- "#808080"

            # CI positions
            ci_l <- row$Beta_CI_lower
            ci_h <- row$Beta_CI_upper
            xl <- beta_to_x(ci_l)
            xh <- beta_to_x(ci_h)
            xb <- beta_to_x(row$Beta)

            clip_l <- ci_l < display_min
            clip_r <- ci_h > display_max

            # CI line
            segments(xl, y, xh, y, col = cc, lwd = 1.8, lend = 1)

            # Left: arrow or cap
            if (clip_l) {
                polygon(x = c(xl, xl + 0.008, xl + 0.008),
                        y = c(y, y + 0.08, y - 0.08),
                        col = cc, border = NA)
            } else {
                segments(xl, y - 0.08, xl, y + 0.08, col = cc, lwd = 0.8)
            }

            # Right: arrow or cap
            if (clip_r) {
                polygon(x = c(xh, xh - 0.008, xh - 0.008),
                        y = c(y, y + 0.08, y - 0.08),
                        col = cc, border = NA)
            } else {
                segments(xh, y - 0.08, xh, y + 0.08, col = cc, lwd = 0.8)
            }

            # Diamond
            dw <- 0.005; dh <- 0.13
            polygon(x = c(xb - dw, xb, xb + dw, xb),
                    y = c(y, y - dh, y, y + dh),
                    col = cc, border = "black", lwd = 0.4)

            # OR [CI] text
            text(x_or, y + 0.09,
                 sprintf("%.2f", row$OR),
                 cex = 0.58, adj = 0.5,
                 font = ifelse(sig_p, 2, 1))
            text(x_or, y - 0.09,
                 sprintf("[%.2f-%.2f]", row$OR_CI_lower, row$OR_CI_upper),
                 cex = 0.50, adj = 0.5, col = "#777777")

            # FDR
            pv <- row$P_adj
            if (pv < 0.001) {
                ptxt <- formatC(pv, format = "e", digits = 1)
            } else if (pv < 0.01) {
                ptxt <- formatC(pv, format = "f", digits = 3)
            } else {
                ptxt <- formatC(pv, format = "f", digits = 2)
            }
            text(x_fdr, y, ptxt,
                 cex = 0.58, adj = 0.5,
                 font = ifelse(sig_fdr, 2, 1),
                 col = ifelse(sig_fdr, "#C44E52", "#333333"))
        }

        # X-axis
        ya <- -0.45
        segments(x_fl, ya, x_fr, ya, lwd = 0.6)

        for (t in ticks) {
            xt <- beta_to_x(log(t))
            segments(xt, ya, xt, ya - 0.06, lwd = 0.5)
            text(xt, ya - 0.15, t, cex = 0.5, adj = 0.5)
        }

        text((x_fl + x_fr) / 2, ya - 0.35,
             "Odds Ratio", cex = 0.55, font = 2, adj = 0.5)

        # Minimal footer
        segments(0, ya - 0.5, 0.99, ya - 0.5, lwd = 0.3, col = "#dddddd")
        text(x_sub, ya - 0.7, "* FDR < 0.05",
             cex = 0.48, adj = 0, col = "#999999")

        if (any(df_plot$Beta_CI_lower < display_min) |
            any(df_plot$Beta_CI_upper > display_max)) {
            text(x_fr, ya - 0.7, "< > CI clipped",
                 cex = 0.48, adj = 1, col = "#999999")
        }
    }

    # Display
    options(repr.plot.width = fig_width, repr.plot.height = fig_height)
    draw_forest()

    # Save
    forest_file <- file.path(FIGURES_DIR,
                             paste0(DATASET, "_", cell_type_label, "_subtype_glmm_forest.svg"))
    svg(forest_file, width = fig_width, height = fig_height)
    draw_forest()
    dev.off()

    cat("Saved:", forest_file, "\n")
}

cat("\n════════════════════════════════════════════════════════════════════════\n")
cat("STEP 3 COMPLETE\n")
cat("════════════════════════════════════════════════════════════════════════\n")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# SAVE ANNOTATED OBJECT
# ════════════════════════════════════════════════════════════════════════════════

cat("\n", paste(rep("=", 80), collapse = ""), "\n")
cat(sprintf("SAVING ANNOTATED %s OBJECT\n", toupper(cell_type_label)))
cat(paste(rep("=", 80), collapse = ""), "\n")

save_file <- file.path(OUTPUT_DIR, paste0(DATASET, "_", cell_type_label, "_annotated.qs"))
qsave(seurat_obj, save_file)

cat("\n✓ Saved:", save_file, "\n")
cat("  Cells:", ncol(seurat_obj), "\n")
cat("  States:", length(unique(seurat_obj@meta.data[[SUBCLUSTER_COL]])), "\n")
cat("  Clusters:", length(unique(seurat_obj$seurat_clusters)), "\n")

# ════════════════════════════════════════════════════════════════════════════════
# MODULE SUMMARY
# ════════════════════════════════════════════════════════════════════════════════

cat("\n", paste(rep("=", 80), collapse = ""), "\n")
cat(sprintf("SUMMARY: %s SUBCLUSTERING & ANNOTATION\n", toupper(cell_type_label)))
cat(paste(rep("=", 80), collapse = ""), "\n")

cat("\n--- Data Summary ---\n")
cat("  Total cells:", ncol(seurat_obj), "\n")
cat("  Donors:", length(unique(seurat_obj@meta.data[[DONOR_COL]])), "\n")
cat("  Study groups:", paste(sg_order, collapse = ", "), "\n")

cat(sprintf("\n--- %s States ---\n", tools::toTitleCase(cell_type_label)))
state_counts <- table(seurat_obj@meta.data[[SUBCLUSTER_COL]])
for (s in names(sort(state_counts, decreasing = TRUE))) {
    cat(sprintf("  %s: %d (%.1f%%)\n", s, state_counts[s], state_counts[s] / sum(state_counts) * 100))
}

cat("\n--- Senescence ---\n")
cat(sprintf("  Label column: %s\n", SENESCENCE_LABEL_COL))
snc_counts <- table(seurat_obj@meta.data[[SENESCENCE_LABEL_COL]])
cat(sprintf("  SnC: %d (%.1f%%)\n", snc_counts["SnC"], snc_counts["SnC"] / sum(snc_counts) * 100))
cat(sprintf("  Non-SnC: %d (%.1f%%)\n", snc_counts["Non-SnC"], snc_counts["Non-SnC"] / sum(snc_counts) * 100))

if ("senescence_label_state" %in% colnames(seurat_obj@meta.data) && 
    SENESCENCE_LABEL_COL == "senescence_label_state") {
    cat("\n  (Original labels for comparison:)\n")
    orig_counts <- table(seurat_obj@meta.data$senescence_label)
    cat(sprintf("  SnC: %d (%.1f%%)\n", orig_counts["SnC"], orig_counts["SnC"] / sum(orig_counts) * 100))
}

cat("\n--- Output Files ---\n")
cat("  Data:\n")
cat("    •", save_file, "\n")

cat("\n  Figures:\n")
fig_files <- list.files(FIGURES_DIR, pattern = paste0("^", DATASET, "_", cell_type_label), full.names = FALSE)
for (f in fig_files) cat("    •", f, "\n")

cat("\n  Results:\n")
res_files <- list.files(RESULTS_DIR, pattern = paste0("^", DATASET, "_", cell_type_label), full.names = FALSE)
for (f in res_files) cat("    •", f, "\n")

cat("\n", paste(rep("=", 80), collapse = ""), "\n")
cat(sprintf("%s COMPLETE\n", toupper(cell_type_label)))
cat(paste(rep("=", 80), collapse = ""), "\n")

---

## Post-Annotation Validation: Aberrant Cluster QC

**Purpose:** Systematic validation of annotated subclusters/states before downstream analysis (DEG, GSEA). Confirms that assigned states are biologically real and not driven by technical artifacts.

**5-Step Validation:**
1. **QC Metrics** — UMI, genes, %MT, %ribo per state vs global. Flags low-quality or doublet-like states.
2. **Canonical Marker Expression** — Are identity markers expressed in all states? Any state lacking lineage markers?
3. **Sample Composition** — Is any state dominated by 1–2 donors? (batch artifact)
4. **Cell Cycle** — Is any state enriched for S/G2M phase? (cycling subpopulation)
5. **Cluster Stability** — Are states stable across clustering resolutions? (clustree)

**Input:** Annotated Seurat object (`seurat_obj`) from save step above.

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# POST-ANNOTATION VALIDATION: SETUP & INSPECT
# ════════════════════════════════════════════════════════════════════════════════

cat("\n", paste(rep("=", 80), collapse = ""), "\n")
cat("POST-ANNOTATION VALIDATION: SETUP\n")
cat(paste(rep("=", 80), collapse = ""), "\n")

suppressPackageStartupMessages({
    library(Seurat)
    library(Matrix)
    library(dplyr)
    library(tidyr)
    library(ggplot2)
    library(qs)
    library(patchwork)
    library(RColorBrewer)
    library(scales)
})

cat("\n✓ Libraries loaded\n")

# ════════════════════════════════════════════════════════════════════════════════
# DATASET SELECTION
# ════════════════════════════════════════════════════════════════════════════════

DATASET <- "psychad_aging"

# ════════════════════════════════════════════════════════════════════════════════
# DATASET-SPECIFIC CONFIGURATION
# ════════════════════════════════════════════════════════════════════════════════

DATASET_CONFIG <- list(
    'psychad_aging' = list(
        cell_type_col = 'subclass',
        donor_col = 'Sample',
        study_group_col = 'Study_Group',
        sex_col = 'Sex',
        cohort_col = 'Cohort',
        study_type = 'aging',
        primary_var = 'Age',
        primary_var_type = 'continuous',
        covariates = c('Sex', 'Cohort'),
        group_order = c('Age_20_29', 'Age_30_39', 'Age_40_49', 'Age_50_59',
                        'Age_60_69', 'Age_70_79', 'Age_80_100')
    ),
    'psychad_ad' = list(
        cell_type_col = 'subclass',
        donor_col = 'Sample',
        study_group_col = 'Study_Group',
        sex_col = 'Sex',
        cohort_col = 'Cohort',
        study_type = 'disease',
        primary_var = 'Study_Group',
        primary_var_type = 'categorical',
        reference_group = 'Control',
        covariates = c('Age', 'Sex', 'Cohort'),
        group_order = c('Control', 'MCI', 'AD')
    ),
    'psychencode' = list(
        cell_type_col = 'cell_type',
        donor_col = 'sample_id',
        study_group_col = 'Study_Group',
        sex_col = 'Biological_Sex',
        cohort_col = 'Batch',
        study_type = 'aging',
        primary_var = 'Age_death',
        primary_var_type = 'continuous',
        covariates = c('Sex', 'Batch'),
        group_order = c('Age_20_29', 'Age_30_39', 'Age_40_49', 'Age_50_59',
                        'Age_60_69', 'Age_70_79', 'Age_80_100')
    ),
    'mathys' = list(
        cell_type_col = 'broad.cell.type',
        donor_col = 'Subject',
        study_group_col = 'Study_Group',
        sex_col = 'sex',
        cohort_col = 'batch',
        study_type = 'disease',
        primary_var = 'Study_Group',
        primary_var_type = 'categorical',
        reference_group = 'Control',
        covariates = c('Age', 'sex', 'batch'),
        group_order = c('Control', 'AD')
    )
)

config <- DATASET_CONFIG[[DATASET]]

# ─────────────────────────────────────────────────────────────────────────────
# COLUMN NAMES
# ─────────────────────────────────────────────────────────────────────────────

CELL_TYPE_COL     <- config$cell_type_col
DONOR_COL         <- config$donor_col
STUDY_GROUP_COL   <- config$study_group_col
SEX_COL           <- config$sex_col
COHORT_COL        <- config$cohort_col
SENESCENCE_LABEL_COL <- "senescence_label"

# ─────────────────────────────────────────────────────────────────────────────
# STUDY DESIGN PARAMETERS
# ─────────────────────────────────────────────────────────────────────────────

STUDY_TYPE       <- config$study_type
PRIMARY_VAR      <- config$primary_var
PRIMARY_VAR_TYPE <- config$primary_var_type
COVARIATES       <- config$covariates
GROUP_ORDER      <- config$group_order

# ─────────────────────────────────────────────────────────────────────────────
# CELL TYPE-SPECIFIC CONFIG
# ─────────────────────────────────────────────────────────────────────────────

cell_type_label    <- "astrocyte"          # astrocyte, microglia, opc, ol
SUBCLUSTER_COL     <- "astrocyte_state"    # microglia_state, opc_state, ol_state
CLUSTERING_RESOLUTION <- 0.6

# ════════════════════════════════════════════════════════════════════════════════
# COLOR PALETTES
# ════════════════════════════════════════════════════════════════════════════════

STUDY_GROUP_COLORS <- c(
    'Age_20_29' = '#2E86AB', 'Age_30_39' = '#4A90E2', 'Age_40_49' = '#50C878',
    'Age_50_59' = '#FFB347', 'Age_60_69' = '#FF8C00', 'Age_70_79' = '#E24A4A',
    'Age_80_100' = '#8B0000',
    'Control' = '#4E79A7', 'MCI' = '#F28E2B', 'AD' = '#E15759', 'NCI' = '#4E79A7'
)

SENESCENCE_COLORS <- c('Non-SnC' = '#D3D3D3', 'SnC' = '#C44E52')
SEX_COLORS <- c('Male' = '#4878CF', 'Female' = '#E97B8A')

BASE_STATE_PALETTE <- c("#4E79A7", "#59A14F", "#E15759", "#F28E2B", "#EDC948",
                        "#76B7B2", "#FFBE7D", "#BAB0AC", "#B07AA1", "#FF9DA7",
                        "#A0CBE8", "#D37295", "#9C755F", "#8B0000")

get_state_colors <- function(states) {
    setNames(BASE_STATE_PALETTE[1:length(states)], states)
}

# ════════════════════════════════════════════════════════════════════════════════
# PATHS
# ════════════════════════════════════════════════════════════════════════════════

BASE_DIR    <- "/fs/scratch/PAS2598/senescence_analysis"
DATA_DIR    <- file.path(BASE_DIR, "data", paste0("04_", cell_type_label), DATASET)
OUTPUT_DIR  <- DATA_DIR
FIGURES_DIR <- file.path(BASE_DIR, "figures", paste0("04_", cell_type_label), DATASET)
RESULTS_DIR <- file.path(BASE_DIR, "results", paste0("04_", cell_type_label), DATASET)

# Validation subdirectories
VAL_FIGURES_DIR <- file.path(FIGURES_DIR, "validation")
VAL_RESULTS_DIR <- file.path(RESULTS_DIR, "validation")

dir.create(VAL_FIGURES_DIR, recursive = TRUE, showWarnings = FALSE)
dir.create(VAL_RESULTS_DIR, recursive = TRUE, showWarnings = FALSE)

FILE_PREFIX <- paste0(DATASET, "_", cell_type_label, "_validation")

# ════════════════════════════════════════════════════════════════════════════════
# LOAD ANNOTATED OBJECT
# ════════════════════════════════════════════════════════════════════════════════

INPUT_FILE <- file.path(DATA_DIR, paste0(DATASET, "_", cell_type_label, "_annotated.qs"))

cat("\n▸ Loading:", INPUT_FILE, "\n")

if (!file.exists(INPUT_FILE)) stop(paste("File not found:", INPUT_FILE))

seurat_obj <- qread(INPUT_FILE)
cat("  ✓ Loaded:", ncol(seurat_obj), "cells,", nrow(seurat_obj), "genes\n")

# ════════════════════════════════════════════════════════════════════════════════
# INSPECT OBJECT
# ════════════════════════════════════════════════════════════════════════════════

cat("\n▸ Object summary:\n")
cat("  Cells:", ncol(seurat_obj), "\n")
cat("  Genes:", nrow(seurat_obj), "\n")
cat("  Dataset:", DATASET, "\n")
cat("  Cell type:", toupper(cell_type_label), "\n")

# States
cat("\n▸ Annotated states (", SUBCLUSTER_COL, "):\n", sep = "")
states <- sort(unique(seurat_obj@meta.data[[SUBCLUSTER_COL]]))
n_states <- length(states)
for (s in states) {
    n <- sum(seurat_obj@meta.data[[SUBCLUSTER_COL]] == s)
    cat(sprintf("    %s: %d (%.1f%%)\n", s, n, n / ncol(seurat_obj) * 100))
}

# Clusters
cat("\n▸ Seurat clusters:\n")
n_clusters <- length(unique(seurat_obj$seurat_clusters))
cat("  N clusters:", n_clusters, "\n")

# State ↔ Cluster mapping
cat("\n▸ State ↔ Cluster mapping:\n")
mapping <- table(seurat_obj@meta.data[[SUBCLUSTER_COL]], seurat_obj$seurat_clusters)
for (s in rownames(mapping)) {
    counts <- mapping[s, mapping[s, ] > 0]
    detail <- paste(sprintf("%s(n=%d)", names(counts), counts), collapse = ", ")
    cat(sprintf("    %s → %s\n", s, detail))
}

# ════════════════════════════════════════════════════════════════════════════════
# DETECT QC COLUMNS
# ════════════════════════════════════════════════════════════════════════════════

cat("\n▸ Detecting QC columns...\n")
md_cols <- colnames(seurat_obj@meta.data)

MT_COL <- NA
for (mc in c("percent.mt", "percent_mt", "pct_counts_mt", "mitoRatio")) {
    if (mc %in% md_cols) { MT_COL <- mc; break }
}
RIBO_COL <- NA
for (rc in c("percent.ribo", "percent_ribo", "pct_counts_ribo")) {
    if (rc %in% md_cols) { RIBO_COL <- rc; break }
}

QC_METRICS <- c("nCount_RNA", "nFeature_RNA")
QC_LABELS <- c("UMI Counts", "Gene Counts")
if (!is.na(MT_COL)) { QC_METRICS <- c(QC_METRICS, MT_COL); QC_LABELS <- c(QC_LABELS, "% MT") }
if (!is.na(RIBO_COL)) { QC_METRICS <- c(QC_METRICS, RIBO_COL); QC_LABELS <- c(QC_LABELS, "% Ribo") }
names(QC_LABELS) <- QC_METRICS

cat("  MT:", ifelse(is.na(MT_COL), "NONE", MT_COL), "\n")
cat("  Ribo:", ifelse(is.na(RIBO_COL), "NONE", RIBO_COL), "\n")
cat("  Active metrics:", paste(QC_METRICS, collapse = ", "), "\n")

# Donors
n_donors <- length(unique(seurat_obj@meta.data[[DONOR_COL]]))
cat("\n▸ Donors:", n_donors, "\n")

# Reductions
UMAP_RED <- ifelse("umap" %in% Reductions(seurat_obj), "umap",
            ifelse("tsne" %in% Reductions(seurat_obj), "tsne", Reductions(seurat_obj)[1]))
cat("▸ Reductions:", paste(Reductions(seurat_obj), collapse = ", "), "\n")
cat("  Plot reduction:", UMAP_RED, "\n")

# Global medians
GLOBAL_MEDIANS <- sapply(QC_METRICS, function(m) median(seurat_obj@meta.data[[m]], na.rm = TRUE))

cat("\n▸ Global QC medians:\n")
for (i in seq_along(QC_METRICS)) {
    cat(sprintf("    %s: %.1f\n", QC_LABELS[i], GLOBAL_MEDIANS[i]))
}

# Derived orders
is_aging <- STUDY_TYPE == "aging"
sg_detected <- unique(seurat_obj@meta.data[[STUDY_GROUP_COL]])
sg_order <- names(STUDY_GROUP_COLORS)[names(STUDY_GROUP_COLORS) %in% sg_detected]

state_order <- seurat_obj@meta.data %>%
    count(.data[[SUBCLUSTER_COL]]) %>%
    arrange(desc(n)) %>%
    pull(.data[[SUBCLUSTER_COL]])

state_colors <- get_state_colors(state_order)

cat("\n▸ Study groups:", paste(sg_order, collapse = ", "), "\n")
cat("▸ States (by freq):", paste(state_order, collapse = ", "), "\n")
cat("▸ State colors:", paste(sprintf("%s=%s", names(state_colors), state_colors), collapse = ", "), "\n")

cat("\n", paste(rep("=", 80), collapse = ""), "\n")
cat("✓ Setup complete — ready for 5-step validation\n")
cat(paste(rep("=", 80), collapse = ""), "\n")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# STEP 1: QC METRICS PER STATE & CLUSTER
# ════════════════════════════════════════════════════════════════════════════════

cat("\n", paste(rep("=", 80), collapse = ""), "\n")
cat("STEP 1: QC METRICS\n")
cat(paste(rep("=", 80), collapse = ""), "\n")

# ─────────────────────────────────────────────────────────────────────────────────
# 1A: QC summary table — per cluster
# ─────────────────────────────────────────────────────────────────────────────────

cat("\n▸ Per-cluster QC summary:\n")

qc_by_cluster <- do.call(rbind, lapply(sort(unique(seurat_obj$seurat_clusters)), function(cl) {
    cells <- WhichCells(seurat_obj, idents = cl)
    md <- seurat_obj@meta.data[cells, , drop = FALSE]
    
    row <- data.frame(
        Cluster = cl,
        N = length(cells),
        Pct = round(length(cells) / ncol(seurat_obj) * 100, 1),
        State = names(which.max(table(md[[SUBCLUSTER_COL]]))),
        median_UMI = median(md$nCount_RNA),
        median_genes = median(md$nFeature_RNA),
        stringsAsFactors = FALSE
    )
    
    if (!is.na(MT_COL)) row$median_mt <- round(median(md[[MT_COL]], na.rm = TRUE), 3)
    if (!is.na(RIBO_COL)) row$median_ribo <- round(median(md[[RIBO_COL]], na.rm = TRUE), 3)
    
    row
}))

print(qc_by_cluster, row.names = FALSE)

# Save
write.csv(qc_by_cluster, file.path(VAL_RESULTS_DIR, paste0(FILE_PREFIX, "_qc_by_cluster.csv")),
          row.names = FALSE)
cat("\n  ✓ Saved: ", FILE_PREFIX, "_qc_by_cluster.csv\n", sep = "")

# ─────────────────────────────────────────────────────────────────────────────────
# 1B: QC summary table — per state
# ─────────────────────────────────────────────────────────────────────────────────

cat("\n▸ Per-state QC summary:\n")

qc_by_state <- do.call(rbind, lapply(states, function(s) {
    cells <- colnames(seurat_obj)[seurat_obj@meta.data[[SUBCLUSTER_COL]] == s]
    md <- seurat_obj@meta.data[cells, , drop = FALSE]
    
    row <- data.frame(
        State = s,
        N = length(cells),
        Pct = round(length(cells) / ncol(seurat_obj) * 100, 1),
        median_UMI = median(md$nCount_RNA),
        median_genes = median(md$nFeature_RNA),
        stringsAsFactors = FALSE
    )
    
    if (!is.na(MT_COL)) row$median_mt <- round(median(md[[MT_COL]], na.rm = TRUE), 3)
    if (!is.na(RIBO_COL)) row$median_ribo <- round(median(md[[RIBO_COL]], na.rm = TRUE), 3)
    
    row
}))

print(qc_by_state, row.names = FALSE)

write.csv(qc_by_state, file.path(VAL_RESULTS_DIR, paste0(FILE_PREFIX, "_qc_by_state.csv")),
          row.names = FALSE)
cat("\n  ✓ Saved: ", FILE_PREFIX, "_qc_by_state.csv\n", sep = "")

# ─────────────────────────────────────────────────────────────────────────────────
# 1C: Flag outlier clusters
# ─────────────────────────────────────────────────────────────────────────────────

cat("\n▸ Flagging outlier clusters:\n")

flags_list <- list()

for (i in 1:nrow(qc_by_cluster)) {
    cl <- qc_by_cluster$Cluster[i]
    flags <- c()
    
    # Low UMI
    if (qc_by_cluster$median_UMI[i] < GLOBAL_MEDIANS["nCount_RNA"] * 0.5)
        flags <- c(flags, "LOW UMI")
    
    # Low genes
    if (qc_by_cluster$median_genes[i] < GLOBAL_MEDIANS["nFeature_RNA"] * 0.5)
        flags <- c(flags, "LOW GENES")
    
    # High MT
    if (!is.na(MT_COL) && "median_mt" %in% colnames(qc_by_cluster)) {
        if (qc_by_cluster$median_mt[i] > GLOBAL_MEDIANS[MT_COL] * 2)
            flags <- c(flags, "HIGH MT")
    }
    
    # Possible doublet (high UMI + high genes)
    if (qc_by_cluster$median_UMI[i] > GLOBAL_MEDIANS["nCount_RNA"] * 2 &&
        qc_by_cluster$median_genes[i] > GLOBAL_MEDIANS["nFeature_RNA"] * 1.5)
        flags <- c(flags, "POSSIBLE DOUBLET")
    
    # Very small cluster
    if (qc_by_cluster$N[i] < ncol(seurat_obj) * 0.005)
        flags <- c(flags, sprintf("TINY (%.1f%%)", qc_by_cluster$Pct[i]))
    
    if (length(flags) > 0) {
        flags_list[[cl]] <- flags
        cat(sprintf("  ⚠ Cluster %s (%s): %s\n", cl, qc_by_cluster$State[i],
                    paste(flags, collapse = ", ")))
    }
}

if (length(flags_list) == 0) cat("  ✓ No clusters flagged\n")

# ─────────────────────────────────────────────────────────────────────────────────
# 1D: Violin plots — QC per cluster
# ─────────────────────────────────────────────────────────────────────────────────

cat("\n▸ Generating QC violin plots...\n")

options(repr.plot.width = 14, repr.plot.height = 3 * length(QC_METRICS))

plots_qc_cluster <- lapply(seq_along(QC_METRICS), function(i) {
    m <- QC_METRICS[i]
    VlnPlot(seurat_obj, features = m, group.by = "seurat_clusters", pt.size = 0) +
        geom_hline(yintercept = GLOBAL_MEDIANS[m], linetype = "dashed",
                   color = "red", linewidth = 0.5) +
        theme(legend.position = "none", axis.title.x = element_blank(),
              plot.title = element_text(size = 10, face = "bold")) +
        ggtitle(paste0(QC_LABELS[i], " (by cluster) — red = global median"))
})

p_qc_cluster <- wrap_plots(plots_qc_cluster, ncol = 1)
print(p_qc_cluster)

ggsave(file.path(VAL_FIGURES_DIR, paste0(FILE_PREFIX, "_qc_violin_cluster.pdf")),
       p_qc_cluster, width = 14, height = 3 * length(QC_METRICS))
ggsave(file.path(VAL_FIGURES_DIR, paste0(FILE_PREFIX, "_qc_violin_cluster.svg")),
       p_qc_cluster, width = 14, height = 3 * length(QC_METRICS))
cat("  ✓ Saved violin plots (by cluster)\n")

# ─────────────────────────────────────────────────────────────────────────────────
# 1E: Violin plots — QC per state
# ─────────────────────────────────────────────────────────────────────────────────

plots_qc_state <- lapply(seq_along(QC_METRICS), function(i) {
    m <- QC_METRICS[i]
    VlnPlot(seurat_obj, features = m, group.by = SUBCLUSTER_COL, pt.size = 0) +
        geom_hline(yintercept = GLOBAL_MEDIANS[m], linetype = "dashed",
                   color = "red", linewidth = 0.5) +
        theme(legend.position = "none", axis.title.x = element_blank(),
              plot.title = element_text(size = 10, face = "bold")) +
        ggtitle(paste0(QC_LABELS[i], " (by state) — red = global median"))
})

p_qc_state <- wrap_plots(plots_qc_state, ncol = 1)
print(p_qc_state)

ggsave(file.path(VAL_FIGURES_DIR, paste0(FILE_PREFIX, "_qc_violin_state.pdf")),
       p_qc_state, width = 10, height = 3 * length(QC_METRICS))
ggsave(file.path(VAL_FIGURES_DIR, paste0(FILE_PREFIX, "_qc_violin_state.svg")),
       p_qc_state, width = 10, height = 3 * length(QC_METRICS))
cat("  ✓ Saved violin plots (by state)\n")

# ─────────────────────────────────────────────────────────────────────────────────
# 1F: UMAP colored by QC metrics
# ─────────────────────────────────────────────────────────────────────────────────

cat("\n▸ Generating QC feature UMAPs...\n")

options(repr.plot.width = 12, repr.plot.height = 3.5)

p_qc_umap <- FeaturePlot(seurat_obj, features = QC_METRICS, reduction = UMAP_RED,
                          ncol = length(QC_METRICS), pt.size = 0.1, order = TRUE) &
    scale_color_viridis_c() &
    theme(plot.title = element_text(size = 9, face = "bold"),
          axis.text = element_blank(), axis.ticks = element_blank())

print(p_qc_umap)

ggsave(file.path(VAL_FIGURES_DIR, paste0(FILE_PREFIX, "_qc_umap.pdf")),
       p_qc_umap, width = 3.5 * length(QC_METRICS), height = 3.5)
ggsave(file.path(VAL_FIGURES_DIR, paste0(FILE_PREFIX, "_qc_umap.svg")),
       p_qc_umap, width = 3.5 * length(QC_METRICS), height = 3.5)
cat("  ✓ Saved QC UMAPs\n")

# ─────────────────────────────────────────────────────────────────────────────────
# Step 1 Summary
# ─────────────────────────────────────────────────────────────────────────────────

cat("\n", paste(rep("-", 80), collapse = ""), "\n")
cat("STEP 1 SUMMARY\n")
cat(paste(rep("-", 80), collapse = ""), "\n")
cat("  Clusters:", n_clusters, "\n")
cat("  States:", n_states, "\n")
cat("  Flagged clusters:", length(flags_list), "\n")
if (length(flags_list) > 0) {
    for (cl in names(flags_list)) {
        cat(sprintf("    Cluster %s: %s\n", cl, paste(flags_list[[cl]], collapse = ", ")))
    }
}
cat("\n  Files saved:\n")
cat("    •", paste0(FILE_PREFIX, "_qc_by_cluster.csv"), "\n")
cat("    •", paste0(FILE_PREFIX, "_qc_by_state.csv"), "\n")
cat("    •", paste0(FILE_PREFIX, "_qc_violin_cluster.{pdf,svg}"), "\n")
cat("    •", paste0(FILE_PREFIX, "_qc_violin_state.{pdf,svg}"), "\n")
cat("    •", paste0(FILE_PREFIX, "_qc_umap.{pdf,svg}"), "\n")
cat(paste(rep("=", 80), collapse = ""), "\n")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# STEP 2: CANONICAL MARKER EXPRESSION
# ════════════════════════════════════════════════════════════════════════════════

cat("\n", paste(rep("=", 80), collapse = ""), "\n")
cat("STEP 2: CANONICAL MARKER EXPRESSION\n")
cat(paste(rep("=", 80), collapse = ""), "\n")

library(dplyr)

# ─────────────────────────────────────────────────────────────────────────────────
# Define markers: positive (identity) and negative (contamination)
# ─────────────────────────────────────────────────────────────────────────────────

POSITIVE_MARKERS <- list(
    "astrocyte" = list(
        'Identity'      = c('GFAP', 'AQP4', 'SLC1A2', 'GLUL', 'ALDH1L1'),
        'Homeostatic'   = c('GLUL', 'SLC1A2', 'GRM3', 'GJA1', 'KCNJ10'),
        'Intermediate'  = c('ETNPPL', 'GLUD1', 'NTRK2', 'GABRB1'),
        'Metabolic'     = c('ENO1', 'LDHA', 'MT1X', 'HSP90AA1'),
        'Trophic'       = c('FGF1', 'FGF2', 'CHI3L1', 'SOCS3'),
        'Reactive'      = c('GFAP', 'C3', 'CD44', 'COL21A1', 'SPARC')
    ),
    "microglia" = list(
        'Identity'              = c('CSF1R', 'P2RY12', 'CX3CR1', 'TMEM119', 'AIF1'),
        'Homeostatic'           = c('TMEM119', 'P2RY12', 'CX3CR1'),
        'Activated'             = c('CD68', 'APOE', 'TREM2'),
        'IFN-I'                 = c('IFITM3', 'IFIT3', 'ISG15'),
        'IFN-II'                = c('GBP2', 'STAT1'),
        'IFN-III'               = c('IFI44L', 'IRF7'),
        'MCHII'                 = c('HLA-DRA', 'HLA-DRB1', 'CD74'),
        'Stress'                = c('HSP90AA1', 'HSPA1A', 'FOS'),
        'Neuronal_Surveillance' = c('MT-ND3', 'MT-CYB', 'MT-CO2'),
        'Lipid_Processing'      = c('APOC1', 'LPL', 'ABCA1'),
        'Cycling'               = c('MKI67', 'TOP2A', 'CDK1'),
        'Mic-reduced'           = c('LINGO1', 'GRID2', 'ELAVL2')
    ),
    "opc" = list(
        'Identity'    = c('PDGFRA', 'CSPG4', 'OLIG1', 'OLIG2', 'SOX10'),
        'Homeostatic' = c('PDGFRA', 'CSPG4', 'PTPRZ1', 'BCAN', 'PCDH15'),
        'COP'         = c('GPR17', 'BCAS1', 'NEU4', 'TNS3', 'FYN'),
        'Reactive'    = c('HLA-A', 'HLA-B', 'HLA-C', 'B2M', 'CD74', 'SERPINA3')
    ),
    "ol" = list(
        'Identity'    = c('MBP', 'MOG', 'PLP1', 'MAG', 'MOBP'),
        'hOligo1'     = c('RASGRF1', 'RASGRF2', 'KLK6'),
        'hOligo2'     = c('PLXDC2', 'PALM2', 'OPALIN', 'QDPR'),
        'DA1_Immune'  = c('SERPINA3', 'C4B', 'B2M', 'HLA-A', 'HLA-B', 'HLA-C', 'CD74'),
        'IFN'         = c('IFIT1', 'IFIT3', 'OAS1', 'OAS2')
    )
)

NEGATIVE_MARKERS_ALL <- list(
    'Neuron'          = c('RBFOX3', 'SYT1', 'SNAP25'),
    'Endothelial'     = c('CLDN5', 'FLT1', 'PECAM1'),
    'Astrocyte'       = c('GFAP', 'AQP4', 'SLC1A2'),
    'Microglia'       = c('CSF1R', 'P2RY12', 'CX3CR1'),
    'OPC'             = c('PDGFRA', 'CSPG4', 'GPR17'),
    'Oligodendrocyte' = c('MBP', 'MOG', 'PLP1')
)

# ── Remove self from negative markers ──
SELF_LABEL_MAP <- c(
    "astrocyte" = "Astrocyte",
    "microglia" = "Microglia",
    "opc"       = "OPC",
    "ol"        = "Oligodendrocyte"
)
self_label <- SELF_LABEL_MAP[[cell_type_label]]
neg_markers_list <- NEGATIVE_MARKERS_ALL[names(NEGATIVE_MARKERS_ALL) != self_label]

# Get positive markers for this cell type
pos_markers_list <- POSITIVE_MARKERS[[cell_type_label]]

# Filter to genes present in object
pos_markers_list <- lapply(pos_markers_list, function(genes) {
    genes[genes %in% rownames(seurat_obj)]
})
pos_markers_list <- pos_markers_list[sapply(pos_markers_list, length) > 0]

neg_markers_list <- lapply(neg_markers_list, function(genes) {
    genes[genes %in% rownames(seurat_obj)]
})
neg_markers_list <- neg_markers_list[sapply(neg_markers_list, length) > 0]

# Remove any genes that overlap between positive and negative
pos_genes_all <- unlist(pos_markers_list)
overlap_genes <- c()
neg_markers_list <- lapply(neg_markers_list, function(genes) {
    ov <- genes[genes %in% pos_genes_all]
    if (length(ov) > 0) overlap_genes <<- c(overlap_genes, ov)
    genes[!genes %in% pos_genes_all]
})
neg_markers_list <- neg_markers_list[sapply(neg_markers_list, length) > 0]

if (length(overlap_genes) > 0) {
    cat("\n  Note: Removed", length(unique(overlap_genes)), "overlapping gene(s) from negatives:",
        paste(unique(overlap_genes), collapse = ", "), "\n")
}

cat("\n▸ Positive markers:\n")
for (cat_name in names(pos_markers_list)) {
    cat(sprintf("  %s: %s\n", cat_name, paste(pos_markers_list[[cat_name]], collapse = ", ")))
}

cat("\n▸ Negative markers:\n")
for (cat_name in names(neg_markers_list)) {
    cat(sprintf("  %s: %s\n", cat_name, paste(neg_markers_list[[cat_name]], collapse = ", ")))
}

# ─────────────────────────────────────────────────────────────────────────────────
# 2A: AddModuleScore — identity + per-lineage contamination
# ─────────────────────────────────────────────────────────────────────────────────

cat("\n▸ Computing module scores (AddModuleScore)...\n")

# Identity: all positive markers as one module
all_pos_genes <- unique(unlist(pos_markers_list))
seurat_obj <- AddModuleScore(seurat_obj,
                              features = list(all_pos_genes),
                              name = "identity_score",
                              ctrl = 50, seed = 42)

cat("  ✓ Identity score: ", length(all_pos_genes), " genes\n", sep = "")

# Per-lineage contamination scores
contam_score_cols <- c()
for (lineage in names(neg_markers_list)) {
    genes <- neg_markers_list[[lineage]]
    col_name <- paste0("contam_", gsub(" ", "_", lineage))
    
    seurat_obj <- AddModuleScore(seurat_obj,
                                  features = list(genes),
                                  name = col_name,
                                  ctrl = 50, seed = 42)
    actual_col <- paste0(col_name, "1")
    contam_score_cols <- c(contam_score_cols, setNames(actual_col, lineage))
    cat("  ✓ ", lineage, " contamination: ", length(genes), " genes -> ", actual_col, "\n", sep = "")
}

# ─────────────────────────────────────────────────────────────────────────────────
# 2B: Compute dataset-wide baselines per lineage
# ─────────────────────────────────────────────────────────────────────────────────

cat("\n▸ Computing dataset-wide baselines per lineage...\n")

lineage_baselines <- data.frame(
    Lineage = names(contam_score_cols),
    Median = NA_real_,
    MAD = NA_real_,
    Threshold = NA_real_,
    stringsAsFactors = FALSE
)

for (i in seq_along(contam_score_cols)) {
    vals <- seurat_obj@meta.data[[contam_score_cols[i]]]
    med <- median(vals, na.rm = TRUE)
    mad_val <- mad(vals, na.rm = TRUE)
    lineage_baselines$Median[i] <- round(med, 4)
    lineage_baselines$MAD[i] <- round(mad_val, 4)
    lineage_baselines$Threshold[i] <- round(med + 3 * mad_val, 4)
}

cat("  Dataset-wide contamination baselines (outlier = median + 3×MAD):\n")
print(lineage_baselines, row.names = FALSE)

# ─────────────────────────────────────────────────────────────────────────────────
# 2C: Aggregate scores per cluster and per state
# ─────────────────────────────────────────────────────────────────────────────────

cat("\n▸ Aggregating scores per cluster...\n")

Idents(seurat_obj) <- "seurat_clusters"
cluster_order <- as.character(sort(as.numeric(levels(Idents(seurat_obj)))))

score_by_cluster <- do.call(rbind, lapply(cluster_order, function(cl) {
    cells <- which(as.character(seurat_obj$seurat_clusters) == cl)
    
    identity_med <- median(seurat_obj$identity_score1[cells])
    
    contam_meds <- sapply(contam_score_cols, function(col) {
        median(seurat_obj@meta.data[[col]][cells])
    })
    
    max_contam_lineage <- names(which.max(contam_meds))
    max_contam_val <- max(contam_meds)
    mean_contam <- mean(contam_meds)
    
    row <- data.frame(
        Cluster = cl,
        State = qc_by_cluster$State[qc_by_cluster$Cluster == cl],
        N_cells = length(cells),
        Identity = round(identity_med, 4),
        stringsAsFactors = FALSE
    )
    
    for (j in seq_along(contam_meds)) {
        row[[names(contam_score_cols)[j]]] <- round(contam_meds[j], 4)
    }
    
    row$Max_contam_lineage <- max_contam_lineage
    row$Max_contam_score <- round(max_contam_val, 4)
    row$Mean_contam <- round(mean_contam, 4)
    row
}))

cat("\n  Module scores per cluster:\n")
print_cols <- c("Cluster", "State", "N_cells", "Identity",
                names(neg_markers_list), "Max_contam_lineage", "Max_contam_score")
print_cols <- print_cols[print_cols %in% colnames(score_by_cluster)]
print(score_by_cluster[, print_cols], row.names = FALSE)

# Per-state
cat("\n▸ Aggregating scores per state...\n")

score_by_state <- do.call(rbind, lapply(state_order, function(st) {
    cells <- which(seurat_obj@meta.data[[SUBCLUSTER_COL]] == st)
    
    identity_med <- median(seurat_obj$identity_score1[cells])
    
    contam_meds <- sapply(contam_score_cols, function(col) {
        median(seurat_obj@meta.data[[col]][cells])
    })
    
    row <- data.frame(
        State = st,
        N_cells = length(cells),
        Identity = round(identity_med, 4),
        stringsAsFactors = FALSE
    )
    
    for (j in seq_along(contam_meds)) {
        row[[names(contam_score_cols)[j]]] <- round(contam_meds[j], 4)
    }
    
    row$Max_contam_lineage <- names(which.max(contam_meds))
    row$Max_contam_score <- round(max(contam_meds), 4)
    row
}))

cat("\n  Module scores per state:\n")
print_cols_state <- c("State", "N_cells", "Identity",
                       names(neg_markers_list), "Max_contam_lineage", "Max_contam_score")
print_cols_state <- print_cols_state[print_cols_state %in% colnames(score_by_state)]
print(score_by_state[, print_cols_state], row.names = FALSE)

# ─────────────────────────────────────────────────────────────────────────────────
# 2D: Flag clusters — absolute + outlier thresholds
# ─────────────────────────────────────────────────────────────────────────────────

cat("\n▸ Flagging clusters:\n")
cat("  Criteria:\n")
cat("    WEAK IDENTITY:  median identity score < 0 (below background)\n")
cat("    CONTAMINATION:  lineage score > dataset baseline + 3×MAD AND > identity × 0.5\n")
cat("    DOUBLET-LIKE:   lineage score > dataset baseline + 3×MAD AND > identity\n")

flagged_id <- FALSE
flagged_contam <- FALSE
any_flagged <- FALSE

for (i in 1:nrow(score_by_cluster)) {
    flags <- c()
    cl <- score_by_cluster$Cluster[i]
    id_score <- score_by_cluster$Identity[i]
    
    # Weak identity: below background
    if (id_score < 0) {
        flags <- c(flags, sprintf("WEAK IDENTITY (%.4f < 0)", id_score))
        flagged_id <- TRUE
    }
    
    # Per-lineage contamination: must exceed BOTH baseline threshold AND relative threshold
    for (lineage in names(neg_markers_list)) {
        if (!lineage %in% colnames(score_by_cluster)) next
        contam_val <- score_by_cluster[[lineage]][i]
        
        baseline_row <- which(lineage_baselines$Lineage == lineage)
        baseline_thresh <- lineage_baselines$Threshold[baseline_row]
        
        # Only flag if above dataset-wide outlier threshold
        if (contam_val > baseline_thresh) {
            if (id_score > 0 && contam_val > id_score) {
                flags <- c(flags, sprintf("DOUBLET-LIKE %s (%.3f > identity %.3f, baseline %.3f)",
                                           lineage, contam_val, id_score, baseline_thresh))
                flagged_contam <- TRUE
            } else if (id_score > 0 && contam_val > id_score * 0.5) {
                flags <- c(flags, sprintf("HIGH %s (%.3f > 50%% identity, baseline %.3f)",
                                           lineage, contam_val, baseline_thresh))
                flagged_contam <- TRUE
            }
        }
    }
    
    if (length(flags) > 0) {
        cat(sprintf("    ⚠ Cluster %s (%s): %s\n",
                    cl, score_by_cluster$State[i], paste(flags, collapse = "; ")))
        any_flagged <- TRUE
    }
}

if (!any_flagged) cat("    ✓ All clusters pass identity and contamination checks\n")

# Build flag_df for downstream compatibility (Summary cell)
flag_df <- data.frame(
    Cluster = score_by_cluster$Cluster,
    State = score_by_cluster$State,
    mean_identity = score_by_cluster$Identity,
    mean_contam = score_by_cluster$Mean_contam,
    max_neg_gene = score_by_cluster$Max_contam_lineage,
    max_neg_val = score_by_cluster$Max_contam_score,
    stringsAsFactors = FALSE
)

write.csv(score_by_cluster, file.path(VAL_RESULTS_DIR, paste0(FILE_PREFIX, "_identity_scores.csv")),
          row.names = FALSE)
write.csv(score_by_state, file.path(VAL_RESULTS_DIR, paste0(FILE_PREFIX, "_identity_scores_state.csv")),
          row.names = FALSE)

# ─────────────────────────────────────────────────────────────────────────────────
# 2E: Compute raw expression for DotPlots
# ─────────────────────────────────────────────────────────────────────────────────

cat("\n▸ Computing expression for DotPlots...\n")

expr_data <- GetAssayData(seurat_obj, layer = "data")

all_marker_cats <- c(pos_markers_list, neg_markers_list)
gene_order <- unique(unlist(all_marker_cats))

# By cluster
dot_list <- list()
idx <- 1

for (cluster in cluster_order) {
    cells <- WhichCells(seurat_obj, idents = cluster)
    if (length(cells) == 0) next
    
    for (category in names(all_marker_cats)) {
        for (gene in all_marker_cats[[category]]) {
            expr <- expr_data[gene, cells]
            
            dot_list[[idx]] <- data.frame(
                Cluster = cluster,
                Gene = gene,
                Category = category,
                Pct_Exp = sum(expr > 0) / length(expr) * 100,
                Avg_Exp = mean(expr),
                stringsAsFactors = FALSE
            )
            idx <- idx + 1
        }
    }
}

dot_data_cluster <- do.call(rbind, dot_list)

dot_data_cluster <- dot_data_cluster %>%
    group_by(Gene) %>%
    mutate(Scaled_Exp = as.numeric(scale(Avg_Exp))) %>%
    ungroup()

dot_data_cluster$Scaled_Exp <- pmax(pmin(dot_data_cluster$Scaled_Exp, 2.5), -2.5)
dot_data_cluster$Cluster <- factor(dot_data_cluster$Cluster, levels = rev(cluster_order))
dot_data_cluster$Gene <- factor(dot_data_cluster$Gene, levels = gene_order)
dot_data_cluster$Category <- factor(dot_data_cluster$Category, levels = names(all_marker_cats))

pos_cats <- names(pos_markers_list)
dot_data_cluster$Marker_Type <- ifelse(dot_data_cluster$Category %in% pos_cats,
                                        "Positive", "Negative")

cat("  ✓ Cluster dotplot:", nrow(dot_data_cluster), "data points,",
    length(unique(dot_data_cluster$Gene)), "genes,",
    length(cluster_order), "clusters\n")

# By state
dot_list_state <- list()
idx <- 1

for (state in states) {
    cells <- colnames(seurat_obj)[seurat_obj@meta.data[[SUBCLUSTER_COL]] == state]
    if (length(cells) == 0) next
    
    for (category in names(all_marker_cats)) {
        for (gene in all_marker_cats[[category]]) {
            expr <- expr_data[gene, cells]
            
            dot_list_state[[idx]] <- data.frame(
                State = state,
                Gene = gene,
                Category = category,
                Pct_Exp = sum(expr > 0) / length(expr) * 100,
                Avg_Exp = mean(expr),
                stringsAsFactors = FALSE
            )
            idx <- idx + 1
        }
    }
}

dot_data_state <- do.call(rbind, dot_list_state)

dot_data_state <- dot_data_state %>%
    group_by(Gene) %>%
    mutate(Scaled_Exp = as.numeric(scale(Avg_Exp))) %>%
    ungroup()

dot_data_state$Scaled_Exp <- pmax(pmin(dot_data_state$Scaled_Exp, 2.5), -2.5)
dot_data_state$State <- factor(dot_data_state$State, levels = rev(states))
dot_data_state$Gene <- factor(dot_data_state$Gene, levels = gene_order)
dot_data_state$Category <- factor(dot_data_state$Category, levels = names(all_marker_cats))

cat("  ✓ State dotplot:", nrow(dot_data_state), "data points\n")

# ─────────────────────────────────────────────────────────────────────────────────
# 2F: DotPlot — by cluster
# ─────────────────────────────────────────────────────────────────────────────────

cat("\n▸ Generating DotPlots...\n")

options(repr.plot.width = 16, repr.plot.height = 10)

p_dot_cluster <- ggplot(dot_data_cluster, aes(x = Gene, y = Cluster)) +
    geom_point(aes(size = Pct_Exp, fill = Scaled_Exp), shape = 21, color = "black", stroke = 0.3) +
    scale_size_continuous(
        range = c(1, 6), limits = c(0, 100),
        breaks = c(0, 25, 50, 75, 100), name = "% Expr"
    ) +
    scale_fill_gradientn(
        colors = c("#313695", "#4575B4", "#74ADD1", "#FFFFBF", "#FDAE61", "#F46D43", "#A50026"),
        limits = c(-2.5, 2.5), name = "Scaled\nExpr"
    ) +
    facet_grid(cols = vars(Category), scales = "free_x", space = "free_x") +
    labs(x = NULL, y = "Cluster",
         title = paste0(toupper(cell_type_label), ": Identity & Contamination Markers (by cluster)")) +
    theme_bw(base_size = 10) +
    theme(
        axis.text.x = element_text(angle = 45, hjust = 1, size = 7, face = "italic"),
        axis.text.y = element_text(size = 9),
        axis.title.y = element_text(size = 10, face = "bold"),
        strip.text = element_text(size = 7, face = "bold"),
        strip.background = element_rect(fill = "gray70", color = "black", linewidth = 0.5),
        panel.grid = element_blank(),
        panel.spacing = unit(0.2, "lines"),
        panel.border = element_rect(color = "black", linewidth = 0.5),
        legend.position = "right",
        legend.title = element_text(size = 8),
        legend.text = element_text(size = 7),
        legend.key.size = unit(0.4, "cm"),
        plot.title = element_text(size = 11, face = "bold"),
        plot.margin = margin(10, 10, 10, 10)
    )

print(p_dot_cluster)

ggsave(file.path(VAL_FIGURES_DIR, paste0(FILE_PREFIX, "_dotplot_cluster.pdf")),
       p_dot_cluster, width = 16, height = 10)
ggsave(file.path(VAL_FIGURES_DIR, paste0(FILE_PREFIX, "_dotplot_cluster.svg")),
       p_dot_cluster, width = 16, height = 10)

# ─────────────────────────────────────────────────────────────────────────────────
# 2G: DotPlot — by state
# ─────────────────────────────────────────────────────────────────────────────────

options(repr.plot.width = 16, repr.plot.height = 5)

p_dot_state <- ggplot(dot_data_state, aes(x = Gene, y = State)) +
    geom_point(aes(size = Pct_Exp, fill = Scaled_Exp), shape = 21, color = "black", stroke = 0.3) +
    scale_size_continuous(
        range = c(1, 6), limits = c(0, 100),
        breaks = c(0, 25, 50, 75, 100), name = "% Expr"
    ) +
    scale_fill_gradientn(
        colors = c("#313695", "#4575B4", "#74ADD1", "#FFFFBF", "#FDAE61", "#F46D43", "#A50026"),
        limits = c(-2.5, 2.5), name = "Scaled\nExpr"
    ) +
    facet_grid(cols = vars(Category), scales = "free_x", space = "free_x") +
    labs(x = NULL, y = "State",
         title = paste0(toupper(cell_type_label), ": Identity & Contamination Markers (by state)")) +
    theme_bw(base_size = 10) +
    theme(
        axis.text.x = element_text(angle = 45, hjust = 1, size = 7, face = "italic"),
        axis.text.y = element_text(size = 9),
        axis.title.y = element_text(size = 10, face = "bold"),
        strip.text = element_text(size = 7, face = "bold"),
        strip.background = element_rect(fill = "gray70", color = "black", linewidth = 0.5),
        panel.grid = element_blank(),
        panel.spacing = unit(0.2, "lines"),
        panel.border = element_rect(color = "black", linewidth = 0.5),
        legend.position = "right",
        legend.title = element_text(size = 8),
        legend.text = element_text(size = 7),
        legend.key.size = unit(0.4, "cm"),
        plot.title = element_text(size = 11, face = "bold"),
        plot.margin = margin(10, 10, 10, 10)
    )

print(p_dot_state)

ggsave(file.path(VAL_FIGURES_DIR, paste0(FILE_PREFIX, "_dotplot_state.pdf")),
       p_dot_state, width = 16, height = 5)
ggsave(file.path(VAL_FIGURES_DIR, paste0(FILE_PREFIX, "_dotplot_state.svg")),
       p_dot_state, width = 16, height = 5)

cat("  ✓ Saved DotPlots\n")

# ─────────────────────────────────────────────────────────────────────────────────
# 2H: Module Score Heatmap — clusters × lineages
# ─────────────────────────────────────────────────────────────────────────────────

cat("\n▸ Generating module score heatmap...\n")

score_mat <- data.frame(
    Cluster = score_by_cluster$Cluster,
    Identity = score_by_cluster$Identity
)
for (lineage in names(neg_markers_list)) {
    if (lineage %in% colnames(score_by_cluster)) {
        score_mat[[lineage]] <- score_by_cluster[[lineage]]
    }
}

score_long <- tidyr::pivot_longer(score_mat, cols = -Cluster,
                                   names_to = "Module", values_to = "Score")
score_long$Cluster <- factor(score_long$Cluster, levels = rev(cluster_order))
module_order <- c("Identity", names(neg_markers_list))
module_order <- module_order[module_order %in% unique(score_long$Module)]
score_long$Module <- factor(score_long$Module, levels = module_order)

score_long$State <- sapply(as.character(score_long$Cluster), function(cl) {
    row <- which(qc_by_cluster$Cluster == cl)
    if (length(row) > 0) qc_by_cluster$State[row] else ""
})
score_long$Y_label <- paste0(score_long$Cluster, " (", score_long$State, ")")
y_label_order <- rev(paste0(cluster_order, " (",
    sapply(cluster_order, function(cl) {
        row <- which(qc_by_cluster$Cluster == cl)
        if (length(row) > 0) qc_by_cluster$State[row] else ""
    }), ")"))
score_long$Y_label <- factor(score_long$Y_label, levels = y_label_order)

options(repr.plot.width = 8, repr.plot.height = max(6, length(cluster_order) * 0.4))

p_heatmap <- ggplot(score_long, aes(x = Module, y = Y_label, fill = Score)) +
    geom_tile(color = "white", linewidth = 0.5) +
    geom_text(aes(label = sprintf("%.3f", Score)), size = 2.5, color = "black") +
    scale_fill_gradient2(low = "#313695", mid = "#FFFFBF", high = "#A50026",
                          midpoint = 0, name = "Module\nScore") +
    labs(x = NULL, y = NULL,
         title = paste0(toupper(cell_type_label),
                        ": Identity vs Contamination Module Scores"),
         subtitle = "AddModuleScore (median per cluster, 0 = background)") +
    theme_minimal(base_size = 10) +
    theme(
        axis.text.x = element_text(angle = 45, hjust = 1, size = 9, face = "bold"),
        axis.text.y = element_text(size = 8),
        plot.title = element_text(size = 11, face = "bold"),
        plot.subtitle = element_text(size = 9, color = "#666666"),
        panel.grid = element_blank(),
        legend.position = "right"
    )

print(p_heatmap)

ggsave(file.path(VAL_FIGURES_DIR, paste0(FILE_PREFIX, "_module_score_heatmap.pdf")),
       p_heatmap, width = 8, height = max(6, length(cluster_order) * 0.4))
ggsave(file.path(VAL_FIGURES_DIR, paste0(FILE_PREFIX, "_module_score_heatmap.svg")),
       p_heatmap, width = 8, height = max(6, length(cluster_order) * 0.4))

cat("  ✓ Saved module score heatmap\n")

# ─────────────────────────────────────────────────────────────────────────────────
# Step 2 Summary
# ─────────────────────────────────────────────────────────────────────────────────

cat("\n", paste(rep("-", 80), collapse = ""), "\n")
cat("STEP 2 SUMMARY\n")
cat(paste(rep("-", 80), collapse = ""), "\n")
cat("  Method: AddModuleScore (expression-level controlled)\n")
cat("  Positive markers:", length(all_pos_genes), "\n")
cat("  Positive categories:", paste(names(pos_markers_list), collapse = ", "), "\n")
cat("  Negative lineages:", length(neg_markers_list),
    "(", paste(names(neg_markers_list), collapse = ", "), ")\n")
cat("  Negative markers:", length(unlist(neg_markers_list)), "\n")
cat("  Weak identity (score < 0):", if(flagged_id) "YES" else "NONE", "\n")
cat("  Contamination (outlier):", if(flagged_contam) "YES" else "NONE", "\n")
cat("  Global median identity:", round(median(seurat_obj$identity_score1), 4), "\n")
cat("\n  Contamination baselines:\n")
for (i in 1:nrow(lineage_baselines)) {
    cat(sprintf("    %s: median=%.4f, threshold=%.4f (median + 3×MAD)\n",
                lineage_baselines$Lineage[i],
                lineage_baselines$Median[i],
                lineage_baselines$Threshold[i]))
}
cat("\n  Files saved:\n")
cat("    •", paste0(FILE_PREFIX, "_identity_scores.csv"), "\n")
cat("    •", paste0(FILE_PREFIX, "_identity_scores_state.csv"), "\n")
cat("    •", paste0(FILE_PREFIX, "_dotplot_cluster.{pdf,svg}"), "\n")
cat("    •", paste0(FILE_PREFIX, "_dotplot_state.{pdf,svg}"), "\n")
cat("    •", paste0(FILE_PREFIX, "_module_score_heatmap.{pdf,svg}"), "\n")
cat(paste(rep("=", 80), collapse = ""), "\n")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# STEP 3: SAMPLE COMPOSITION
# ════════════════════════════════════════════════════════════════════════════════

cat("\n", paste(rep("=", 80), collapse = ""), "\n")
cat("STEP 3: SAMPLE COMPOSITION\n")
cat(paste(rep("=", 80), collapse = ""), "\n")

# ─────────────────────────────────────────────────────────────────────────────────
# Thresholds
# ─────────────────────────────────────────────────────────────────────────────────

DONOR_DOMINANCE_THRESHOLD <- 60
EVENNESS_FLAG_THRESHOLD <- 0.40
FEW_DONORS_THRESHOLD <- 3

# ─────────────────────────────────────────────────────────────────────────────────
# 3A: Per-cluster donor composition
# ─────────────────────────────────────────────────────────────────────────────────

cat("\n▸ Analyzing donor composition per cluster...\n")

comp_cluster <- table(seurat_obj$seurat_clusters, seurat_obj@meta.data[[DONOR_COL]])

cluster_comp_summary <- do.call(rbind, lapply(cluster_order, function(cl) {
    cl_counts <- comp_cluster[cl, ]
    cl_counts <- cl_counts[cl_counts > 0]
    n_donors_cl <- length(cl_counts)
    total <- sum(cl_counts)
    sorted_counts <- sort(cl_counts, decreasing = TRUE)
    top1_pct <- sorted_counts[1] / total * 100
    top3_pct <- sum(sorted_counts[1:min(3, length(sorted_counts))]) / total * 100
    top_donor <- names(sorted_counts)[1]
    
    props <- cl_counts / total
    shannon <- -sum(props * log(props))
    max_shannon <- log(n_donors_cl)
    evenness <- if (max_shannon > 0) shannon / max_shannon else 0
    
    data.frame(
        Cluster = cl,
        State = qc_by_cluster$State[qc_by_cluster$Cluster == cl],
        N_cells = total,
        N_donors = n_donors_cl,
        Top_donor = top_donor,
        Top1_pct = round(top1_pct, 1),
        Top3_pct = round(top3_pct, 1),
        Shannon = round(shannon, 2),
        Evenness = round(evenness, 2),
        stringsAsFactors = FALSE
    )
}))

cat("\n  Cluster composition summary:\n")
print(cluster_comp_summary, row.names = FALSE)

write.csv(cluster_comp_summary,
          file.path(VAL_RESULTS_DIR, paste0(FILE_PREFIX, "_sample_comp_cluster.csv")),
          row.names = FALSE)

# ─────────────────────────────────────────────────────────────────────────────────
# 3B: Per-state donor composition
# ─────────────────────────────────────────────────────────────────────────────────

cat("\n▸ Analyzing donor composition per state...\n")

comp_state <- table(seurat_obj@meta.data[[SUBCLUSTER_COL]], seurat_obj@meta.data[[DONOR_COL]])

state_comp_summary <- do.call(rbind, lapply(states, function(s) {
    s_counts <- comp_state[s, ]
    s_counts <- s_counts[s_counts > 0]
    n_donors_s <- length(s_counts)
    total <- sum(s_counts)
    sorted_counts <- sort(s_counts, decreasing = TRUE)
    top1_pct <- sorted_counts[1] / total * 100
    top3_pct <- sum(sorted_counts[1:min(3, length(sorted_counts))]) / total * 100
    
    props <- s_counts / total
    shannon <- -sum(props * log(props))
    max_shannon <- log(n_donors_s)
    evenness <- if (max_shannon > 0) shannon / max_shannon else 0
    
    data.frame(
        State = s,
        N_cells = total,
        N_donors = n_donors_s,
        Top1_pct = round(top1_pct, 1),
        Top3_pct = round(top3_pct, 1),
        Shannon = round(shannon, 2),
        Evenness = round(evenness, 2),
        stringsAsFactors = FALSE
    )
}))

cat("\n  State composition summary:\n")
print(state_comp_summary, row.names = FALSE)

write.csv(state_comp_summary,
          file.path(VAL_RESULTS_DIR, paste0(FILE_PREFIX, "_sample_comp_state.csv")),
          row.names = FALSE)

# ─────────────────────────────────────────────────────────────────────────────────
# 3C: Flag sample-dominated clusters
# ─────────────────────────────────────────────────────────────────────────────────

cat("\n▸ Flagging sample-dominated clusters:\n")
cat(sprintf("  Thresholds: top donor >%d%%, evenness <%.2f, donors <=%d\n",
            DONOR_DOMINANCE_THRESHOLD, EVENNESS_FLAG_THRESHOLD, FEW_DONORS_THRESHOLD))

flagged_sample <- FALSE
flagged_clusters <- c()

for (i in 1:nrow(cluster_comp_summary)) {
    flags_list <- c()
    
    if (cluster_comp_summary$Top1_pct[i] > DONOR_DOMINANCE_THRESHOLD) {
        flags_list <- c(flags_list, sprintf("TOP DONOR >%d%% (%s = %.1f%%)",
                                   DONOR_DOMINANCE_THRESHOLD,
                                   cluster_comp_summary$Top_donor[i],
                                   cluster_comp_summary$Top1_pct[i]))
    }
    if (cluster_comp_summary$N_donors[i] <= FEW_DONORS_THRESHOLD) {
        flags_list <- c(flags_list, sprintf("FEW DONORS (n=%d)", cluster_comp_summary$N_donors[i]))
    }
    if (cluster_comp_summary$Evenness[i] < EVENNESS_FLAG_THRESHOLD) {
        flags_list <- c(flags_list, sprintf("LOW EVENNESS (%.2f)", cluster_comp_summary$Evenness[i]))
    }
    
    if (length(flags_list) > 0) {
        cat(sprintf("  ⚠ Cluster %s (%s, n=%d): %s\n",
                    cluster_comp_summary$Cluster[i],
                    cluster_comp_summary$State[i],
                    cluster_comp_summary$N_cells[i],
                    paste(flags_list, collapse = "; ")))
        flagged_sample <- TRUE
        flagged_clusters <- c(flagged_clusters, as.character(cluster_comp_summary$Cluster[i]))
    }
}
if (!flagged_sample) cat("  ✓ No sample-dominated clusters\n")

# ─────────────────────────────────────────────────────────────────────────────────
# 3D: State robustness after excluding flagged clusters
# ─────────────────────────────────────────────────────────────────────────────────

cat("\n▸ State robustness check (excluding flagged clusters)...\n")
cat("  Goal: Verify each state retains multi-donor support after removing\n")
cat("  sample-dominated clusters. If a state is defined primarily by flagged\n")
cat("  clusters, it may be a donor artifact rather than real biology.\n")

if (length(flagged_clusters) > 0) {
    flagged_cells <- as.character(seurat_obj$seurat_clusters) %in% flagged_clusters
    clean_meta <- seurat_obj@meta.data[!flagged_cells, ]
    
    state_robustness <- do.call(rbind, lapply(state_order, function(st) {
        all_cells <- seurat_obj@meta.data[[SUBCLUSTER_COL]] == st
        n_total <- sum(all_cells)
        n_donors_total <- length(unique(seurat_obj@meta.data[[DONOR_COL]][all_cells]))
        n_clusters_total <- length(unique(as.character(seurat_obj$seurat_clusters[all_cells])))
        
        clean_cells <- clean_meta[[SUBCLUSTER_COL]] == st
        n_clean <- sum(clean_cells)
        n_donors_clean <- length(unique(clean_meta[[DONOR_COL]][clean_cells]))
        n_clusters_clean <- length(unique(as.character(clean_meta$seurat_clusters[clean_cells])))
        
        pct_retained <- round(n_clean / n_total * 100, 1)
        
        donor_table <- table(clean_meta[[DONOR_COL]][clean_cells])
        donor_props <- as.numeric(donor_table) / sum(donor_table)
        evenness <- if (length(donor_props) > 1) {
            -sum(donor_props * log(donor_props)) / log(length(donor_props))
        } else { 0 }
        
        top_donor_pct <- round(max(donor_table) / sum(donor_table) * 100, 1)
        
        data.frame(
            State = st,
            Cells_total = n_total,
            Cells_clean = n_clean,
            Pct_retained = pct_retained,
            Donors_total = n_donors_total,
            Donors_clean = n_donors_clean,
            Clusters_total = n_clusters_total,
            Clusters_clean = n_clusters_clean,
            Top_donor_pct = top_donor_pct,
            Evenness = round(evenness, 2),
            stringsAsFactors = FALSE
        )
    }))
    
    cat("\n")
    print(state_robustness, row.names = FALSE)
    
    at_risk <- state_robustness[state_robustness$Donors_clean < 5 | 
                                 state_robustness$Pct_retained < 50 |
                                 state_robustness$Evenness < 0.5, ]
    
    if (nrow(at_risk) > 0) {
        cat("\n  ⚠ States at risk after removing flagged clusters:\n")
        for (i in 1:nrow(at_risk)) {
            reasons <- c()
            if (at_risk$Donors_clean[i] < 5) reasons <- c(reasons, 
                sprintf("only %d donors remain", at_risk$Donors_clean[i]))
            if (at_risk$Pct_retained[i] < 50) reasons <- c(reasons, 
                sprintf("only %.1f%% cells retained", at_risk$Pct_retained[i]))
            if (at_risk$Evenness[i] < 0.5) reasons <- c(reasons, 
                sprintf("low evenness (%.2f)", at_risk$Evenness[i]))
            cat(sprintf("    %s: %s\n", at_risk$State[i], paste(reasons, collapse = ", ")))
        }
    } else {
        cat("\n  ✓ All states retain multi-donor support after removing flagged clusters\n")
    }
    
    write.csv(state_robustness,
              file.path(VAL_RESULTS_DIR, paste0(FILE_PREFIX, "_state_robustness.csv")),
              row.names = FALSE)
} else {
    cat("  No flagged clusters -- skipping robustness check\n")
}

# ─────────────────────────────────────────────────────────────────────────────────
# 3E: Multi-resolution clustering + State stability across resolutions
# ─────────────────────────────────────────────────────────────────────────────────

cat("\n▸ State stability across resolutions...\n")
cat("  Goal: Verify each state emerges as a distinct cluster (or set of clusters)\n")
cat("  at multiple resolutions. A real state should be recoverable across\n")
cat("  resolutions, not only at the chosen one.\n")

# ── Ensure multi-resolution clustering exists ──
res_seq <- c(0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 1.0, 1.2)
existing_res <- grep("RNA_snn_res\\.", colnames(seurat_obj@meta.data), value = TRUE)
existing_values <- as.numeric(gsub("RNA_snn_res\\.", "", existing_res))
missing_res <- setdiff(res_seq, existing_values)

if (length(missing_res) > 0) {
    cat("  Computing clustering at", length(missing_res), "additional resolutions...\n")
    
    # Preserve original clustering
    seurat_obj$original_clusters <- seurat_obj$seurat_clusters
    
    # Determine reduction and dims
    cluster_reduction <- ifelse("harmony" %in% Reductions(seurat_obj), "harmony", "pca")
    if (!exists("N_DIMS_USE")) {
        N_DIMS_USE <- min(ncol(Embeddings(seurat_obj, cluster_reduction)), 30)
        cat("  N_DIMS_USE auto-detected:", N_DIMS_USE, "\n")
    }
    cat("  Using reduction:", cluster_reduction, ", dims: 1:", N_DIMS_USE, "\n")
    
    # Rebuild neighbors if needed
    if (length(seurat_obj@graphs) == 0) {
        cat("  Building neighbor graph...\n")
        seurat_obj <- FindNeighbors(seurat_obj, reduction = cluster_reduction,
                                     dims = 1:N_DIMS_USE, verbose = FALSE)
    }
    
    for (res in sort(missing_res)) {
        seurat_obj <- FindClusters(seurat_obj, resolution = res, verbose = FALSE)
        col_name <- paste0("RNA_snn_res.", res)
        n_cl <- length(unique(seurat_obj@meta.data[[col_name]]))
        cat(sprintf("    res = %.1f -> %d clusters\n", res, n_cl))
    }
    
    # Restore original clustering
    seurat_obj$seurat_clusters <- seurat_obj$original_clusters
    Idents(seurat_obj) <- seurat_obj$seurat_clusters
    seurat_obj$original_clusters <- NULL
    cat("  ✓ Restored original seurat_clusters (",
        length(unique(seurat_obj$seurat_clusters)), " clusters)\n", sep = "")
} else {
    cat("  All", length(res_seq), "resolutions already computed\n")
}

# Print resolution summary
available_res <- grep("RNA_snn_res\\.", colnames(seurat_obj@meta.data), value = TRUE)
res_values <- sort(as.numeric(gsub("RNA_snn_res\\.", "", available_res)))
cat("  Available resolutions:", paste(res_values, collapse = ", "), "\n\n")

for (res in res_seq) {
    col_name <- paste0("RNA_snn_res.", res)
    if (col_name %in% colnames(seurat_obj@meta.data)) {
        n_cl <- length(unique(seurat_obj@meta.data[[col_name]]))
        marker <- ifelse(res == CLUSTERING_RESOLUTION, " <- CURRENT", "")
        cat(sprintf("    res = %.1f -> %d clusters%s\n", res, n_cl, marker))
    }
}

# ── Compute state recovery per resolution ──
cat("\n  Computing state recovery per resolution...\n")

state_by_res <- do.call(rbind, lapply(res_values, function(res) {
    col_name <- paste0("RNA_snn_res.", res)
    
    do.call(rbind, lapply(state_order, function(st) {
        state_cells <- seurat_obj@meta.data[[SUBCLUSTER_COL]] == st
        
        majority_clusters <- c()
        for (cl in unique(seurat_obj@meta.data[[col_name]])) {
            cl_cells <- seurat_obj@meta.data[[col_name]] == cl
            cl_states <- table(seurat_obj@meta.data[[SUBCLUSTER_COL]][cl_cells])
            if (names(which.max(cl_states)) == st) {
                majority_clusters <- c(majority_clusters, cl)
            }
        }
        
        n_majority <- length(majority_clusters)
        in_majority <- sum(seurat_obj@meta.data[[col_name]][state_cells] %in% majority_clusters)
        recovery_pct <- round(in_majority / sum(state_cells) * 100, 1)
        
        data.frame(
            Resolution = res,
            State = st,
            N_majority_clusters = n_majority,
            Recovery_pct = recovery_pct,
            stringsAsFactors = FALSE
        )
    }))
}))

recovery_wide <- reshape(state_by_res[, c("Resolution", "State", "Recovery_pct")],
                          idvar = "Resolution", timevar = "State",
                          direction = "wide")
colnames(recovery_wide) <- gsub("Recovery_pct\\.", "", colnames(recovery_wide))
cat("\n  Recovery % per state across resolutions:\n")
print(recovery_wide, row.names = FALSE)

# ── Interpret per state ──
cat("\n")
for (st in state_order) {
    st_data <- state_by_res[state_by_res$State == st, ]
    min_rec <- min(st_data$Recovery_pct)
    low_recovery <- st_data[st_data$Recovery_pct < 50, ]
    
    if (nrow(low_recovery) > 0) {
        low_res <- low_recovery$Resolution
        chosen_rec <- st_data$Recovery_pct[st_data$Resolution == CLUSTERING_RESOLUTION]
        if (length(chosen_rec) == 0) chosen_rec <- NA
        
        all_below_chosen <- all(low_res < CLUSTERING_RESOLUTION)
        at_and_above <- st_data[st_data$Resolution >= CLUSTERING_RESOLUTION, ]
        stable_above <- all(at_and_above$Recovery_pct >= 70)
        
        if (all_below_chosen && stable_above && !is.na(chosen_rec) && chosen_rec >= 70) {
            first_stable <- min(st_data$Resolution[st_data$Recovery_pct >= 50])
            n_state_cells <- sum(seurat_obj@meta.data[[SUBCLUSTER_COL]] == st)
            size_note <- ifelse(n_state_cells < 2000, "a small state", "this state")
            cat(sprintf("  ✓ %s: stable at working resolutions (min %.1f%% at coarse res %s)\n",
                        st, min_rec, paste(low_res, collapse = ", ")))
            cat(sprintf("    Emerges from res %.1f onward — expected for %s (%d cells)\n",
                        first_stable, size_note, n_state_cells))
        } else if (!is.na(chosen_rec) && chosen_rec >= 70) {
            cat(sprintf("  ~ %s: partially stable -- <50%% at res %s, but %.1f%% at chosen res\n",
                        st, paste(low_res, collapse = ", "), chosen_rec))
            if (all_below_chosen) {
                cat("    Absorbed into neighboring states at coarse resolutions\n")
            } else {
                cat("    Variable recovery suggests resolution-sensitive boundaries\n")
            }
        } else {
            cat(sprintf("  ⚠ %s: UNSTABLE -- %.1f%% at chosen res, <50%% at res %s\n",
                        st, ifelse(is.na(chosen_rec), NA, chosen_rec),
                        paste(low_res, collapse = ", ")))
            cat("    This state may not be well-supported at the current resolution\n")
        }
    } else {
        cat(sprintf("  ✓ %s: stable across all resolutions (min recovery: %.1f%%)\n",
                    st, min_rec))
    }
}

write.csv(state_by_res,
          file.path(VAL_RESULTS_DIR, paste0(FILE_PREFIX, "_state_resolution_stability.csv")),
          row.names = FALSE)

# ─────────────────────────────────────────────────────────────────────────────────
# 3F: DONOR COMPOSITION STABILITY ACROSS RESOLUTIONS
# ─────────────────────────────────────────────────────────────────────────────────
# Goal: For each resolution, find every cluster and compute its donor composition.
# Then check: do donor-dominated clusters persistently appear across resolutions,
# or do those cells merge into well-mixed clusters at other resolutions?
# This answers whether the sample dominance is a stable structural feature
# (those cells always segregate) or a resolution artifact (they only split off
# at one resolution).
# ─────────────────────────────────────────────────────────────────────────────────

cat("\n▸ Donor composition stability across resolutions...\n")
cat("  Goal: Track whether donor-dominated clusters persist, resolve, or worsen\n")
cat("  across resolutions. If flagged cells always form their own donor-dominated\n")
cat("  cluster, the dominance is structural. If they merge into well-mixed clusters\n")
cat("  at other resolutions, it is a resolution artifact.\n\n")

# ── 3F-i: Per-cluster donor composition at every resolution ──

multires_donor_comp <- do.call(rbind, lapply(res_values, function(res) {
    col_name <- paste0("RNA_snn_res.", res)
    clusters_at_res <- sort(unique(seurat_obj@meta.data[[col_name]]))
    
    do.call(rbind, lapply(clusters_at_res, function(cl) {
        cl_cells <- seurat_obj@meta.data[[col_name]] == cl
        donor_counts <- table(seurat_obj@meta.data[[DONOR_COL]][cl_cells])
        donor_counts <- donor_counts[donor_counts > 0]
        
        n_cells <- sum(cl_cells)
        n_donors <- length(donor_counts)
        sorted <- sort(donor_counts, decreasing = TRUE)
        top1_pct <- sorted[1] / n_cells * 100
        top_donor <- names(sorted)[1]
        
        props <- donor_counts / n_cells
        shannon <- -sum(props * log(props))
        max_shannon <- log(n_donors)
        evenness <- if (max_shannon > 0) shannon / max_shannon else 0
        
        # Majority annotated state for this cluster
        state_table <- table(seurat_obj@meta.data[[SUBCLUSTER_COL]][cl_cells])
        majority_state <- names(which.max(state_table))
        state_purity <- round(max(state_table) / n_cells * 100, 1)
        
        data.frame(
            Resolution = res,
            Cluster = as.character(cl),
            N_cells = n_cells,
            Majority_state = majority_state,
            State_purity = state_purity,
            N_donors = n_donors,
            Top_donor = top_donor,
            Top1_pct = round(top1_pct, 1),
            Evenness = round(evenness, 2),
            Flagged = (top1_pct > DONOR_DOMINANCE_THRESHOLD | evenness < EVENNESS_FLAG_THRESHOLD),
            stringsAsFactors = FALSE
        )
    }))
}))

# Save full table
write.csv(multires_donor_comp,
          file.path(VAL_RESULTS_DIR, paste0(FILE_PREFIX, "_multires_donor_comp.csv")),
          row.names = FALSE)

# ── 3F-ii: Summary — how many flagged clusters per resolution? ──

cat("  Flagged clusters per resolution:\n")
flag_summary <- do.call(rbind, lapply(res_values, function(res) {
    res_data <- multires_donor_comp[multires_donor_comp$Resolution == res, ]
    n_total <- nrow(res_data)
    n_flagged <- sum(res_data$Flagged)
    n_cells_flagged <- sum(res_data$N_cells[res_data$Flagged])
    pct_cells_flagged <- round(n_cells_flagged / ncol(seurat_obj) * 100, 1)
    
    data.frame(
        Resolution = res,
        N_clusters = n_total,
        N_flagged = n_flagged,
        Cells_in_flagged = n_cells_flagged,
        Pct_cells_flagged = pct_cells_flagged,
        stringsAsFactors = FALSE
    )
}))
print(flag_summary, row.names = FALSE)

write.csv(flag_summary,
          file.path(VAL_RESULTS_DIR, paste0(FILE_PREFIX, "_multires_flag_summary.csv")),
          row.names = FALSE)

# ── 3F-iii: Track flagged donors' cells across resolutions ──
# For each donor that dominated a cluster at the current resolution,
# trace where those cells end up at every other resolution.

if (length(flagged_clusters) > 0) {
    cat("\n  Tracking flagged donors' cells across resolutions...\n")
    
    # Identify the dominant donor + their cells for each flagged cluster
    flagged_donor_info <- lapply(flagged_clusters, function(cl) {
        cl_data <- cluster_comp_summary[cluster_comp_summary$Cluster == cl, ]
        dominant_donor <- cl_data$Top_donor
        # Get cell barcodes from this donor that were in this cluster
        in_cluster <- as.character(seurat_obj$seurat_clusters) == cl
        from_donor <- seurat_obj@meta.data[[DONOR_COL]] == dominant_donor
        cell_barcodes <- colnames(seurat_obj)[in_cluster & from_donor]
        
        list(
            cluster = cl,
            state = cl_data$State,
            donor = dominant_donor,
            n_cells = length(cell_barcodes),
            barcodes = cell_barcodes
        )
    })
    names(flagged_donor_info) <- flagged_clusters
    
    # For each flagged cluster's dominant-donor cells, check where they land
    # at every resolution
    donor_tracking <- do.call(rbind, lapply(flagged_donor_info, function(info) {
        cell_idx <- match(info$barcodes, colnames(seurat_obj))
        
        do.call(rbind, lapply(res_values, function(res) {
            col_name <- paste0("RNA_snn_res.", res)
            assignments <- seurat_obj@meta.data[[col_name]][cell_idx]
            
            # Where do these cells go?
            dest_table <- sort(table(assignments), decreasing = TRUE)
            n_dest_clusters <- length(dest_table)
            top_dest <- names(dest_table)[1]
            top_dest_pct <- round(dest_table[1] / length(assignments) * 100, 1)
            
            # In the cluster they land in, what fraction are they?
            top_dest_all_cells <- seurat_obj@meta.data[[col_name]] == top_dest
            n_top_dest_total <- sum(top_dest_all_cells)
            pct_of_dest <- round(dest_table[1] / n_top_dest_total * 100, 1)
            
            # Is that destination cluster also flagged?
            dest_data <- multires_donor_comp[multires_donor_comp$Resolution == res &
                                              multires_donor_comp$Cluster == top_dest, ]
            dest_flagged <- if (nrow(dest_data) > 0) dest_data$Flagged[1] else NA
            dest_evenness <- if (nrow(dest_data) > 0) dest_data$Evenness[1] else NA
            
            data.frame(
                Source_cluster = info$cluster,
                Source_state = info$state,
                Dominant_donor = info$donor,
                N_tracked_cells = length(info$barcodes),
                Resolution = res,
                N_dest_clusters = n_dest_clusters,
                Top_destination = top_dest,
                Pct_cells_in_top_dest = top_dest_pct,
                Pct_of_dest_cluster = pct_of_dest,
                Dest_evenness = dest_evenness,
                Dest_flagged = dest_flagged,
                stringsAsFactors = FALSE
            )
        }))
    }))
    
    # Print tracking results per flagged cluster
    for (cl in flagged_clusters) {
        cl_track <- donor_tracking[donor_tracking$Source_cluster == cl, ]
        info <- flagged_donor_info[[cl]]
        
        cat(sprintf("\n  ── Cluster %s (%s) — dominant donor: %s (%d cells) ──\n",
                    cl, info$state, info$donor, info$n_cells))
        
        print_cols <- c("Resolution", "N_dest_clusters", "Top_destination",
                        "Pct_cells_in_top_dest", "Pct_of_dest_cluster",
                        "Dest_evenness", "Dest_flagged")
        print(cl_track[, print_cols], row.names = FALSE)
        
        # Interpret: do these cells always segregate?
        always_dominant <- all(cl_track$Pct_of_dest_cluster > DONOR_DOMINANCE_THRESHOLD)
        always_together <- all(cl_track$Pct_cells_in_top_dest > 80)
        never_flagged_elsewhere <- all(!cl_track$Dest_flagged, na.rm = TRUE)
        
        if (always_dominant & always_together) {
            cat(sprintf("    → STRUCTURAL: These %d cells always form a donor-dominated cluster.\n",
                        info$n_cells))
            cat("      The dominance persists across all resolutions — not a resolution artifact.\n")
            cat("      Recommendation: REMOVE or treat as donor-specific.\n")
        } else if (never_flagged_elsewhere) {
            cat("    → RESOLUTION ARTIFACT: These cells merge into well-mixed clusters\n")
            cat("      at other resolutions. The dominance is specific to the current resolution.\n")
            cat("      Recommendation: Likely safe to KEEP (dominance dilutes at other resolutions).\n")
        } else {
            cat("    → MIXED: Dominance varies across resolutions.\n")
            # Count how many resolutions show flagged destination
            n_flagged_res <- sum(cl_track$Dest_flagged, na.rm = TRUE)
            cat(sprintf("      Flagged at %d/%d resolutions.\n",
                        n_flagged_res, nrow(cl_track)))
            if (n_flagged_res > nrow(cl_track) / 2) {
                cat("      Recommendation: Lean toward REMOVE (dominant at most resolutions).\n")
            } else {
                cat("      Recommendation: MONITOR — verify DEG results aren't donor-driven.\n")
            }
        }
    }
    
    write.csv(donor_tracking,
              file.path(VAL_RESULTS_DIR, paste0(FILE_PREFIX, "_donor_tracking_across_res.csv")),
              row.names = FALSE)
    
} else {
    cat("  No flagged clusters — skipping donor tracking.\n")
}

# ── 3F-iv: Visualization — Donor dominance heatmap across resolutions ──

cat("\n▸ Generating multi-resolution donor composition plots...\n")

# Heatmap: top donor % per cluster per resolution
# Only show resolutions with a manageable number of clusters

library(ggplot2)

# Plot 1: Number of flagged clusters per resolution
options(repr.plot.width = 10, repr.plot.height = 4)

p_flags_per_res <- ggplot(flag_summary, aes(x = factor(Resolution), y = N_flagged)) +
    geom_col(fill = "#E15759", alpha = 0.8, width = 0.6) +
    geom_text(aes(label = N_flagged), vjust = -0.3, size = 3) +
    geom_hline(yintercept = 0, linewidth = 0.3) +
    scale_y_continuous(expand = expansion(mult = c(0, 0.15))) +
    labs(x = "Resolution", y = "N flagged clusters",
         title = paste0(toupper(cell_type_label),
                        ": Donor-dominated clusters across resolutions"),
         subtitle = sprintf("Flagged = top donor >%d%% OR evenness <%.2f",
                           DONOR_DOMINANCE_THRESHOLD, EVENNESS_FLAG_THRESHOLD)) +
    theme_bw(base_size = 10) +
    theme(
        plot.title = element_text(size = 11, face = "bold"),
        plot.subtitle = element_text(size = 9, color = "grey40"),
        panel.grid.major.x = element_blank()
    )

print(p_flags_per_res)

ggsave(file.path(VAL_FIGURES_DIR, paste0(FILE_PREFIX, "_multires_flagged_count.pdf")),
       p_flags_per_res, width = 10, height = 4)
ggsave(file.path(VAL_FIGURES_DIR, paste0(FILE_PREFIX, "_multires_flagged_count.svg")),
       p_flags_per_res, width = 10, height = 4)

# Plot 2: Tracking plot — for each flagged cluster, show the % of destination
# cluster that is the dominant donor across resolutions

if (exists("donor_tracking") && nrow(donor_tracking) > 0) {
    options(repr.plot.width = 12, repr.plot.height = 4 + length(flagged_clusters) * 1.5)
    
    donor_tracking$label <- paste0("Cl.", donor_tracking$Source_cluster,
                                    " (", donor_tracking$Source_state, ")")
    
    p_tracking <- ggplot(donor_tracking,
                          aes(x = factor(Resolution), y = Pct_of_dest_cluster)) +
        geom_line(aes(group = label), color = "#4E79A7", linewidth = 0.8) +
        geom_point(aes(color = Dest_flagged), size = 2.5) +
        geom_hline(yintercept = DONOR_DOMINANCE_THRESHOLD,
                   linetype = "dashed", color = "red", linewidth = 0.4) +
        scale_color_manual(values = c("TRUE" = "#E15759", "FALSE" = "#59A14F"),
                          labels = c("TRUE" = "Flagged", "FALSE" = "Not flagged"),
                          name = "Destination cluster") +
        facet_wrap(~label, ncol = 1, scales = "free_y") +
        labs(x = "Resolution",
             y = "Dominant donor's % of destination cluster",
             title = paste0(toupper(cell_type_label),
                            ": Donor dominance tracking across resolutions"),
             subtitle = "Tracks where the dominant donor's cells land at each resolution") +
        theme_bw(base_size = 10) +
        theme(
            plot.title = element_text(size = 11, face = "bold"),
            plot.subtitle = element_text(size = 9, color = "grey40"),
            strip.text = element_text(size = 9, face = "bold"),
            legend.position = "bottom"
        )
    
    print(p_tracking)
    
    ggsave(file.path(VAL_FIGURES_DIR, paste0(FILE_PREFIX, "_donor_tracking_across_res.pdf")),
           p_tracking, width = 12, height = 4 + length(flagged_clusters) * 1.5)
    ggsave(file.path(VAL_FIGURES_DIR, paste0(FILE_PREFIX, "_donor_tracking_across_res.svg")),
           p_tracking, width = 12, height = 4 + length(flagged_clusters) * 1.5)
}

cat("  ✓ Saved multi-resolution donor composition plots\n")

# ─────────────────────────────────────────────────────────────────────────────────
# 3G: Stacked bar plot — by cluster
# ─────────────────────────────────────────────────────────────────────────────────

cat("\n▸ Generating composition plots...\n")

comp_df <- as.data.frame(comp_cluster)
colnames(comp_df) <- c("Cluster", "Donor", "Count")
comp_df <- comp_df[comp_df$Count > 0, ]
comp_df$Cluster <- factor(comp_df$Cluster, levels = cluster_order)

n_unique_donors <- length(unique(comp_df$Donor))

options(repr.plot.width = 12, repr.plot.height = 5)

p_comp_cluster <- ggplot(comp_df, aes(x = Cluster, y = Count, fill = Donor)) +
    geom_bar(stat = "identity", position = "fill", width = 0.8) +
    scale_y_continuous(labels = scales::percent, expand = c(0, 0)) +
    labs(x = "Cluster", y = "Proportion",
         title = paste0(toupper(cell_type_label), ": Donor composition per cluster")) +
    theme_bw(base_size = 10) +
    theme(
        legend.position = "none",
        axis.text.x = element_text(size = 9),
        plot.title = element_text(size = 11, face = "bold"),
        panel.grid.major.x = element_blank()
    )

print(p_comp_cluster)

ggsave(file.path(VAL_FIGURES_DIR, paste0(FILE_PREFIX, "_donor_comp_cluster.pdf")),
       p_comp_cluster, width = 12, height = 5)
ggsave(file.path(VAL_FIGURES_DIR, paste0(FILE_PREFIX, "_donor_comp_cluster.svg")),
       p_comp_cluster, width = 12, height = 5)

# ─────────────────────────────────────────────────────────────────────────────────
# 3H: Stacked bar plot — by state
# ─────────────────────────────────────────────────────────────────────────────────

comp_state_df <- as.data.frame(comp_state)
colnames(comp_state_df) <- c("State", "Donor", "Count")
comp_state_df <- comp_state_df[comp_state_df$Count > 0, ]
comp_state_df$State <- factor(comp_state_df$State, levels = states)

p_comp_state <- ggplot(comp_state_df, aes(x = State, y = Count, fill = Donor)) +
    geom_bar(stat = "identity", position = "fill", width = 0.8) +
    scale_y_continuous(labels = scales::percent, expand = c(0, 0)) +
    labs(x = "State", y = "Proportion",
         title = paste0(toupper(cell_type_label), ": Donor composition per state")) +
    theme_bw(base_size = 10) +
    theme(
        legend.position = "none",
        axis.text.x = element_text(angle = 30, hjust = 1, size = 9),
        plot.title = element_text(size = 11, face = "bold"),
        panel.grid.major.x = element_blank()
    )

print(p_comp_state)

ggsave(file.path(VAL_FIGURES_DIR, paste0(FILE_PREFIX, "_donor_comp_state.pdf")),
       p_comp_state, width = 8, height = 5)
ggsave(file.path(VAL_FIGURES_DIR, paste0(FILE_PREFIX, "_donor_comp_state.svg")),
       p_comp_state, width = 8, height = 5)

# ─────────────────────────────────────────────────────────────────────────────────
# 3I: Evenness bar plot
# ─────────────────────────────────────────────────────────────────────────────────

options(repr.plot.width = 10, repr.plot.height = 4)

cluster_comp_summary$Cluster <- factor(cluster_comp_summary$Cluster, levels = cluster_order)

p_evenness <- ggplot(cluster_comp_summary, aes(x = Cluster, y = Evenness, fill = State)) +
    geom_col(width = 0.7, alpha = 0.8) +
    geom_hline(yintercept = EVENNESS_FLAG_THRESHOLD, linetype = "dashed", color = "red", linewidth = 0.5) +
    scale_y_continuous(limits = c(0, 1), expand = c(0, 0)) +
    labs(x = "Cluster", y = "Shannon Evenness",
         title = paste0(toupper(cell_type_label), 
                        ": Donor evenness per cluster (red = ", EVENNESS_FLAG_THRESHOLD, " threshold)")) +
    theme_bw(base_size = 10) +
    theme(
        axis.text.x = element_text(size = 9),
        plot.title = element_text(size = 10, face = "bold"),
        legend.position = "bottom",
        legend.title = element_blank(),
        legend.text = element_text(size = 8)
    )

print(p_evenness)

ggsave(file.path(VAL_FIGURES_DIR, paste0(FILE_PREFIX, "_donor_evenness.pdf")),
       p_evenness, width = 10, height = 4)
ggsave(file.path(VAL_FIGURES_DIR, paste0(FILE_PREFIX, "_donor_evenness.svg")),
       p_evenness, width = 10, height = 4)

cat("  ✓ Saved composition plots\n")

# ─────────────────────────────────────────────────────────────────────────────────
# Step 3 Summary
# ─────────────────────────────────────────────────────────────────────────────────

cat("\n", paste(rep("-", 80), collapse = ""), "\n")
cat("STEP 3 SUMMARY\n")
cat(paste(rep("-", 80), collapse = ""), "\n")
cat("  Total donors:", n_donors, "\n")
cat("  Sample-dominated clusters (res", CLUSTERING_RESOLUTION, "):", 
    if(flagged_sample) paste(flagged_clusters, collapse = ", ") else "NONE", "\n")
cat("  Min evenness:", min(cluster_comp_summary$Evenness),
    "(cluster", cluster_comp_summary$Cluster[which.min(cluster_comp_summary$Evenness)], ")\n")
cat("  Max top-1 donor %:", max(cluster_comp_summary$Top1_pct),
    "(cluster", cluster_comp_summary$Cluster[which.max(cluster_comp_summary$Top1_pct)], ")\n")

# Multi-resolution donor stability summary
if (exists("flag_summary")) {
    always_flagged_res <- sum(flag_summary$N_flagged > 0)
    cat("  Multi-resolution: donor-dominated clusters present at",
        always_flagged_res, "/", nrow(flag_summary), "resolutions\n")
}
if (exists("donor_tracking") && nrow(donor_tracking) > 0) {
    for (cl in flagged_clusters) {
        cl_track <- donor_tracking[donor_tracking$Source_cluster == cl, ]
        n_struct <- sum(cl_track$Dest_flagged, na.rm = TRUE)
        cat(sprintf("    Cluster %s dominant donor: flagged at %d/%d resolutions",
                    cl, n_struct, nrow(cl_track)))
        if (n_struct == nrow(cl_track)) {
            cat(" → STRUCTURAL\n")
        } else if (n_struct > nrow(cl_track) / 2) {
            cat(" → MOSTLY STRUCTURAL\n")
        } else {
            cat(" → RESOLUTION-DEPENDENT\n")
        }
    }
}

if (exists("state_robustness")) {
    at_risk_states <- state_robustness$State[state_robustness$Donors_clean < 5 | 
                                              state_robustness$Pct_retained < 50 |
                                              state_robustness$Evenness < 0.5]
    if (length(at_risk_states) > 0) {
        cat("  States at risk:", paste(at_risk_states, collapse = ", "), "\n")
    } else {
        cat("  State robustness: All states retain multi-donor support\n")
    }
}
cat("  Resolutions computed:", paste(res_values, collapse = ", "), "\n")
cat("\n  Files saved:\n")
cat("    •", paste0(FILE_PREFIX, "_sample_comp_cluster.csv"), "\n")
cat("    •", paste0(FILE_PREFIX, "_sample_comp_state.csv"), "\n")
if (exists("state_robustness")) cat("    •", paste0(FILE_PREFIX, "_state_robustness.csv"), "\n")
cat("    •", paste0(FILE_PREFIX, "_state_resolution_stability.csv"), "\n")
cat("    •", paste0(FILE_PREFIX, "_multires_donor_comp.csv"), "\n")
cat("    •", paste0(FILE_PREFIX, "_multires_flag_summary.csv"), "\n")
if (exists("donor_tracking")) cat("    •", paste0(FILE_PREFIX, "_donor_tracking_across_res.csv"), "\n")
cat("    •", paste0(FILE_PREFIX, "_donor_comp_cluster.{pdf,svg}"), "\n")
cat("    •", paste0(FILE_PREFIX, "_donor_comp_state.{pdf,svg}"), "\n")
cat("    •", paste0(FILE_PREFIX, "_donor_evenness.{pdf,svg}"), "\n")
cat("    •", paste0(FILE_PREFIX, "_multires_flagged_count.{pdf,svg}"), "\n")
if (exists("donor_tracking")) cat("    •", paste0(FILE_PREFIX, "_donor_tracking_across_res.{pdf,svg}"), "\n")
cat(paste(rep("=", 80), collapse = ""), "\n")

In [ ]:
# Per-cluster sample composition
table_df <- as.data.frame.matrix(table(seurat_obj$seurat_clusters, seurat_obj$Sample))
prop_df <- table_df / rowSums(table_df)

# Top contributing sample per cluster
cat("Top sample per cluster:\n")
print(apply(prop_df, 1, function(x) names(which.max(x))))

cat("\nMax sample fraction per cluster:\n")
print(sort(apply(prop_df, 1, max), decreasing = TRUE))

# Also check metadata for the dominant samples
cat("\nSample cell counts:\n")
print(sort(table(seurat_obj$Sample), decreasing = TRUE))

In [ ]:
# Which clusters are flagged and by which donor?
cat("Flagged clusters:\n")
print(cluster_comp_summary[cluster_comp_summary$Cluster %in% flagged_clusters, 
      c("Cluster", "State", "N_cells", "Top_donor", "Top1_pct", "Evenness")])

# Pull metadata for the dominant donor(s)
dominant_donors <- unique(cluster_comp_summary$Top_donor[cluster_comp_summary$Cluster %in% flagged_clusters])
cat("\nDominant donor(s):", paste(dominant_donors, collapse = ", "), "\n")

# Check their metadata — adjust column names to match your Seurat object
meta_cols <- intersect(c("age", "Age", "sex", "Sex", "Study_Group", "diagnosis", 
                          "brain_region", "Braak", "braak", "PMI", "pmi",
                          "cohort", "Cohort", "batch", "Batch"), 
                       colnames(seurat_obj@meta.data))

cat("\nMetadata for dominant donor(s) vs all donors:\n")
for (d in dominant_donors) {
    donor_cells <- seurat_obj@meta.data[[DONOR_COL]] == d
    cat(sprintf("\n--- Donor: %s (%d cells) ---\n", d, sum(donor_cells)))
    for (col in meta_cols) {
        vals <- unique(seurat_obj@meta.data[[col]][donor_cells])
        cat(sprintf("  %s: %s\n", col, paste(vals, collapse = ", ")))
    }
}

# Compare: are these donors outliers in cell count?
cat("\nCell counts per donor (top 10):\n")
donor_counts <- sort(table(seurat_obj@meta.data[[DONOR_COL]]), decreasing = TRUE)
print(head(donor_counts, 10))

In [ ]:
donor_meta <- seurat_obj@meta.data[!duplicated(seurat_obj@meta.data[[DONOR_COL]]), ]

# Are there OTHER old donors (75+) who also contribute to these clusters?
old_donors <- donor_meta[[DONOR_COL]][as.numeric(donor_meta$Age) >= 75]
cat("Donors aged 75+:", length(old_donors), "\n\n")

# For each flagged cluster, check if other old donors are present
for (cl in flagged_clusters) {
    cl_cells <- seurat_obj$seurat_clusters == cl
    cl_donors <- table(seurat_obj@meta.data[[DONOR_COL]][cl_cells])
    cl_donors <- sort(cl_donors, decreasing = TRUE)
    
    cat(sprintf("Cluster %s (%s) — all donors:\n", cl, 
                cluster_comp_summary$State[cluster_comp_summary$Cluster == cl]))
    
    for (d in names(cl_donors)) {
        age <- unique(seurat_obj@meta.data$Age[seurat_obj@meta.data[[DONOR_COL]] == d])
        cohort <- unique(seurat_obj@meta.data$Cohort[seurat_obj@meta.data[[DONOR_COL]] == d])
        cat(sprintf("  %s: %d cells (age=%s, %s)\n", d, cl_donors[d], age, cohort))
    }
    cat("\n")
}

In [ ]:
# How much of each state comes from flagged clusters?
for (st in c("Intermediate", "Reactive", "Trophic")) {
    total <- sum(seurat_obj@meta.data[[SUBCLUSTER_COL]] == st)
    in_flagged <- sum(seurat_obj@meta.data[[SUBCLUSTER_COL]] == st & 
                      seurat_obj$seurat_clusters %in% flagged_clusters)
    cat(sprintf("%s: %d/%d cells (%.1f%%) in flagged clusters\n", 
                st, in_flagged, total, in_flagged/total*100))
}

In [ ]:
# Impact of removing all 4 donors
all_donors_to_remove <- c("AMPAD_MSSM_0000035312", "AMPAD_MSSM_0000066132", 
                           "AMPAD_MSSM_0000006362", "AMPAD_HBCC_0000000126")

cells_to_remove <- seurat_obj@meta.data[[DONOR_COL]] %in% all_donors_to_remove

cat(sprintf("Total cells to remove: %d (%.1f%%)\n", 
            sum(cells_to_remove), sum(cells_to_remove)/ncol(seurat_obj)*100))

for (st in states) {
    in_state <- seurat_obj@meta.data[[SUBCLUSTER_COL]] == st
    lost <- sum(in_state & cells_to_remove)
    total <- sum(in_state)
    cat(sprintf("  %s: loses %d/%d (%.1f%%)\n", st, lost, total, lost/total*100))
}

In [ ]:
# Impact of removing the 3 donors entirely
donors_to_remove <- c("AMPAD_MSSM_0000035312", "AMPAD_MSSM_0000066132", "AMPAD_MSSM_0000006362", "AMPAD_HBCC_0000000126")

n_before <- ncol(seurat_obj)
cells_to_remove <- seurat_obj@meta.data[[DONOR_COL]] %in% donors_to_remove
n_remove <- sum(cells_to_remove)

cat(sprintf("Removing %d cells from %d donors (%.1f%% of total)\n", 
            n_remove, length(donors_to_remove), n_remove/n_before*100))

# How does this affect each state?
state_impact <- do.call(rbind, lapply(states, function(st) {
    in_state <- seurat_obj@meta.data[[SUBCLUSTER_COL]] == st
    lost <- sum(in_state & cells_to_remove)
    total <- sum(in_state)
    data.frame(State = st, Total = total, Lost = lost, 
               Pct_lost = round(lost/total*100, 1))
}))
print(state_impact)

In [ ]:
# Remove the 3 donors
donors_to_remove <- c("AMPAD_MSSM_0000035312", "AMPAD_MSSM_0000066132", "AMPAD_MSSM_0000006362", "AMPAD_HBCC_0000000126")
seurat_obj <- subset(seurat_obj, cells = colnames(seurat_obj)[!seurat_obj@meta.data[[DONOR_COL]] %in% donors_to_remove])

cat(sprintf("After removal: %d cells, %d donors\n", 
            ncol(seurat_obj), length(unique(seurat_obj@meta.data[[DONOR_COL]]))))

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# STEP 3: SAMPLE COMPOSITION
# ════════════════════════════════════════════════════════════════════════════════

cat("\n", paste(rep("=", 80), collapse = ""), "\n")
cat("STEP 3: SAMPLE COMPOSITION\n")
cat(paste(rep("=", 80), collapse = ""), "\n")

# ─────────────────────────────────────────────────────────────────────────────────
# Thresholds
# ─────────────────────────────────────────────────────────────────────────────────

DONOR_DOMINANCE_THRESHOLD <- 60
EVENNESS_FLAG_THRESHOLD <- 0.40
FEW_DONORS_THRESHOLD <- 3

# ─────────────────────────────────────────────────────────────────────────────────
# 3A: Per-cluster donor composition
# ─────────────────────────────────────────────────────────────────────────────────

cat("\n▸ Analyzing donor composition per cluster...\n")

comp_cluster <- table(seurat_obj$seurat_clusters, seurat_obj@meta.data[[DONOR_COL]])

cluster_comp_summary <- do.call(rbind, lapply(cluster_order, function(cl) {
    cl_counts <- comp_cluster[cl, ]
    cl_counts <- cl_counts[cl_counts > 0]
    n_donors_cl <- length(cl_counts)
    total <- sum(cl_counts)
    sorted_counts <- sort(cl_counts, decreasing = TRUE)
    top1_pct <- sorted_counts[1] / total * 100
    top3_pct <- sum(sorted_counts[1:min(3, length(sorted_counts))]) / total * 100
    top_donor <- names(sorted_counts)[1]
    
    props <- cl_counts / total
    shannon <- -sum(props * log(props))
    max_shannon <- log(n_donors_cl)
    evenness <- if (max_shannon > 0) shannon / max_shannon else 0
    
    data.frame(
        Cluster = cl,
        State = qc_by_cluster$State[qc_by_cluster$Cluster == cl],
        N_cells = total,
        N_donors = n_donors_cl,
        Top_donor = top_donor,
        Top1_pct = round(top1_pct, 1),
        Top3_pct = round(top3_pct, 1),
        Shannon = round(shannon, 2),
        Evenness = round(evenness, 2),
        stringsAsFactors = FALSE
    )
}))

cat("\n  Cluster composition summary:\n")
print(cluster_comp_summary, row.names = FALSE)

write.csv(cluster_comp_summary,
          file.path(VAL_RESULTS_DIR, paste0(FILE_PREFIX, "_sample_comp_cluster.csv")),
          row.names = FALSE)

# ─────────────────────────────────────────────────────────────────────────────────
# 3B: Per-state donor composition
# ─────────────────────────────────────────────────────────────────────────────────

cat("\n▸ Analyzing donor composition per state...\n")

comp_state <- table(seurat_obj@meta.data[[SUBCLUSTER_COL]], seurat_obj@meta.data[[DONOR_COL]])

state_comp_summary <- do.call(rbind, lapply(states, function(s) {
    s_counts <- comp_state[s, ]
    s_counts <- s_counts[s_counts > 0]
    n_donors_s <- length(s_counts)
    total <- sum(s_counts)
    sorted_counts <- sort(s_counts, decreasing = TRUE)
    top1_pct <- sorted_counts[1] / total * 100
    top3_pct <- sum(sorted_counts[1:min(3, length(sorted_counts))]) / total * 100
    
    props <- s_counts / total
    shannon <- -sum(props * log(props))
    max_shannon <- log(n_donors_s)
    evenness <- if (max_shannon > 0) shannon / max_shannon else 0
    
    data.frame(
        State = s,
        N_cells = total,
        N_donors = n_donors_s,
        Top1_pct = round(top1_pct, 1),
        Top3_pct = round(top3_pct, 1),
        Shannon = round(shannon, 2),
        Evenness = round(evenness, 2),
        stringsAsFactors = FALSE
    )
}))

cat("\n  State composition summary:\n")
print(state_comp_summary, row.names = FALSE)

write.csv(state_comp_summary,
          file.path(VAL_RESULTS_DIR, paste0(FILE_PREFIX, "_sample_comp_state.csv")),
          row.names = FALSE)

# ─────────────────────────────────────────────────────────────────────────────────
# 3C: Flag sample-dominated clusters
# ─────────────────────────────────────────────────────────────────────────────────

cat("\n▸ Flagging sample-dominated clusters:\n")
cat(sprintf("  Thresholds: top donor >%d%%, evenness <%.2f, donors <=%d\n",
            DONOR_DOMINANCE_THRESHOLD, EVENNESS_FLAG_THRESHOLD, FEW_DONORS_THRESHOLD))

flagged_sample <- FALSE
flagged_clusters <- c()

for (i in 1:nrow(cluster_comp_summary)) {
    flags_list <- c()
    
    if (cluster_comp_summary$Top1_pct[i] > DONOR_DOMINANCE_THRESHOLD) {
        flags_list <- c(flags_list, sprintf("TOP DONOR >%d%% (%s = %.1f%%)",
                                   DONOR_DOMINANCE_THRESHOLD,
                                   cluster_comp_summary$Top_donor[i],
                                   cluster_comp_summary$Top1_pct[i]))
    }
    if (cluster_comp_summary$N_donors[i] <= FEW_DONORS_THRESHOLD) {
        flags_list <- c(flags_list, sprintf("FEW DONORS (n=%d)", cluster_comp_summary$N_donors[i]))
    }
    if (cluster_comp_summary$Evenness[i] < EVENNESS_FLAG_THRESHOLD) {
        flags_list <- c(flags_list, sprintf("LOW EVENNESS (%.2f)", cluster_comp_summary$Evenness[i]))
    }
    
    if (length(flags_list) > 0) {
        cat(sprintf("  ⚠ Cluster %s (%s, n=%d): %s\n",
                    cluster_comp_summary$Cluster[i],
                    cluster_comp_summary$State[i],
                    cluster_comp_summary$N_cells[i],
                    paste(flags_list, collapse = "; ")))
        flagged_sample <- TRUE
        flagged_clusters <- c(flagged_clusters, as.character(cluster_comp_summary$Cluster[i]))
    }
}
if (!flagged_sample) cat("  ✓ No sample-dominated clusters\n")

# ─────────────────────────────────────────────────────────────────────────────────
# 3D: State robustness after excluding flagged clusters
# ─────────────────────────────────────────────────────────────────────────────────

cat("\n▸ State robustness check (excluding flagged clusters)...\n")
cat("  Goal: Verify each state retains multi-donor support after removing\n")
cat("  sample-dominated clusters. If a state is defined primarily by flagged\n")
cat("  clusters, it may be a donor artifact rather than real biology.\n")

if (length(flagged_clusters) > 0) {
    flagged_cells <- as.character(seurat_obj$seurat_clusters) %in% flagged_clusters
    clean_meta <- seurat_obj@meta.data[!flagged_cells, ]
    
    state_robustness <- do.call(rbind, lapply(state_order, function(st) {
        all_cells <- seurat_obj@meta.data[[SUBCLUSTER_COL]] == st
        n_total <- sum(all_cells)
        n_donors_total <- length(unique(seurat_obj@meta.data[[DONOR_COL]][all_cells]))
        n_clusters_total <- length(unique(as.character(seurat_obj$seurat_clusters[all_cells])))
        
        clean_cells <- clean_meta[[SUBCLUSTER_COL]] == st
        n_clean <- sum(clean_cells)
        n_donors_clean <- length(unique(clean_meta[[DONOR_COL]][clean_cells]))
        n_clusters_clean <- length(unique(as.character(clean_meta$seurat_clusters[clean_cells])))
        
        pct_retained <- round(n_clean / n_total * 100, 1)
        
        donor_table <- table(clean_meta[[DONOR_COL]][clean_cells])
        donor_props <- as.numeric(donor_table) / sum(donor_table)
        evenness <- if (length(donor_props) > 1) {
            -sum(donor_props * log(donor_props)) / log(length(donor_props))
        } else { 0 }
        
        top_donor_pct <- round(max(donor_table) / sum(donor_table) * 100, 1)
        
        data.frame(
            State = st,
            Cells_total = n_total,
            Cells_clean = n_clean,
            Pct_retained = pct_retained,
            Donors_total = n_donors_total,
            Donors_clean = n_donors_clean,
            Clusters_total = n_clusters_total,
            Clusters_clean = n_clusters_clean,
            Top_donor_pct = top_donor_pct,
            Evenness = round(evenness, 2),
            stringsAsFactors = FALSE
        )
    }))
    
    cat("\n")
    print(state_robustness, row.names = FALSE)
    
    at_risk <- state_robustness[state_robustness$Donors_clean < 5 | 
                                 state_robustness$Pct_retained < 50 |
                                 state_robustness$Evenness < 0.5, ]
    
    if (nrow(at_risk) > 0) {
        cat("\n  ⚠ States at risk after removing flagged clusters:\n")
        for (i in 1:nrow(at_risk)) {
            reasons <- c()
            if (at_risk$Donors_clean[i] < 5) reasons <- c(reasons, 
                sprintf("only %d donors remain", at_risk$Donors_clean[i]))
            if (at_risk$Pct_retained[i] < 50) reasons <- c(reasons, 
                sprintf("only %.1f%% cells retained", at_risk$Pct_retained[i]))
            if (at_risk$Evenness[i] < 0.5) reasons <- c(reasons, 
                sprintf("low evenness (%.2f)", at_risk$Evenness[i]))
            cat(sprintf("    %s: %s\n", at_risk$State[i], paste(reasons, collapse = ", ")))
        }
    } else {
        cat("\n  ✓ All states retain multi-donor support after removing flagged clusters\n")
    }
    
    write.csv(state_robustness,
              file.path(VAL_RESULTS_DIR, paste0(FILE_PREFIX, "_state_robustness.csv")),
              row.names = FALSE)
} else {
    cat("  No flagged clusters -- skipping robustness check\n")
}

# ─────────────────────────────────────────────────────────────────────────────────
# 3E: Multi-resolution clustering + State stability across resolutions
# ─────────────────────────────────────────────────────────────────────────────────

cat("\n▸ State stability across resolutions...\n")
cat("  Goal: Verify each state emerges as a distinct cluster (or set of clusters)\n")
cat("  at multiple resolutions. A real state should be recoverable across\n")
cat("  resolutions, not only at the chosen one.\n")

# ── Ensure multi-resolution clustering exists ──
res_seq <- c(0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 1.0, 1.2)
existing_res <- grep("RNA_snn_res\\.", colnames(seurat_obj@meta.data), value = TRUE)
existing_values <- as.numeric(gsub("RNA_snn_res\\.", "", existing_res))
missing_res <- setdiff(res_seq, existing_values)

if (length(missing_res) > 0) {
    cat("  Computing clustering at", length(missing_res), "additional resolutions...\n")
    
    # Preserve original clustering
    seurat_obj$original_clusters <- seurat_obj$seurat_clusters
    
    # Determine reduction and dims
    cluster_reduction <- ifelse("harmony" %in% Reductions(seurat_obj), "harmony", "pca")
    if (!exists("N_DIMS_USE")) {
        N_DIMS_USE <- min(ncol(Embeddings(seurat_obj, cluster_reduction)), 30)
        cat("  N_DIMS_USE auto-detected:", N_DIMS_USE, "\n")
    }
    cat("  Using reduction:", cluster_reduction, ", dims: 1:", N_DIMS_USE, "\n")
    
    # Rebuild neighbors if needed
    if (length(seurat_obj@graphs) == 0) {
        cat("  Building neighbor graph...\n")
        seurat_obj <- FindNeighbors(seurat_obj, reduction = cluster_reduction,
                                     dims = 1:N_DIMS_USE, verbose = FALSE)
    }
    
    for (res in sort(missing_res)) {
        seurat_obj <- FindClusters(seurat_obj, resolution = res, verbose = FALSE)
        col_name <- paste0("RNA_snn_res.", res)
        n_cl <- length(unique(seurat_obj@meta.data[[col_name]]))
        cat(sprintf("    res = %.1f -> %d clusters\n", res, n_cl))
    }
    
    # Restore original clustering
    seurat_obj$seurat_clusters <- seurat_obj$original_clusters
    Idents(seurat_obj) <- seurat_obj$seurat_clusters
    seurat_obj$original_clusters <- NULL
    cat("  ✓ Restored original seurat_clusters (",
        length(unique(seurat_obj$seurat_clusters)), " clusters)\n", sep = "")
} else {
    cat("  All", length(res_seq), "resolutions already computed\n")
}

# Print resolution summary
available_res <- grep("RNA_snn_res\\.", colnames(seurat_obj@meta.data), value = TRUE)
res_values <- sort(as.numeric(gsub("RNA_snn_res\\.", "", available_res)))
cat("  Available resolutions:", paste(res_values, collapse = ", "), "\n\n")

for (res in res_seq) {
    col_name <- paste0("RNA_snn_res.", res)
    if (col_name %in% colnames(seurat_obj@meta.data)) {
        n_cl <- length(unique(seurat_obj@meta.data[[col_name]]))
        marker <- ifelse(res == CLUSTERING_RESOLUTION, " <- CURRENT", "")
        cat(sprintf("    res = %.1f -> %d clusters%s\n", res, n_cl, marker))
    }
}

# ── Compute state recovery per resolution ──
cat("\n  Computing state recovery per resolution...\n")

state_by_res <- do.call(rbind, lapply(res_values, function(res) {
    col_name <- paste0("RNA_snn_res.", res)
    
    do.call(rbind, lapply(state_order, function(st) {
        state_cells <- seurat_obj@meta.data[[SUBCLUSTER_COL]] == st
        
        majority_clusters <- c()
        for (cl in unique(seurat_obj@meta.data[[col_name]])) {
            cl_cells <- seurat_obj@meta.data[[col_name]] == cl
            cl_states <- table(seurat_obj@meta.data[[SUBCLUSTER_COL]][cl_cells])
            if (names(which.max(cl_states)) == st) {
                majority_clusters <- c(majority_clusters, cl)
            }
        }
        
        n_majority <- length(majority_clusters)
        in_majority <- sum(seurat_obj@meta.data[[col_name]][state_cells] %in% majority_clusters)
        recovery_pct <- round(in_majority / sum(state_cells) * 100, 1)
        
        data.frame(
            Resolution = res,
            State = st,
            N_majority_clusters = n_majority,
            Recovery_pct = recovery_pct,
            stringsAsFactors = FALSE
        )
    }))
}))

recovery_wide <- reshape(state_by_res[, c("Resolution", "State", "Recovery_pct")],
                          idvar = "Resolution", timevar = "State",
                          direction = "wide")
colnames(recovery_wide) <- gsub("Recovery_pct\\.", "", colnames(recovery_wide))
cat("\n  Recovery % per state across resolutions:\n")
print(recovery_wide, row.names = FALSE)

# ── Interpret per state ──
cat("\n")
for (st in state_order) {
    st_data <- state_by_res[state_by_res$State == st, ]
    min_rec <- min(st_data$Recovery_pct)
    low_recovery <- st_data[st_data$Recovery_pct < 50, ]
    
    if (nrow(low_recovery) > 0) {
        low_res <- low_recovery$Resolution
        chosen_rec <- st_data$Recovery_pct[st_data$Resolution == CLUSTERING_RESOLUTION]
        if (length(chosen_rec) == 0) chosen_rec <- NA
        
        all_below_chosen <- all(low_res < CLUSTERING_RESOLUTION)
        at_and_above <- st_data[st_data$Resolution >= CLUSTERING_RESOLUTION, ]
        stable_above <- all(at_and_above$Recovery_pct >= 70)
        
        if (all_below_chosen && stable_above && !is.na(chosen_rec) && chosen_rec >= 70) {
            first_stable <- min(st_data$Resolution[st_data$Recovery_pct >= 50])
            n_state_cells <- sum(seurat_obj@meta.data[[SUBCLUSTER_COL]] == st)
            size_note <- ifelse(n_state_cells < 2000, "a small state", "this state")
            cat(sprintf("  ✓ %s: stable at working resolutions (min %.1f%% at coarse res %s)\n",
                        st, min_rec, paste(low_res, collapse = ", ")))
            cat(sprintf("    Emerges from res %.1f onward — expected for %s (%d cells)\n",
                        first_stable, size_note, n_state_cells))
        } else if (!is.na(chosen_rec) && chosen_rec >= 70) {
            cat(sprintf("  ~ %s: partially stable -- <50%% at res %s, but %.1f%% at chosen res\n",
                        st, paste(low_res, collapse = ", "), chosen_rec))
            if (all_below_chosen) {
                cat("    Absorbed into neighboring states at coarse resolutions\n")
            } else {
                cat("    Variable recovery suggests resolution-sensitive boundaries\n")
            }
        } else {
            cat(sprintf("  ⚠ %s: UNSTABLE -- %.1f%% at chosen res, <50%% at res %s\n",
                        st, ifelse(is.na(chosen_rec), NA, chosen_rec),
                        paste(low_res, collapse = ", ")))
            cat("    This state may not be well-supported at the current resolution\n")
        }
    } else {
        cat(sprintf("  ✓ %s: stable across all resolutions (min recovery: %.1f%%)\n",
                    st, min_rec))
    }
}

write.csv(state_by_res,
          file.path(VAL_RESULTS_DIR, paste0(FILE_PREFIX, "_state_resolution_stability.csv")),
          row.names = FALSE)

# ─────────────────────────────────────────────────────────────────────────────────
# 3F: DONOR COMPOSITION STABILITY ACROSS RESOLUTIONS
# ─────────────────────────────────────────────────────────────────────────────────
# Goal: For each resolution, find every cluster and compute its donor composition.
# Then check: do donor-dominated clusters persistently appear across resolutions,
# or do those cells merge into well-mixed clusters at other resolutions?
# This answers whether the sample dominance is a stable structural feature
# (those cells always segregate) or a resolution artifact (they only split off
# at one resolution).
# ─────────────────────────────────────────────────────────────────────────────────

cat("\n▸ Donor composition stability across resolutions...\n")
cat("  Goal: Track whether donor-dominated clusters persist, resolve, or worsen\n")
cat("  across resolutions. If flagged cells always form their own donor-dominated\n")
cat("  cluster, the dominance is structural. If they merge into well-mixed clusters\n")
cat("  at other resolutions, it is a resolution artifact.\n\n")

# ── 3F-i: Per-cluster donor composition at every resolution ──

multires_donor_comp <- do.call(rbind, lapply(res_values, function(res) {
    col_name <- paste0("RNA_snn_res.", res)
    clusters_at_res <- sort(unique(seurat_obj@meta.data[[col_name]]))
    
    do.call(rbind, lapply(clusters_at_res, function(cl) {
        cl_cells <- seurat_obj@meta.data[[col_name]] == cl
        donor_counts <- table(seurat_obj@meta.data[[DONOR_COL]][cl_cells])
        donor_counts <- donor_counts[donor_counts > 0]
        
        n_cells <- sum(cl_cells)
        n_donors <- length(donor_counts)
        sorted <- sort(donor_counts, decreasing = TRUE)
        top1_pct <- sorted[1] / n_cells * 100
        top_donor <- names(sorted)[1]
        
        props <- donor_counts / n_cells
        shannon <- -sum(props * log(props))
        max_shannon <- log(n_donors)
        evenness <- if (max_shannon > 0) shannon / max_shannon else 0
        
        # Majority annotated state for this cluster
        state_table <- table(seurat_obj@meta.data[[SUBCLUSTER_COL]][cl_cells])
        majority_state <- names(which.max(state_table))
        state_purity <- round(max(state_table) / n_cells * 100, 1)
        
        data.frame(
            Resolution = res,
            Cluster = as.character(cl),
            N_cells = n_cells,
            Majority_state = majority_state,
            State_purity = state_purity,
            N_donors = n_donors,
            Top_donor = top_donor,
            Top1_pct = round(top1_pct, 1),
            Evenness = round(evenness, 2),
            Flagged = (top1_pct > DONOR_DOMINANCE_THRESHOLD | evenness < EVENNESS_FLAG_THRESHOLD),
            stringsAsFactors = FALSE
        )
    }))
}))

# Save full table
write.csv(multires_donor_comp,
          file.path(VAL_RESULTS_DIR, paste0(FILE_PREFIX, "_multires_donor_comp.csv")),
          row.names = FALSE)

# ── 3F-ii: Summary — how many flagged clusters per resolution? ──

cat("  Flagged clusters per resolution:\n")
flag_summary <- do.call(rbind, lapply(res_values, function(res) {
    res_data <- multires_donor_comp[multires_donor_comp$Resolution == res, ]
    n_total <- nrow(res_data)
    n_flagged <- sum(res_data$Flagged)
    n_cells_flagged <- sum(res_data$N_cells[res_data$Flagged])
    pct_cells_flagged <- round(n_cells_flagged / ncol(seurat_obj) * 100, 1)
    
    data.frame(
        Resolution = res,
        N_clusters = n_total,
        N_flagged = n_flagged,
        Cells_in_flagged = n_cells_flagged,
        Pct_cells_flagged = pct_cells_flagged,
        stringsAsFactors = FALSE
    )
}))
print(flag_summary, row.names = FALSE)

write.csv(flag_summary,
          file.path(VAL_RESULTS_DIR, paste0(FILE_PREFIX, "_multires_flag_summary.csv")),
          row.names = FALSE)

# ── 3F-iii: Track flagged donors' cells across resolutions ──
# For each donor that dominated a cluster at the current resolution,
# trace where those cells end up at every other resolution.

if (length(flagged_clusters) > 0) {
    cat("\n  Tracking flagged donors' cells across resolutions...\n")
    
    # Identify the dominant donor + their cells for each flagged cluster
    flagged_donor_info <- lapply(flagged_clusters, function(cl) {
        cl_data <- cluster_comp_summary[cluster_comp_summary$Cluster == cl, ]
        dominant_donor <- cl_data$Top_donor
        # Get cell barcodes from this donor that were in this cluster
        in_cluster <- as.character(seurat_obj$seurat_clusters) == cl
        from_donor <- seurat_obj@meta.data[[DONOR_COL]] == dominant_donor
        cell_barcodes <- colnames(seurat_obj)[in_cluster & from_donor]
        
        list(
            cluster = cl,
            state = cl_data$State,
            donor = dominant_donor,
            n_cells = length(cell_barcodes),
            barcodes = cell_barcodes
        )
    })
    names(flagged_donor_info) <- flagged_clusters
    
    # For each flagged cluster's dominant-donor cells, check where they land
    # at every resolution
    donor_tracking <- do.call(rbind, lapply(flagged_donor_info, function(info) {
        cell_idx <- match(info$barcodes, colnames(seurat_obj))
        
        do.call(rbind, lapply(res_values, function(res) {
            col_name <- paste0("RNA_snn_res.", res)
            assignments <- seurat_obj@meta.data[[col_name]][cell_idx]
            
            # Where do these cells go?
            dest_table <- sort(table(assignments), decreasing = TRUE)
            n_dest_clusters <- length(dest_table)
            top_dest <- names(dest_table)[1]
            top_dest_pct <- round(dest_table[1] / length(assignments) * 100, 1)
            
            # In the cluster they land in, what fraction are they?
            top_dest_all_cells <- seurat_obj@meta.data[[col_name]] == top_dest
            n_top_dest_total <- sum(top_dest_all_cells)
            pct_of_dest <- round(dest_table[1] / n_top_dest_total * 100, 1)
            
            # Is that destination cluster also flagged?
            dest_data <- multires_donor_comp[multires_donor_comp$Resolution == res &
                                              multires_donor_comp$Cluster == top_dest, ]
            dest_flagged <- if (nrow(dest_data) > 0) dest_data$Flagged[1] else NA
            dest_evenness <- if (nrow(dest_data) > 0) dest_data$Evenness[1] else NA
            
            data.frame(
                Source_cluster = info$cluster,
                Source_state = info$state,
                Dominant_donor = info$donor,
                N_tracked_cells = length(info$barcodes),
                Resolution = res,
                N_dest_clusters = n_dest_clusters,
                Top_destination = top_dest,
                Pct_cells_in_top_dest = top_dest_pct,
                Pct_of_dest_cluster = pct_of_dest,
                Dest_evenness = dest_evenness,
                Dest_flagged = dest_flagged,
                stringsAsFactors = FALSE
            )
        }))
    }))
    
    # Print tracking results per flagged cluster
    for (cl in flagged_clusters) {
        cl_track <- donor_tracking[donor_tracking$Source_cluster == cl, ]
        info <- flagged_donor_info[[cl]]
        
        cat(sprintf("\n  ── Cluster %s (%s) — dominant donor: %s (%d cells) ──\n",
                    cl, info$state, info$donor, info$n_cells))
        
        print_cols <- c("Resolution", "N_dest_clusters", "Top_destination",
                        "Pct_cells_in_top_dest", "Pct_of_dest_cluster",
                        "Dest_evenness", "Dest_flagged")
        print(cl_track[, print_cols], row.names = FALSE)
        
        # Interpret: do these cells always segregate?
        always_dominant <- all(cl_track$Pct_of_dest_cluster > DONOR_DOMINANCE_THRESHOLD)
        always_together <- all(cl_track$Pct_cells_in_top_dest > 80)
        never_flagged_elsewhere <- all(!cl_track$Dest_flagged, na.rm = TRUE)
        
        if (always_dominant & always_together) {
            cat(sprintf("    → STRUCTURAL: These %d cells always form a donor-dominated cluster.\n",
                        info$n_cells))
            cat("      The dominance persists across all resolutions — not a resolution artifact.\n")
            cat("      Recommendation: REMOVE or treat as donor-specific.\n")
        } else if (never_flagged_elsewhere) {
            cat("    → RESOLUTION ARTIFACT: These cells merge into well-mixed clusters\n")
            cat("      at other resolutions. The dominance is specific to the current resolution.\n")
            cat("      Recommendation: Likely safe to KEEP (dominance dilutes at other resolutions).\n")
        } else {
            cat("    → MIXED: Dominance varies across resolutions.\n")
            # Count how many resolutions show flagged destination
            n_flagged_res <- sum(cl_track$Dest_flagged, na.rm = TRUE)
            cat(sprintf("      Flagged at %d/%d resolutions.\n",
                        n_flagged_res, nrow(cl_track)))
            if (n_flagged_res > nrow(cl_track) / 2) {
                cat("      Recommendation: Lean toward REMOVE (dominant at most resolutions).\n")
            } else {
                cat("      Recommendation: MONITOR — verify DEG results aren't donor-driven.\n")
            }
        }
    }
    
    write.csv(donor_tracking,
              file.path(VAL_RESULTS_DIR, paste0(FILE_PREFIX, "_donor_tracking_across_res.csv")),
              row.names = FALSE)
    
} else {
    cat("  No flagged clusters — skipping donor tracking.\n")
}

# ── 3F-iv: Visualization — Donor dominance heatmap across resolutions ──

cat("\n▸ Generating multi-resolution donor composition plots...\n")

# Heatmap: top donor % per cluster per resolution
# Only show resolutions with a manageable number of clusters

library(ggplot2)

# Plot 1: Number of flagged clusters per resolution
options(repr.plot.width = 10, repr.plot.height = 4)

p_flags_per_res <- ggplot(flag_summary, aes(x = factor(Resolution), y = N_flagged)) +
    geom_col(fill = "#E15759", alpha = 0.8, width = 0.6) +
    geom_text(aes(label = N_flagged), vjust = -0.3, size = 3) +
    geom_hline(yintercept = 0, linewidth = 0.3) +
    scale_y_continuous(expand = expansion(mult = c(0, 0.15))) +
    labs(x = "Resolution", y = "N flagged clusters",
         title = paste0(toupper(cell_type_label),
                        ": Donor-dominated clusters across resolutions"),
         subtitle = sprintf("Flagged = top donor >%d%% OR evenness <%.2f",
                           DONOR_DOMINANCE_THRESHOLD, EVENNESS_FLAG_THRESHOLD)) +
    theme_bw(base_size = 10) +
    theme(
        plot.title = element_text(size = 11, face = "bold"),
        plot.subtitle = element_text(size = 9, color = "grey40"),
        panel.grid.major.x = element_blank()
    )

print(p_flags_per_res)

ggsave(file.path(VAL_FIGURES_DIR, paste0(FILE_PREFIX, "_multires_flagged_count.pdf")),
       p_flags_per_res, width = 10, height = 4)
ggsave(file.path(VAL_FIGURES_DIR, paste0(FILE_PREFIX, "_multires_flagged_count.svg")),
       p_flags_per_res, width = 10, height = 4)

# Plot 2: Tracking plot — for each flagged cluster, show the % of destination
# cluster that is the dominant donor across resolutions

if (exists("donor_tracking") && nrow(donor_tracking) > 0) {
    options(repr.plot.width = 12, repr.plot.height = 4 + length(flagged_clusters) * 1.5)
    
    donor_tracking$label <- paste0("Cl.", donor_tracking$Source_cluster,
                                    " (", donor_tracking$Source_state, ")")
    
    p_tracking <- ggplot(donor_tracking,
                          aes(x = factor(Resolution), y = Pct_of_dest_cluster)) +
        geom_line(aes(group = label), color = "#4E79A7", linewidth = 0.8) +
        geom_point(aes(color = Dest_flagged), size = 2.5) +
        geom_hline(yintercept = DONOR_DOMINANCE_THRESHOLD,
                   linetype = "dashed", color = "red", linewidth = 0.4) +
        scale_color_manual(values = c("TRUE" = "#E15759", "FALSE" = "#59A14F"),
                          labels = c("TRUE" = "Flagged", "FALSE" = "Not flagged"),
                          name = "Destination cluster") +
        facet_wrap(~label, ncol = 1, scales = "free_y") +
        labs(x = "Resolution",
             y = "Dominant donor's % of destination cluster",
             title = paste0(toupper(cell_type_label),
                            ": Donor dominance tracking across resolutions"),
             subtitle = "Tracks where the dominant donor's cells land at each resolution") +
        theme_bw(base_size = 10) +
        theme(
            plot.title = element_text(size = 11, face = "bold"),
            plot.subtitle = element_text(size = 9, color = "grey40"),
            strip.text = element_text(size = 9, face = "bold"),
            legend.position = "bottom"
        )
    
    print(p_tracking)
    
    ggsave(file.path(VAL_FIGURES_DIR, paste0(FILE_PREFIX, "_donor_tracking_across_res.pdf")),
           p_tracking, width = 12, height = 4 + length(flagged_clusters) * 1.5)
    ggsave(file.path(VAL_FIGURES_DIR, paste0(FILE_PREFIX, "_donor_tracking_across_res.svg")),
           p_tracking, width = 12, height = 4 + length(flagged_clusters) * 1.5)
}

cat("  ✓ Saved multi-resolution donor composition plots\n")

# ─────────────────────────────────────────────────────────────────────────────────
# 3G: Stacked bar plot — by cluster
# ─────────────────────────────────────────────────────────────────────────────────

cat("\n▸ Generating composition plots...\n")

comp_df <- as.data.frame(comp_cluster)
colnames(comp_df) <- c("Cluster", "Donor", "Count")
comp_df <- comp_df[comp_df$Count > 0, ]
comp_df$Cluster <- factor(comp_df$Cluster, levels = cluster_order)

n_unique_donors <- length(unique(comp_df$Donor))

options(repr.plot.width = 12, repr.plot.height = 5)

p_comp_cluster <- ggplot(comp_df, aes(x = Cluster, y = Count, fill = Donor)) +
    geom_bar(stat = "identity", position = "fill", width = 0.8) +
    scale_y_continuous(labels = scales::percent, expand = c(0, 0)) +
    labs(x = "Cluster", y = "Proportion",
         title = paste0(toupper(cell_type_label), ": Donor composition per cluster")) +
    theme_bw(base_size = 10) +
    theme(
        legend.position = "none",
        axis.text.x = element_text(size = 9),
        plot.title = element_text(size = 11, face = "bold"),
        panel.grid.major.x = element_blank()
    )

print(p_comp_cluster)

ggsave(file.path(VAL_FIGURES_DIR, paste0(FILE_PREFIX, "_donor_comp_cluster.pdf")),
       p_comp_cluster, width = 12, height = 5)
ggsave(file.path(VAL_FIGURES_DIR, paste0(FILE_PREFIX, "_donor_comp_cluster.svg")),
       p_comp_cluster, width = 12, height = 5)

# ─────────────────────────────────────────────────────────────────────────────────
# 3H: Stacked bar plot — by state
# ─────────────────────────────────────────────────────────────────────────────────

comp_state_df <- as.data.frame(comp_state)
colnames(comp_state_df) <- c("State", "Donor", "Count")
comp_state_df <- comp_state_df[comp_state_df$Count > 0, ]
comp_state_df$State <- factor(comp_state_df$State, levels = states)

p_comp_state <- ggplot(comp_state_df, aes(x = State, y = Count, fill = Donor)) +
    geom_bar(stat = "identity", position = "fill", width = 0.8) +
    scale_y_continuous(labels = scales::percent, expand = c(0, 0)) +
    labs(x = "State", y = "Proportion",
         title = paste0(toupper(cell_type_label), ": Donor composition per state")) +
    theme_bw(base_size = 10) +
    theme(
        legend.position = "none",
        axis.text.x = element_text(angle = 30, hjust = 1, size = 9),
        plot.title = element_text(size = 11, face = "bold"),
        panel.grid.major.x = element_blank()
    )

print(p_comp_state)

ggsave(file.path(VAL_FIGURES_DIR, paste0(FILE_PREFIX, "_donor_comp_state.pdf")),
       p_comp_state, width = 8, height = 5)
ggsave(file.path(VAL_FIGURES_DIR, paste0(FILE_PREFIX, "_donor_comp_state.svg")),
       p_comp_state, width = 8, height = 5)

# ─────────────────────────────────────────────────────────────────────────────────
# 3I: Evenness bar plot
# ─────────────────────────────────────────────────────────────────────────────────

options(repr.plot.width = 10, repr.plot.height = 4)

cluster_comp_summary$Cluster <- factor(cluster_comp_summary$Cluster, levels = cluster_order)

p_evenness <- ggplot(cluster_comp_summary, aes(x = Cluster, y = Evenness, fill = State)) +
    geom_col(width = 0.7, alpha = 0.8) +
    geom_hline(yintercept = EVENNESS_FLAG_THRESHOLD, linetype = "dashed", color = "red", linewidth = 0.5) +
    scale_y_continuous(limits = c(0, 1), expand = c(0, 0)) +
    labs(x = "Cluster", y = "Shannon Evenness",
         title = paste0(toupper(cell_type_label), 
                        ": Donor evenness per cluster (red = ", EVENNESS_FLAG_THRESHOLD, " threshold)")) +
    theme_bw(base_size = 10) +
    theme(
        axis.text.x = element_text(size = 9),
        plot.title = element_text(size = 10, face = "bold"),
        legend.position = "bottom",
        legend.title = element_blank(),
        legend.text = element_text(size = 8)
    )

print(p_evenness)

ggsave(file.path(VAL_FIGURES_DIR, paste0(FILE_PREFIX, "_donor_evenness.pdf")),
       p_evenness, width = 10, height = 4)
ggsave(file.path(VAL_FIGURES_DIR, paste0(FILE_PREFIX, "_donor_evenness.svg")),
       p_evenness, width = 10, height = 4)

cat("  ✓ Saved composition plots\n")

# ─────────────────────────────────────────────────────────────────────────────────
# Step 3 Summary
# ─────────────────────────────────────────────────────────────────────────────────

cat("\n", paste(rep("-", 80), collapse = ""), "\n")
cat("STEP 3 SUMMARY\n")
cat(paste(rep("-", 80), collapse = ""), "\n")
cat("  Total donors:", n_donors, "\n")
cat("  Sample-dominated clusters (res", CLUSTERING_RESOLUTION, "):", 
    if(flagged_sample) paste(flagged_clusters, collapse = ", ") else "NONE", "\n")
cat("  Min evenness:", min(cluster_comp_summary$Evenness),
    "(cluster", cluster_comp_summary$Cluster[which.min(cluster_comp_summary$Evenness)], ")\n")
cat("  Max top-1 donor %:", max(cluster_comp_summary$Top1_pct),
    "(cluster", cluster_comp_summary$Cluster[which.max(cluster_comp_summary$Top1_pct)], ")\n")

# Multi-resolution donor stability summary
if (exists("flag_summary")) {
    always_flagged_res <- sum(flag_summary$N_flagged > 0)
    cat("  Multi-resolution: donor-dominated clusters present at",
        always_flagged_res, "/", nrow(flag_summary), "resolutions\n")
}
if (exists("donor_tracking") && nrow(donor_tracking) > 0) {
    for (cl in flagged_clusters) {
        cl_track <- donor_tracking[donor_tracking$Source_cluster == cl, ]
        n_struct <- sum(cl_track$Dest_flagged, na.rm = TRUE)
        cat(sprintf("    Cluster %s dominant donor: flagged at %d/%d resolutions",
                    cl, n_struct, nrow(cl_track)))
        if (n_struct == nrow(cl_track)) {
            cat(" → STRUCTURAL\n")
        } else if (n_struct > nrow(cl_track) / 2) {
            cat(" → MOSTLY STRUCTURAL\n")
        } else {
            cat(" → RESOLUTION-DEPENDENT\n")
        }
    }
}

if (exists("state_robustness")) {
    at_risk_states <- state_robustness$State[state_robustness$Donors_clean < 5 | 
                                              state_robustness$Pct_retained < 50 |
                                              state_robustness$Evenness < 0.5]
    if (length(at_risk_states) > 0) {
        cat("  States at risk:", paste(at_risk_states, collapse = ", "), "\n")
    } else {
        cat("  State robustness: All states retain multi-donor support\n")
    }
}
cat("  Resolutions computed:", paste(res_values, collapse = ", "), "\n")
cat("\n  Files saved:\n")
cat("    •", paste0(FILE_PREFIX, "_sample_comp_cluster.csv"), "\n")
cat("    •", paste0(FILE_PREFIX, "_sample_comp_state.csv"), "\n")
if (exists("state_robustness")) cat("    •", paste0(FILE_PREFIX, "_state_robustness.csv"), "\n")
cat("    •", paste0(FILE_PREFIX, "_state_resolution_stability.csv"), "\n")
cat("    •", paste0(FILE_PREFIX, "_multires_donor_comp.csv"), "\n")
cat("    •", paste0(FILE_PREFIX, "_multires_flag_summary.csv"), "\n")
if (exists("donor_tracking")) cat("    •", paste0(FILE_PREFIX, "_donor_tracking_across_res.csv"), "\n")
cat("    •", paste0(FILE_PREFIX, "_donor_comp_cluster.{pdf,svg}"), "\n")
cat("    •", paste0(FILE_PREFIX, "_donor_comp_state.{pdf,svg}"), "\n")
cat("    •", paste0(FILE_PREFIX, "_donor_evenness.{pdf,svg}"), "\n")
cat("    •", paste0(FILE_PREFIX, "_multires_flagged_count.{pdf,svg}"), "\n")
if (exists("donor_tracking")) cat("    •", paste0(FILE_PREFIX, "_donor_tracking_across_res.{pdf,svg}"), "\n")
cat(paste(rep("=", 80), collapse = ""), "\n")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# STEP 4: CELL CYCLE SCORING
# ════════════════════════════════════════════════════════════════════════════════

cat("\n", paste(rep("=", 80), collapse = ""), "\n")
cat("STEP 4: CELL CYCLE SCORING\n")
cat(paste(rep("=", 80), collapse = ""), "\n")

# ─────────────────────────────────────────────────────────────────────────────────
# 4A: Score cell cycle
# ─────────────────────────────────────────────────────────────────────────────────

cat("\n▸ Scoring cell cycle phases...\n")

s_genes <- cc.genes.updated.2019$s.genes
g2m_genes <- cc.genes.updated.2019$g2m.genes

s_found <- s_genes[s_genes %in% rownames(seurat_obj)]
g2m_found <- g2m_genes[g2m_genes %in% rownames(seurat_obj)]

cat("  S genes found:", length(s_found), "/", length(s_genes), "\n")
cat("  G2M genes found:", length(g2m_found), "/", length(g2m_genes), "\n")

seurat_obj <- CellCycleScoring(seurat_obj, s.features = s_found, g2m.features = g2m_found)

cat("  ✓ Cell cycle scoring complete\n")

# ─────────────────────────────────────────────────────────────────────────────────
# 4B: Phase distribution per cluster
# ─────────────────────────────────────────────────────────────────────────────────

cat("\n▸ Phase distribution per cluster:\n")

cl_ids <- as.character(seurat_obj$seurat_clusters)
phase_table <- table(cl_ids, seurat_obj$Phase)
phase_pct <- prop.table(phase_table, margin = 1) * 100

phase_df <- as.data.frame(round(phase_pct, 1))
colnames(phase_df) <- c("Cluster", "Phase", "Pct")

# Wide format for printing
phase_wide <- as.data.frame.matrix(round(phase_pct, 1))
phase_wide$Cluster <- rownames(phase_wide)
phase_wide$Cycling <- round(phase_pct[, "S"] + phase_pct[, "G2M"], 1)
phase_wide$State <- sapply(phase_wide$Cluster, function(cl) {
    cluster_comp_summary$State[cluster_comp_summary$Cluster == cl]
})
phase_wide <- phase_wide[match(cluster_order, phase_wide$Cluster), 
                          c("Cluster", "State", "G1", "G2M", "S", "Cycling")]

print(phase_wide, row.names = FALSE)

write.csv(phase_wide, file.path(VAL_RESULTS_DIR, paste0(FILE_PREFIX, "_cellcycle_cluster.csv")),
          row.names = FALSE)

# ─────────────────────────────────────────────────────────────────────────────────
# 4C: Phase distribution per state
# ─────────────────────────────────────────────────────────────────────────────────

cat("\n▸ Phase distribution per state:\n")

state_ids <- as.character(seurat_obj@meta.data[[SUBCLUSTER_COL]])
phase_state <- table(state_ids, seurat_obj$Phase)
phase_state_pct <- prop.table(phase_state, margin = 1) * 100

phase_state_wide <- as.data.frame.matrix(round(phase_state_pct, 1))
phase_state_wide$State <- rownames(phase_state_wide)
phase_state_wide$Cycling <- round(phase_state_pct[, "S"] + phase_state_pct[, "G2M"], 1)
phase_state_wide <- phase_state_wide[match(state_order, phase_state_wide$State),
                                      c("State", "G1", "G2M", "S", "Cycling")]

print(phase_state_wide, row.names = FALSE)

write.csv(phase_state_wide, file.path(VAL_RESULTS_DIR, paste0(FILE_PREFIX, "_cellcycle_state.csv")),
          row.names = FALSE)

# ─────────────────────────────────────────────────────────────────────────────────
# 4D: Flag cycling clusters (>30% S/G2M)
# ─────────────────────────────────────────────────────────────────────────────────

cat("\n▸ Flagging cycling clusters (>30% S/G2M):\n")

flagged_cc <- FALSE
for (i in 1:nrow(phase_wide)) {
    if (phase_wide$Cycling[i] > 30) {
        cat(sprintf("  ⚠ Cluster %s (%s): %.1f%% cycling (S=%.1f%%, G2M=%.1f%%)\n",
                    phase_wide$Cluster[i], phase_wide$State[i], phase_wide$Cycling[i],
                    phase_wide$S[i], phase_wide$G2M[i]))
        flagged_cc <- TRUE
    }
}
if (!flagged_cc) cat("  ✓ No heavily cycling clusters\n")

# ─────────────────────────────────────────────────────────────────────────────────
# 4E: Stacked bar — cell cycle per cluster
# ─────────────────────────────────────────────────────────────────────────────────

cat("\n▸ Generating cell cycle plots...\n")

PHASE_COLORS <- c("G1" = "#B0B0B0", "S" = "#E15759", "G2M" = "#4E79A7")

phase_plot_df <- as.data.frame(phase_table)
colnames(phase_plot_df) <- c("Cluster", "Phase", "Count")
phase_plot_df$Cluster <- factor(phase_plot_df$Cluster, levels = cluster_order)
phase_plot_df$Phase <- factor(phase_plot_df$Phase, levels = c("G1", "S", "G2M"))

options(repr.plot.width = 12, repr.plot.height = 5)

p_cc_cluster <- ggplot(phase_plot_df, aes(x = Cluster, y = Count, fill = Phase)) +
    geom_col(position = "fill", width = 0.8, color = "white", linewidth = 0.2) +
    scale_fill_manual(values = PHASE_COLORS) +
    scale_y_continuous(labels = percent, expand = c(0, 0)) +
    labs(x = "Cluster", y = "Proportion", fill = "Cell Cycle Phase",
         title = paste0(toupper(cell_type_label), ": Cell cycle phase per cluster")) +
    theme_minimal(base_size = 14) +
    theme(
        axis.text.x = element_text(size = 12, color = "black"),
        axis.text.y = element_text(size = 12, color = "black"),
        axis.title.x = element_text(size = 14, face = "bold", margin = margin(t = 10)),
        axis.title.y = element_text(size = 14, face = "bold", margin = margin(r = 10)),
        axis.line = element_line(color = "black", linewidth = 0.5),
        axis.ticks = element_line(color = "black", linewidth = 0.3),
        axis.ticks.length = unit(0.15, "cm"),
        plot.title = element_text(size = 14, face = "bold"),
        legend.position = "right",
        legend.title = element_text(size = 11, face = "bold"),
        legend.text = element_text(size = 10),
        panel.grid = element_blank(),
        plot.margin = margin(15, 15, 15, 15)
    )

print(p_cc_cluster)

ggsave(file.path(VAL_FIGURES_DIR, paste0(FILE_PREFIX, "_cellcycle_cluster.pdf")),
       p_cc_cluster, width = 12, height = 5)
ggsave(file.path(VAL_FIGURES_DIR, paste0(FILE_PREFIX, "_cellcycle_cluster.svg")),
       p_cc_cluster, width = 12, height = 5)

# ─────────────────────────────────────────────────────────────────────────────────
# 4F: Stacked bar — cell cycle per state
# ─────────────────────────────────────────────────────────────────────────────────

phase_state_plot <- as.data.frame(phase_state)
colnames(phase_state_plot) <- c("State", "Phase", "Count")
phase_state_plot$State <- factor(phase_state_plot$State, levels = state_order)
phase_state_plot$Phase <- factor(phase_state_plot$Phase, levels = c("G1", "S", "G2M"))

options(repr.plot.width = 8, repr.plot.height = 5)

p_cc_state <- ggplot(phase_state_plot, aes(x = State, y = Count, fill = Phase)) +
    geom_col(position = "fill", width = 0.8, color = "white", linewidth = 0.2) +
    scale_fill_manual(values = PHASE_COLORS) +
    scale_y_continuous(labels = percent, expand = c(0, 0)) +
    labs(x = paste0(tools::toTitleCase(cell_type_label), " State"), y = "Proportion",
         fill = "Cell Cycle Phase",
         title = paste0(toupper(cell_type_label), ": Cell cycle phase per state")) +
    theme_minimal(base_size = 14) +
    theme(
        axis.text.x = element_text(size = 12, color = "black", angle = 30, hjust = 1),
        axis.text.y = element_text(size = 12, color = "black"),
        axis.title.x = element_text(size = 14, face = "bold", margin = margin(t = 10)),
        axis.title.y = element_text(size = 14, face = "bold", margin = margin(r = 10)),
        axis.line = element_line(color = "black", linewidth = 0.5),
        axis.ticks = element_line(color = "black", linewidth = 0.3),
        axis.ticks.length = unit(0.15, "cm"),
        plot.title = element_text(size = 14, face = "bold"),
        legend.position = "right",
        legend.title = element_text(size = 11, face = "bold"),
        legend.text = element_text(size = 10),
        panel.grid = element_blank(),
        plot.margin = margin(15, 15, 15, 15)
    )

print(p_cc_state)

ggsave(file.path(VAL_FIGURES_DIR, paste0(FILE_PREFIX, "_cellcycle_state.pdf")),
       p_cc_state, width = 8, height = 5)
ggsave(file.path(VAL_FIGURES_DIR, paste0(FILE_PREFIX, "_cellcycle_state.svg")),
       p_cc_state, width = 8, height = 5)

# ─────────────────────────────────────────────────────────────────────────────────
# 4G: UMAP — Phase + S.Score + G2M.Score (with corner arrows)
# ─────────────────────────────────────────────────────────────────────────────────

if (!exists("add_corner_arrows")) {
    add_corner_arrows <- function(p, label_size = 3) {
        build <- ggplot_build(p)
        x_range <- build$layout$panel_params[[1]]$x.range
        y_range <- build$layout$panel_params[[1]]$y.range
        
        x_start <- x_range[1] + diff(x_range) * 0.02
        y_start <- y_range[1] + diff(y_range) * 0.02
        x_arrow <- diff(x_range) * 0.12
        y_arrow <- diff(y_range) * 0.12
        
        p + 
            annotate("segment", x = x_start, xend = x_start + x_arrow,
                     y = y_start, yend = y_start,
                     arrow = arrow(length = unit(0.12, "cm"), type = "closed"),
                     linewidth = 0.4) +
            annotate("text", x = x_start + x_arrow/2,
                     y = y_start - diff(y_range) * 0.04,
                     label = "UMAP1", size = label_size,
                     hjust = 0.5, vjust = 1, fontface = "bold") +
            annotate("segment", x = x_start, xend = x_start,
                     y = y_start, yend = y_start + y_arrow,
                     arrow = arrow(length = unit(0.12, "cm"), type = "closed"),
                     linewidth = 0.4) +
            annotate("text", x = x_start - diff(x_range) * 0.04,
                     y = y_start + y_arrow/2,
                     label = "UMAP2", size = label_size,
                     hjust = 1, vjust = 0.5, angle = 90, fontface = "bold") +
            coord_cartesian(clip = "off")
    }
    
    umap_theme <- theme_void(base_size = 12) +
        theme(
            plot.title = element_text(size = 11, face = "bold", hjust = 0.5),
            legend.text = element_text(size = 9),
            legend.title = element_blank(),
            plot.margin = margin(15, 15, 20, 20)
        )
    
    umap_theme_noleg <- umap_theme +
        theme(legend.position = "none")
}

options(repr.plot.width = 14, repr.plot.height = 4.5)

p_phase_umap <- DimPlot(seurat_obj, group.by = "Phase", reduction = UMAP_RED,
                         cols = PHASE_COLORS, pt.size = 0.1) +
    ggtitle("Cell Cycle Phase") +
    umap_theme
p_phase_umap <- add_corner_arrows(p_phase_umap, label_size = 2.5)

p_s_score <- FeaturePlot(seurat_obj, features = "S.Score", reduction = UMAP_RED,
                          pt.size = 0.1, order = TRUE) +
    scale_color_viridis_c() +
    ggtitle("S.Score") +
    umap_theme_noleg
p_s_score <- add_corner_arrows(p_s_score, label_size = 2.5)

p_g2m_score <- FeaturePlot(seurat_obj, features = "G2M.Score", reduction = UMAP_RED,
                            pt.size = 0.1, order = TRUE) +
    scale_color_viridis_c() +
    ggtitle("G2M.Score") +
    umap_theme_noleg
p_g2m_score <- add_corner_arrows(p_g2m_score, label_size = 2.5)

p_cc_umap <- cowplot::plot_grid(p_phase_umap, p_s_score, p_g2m_score, ncol = 3)

print(p_cc_umap)

ggsave(file.path(VAL_FIGURES_DIR, paste0(FILE_PREFIX, "_cellcycle_umap.pdf")),
       p_cc_umap, width = 14, height = 4.5)
ggsave(file.path(VAL_FIGURES_DIR, paste0(FILE_PREFIX, "_cellcycle_umap.svg")),
       p_cc_umap, width = 14, height = 4.5)

cat("  ✓ Saved cell cycle plots\n")

# ─────────────────────────────────────────────────────────────────────────────────
# Step 4 Summary
# ─────────────────────────────────────────────────────────────────────────────────

cat("\n", paste(rep("-", 80), collapse = ""), "\n")
cat("STEP 4 SUMMARY\n")
cat(paste(rep("-", 80), collapse = ""), "\n")
cat("  Overall cycling (S+G2M):",
    round(sum(seurat_obj$Phase %in% c("S", "G2M")) / ncol(seurat_obj) * 100, 1), "%\n")
cat("  Cycling clusters (>30%):", if(flagged_cc) "YES" else "NONE", "\n")

# Top cycling clusters regardless of threshold
top_cycling <- phase_wide[order(-phase_wide$Cycling), ][1:3, ]
cat("  Top 3 cycling clusters:\n")
for (i in 1:3) {
    cat(sprintf("    Cluster %s (%s): %.1f%%\n",
                top_cycling$Cluster[i], top_cycling$State[i], top_cycling$Cycling[i]))
}

cat("\n  Files saved:\n")
cat("    •", paste0(FILE_PREFIX, "_cellcycle_cluster.csv"), "\n")
cat("    •", paste0(FILE_PREFIX, "_cellcycle_state.csv"), "\n")
cat("    •", paste0(FILE_PREFIX, "_cellcycle_cluster.{pdf,svg}"), "\n")
cat("    •", paste0(FILE_PREFIX, "_cellcycle_state.{pdf,svg}"), "\n")
cat("    •", paste0(FILE_PREFIX, "_cellcycle_umap.{pdf,svg}"), "\n")
cat(paste(rep("=", 80), collapse = ""), "\n")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# STEP 5: CLUSTER STABILITY (CLUSTREE)
# ════════════════════════════════════════════════════════════════════════════════

cat("\n", paste(rep("=", 80), collapse = ""), "\n")
cat("STEP 5: CLUSTER STABILITY\n")
cat(paste(rep("=", 80), collapse = ""), "\n")

library(clustree)

# Safety check
if (!exists("N_DIMS_USE")) {
    if ("harmony" %in% Reductions(seurat_obj)) {
        N_DIMS_USE <- min(ncol(Embeddings(seurat_obj, "harmony")), 30)
    } else if ("pca" %in% Reductions(seurat_obj)) {
        N_DIMS_USE <- min(ncol(Embeddings(seurat_obj, "pca")), 30)
    } else {
        N_DIMS_USE <- 15
    }
    cat("  N_DIMS_USE detected:", N_DIMS_USE, "\n")
} else {
    cat("  N_DIMS_USE:", N_DIMS_USE, "\n")
}

# ─────────────────────────────────────────────────────────────────────────────────
# 5A: Ensure multi-resolution clustering exists
# ─────────────────────────────────────────────────────────────────────────────────

cat("\n▸ Multi-resolution clustering...\n")

res_seq <- c(0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 1.0, 1.2)
existing_res <- grep("RNA_snn_res\\.", colnames(seurat_obj@meta.data), value = TRUE)
existing_values <- as.numeric(gsub("RNA_snn_res\\.", "", existing_res))
missing_res <- setdiff(res_seq, existing_values)

if (length(missing_res) > 0) {
    cat("  Computing", length(missing_res), "missing resolutions...\n")
    
    seurat_obj$original_clusters <- seurat_obj$seurat_clusters
    
    cluster_reduction <- ifelse("harmony" %in% Reductions(seurat_obj), "harmony", "pca")
    
    if (length(seurat_obj@graphs) == 0) {
        cat("  Building neighbor graph...\n")
        seurat_obj <- FindNeighbors(seurat_obj, reduction = cluster_reduction,
                                     dims = 1:N_DIMS_USE, verbose = FALSE)
    }
    
    for (res in sort(missing_res)) {
        seurat_obj <- FindClusters(seurat_obj, resolution = res, verbose = FALSE)
        col_name <- paste0("RNA_snn_res.", res)
        n_cl <- length(unique(seurat_obj@meta.data[[col_name]]))
        cat(sprintf("    res = %.1f -> %d clusters\n", res, n_cl))
    }
    
    seurat_obj$seurat_clusters <- seurat_obj$original_clusters
    Idents(seurat_obj) <- seurat_obj$seurat_clusters
    seurat_obj$original_clusters <- NULL
    cat("  ✓ Restored original seurat_clusters\n")
} else {
    cat("  All resolutions already computed (from Step 3)\n")
}

# Print summary
for (res in res_seq) {
    col_name <- paste0("RNA_snn_res.", res)
    n_cl <- length(unique(seurat_obj@meta.data[[col_name]]))
    marker <- ifelse(res == CLUSTERING_RESOLUTION, " <- CURRENT", "")
    cat(sprintf("    res = %.1f -> %d clusters%s\n", res, n_cl, marker))
}

# ─────────────────────────────────────────────────────────────────────────────────
# 5B: Resolution summary table
# ─────────────────────────────────────────────────────────────────────────────────

cat("\n▸ Resolution summary:\n")

res_summary <- do.call(rbind, lapply(res_seq, function(res) {
    col_name <- paste0("RNA_snn_res.", res)
    cl_sizes <- table(seurat_obj@meta.data[[col_name]])
    data.frame(
        Resolution = res,
        N_clusters = length(cl_sizes),
        Min_size = min(cl_sizes),
        Max_size = max(cl_sizes),
        Median_size = median(cl_sizes),
        stringsAsFactors = FALSE
    )
}))

print(res_summary, row.names = FALSE)

write.csv(res_summary, file.path(VAL_RESULTS_DIR, paste0(FILE_PREFIX, "_resolution_summary.csv")),
          row.names = FALSE)

# ─────────────────────────────────────────────────────────────────────────────────
# 5C: Clustree plot
# ─────────────────────────────────────────────────────────────────────────────────

cat("\n▸ Generating clustree...\n")

options(repr.plot.width = 12, repr.plot.height = 10)

p_tree <- clustree(seurat_obj, prefix = "RNA_snn_res.") +
    ggtitle(paste0(toupper(cell_type_label),
                   ": Cluster Stability Across Resolutions")) +
    theme(plot.title = element_text(size = 14, face = "bold"),
          legend.text = element_text(size = 9),
          legend.title = element_text(size = 10))

print(p_tree)

ggsave(file.path(VAL_FIGURES_DIR, paste0(FILE_PREFIX, "_clustree.pdf")),
       p_tree, width = 12, height = 10)
ggsave(file.path(VAL_FIGURES_DIR, paste0(FILE_PREFIX, "_clustree.svg")),
       p_tree, width = 12, height = 10)

# ─────────────────────────────────────────────────────────────────────────────────
# 5D: Clustree with state overlay
# ─────────────────────────────────────────────────────────────────────────────────

cat("\n▸ Generating clustree with state overlay...\n")

state_numeric <- as.numeric(factor(seurat_obj@meta.data[[SUBCLUSTER_COL]],
                                    levels = state_order))
seurat_obj$state_numeric <- state_numeric

options(repr.plot.width = 12, repr.plot.height = 10)

p_tree_state <- clustree(seurat_obj, prefix = "RNA_snn_res.",
                          node_colour = "state_numeric",
                          node_colour_aggr = "median") +
    scale_color_gradientn(colors = state_colors,
                          breaks = 1:length(state_order),
                          labels = state_order,
                          name = "Dominant\nState") +
    ggtitle(paste0(toupper(cell_type_label),
                   ": Cluster Stability (colored by dominant state)")) +
    theme(plot.title = element_text(size = 14, face = "bold"),
          legend.text = element_text(size = 9),
          legend.title = element_text(size = 10))

print(p_tree_state)

ggsave(file.path(VAL_FIGURES_DIR, paste0(FILE_PREFIX, "_clustree_state.pdf")),
       p_tree_state, width = 12, height = 10)
ggsave(file.path(VAL_FIGURES_DIR, paste0(FILE_PREFIX, "_clustree_state.svg")),
       p_tree_state, width = 12, height = 10)

# ─────────────────────────────────────────────────────────────────────────────────
# 5E: State stability across resolutions
# ─────────────────────────────────────────────────────────────────────────────────

cat("\n▸ Assessing state stability across resolutions...\n")

stability_results <- list()

for (res in res_seq) {
    col_name <- paste0("RNA_snn_res.", res)
    clusters_at_res <- seurat_obj@meta.data[[col_name]]
    states_at_res <- seurat_obj@meta.data[[SUBCLUSTER_COL]]
    
    cluster_purity <- tapply(states_at_res, clusters_at_res, function(x) {
        max(table(x)) / length(x) * 100
    })
    
    stability_results[[as.character(res)]] <- data.frame(
        Resolution = res,
        N_clusters = length(cluster_purity),
        Mean_purity = round(mean(cluster_purity), 1),
        Min_purity = round(min(cluster_purity), 1),
        N_mixed = sum(cluster_purity < 70),
        stringsAsFactors = FALSE
    )
}

stability_df <- do.call(rbind, stability_results)
cat("\n  State purity per resolution:\n")
print(stability_df, row.names = FALSE)

write.csv(stability_df, file.path(VAL_RESULTS_DIR, paste0(FILE_PREFIX, "_state_stability.csv")),
          row.names = FALSE)

# Save cluster_purity at current resolution for Summary cell
current_col <- paste0("RNA_snn_res.", CLUSTERING_RESOLUTION)
if (current_col %in% colnames(seurat_obj@meta.data)) {
    cluster_purity <- tapply(
        seurat_obj@meta.data[[SUBCLUSTER_COL]],
        seurat_obj@meta.data[[current_col]],
        function(x) max(table(x)) / length(x) * 100
    )
}

# ─────────────────────────────────────────────────────────────────────────────────
# 5F: Purity plot
# ─────────────────────────────────────────────────────────────────────────────────

options(repr.plot.width = 8, repr.plot.height = 4.5)

p_purity <- ggplot(stability_df, aes(x = Resolution)) +
    geom_line(aes(y = Mean_purity), color = "#4E79A7", linewidth = 1) +
    geom_point(aes(y = Mean_purity), color = "#4E79A7", size = 3) +
    geom_line(aes(y = Min_purity), color = "#E15759", linewidth = 0.8, linetype = "dashed") +
    geom_point(aes(y = Min_purity), color = "#E15759", size = 2) +
    geom_vline(xintercept = CLUSTERING_RESOLUTION, linetype = "dotted",
               color = "black", linewidth = 0.8) +
    annotate("text", x = CLUSTERING_RESOLUTION + 0.03, y = 50,
             label = paste0("Current (", CLUSTERING_RESOLUTION, ")"),
             hjust = 0, size = 3.5, fontface = "bold") +
    scale_y_continuous(limits = c(0, 100)) +
    labs(x = "Clustering Resolution", y = "State Purity (%)",
         title = paste0(toupper(cell_type_label),
                        ": State purity across resolutions"),
         subtitle = "Blue = mean purity, Red dashed = min purity") +
    theme_minimal(base_size = 14) +
    theme(
        axis.text.x = element_text(size = 12, color = "black"),
        axis.text.y = element_text(size = 12, color = "black"),
        axis.title.x = element_text(size = 14, face = "bold", margin = margin(t = 10)),
        axis.title.y = element_text(size = 14, face = "bold", margin = margin(r = 10)),
        axis.line = element_line(color = "black", linewidth = 0.5),
        axis.ticks = element_line(color = "black", linewidth = 0.3),
        axis.ticks.length = unit(0.15, "cm"),
        plot.title = element_text(size = 13, face = "bold"),
        plot.subtitle = element_text(size = 10, color = "#666666"),
        panel.grid = element_blank(),
        plot.margin = margin(15, 15, 15, 15)
    )

print(p_purity)

ggsave(file.path(VAL_FIGURES_DIR, paste0(FILE_PREFIX, "_state_purity.pdf")),
       p_purity, width = 8, height = 4.5)
ggsave(file.path(VAL_FIGURES_DIR, paste0(FILE_PREFIX, "_state_purity.svg")),
       p_purity, width = 8, height = 4.5)

cat("  ✓ Saved stability plots\n")

# ─────────────────────────────────────────────────────────────────────────────────
# 5G: Verify original clustering is intact
# ─────────────────────────────────────────────────────────────────────────────────

Idents(seurat_obj) <- seurat_obj$seurat_clusters
cat("\n  ✓ Active identity: seurat_clusters (",
    length(unique(Idents(seurat_obj))), " clusters)\n", sep = "")

# ─────────────────────────────────────────────────────────────────────────────────
# Step 5 Summary
# ─────────────────────────────────────────────────────────────────────────────────

cat("\n", paste(rep("-", 80), collapse = ""), "\n")
cat("STEP 5 SUMMARY\n")
cat(paste(rep("-", 80), collapse = ""), "\n")

current_res <- stability_df[stability_df$Resolution == CLUSTERING_RESOLUTION, ]
cat("  Current resolution:", CLUSTERING_RESOLUTION, "\n")
if (nrow(current_res) > 0) {
    cat("  Clusters at current res:", current_res$N_clusters, "\n")
    cat("  Mean state purity:", current_res$Mean_purity, "%\n")
    cat("  Min state purity:", current_res$Min_purity, "%\n")
    cat("  Mixed clusters (<70% purity):", current_res$N_mixed, "\n")
} else {
    cat("  Note: CLUSTERING_RESOLUTION", CLUSTERING_RESOLUTION,
        "not in res_seq, using original clusters\n")
    cat("  Original clusters:", length(unique(seurat_obj$seurat_clusters)), "\n")
}

cat("\n  Verification: seurat_clusters has",
    length(unique(seurat_obj$seurat_clusters)), "clusters (original preserved)\n")

cat("\n  Files saved:\n")
cat("    •", paste0(FILE_PREFIX, "_resolution_summary.csv"), "\n")
cat("    •", paste0(FILE_PREFIX, "_state_stability.csv"), "\n")
cat("    •", paste0(FILE_PREFIX, "_clustree.{pdf,svg}"), "\n")
cat("    •", paste0(FILE_PREFIX, "_clustree_state.{pdf,svg}"), "\n")
cat("    •", paste0(FILE_PREFIX, "_state_purity.{pdf,svg}"), "\n")
cat(paste(rep("=", 80), collapse = ""), "\n")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# VALIDATION UMAPs
# ════════════════════════════════════════════════════════════════════════════════

cat("\n", paste(rep("=", 80), collapse = ""), "\n")
cat("VALIDATION UMAPs\n")
cat(paste(rep("=", 80), collapse = ""), "\n")

# ─────────────────────────────────────────────────────────────────────────────────
# Panel 1: Clusters + States + Flagged
# ─────────────────────────────────────────────────────────────────────────────────

options(repr.plot.width = 16, repr.plot.height = 5)

# Cluster UMAP
p_clusters <- DimPlot(seurat_obj, group.by = "seurat_clusters", reduction = UMAP_RED,
                       label = TRUE, label.size = 3, pt.size = 0.1, repel = TRUE) +
    ggtitle("Clusters") +
    umap_theme_noleg
p_clusters <- add_corner_arrows(p_clusters, label_size = 2.5)

# State UMAP
p_states <- DimPlot(seurat_obj, group.by = SUBCLUSTER_COL, reduction = UMAP_RED,
                     cols = state_colors, pt.size = 0.1) +
    ggtitle("Annotated States") +
    umap_theme
p_states <- add_corner_arrows(p_states, label_size = 2.5)

# Flagged clusters highlighted
flagged_clusters <- c("16", "12", "13")
seurat_obj$flagged <- ifelse(
    as.character(seurat_obj$seurat_clusters) == "16", "Remove (Cl.16)",
    ifelse(as.character(seurat_obj$seurat_clusters) %in% c("12", "13"), "Monitor (Cl.12,13)",
           "Pass"))
seurat_obj$flagged <- factor(seurat_obj$flagged,
                              levels = c("Pass", "Monitor (Cl.12,13)", "Remove (Cl.16)"))

FLAG_COLORS <- c("Pass" = "#D3D3D3", "Monitor (Cl.12,13)" = "#F28E2B", "Remove (Cl.16)" = "#E15759")

p_flagged <- DimPlot(seurat_obj, group.by = "flagged", reduction = UMAP_RED,
                      cols = FLAG_COLORS, pt.size = 0.1, order = c("Remove (Cl.16)", "Monitor (Cl.12,13)")) +
    ggtitle("Flagged Clusters") +
    umap_theme
p_flagged <- add_corner_arrows(p_flagged, label_size = 2.5)

p_panel1 <- cowplot::plot_grid(p_clusters, p_states, p_flagged, ncol = 3)
print(p_panel1)

ggsave(file.path(VAL_FIGURES_DIR, paste0(FILE_PREFIX, "_umap_panel1.pdf")),
       p_panel1, width = 16, height = 5)
ggsave(file.path(VAL_FIGURES_DIR, paste0(FILE_PREFIX, "_umap_panel1.svg")),
       p_panel1, width = 16, height = 5)

# ─────────────────────────────────────────────────────────────────────────────────
# Panel 2: QC metrics on UMAP
# ─────────────────────────────────────────────────────────────────────────────────

options(repr.plot.width = 14, repr.plot.height = 4)

qc_umap_list <- lapply(QC_METRICS, function(feat) {
    p <- FeaturePlot(seurat_obj, features = feat, reduction = UMAP_RED,
                      pt.size = 0.1, order = TRUE) +
        scale_color_viridis_c() +
        ggtitle(feat) +
        umap_theme_noleg
    add_corner_arrows(p, label_size = 2)
})

p_qc_umap <- cowplot::plot_grid(plotlist = qc_umap_list, ncol = length(QC_METRICS))
print(p_qc_umap)

ggsave(file.path(VAL_FIGURES_DIR, paste0(FILE_PREFIX, "_umap_qc.pdf")),
       p_qc_umap, width = 3.5 * length(QC_METRICS), height = 4)
ggsave(file.path(VAL_FIGURES_DIR, paste0(FILE_PREFIX, "_umap_qc.svg")),
       p_qc_umap, width = 3.5 * length(QC_METRICS), height = 4)

# ─────────────────────────────────────────────────────────────────────────────────
# Panel 3: Donor + Study Group + Senescence
# ─────────────────────────────────────────────────────────────────────────────────

options(repr.plot.width = 16, repr.plot.height = 5)

p_donor <- DimPlot(seurat_obj, group.by = DONOR_COL, reduction = UMAP_RED,
                    pt.size = 0.1) +
    ggtitle("Donor") +
    umap_theme_noleg
p_donor <- add_corner_arrows(p_donor, label_size = 2.5)

study_colors <- STUDY_GROUP_COLORS[names(STUDY_GROUP_COLORS) %in% sg_order]
p_sg <- DimPlot(seurat_obj, group.by = STUDY_GROUP_COL, reduction = UMAP_RED,
                 cols = study_colors, pt.size = 0.1) +
    ggtitle("Study Group") +
    umap_theme +
    theme(legend.key.size = unit(0.3, "cm"),
          legend.text = element_text(size = 7))
p_sg <- add_corner_arrows(p_sg, label_size = 2.5)

snc_col <- ifelse("senescence_label_state" %in% colnames(seurat_obj@meta.data),
                   "senescence_label_state", "senescence_label")
p_snc <- DimPlot(seurat_obj, group.by = snc_col, reduction = UMAP_RED,
                  cols = SENESCENCE_COLORS, pt.size = 0.1,
                  order = "SnC") +
    ggtitle("Senescence") +
    umap_theme
p_snc <- add_corner_arrows(p_snc, label_size = 2.5)

p_panel3 <- cowplot::plot_grid(p_donor, p_sg, p_snc, ncol = 3)
print(p_panel3)

ggsave(file.path(VAL_FIGURES_DIR, paste0(FILE_PREFIX, "_umap_panel3.pdf")),
       p_panel3, width = 16, height = 5)
ggsave(file.path(VAL_FIGURES_DIR, paste0(FILE_PREFIX, "_umap_panel3.svg")),
       p_panel3, width = 16, height = 5)

cat("\n  ✓ Saved UMAP panels\n")

# ─────────────────────────────────────────────────────────────────────────────────
# Panel 4: Cell cycle on UMAP
# ─────────────────────────────────────────────────────────────────────────────────

options(repr.plot.width = 14, repr.plot.height = 4.5)

PHASE_COLORS <- c("G1" = "#B0B0B0", "S" = "#E15759", "G2M" = "#4E79A7")

p_phase <- DimPlot(seurat_obj, group.by = "Phase", reduction = UMAP_RED,
                    cols = PHASE_COLORS, pt.size = 0.1) +
    ggtitle("Cell Cycle Phase") +
    umap_theme
p_phase <- add_corner_arrows(p_phase, label_size = 2.5)

p_s <- FeaturePlot(seurat_obj, features = "S.Score", reduction = UMAP_RED,
                    pt.size = 0.1, order = TRUE) +
    scale_color_viridis_c() +
    ggtitle("S.Score") +
    umap_theme_noleg
p_s <- add_corner_arrows(p_s, label_size = 2.5)

p_g2m <- FeaturePlot(seurat_obj, features = "G2M.Score", reduction = UMAP_RED,
                      pt.size = 0.1, order = TRUE) +
    scale_color_viridis_c() +
    ggtitle("G2M.Score") +
    umap_theme_noleg
p_g2m <- add_corner_arrows(p_g2m, label_size = 2.5)

p_panel4 <- cowplot::plot_grid(p_phase, p_s, p_g2m, ncol = 3)
print(p_panel4)

ggsave(file.path(VAL_FIGURES_DIR, paste0(FILE_PREFIX, "_umap_cellcycle.pdf")),
       p_panel4, width = 14, height = 4.5)
ggsave(file.path(VAL_FIGURES_DIR, paste0(FILE_PREFIX, "_umap_cellcycle.svg")),
       p_panel4, width = 14, height = 4.5)

cat("  ✓ Saved all UMAP panels\n")
cat("\n  Files:\n")
cat("    •", paste0(FILE_PREFIX, "_umap_panel1.{pdf,svg} (clusters + states + flagged)"), "\n")
cat("    •", paste0(FILE_PREFIX, "_umap_qc.{pdf,svg} (QC metrics)"), "\n")
cat("    •", paste0(FILE_PREFIX, "_umap_panel3.{pdf,svg} (donor + study group + senescence)"), "\n")
cat("    •", paste0(FILE_PREFIX, "_umap_cellcycle.{pdf,svg} (cell cycle)"), "\n")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# MULTI-RESOLUTION UMAP PANEL
# ════════════════════════════════════════════════════════════════════════════════

cat("\n", paste(rep("=", 80), collapse = ""), "\n")
cat("MULTI-RESOLUTION UMAPs\n")
cat(paste(rep("=", 80), collapse = ""), "\n")

res_to_plot <- c(0.1, 0.2, 0.4, 0.6, 0.8, 1.0, 1.2)

cat("  Resolutions:", paste(res_to_plot, collapse = ", "), "\n")
cat("  Current resolution:", CLUSTERING_RESOLUTION, "\n")

# ─────────────────────────────────────────────────────────────────────────────────
# Generate one UMAP per resolution
# ─────────────────────────────────────────────────────────────────────────────────

umap_list <- lapply(res_to_plot, function(res) {
    col_name <- paste0("RNA_snn_res.", res)
    n_cl <- length(unique(seurat_obj@meta.data[[col_name]]))
    
    title_label <- sprintf("res = %.1f (%d clusters)", res, n_cl)
    if (res == CLUSTERING_RESOLUTION) title_label <- paste0(title_label, " *")
    
    p <- DimPlot(seurat_obj, group.by = col_name, reduction = UMAP_RED,
            label = TRUE, label.size = 2.5, pt.size = 0.05, repel = TRUE) +
        ggtitle(title_label) +
        umap_theme_noleg +
        theme(plot.title = element_text(size = 10, face = "bold",
                                         color = ifelse(res == CLUSTERING_RESOLUTION,
                                                        "#E15759", "black")))
    add_corner_arrows(p, label_size = 2)
})

# ─────────────────────────────────────────────────────────────────────────────────
# Add state reference UMAP
# ─────────────────────────────────────────────────────────────────────────────────

p_state_ref <- DimPlot(seurat_obj, group.by = SUBCLUSTER_COL, reduction = UMAP_RED,
                        cols = state_colors, pt.size = 0.05) +
    ggtitle("Annotated States (reference)") +
    umap_theme +
    theme(plot.title = element_text(size = 10, face = "bold", color = "#4E79A7"),
          legend.text = element_text(size = 8),
          legend.key.size = unit(0.3, "cm"))
p_state_ref <- add_corner_arrows(p_state_ref, label_size = 2)

umap_list <- c(umap_list, list(p_state_ref))

# ─────────────────────────────────────────────────────────────────────────────────
# Combine
# ─────────────────────────────────────────────────────────────────────────────────

n_plots <- length(umap_list)
n_cols <- 4
n_rows <- ceiling(n_plots / n_cols)

options(repr.plot.width = 16, repr.plot.height = 4.5 * n_rows)

p_multi_res <- cowplot::plot_grid(plotlist = umap_list, ncol = n_cols)

# Add title
title_grob <- cowplot::ggdraw() +
    cowplot::draw_label(
        paste0(toupper(cell_type_label), ": Clustering Across Resolutions (* = current)"),
        size = 14, fontface = "bold"
    )

p_multi_res <- cowplot::plot_grid(title_grob, p_multi_res, ncol = 1, rel_heights = c(0.05, 1))

print(p_multi_res)

ggsave(file.path(VAL_FIGURES_DIR, paste0(FILE_PREFIX, "_umap_multi_resolution.pdf")),
       p_multi_res, width = 16, height = 4.5 * n_rows)
ggsave(file.path(VAL_FIGURES_DIR, paste0(FILE_PREFIX, "_umap_multi_resolution.svg")),
       p_multi_res, width = 16, height = 4.5 * n_rows)

cat("\n  ✓ Saved:", paste0(FILE_PREFIX, "_umap_multi_resolution.{pdf,svg}"), "\n")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# MULTI-RESOLUTION UMAP PANEL (Clusters + Majority-State side by side)
# ════════════════════════════════════════════════════════════════════════════════

cat("\n", paste(rep("=", 80), collapse = ""), "\n")
cat("MULTI-RESOLUTION UMAPs\n")
cat(paste(rep("=", 80), collapse = ""), "\n")

# Auto-detect available resolutions
available_res <- grep("RNA_snn_res\\.", colnames(seurat_obj@meta.data), value = TRUE)
res_to_plot <- sort(as.numeric(gsub("RNA_snn_res\\.", "", available_res)))

cat("  Available resolutions:", paste(res_to_plot, collapse = ", "), "\n")
cat("  Current resolution:", CLUSTERING_RESOLUTION, "\n")

# ─────────────────────────────────────────────────────────────────────────────────
# Generate paired UMAPs per resolution
# ─────────────────────────────────────────────────────────────────────────────────

umap_list <- list()
state_summary_list <- list()

for (res in res_to_plot) {
    col_name <- paste0("RNA_snn_res.", res)
    n_cl <- length(unique(seurat_obj@meta.data[[col_name]]))
    is_current <- (res == CLUSTERING_RESOLUTION)
    title_color <- ifelse(is_current, "#E15759", "black")
    
    # --- Compute majority state per cluster at this resolution ---
    majority_col <- paste0("majority_state_res_", gsub("\\.", "_", as.character(res)))
    
    cluster_to_state <- seurat_obj@meta.data %>%
        group_by(across(all_of(col_name)), across(all_of(SUBCLUSTER_COL))) %>%
        summarise(n = n(), .groups = "drop") %>%
        group_by(across(all_of(col_name))) %>%
        slice_max(n, n = 1, with_ties = FALSE) %>%
        ungroup()
    
    # Build lookup: cluster -> majority state
    state_lookup <- setNames(cluster_to_state[[SUBCLUSTER_COL]], 
                              as.character(cluster_to_state[[col_name]]))
    
    # Assign majority state to each cell
    seurat_obj@meta.data[[majority_col]] <- state_lookup[as.character(seurat_obj@meta.data[[col_name]])]
    
    # Count unique states at this resolution
    n_states <- length(unique(seurat_obj@meta.data[[majority_col]]))
    
    # Track for summary table
    state_summary_list[[as.character(res)]] <- data.frame(
        Resolution = res,
        n_clusters = n_cl,
        n_states = n_states,
        states = paste(sort(unique(seurat_obj@meta.data[[majority_col]])), collapse = ", ")
    )
    
    # --- Cluster UMAP ---
    cluster_label <- sprintf("res = %.1f (%d clusters)", res, n_cl)
    if (is_current) cluster_label <- paste0(cluster_label, " *")
    
    p_cl <- DimPlot(seurat_obj, group.by = col_name, reduction = UMAP_RED,
                     label = TRUE, label.size = 2.5, pt.size = 0.05, repel = TRUE) +
        ggtitle(cluster_label) +
        umap_theme_noleg +
        theme(plot.title = element_text(size = 10, face = "bold", color = title_color))
    p_cl <- add_corner_arrows(p_cl, label_size = 2)
    
    # --- Majority State UMAP ---
    p_st <- DimPlot(seurat_obj, group.by = majority_col, reduction = UMAP_RED,
                     cols = state_colors, pt.size = 0.05) +
        ggtitle(sprintf("Majority states (%d)", n_states)) +
        umap_theme_noleg +
        theme(plot.title = element_text(size = 10, face = "bold", color = title_color))
    p_st <- add_corner_arrows(p_st, label_size = 2)
    
    umap_list <- c(umap_list, list(p_cl, p_st))
}

# ─────────────────────────────────────────────────────────────────────────────────
# Print summary table
# ─────────────────────────────────────────────────────────────────────────────────

state_summary <- do.call(rbind, state_summary_list)
cat("\n--- State counts per resolution ---\n")
print(state_summary[, c("Resolution", "n_clusters", "n_states")], row.names = FALSE)

write.csv(state_summary, 
          file.path(VAL_RESULTS_DIR, paste0(FILE_PREFIX, "_multires_state_summary.csv")),
          row.names = FALSE)

# ─────────────────────────────────────────────────────────────────────────────────
# Shared state legend
# ─────────────────────────────────────────────────────────────────────────────────

p_legend_src <- DimPlot(seurat_obj, group.by = SUBCLUSTER_COL, reduction = UMAP_RED,
                         cols = state_colors, pt.size = 0.5) +
    theme_void() +
    theme(
        legend.position = "right",
        legend.title = element_text(size = 10, face = "bold"),
        legend.text = element_text(size = 8),
        legend.key.size = unit(0.4, "cm")
    ) +
    labs(color = paste0(tools::toTitleCase(cell_type_label), " State")) +
    guides(color = guide_legend(ncol = 1, override.aes = list(size = 3)))

legend_grob <- cowplot::get_legend(p_legend_src)

# ─────────────────────────────────────────────────────────────────────────────────
# Combine
# ─────────────────────────────────────────────────────────────────────────────────

n_cols <- 4  # 2 pairs per row
n_rows <- ceiling(length(umap_list) / n_cols)

options(repr.plot.width = 18, repr.plot.height = 4.5 * n_rows)

umap_grid <- cowplot::plot_grid(plotlist = umap_list, ncol = n_cols)

umap_with_legend <- cowplot::plot_grid(umap_grid, legend_grob, 
                                        ncol = 2, rel_widths = c(1, 0.15))

title_grob <- cowplot::ggdraw() +
    cowplot::draw_label(
        paste0(toupper(cell_type_label), 
               ": Clustering & Majority States Across Resolutions (* = current)"),
        size = 14, fontface = "bold"
    )

p_multi_res <- cowplot::plot_grid(title_grob, umap_with_legend, 
                                   ncol = 1, rel_heights = c(0.05, 1))

print(p_multi_res)

ggsave(file.path(VAL_FIGURES_DIR, paste0(FILE_PREFIX, "_umap_multi_resolution.pdf")),
       p_multi_res, width = 18, height = 4.5 * n_rows)
ggsave(file.path(VAL_FIGURES_DIR, paste0(FILE_PREFIX, "_umap_multi_resolution.svg")),
       p_multi_res, width = 18, height = 4.5 * n_rows)

cat("\n  ✓ Saved:", paste0(FILE_PREFIX, "_umap_multi_resolution.{pdf,svg}"), "\n")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# OVERALL VALIDATION SUMMARY
# ════════════════════════════════════════════════════════════════════════════════

mt_threshold <- 0.15
MT_REMOVE_THRESHOLD <- 0.25

# ── Sanity checks: make sure all variables are from the same run ──
cat("\n▸ Pre-summary sanity checks...\n")

# Verify cluster_order matches seurat_clusters
actual_clusters <- as.character(sort(as.numeric(as.character(unique(seurat_obj$seurat_clusters)))))
if (!identical(cluster_order, actual_clusters)) {
    cat("  ⚠ cluster_order mismatch — rebuilding from seurat_obj\n")
    cluster_order <- actual_clusters
    n_clusters <- length(cluster_order)
}

# Verify qc_by_cluster matches
if (exists("qc_by_cluster") && !all(cluster_order %in% qc_by_cluster$Cluster)) {
    missing <- cluster_order[!cluster_order %in% qc_by_cluster$Cluster]
    cat("  ⚠ qc_by_cluster missing clusters:", paste(missing, collapse = ", "), "\n")
    cat("  → Rerun Step 1 before Summary\n")
}

# Verify cluster_comp_summary matches
if (exists("cluster_comp_summary") && !all(cluster_order %in% cluster_comp_summary$Cluster)) {
    missing <- cluster_order[!cluster_order %in% cluster_comp_summary$Cluster]
    cat("  ⚠ cluster_comp_summary missing clusters:", paste(missing, collapse = ", "), "\n")
    cat("  → Rerun Step 3 before Summary\n")
}

# Verify flagged_clusters are valid
if (exists("flagged_clusters") && length(flagged_clusters) > 0) {
    invalid <- flagged_clusters[!flagged_clusters %in% cluster_order]
    if (length(invalid) > 0) {
        cat("  ⚠ flagged_clusters contains invalid clusters:", paste(invalid, collapse = ", "), "\n")
        cat("  → Filtering to valid clusters only\n")
        flagged_clusters <- flagged_clusters[flagged_clusters %in% cluster_order]
    }
}

# Ensure cluster_purity exists
if (!exists("cluster_purity") || length(cluster_purity) == 0) {
    cluster_purity <- tapply(
        seurat_obj@meta.data[[SUBCLUSTER_COL]],
        seurat_obj$seurat_clusters,
        function(x) max(table(x)) / length(x) * 100
    )
}

cat("  ✓ Using", n_clusters, "clusters:", paste(cluster_order, collapse = ", "), "\n")

# ════════════════════════════════════════════════════════════════════════════════

cat("\n", paste(rep("=", 80), collapse = ""), "\n")
cat(sprintf("VALIDATION SUMMARY: %s (%s)\n", toupper(cell_type_label), DATASET))
cat(paste(rep("=", 80), collapse = ""), "\n")

# ── Object ──
cat("\n-- Object --\n")
cat("  Cells:", ncol(seurat_obj), "\n")
cat("  Clusters:", n_clusters, "\n")
cat("  States:", n_states, "(", paste(state_order, collapse = ", "), ")\n")
cat("  Donors:", n_donors, "\n")

# ── Step 1: QC Metrics ──
cat("\n-- Step 1: QC Metrics --\n")
cat("  Goal: Identify clusters driven by low-quality cells (high MT%, low genes/UMI)\n")
cat("  Rationale: Clusters enriched for damaged or dying cells produce spurious\n")
cat("  transcriptional signatures. For postmortem brain snRNA-seq, MT% is generally\n")
cat("  higher than other tissues due to nuclear isolation; moderate elevations are\n")
cat("  expected but extreme values (>0.3) still indicate damaged nuclei.\n")
high_mt <- qc_by_cluster[qc_by_cluster$median_mt > mt_threshold, ]
if (nrow(high_mt) > 0) {
    cat("  Result:", nrow(high_mt), "cluster(s) with elevated MT (>", mt_threshold, "):\n")
    for (i in 1:nrow(high_mt)) {
        severity <- ifelse(high_mt$median_mt[i] > 0.3, "SEVERE", 
                    ifelse(high_mt$median_mt[i] > 0.2, "MODERATE", "MILD"))
        cat(sprintf("    Cluster %s (%s): median MT = %.3f [%s], UMI = %s, genes = %s\n",
                    high_mt$Cluster[i], high_mt$State[i], high_mt$median_mt[i], severity,
                    format(high_mt$median_UMI[i], big.mark = ","),
                    format(high_mt$median_genes[i], big.mark = ",")))
    }
} else {
    cat("  Result: PASS -- No clusters with elevated MT (>", mt_threshold, ")\n")
}

# ── Step 2: Canonical Markers ──
cat("\n-- Step 2: Canonical Markers --\n")
cat("  Goal: Confirm cell type identity and detect contamination from other lineages\n")
cat("  Rationale: Every cluster should express known", cell_type_label, "markers.\n")
cat("  High expression of markers from other cell types indicates doublets or\n")
cat("  mis-classified cells.\n")
low_identity <- flag_df[flag_df$mean_identity < 0.5, ]
high_contam <- flag_df[flag_df$mean_contam > 0.5, ]
if (nrow(low_identity) > 0) {
    cat("  Identity: FAIL -- Low identity clusters:\n")
    for (i in 1:nrow(low_identity)) {
        cat(sprintf("    Cluster %s (%s): mean identity = %.2f\n",
                    low_identity$Cluster[i], low_identity$State[i], low_identity$mean_identity[i]))
    }
} else {
    cat("  Identity: PASS -- All clusters express", cell_type_label, "markers\n")
}
if (nrow(high_contam) > 0) {
    cat("  Contamination: FAIL -- High contamination clusters:\n")
    for (i in 1:nrow(high_contam)) {
        cat(sprintf("    Cluster %s (%s): mean contam = %.2f, top negative = %s (%.2f)\n",
                    high_contam$Cluster[i], high_contam$State[i], high_contam$mean_contam[i],
                    high_contam$max_neg_gene[i], high_contam$max_neg_val[i]))
    }
} else {
    cat("  Contamination: PASS -- No clusters with high off-target signal\n")
}

# ── Step 3: Sample Composition ──
cat("\n-- Step 3: Sample Composition --\n")
cat("  Goal: Detect clusters dominated by a single donor (batch/sample artifacts)\n")
cat("  Rationale: Clusters driven by one donor reflect individual-specific effects,\n")
cat("  not generalizable biology. If a state is defined primarily by such clusters,\n")
cat("  the state itself may be a donor artifact.\n")
if (length(flagged_clusters) > 0) {
    cat("  Flagged clusters:", paste(flagged_clusters, collapse = ", "), "\n")
    for (cl in flagged_clusters) {
        cl_row <- cluster_comp_summary[cluster_comp_summary$Cluster == cl, ]
        if (nrow(cl_row) > 0) {
            cat(sprintf("    Cluster %s (%s, n=%d): top donor = %.1f%%, evenness = %.2f\n",
                        cl, cl_row$State, cl_row$N_cells, cl_row$Top1_pct, cl_row$Evenness))
        }
    }
} else {
    cat("  Result: PASS -- No clusters with single-donor dominance\n")
}

# State robustness
if (exists("state_robustness") && length(flagged_clusters) > 0) {
    at_risk <- state_robustness[state_robustness$Donors_clean < 5 | 
                                 state_robustness$Pct_retained < 50 |
                                 state_robustness$Evenness < 0.5, ]
    if (nrow(at_risk) > 0) {
        cat("  State robustness: WARNING\n")
        for (i in 1:nrow(at_risk)) {
            reasons <- c()
            if (at_risk$Donors_clean[i] < 5) reasons <- c(reasons, 
                sprintf("%d donors remain", at_risk$Donors_clean[i]))
            if (at_risk$Pct_retained[i] < 50) reasons <- c(reasons, 
                sprintf("%.1f%% cells retained", at_risk$Pct_retained[i]))
            if (at_risk$Evenness[i] < 0.5) reasons <- c(reasons, 
                sprintf("evenness %.2f", at_risk$Evenness[i]))
            cat(sprintf("    %s: %s\n", at_risk$State[i], paste(reasons, collapse = ", ")))
        }
    } else {
        cat("  State robustness: PASS -- All states retain multi-donor support\n")
    }
}

# State resolution stability
if (exists("state_by_res") && nrow(state_by_res) > 0) {
    cat("  State resolution stability:\n")
    for (st in state_order) {
        st_data <- state_by_res[state_by_res$State == st, ]
        if (nrow(st_data) == 0) next
        min_rec <- min(st_data$Recovery_pct)
        low_recovery <- st_data[st_data$Recovery_pct < 50, ]
        
        if (nrow(low_recovery) > 0) {
            low_res <- low_recovery$Resolution
            chosen_rec <- st_data$Recovery_pct[st_data$Resolution == CLUSTERING_RESOLUTION]
            if (length(chosen_rec) == 0) chosen_rec <- NA
            
            all_below_chosen <- all(low_res < CLUSTERING_RESOLUTION)
            at_and_above <- st_data[st_data$Resolution >= CLUSTERING_RESOLUTION, ]
            stable_above <- all(at_and_above$Recovery_pct >= 70)
            
            if (all_below_chosen && stable_above && !is.na(chosen_rec) && chosen_rec >= 70) {
                first_stable <- min(st_data$Resolution[st_data$Recovery_pct >= 50])
                cat(sprintf("    %s: stable at working resolutions (emerges at res %.1f, min %.1f%% at coarse res)\n",
                            st, first_stable, min_rec))
            } else if (!is.na(chosen_rec) && chosen_rec >= 70) {
                cat(sprintf("    %s: transitional -- <50%% at res %s, but %.1f%% at chosen res\n",
                            st, paste(low_res, collapse = ", "), chosen_rec))
            } else {
                cat(sprintf("    %s: UNSTABLE -- %.1f%% at chosen res, <50%% at res %s\n",
                            st, ifelse(is.na(chosen_rec), NA, chosen_rec),
                            paste(low_res, collapse = ", ")))
            }
        } else {
            cat(sprintf("    %s: stable across all resolutions (min recovery: %.1f%%)\n",
                        st, min_rec))
        }
    }
}

# ── Step 4: Cell Cycle ──
cat("\n-- Step 4: Cell Cycle --\n")
cat("  Goal: Identify clusters driven by cell cycle state rather than biology\n")
cat("  Rationale: If a cluster is enriched for S/G2M phase relative to others, its\n")
cat("  transcriptional identity may reflect proliferation rather than a distinct\n")
cat("  cell state. Uniform cycling across clusters is not concerning. Note: snRNA-seq\n")
cat("  from brain tissue often shows high overall cycling scores because nuclear RNA\n")
cat("  captures cell cycle transcripts even in post-mitotic cells.\n")
overall_cycling <- round(sum(seurat_obj$Phase %in% c("S", "G2M")) / ncol(seurat_obj) * 100, 1)
cat("  Overall cycling (S+G2M):", overall_cycling, "%\n")
if (flagged_cc) {
    n_flagged_cc <- sum(phase_wide$Cycling > 30)
    cycling_range <- range(phase_wide$Cycling)
    cat("  Clusters >30% cycling:", n_flagged_cc, "/", nrow(phase_wide), "\n")
    cat("  Cycling range:", cycling_range[1], "-", cycling_range[2], "%\n")
    if ((cycling_range[2] - cycling_range[1]) < 20) {
        cat("  Result: PASS -- Uniformly distributed, not a cluster-specific artifact\n")
    } else {
        cat("  Result: WARNING -- Variable across clusters:\n")
        top3 <- phase_wide[order(-phase_wide$Cycling), ][1:min(3, nrow(phase_wide)), ]
        for (i in 1:nrow(top3)) {
            cat(sprintf("    Cluster %s (%s): %.1f%%\n",
                        top3$Cluster[i], top3$State[i], top3$Cycling[i]))
        }
    }
} else {
    cat("  Result: PASS -- No clusters with >30% cycling\n")
}

# ── Step 5: Cluster Stability ──
cat("\n-- Step 5: Cluster Stability --\n")
cat("  Goal: Assess whether clusters and states are robust across resolutions\n")
cat("  Rationale: Stable clusters appear as clean, single-path branches in the\n")
cat("  clustree (cell flow across resolutions). Unstable clusters show messy splits\n")
cat("  where cells scatter to multiple destinations, indicating over-clustering.\n")
cat("  State purity measures how well each cluster maps to a single annotated state;\n")
cat("  low purity suggests boundary clusters between two states.\n")
cat("  Resolution:", CLUSTERING_RESOLUTION, "->", n_clusters, "clusters\n")

current_res_row <- if (exists("stability_df")) {
    stability_df[stability_df$Resolution == CLUSTERING_RESOLUTION, ]
} else {
    data.frame()
}

if (nrow(current_res_row) > 0) {
    cat("  Mean state purity:", current_res_row$Mean_purity, "%\n")
    cat("  Min state purity:", current_res_row$Min_purity, "%\n")
    cat("  Mixed clusters (<70% purity):", current_res_row$N_mixed, "\n")
} else {
    # Compute purity against original seurat_clusters
    orig_purity <- tapply(
        seurat_obj@meta.data[[SUBCLUSTER_COL]],
        seurat_obj$seurat_clusters,
        function(x) max(table(x)) / length(x) * 100
    )
    cat("  Mean state purity:", round(mean(orig_purity), 1), "%\n")
    cat("  Min state purity:", round(min(orig_purity), 1), "%\n")
    cat("  Mixed clusters (<70% purity):", sum(orig_purity < 70), "\n")
}

purity_sorted <- sort(cluster_purity)
low_purity <- purity_sorted[purity_sorted < 70]
if (length(low_purity) > 0) {
    cat("  Low purity clusters:\n")
    for (cl in names(low_purity)) {
        cl_cells <- as.character(seurat_obj$seurat_clusters) == cl
        if (sum(cl_cells) == 0) next
        cl_state_table <- sort(table(seurat_obj@meta.data[[SUBCLUSTER_COL]][cl_cells]), decreasing = TRUE)
        cl_state_pcts <- round(cl_state_table / sum(cl_state_table) * 100, 1)
        top2 <- paste(sprintf("%s (%.1f%%)", names(cl_state_pcts)[1:min(2, length(cl_state_pcts))],
                              cl_state_pcts[1:min(2, length(cl_state_pcts))]), collapse = " / ")
        cat(sprintf("    Cluster %s: %.1f%% purity -- %s\n", cl, low_purity[cl], top2))
    }
    cat("  Note: Low purity clusters are typically transition zones between states\n")
    cat("  and do not require action unless they distort downstream results.\n")
} else {
    cat("  All clusters >70% purity\n")
}

if (exists("stability_df")) {
    stable_range <- stability_df[stability_df$Mean_purity > 85, ]
    if (nrow(stable_range) > 0) {
        cat("  Stable resolution range (>85% mean purity): ",
            min(stable_range$Resolution), " - ", max(stable_range$Resolution), "\n", sep = "")
    }
}

# ── Actionable Findings ──
cat("\n", paste(rep("-", 80), collapse = ""), "\n")
cat("ACTIONABLE FINDINGS\n")
cat(paste(rep("-", 80), collapse = ""), "\n")

remove_list <- c()
monitor_list <- c()

for (cl in flagged_clusters) {
    mt_row <- which(qc_by_cluster$Cluster == cl)
    mt_val <- if (length(mt_row) > 0) qc_by_cluster$median_mt[mt_row] else NA
    
    purity_val <- if (cl %in% names(cluster_purity)) cluster_purity[cl] else NA
    
    comp_row <- which(cluster_comp_summary$Cluster == cl)
    top_donor <- if (length(comp_row) > 0) cluster_comp_summary$Top1_pct[comp_row] else NA
    
    is_severe_mt <- !is.na(mt_val) && mt_val > MT_REMOVE_THRESHOLD
    is_low_purity <- !is.na(purity_val) && purity_val < 60
    is_donor_dominated <- !is.na(top_donor) && top_donor > 80
    is_elevated_mt <- !is.na(mt_val) && mt_val > mt_threshold
    
    if (is_severe_mt || (is_donor_dominated && (is_elevated_mt || is_low_purity))) {
        remove_list <- c(remove_list, cl)
    } else {
        monitor_list <- c(monitor_list, cl)
    }
}

if (length(remove_list) > 0) {
    cat("\n  REMOVE:\n")
    for (cl in remove_list) {
        mt_row <- which(qc_by_cluster$Cluster == cl)
        comp_row <- which(cluster_comp_summary$Cluster == cl)
        cl_state <- if (length(mt_row) > 0) qc_by_cluster$State[mt_row] else "unknown"
        cl_n <- if (length(mt_row) > 0) qc_by_cluster$N[mt_row] else NA
        cl_mt <- if (length(mt_row) > 0) qc_by_cluster$median_mt[mt_row] else NA
        cl_top <- if (length(comp_row) > 0) cluster_comp_summary$Top1_pct[comp_row] else NA
        
        reasons <- c()
        if (!is.na(cl_mt) && cl_mt > MT_REMOVE_THRESHOLD) {
            reasons <- c(reasons, sprintf("severe MT (%.3f)", cl_mt))
        }
        if (!is.na(cl_top) && cl_top > 80) {
            reasons <- c(reasons, sprintf("donor dominated (%.1f%%)", cl_top))
        }
        if (length(reasons) == 0) reasons <- "multiple flags"
        cat(sprintf("    Cluster %s (%s, n=%s): %s\n",
                    cl, cl_state, ifelse(is.na(cl_n), "?", cl_n),
                    paste(reasons, collapse = " + ")))
    }
} else {
    cat("\n  REMOVE: None\n")
}

if (length(monitor_list) > 0) {
    cat("\n  MONITOR:\n")
    for (cl in monitor_list) {
        mt_row <- which(qc_by_cluster$Cluster == cl)
        comp_row <- which(cluster_comp_summary$Cluster == cl)
        cl_state <- if (length(mt_row) > 0) qc_by_cluster$State[mt_row] else "unknown"
        cl_n <- if (length(mt_row) > 0) qc_by_cluster$N[mt_row] else NA
        cl_top <- if (length(comp_row) > 0) cluster_comp_summary$Top1_pct[comp_row] else NA
        cl_eve <- if (length(comp_row) > 0) cluster_comp_summary$Evenness[comp_row] else NA
        
        reasons <- c()
        if (!is.na(cl_top) && cl_top > DONOR_DOMINANCE_THRESHOLD) {
            reasons <- c(reasons, sprintf("top donor %.1f%%", cl_top))
        }
        if (!is.na(cl_eve) && cl_eve < EVENNESS_FLAG_THRESHOLD) {
            reasons <- c(reasons, sprintf("evenness %.2f", cl_eve))
        }
        if (length(reasons) == 0) reasons <- "flagged in composition"
        cat(sprintf("    Cluster %s (%s, n=%s): %s\n",
                    cl, cl_state, ifelse(is.na(cl_n), "?", cl_n),
                    paste(reasons, collapse = ", ")))
    }
    cat("  Action: verify DEG results from these clusters are not single-donor driven\n")
} else {
    cat("\n  MONITOR: None\n")
}

n_pass <- n_clusters - length(remove_list) - length(monitor_list)
cat("\n  PASS:", n_pass, "clusters\n")

# ── Recommendation ──
cat("\n", paste(rep("-", 80), collapse = ""), "\n")
cat("RECOMMENDATION\n")
cat(paste(rep("-", 80), collapse = ""), "\n")
rec_num <- 1
if (length(remove_list) > 0) {
    affected_states <- c()
    for (cl in remove_list) {
        mt_row <- which(qc_by_cluster$Cluster == cl)
        if (length(mt_row) > 0) affected_states <- c(affected_states, qc_by_cluster$State[mt_row])
    }
    affected_states <- unique(affected_states)
    cat(sprintf("  %d. Remove cluster(s) %s and re-evaluate %s state annotation\n",
                rec_num, paste(remove_list, collapse = ", "),
                paste(affected_states, collapse = ", ")))
    rec_num <- rec_num + 1
}
if (length(monitor_list) > 0) {
    cat(sprintf("  %d. Verify DEG results for cluster(s) %s are not single-donor driven\n",
                rec_num, paste(monitor_list, collapse = ", ")))
    rec_num <- rec_num + 1
}
if (nrow(current_res_row) > 0) {
    cat(sprintf("  %d. Resolution %s is appropriate (%d clusters, mean purity %.1f%%)\n",
                rec_num, CLUSTERING_RESOLUTION, n_clusters, current_res_row$Mean_purity))
} else {
    cat(sprintf("  %d. Original clustering (%d clusters) used for annotation\n",
                rec_num, n_clusters))
}
rec_num <- rec_num + 1
cat(sprintf("  %d. All %d annotated states are biologically supported\n", rec_num, n_states))

cat("\n", paste(rep("=", 80), collapse = ""), "\n")
cat("VALIDATION COMPLETE\n")
cat(paste(rep("=", 80), collapse = ""), "\n")

# ─────────────────────────────────────────────────────────────────────────────────
# List all validation outputs
# ─────────────────────────────────────────────────────────────────────────────────

cat("\n-- All Validation Files --\n")

cat("\n  Results (", VAL_RESULTS_DIR, "):\n", sep = "")
val_csv <- list.files(VAL_RESULTS_DIR, pattern = "\\.csv$", full.names = FALSE)
for (f in val_csv) cat("    -", f, "\n")

cat("\n  Figures (", VAL_FIGURES_DIR, "):\n", sep = "")
val_figs <- list.files(VAL_FIGURES_DIR, pattern = "\\.(pdf|svg)$", full.names = FALSE)
for (f in val_figs) cat("    -", f, "\n")

# Module 04F: OPC Subclustering & Annotation

## Overview
Subcluster OPCs into functional states using literature-derived gene signatures, then characterize senescence patterns across states and study groups.

## OPC Subclustering Notes

### Annotation: Cluster-Level, Z-Scored

Same approach as astrocytes. Clusters scored via AddModuleScore, z-scored across clusters, assigned to highest z-scored state.

### 3 States (Cross-Species Consensus)

- Homeostatic: PDGFRA, CSPG4, PTPRZ1, BCAN, PCDH15 — quiescent progenitors
- COP: GPR17, BCAS1, NEU4, TNS3, FYN — committed oligodendrocyte precursors
- Reactive: HLA-A/B/C, B2M, CD74, SERPINA3 — immune/antigen-presenting OPCs

### Workflow

Subset → QC (doublet removal) → Harmony (Cohort) → t-SNE/UMAP → Cluster → AddModuleScore → Z-score per cluster → Assign states → Validate → GLMM

In [ ]:
cat("\n================================================================================\n")
cat("04F: OPC SUBCLUSTERING & ANNOTATION\n")
cat("================================================================================\n")

# ════════════════════════════════════════════════════════════════════════════════
# LOAD LIBRARIES
# ════════════════════════════════════════════════════════════════════════════════

suppressPackageStartupMessages({
    library(Seurat)
    library(Matrix)
    library(dplyr)
    library(tidyr)
    library(ggplot2)
    library(qs)
    library(pheatmap)
    library(patchwork)
    library(RColorBrewer)
})

cat("\n✓ Libraries loaded\n")

# ════════════════════════════════════════════════════════════════════════════════
# DATASET SELECTION
# ════════════════════════════════════════════════════════════════════════════════

DATASET <- "psychad_aging"  # Options: 'psychad_aging', 'psychad_ad', 'psychencode', 'mathys'

# ════════════════════════════════════════════════════════════════════════════════
# DATASET-SPECIFIC CONFIGURATION
# ════════════════════════════════════════════════════════════════════════════════

DATASET_CONFIG <- list(
    'psychad_aging' = list(
        cell_type_col = 'subclass',
        donor_col = 'Sample',
        study_group_col = 'Study_Group',
        sex_col = 'Sex',
        cohort_col = 'Cohort',
        study_type = 'aging',
        primary_var = 'Age',
        primary_var_type = 'continuous',
        covariates = c('Sex', 'Cohort'),
        group_order = c('Age_20_29', 'Age_30_39', 'Age_40_49', 'Age_50_59', 
                        'Age_60_69', 'Age_70_79', 'Age_80_100')
    ),
    'psychad_ad' = list(
        cell_type_col = 'subclass',
        donor_col = 'Sample',
        study_group_col = 'Study_Group',
        sex_col = 'Sex',
        cohort_col = 'Cohort',
        study_type = 'disease',
        primary_var = 'Study_Group',
        primary_var_type = 'categorical',
        reference_group = 'Control',
        covariates = c('Age', 'Sex', 'Cohort'),
        group_order = c('Control', 'MCI', 'AD')
    ),
    'psychencode' = list(
        cell_type_col = 'major_celltype',
        donor_col = 'sample_id',
        study_group_col = 'Study_Group',
        sex_col = 'Biological_Sex',
        cohort_col = 'Batch',
        study_type = 'aging',
        primary_var = 'Age_death',
        primary_var_type = 'continuous',
        covariates = c('Sex', 'Batch'),
        group_order = c('Age_20_29', 'Age_30_39', 'Age_40_49', 'Age_50_59', 
                        'Age_60_69', 'Age_70_79', 'Age_80_100')
    ),
    'mathys' = list(
        cell_type_col = 'broad.cell.type',
        donor_col = 'Subject',
        study_group_col = 'Study_Group',
        sex_col = 'sex',
        cohort_col = 'batch',
        study_type = 'disease',
        primary_var = 'Study_Group',
        primary_var_type = 'categorical',
        reference_group = 'NCI',
        covariates = c('Age', 'sex', 'batch'),
        group_order = c('NCI', 'MCI', 'AD')
    )
)

config <- DATASET_CONFIG[[DATASET]]

# ─────────────────────────────────────────────────────────────────────────────
# COLUMN NAMES
# ─────────────────────────────────────────────────────────────────────────────

CELL_TYPE_COL <- config$cell_type_col
DONOR_COL <- config$donor_col
STUDY_GROUP_COL <- config$study_group_col
SEX_COL <- config$sex_col
COHORT_COL <- config$cohort_col
SENESCENCE_LABEL_COL <- "senescence_label"

# ─────────────────────────────────────────────────────────────────────────────
# STUDY DESIGN PARAMETERS
# ─────────────────────────────────────────────────────────────────────────────

STUDY_TYPE <- config$study_type
PRIMARY_VAR <- config$primary_var
PRIMARY_VAR_TYPE <- config$primary_var_type
COVARIATES <- config$covariates
GROUP_ORDER <- config$group_order

# Processing parameters
N_VARIABLE_FEATURES <- 2000
N_PCS <- 50
N_DIMS_USE <- 30
CLUSTERING_RESOLUTION <- 0.7

# ════════════════════════════════════════════════════════════════════════════════
# OPC SIGNATURES
# ════════════════════════════════════════════════════════════════════════════════

OPC_SIGNATURES <- list(
    Homeostatic = c("PDGFRA", "CSPG4", "PTPRZ1", "BCAN", "PCDH15"),
    COP         = c("GPR17", "BCAS1", "NEU4", "TNS3", "FYN"),
    Reactive    = c("HLA-A", "HLA-B", "HLA-C", "B2M", "CD74", "C4B", "SERPINA3", "TLR3")
)

# ════════════════════════════════════════════════════════════════════════════════
# COLOR PALETTES
# ════════════════════════════════════════════════════════════════════════════════

STUDY_GROUP_COLORS <- c(
    'Age_20_29' = '#2E86AB',
    'Age_30_39' = '#4A90E2',
    'Age_40_49' = '#50C878',
    'Age_50_59' = '#FFB347',
    'Age_60_69' = '#FF8C00',
    'Age_70_79' = '#E24A4A',
    'Age_80_100' = '#8B0000',
    'Control' = '#4E79A7',
    'MCI' = '#F28E2B',
    'AD' = '#E15759',
    'NCI' = '#4E79A7'
)

study_colors <- STUDY_GROUP_COLORS[names(STUDY_GROUP_COLORS) %in% GROUP_ORDER]

SENESCENCE_COLORS <- c('Non-SnC' = '#D3D3D3', 'SnC' = '#C44E52')

SEX_COLORS <- c('Male' = '#4878CF', 'Female' = '#E97B8A')

# Base state palette (same colors for all cell types, assigned by position)
BASE_STATE_PALETTE <- c("#4E79A7", "#59A14F", "#E15759", "#F28E2B", "#EDC948",
                        "#76B7B2", "#FFBE7D", "#BAB0AC", "#B07AA1", "#FF9DA7",
                        "#A0CBE8", "#D37295", "#9C755F", "#8B0000")

get_state_colors <- function(states) {
    setNames(BASE_STATE_PALETTE[1:length(states)], states)
}

# ════════════════════════════════════════════════════════════════════════════════
# PATHS
# ════════════════════════════════════════════════════════════════════════════════

BASE_DIR <- "/fs/scratch/PAS2598/senescence_analysis"

INPUT_FILE <- file.path(BASE_DIR, "data", "04_subsetting", DATASET, paste0(DATASET, "_seurat.qs"))
OUTPUT_DIR <- file.path(BASE_DIR, "data", "04_opc", DATASET)
RESULTS_DIR <- file.path(BASE_DIR, "results", "04_opc", DATASET)
FIGURES_DIR <- file.path(BASE_DIR, "figures", "04_opc", DATASET)

dir.create(OUTPUT_DIR, recursive = TRUE, showWarnings = FALSE)
dir.create(RESULTS_DIR, recursive = TRUE, showWarnings = FALSE)
dir.create(FIGURES_DIR, recursive = TRUE, showWarnings = FALSE)

# ════════════════════════════════════════════════════════════════════════════════
# PRINT CONFIGURATION
# ════════════════════════════════════════════════════════════════════════════════

cat("\nConfiguration:\n")
cat("  Dataset:", DATASET, "\n")
cat("  Cell type column:", CELL_TYPE_COL, "\n")
cat("  Donor column:", DONOR_COL, "\n")
cat("  Study group column:", STUDY_GROUP_COL, "\n")
cat("  Sex column:", SEX_COL, "\n")
cat("  Cohort column:", COHORT_COL, "\n")
cat("  Study type:", STUDY_TYPE, "\n")
cat("  Primary variable:", PRIMARY_VAR, "(", PRIMARY_VAR_TYPE, ")\n")
cat("  Covariates:", paste(COVARIATES, collapse = ", "), "\n")
cat("  Group order:", paste(GROUP_ORDER, collapse = ", "), "\n")
cat("  Clustering resolution:", CLUSTERING_RESOLUTION, "\n")

cat("\nOPC Signatures:\n")
for (sig in names(OPC_SIGNATURES)) {
    cat(sprintf("  %s: %s\n", sig, paste(OPC_SIGNATURES[[sig]], collapse = ", ")))
}

cat("\nPaths:\n")
cat("  Input:", INPUT_FILE, "\n")
cat("  Output:", OUTPUT_DIR, "\n")
cat("  Results:", RESULTS_DIR, "\n")
cat("  Figures:", FIGURES_DIR, "\n")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# SUBSET OPCs FROM SHARED SEURAT OBJECT
# ════════════════════════════════════════════════════════════════════════════════

cat("\n================================================================================\n")
cat("SUBSETTING OPCs\n")
cat("================================================================================\n")

data <- qread(INPUT_FILE)
cat("Loaded:", ncol(data), "cells\n")

cat("\nCell types available:\n")
print(table(data[[CELL_TYPE_COL]]))

opc <- subset(data, cells = colnames(data)[data[[CELL_TYPE_COL]] == "OPC"])
cat("\nOPC subset:", ncol(opc), "cells\n")

rm(data); gc(verbose = FALSE)

# ════════════════════════════════════════════════════════════════════════════════
# STEP 1: QC FILTERING (POST-SUBSET)
# ════════════════════════════════════════════════════════════════════════════════

cat("\n================================================================================\n")
cat("STEP 1: QC FILTERING — OPC SUBSET\n")
cat("================================================================================\n")

cat("\nBefore filtering:", ncol(opc), "cells\n")

# Remove low-quality cells
opc <- subset(opc,
              nFeature_RNA > 200 &
              nFeature_RNA < quantile(opc$nFeature_RNA, 0.99) &
              nCount_RNA > 500 &
              nCount_RNA < quantile(opc$nCount_RNA, 0.99))
cat("After quality filter:", ncol(opc), "cells\n")

# Remove high mito
if ("percent.mt" %in% colnames(opc@meta.data)) {
    opc <- subset(opc, percent.mt < 10)
    cat("After mito filter:", ncol(opc), "cells\n")
}

# Remove doublets — cells expressing non-OPC markers
cat("\nScoring contamination from other cell types...\n")

contam_signatures <- list(
    neuron = c("RBFOX3", "SYT1", "SNAP25", "STMN2"),
    astro  = c("GFAP", "AQP4", "GLUL", "SLC1A2"),
    micro  = c("CSF1R", "P2RY12", "CX3CR1", "TMEM119"),
    oligo  = c("MBP", "MOG", "PLP1", "MAG"),
    endo   = c("CLDN5", "FLT1", "PECAM1", "VWF")
)

for (ct in names(contam_signatures)) {
    genes <- contam_signatures[[ct]][contam_signatures[[ct]] %in% rownames(opc)]
    if (length(genes) >= 2) {
        opc <- AddModuleScore(opc, features = list(genes), name = paste0(ct, "_contam"))
        cat(sprintf("  %s: %d/%d markers found\n", ct, length(genes), length(contam_signatures[[ct]])))
    }
}

contam_cols <- grep("_contam1$", colnames(opc@meta.data), value = TRUE)
if (length(contam_cols) > 0) {
    keep <- rep(TRUE, ncol(opc))
    for (col in contam_cols) {
        threshold <- quantile(opc@meta.data[[col]], 0.95)
        keep <- keep & (opc@meta.data[[col]] < threshold)
        cat(sprintf("  Removing top 5%% %s (threshold = %.3f)\n", col, threshold))
    }
    opc <- opc[, keep]
}

cat("\nAfter all QC:", ncol(opc), "cells\n")

# ════════════════════════════════════════════════════════════════════════════════
# STEP 2: REPROCESS
# ════════════════════════════════════════════════════════════════════════════════

library(harmony)

cat("\n================================================================================\n")
cat("STEP 2: REPROCESSING OPCs\n")
cat("================================================================================\n")

cat("\n1. Normalizing...\n")
opc <- NormalizeData(opc, verbose = FALSE)

cat("2. Finding variable features...\n")
opc <- FindVariableFeatures(opc, nfeatures = N_VARIABLE_FEATURES, verbose = FALSE)

cat("3. Scaling (regressing nCount_RNA)...\n")
opc <- ScaleData(opc,
                 features = VariableFeatures(opc),
                 vars.to.regress = "nCount_RNA",
                 verbose = FALSE)

cat("4. Running PCA...\n")
opc <- RunPCA(opc, npcs = N_PCS, verbose = FALSE)

cat("5. Running Harmony (", COHORT_COL, ", theta=2)...\n")
opc <- RunHarmony(
    opc,
    group.by.vars = COHORT_COL,
    theta = c(2),
    max_iter = 30,
    verbose = TRUE
)

cat("6. Running t-SNE...\n")
opc <- RunTSNE(opc, reduction = "harmony", dims = 1:N_DIMS_USE,
               perplexity = 20, verbose = FALSE)

cat("7. Running UMAP...\n")
opc <- RunUMAP(opc, reduction = "harmony", dims = 1:N_DIMS_USE,
               verbose = FALSE)

cat("8. Finding neighbors...\n")
opc <- FindNeighbors(opc, reduction = "harmony", dims = 1:N_DIMS_USE,
                     verbose = FALSE)

cat("9. Clustering (resolution =", CLUSTERING_RESOLUTION, ")...\n")
opc <- FindClusters(opc, resolution = CLUSTERING_RESOLUTION, verbose = FALSE)

n_clusters <- length(unique(Idents(opc)))
cat("\nReprocessing complete:", ncol(opc), "cells,", n_clusters, "clusters\n")

cat("\nCells per cluster:\n")
cluster_sizes <- sort(table(Idents(opc)), decreasing = TRUE)
for (cl in names(cluster_sizes)) {
    cat(sprintf("  Cluster %s: %d cells (%.1f%%)\n",
                cl, cluster_sizes[cl], cluster_sizes[cl] / ncol(opc) * 100))
}

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# MODULE SCORES
# ════════════════════════════════════════════════════════════════════════════════

cat("\n================================================================================\n")
cat("MODULE SCORES\n")
cat("================================================================================\n")

DefaultAssay(opc) <- "RNA"

# Filter signatures to available genes
signatures_filtered <- lapply(OPC_SIGNATURES, function(genes) {
    genes[genes %in% rownames(opc)]
})

cat("\nSignature genes available:\n")
for (sig in names(signatures_filtered)) {
    missing <- setdiff(OPC_SIGNATURES[[sig]], signatures_filtered[[sig]])
    cat(sprintf("  %s: %d/%d genes", sig, length(signatures_filtered[[sig]]),
                length(OPC_SIGNATURES[[sig]])))
    if (length(missing) > 0) cat(sprintf(" (missing: %s)", paste(missing, collapse = ", ")))
    cat("\n")
}

# Add module scores
for (i in seq_along(signatures_filtered)) {
    sig_name <- names(signatures_filtered)[i]
    sig_genes <- signatures_filtered[[sig_name]]

    if (length(sig_genes) < 2) {
        cat(sprintf("  %s: skipped (only %d gene)\n", sig_name, length(sig_genes)))
        next
    }

    opc <- AddModuleScore(opc, features = list(sig_genes),
                          name = paste0(sig_name, "_"), verbose = FALSE)

    score_col <- paste0(sig_name, "_1")

    cat(sprintf("  %s: %.4f +/- %.4f\n",
                sig_name,
                mean(opc@meta.data[[score_col]], na.rm = TRUE),
                sd(opc@meta.data[[score_col]], na.rm = TRUE)))
}

# Score columns: Homeostatic_1, COP_1, Reactive_1
score_cols <- paste0(names(signatures_filtered), "_1")
score_cols <- score_cols[score_cols %in% colnames(opc@meta.data)]

cat("\nScore columns:", paste(score_cols, collapse = ", "), "\n")

In [ ]:
get_state_colors <- function(states) {
    setNames(BASE_STATE_PALETTE[1:length(states)], states)
}
cat("✓ get_state_colors updated\n")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# ANNOTATE OPCs (CLUSTER-LEVEL, Z-SCORED)
# ════════════════════════════════════════════════════════════════════════════════

cat("\n================================================================================\n")
cat("ANNOTATING OPCs (Cluster-Level, Z-Scored Assignment)\n")
cat("================================================================================\n")

# ════════════════════════════════════════════════════════════════════════════════
# STEP 1: GET SCORE COLUMNS
# ════════════════════════════════════════════════════════════════════════════════

score_cols <- paste0(names(signatures_filtered), "_1")
score_cols <- score_cols[score_cols %in% colnames(opc@meta.data)]
sig_names <- gsub("_1$", "", score_cols)

cat("Using", length(score_cols), "signatures:", paste(sig_names, collapse = ", "), "\n")

# ════════════════════════════════════════════════════════════════════════════════
# STEP 2: COMPUTE MEAN SCORES PER CLUSTER
# ════════════════════════════════════════════════════════════════════════════════

cat("\n--- Computing mean scores per cluster ---\n")

cluster_ids <- sort(unique(opc$seurat_clusters))

cluster_scores <- data.frame(cluster = cluster_ids)
rownames(cluster_scores) <- cluster_ids

for (col in score_cols) {
    means <- tapply(opc@meta.data[[col]], opc$seurat_clusters, mean)
    cluster_scores[[col]] <- means[as.character(cluster_ids)]
}

# ════════════════════════════════════════════════════════════════════════════════
# STEP 3: Z-SCORE ACROSS CLUSTERS → ASSIGN
# ════════════════════════════════════════════════════════════════════════════════

score_matrix <- as.matrix(cluster_scores[, score_cols])
z_cluster_scores <- scale(score_matrix)
colnames(z_cluster_scores) <- sig_names

cluster_scores$assigned_state <- sig_names[apply(z_cluster_scores, 1, which.max)]
cluster_scores$top_zscore <- apply(z_cluster_scores, 1, max)
cluster_scores$margin <- apply(z_cluster_scores, 1, function(x) {
    sorted <- sort(x, decreasing = TRUE)
    sorted[1] - sorted[2]
})

cat("\nCluster assignments (z-scored):\n")
print(cluster_scores[, c("cluster", "assigned_state", "top_zscore", "margin")])

# Flag low-confidence clusters
low_conf <- cluster_scores$margin < 0.3
if (any(low_conf)) {
    cat("\nLow-confidence clusters (margin < 0.3):\n")
    print(cluster_scores[low_conf, c("cluster", "assigned_state", "margin")])
}

# ════════════════════════════════════════════════════════════════════════════════
# STEP 4: MAP STATE TO CELLS
# ════════════════════════════════════════════════════════════════════════════════

state_map <- setNames(cluster_scores$assigned_state, as.character(cluster_scores$cluster))
opc$opc_state <- unname(state_map[as.character(opc$seurat_clusters)])

SUBCLUSTER_COL <- "opc_state"

# ════════════════════════════════════════════════════════════════════════════════
# STATE DISTRIBUTION
# ════════════════════════════════════════════════════════════════════════════════

cat("\nState distribution:\n")
print(table(opc@meta.data[[SUBCLUSTER_COL]]))

cat("\nPercentages:\n")
print(round(prop.table(table(opc@meta.data[[SUBCLUSTER_COL]])) * 100, 1))

# Check coverage
states_found <- unique(cluster_scores$assigned_state)
states_missing <- setdiff(sig_names, states_found)
cat("\nStates found:", paste(states_found, collapse = ", "), "\n")
if (length(states_missing) > 0) {
    cat("States not detected:", paste(states_missing, collapse = ", "), "\n")
    cat("  (No cluster scored highest for these — may not be present in data)\n")
}

# ════════════════════════════════════════════════════════════════════════════════
# SAVE CLUSTER SCORES
# ════════════════════════════════════════════════════════════════════════════════

cell_type_label <- gsub("_state$", "", SUBCLUSTER_COL)

scores_file <- file.path(RESULTS_DIR, paste0(DATASET, "_", cell_type_label, "_cluster_scores.csv"))
write.csv(cluster_scores, scores_file, row.names = FALSE)
cat("Saved:", scores_file, "\n")

cluster_summary <- data.frame(
    cluster = names(state_map),
    assigned_state = unname(state_map),
    n_cells = as.numeric(table(opc$seurat_clusters)[names(state_map)]),
    margin = cluster_scores$margin,
    row.names = NULL
)

cluster_summary_file <- file.path(RESULTS_DIR, paste0(DATASET, "_", cell_type_label, "_cluster_assignments.csv"))
write.csv(cluster_summary, cluster_summary_file, row.names = FALSE)
cat("Saved:", cluster_summary_file, "\n")

# ════════════════════════════════════════════════════════════════════════════════
# STATE COLORS & ORDER
# ════════════════════════════════════════════════════════════════════════════════

state_order <- names(sort(table(opc@meta.data[[SUBCLUSTER_COL]]), decreasing = TRUE))
state_colors <- get_state_colors(state_order)

cat("\nState order (by frequency):\n")
cat("  ", paste(state_order, collapse = " > "), "\n")

# ════════════════════════════════════════════════════════════════════════════════
# SET seurat_obj FOR DOWNSTREAM
# ════════════════════════════════════════════════════════════════════════════════

seurat_obj <- opc

save_file <- file.path(OUTPUT_DIR, paste0(DATASET, "_", cell_type_label, "_annotated.qs"))
qsave(seurat_obj, save_file)

cat("\nSaved:", save_file, "\n")
cat("  Cells:", ncol(seurat_obj), "\n")
cat("  Clusters:", length(cluster_ids), "\n")
cat("  States:", length(states_found), "\n")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# VALIDATION: UMAP PANEL
# ════════════════════════════════════════════════════════════════════════════════

cat("\n", paste(rep("=", 80), collapse = ""), "\n")
cat("VALIDATION: UMAP PANEL\n")
cat(paste(rep("=", 80), collapse = ""), "\n")

cell_type_label <- gsub("_state$", "", SUBCLUSTER_COL)
state_colors <- get_state_colors(unique(seurat_obj@meta.data[[SUBCLUSTER_COL]]))

add_corner_arrows <- function(p, label_size = 2.5) {
    build <- ggplot_build(p)
    x_range <- build$layout$panel_params[[1]]$x.range
    y_range <- build$layout$panel_params[[1]]$y.range
    
    x_start <- x_range[1] + diff(x_range) * 0.02
    y_start <- y_range[1] + diff(y_range) * 0.02
    x_arrow <- diff(x_range) * 0.12
    y_arrow <- diff(y_range) * 0.12
    
    p + 
        annotate("segment", x = x_start, xend = x_start + x_arrow, y = y_start, yend = y_start,
                 arrow = arrow(length = unit(0.1, "cm"), type = "closed"), linewidth = 0.3) +
        annotate("text", x = x_start + x_arrow/2, y = y_start - diff(y_range) * 0.04,
                 label = "UMAP1", size = label_size, hjust = 0.5, vjust = 1) +
        annotate("segment", x = x_start, xend = x_start, y = y_start, yend = y_start + y_arrow,
                 arrow = arrow(length = unit(0.1, "cm"), type = "closed"), linewidth = 0.3) +
        annotate("text", x = x_start - diff(x_range) * 0.04, y = y_start + y_arrow/2,
                 label = "UMAP2", size = label_size, hjust = 1, vjust = 0.5, angle = 90) +
        coord_cartesian(clip = "off")
}

options(repr.plot.width = 14, repr.plot.height = 12)

umap_theme <- theme_void(base_size = 10) +
    theme(
        plot.title = element_text(size = 11, face = "bold", hjust = 0.5),
        legend.position = "right",
        legend.title = element_blank(),
        legend.text = element_text(size = 8),
        legend.key.size = unit(0.3, "cm"),
        plot.margin = margin(15, 15, 20, 20)
    )

p1 <- DimPlot(seurat_obj, reduction = "umap", group.by = "seurat_clusters", 
              label = TRUE, label.size = 3, pt.size = 0.3) +
    ggtitle("Clusters") + umap_theme + theme(legend.position = "none")
p1 <- add_corner_arrows(p1)

p2 <- DimPlot(seurat_obj, reduction = "umap", group.by = SUBCLUSTER_COL, 
              pt.size = 0.3) +
    scale_color_manual(values = state_colors) +
    ggtitle("Cell States") + umap_theme
p2 <- add_corner_arrows(p2)

p3 <- DimPlot(seurat_obj, reduction = "umap", group.by = SENESCENCE_LABEL_COL, 
              pt.size = 0.3, order = c("SnC", "Non-SnC")) +
    scale_color_manual(values = SENESCENCE_COLORS) +
    ggtitle("Senescence") + umap_theme
p3 <- add_corner_arrows(p3)

p4 <- DimPlot(seurat_obj, reduction = "umap", group.by = STUDY_GROUP_COL, pt.size = 0.3) +
    scale_color_manual(values = study_colors) +
    ggtitle("Study Group") + umap_theme
p4 <- add_corner_arrows(p4)

umap_panel <- (p1 | p2) / (p3 | p4) +
    plot_annotation(
        title = paste0(tools::toTitleCase(cell_type_label), " Subclustering — ", DATASET),
        theme = theme(plot.title = element_text(size = 14, face = "bold", hjust = 0.5))
    )

print(umap_panel)

umap_file <- file.path(FIGURES_DIR, paste0(DATASET, "_", cell_type_label, "_umap_panel.svg"))
ggsave(umap_file, umap_panel, width = 14, height = 12)
cat("✓ Saved:", umap_file, "\n")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# VALIDATION: SIGNATURE HEATMAP
# ════════════════════════════════════════════════════════════════════════════════

cat("\n================================================================================\n")
cat("VALIDATION: SIGNATURE HEATMAP\n")
cat("================================================================================\n")

cell_type_label <- gsub("_state$", "", SUBCLUSTER_COL)

# Extract only numeric score columns
score_cols_only <- score_cols[score_cols %in% colnames(cluster_scores)]
heatmap_data <- cluster_scores[, score_cols_only]
colnames(heatmap_data) <- gsub("_1$", "", colnames(heatmap_data))
rownames(heatmap_data) <- cluster_scores$cluster

# Z-scale for visualization
cluster_scores_scaled <- t(scale(t(as.matrix(heatmap_data))))

# Order clusters by dominant state
cluster_dominant <- colnames(heatmap_data)[apply(heatmap_data, 1, which.max)]
cluster_order_df <- data.frame(
    cluster = rownames(cluster_scores_scaled),
    dominant = cluster_dominant,
    max_score = apply(cluster_scores_scaled, 1, max)
)

state_priority <- names(sort(table(seurat_obj@meta.data[[SUBCLUSTER_COL]]), decreasing = TRUE))
state_priority <- state_priority[state_priority %in% unique(cluster_dominant)]
cluster_order_df$dominant <- factor(cluster_order_df$dominant, levels = state_priority)
cluster_order_df <- cluster_order_df[order(cluster_order_df$dominant, -cluster_order_df$max_score), ]
cluster_order <- cluster_order_df$cluster

cluster_scores_scaled <- cluster_scores_scaled[cluster_order, ]

sig_order <- state_priority[state_priority %in% colnames(cluster_scores_scaled)]
sig_remaining <- colnames(cluster_scores_scaled)[!colnames(cluster_scores_scaled) %in% sig_order]
sig_order <- c(sig_order, sig_remaining)
cluster_scores_scaled <- cluster_scores_scaled[, sig_order]

row_annotation <- data.frame(
    State = cluster_order_df$dominant[match(rownames(cluster_scores_scaled), cluster_order_df$cluster)]
)
rownames(row_annotation) <- rownames(cluster_scores_scaled)

ann_colors <- list(
    State = get_state_colors(state_priority)
)

options(repr.plot.width = 10, repr.plot.height = 10)

p_heatmap <- pheatmap(
    cluster_scores_scaled,
    cluster_rows = FALSE,
    cluster_cols = FALSE,
    color = colorRampPalette(c("#313695", "#4575B4", "#74ADD1", "white", "#FDAE61", "#F46D43", "#A50026"))(100),
    breaks = seq(-2.5, 2.5, length.out = 101),
    annotation_row = row_annotation,
    annotation_colors = ann_colors,
    fontsize = 10,
    fontsize_row = 8,
    fontsize_col = 10,
    angle_col = 45,
    border_color = "gray90",
    cellwidth = 35,
    cellheight = 12,
    gaps_row = cumsum(table(cluster_order_df$dominant)),
    main = paste0(tools::toTitleCase(cell_type_label), " Signature Enrichment — ", DATASET),
    silent = TRUE
)

print(p_heatmap)

heatmap_file <- file.path(FIGURES_DIR, paste0(DATASET, "_", cell_type_label, "_signature_heatmap.svg"))
svg(heatmap_file, width = 10, height = 10)
print(p_heatmap)
dev.off()

cat("Saved:", heatmap_file, "\n")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# VALIDATION: CANONICAL MARKERS DOTPLOT
# ════════════════════════════════════════════════════════════════════════════════

cat("\n================================================================================\n")
cat("VALIDATION: CANONICAL MARKERS DOTPLOT\n")
cat("================================================================================\n")

cell_type_label <- gsub("_state$", "", SUBCLUSTER_COL)

canonical_markers <- list(
    'Homeostatic' = c('PDGFRA', 'CSPG4', 'PTPRZ1', 'BCAN', 'PCDH15'),
    'COP'         = c('GPR17', 'BCAS1', 'NEU4', 'TNS3', 'FYN'),
    'Reactive'    = c('HLA-A', 'HLA-B', 'B2M', 'CD74', 'SERPINA3')
)

canonical_markers_filtered <- lapply(canonical_markers, function(genes) {
    genes[genes %in% rownames(seurat_obj)]
})
canonical_markers_filtered <- canonical_markers_filtered[sapply(canonical_markers_filtered, length) > 0]

cat("\nMarkers available:\n")
for (ct in names(canonical_markers_filtered)) {
    cat(sprintf("  %s: %s\n", ct, paste(canonical_markers_filtered[[ct]], collapse = ", ")))
}

all_markers <- unlist(canonical_markers_filtered)

Idents(seurat_obj) <- "seurat_clusters"
cluster_order <- as.character(sort(as.numeric(levels(Idents(seurat_obj)))))

cat("\nCalculating expression statistics...\n")

dot_list <- list()
idx <- 1

for (cluster in cluster_order) {
    cells <- WhichCells(seurat_obj, idents = cluster)
    if (length(cells) == 0) next

    for (category in names(canonical_markers_filtered)) {
        for (gene in canonical_markers_filtered[[category]]) {
            expr <- GetAssayData(seurat_obj, slot = "data")[gene, cells]

            dot_list[[idx]] <- data.frame(
                Cluster = cluster,
                Gene = gene,
                Category = category,
                Pct_Exp = sum(expr > 0) / length(expr) * 100,
                Avg_Exp = mean(expr),
                stringsAsFactors = FALSE
            )
            idx <- idx + 1
        }
    }
}

dot_data <- do.call(rbind, dot_list)
cat("Collected", nrow(dot_data), "data points\n")

dot_data <- dot_data %>%
    group_by(Gene) %>%
    mutate(Scaled_Exp = as.numeric(scale(Avg_Exp))) %>%
    ungroup()

dot_data$Scaled_Exp <- pmax(pmin(dot_data$Scaled_Exp, 2.5), -2.5)
dot_data$Cluster <- factor(dot_data$Cluster, levels = rev(cluster_order))
dot_data$Gene <- factor(dot_data$Gene, levels = all_markers)
dot_data$Category <- factor(dot_data$Category, levels = names(canonical_markers_filtered))

options(repr.plot.width = 10, repr.plot.height = 10)

p_dotplot <- ggplot(dot_data, aes(x = Gene, y = Cluster)) +
    geom_point(aes(size = Pct_Exp, fill = Scaled_Exp), shape = 21, color = "black", stroke = 0.3) +
    scale_size_continuous(
        range = c(1, 6),
        limits = c(0, 100),
        breaks = c(0, 25, 50, 75, 100),
        name = "% Expr"
    ) +
    scale_fill_gradientn(
        colors = c("#313695", "#4575B4", "#74ADD1", "#FFFFBF", "#FDAE61", "#F46D43", "#A50026"),
        limits = c(-2.5, 2.5),
        name = "Scaled\nExpr"
    ) +
    facet_grid(cols = vars(Category), scales = "free_x", space = "free_x") +
    labs(x = NULL, y = "Cluster") +
    theme_bw(base_size = 10) +
    theme(
        axis.text.x = element_text(angle = 45, hjust = 1, size = 8, face = "italic"),
        axis.text.y = element_text(size = 9),
        axis.title.y = element_text(size = 10, face = "bold"),
        strip.text = element_text(size = 8, face = "bold"),
        strip.background = element_rect(fill = "gray70", color = "black", linewidth = 0.5),
        panel.grid = element_blank(),
        panel.spacing = unit(0.2, "lines"),
        panel.border = element_rect(color = "black", linewidth = 0.5),
        legend.position = "right",
        legend.title = element_text(size = 8),
        legend.text = element_text(size = 7),
        legend.key.size = unit(0.4, "cm"),
        plot.margin = margin(10, 10, 10, 10)
    )

print(p_dotplot)

dotplot_file <- file.path(FIGURES_DIR, paste0(DATASET, "_", cell_type_label, "_dotplot_canonical.svg"))
ggsave(dotplot_file, p_dotplot, width = 10, height = 10)
cat("Saved:", dotplot_file, "\n")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# SENESCENCE RE-THRESHOLDING AT CELL STATE LEVEL
# ════════════════════════════════════════════════════════════════════════════════

cat("\n", paste(rep("=", 80), collapse = ""), "\n")
cat("SENESCENCE RE-THRESHOLDING AT CELL STATE LEVEL\n")
cat(paste(rep("=", 80), collapse = ""), "\n")

# ─────────────────────────────────────────────────────────────────────────────
# CONFIGURATION
# ─────────────────────────────────────────────────────────────────────────────

SD_THRESHOLD <- 2
SCORE_COL <- "senescence_score"

if (STUDY_TYPE == "aging") {
    REFERENCE_GROUP <- GROUP_ORDER[1]
} else {
    REFERENCE_GROUP <- config$reference_group
}

cat(sprintf("\n  Seurat object: %s cells\n", format(ncol(seurat_obj), big.mark = ",")))
cat(sprintf("  Score column: %s\n", SCORE_COL))
cat(sprintf("  State column: %s\n", SUBCLUSTER_COL))
cat(sprintf("  Study type: %s\n", STUDY_TYPE))
cat(sprintf("  Reference group: %s\n", REFERENCE_GROUP))
cat(sprintf("  Threshold: Mean + %s SD per cell state\n", SD_THRESHOLD))

# ─────────────────────────────────────────────────────────────────────────────
# VALIDATE
# ─────────────────────────────────────────────────────────────────────────────

required_cols <- c(SCORE_COL, SENESCENCE_LABEL_COL, STUDY_GROUP_COL, SUBCLUSTER_COL)
missing <- required_cols[!required_cols %in% colnames(seurat_obj@meta.data)]
if (length(missing) > 0) {
    stop(sprintf("Missing columns: %s", paste(missing, collapse = ", ")))
}

ref_mask <- seurat_obj@meta.data[[STUDY_GROUP_COL]] == REFERENCE_GROUP
n_ref <- sum(ref_mask)
if (n_ref == 0) {
    stop(sprintf("Reference group '%s' not found! Available: %s",
                 REFERENCE_GROUP,
                 paste(unique(seurat_obj@meta.data[[STUDY_GROUP_COL]]), collapse = ", ")))
}
cat(sprintf("  Reference cells: %s\n", format(n_ref, big.mark = ",")))

# ─────────────────────────────────────────────────────────────────────────────
# ORIGINAL (SUBCLASS-LEVEL) SUMMARY
# ─────────────────────────────────────────────────────────────────────────────

orig_snc <- sum(seurat_obj@meta.data[[SENESCENCE_LABEL_COL]] == "SnC")
n_total <- nrow(seurat_obj@meta.data)

cat(sprintf("\n  Original (subclass-level): %s SnC / %s (%.1f%%)\n",
            format(orig_snc, big.mark = ","),
            format(n_total, big.mark = ","),
            orig_snc / n_total * 100))

# ─────────────────────────────────────────────────────────────────────────────
# CALCULATE STATE-LEVEL THRESHOLDS
# ─────────────────────────────────────────────────────────────────────────────

cat("\n  Calculating thresholds per cell state...\n")

states <- unique(seurat_obj@meta.data[[SUBCLUSTER_COL]])
thresholds <- list()

for (state in states) {
    state_ref_mask <- ref_mask & (seurat_obj@meta.data[[SUBCLUSTER_COL]] == state)
    ref_scores <- seurat_obj@meta.data[state_ref_mask, SCORE_COL]
    
    if (length(ref_scores) >= 10) {
        state_mean <- mean(ref_scores)
        state_sd <- sd(ref_scores)
        thresholds[[state]] <- state_mean + (SD_THRESHOLD * state_sd)
        
        cat(sprintf("    %s:\n", state))
        cat(sprintf("      n_ref=%s, mean=%.4f, sd=%.4f, threshold=%.4f\n",
                    format(length(ref_scores), big.mark = ","),
                    state_mean, state_sd, thresholds[[state]]))
    } else {
        all_state_scores <- seurat_obj@meta.data[
            seurat_obj@meta.data[[SUBCLUSTER_COL]] == state, SCORE_COL
        ]
        state_mean <- mean(all_state_scores)
        state_sd <- sd(all_state_scores)
        thresholds[[state]] <- state_mean + (SD_THRESHOLD * state_sd)
        
        cat(sprintf("    %s: (fallback — only %d ref cells, using all %s cells)\n",
                    state, length(ref_scores),
                    format(length(all_state_scores), big.mark = ",")))
        cat(sprintf("      mean=%.4f, sd=%.4f, threshold=%.4f\n",
                    state_mean, state_sd, thresholds[[state]]))
    }
}

# ─────────────────────────────────────────────────────────────────────────────
# APPLY STATE-LEVEL THRESHOLDS
# ─────────────────────────────────────────────────────────────────────────────

cat("\n  Applying state-level thresholds...\n")

seurat_obj@meta.data$is_senescent_state <- FALSE

for (state in names(thresholds)) {
    mask <- (seurat_obj@meta.data[[SUBCLUSTER_COL]] == state) &
            (seurat_obj@meta.data[[SCORE_COL]] >= thresholds[[state]])
    seurat_obj@meta.data$is_senescent_state[mask] <- TRUE
}

seurat_obj@meta.data$senescence_label_state <- ifelse(
    seurat_obj@meta.data$is_senescent_state, "SnC", "Non-SnC"
)

# ─────────────────────────────────────────────────────────────────────────────
# COMPARISON: ORIGINAL VS STATE-LEVEL
# ─────────────────────────────────────────────────────────────────────────────

new_snc <- sum(seurat_obj@meta.data$is_senescent_state)

cat("\n  ✓ Re-thresholding complete\n")
cat(sprintf("\n  %-25s %12s %12s\n", "", "Subclass", "State-level"))
cat(paste(rep("-", 55), collapse = ""), "\n")
cat(sprintf("  %-25s %12s %12s\n",
            "Total SnC",
            format(orig_snc, big.mark = ","),
            format(new_snc, big.mark = ",")))
cat(sprintf("  %-25s %11.1f%% %11.1f%%\n",
            "%SnC overall",
            orig_snc / n_total * 100,
            new_snc / n_total * 100))

cat("\n  Per state:\n")
cat(sprintf("  %-25s %6s %8s %8s %8s %8s %8s\n",
            "State", "Total", "Orig_n", "Orig_%", "New_n", "New_%", "Change"))
cat(paste(rep("-", 80), collapse = ""), "\n")

for (state in sort(states)) {
    state_mask <- seurat_obj@meta.data[[SUBCLUSTER_COL]] == state
    state_total <- sum(state_mask)
    
    orig_n <- sum(state_mask & (seurat_obj@meta.data[[SENESCENCE_LABEL_COL]] == "SnC"))
    new_n <- sum(state_mask & seurat_obj@meta.data$is_senescent_state)
    
    orig_pct <- orig_n / state_total * 100
    new_pct <- new_n / state_total * 100
    change <- new_pct - orig_pct
    
    flag <- if (abs(change) > 2) sprintf("%+.1f ←", change) else sprintf("%+.1f", change)
    
    cat(sprintf("  %-25s %6s %8s %7.1f%% %8s %7.1f%% %8s\n",
                state,
                format(state_total, big.mark = ","),
                format(orig_n, big.mark = ","), orig_pct,
                format(new_n, big.mark = ","), new_pct,
                flag))
}

cat("\n", paste(rep("=", 80), collapse = ""), "\n")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# ACTIVATE STATE-LEVEL LABELS FOR DOWNSTREAM PLOTS
# ─────────────────────────────────────────────────────────────────────────────

SENESCENCE_LABEL_COL <- "senescence_label_state"

cat(sprintf("\n  Active senescence column: %s\n", SENESCENCE_LABEL_COL))
cat(sprintf("  SnC: %s\n", format(sum(seurat_obj@meta.data[[SENESCENCE_LABEL_COL]] == "SnC"), big.mark = ",")))
cat(sprintf("  Non-SnC: %s\n", format(sum(seurat_obj@meta.data[[SENESCENCE_LABEL_COL]] == "Non-SnC"), big.mark = ",")))
cat("\n  All downstream plots will use state-level labels.\n")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# POST-SUBCLUSTERING ANALYSIS: COMPOSITION BY STUDY GROUP
# ════════════════════════════════════════════════════════════════════════════════

cat("\n================================================================================\n")
cat("POST-SUBCLUSTERING ANALYSIS\n")
cat("================================================================================\n")

# ─────────────────────────────────────────────────────────────────────────────
# SETUP
# ─────────────────────────────────────────────────────────────────────────────

cell_type_label <- gsub("_state$", "", SUBCLUSTER_COL)

state_order <- seurat_obj@meta.data %>%
    count(.data[[SUBCLUSTER_COL]]) %>%
    arrange(desc(n)) %>%
    pull(.data[[SUBCLUSTER_COL]])

n_states <- length(state_order)
state_colors <- get_state_colors(state_order)

cat("\nCell type:", cell_type_label, "\n")
cat("States detected:", n_states, "\n")
cat("  ", paste(state_order, collapse = ", "), "\n")
cat("Colors mapped:", length(state_colors), "\n")

# ════════════════════════════════════════════════════════════════════════════════
# PLOT 1: COMPOSITION BY STUDY GROUP
# ════════════════════════════════════════════════════════════════════════════════

cat("\n--- Plot 1: Composition by Study Group ---\n")

comp_studygroup <- seurat_obj@meta.data %>%
    group_by(.data[[STUDY_GROUP_COL]], .data[[SUBCLUSTER_COL]]) %>%
    summarise(n = n(), .groups = "drop") %>%
    group_by(.data[[STUDY_GROUP_COL]]) %>%
    mutate(pct = n / sum(n) * 100) %>%
    ungroup()

colnames(comp_studygroup)[1:2] <- c("Study_Group", "State")

# ─────────────────────────────────────────────────────────────────────────────
# TABLE OUTPUT
# ─────────────────────────────────────────────────────────────────────────────

comp_wide <- comp_studygroup %>%
    select(Study_Group, State, pct) %>%
    pivot_wider(names_from = Study_Group, values_from = pct, values_fill = 0) %>%
    mutate(across(where(is.numeric), ~ round(.x, 1)))

cat("\nState Composition by Study Group (%):\n")
print(as.data.frame(comp_wide))

write.csv(comp_wide, 
          file.path(RESULTS_DIR, paste0(DATASET, "_", cell_type_label, "_composition_by_studygroup.csv")),
          row.names = FALSE)

# ─────────────────────────────────────────────────────────────────────────────
# PREPARE FACTORS
# ─────────────────────────────────────────────────────────────────────────────

sg_detected <- unique(comp_studygroup$Study_Group)
is_aging <- STUDY_TYPE == "aging"

sg_order <- names(STUDY_GROUP_COLORS)[names(STUDY_GROUP_COLORS) %in% sg_detected]
comp_studygroup$Study_Group <- factor(comp_studygroup$Study_Group, levels = sg_order)
comp_studygroup$State <- factor(comp_studygroup$State, levels = rev(state_order))

if (is_aging) {
    x_labels <- setNames(gsub("Age_", "", gsub("_", "-", sg_order)), sg_order)
    x_title <- "Age Group (years)"
} else {
    x_labels <- setNames(gsub("_", " ", sg_order), sg_order)
    x_title <- "Disease Status"
}

# ─────────────────────────────────────────────────────────────────────────────
# PLOT
# ─────────────────────────────────────────────────────────────────────────────

options(repr.plot.width = 8, repr.plot.height = 6)

p_comp_sg <- ggplot(comp_studygroup, aes(x = Study_Group, y = pct, fill = State)) +
    geom_col(width = 0.75, color = "white", linewidth = 0.3) +
    scale_fill_manual(values = state_colors) +
    scale_x_discrete(labels = x_labels) +
    scale_y_continuous(expand = expansion(mult = c(0, 0.02))) +
    labs(
        x = x_title, 
        y = "Proportion (%)",
        fill = paste0(tools::toTitleCase(cell_type_label), " State")
    ) +
    theme_minimal(base_size = 14) +
    theme(
        axis.text.x = element_text(size = 13, color = "black", face = "bold",
                                   angle = 45, hjust = 1, vjust = 1),
        axis.text.y = element_text(size = 12, color = "black"),
        axis.title.x = element_text(size = 15, face = "bold", margin = margin(t = 12)),
        axis.title.y = element_text(size = 15, face = "bold", margin = margin(r = 12)),
        axis.line = element_line(color = "black", linewidth = 0.5),
        axis.ticks = element_line(color = "black", linewidth = 0.3),
        axis.ticks.length = unit(0.15, "cm"),
        legend.position = "right",
        legend.title = element_text(size = 12, face = "bold"),
        legend.text = element_text(size = 10),
        legend.key.size = unit(0.5, "cm"),
        panel.grid = element_blank(),
        panel.background = element_blank(),
        plot.margin = margin(15, 15, 15, 15)
    ) +
    guides(fill = guide_legend(ncol = 1, reverse = TRUE))

print(p_comp_sg)

ggsave(file.path(FIGURES_DIR, paste0(DATASET, "_", cell_type_label, "_composition_by_studygroup.svg")), 
       p_comp_sg, width = 8, height = 6, dpi = 300)

cat("\n✓ Saved:", paste0(DATASET, "_", cell_type_label, "_composition_by_studygroup.svg"), "\n")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# PLOT: SnC COMPOSITION BY STUDY GROUP
# ════════════════════════════════════════════════════════════════════════════════

cat(sprintf("\n--- Plot: SnC Composition by Study Group (%s) ---\n", tools::toTitleCase(cell_type_label)))

snc_comp_sg <- seurat_obj@meta.data %>%
    filter(.data[[SENESCENCE_LABEL_COL]] == "SnC") %>%
    group_by(.data[[STUDY_GROUP_COL]], .data[[SUBCLUSTER_COL]]) %>%
    summarise(n = n(), .groups = "drop") %>%
    complete(.data[[STUDY_GROUP_COL]], .data[[SUBCLUSTER_COL]], fill = list(n = 0)) %>%
    group_by(.data[[STUDY_GROUP_COL]]) %>%
    mutate(pct = n / sum(n) * 100) %>%
    ungroup()

colnames(snc_comp_sg)[1:2] <- c("Study_Group", "State")
snc_comp_sg$pct[is.nan(snc_comp_sg$pct)] <- 0

sg_detected <- unique(snc_comp_sg$Study_Group)
is_aging <- STUDY_TYPE == "aging"

sg_order <- names(STUDY_GROUP_COLORS)[names(STUDY_GROUP_COLORS) %in% sg_detected]
snc_comp_sg$Study_Group <- factor(snc_comp_sg$Study_Group, levels = sg_order)
snc_comp_sg$State <- factor(snc_comp_sg$State, levels = rev(state_order))

if (is_aging) {
    x_labels <- setNames(gsub("Age_", "", gsub("_", "-", sg_order)), sg_order)
    x_title <- "Age Group (years)"
} else {
    x_labels <- setNames(gsub("_", " ", sg_order), sg_order)
    x_title <- "Disease Status"
}

snc_comp_sg_wide <- snc_comp_sg %>%
    select(Study_Group, State, pct) %>%
    pivot_wider(names_from = Study_Group, values_from = pct, values_fill = 0) %>%
    mutate(across(where(is.numeric), ~ round(.x, 1)))

cat("\nSnC Composition by Study Group (%):\n")
print(as.data.frame(snc_comp_sg_wide))

write.csv(snc_comp_sg_wide, 
          file.path(RESULTS_DIR, paste0(DATASET, "_", cell_type_label, "_snc_composition_by_studygroup.csv")),
          row.names = FALSE)

snc_totals <- seurat_obj@meta.data %>%
    filter(.data[[SENESCENCE_LABEL_COL]] == "SnC") %>%
    group_by(.data[[STUDY_GROUP_COL]]) %>%
    summarise(n = n(), .groups = "drop")

colnames(snc_totals)[1] <- "Study_Group"

cat("\nTotal SnC cells per study group:\n")
print(as.data.frame(snc_totals))

options(repr.plot.width = 8, repr.plot.height = 6)

p_snc_comp_sg <- ggplot(snc_comp_sg, aes(x = Study_Group, y = pct, fill = State)) +
    geom_col(width = 0.75, color = "white", linewidth = 0.3) +
    scale_fill_manual(values = state_colors) +
    scale_x_discrete(labels = x_labels) +
    scale_y_continuous(limits = c(0, 100.1), expand = c(0, 0)) +
    labs(
        x = x_title, 
        y = paste0("Proportion of SnC ", tools::toTitleCase(cell_type_label), " (%)"),
        fill = paste0(tools::toTitleCase(cell_type_label), " State")
    ) +
    theme_minimal(base_size = 14) +
    theme(
        axis.text.x = element_text(size = 13, color = "black", face = "bold",
                                   angle = 45, hjust = 1, vjust = 1),
        axis.text.y = element_text(size = 12, color = "black"),
        axis.title.x = element_text(size = 15, face = "bold", margin = margin(t = 12)),
        axis.title.y = element_text(size = 15, face = "bold", margin = margin(r = 12)),
        axis.line = element_line(color = "black", linewidth = 0.5),
        axis.ticks = element_line(color = "black", linewidth = 0.3),
        axis.ticks.length = unit(0.15, "cm"),
        legend.position = "right",
        legend.title = element_text(size = 12, face = "bold"),
        legend.text = element_text(size = 10),
        legend.key.size = unit(0.5, "cm"),
        panel.grid = element_blank(),
        panel.background = element_blank(),
        plot.margin = margin(15, 15, 15, 15)
    ) +
    guides(fill = guide_legend(ncol = 1, reverse = TRUE))

print(p_snc_comp_sg)

ggsave(file.path(FIGURES_DIR, paste0(DATASET, "_", cell_type_label, "_snc_composition_by_studygroup.svg")), 
       p_snc_comp_sg, width = 8, height = 6, dpi = 300)

cat("\nSaved:", paste0(DATASET, "_", cell_type_label, "_snc_composition_by_studygroup.svg"), "\n")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# PLOT 2: COMPOSITION BY SAMPLE (SPLIT BY STUDY GROUP)
# ════════════════════════════════════════════════════════════════════════════════

cat("\n--- Plot 2: Composition by Sample (Split by Study Group) ---\n")

cell_type_label <- gsub("_state$", "", SUBCLUSTER_COL)
is_aging <- STUDY_TYPE == "aging"

# ─────────────────────────────────────────────────────────────────────────────
# STEP 1: CALCULATE COMPOSITION
# ─────────────────────────────────────────────────────────────────────────────

comp_sample <- seurat_obj@meta.data %>%
    group_by(.data[[DONOR_COL]], .data[[STUDY_GROUP_COL]], .data[[SUBCLUSTER_COL]]) %>%
    summarise(n = n(), .groups = "drop") %>%
    group_by(.data[[DONOR_COL]]) %>%
    mutate(pct = n / sum(n) * 100) %>%
    ungroup()

colnames(comp_sample)[1:3] <- c("Donor", "Study_Group", "State")

comp_sample$Donor <- as.character(comp_sample$Donor)
comp_sample$Study_Group <- as.character(comp_sample$Study_Group)
comp_sample$State <- as.character(comp_sample$State)

cat("\nStep 1 - Initial calculation:\n")
cat("  Rows:", nrow(comp_sample), "\n")
cat("  Unique donors:", n_distinct(comp_sample$Donor), "\n")
cat("  Unique states:", n_distinct(comp_sample$State), "\n")

# ─────────────────────────────────────────────────────────────────────────────
# STEP 2: COMPLETE MISSING COMBINATIONS
# ─────────────────────────────────────────────────────────────────────────────

donor_sg <- comp_sample %>% select(Donor, Study_Group) %>% distinct()
all_states <- unique(comp_sample$State)
complete_grid <- donor_sg %>% crossing(State = all_states)

comp_sample <- complete_grid %>%
    left_join(comp_sample %>% select(Donor, State, n, pct), 
              by = c("Donor", "State")) %>%
    mutate(n = replace_na(n, 0), pct = replace_na(pct, 0))

cat("\nStep 2 - After grid completion:\n")
cat("  Rows:", nrow(comp_sample), "\n")
cat("  Expected:", n_distinct(donor_sg$Donor), "x", length(all_states), "=", 
    n_distinct(donor_sg$Donor) * length(all_states), "\n")

sample_totals <- comp_sample %>%
    group_by(Donor) %>%
    summarise(total_pct = sum(pct), .groups = "drop")

cat("  Sample totals - Min:", round(min(sample_totals$total_pct), 1), 
    "Max:", round(max(sample_totals$total_pct), 1), "\n")

# ─────────────────────────────────────────────────────────────────────────────
# STEP 3: DETECT COHORT TYPE AND SET ORDERS
# ─────────────────────────────────────────────────────────────────────────────

sg_detected <- unique(comp_sample$Study_Group)
sg_order <- names(STUDY_GROUP_COLORS)[names(STUDY_GROUP_COLORS) %in% sg_detected]

state_order <- comp_sample %>%
    group_by(State) %>%
    summarise(total_n = sum(n), .groups = "drop") %>%
    arrange(desc(total_n)) %>%
    pull(State)

sample_order <- seurat_obj@meta.data %>%
    select(all_of(c(DONOR_COL, STUDY_GROUP_COL))) %>%
    distinct() %>%
    mutate(sg_factor = factor(.data[[STUDY_GROUP_COL]], levels = sg_order)) %>%
    arrange(sg_factor, .data[[DONOR_COL]]) %>%
    pull(.data[[DONOR_COL]])

cat("\nStep 3 - Orders defined:\n")
cat("  Study groups:", paste(sg_order, collapse = ", "), "\n")
cat("  States (by freq):", paste(state_order, collapse = ", "), "\n")
cat("  Samples:", length(sample_order), "\n")

# ─────────────────────────────────────────────────────────────────────────────
# STEP 4: TABLE
# ─────────────────────────────────────────────────────────────────────────────

comp_summary <- comp_sample %>%
    group_by(Study_Group, State) %>%
    summarise(mean_pct = mean(pct), sd_pct = sd(pct), .groups = "drop") %>%
    mutate(display = sprintf("%.1f ± %.1f", mean_pct, sd_pct)) %>%
    select(Study_Group, State, display) %>%
    pivot_wider(names_from = Study_Group, values_from = display)

cat("\nComposition Summary (Mean ± SD):\n")
print(as.data.frame(comp_summary))

# ─────────────────────────────────────────────────────────────────────────────
# STEP 5: STATISTICAL TEST
# ─────────────────────────────────────────────────────────────────────────────

cat("\n--- Statistical Test ---\n")

has_sex <- SEX_COL %in% colnames(seurat_obj@meta.data)
has_cohort <- COHORT_COL %in% colnames(seurat_obj@meta.data)

cat("\nCovariates:\n")
cat("  Sex column (", SEX_COL, "):", ifelse(has_sex, "FOUND", "NOT FOUND"), "\n")
cat("  Cohort column (", COHORT_COL, "):", ifelse(has_cohort, "FOUND", "NOT FOUND"), "\n")

rhs <- "Predictor"
if (has_sex) rhs <- paste0(rhs, " + Sex")
if (has_cohort) rhs <- paste0(rhs, " + Cohort")
formula_text <- paste0("CubeRoot(State_%) ~ ", rhs)

cat("\nModel:", formula_text, "\n")
cat("Transformation: (proportion / 100)^(1/3)\n")

covariate_cols <- c(DONOR_COL)
if (has_sex) covariate_cols <- c(covariate_cols, SEX_COL)
if (has_cohort) covariate_cols <- c(covariate_cols, COHORT_COL)

donor_meta <- seurat_obj@meta.data %>%
    select(all_of(covariate_cols)) %>%
    distinct()

colnames(donor_meta)[1] <- "Donor"
if (has_sex) colnames(donor_meta)[colnames(donor_meta) == SEX_COL] <- "Sex"
if (has_cohort) colnames(donor_meta)[colnames(donor_meta) == COHORT_COL] <- "Cohort"

comp_stats_data <- comp_sample %>%
    mutate(pct_cuberoot = (pct / 100)^(1/3)) %>%
    left_join(donor_meta, by = "Donor")

if (is_aging) {
    comp_stats_data <- comp_stats_data %>%
        mutate(Predictor = as.numeric(gsub("Age_([0-9]+)_.*", "\\1", Study_Group)))
    cat("Cohort type: AGING (testing linear trend with age)\n\n")
} else {
    disease_order <- sg_order
    comp_stats_data <- comp_stats_data %>%
        mutate(Predictor = as.numeric(factor(Study_Group, levels = disease_order)) - 1)
    cat("Cohort type: DISEASE (testing trend across disease stages)\n\n")
}

if (has_sex) comp_stats_data$Sex <- as.factor(comp_stats_data$Sex)
if (has_cohort) comp_stats_data$Cohort <- as.factor(comp_stats_data$Cohort)

model_formula_comp <- as.formula(paste0("pct_cuberoot ~ ", rhs))

results_list <- list()
for (state in state_order) {
    df_state <- comp_stats_data %>% filter(State == state)
    model <- lm(model_formula_comp, data = df_state)
    coef_summary <- summary(model)$coefficients
    results_list[[state]] <- data.frame(
        State = state,
        estimate = coef_summary["Predictor", "Estimate"],
        std_error = coef_summary["Predictor", "Std. Error"],
        p_value = coef_summary["Predictor", "Pr(>|t|)"]
    )
}

comp_stats <- do.call(rbind, results_list) %>%
    mutate(
        p_adj = p.adjust(p_value, method = "BH"),
        direction = case_when(estimate > 0 ~ "Up", estimate < 0 ~ "Down", TRUE ~ ""),
        sig = case_when(p_adj < 0.001 ~ "***", p_adj < 0.01 ~ "**", p_adj < 0.05 ~ "*", TRUE ~ "")
    ) %>%
    arrange(p_adj)

cat("Results (BH-adjusted):\n")
print(comp_stats %>% select(State, estimate, p_value, p_adj, direction, sig))

write.csv(comp_stats, 
          file.path(RESULTS_DIR, paste0(DATASET, "_", cell_type_label, "_composition_stats.csv")),
          row.names = FALSE)

# ─────────────────────────────────────────────────────────────────────────────
# STEP 6: CONVERT TO FACTORS
# ─────────────────────────────────────────────────────────────────────────────

comp_sample$Donor <- factor(comp_sample$Donor, levels = sample_order)
comp_sample$Study_Group <- factor(comp_sample$Study_Group, levels = sg_order)
comp_sample$State <- factor(comp_sample$State, levels = rev(state_order))

cat("\nStep 6 - Factor check (should all be 0):\n")
cat("  NA in Donor:", sum(is.na(comp_sample$Donor)), "\n")
cat("  NA in Study_Group:", sum(is.na(comp_sample$Study_Group)), "\n")
cat("  NA in State:", sum(is.na(comp_sample$State)), "\n")

samples_per_group <- comp_sample %>%
    select(Donor, Study_Group) %>%
    distinct() %>%
    count(Study_Group)

cat("\nSamples per study group:\n")
print(as.data.frame(samples_per_group))

# ─────────────────────────────────────────────────────────────────────────────
# STEP 7: COLORS AND LABELS
# ─────────────────────────────────────────────────────────────────────────────

state_colors <- get_state_colors(state_order)

if (is_aging) {
    facet_labels <- setNames(gsub("Age_", "", gsub("_", "-", sg_order)), sg_order)
} else {
    facet_labels <- setNames(gsub("_", " ", sg_order), sg_order)
}

sig_lookup <- setNames(comp_stats$sig, comp_stats$State)
dir_lookup <- setNames(comp_stats$direction, comp_stats$State)

state_labels <- sapply(state_order, function(s) {
    sig <- sig_lookup[s]
    dir <- dir_lookup[s]
    if (!is.na(sig) && sig != "") {
        dir_symbol <- ifelse(dir == "Up", "(+)", "(-)")
        paste0(s, " ", dir_symbol, sig)
    } else { s }
})

# ─────────────────────────────────────────────────────────────────────────────
# STEP 8: PLOT
# ─────────────────────────────────────────────────────────────────────────────

options(repr.plot.width = 14, repr.plot.height = 6)

p_comp_sample <- ggplot(comp_sample, aes(x = Donor, y = pct, fill = State)) +
    geom_col(width = 0.85, color = "white", linewidth = 0.2) +
    scale_fill_manual(values = state_colors, labels = state_labels) +
    scale_y_continuous(expand = c(0, 0)) +
    coord_cartesian(ylim = c(0, 100)) +
    facet_grid(cols = vars(Study_Group), scales = "free_x", space = "free_x",
               labeller = labeller(Study_Group = facet_labels)) +
    labs(
        x = "Samples", 
        y = "Proportion (%)",
        fill = paste0(tools::toTitleCase(cell_type_label), " State")
    ) +
    theme_minimal(base_size = 14) +
    theme(
        axis.text.x = element_blank(),
        axis.ticks.x = element_blank(),
        axis.text.y = element_text(size = 12, color = "black"),
        axis.title.x = element_text(size = 15, face = "bold", margin = margin(t = 10)),
        axis.title.y = element_text(size = 15, face = "bold", margin = margin(r = 10)),
        axis.line = element_line(color = "black", linewidth = 0.5),
        axis.ticks.y = element_line(color = "black", linewidth = 0.3),
        axis.ticks.length = unit(0.15, "cm"),
        strip.text = element_text(size = 13, face = "bold", color = "black"),
        strip.background = element_rect(fill = "gray80", color = "black", linewidth = 0.5),
        panel.spacing = unit(0.5, "lines"),
        panel.border = element_rect(color = "black", fill = NA, linewidth = 0.5),
        legend.position = "right",
        legend.title = element_text(size = 12, face = "bold"),
        legend.text = element_text(size = 10),
        legend.key.size = unit(0.45, "cm"),
        panel.grid = element_blank(),
        plot.margin = margin(15, 15, 15, 15)
    ) +
    guides(fill = guide_legend(ncol = 1, reverse = TRUE))

print(p_comp_sample)

ggsave(file.path(FIGURES_DIR, paste0(DATASET, "_", cell_type_label, "_composition_by_sample.svg")), 
       p_comp_sample, width = 14, height = 6, dpi = 300)

write.csv(comp_sample, 
          file.path(RESULTS_DIR, paste0(DATASET, "_", cell_type_label, "_composition_by_sample.csv")),
          row.names = FALSE)

cat("\n✓ Saved:", paste0(DATASET, "_", cell_type_label, "_composition_by_sample.svg"), "\n")
cat("✓ Saved:", paste0(DATASET, "_", cell_type_label, "_composition_stats.csv"), "\n")

# ─────────────────────────────────────────────────────────────────────────────
# SUMMARY
# ─────────────────────────────────────────────────────────────────────────────

sig_states <- comp_stats %>% filter(p_adj < 0.05)

cat("\n================================================================================\n")
cat("SIGNIFICANT CHANGES (p_adj < 0.05):\n")
cat("================================================================================\n")

if (nrow(sig_states) > 0) {
    direction_text <- if (is_aging) {
        c("Up" = "increases with age", "Down" = "decreases with age")
    } else {
        c("Up" = "increases with disease", "Down" = "decreases with disease")
    }
    for (i in 1:nrow(sig_states)) {
        dir_desc <- direction_text[sig_states$direction[i]]
        cat(sprintf("  - %s %s (p_adj = %.2e)\n", sig_states$State[i], dir_desc, sig_states$p_adj[i]))
    }
} else {
    cat("  No significant changes detected\n")
}

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# SECTION 1: DATA PREPARATION
# ════════════════════════════════════════════════════════════════════════════════

cat("\n", paste(rep("=", 80), collapse = ""), "\n")
cat("SECTION 1: DATA PREPARATION\n")
cat(paste(rep("=", 80), collapse = ""), "\n")

# ─────────────────────────────────────────────────────────────────────────────
# STEP 1.1: CALCULATE %SnC PER STATE PER DONOR
# ─────────────────────────────────────────────────────────────────────────────

cat("\n[Step 1.1] Calculating %SnC per state per donor...\n")

snc_by_state_donor <- seurat_obj@meta.data %>%
    group_by(.data[[DONOR_COL]], .data[[STUDY_GROUP_COL]], .data[[SUBCLUSTER_COL]]) %>%
    summarise(
        n_total = n(),
        n_snc = sum(.data[[SENESCENCE_LABEL_COL]] == "SnC"),
        pct_snc = n_snc / n_total * 100,
        mean_log_umi = mean(log10(total_counts + 1)),
        .groups = "drop"
    )

colnames(snc_by_state_donor)[1:3] <- c("Donor", "Study_Group", "State")

snc_by_state_donor$Donor <- as.character(snc_by_state_donor$Donor)
snc_by_state_donor$Study_Group <- as.character(snc_by_state_donor$Study_Group)
snc_by_state_donor$State <- as.character(snc_by_state_donor$State)

cat("  Rows:", nrow(snc_by_state_donor), "\n")
cat("  Unique donors:", n_distinct(snc_by_state_donor$Donor), "\n")
cat("  Unique states:", n_distinct(snc_by_state_donor$State), "\n")

# ─────────────────────────────────────────────────────────────────────────────
# STEP 1.2: COMPLETE MISSING COMBINATIONS (fill with 0% SnC)
# ─────────────────────────────────────────────────────────────────────────────

cat("\n[Step 1.2] Completing missing donor-state combinations...\n")

donor_sg <- snc_by_state_donor %>%
    select(Donor, Study_Group) %>%
    distinct()

all_states <- unique(snc_by_state_donor$State)

complete_grid <- donor_sg %>%
    crossing(State = all_states)

snc_by_state_donor <- complete_grid %>%
    left_join(snc_by_state_donor %>% select(Donor, State, n_total, n_snc, pct_snc, mean_log_umi), 
              by = c("Donor", "State")) %>%
    mutate(
        n_total = replace_na(n_total, 0),
        n_snc = replace_na(n_snc, 0),
        pct_snc = replace_na(pct_snc, 0)
    )

cat("  Rows after completion:", nrow(snc_by_state_donor), "\n")
cat("  Expected:", n_distinct(donor_sg$Donor), "x", length(all_states), "=", 
    n_distinct(donor_sg$Donor) * length(all_states), "\n")

# ─────────────────────────────────────────────────────────────────────────────
# STEP 1.3: MERGE DONOR-LEVEL METADATA (Age, Sex, Cohort)
# ─────────────────────────────────────────────────────────────────────────────

cat("\n[Step 1.3] Merging donor-level metadata...\n")

available_cols <- colnames(seurat_obj@meta.data)

cols_to_extract <- c(DONOR_COL)

has_primary_var <- PRIMARY_VAR %in% available_cols
if (has_primary_var) {
    cols_to_extract <- c(cols_to_extract, PRIMARY_VAR)
    cat("  Found PRIMARY_VAR:", PRIMARY_VAR, "\n")
} else {
    cat("  WARNING: PRIMARY_VAR", PRIMARY_VAR, "not found in metadata\n")
}

has_sex <- SEX_COL %in% available_cols
if (has_sex) {
    cols_to_extract <- c(cols_to_extract, SEX_COL)
    cat("  Found SEX_COL:", SEX_COL, "\n")
}

has_cohort <- COHORT_COL %in% available_cols
if (has_cohort) {
    cols_to_extract <- c(cols_to_extract, COHORT_COL)
    cat("  Found COHORT_COL:", COHORT_COL, "\n")
}

donor_meta <- seurat_obj@meta.data %>%
    select(all_of(cols_to_extract)) %>%
    distinct()

colnames(donor_meta)[1] <- "Donor"
if (has_primary_var) colnames(donor_meta)[colnames(donor_meta) == PRIMARY_VAR] <- "PrimaryVar"
if (has_sex) colnames(donor_meta)[colnames(donor_meta) == SEX_COL] <- "Sex"
if (has_cohort) colnames(donor_meta)[colnames(donor_meta) == COHORT_COL] <- "Cohort"

donor_meta$Donor <- as.character(donor_meta$Donor)

cat("  Donor metadata rows:", nrow(donor_meta), "\n")
cat("  Donor metadata columns:", paste(colnames(donor_meta), collapse = ", "), "\n")

snc_by_state_donor <- snc_by_state_donor %>%
    left_join(donor_meta, by = "Donor")

cat("  Final columns:", paste(colnames(snc_by_state_donor), collapse = ", "), "\n")

# ─────────────────────────────────────────────────────────────────────────────
# STEP 1.4: CREATE NUMERIC PREDICTOR
# ─────────────────────────────────────────────────────────────────────────────

cat("\n[Step 1.4] Creating numeric predictor...\n")

cat("  Study type:", STUDY_TYPE, "\n")
cat("  Primary variable:", PRIMARY_VAR, "(", PRIMARY_VAR_TYPE, ")\n")

is_aging <- STUDY_TYPE == "aging"

if (PRIMARY_VAR_TYPE == "continuous" && has_primary_var) {
    snc_by_state_donor$Predictor <- as.numeric(snc_by_state_donor$PrimaryVar)
    predictor_desc <- paste0(PRIMARY_VAR, " (continuous)")
    cat("  Using continuous predictor:", PRIMARY_VAR, "\n")
    cat("  Range:", min(snc_by_state_donor$Predictor, na.rm = TRUE), "-", 
        max(snc_by_state_donor$Predictor, na.rm = TRUE), "\n")
} else {
    sg_order <- names(STUDY_GROUP_COLORS)[names(STUDY_GROUP_COLORS) %in% unique(snc_by_state_donor$Study_Group)]
    snc_by_state_donor$Study_Group <- factor(snc_by_state_donor$Study_Group, levels = sg_order)
    snc_by_state_donor$Predictor <- as.numeric(snc_by_state_donor$Study_Group)
    predictor_desc <- paste0("Study_Group (ordinal)")
    cat("  Using ordinal predictor from Study_Group\n")
    cat("  Levels:", paste(sg_order, collapse = ", "), "\n")
}

# ─────────────────────────────────────────────────────────────────────────────
# STEP 1.5: SETUP ORDERS AND LABELS
# ─────────────────────────────────────────────────────────────────────────────

cat("\n[Step 1.5] Setting up orders and labels...\n")

sg_detected <- unique(snc_by_state_donor$Study_Group)
sg_order <- names(STUDY_GROUP_COLORS)[names(STUDY_GROUP_COLORS) %in% sg_detected]

state_order <- snc_by_state_donor %>%
    group_by(State) %>%
    summarise(median_snc = median(pct_snc), .groups = "drop") %>%
    arrange(desc(median_snc)) %>%
    pull(State)

if (is_aging) {
    x_labels <- setNames(gsub("Age_", "", gsub("_", "-", sg_order)), sg_order)
    x_title <- "Age Group (years)"
} else {
    x_labels <- setNames(gsub("_", " ", sg_order), sg_order)
    x_title <- "Disease Status"
}

snc_by_state_donor$Study_Group <- factor(snc_by_state_donor$Study_Group, levels = sg_order)
snc_by_state_donor$State <- factor(snc_by_state_donor$State, levels = state_order)

cat("  Study groups:", paste(sg_order, collapse = ", "), "\n")
cat("  States (by median %SnC):", paste(state_order, collapse = ", "), "\n")

cat("\n[Section 1 Complete]\n")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# SECTION 2: STATISTICAL MODELING
# ════════════════════════════════════════════════════════════════════════════════

cat("\n", paste(rep("=", 80), collapse = ""), "\n")
cat("SECTION 2: STATISTICAL MODELING\n")
cat(paste(rep("=", 80), collapse = ""), "\n")

cell_type_label <- gsub("_state$", "", SUBCLUSTER_COL)

# ─────────────────────────────────────────────────────────────────────────────
# STEP 2.1: CHECK COVARIATES
# ─────────────────────────────────────────────────────────────────────────────

cat("\n[Step 2.1] Checking covariates...\n")

has_sex <- "Sex" %in% colnames(snc_by_state_donor)
has_cohort <- "Cohort" %in% colnames(snc_by_state_donor)
has_umi <- "mean_log_umi" %in% colnames(snc_by_state_donor)

cat("  Sex:", ifelse(has_sex, "FOUND", "NOT FOUND"), "\n")
cat("  Cohort:", ifelse(has_cohort, "FOUND", "NOT FOUND"), "\n")
cat("  Log UMI:", ifelse(has_umi, "FOUND", "NOT FOUND"), "\n")

# ─────────────────────────────────────────────────────────────────────────────
# STEP 2.2: BUILD FORMULA AND DESCRIPTOR STRINGS
# ─────────────────────────────────────────────────────────────────────────────

cat("\n[Step 2.2] Building model formula...\n")

covariates_used <- c()
if (has_sex) covariates_used <- c(covariates_used, "Sex")
if (has_cohort) covariates_used <- c(covariates_used, "Cohort")
if (has_umi) covariates_used <- c(covariates_used, "mean_log_umi")

if (length(covariates_used) == 0) {
    cov_part <- ""
    covariates_display <- "none"
} else {
    cov_part <- paste0(" + ", paste(covariates_used, collapse = " + "))
    covariates_display <- paste(covariates_used, collapse = ", ")
}

formula_lm <- paste0("pct_cuberoot ~ Predictor", cov_part)
model_formula <- as.formula(formula_lm)

if (is_aging) {
    predictor_desc <- "Age (continuous)"
    PRIMARY_VAR <- "Age"
    x_title <- "Age Group"
} else {
    predictor_desc <- "Disease stage (ordinal)"
    PRIMARY_VAR <- "Disease"
    x_title <- "Disease Stage"
}

cat("  Formula:", formula_lm, "\n")
cat("  Predictor:", predictor_desc, "\n")
cat("  Covariates:", covariates_display, "\n")
cat("  PRIMARY_VAR:", PRIMARY_VAR, "\n")

# ─────────────────────────────────────────────────────────────────────────────
# STEP 2.3: SET UP COLORS AND LABELS FOR VISUALIZATION
# ─────────────────────────────────────────────────────────────────────────────

cat("\n[Step 2.3] Setting up colors and labels...\n")

if (is_aging) {
    x_labels <- setNames(gsub("Age_", "", gsub("_", "-", sg_order)), sg_order)
} else {
    x_labels <- setNames(gsub("_", " ", sg_order), sg_order)
}

cat("  X-axis labels:", paste(x_labels, collapse = ", "), "\n")
cat("  Study group colors:", length(STUDY_GROUP_COLORS), "defined\n")

# ─────────────────────────────────────────────────────────────────────────────
# STEP 2.4: PREPARE DATA FOR MODELING
# ─────────────────────────────────────────────────────────────────────────────

cat("\n[Step 2.4] Preparing data for modeling...\n")

snc_model_data <- snc_by_state_donor %>%
    mutate(pct_cuberoot = (pct_snc / 100)^(1/3))

if (!"Predictor" %in% colnames(snc_model_data)) {
    cat("  Adding Predictor column...\n")
    if (is_aging) {
        snc_model_data <- snc_model_data %>%
            mutate(Predictor = PrimaryVar)
    } else {
        disease_order <- sg_order
        snc_model_data <- snc_model_data %>%
            mutate(Predictor = as.numeric(factor(Study_Group, levels = disease_order)) - 1)
    }
}

if (has_sex) snc_model_data$Sex <- as.factor(snc_model_data$Sex)
if (has_cohort) snc_model_data$Cohort <- as.factor(snc_model_data$Cohort)

cat("  Rows:", nrow(snc_model_data), "\n")
cat("  Predictor range:", min(snc_model_data$Predictor, na.rm = TRUE), "-", 
    max(snc_model_data$Predictor, na.rm = TRUE), "\n")
if (has_umi) {
    cat("  Log UMI range:", 
        sprintf("%.2f - %.2f", 
                min(snc_model_data$mean_log_umi, na.rm = TRUE),
                max(snc_model_data$mean_log_umi, na.rm = TRUE)), "\n")
}

# ─────────────────────────────────────────────────────────────────────────────
# STEP 2.5: RUN LINEAR MODEL PER STATE
# ─────────────────────────────────────────────────────────────────────────────

cat("\n[Step 2.5] Running linear models per state...\n")
cat("  Formula:", formula_lm, "\n\n")

results_list <- list()

for (state in state_order) {
    df_state <- snc_model_data %>% filter(State == state)
    n_donors <- n_distinct(df_state$Donor)
    
    model <- tryCatch(
        lm(model_formula, data = df_state),
        error = function(e) {
            cat(sprintf("    %s: MODEL FAILED — %s\n", state, e$message))
            return(NULL)
        }
    )
    
    if (is.null(model)) next
    
    coef_summary <- summary(model)$coefficients
    
    results_list[[state]] <- data.frame(
        State = state,
        n_donors = n_donors,
        estimate = coef_summary["Predictor", "Estimate"],
        std_error = coef_summary["Predictor", "Std. Error"],
        p_value = coef_summary["Predictor", "Pr(>|t|)"]
    )
    
    cat(sprintf("    %s (n=%d): b=%.4f, p=%.2e\n", 
                state, n_donors, 
                coef_summary["Predictor", "Estimate"],
                coef_summary["Predictor", "Pr(>|t|)"]))
}

# ─────────────────────────────────────────────────────────────────────────────
# STEP 2.6: MULTIPLE TESTING CORRECTION
# ─────────────────────────────────────────────────────────────────────────────

cat("\n[Step 2.6] Applying BH (FDR) correction...\n")

snc_state_stats <- do.call(rbind, results_list) %>%
    mutate(
        p_adj = p.adjust(p_value, method = "BH"),
        direction = case_when(
            estimate > 0 ~ "Up",
            estimate < 0 ~ "Down",
            TRUE ~ ""
        ),
        sig = case_when(
            p_adj < 0.001 ~ "***",
            p_adj < 0.01 ~ "**",
            p_adj < 0.05 ~ "*",
            TRUE ~ ""
        ),
        Significant = p_adj < 0.05
    ) %>%
    arrange(p_adj)

rownames(snc_state_stats) <- NULL

cat("\n  Results (BH-adjusted):\n")
print(snc_state_stats %>% select(State, n_donors, estimate, std_error, p_value, p_adj, direction, sig))

sig_results <- snc_state_stats %>% filter(Significant)

cat("\n  Significant states (FDR < 0.05):", nrow(sig_results), "of", nrow(snc_state_stats), "\n")

if (nrow(sig_results) > 0) {
    for (i in 1:nrow(sig_results)) {
        dir_text <- ifelse(sig_results$direction[i] == "Up",
                           paste0("increases with ", PRIMARY_VAR),
                           paste0("decreases with ", PRIMARY_VAR))
        cat(sprintf("    • %s: SnC %s (b=%.4f, p_adj=%.3f)\n",
                    sig_results$State[i], dir_text,
                    sig_results$estimate[i], sig_results$p_adj[i]))
    }
}

# ─────────────────────────────────────────────────────────────────────────────
# STEP 2.7: SAVE RESULTS
# ─────────────────────────────────────────────────────────────────────────────

cat("\n[Step 2.7] Saving results...\n")

write.csv(snc_state_stats, 
          file.path(RESULTS_DIR, paste0(DATASET, "_", cell_type_label, "_snc_state_stats.csv")),
          row.names = FALSE)

cat("  Saved:", paste0(DATASET, "_", cell_type_label, "_snc_state_stats.csv"), "\n")

write.csv(snc_by_state_donor %>%
              group_by(State, Study_Group) %>%
              summarise(
                  mean_pct = mean(pct_snc),
                  sd_pct = sd(pct_snc),
                  n = n(),
                  .groups = "drop"
              ),
          file.path(RESULTS_DIR, paste0(DATASET, "_", cell_type_label, "_snc_by_state_studygroup.csv")),
          row.names = FALSE)

cat("  Saved:", paste0(DATASET, "_", cell_type_label, "_snc_by_state_studygroup.csv"), "\n")

cat("\n[Section 2 Complete]\n")
cat("  Formula:", formula_lm, "\n")
cat("  Covariates:", covariates_display, "\n")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# SECTION 3: VISUALIZATION
# ════════════════════════════════════════════════════════════════════════════════

cat("\n", paste(rep("=", 80), collapse = ""), "\n")
cat("SECTION 3: VISUALIZATION\n")
cat(paste(rep("=", 80), collapse = ""), "\n")

cell_type_label <- gsub("_state$", "", SUBCLUSTER_COL)

# ─────────────────────────────────────────────────────────────────────────────
# STEP 3.1: DISPLAY SUMMARY TABLE
# ─────────────────────────────────────────────────────────────────────────────

cat("\n[Step 3.1] Summary table: Mean %SnC by State x Study Group\n\n")

snc_summary_display <- snc_by_state_donor %>%
    group_by(State, Study_Group) %>%
    summarise(
        mean_pct = mean(pct_snc),
        sd_pct = sd(pct_snc),
        n = n(),
        .groups = "drop"
    ) %>%
    mutate(display = sprintf("%.1f ± %.1f", mean_pct, sd_pct)) %>%
    select(State, Study_Group, display) %>%
    pivot_wider(names_from = Study_Group, values_from = display)

print(as.data.frame(snc_summary_display))

# ─────────────────────────────────────────────────────────────────────────────
# STEP 3.2: DISPLAY MODEL FORMULA AND RESULTS
# ─────────────────────────────────────────────────────────────────────────────

cat("\n[Step 3.2] Model specification\n")

cat("\n", paste(rep("-", 60), collapse = ""), "\n")
cat("MODEL FORMULA\n")
cat(paste(rep("-", 60), collapse = ""), "\n")

cat("\n  Response: pct_cuberoot = (pct_snc / 100)^(1/3)\n")
cat("  Formula:", formula_lm, "\n")
cat("  Predictor:", predictor_desc, "\n")
cat("  Covariates:", paste(covariates_used, collapse = ", "), "\n")
cat("  Multiple testing: BH (FDR) correction\n")

cat("\n", paste(rep("-", 60), collapse = ""), "\n")
cat("RESULTS\n")
cat(paste(rep("-", 60), collapse = ""), "\n\n")

results_display <- snc_state_stats %>%
    select(State, n_donors, estimate, std_error, p_value, p_adj, direction, sig) %>%
    mutate(
        estimate = sprintf("%.4f", estimate),
        std_error = sprintf("%.4f", std_error),
        p_value = sprintf("%.2e", p_value),
        p_adj = sprintf("%.3f", p_adj)
    )

print(as.data.frame(results_display))

if (nrow(sig_results) > 0) {
    cat("\nSignificant associations (FDR < 0.05):\n")
    for (i in 1:nrow(sig_results)) {
        dir_text <- ifelse(sig_results$direction[i] == "Up", 
                           paste0("increases with ", PRIMARY_VAR),
                           paste0("decreases with ", PRIMARY_VAR))
        cat(sprintf("  • %s: SnC %s (b=%s, p_adj=%s)\n", 
                    sig_results$State[i], 
                    dir_text,
                    sprintf("%.4f", sig_results$estimate[i]),
                    sprintf("%.3f", sig_results$p_adj[i])))
    }
} else {
    cat("\nNo significant associations (FDR < 0.05)\n")
}

# ─────────────────────────────────────────────────────────────────────────────
# STEP 3.3: CHECK MAX %SNC PER STATE (DONOR-LEVEL)
# ─────────────────────────────────────────────────────────────────────────────

cat("\n[Step 3.3] Checking max %SnC per state (donor-level)...\n\n")

state_y_stats <- snc_by_state_donor %>%
    group_by(State) %>%
    summarise(
        n_donors = n(),
        min_pct = min(pct_snc, na.rm = TRUE),
        max_pct = max(pct_snc, na.rm = TRUE),
        mean_pct = mean(pct_snc, na.rm = TRUE),
        median_pct = median(pct_snc, na.rm = TRUE),
        .groups = "drop"
    ) %>%
    arrange(desc(max_pct))

cat("  Max %SnC per state (donor-level):\n")
cat("  ", paste(rep("-", 60), collapse = ""), "\n")
cat(sprintf("  %-25s %8s %8s %8s\n", "State", "Min", "Max", "Mean"))
cat("  ", paste(rep("-", 60), collapse = ""), "\n")

for (i in 1:nrow(state_y_stats)) {
    cat(sprintf("  %-25s %7.1f%% %7.1f%% %7.1f%%\n", 
                state_y_stats$State[i],
                state_y_stats$min_pct[i],
                state_y_stats$max_pct[i],
                state_y_stats$mean_pct[i]))
}
cat("  ", paste(rep("-", 60), collapse = ""), "\n")

y_max_lookup <- state_y_stats %>%
    mutate(y_limit = max_pct * 1.15) %>%
    select(State, max_pct, y_limit)

cat("\n  Y-axis limits to use:\n")
for (i in 1:nrow(y_max_lookup)) {
    cat(sprintf("    %s: 0 - %.1f%% (max=%.1f%%)\n", 
                y_max_lookup$State[i],
                y_max_lookup$y_limit[i],
                y_max_lookup$max_pct[i]))
}

# ─────────────────────────────────────────────────────────────────────────────
# STEP 3.4: PREPARE PLOT DATA
# ─────────────────────────────────────────────────────────────────────────────

cat("\n[Step 3.4] Preparing plot data...\n")

sig_lookup <- setNames(snc_state_stats$sig, snc_state_stats$State)
dir_lookup <- setNames(snc_state_stats$direction, snc_state_stats$State)

facet_labels <- sapply(state_order, function(s) {
    sig <- sig_lookup[s]
    dir <- dir_lookup[s]
    if (!is.na(sig) && sig != "") {
        dir_symbol <- ifelse(dir == "Up", "(+)", "(-)")
        paste0(s, " ", dir_symbol, sig)
    } else {
        s
    }
})

label_map <- setNames(facet_labels, state_order)

# ─────────────────────────────────────────────────────────────────────────────
# STEP 3.5: CREATE INDIVIDUAL PLOTS PER STATE
# ─────────────────────────────────────────────────────────────────────────────

cat("\n[Step 3.5] Creating individual plots per state...\n")

suppressPackageStartupMessages({
    if (!require(cowplot, quietly = TRUE)) {
        install.packages("cowplot", quiet = TRUE)
        library(cowplot)
    }
})

plot_list <- list()

for (s in state_order) {
    
    state_data <- snc_by_state_donor %>% filter(State == s)
    state_stats <- snc_state_stats %>% filter(State == s)
    y_limit <- y_max_lookup %>% filter(State == s) %>% pull(y_limit)
    
    p_fmt <- if (state_stats$p_value < 0.001) {
        sprintf("%.1e", state_stats$p_value)
    } else if (state_stats$p_value < 0.01) {
        sprintf("%.3f", state_stats$p_value)
    } else {
        sprintf("%.2f", state_stats$p_value)
    }
    
    label_text <- paste0("b = ", sprintf("%.4f", state_stats$estimate), "\np = ", p_fmt)
    label_color <- ifelse(state_stats$Significant, "#333333", "#888888")
    label_face <- ifelse(state_stats$Significant, "bold", "plain")
    facet_title <- label_map[s]
    
    cat(sprintf("    %s: y_limit = %.1f%%\n", s, y_limit))
    
    p <- ggplot(state_data, aes(x = Study_Group, y = pct_snc)) +
        geom_boxplot(aes(fill = Study_Group), 
                     alpha = 0.7, outlier.shape = NA, width = 0.6,
                     linewidth = 0.5, color = "black") +
        geom_jitter(aes(color = Study_Group), 
                    width = 0.15, size = 1.2, alpha = 0.6) +
        geom_smooth(aes(x = as.numeric(Study_Group), y = pct_snc),
                    method = "lm", se = FALSE,
                    color = "black", linetype = "dashed", linewidth = 0.8) +
        annotate("text", x = 1, y = y_limit * 0.92, 
                 label = label_text, hjust = 0, vjust = 1,
                 size = 2.5, color = label_color, fontface = label_face) +
        coord_cartesian(ylim = c(0, y_limit)) +
        scale_fill_manual(values = STUDY_GROUP_COLORS) +
        scale_color_manual(values = STUDY_GROUP_COLORS) +
        scale_x_discrete(labels = x_labels) +
        labs(title = facet_title, x = NULL, y = NULL) +
        theme_minimal(base_size = 10) +
        theme(
            plot.title = element_text(size = 10, face = "bold", hjust = 0.5),
            axis.text.x = element_text(angle = 45, hjust = 1, vjust = 1, size = 8, 
                                       color = "black", face = "bold"),
            axis.text.y = element_text(size = 8, color = "black"),
            axis.line = element_line(color = "black", linewidth = 0.5),
            axis.ticks = element_line(color = "black", linewidth = 0.3),
            panel.grid = element_blank(),
            panel.border = element_rect(color = "black", fill = NA, linewidth = 0.5),
            legend.position = "none",
            plot.margin = margin(5, 5, 5, 5)
        )
    
    plot_list[[s]] <- p
}

# ─────────────────────────────────────────────────────────────────────────────
# STEP 3.6: COMBINE PLOTS
# ─────────────────────────────────────────────────────────────────────────────

cat("\n[Step 3.6] Combining plots...\n")

n_states <- length(state_order)
n_cols <- min(4, n_states)
n_rows <- ceiling(n_states / n_cols)

fig_width <- 3 * n_cols
fig_height <- 3 * n_rows

options(repr.plot.width = fig_width, repr.plot.height = fig_height)

p_combined <- plot_grid(
    plotlist = plot_list,
    ncol = n_cols,
    align = "hv"
)

p_final <- ggdraw() +
    draw_plot(p_combined, x = 0.03, y = 0.03, width = 0.95, height = 0.88) +
    draw_label(x_title, x = 0.5, y = 0.01, size = 12, fontface = "bold") +
    draw_label("Senescent Cells (%)", x = 0.01, y = 0.5, size = 12, fontface = "bold", angle = 90) +
    draw_label(paste0("Senescent Cell Proportion by ", tools::toTitleCase(cell_type_label), " State — ", DATASET), 
               x = 0.5, y = 0.97, size = 14, fontface = "bold") +
    draw_label(paste0("Model: ", predictor_desc, " + ", paste(covariates_used, collapse = " + ")),
               x = 0.5, y = 0.93, size = 10, color = "gray40")

print(p_final)

# ─────────────────────────────────────────────────────────────────────────────
# STEP 3.7: SAVE FIGURE
# ─────────────────────────────────────────────────────────────────────────────

cat("\n[Step 3.7] Saving figure...\n")

ggsave(file.path(FIGURES_DIR, paste0(DATASET, "_", cell_type_label, "_snc_by_state_boxplot.svg")), 
       p_final, width = fig_width, height = fig_height, dpi = 300)

cat("  Saved:", paste0(DATASET, "_", cell_type_label, "_snc_by_state_boxplot.svg"), "\n")

# ════════════════════════════════════════════════════════════════════════════════
# SUMMARY
# ════════════════════════════════════════════════════════════════════════════════

cat("\n", paste(rep("=", 80), collapse = ""), "\n")
cat("PLOT 4 COMPLETE\n")
cat(paste(rep("=", 80), collapse = ""), "\n")

cat("\nOutputs:\n")
cat("  ", file.path(RESULTS_DIR, paste0(DATASET, "_", cell_type_label, "_snc_state_stats.csv")), "\n")
cat("  ", file.path(RESULTS_DIR, paste0(DATASET, "_", cell_type_label, "_snc_by_state_studygroup.csv")), "\n")
cat("  ", file.path(FIGURES_DIR, paste0(DATASET, "_", cell_type_label, "_snc_by_state_boxplot.svg")), "\n")

cat("\n", paste(rep("=", 80), collapse = ""), "\n")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# STEP 1: DATA PREPARATION — SUBTYPE-LEVEL LOGISTIC GLMM
# ════════════════════════════════════════════════════════════════════════════════

cat("════════════════════════════════════════════════════════════════════════\n")
cat("STEP 1: DATA PREPARATION FOR SUBTYPE GLMM\n")
cat("════════════════════════════════════════════════════════════════════════\n\n")

library(lme4)

COL_DONOR      <- DONOR_COL
COL_SUBTYPE    <- SUBCLUSTER_COL
COL_SEX        <- SEX_COL
COL_COHORT     <- COHORT_COL
COL_SENESCENT  <- "is_senescent"
COL_STUDYGROUP <- STUDY_GROUP_COL

cat("▸ Extracting metadata from seurat_obj...\n")
cat(sprintf("  Study type: %s | Primary variable: %s (%s)\n",
            STUDY_TYPE, PRIMARY_VAR, PRIMARY_VAR_TYPE))

df_glmm <- seurat_obj@meta.data

if (!COL_SENESCENT %in% colnames(df_glmm)) {
    if (SENESCENCE_LABEL_COL %in% colnames(df_glmm)) {
        df_glmm$is_senescent <- as.integer(df_glmm[[SENESCENCE_LABEL_COL]] == "SnC")
        cat("  ✓ Created is_senescent from", SENESCENCE_LABEL_COL, "\n")
    } else {
        stop("No senescence column found.")
    }
} else {
    sen_vals <- df_glmm[[COL_SENESCENT]]
    if (is.character(sen_vals)) {
        df_glmm[[COL_SENESCENT]] <- as.integer(sen_vals %in% c("True", "TRUE", "SnC"))
        cat("  ✓ Converted senescence from character to 0/1\n")
    } else if (is.logical(sen_vals)) {
        df_glmm[[COL_SENESCENT]] <- as.integer(sen_vals)
        cat("  ✓ Converted senescence from logical to 0/1\n")
    }
}

required_cols <- c(COL_DONOR, COL_SUBTYPE, COL_SEX, COL_COHORT, COL_SENESCENT)
if (PRIMARY_VAR_TYPE == "continuous") {
    required_cols <- c(required_cols, PRIMARY_VAR)
} else {
    required_cols <- c(required_cols, COL_STUDYGROUP)
    if (PRIMARY_VAR %in% colnames(df_glmm)) {
        required_cols <- c(required_cols, PRIMARY_VAR)
    }
}

missing <- required_cols[!required_cols %in% colnames(df_glmm)]
if (length(missing) > 0) {
    stop(paste("Missing columns:", paste(missing, collapse = ", ")))
}

cat("▸ Cleaning data...\n")

df_glmm[[COL_DONOR]]   <- as.factor(df_glmm[[COL_DONOR]])
df_glmm[[COL_SUBTYPE]] <- as.factor(df_glmm[[COL_SUBTYPE]])
df_glmm[[COL_SEX]]     <- as.factor(df_glmm[[COL_SEX]])
df_glmm[[COL_COHORT]]  <- as.factor(df_glmm[[COL_COHORT]])

if (PRIMARY_VAR %in% colnames(df_glmm)) {
    df_glmm[[PRIMARY_VAR]] <- as.numeric(df_glmm[[PRIMARY_VAR]])
    df_glmm$Age_scaled <- df_glmm[[PRIMARY_VAR]] / 10
    cat(sprintf("  ✓ Age_scaled computed from %s\n", PRIMARY_VAR))
}

if (PRIMARY_VAR_TYPE == "categorical") {
    df_glmm[[COL_STUDYGROUP]] <- as.factor(df_glmm[[COL_STUDYGROUP]])
    if (!is.null(config$reference_group)) {
        df_glmm[[COL_STUDYGROUP]] <- relevel(df_glmm[[COL_STUDYGROUP]], ref = config$reference_group)
        cat(sprintf("  ✓ Reference group set to: %s\n", config$reference_group))
    }
}

has_counts <- "nCount_RNA" %in% colnames(df_glmm)
if (has_counts) {
    df_glmm$log10_total_counts <- log10(df_glmm$nCount_RNA + 1)
    cat("  ✓ log10_total_counts computed from nCount_RNA\n")
}

check_cols <- c(COL_DONOR, COL_SUBTYPE, COL_SEX, COL_COHORT, COL_SENESCENT)
if ("Age_scaled" %in% colnames(df_glmm)) check_cols <- c(check_cols, "Age_scaled")
if (PRIMARY_VAR_TYPE == "categorical") check_cols <- c(check_cols, COL_STUDYGROUP)

n_before <- nrow(df_glmm)
df_glmm <- df_glmm[complete.cases(df_glmm[, check_cols]), ]
n_after <- nrow(df_glmm)
if (n_before != n_after) {
    cat(sprintf("  ⚠ Dropped %d rows with missing values\n", n_before - n_after))
} else {
    cat(sprintf("  ✓ No missing values — all %s cells retained\n", format(n_after, big.mark = ",")))
}

if (PRIMARY_VAR_TYPE == "continuous") {
    fixed_parts <- "Age_scaled"
} else {
    fixed_parts <- COL_STUDYGROUP
    if ("Age_scaled" %in% colnames(df_glmm)) {
        fixed_parts <- c(fixed_parts, "Age_scaled")
    }
}

if (length(levels(df_glmm[[COL_SEX]])) > 1) fixed_parts <- c(fixed_parts, COL_SEX)
if (length(levels(df_glmm[[COL_COHORT]])) > 1) fixed_parts <- c(fixed_parts, COL_COHORT)
if (has_counts) fixed_parts <- c(fixed_parts, "log10_total_counts")

formula_str <- paste0(COL_SENESCENT, " ~ ", paste(fixed_parts, collapse = " + "),
                      " + (1|", COL_DONOR, ")")
formula_glmm <- as.formula(formula_str)

cat(sprintf("\n▸ Data ready: %s cells, %d donors\n",
            format(nrow(df_glmm), big.mark = ","),
            length(unique(df_glmm[[COL_DONOR]]))))
cat(sprintf("  SnC rate: %.2f%%\n", mean(df_glmm[[COL_SENESCENT]]) * 100))
cat(sprintf("  SnC cells: %s / %s\n",
            format(sum(df_glmm[[COL_SENESCENT]]), big.mark = ","),
            format(nrow(df_glmm), big.mark = ",")))
cat(sprintf("  Formula: %s\n", formula_str))

if (PRIMARY_VAR_TYPE == "categorical") {
    cat(sprintf("  Study groups: %s\n",
                paste(levels(df_glmm[[COL_STUDYGROUP]]), collapse = " vs ")))
}

cat(sprintf("\n▸ Subtypes (%d):\n", length(levels(df_glmm[[COL_SUBTYPE]]))))

for (st in levels(df_glmm[[COL_SUBTYPE]])) {
    sub <- df_glmm[df_glmm[[COL_SUBTYPE]] == st, ]
    n <- nrow(sub)
    nd <- length(unique(sub[[COL_DONOR]]))
    sr <- mean(sub[[COL_SENESCENT]]) * 100
    flag <- if (n < 100 | nd < 15) " ⚠ (may be skipped)" else ""
    cat(sprintf("  %-25s n=%6s  donors=%3d  SnC=%.1f%%%s\n",
                st, format(n, big.mark = ","), nd, sr, flag))
}

cat("\n════════════════════════════════════════════════════════════════════════\n")
cat("✓ STEP 1 COMPLETE — df_glmm and formula_glmm ready\n")
cat("════════════════════════════════════════════════════════════════════════\n")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# STEP 2: SUBTYPE-LEVEL LOGISTIC GLMM — MODEL FITTING
# ════════════════════════════════════════════════════════════════════════════════

cat("════════════════════════════════════════════════════════════════════════\n")
cat("STEP 2: SUBTYPE-LEVEL LOGISTIC GLMM\n")
cat("════════════════════════════════════════════════════════════════════════\n\n")

MIN_CELLS  <- 100
MIN_DONORS <- 15

extract_primary_effect <- function(coefs) {
    if (PRIMARY_VAR_TYPE == "continuous") {
        if ("Age_scaled" %in% rownames(coefs)) {
            return(list(
                term = "Age_scaled",
                beta = coefs["Age_scaled", "Estimate"],
                se   = coefs["Age_scaled", "Std. Error"],
                z    = coefs["Age_scaled", "z value"],
                p    = coefs["Age_scaled", "Pr(>|z|)"]
            ))
        }
    } else {
        sg_rows <- grep(COL_STUDYGROUP, rownames(coefs))
        if (length(sg_rows) > 0) {
            p_vals <- coefs[sg_rows, "Pr(>|z|)"]
            best <- sg_rows[which.min(p_vals)]
            return(list(
                term = rownames(coefs)[best],
                beta = coefs[best, "Estimate"],
                se   = coefs[best, "Std. Error"],
                z    = coefs[best, "z value"],
                p    = coefs[best, "Pr(>|z|)"]
            ))
        }
    }
    return(NULL)
}

# ═══════════════════════════════════════════════════════════════════════════════
# OVERALL GLMM
# ═══════════════════════════════════════════════════════════════════════════════

cat("─────────────────────────────────────────────────────────────────────\n")
cat("1. OVERALL GLMM (ALL CELLS)\n")
cat("─────────────────────────────────────────────────────────────────────\n")

cat(sprintf("\n▸ Formula: %s\n", deparse(formula_glmm)))
cat("▸ Fitting overall model...\n\n")

overall_model <- tryCatch({
    glmer(
        formula_glmm,
        data = df_glmm,
        family = binomial(link = "logit"),
        control = glmerControl(optimizer = "bobyqa", optCtrl = list(maxfun = 100000)),
        nAGQ = 1
    )
}, error = function(e) {
    cat(sprintf("✗ Overall model failed: %s\n", e$message))
    NULL
})

if (!is.null(overall_model)) {
    overall_coefs <- summary(overall_model)$coefficients
    re_var <- as.numeric(VarCorr(overall_model)[[COL_DONOR]])

    cat("✓ Model converged\n")
    cat(sprintf("  Observations: %s  |  Donors: %d  |  AIC: %.1f\n",
                format(nobs(overall_model), big.mark = ","),
                ngrps(overall_model),
                AIC(overall_model)))

    cat(sprintf("\n  %-30s %10s %10s %10s %12s %8s\n",
                "Term", "B", "SE", "z", "p-value", "OR"))
    cat(paste0("  ", paste(rep("-", 85), collapse = ""), "\n"))

    for (i in seq_len(nrow(overall_coefs))) {
        term <- rownames(overall_coefs)[i]
        beta <- overall_coefs[i, "Estimate"]
        se   <- overall_coefs[i, "Std. Error"]
        z    <- overall_coefs[i, "z value"]
        p    <- overall_coefs[i, "Pr(>|z|)"]
        or   <- exp(beta)
        sig  <- ifelse(p < 0.05, "*", "")
        cat(sprintf("  %-30s %10.4f %10.4f %10.3f %12.2e %8.3f %s\n",
                    term, beta, se, z, p, or, sig))
    }

    cat(sprintf("\n  Random Effects: Donor variance = %.4f, SD = %.4f\n",
                re_var, sqrt(re_var)))

    primary_eff <- extract_primary_effect(overall_coefs)
    if (!is.null(primary_eff)) {
        or_val <- exp(primary_eff$beta)
        or_lo  <- exp(primary_eff$beta - 1.96 * primary_eff$se)
        or_hi  <- exp(primary_eff$beta + 1.96 * primary_eff$se)

        if (PRIMARY_VAR_TYPE == "continuous") {
            cat(sprintf("\n▸ Age Effect (per decade):\n"))
        } else {
            cat(sprintf("\n▸ Disease Effect (%s):\n", primary_eff$term))
        }
        cat(sprintf("  B = %.4f (SE = %.4f)\n", primary_eff$beta, primary_eff$se))
        cat(sprintf("  OR = %.3f (95%% CI: %.3f - %.3f)\n", or_val, or_lo, or_hi))
        cat(sprintf("  p = %.2e\n", primary_eff$p))

        if (primary_eff$p < 0.05) {
            direction <- ifelse(primary_eff$beta > 0, "INCREASES", "DECREASES")
            pct <- abs((or_val - 1) * 100)
            cat(sprintf("  ✓ Senescence probability %s (+%.1f%% odds)\n", direction, pct))
        }
    }

    if (PRIMARY_VAR_TYPE == "categorical") {
        sg_rows <- grep(COL_STUDYGROUP, rownames(overall_coefs))
        if (length(sg_rows) > 1) {
            cat(sprintf("\n▸ All disease group effects (vs %s):\n", config$reference_group))
            for (i in sg_rows) {
                term <- gsub(COL_STUDYGROUP, "", rownames(overall_coefs)[i])
                beta <- overall_coefs[i, "Estimate"]
                or   <- exp(beta)
                p    <- overall_coefs[i, "Pr(>|z|)"]
                sig  <- ifelse(p < 0.05, "*", "")
                cat(sprintf("  %s: B=%.4f, OR=%.3f, p=%.2e %s\n", term, beta, or, p, sig))
            }
        }
    }
}

# ═══════════════════════════════════════════════════════════════════════════════
# SUBTYPE-SPECIFIC GLMM
# ═══════════════════════════════════════════════════════════════════════════════

cat("\n─────────────────────────────────────────────────────────────────────\n")
cat("2. SUBTYPE-SPECIFIC GLMM\n")
cat("─────────────────────────────────────────────────────────────────────\n")

subtypes <- levels(df_glmm[[COL_SUBTYPE]])
cat(sprintf("\n▸ Running GLMM for %d subtypes...\n\n", length(subtypes)))

results_list <- list()

for (subtype in subtypes) {

    sub_data <- df_glmm[df_glmm[[COL_SUBTYPE]] == subtype, ]

    n_cells  <- nrow(sub_data)
    n_donors <- length(unique(sub_data[[COL_DONOR]]))
    snc_rate <- mean(sub_data[[COL_SENESCENT]])

    if (n_cells < MIN_CELLS) {
        cat(sprintf("  %-25s SKIPPED (n=%d < %d)\n", subtype, n_cells, MIN_CELLS))
        next
    }
    if (n_donors < MIN_DONORS) {
        cat(sprintf("  %-25s SKIPPED (donors=%d < %d)\n", subtype, n_donors, MIN_DONORS))
        next
    }
    if (snc_rate == 0 | snc_rate == 1) {
        cat(sprintf("  %-25s SKIPPED (no variance)\n", subtype))
        next
    }

    model <- tryCatch({
        glmer(
            formula_glmm,
            data = sub_data,
            family = binomial(link = "logit"),
            control = glmerControl(optimizer = "bobyqa", optCtrl = list(maxfun = 50000)),
            nAGQ = 1
        )
    }, error = function(e) {
        cat(sprintf("  %-25s FAILED - %s\n", subtype, substr(e$message, 1, 60)))
        NULL
    })

    if (!is.null(model)) {
        coefs <- summary(model)$coefficients
        re_var <- as.numeric(VarCorr(model)[[COL_DONOR]])

        if (PRIMARY_VAR_TYPE == "categorical") {
            sg_rows <- grep(COL_STUDYGROUP, rownames(coefs))
            for (idx in sg_rows) {
                term_name <- gsub(COL_STUDYGROUP, "", rownames(coefs)[idx])
                beta <- coefs[idx, "Estimate"]
                se   <- coefs[idx, "Std. Error"]
                z    <- coefs[idx, "z value"]
                p    <- coefs[idx, "Pr(>|z|)"]

                results_list[[paste0(subtype, "_", term_name)]] <- data.frame(
                    Subtype        = subtype,
                    Comparison     = paste0(term_name, " vs ", config$reference_group),
                    N_cells        = n_cells,
                    N_donors       = n_donors,
                    SnC_rate       = snc_rate,
                    Beta           = beta,
                    SE             = se,
                    Beta_CI_lower  = beta - 1.96 * se,
                    Beta_CI_upper  = beta + 1.96 * se,
                    OR             = exp(beta),
                    OR_CI_lower    = exp(beta - 1.96 * se),
                    OR_CI_upper    = exp(beta + 1.96 * se),
                    z_value        = z,
                    P_value        = p,
                    Donor_variance = re_var,
                    AIC            = AIC(model),
                    stringsAsFactors = FALSE
                )
            }
            best_idx <- sg_rows[which.min(coefs[sg_rows, "Pr(>|z|)"])]
            beta <- coefs[best_idx, "Estimate"]
            p    <- coefs[best_idx, "Pr(>|z|)"]
            term <- gsub(COL_STUDYGROUP, "", rownames(coefs)[best_idx])
            sig  <- ifelse(p < 0.05, "*", "")
            cat(sprintf("  %-25s %s: B=%.4f  OR=%.3f  p=%.2e %s\n",
                        subtype, term, beta, exp(beta), p, sig))

        } else {
            primary_eff <- extract_primary_effect(coefs)
            if (!is.null(primary_eff)) {
                beta <- primary_eff$beta
                se   <- primary_eff$se
                z    <- primary_eff$z
                p    <- primary_eff$p
            } else {
                beta <- NA; se <- NA; z <- NA; p <- NA
            }

            results_list[[subtype]] <- data.frame(
                Subtype        = subtype,
                Comparison     = "per decade",
                N_cells        = n_cells,
                N_donors       = n_donors,
                SnC_rate       = snc_rate,
                Beta           = beta,
                SE             = se,
                Beta_CI_lower  = beta - 1.96 * se,
                Beta_CI_upper  = beta + 1.96 * se,
                OR             = exp(beta),
                OR_CI_lower    = exp(beta - 1.96 * se),
                OR_CI_upper    = exp(beta + 1.96 * se),
                z_value        = z,
                P_value        = p,
                Donor_variance = re_var,
                AIC            = AIC(model),
                stringsAsFactors = FALSE
            )

            sig <- ifelse(p < 0.05, "*", "")
            cat(sprintf("  %-25s B=%.4f  OR=%.3f  p=%.2e %s\n",
                        subtype, beta, exp(beta), p, sig))
        }
    }
}

# ─────────────────────────────────────────────────────────────────────────────
# FDR correction
# ─────────────────────────────────────────────────────────────────────────────

df_glmm_results <- do.call(rbind, results_list)
rownames(df_glmm_results) <- NULL

if (nrow(df_glmm_results) > 0) {

    df_glmm_results$P_adj <- p.adjust(df_glmm_results$P_value, method = "BH")
    df_glmm_results$Significant <- df_glmm_results$P_adj < 0.05
    df_glmm_results <- df_glmm_results[order(df_glmm_results$P_value), ]

    n_sig <- sum(df_glmm_results$Significant)
    cat(sprintf("\n▸ Results: %d / %d significant (FDR < 0.05)\n",
                n_sig, nrow(df_glmm_results)))

    cat(sprintf("\n%-25s %-20s %7s %6s %8s %20s %6s %10s %10s %4s\n",
                "Subtype", "Comparison", "N", "%SnC", "B", "95% CI (B)", "OR",
                "P", "FDR", "Sig"))
    cat(paste(rep("-", 130), collapse = ""), "\n")

    for (i in seq_len(nrow(df_glmm_results))) {
        row <- df_glmm_results[i, ]
        sig_mark <- ifelse(row$Significant, "Y", "")
        ci_str <- sprintf("[%.3f, %.3f]", row$Beta_CI_lower, row$Beta_CI_upper)
        cat(sprintf("%-25s %-20s %7s %5.1f%% %8.4f %20s %6.2f %10.2e %10.2e %4s\n",
                    row$Subtype,
                    row$Comparison,
                    format(row$N_cells, big.mark = ","),
                    row$SnC_rate * 100,
                    row$Beta,
                    ci_str,
                    row$OR,
                    row$P_value,
                    row$P_adj,
                    sig_mark))
    }

    results_file <- file.path(RESULTS_DIR, paste0(DATASET, "_", cell_type_label, "_subtype_glmm_results.csv"))
    write.csv(df_glmm_results, results_file, row.names = FALSE)
    cat(sprintf("\n✓ Saved: %s\n", results_file))

    if (PRIMARY_VAR_TYPE == "continuous") {
        var_label <- "Age"; unit_label <- "per decade"
    } else {
        var_label <- "Disease"; unit_label <- paste0("vs ", config$reference_group)
    }

    sig_up   <- df_glmm_results[df_glmm_results$Significant & df_glmm_results$Beta > 0, ]
    sig_down <- df_glmm_results[df_glmm_results$Significant & df_glmm_results$Beta < 0, ]

    if (nrow(sig_up) > 0) {
        cat(sprintf("\n▸ INCREASING senescence (%s):\n", var_label))
        for (i in seq_len(nrow(sig_up))) {
            row <- sig_up[i, ]
            pct <- (row$OR - 1) * 100
            cat(sprintf("   %s [%s]: OR=%.2f (+%.1f%% %s, FDR=%.2e)\n",
                        row$Subtype, row$Comparison, row$OR, pct, unit_label, row$P_adj))
        }
    }

    if (nrow(sig_down) > 0) {
        cat(sprintf("\n▸ DECREASING senescence (%s):\n", var_label))
        for (i in seq_len(nrow(sig_down))) {
            row <- sig_down[i, ]
            pct <- (1 - row$OR) * 100
            cat(sprintf("   %s [%s]: OR=%.2f (-%.1f%% %s, FDR=%.2e)\n",
                        row$Subtype, row$Comparison, row$OR, pct, unit_label, row$P_adj))
        }
    }

    if (nrow(sig_up) == 0 & nrow(sig_down) == 0) {
        cat("\n▸ No significant associations at FDR < 0.05\n")
    }

} else {
    cat("\n⚠ No subtypes passed filters\n")
}

cat("\n════════════════════════════════════════════════════════════════════════\n")
cat("✓ STEP 2 COMPLETE — df_glmm_results ready for plotting\n")
cat("════════════════════════════════════════════════════════════════════════\n")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# STEP 3: FOREST PLOT (TABLE STYLE) — SUBTYPE GLMM
# ════════════════════════════════════════════════════════════════════════════════

cat("════════════════════════════════════════════════════════════════════════\n")
cat("STEP 3: FOREST PLOT — SUBTYPE GLMM\n")
cat("════════════════════════════════════════════════════════════════════════\n\n")

if (nrow(df_glmm_results) == 0) {
    cat("No results to plot.\n")
} else {

    df_plot <- df_glmm_results[order(df_glmm_results$OR, decreasing = TRUE), ]
    n_rows <- nrow(df_plot)

    fig_width  <- 7
    fig_height <- max(3, n_rows * 0.5 + 2)

    x_sub  <- 0.01
    x_fl   <- 0.19
    x_fr   <- 0.58
    x_or   <- 0.72
    x_fdr  <- 0.93

    betas <- df_plot$Beta
    iqr <- IQR(betas)
    center <- median(betas)
    display_min <- min(betas) - max(iqr * 1.5, 0.15)
    display_max <- max(betas) + max(iqr * 1.5, 0.15)

    beta_to_x <- function(val) {
        v <- pmax(pmin(val, display_max), display_min)
        x_fl + (v - display_min) / (display_max - display_min) * (x_fr - x_fl)
    }

    x_null <- beta_to_x(0)

    cands <- c(0.5, 0.6, 0.7, 0.75, 0.8, 0.85, 0.9, 0.95,
               1.0, 1.05, 1.1, 1.15, 1.2, 1.3, 1.5, 2.0)
    valid <- cands[cands >= exp(display_min) & cands <= exp(display_max)]

    min_sp <- (x_fr - x_fl) * 0.08
    ticks <- valid[1]
    for (t in valid[-1]) {
        if (abs(beta_to_x(log(t)) - beta_to_x(log(ticks[length(ticks)]))) >= min_sp) {
            ticks <- c(ticks, t)
        }
    }

    draw_forest <- function() {

        par(mar = c(1.8, 0.3, 1.8, 0.3), family = "sans")
        plot.new()
        plot.window(xlim = c(0, 1), ylim = c(-1, n_rows + 0.8))

        text(0.5, n_rows + 0.5,
             paste0(tools::toTitleCase(cell_type_label),
                    " Subtype — Age vs Senescence (", DATASET, ")"),
             cex = 0.8, font = 2, adj = 0.5)

        yh <- n_rows + 0.05
        text(x_sub, yh, "Subtype", cex = 0.65, font = 2, adj = 0)
        text((x_fl + x_fr) / 2, yh, "log-OR", cex = 0.65, font = 2, adj = 0.5)
        text(x_or, yh, "OR [95% CI]", cex = 0.65, font = 2, adj = 0.5)
        text(x_fdr, yh, "FDR", cex = 0.65, font = 2, adj = 0.5)
        segments(0, yh - 0.15, 0.99, yh - 0.15, lwd = 0.6)

        segments(x_null, -0.3, x_null, n_rows - 0.5,
                 lty = 2, col = "#aaaaaa", lwd = 0.5)

        for (i in seq_len(n_rows)) {
            row <- df_plot[i, ]
            y <- n_rows - i

            sig_fdr <- row$Significant
            sig_p   <- row$P_value < 0.05

            if (i %% 2 == 0) {
                rect(0, y - 0.28, 0.99, y + 0.28,
                     col = "#fafafa", border = NA)
            }

            lbl <- as.character(row$Subtype)
            if (sig_fdr) lbl <- paste0("* ", lbl)
            text(x_sub, y, lbl,
                 cex = 0.62, font = ifelse(sig_p, 2, 1), adj = 0)

            cc <- state_colors[row$Subtype]
            if (is.na(cc)) cc <- "#808080"

            ci_l <- row$Beta_CI_lower
            ci_h <- row$Beta_CI_upper
            xl <- beta_to_x(ci_l)
            xh <- beta_to_x(ci_h)
            xb <- beta_to_x(row$Beta)

            clip_l <- ci_l < display_min
            clip_r <- ci_h > display_max

            segments(xl, y, xh, y, col = cc, lwd = 1.8, lend = 1)

            if (clip_l) {
                polygon(x = c(xl, xl + 0.008, xl + 0.008),
                        y = c(y, y + 0.08, y - 0.08),
                        col = cc, border = NA)
            } else {
                segments(xl, y - 0.08, xl, y + 0.08, col = cc, lwd = 0.8)
            }

            if (clip_r) {
                polygon(x = c(xh, xh - 0.008, xh - 0.008),
                        y = c(y, y + 0.08, y - 0.08),
                        col = cc, border = NA)
            } else {
                segments(xh, y - 0.08, xh, y + 0.08, col = cc, lwd = 0.8)
            }

            dw <- 0.005; dh <- 0.13
            polygon(x = c(xb - dw, xb, xb + dw, xb),
                    y = c(y, y - dh, y, y + dh),
                    col = cc, border = "black", lwd = 0.4)

            text(x_or, y + 0.09,
                 sprintf("%.2f", row$OR),
                 cex = 0.58, adj = 0.5,
                 font = ifelse(sig_p, 2, 1))
            text(x_or, y - 0.09,
                 sprintf("[%.2f-%.2f]", row$OR_CI_lower, row$OR_CI_upper),
                 cex = 0.50, adj = 0.5, col = "#777777")

            pv <- row$P_adj
            if (pv < 0.001) {
                ptxt <- formatC(pv, format = "e", digits = 1)
            } else if (pv < 0.01) {
                ptxt <- formatC(pv, format = "f", digits = 3)
            } else {
                ptxt <- formatC(pv, format = "f", digits = 2)
            }
            text(x_fdr, y, ptxt,
                 cex = 0.58, adj = 0.5,
                 font = ifelse(sig_fdr, 2, 1),
                 col = ifelse(sig_fdr, "#C44E52", "#333333"))
        }

        ya <- -0.45
        segments(x_fl, ya, x_fr, ya, lwd = 0.6)

        for (t in ticks) {
            xt <- beta_to_x(log(t))
            segments(xt, ya, xt, ya - 0.06, lwd = 0.5)
            text(xt, ya - 0.15, t, cex = 0.5, adj = 0.5)
        }

        text((x_fl + x_fr) / 2, ya - 0.35,
             "Odds Ratio", cex = 0.55, font = 2, adj = 0.5)

        segments(0, ya - 0.5, 0.99, ya - 0.5, lwd = 0.3, col = "#dddddd")
        text(x_sub, ya - 0.7, "* FDR < 0.05",
             cex = 0.48, adj = 0, col = "#999999")

        if (any(df_plot$Beta_CI_lower < display_min) |
            any(df_plot$Beta_CI_upper > display_max)) {
            text(x_fr, ya - 0.7, "< > CI clipped",
                 cex = 0.48, adj = 1, col = "#999999")
        }
    }

    options(repr.plot.width = fig_width, repr.plot.height = fig_height)
    draw_forest()

    forest_file <- file.path(FIGURES_DIR,
                             paste0(DATASET, "_", cell_type_label, "_subtype_glmm_forest.svg"))
    svg(forest_file, width = fig_width, height = fig_height)
    draw_forest()
    dev.off()

    cat("Saved:", forest_file, "\n")
}

cat("\n════════════════════════════════════════════════════════════════════════\n")
cat("STEP 3 COMPLETE\n")
cat("════════════════════════════════════════════════════════════════════════\n")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# PLOT 3: UMAP PER STUDY GROUP
# ════════════════════════════════════════════════════════════════════════════════

cat("\n--- Plot 3: UMAP per Study Group ---\n")

cell_type_label <- gsub("_state$", "", SUBCLUSTER_COL)
is_aging <- STUDY_TYPE == "aging"

# ─────────────────────────────────────────────────────────────────────────────
# SETUP
# ─────────────────────────────────────────────────────────────────────────────

sg_detected <- unique(seurat_obj@meta.data[[STUDY_GROUP_COL]])
sg_order <- names(STUDY_GROUP_COLORS)[names(STUDY_GROUP_COLORS) %in% sg_detected]

if (is_aging) {
    sg_labels <- setNames(gsub("Age_", "", gsub("_", "-", sg_order)), sg_order)
} else {
    sg_labels <- setNames(gsub("_", " ", sg_order), sg_order)
}

cat("Study groups:", paste(sg_order, collapse = ", "), "\n")

state_order <- seurat_obj@meta.data %>%
    count(.data[[SUBCLUSTER_COL]]) %>%
    arrange(desc(n)) %>%
    pull(.data[[SUBCLUSTER_COL]])

state_colors <- get_state_colors(state_order)

cat("States:", paste(state_order, collapse = ", "), "\n")

# ─────────────────────────────────────────────────────────────────────────────
# HELPER FUNCTIONS
# ─────────────────────────────────────────────────────────────────────────────

add_corner_arrows <- function(p, label_size = 3) {
    build <- ggplot_build(p)
    x_range <- build$layout$panel_params[[1]]$x.range
    y_range <- build$layout$panel_params[[1]]$y.range
    
    x_start <- x_range[1] + diff(x_range) * 0.02
    y_start <- y_range[1] + diff(y_range) * 0.02
    x_arrow <- diff(x_range) * 0.12
    y_arrow <- diff(y_range) * 0.12
    
    p + 
        annotate("segment", x = x_start, xend = x_start + x_arrow, y = y_start, yend = y_start,
                 arrow = arrow(length = unit(0.12, "cm"), type = "closed"), linewidth = 0.4) +
        annotate("text", x = x_start + x_arrow/2, y = y_start - diff(y_range) * 0.04,
                 label = "UMAP1", size = label_size, hjust = 0.5, vjust = 1, fontface = "bold") +
        annotate("segment", x = x_start, xend = x_start, y = y_start, yend = y_start + y_arrow,
                 arrow = arrow(length = unit(0.12, "cm"), type = "closed"), linewidth = 0.4) +
        annotate("text", x = x_start - diff(x_range) * 0.04, y = y_start + y_arrow/2,
                 label = "UMAP2", size = label_size, hjust = 1, vjust = 0.5, angle = 90, fontface = "bold") +
        coord_cartesian(clip = "off")
}

umap_theme <- theme_void(base_size = 12) +
    theme(
        plot.title = element_text(size = 13, face = "bold", hjust = 0.5),
        legend.position = "none",
        plot.margin = margin(15, 15, 20, 20)
    )

# ─────────────────────────────────────────────────────────────────────────────
# GENERATE UMAPS
# ─────────────────────────────────────────────────────────────────────────────

umap_list <- list()

for (sg in sg_order) {
    cells_sg <- colnames(seurat_obj)[seurat_obj@meta.data[[STUDY_GROUP_COL]] == sg]
    n_cells <- length(cells_sg)
    sg_label <- sg_labels[sg]
    
    p <- DimPlot(seurat_obj, reduction = "umap", cells = cells_sg,
                 group.by = SUBCLUSTER_COL, pt.size = 0.3) +
        scale_color_manual(values = state_colors) +
        ggtitle(paste0(sg_label, "\n(n=", format(n_cells, big.mark = ","), ")")) +
        umap_theme
    
    p <- add_corner_arrows(p, label_size = 2.5)
    umap_list[[sg]] <- p
}

# ─────────────────────────────────────────────────────────────────────────────
# SHARED LEGEND
# ─────────────────────────────────────────────────────────────────────────────

p_legend <- DimPlot(seurat_obj, reduction = "umap", group.by = SUBCLUSTER_COL, pt.size = 0.5) +
    scale_color_manual(values = state_colors) +
    theme_void() +
    theme(
        legend.position = "right",
        legend.title = element_text(size = 12, face = "bold"),
        legend.text = element_text(size = 10),
        legend.key.size = unit(0.5, "cm")
    ) +
    labs(color = paste0(tools::toTitleCase(cell_type_label), " State")) +
    guides(color = guide_legend(ncol = 1, override.aes = list(size = 4)))

legend <- cowplot::get_legend(p_legend)

# ─────────────────────────────────────────────────────────────────────────────
# COMBINE
# ─────────────────────────────────────────────────────────────────────────────

n_groups <- length(sg_order)
n_cols <- 4
n_rows <- ceiling(n_groups / n_cols)

while (length(umap_list) < n_rows * n_cols) {
    umap_list[[length(umap_list) + 1]] <- plot_spacer()
}

options(repr.plot.width = 16, repr.plot.height = 4 * n_rows + 1)

umap_grid <- wrap_plots(umap_list, ncol = n_cols)

title_text <- if (is_aging) {
    paste0(tools::toTitleCase(cell_type_label), " States by Age Group")
} else {
    paste0(tools::toTitleCase(cell_type_label), " States by Disease Group")
}

umap_panel_sg <- (umap_grid | legend) + 
    plot_layout(widths = c(1, 0.15)) +
    plot_annotation(
        title = title_text,
        theme = theme(plot.title = element_text(size = 16, face = "bold", hjust = 0.5))
    )

print(umap_panel_sg)

ggsave(file.path(FIGURES_DIR, paste0(DATASET, "_", cell_type_label, "_umap_by_studygroup.svg")), 
       umap_panel_sg, width = 16, height = 4 * n_rows + 1, dpi = 300)

cat("\n✓ Saved:", paste0(DATASET, "_", cell_type_label, "_umap_by_studygroup.svg"), "\n")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# PLOT: UMAP PER STUDY GROUP (SnC HIGHLIGHT)
# ════════════════════════════════════════════════════════════════════════════════

cat("\n--- Plot: UMAP per Study Group (SnC Highlight) ---\n")

cell_type_label <- gsub("_state$", "", SUBCLUSTER_COL)
is_aging <- STUDY_TYPE == "aging"

sg_detected <- unique(seurat_obj@meta.data[[STUDY_GROUP_COL]])
sg_order <- names(STUDY_GROUP_COLORS)[names(STUDY_GROUP_COLORS) %in% sg_detected]

if (is_aging) {
    sg_labels <- setNames(gsub("Age_", "", gsub("_", "-", sg_order)), sg_order)
} else {
    sg_labels <- setNames(gsub("_", " ", sg_order), sg_order)
}

cat("Study groups:", paste(sg_order, collapse = ", "), "\n")
cat("Senescence column:", SENESCENCE_LABEL_COL, "\n")

add_corner_arrows <- function(p, label_size = 3) {
    build <- ggplot_build(p)
    x_range <- build$layout$panel_params[[1]]$x.range
    y_range <- build$layout$panel_params[[1]]$y.range
    
    x_start <- x_range[1] + diff(x_range) * 0.02
    y_start <- y_range[1] + diff(y_range) * 0.02
    x_arrow <- diff(x_range) * 0.12
    y_arrow <- diff(y_range) * 0.12
    
    p + 
        annotate("segment", x = x_start, xend = x_start + x_arrow, y = y_start, yend = y_start,
                 arrow = arrow(length = unit(0.12, "cm"), type = "closed"), linewidth = 0.4) +
        annotate("text", x = x_start + x_arrow/2, y = y_start - diff(y_range) * 0.04,
                 label = "UMAP1", size = label_size, hjust = 0.5, vjust = 1, fontface = "bold") +
        annotate("segment", x = x_start, xend = x_start, y = y_start, yend = y_start + y_arrow,
                 arrow = arrow(length = unit(0.12, "cm"), type = "closed"), linewidth = 0.4) +
        annotate("text", x = x_start - diff(x_range) * 0.04, y = y_start + y_arrow/2,
                 label = "UMAP2", size = label_size, hjust = 1, vjust = 0.5, angle = 90, fontface = "bold") +
        coord_cartesian(clip = "off")
}

umap_theme <- theme_void(base_size = 12) +
    theme(
        plot.title = element_text(size = 13, face = "bold", hjust = 0.5),
        legend.position = "none",
        plot.margin = margin(15, 15, 20, 20)
    )

umap_list <- list()

for (sg in sg_order) {
    cells_sg <- colnames(seurat_obj)[seurat_obj@meta.data[[STUDY_GROUP_COL]] == sg]
    n_cells <- length(cells_sg)
    n_snc <- sum(seurat_obj@meta.data[cells_sg, SENESCENCE_LABEL_COL] == "SnC")
    sg_label <- sg_labels[sg]
    
    p <- DimPlot(seurat_obj, reduction = "umap", cells = cells_sg,
                 group.by = SENESCENCE_LABEL_COL, pt.size = 0.3,
                 order = c("SnC", "Non-SnC")) +
        scale_color_manual(values = SENESCENCE_COLORS) +
        ggtitle(paste0(sg_label, "\n(n=", format(n_cells, big.mark = ","), 
                       ", SnC=", format(n_snc, big.mark = ","), ")")) +
        umap_theme
    
    p <- add_corner_arrows(p, label_size = 2.5)
    umap_list[[sg]] <- p
}

p_legend <- DimPlot(seurat_obj, reduction = "umap", group.by = SENESCENCE_LABEL_COL, 
                    pt.size = 0.5, order = c("SnC", "Non-SnC")) +
    scale_color_manual(values = SENESCENCE_COLORS) +
    theme_void() +
    theme(
        legend.position = "right",
        legend.title = element_text(size = 12, face = "bold"),
        legend.text = element_text(size = 10),
        legend.key.size = unit(0.5, "cm")
    ) +
    labs(color = "Senescence") +
    guides(color = guide_legend(ncol = 1, override.aes = list(size = 4)))

legend <- cowplot::get_legend(p_legend)

n_groups <- length(sg_order)
n_cols <- 4
n_rows <- ceiling(n_groups / n_cols)

while (length(umap_list) < n_rows * n_cols) {
    umap_list[[length(umap_list) + 1]] <- plot_spacer()
}

options(repr.plot.width = 16, repr.plot.height = 4 * n_rows + 1)

umap_grid <- wrap_plots(umap_list, ncol = n_cols)

title_text <- if (is_aging) {
    paste0(tools::toTitleCase(cell_type_label), " Senescence by Age Group")
} else {
    paste0(tools::toTitleCase(cell_type_label), " Senescence by Disease Group")
}

umap_panel_snc <- (umap_grid | legend) + 
    plot_layout(widths = c(1, 0.15)) +
    plot_annotation(
        title = title_text,
        theme = theme(plot.title = element_text(size = 16, face = "bold", hjust = 0.5))
    )

print(umap_panel_snc)

ggsave(file.path(FIGURES_DIR, paste0(DATASET, "_", cell_type_label, "_umap_snc_by_studygroup.svg")), 
       umap_panel_snc, width = 16, height = 4 * n_rows + 1, dpi = 300)

cat("\n✓ Saved:", paste0(DATASET, "_", cell_type_label, "_umap_snc_by_studygroup.svg"), "\n")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# SAVE ANNOTATED OBJECT
# ════════════════════════════════════════════════════════════════════════════════

cat("\n", paste(rep("=", 80), collapse = ""), "\n")
cat(sprintf("SAVING ANNOTATED %s OBJECT\n", toupper(cell_type_label)))
cat(paste(rep("=", 80), collapse = ""), "\n")

save_file <- file.path(OUTPUT_DIR, paste0(DATASET, "_", cell_type_label, "_annotated.qs"))
qsave(seurat_obj, save_file)

cat("\n✓ Saved:", save_file, "\n")
cat("  Cells:", ncol(seurat_obj), "\n")
cat("  States:", length(unique(seurat_obj@meta.data[[SUBCLUSTER_COL]])), "\n")
cat("  Clusters:", length(unique(seurat_obj$seurat_clusters)), "\n")

# ════════════════════════════════════════════════════════════════════════════════
# MODULE SUMMARY
# ════════════════════════════════════════════════════════════════════════════════

cat("\n", paste(rep("=", 80), collapse = ""), "\n")
cat(sprintf("SUMMARY: %s SUBCLUSTERING & ANNOTATION\n", toupper(cell_type_label)))
cat(paste(rep("=", 80), collapse = ""), "\n")

cat("\n--- Data Summary ---\n")
cat("  Total cells:", ncol(seurat_obj), "\n")
cat("  Donors:", length(unique(seurat_obj@meta.data[[DONOR_COL]])), "\n")
cat("  Study groups:", paste(sg_order, collapse = ", "), "\n")

cat(sprintf("\n--- %s States ---\n", tools::toTitleCase(cell_type_label)))
state_counts <- table(seurat_obj@meta.data[[SUBCLUSTER_COL]])
for (s in names(sort(state_counts, decreasing = TRUE))) {
    cat(sprintf("  %s: %d (%.1f%%)\n", s, state_counts[s], state_counts[s] / sum(state_counts) * 100))
}

cat("\n--- Senescence ---\n")
cat(sprintf("  Label column: %s\n", SENESCENCE_LABEL_COL))
snc_counts <- table(seurat_obj@meta.data[[SENESCENCE_LABEL_COL]])
cat(sprintf("  SnC: %d (%.1f%%)\n", snc_counts["SnC"], snc_counts["SnC"] / sum(snc_counts) * 100))
cat(sprintf("  Non-SnC: %d (%.1f%%)\n", snc_counts["Non-SnC"], snc_counts["Non-SnC"] / sum(snc_counts) * 100))

if ("senescence_label_state" %in% colnames(seurat_obj@meta.data) && 
    SENESCENCE_LABEL_COL == "senescence_label_state") {
    cat("\n  (Original labels for comparison:)\n")
    orig_counts <- table(seurat_obj@meta.data$senescence_label)
    cat(sprintf("  SnC: %d (%.1f%%)\n", orig_counts["SnC"], orig_counts["SnC"] / sum(orig_counts) * 100))
}

cat("\n--- Output Files ---\n")
cat("  Data:\n")
cat("    •", save_file, "\n")

cat("\n  Figures:\n")
fig_files <- list.files(FIGURES_DIR, pattern = paste0("^", DATASET, "_", cell_type_label), full.names = FALSE)
for (f in fig_files) cat("    •", f, "\n")

cat("\n  Results:\n")
res_files <- list.files(RESULTS_DIR, pattern = paste0("^", DATASET, "_", cell_type_label), full.names = FALSE)
for (f in res_files) cat("    •", f, "\n")

cat("\n", paste(rep("=", 80), collapse = ""), "\n")
cat(sprintf("%s COMPLETE\n", toupper(cell_type_label)))
cat(paste(rep("=", 80), collapse = ""), "\n")

---

## Post-Annotation Validation: Aberrant Cluster QC

**Purpose:** Systematic validation of annotated subclusters/states before downstream analysis (DEG, GSEA). Confirms that assigned states are biologically real and not driven by technical artifacts.

**5-Step Validation:**
1. **QC Metrics** — UMI, genes, %MT, %ribo per state vs global. Flags low-quality or doublet-like states.
2. **Canonical Marker Expression** — Are identity markers expressed in all states? Any state lacking lineage markers?
3. **Sample Composition** — Is any state dominated by 1–2 donors? (batch artifact)
4. **Cell Cycle** — Is any state enriched for S/G2M phase? (cycling subpopulation)
5. **Cluster Stability** — Are states stable across clustering resolutions? (clustree)

**Input:** Annotated Seurat object (`seurat_obj`) from save step above.

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# POST-ANNOTATION VALIDATION: SETUP & INSPECT
# ════════════════════════════════════════════════════════════════════════════════

cat("\n", paste(rep("=", 80), collapse = ""), "\n")
cat("POST-ANNOTATION VALIDATION: SETUP\n")
cat(paste(rep("=", 80), collapse = ""), "\n")

suppressPackageStartupMessages({
    library(Seurat)
    library(Matrix)
    library(dplyr)
    library(tidyr)
    library(ggplot2)
    library(qs)
    library(patchwork)
    library(RColorBrewer)
    library(scales)
})

cat("\n✓ Libraries loaded\n")

# ════════════════════════════════════════════════════════════════════════════════
# DATASET SELECTION
# ════════════════════════════════════════════════════════════════════════════════

DATASET <- "psychad_aging"

# ════════════════════════════════════════════════════════════════════════════════
# DATASET-SPECIFIC CONFIGURATION
# ════════════════════════════════════════════════════════════════════════════════

DATASET_CONFIG <- list(
    'psychad_aging' = list(
        cell_type_col = 'subclass',
        donor_col = 'Sample',
        study_group_col = 'Study_Group',
        sex_col = 'Sex',
        cohort_col = 'Cohort',
        study_type = 'aging',
        primary_var = 'Age',
        primary_var_type = 'continuous',
        covariates = c('Sex', 'Cohort'),
        group_order = c('Age_20_29', 'Age_30_39', 'Age_40_49', 'Age_50_59',
                        'Age_60_69', 'Age_70_79', 'Age_80_100')
    ),
    'psychad_ad' = list(
        cell_type_col = 'subclass',
        donor_col = 'Sample',
        study_group_col = 'Study_Group',
        sex_col = 'Sex',
        cohort_col = 'Cohort',
        study_type = 'disease',
        primary_var = 'Study_Group',
        primary_var_type = 'categorical',
        reference_group = 'Control',
        covariates = c('Age', 'Sex', 'Cohort'),
        group_order = c('Control', 'MCI', 'AD')
    ),
    'psychencode' = list(
        cell_type_col = 'cell_type',
        donor_col = 'sample_id',
        study_group_col = 'Study_Group',
        sex_col = 'Biological_Sex',
        cohort_col = 'Batch',
        study_type = 'aging',
        primary_var = 'Age_death',
        primary_var_type = 'continuous',
        covariates = c('Sex', 'Batch'),
        group_order = c('Age_20_29', 'Age_30_39', 'Age_40_49', 'Age_50_59',
                        'Age_60_69', 'Age_70_79', 'Age_80_100')
    ),
    'mathys' = list(
        cell_type_col = 'broad.cell.type',
        donor_col = 'Subject',
        study_group_col = 'Study_Group',
        sex_col = 'sex',
        cohort_col = 'batch',
        study_type = 'disease',
        primary_var = 'Study_Group',
        primary_var_type = 'categorical',
        reference_group = 'Control',
        covariates = c('Age', 'sex', 'batch'),
        group_order = c('Control', 'AD')
    )
)

config <- DATASET_CONFIG[[DATASET]]

# ─────────────────────────────────────────────────────────────────────────────
# COLUMN NAMES
# ─────────────────────────────────────────────────────────────────────────────

CELL_TYPE_COL     <- config$cell_type_col
DONOR_COL         <- config$donor_col
STUDY_GROUP_COL   <- config$study_group_col
SEX_COL           <- config$sex_col
COHORT_COL        <- config$cohort_col
SENESCENCE_LABEL_COL <- "senescence_label"

# ─────────────────────────────────────────────────────────────────────────────
# STUDY DESIGN PARAMETERS
# ─────────────────────────────────────────────────────────────────────────────

STUDY_TYPE       <- config$study_type
PRIMARY_VAR      <- config$primary_var
PRIMARY_VAR_TYPE <- config$primary_var_type
COVARIATES       <- config$covariates
GROUP_ORDER      <- config$group_order

# ─────────────────────────────────────────────────────────────────────────────
# CELL TYPE-SPECIFIC CONFIG
# ─────────────────────────────────────────────────────────────────────────────

cell_type_label    <- "opc"          # astrocyte, microglia, opc, ol
SUBCLUSTER_COL     <- "opc_state"    # microglia_state, opc_state, ol_state
CLUSTERING_RESOLUTION <- 0.6

# ════════════════════════════════════════════════════════════════════════════════
# COLOR PALETTES
# ════════════════════════════════════════════════════════════════════════════════

STUDY_GROUP_COLORS <- c(
    'Age_20_29' = '#2E86AB', 'Age_30_39' = '#4A90E2', 'Age_40_49' = '#50C878',
    'Age_50_59' = '#FFB347', 'Age_60_69' = '#FF8C00', 'Age_70_79' = '#E24A4A',
    'Age_80_100' = '#8B0000',
    'Control' = '#4E79A7', 'MCI' = '#F28E2B', 'AD' = '#E15759', 'NCI' = '#4E79A7'
)

SENESCENCE_COLORS <- c('Non-SnC' = '#D3D3D3', 'SnC' = '#C44E52')
SEX_COLORS <- c('Male' = '#4878CF', 'Female' = '#E97B8A')

BASE_STATE_PALETTE <- c("#4E79A7", "#59A14F", "#E15759", "#F28E2B", "#EDC948",
                        "#76B7B2", "#FFBE7D", "#BAB0AC", "#B07AA1", "#FF9DA7",
                        "#A0CBE8", "#D37295", "#9C755F", "#8B0000")

get_state_colors <- function(states) {
    setNames(BASE_STATE_PALETTE[1:length(states)], states)
}

# ════════════════════════════════════════════════════════════════════════════════
# PATHS
# ════════════════════════════════════════════════════════════════════════════════

BASE_DIR    <- "/fs/scratch/PAS2598/senescence_analysis"
DATA_DIR    <- file.path(BASE_DIR, "data", paste0("04_", cell_type_label), DATASET)
OUTPUT_DIR  <- DATA_DIR
FIGURES_DIR <- file.path(BASE_DIR, "figures", paste0("04_", cell_type_label), DATASET)
RESULTS_DIR <- file.path(BASE_DIR, "results", paste0("04_", cell_type_label), DATASET)

# Validation subdirectories
VAL_FIGURES_DIR <- file.path(FIGURES_DIR, "validation")
VAL_RESULTS_DIR <- file.path(RESULTS_DIR, "validation")

dir.create(VAL_FIGURES_DIR, recursive = TRUE, showWarnings = FALSE)
dir.create(VAL_RESULTS_DIR, recursive = TRUE, showWarnings = FALSE)

FILE_PREFIX <- paste0(DATASET, "_", cell_type_label, "_validation")

# ════════════════════════════════════════════════════════════════════════════════
# LOAD ANNOTATED OBJECT
# ════════════════════════════════════════════════════════════════════════════════

INPUT_FILE <- file.path(DATA_DIR, paste0(DATASET, "_", cell_type_label, "_annotated.qs"))

cat("\n▸ Loading:", INPUT_FILE, "\n")

if (!file.exists(INPUT_FILE)) stop(paste("File not found:", INPUT_FILE))

seurat_obj <- qread(INPUT_FILE)
cat("  ✓ Loaded:", ncol(seurat_obj), "cells,", nrow(seurat_obj), "genes\n")

# ════════════════════════════════════════════════════════════════════════════════
# INSPECT OBJECT
# ════════════════════════════════════════════════════════════════════════════════

cat("\n▸ Object summary:\n")
cat("  Cells:", ncol(seurat_obj), "\n")
cat("  Genes:", nrow(seurat_obj), "\n")
cat("  Dataset:", DATASET, "\n")
cat("  Cell type:", toupper(cell_type_label), "\n")

# States
cat("\n▸ Annotated states (", SUBCLUSTER_COL, "):\n", sep = "")
states <- sort(unique(seurat_obj@meta.data[[SUBCLUSTER_COL]]))
n_states <- length(states)
for (s in states) {
    n <- sum(seurat_obj@meta.data[[SUBCLUSTER_COL]] == s)
    cat(sprintf("    %s: %d (%.1f%%)\n", s, n, n / ncol(seurat_obj) * 100))
}

# Clusters
cat("\n▸ Seurat clusters:\n")
n_clusters <- length(unique(seurat_obj$seurat_clusters))
cat("  N clusters:", n_clusters, "\n")

# State ↔ Cluster mapping
cat("\n▸ State ↔ Cluster mapping:\n")
mapping <- table(seurat_obj@meta.data[[SUBCLUSTER_COL]], seurat_obj$seurat_clusters)
for (s in rownames(mapping)) {
    counts <- mapping[s, mapping[s, ] > 0]
    detail <- paste(sprintf("%s(n=%d)", names(counts), counts), collapse = ", ")
    cat(sprintf("    %s → %s\n", s, detail))
}

# ════════════════════════════════════════════════════════════════════════════════
# DETECT QC COLUMNS
# ════════════════════════════════════════════════════════════════════════════════

cat("\n▸ Detecting QC columns...\n")
md_cols <- colnames(seurat_obj@meta.data)

MT_COL <- NA
for (mc in c("percent.mt", "percent_mt", "pct_counts_mt", "mitoRatio")) {
    if (mc %in% md_cols) { MT_COL <- mc; break }
}
RIBO_COL <- NA
for (rc in c("percent.ribo", "percent_ribo", "pct_counts_ribo")) {
    if (rc %in% md_cols) { RIBO_COL <- rc; break }
}

QC_METRICS <- c("nCount_RNA", "nFeature_RNA")
QC_LABELS <- c("UMI Counts", "Gene Counts")
if (!is.na(MT_COL)) { QC_METRICS <- c(QC_METRICS, MT_COL); QC_LABELS <- c(QC_LABELS, "% MT") }
if (!is.na(RIBO_COL)) { QC_METRICS <- c(QC_METRICS, RIBO_COL); QC_LABELS <- c(QC_LABELS, "% Ribo") }
names(QC_LABELS) <- QC_METRICS

cat("  MT:", ifelse(is.na(MT_COL), "NONE", MT_COL), "\n")
cat("  Ribo:", ifelse(is.na(RIBO_COL), "NONE", RIBO_COL), "\n")
cat("  Active metrics:", paste(QC_METRICS, collapse = ", "), "\n")

# Donors
n_donors <- length(unique(seurat_obj@meta.data[[DONOR_COL]]))
cat("\n▸ Donors:", n_donors, "\n")

# Reductions
UMAP_RED <- ifelse("umap" %in% Reductions(seurat_obj), "umap",
            ifelse("tsne" %in% Reductions(seurat_obj), "tsne", Reductions(seurat_obj)[1]))
cat("▸ Reductions:", paste(Reductions(seurat_obj), collapse = ", "), "\n")
cat("  Plot reduction:", UMAP_RED, "\n")

# Global medians
GLOBAL_MEDIANS <- sapply(QC_METRICS, function(m) median(seurat_obj@meta.data[[m]], na.rm = TRUE))

cat("\n▸ Global QC medians:\n")
for (i in seq_along(QC_METRICS)) {
    cat(sprintf("    %s: %.1f\n", QC_LABELS[i], GLOBAL_MEDIANS[i]))
}

# Derived orders
is_aging <- STUDY_TYPE == "aging"
sg_detected <- unique(seurat_obj@meta.data[[STUDY_GROUP_COL]])
sg_order <- names(STUDY_GROUP_COLORS)[names(STUDY_GROUP_COLORS) %in% sg_detected]

state_order <- seurat_obj@meta.data %>%
    count(.data[[SUBCLUSTER_COL]]) %>%
    arrange(desc(n)) %>%
    pull(.data[[SUBCLUSTER_COL]])

state_colors <- get_state_colors(state_order)

cat("\n▸ Study groups:", paste(sg_order, collapse = ", "), "\n")
cat("▸ States (by freq):", paste(state_order, collapse = ", "), "\n")
cat("▸ State colors:", paste(sprintf("%s=%s", names(state_colors), state_colors), collapse = ", "), "\n")

cat("\n", paste(rep("=", 80), collapse = ""), "\n")
cat("✓ Setup complete — ready for 5-step validation\n")
cat(paste(rep("=", 80), collapse = ""), "\n")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# STEP 1: QC METRICS PER STATE & CLUSTER
# ════════════════════════════════════════════════════════════════════════════════

cat("\n", paste(rep("=", 80), collapse = ""), "\n")
cat("STEP 1: QC METRICS\n")
cat(paste(rep("=", 80), collapse = ""), "\n")

# ─────────────────────────────────────────────────────────────────────────────────
# 1A: QC summary table — per cluster
# ─────────────────────────────────────────────────────────────────────────────────

cat("\n▸ Per-cluster QC summary:\n")

qc_by_cluster <- do.call(rbind, lapply(sort(unique(seurat_obj$seurat_clusters)), function(cl) {
    cells <- WhichCells(seurat_obj, idents = cl)
    md <- seurat_obj@meta.data[cells, , drop = FALSE]
    
    row <- data.frame(
        Cluster = cl,
        N = length(cells),
        Pct = round(length(cells) / ncol(seurat_obj) * 100, 1),
        State = names(which.max(table(md[[SUBCLUSTER_COL]]))),
        median_UMI = median(md$nCount_RNA),
        median_genes = median(md$nFeature_RNA),
        stringsAsFactors = FALSE
    )
    
    if (!is.na(MT_COL)) row$median_mt <- round(median(md[[MT_COL]], na.rm = TRUE), 3)
    if (!is.na(RIBO_COL)) row$median_ribo <- round(median(md[[RIBO_COL]], na.rm = TRUE), 3)
    
    row
}))

print(qc_by_cluster, row.names = FALSE)

# Save
write.csv(qc_by_cluster, file.path(VAL_RESULTS_DIR, paste0(FILE_PREFIX, "_qc_by_cluster.csv")),
          row.names = FALSE)
cat("\n  ✓ Saved: ", FILE_PREFIX, "_qc_by_cluster.csv\n", sep = "")

# ─────────────────────────────────────────────────────────────────────────────────
# 1B: QC summary table — per state
# ─────────────────────────────────────────────────────────────────────────────────

cat("\n▸ Per-state QC summary:\n")

qc_by_state <- do.call(rbind, lapply(states, function(s) {
    cells <- colnames(seurat_obj)[seurat_obj@meta.data[[SUBCLUSTER_COL]] == s]
    md <- seurat_obj@meta.data[cells, , drop = FALSE]
    
    row <- data.frame(
        State = s,
        N = length(cells),
        Pct = round(length(cells) / ncol(seurat_obj) * 100, 1),
        median_UMI = median(md$nCount_RNA),
        median_genes = median(md$nFeature_RNA),
        stringsAsFactors = FALSE
    )
    
    if (!is.na(MT_COL)) row$median_mt <- round(median(md[[MT_COL]], na.rm = TRUE), 3)
    if (!is.na(RIBO_COL)) row$median_ribo <- round(median(md[[RIBO_COL]], na.rm = TRUE), 3)
    
    row
}))

print(qc_by_state, row.names = FALSE)

write.csv(qc_by_state, file.path(VAL_RESULTS_DIR, paste0(FILE_PREFIX, "_qc_by_state.csv")),
          row.names = FALSE)
cat("\n  ✓ Saved: ", FILE_PREFIX, "_qc_by_state.csv\n", sep = "")

# ─────────────────────────────────────────────────────────────────────────────────
# 1C: Flag outlier clusters
# ─────────────────────────────────────────────────────────────────────────────────

cat("\n▸ Flagging outlier clusters:\n")

flags_list <- list()

for (i in 1:nrow(qc_by_cluster)) {
    cl <- qc_by_cluster$Cluster[i]
    flags <- c()
    
    # Low UMI
    if (qc_by_cluster$median_UMI[i] < GLOBAL_MEDIANS["nCount_RNA"] * 0.5)
        flags <- c(flags, "LOW UMI")
    
    # Low genes
    if (qc_by_cluster$median_genes[i] < GLOBAL_MEDIANS["nFeature_RNA"] * 0.5)
        flags <- c(flags, "LOW GENES")
    
    # High MT
    if (!is.na(MT_COL) && "median_mt" %in% colnames(qc_by_cluster)) {
        if (qc_by_cluster$median_mt[i] > GLOBAL_MEDIANS[MT_COL] * 2)
            flags <- c(flags, "HIGH MT")
    }
    
    # Possible doublet (high UMI + high genes)
    if (qc_by_cluster$median_UMI[i] > GLOBAL_MEDIANS["nCount_RNA"] * 2 &&
        qc_by_cluster$median_genes[i] > GLOBAL_MEDIANS["nFeature_RNA"] * 1.5)
        flags <- c(flags, "POSSIBLE DOUBLET")
    
    # Very small cluster
    if (qc_by_cluster$N[i] < ncol(seurat_obj) * 0.005)
        flags <- c(flags, sprintf("TINY (%.1f%%)", qc_by_cluster$Pct[i]))
    
    if (length(flags) > 0) {
        flags_list[[cl]] <- flags
        cat(sprintf("  ⚠ Cluster %s (%s): %s\n", cl, qc_by_cluster$State[i],
                    paste(flags, collapse = ", ")))
    }
}

if (length(flags_list) == 0) cat("  ✓ No clusters flagged\n")

# ─────────────────────────────────────────────────────────────────────────────────
# 1D: Violin plots — QC per cluster
# ─────────────────────────────────────────────────────────────────────────────────

cat("\n▸ Generating QC violin plots...\n")

options(repr.plot.width = 14, repr.plot.height = 3 * length(QC_METRICS))

plots_qc_cluster <- lapply(seq_along(QC_METRICS), function(i) {
    m <- QC_METRICS[i]
    VlnPlot(seurat_obj, features = m, group.by = "seurat_clusters", pt.size = 0) +
        geom_hline(yintercept = GLOBAL_MEDIANS[m], linetype = "dashed",
                   color = "red", linewidth = 0.5) +
        theme(legend.position = "none", axis.title.x = element_blank(),
              plot.title = element_text(size = 10, face = "bold")) +
        ggtitle(paste0(QC_LABELS[i], " (by cluster) — red = global median"))
})

p_qc_cluster <- wrap_plots(plots_qc_cluster, ncol = 1)
print(p_qc_cluster)

ggsave(file.path(VAL_FIGURES_DIR, paste0(FILE_PREFIX, "_qc_violin_cluster.pdf")),
       p_qc_cluster, width = 14, height = 3 * length(QC_METRICS))
ggsave(file.path(VAL_FIGURES_DIR, paste0(FILE_PREFIX, "_qc_violin_cluster.svg")),
       p_qc_cluster, width = 14, height = 3 * length(QC_METRICS))
cat("  ✓ Saved violin plots (by cluster)\n")

# ─────────────────────────────────────────────────────────────────────────────────
# 1E: Violin plots — QC per state
# ─────────────────────────────────────────────────────────────────────────────────

plots_qc_state <- lapply(seq_along(QC_METRICS), function(i) {
    m <- QC_METRICS[i]
    VlnPlot(seurat_obj, features = m, group.by = SUBCLUSTER_COL, pt.size = 0) +
        geom_hline(yintercept = GLOBAL_MEDIANS[m], linetype = "dashed",
                   color = "red", linewidth = 0.5) +
        theme(legend.position = "none", axis.title.x = element_blank(),
              plot.title = element_text(size = 10, face = "bold")) +
        ggtitle(paste0(QC_LABELS[i], " (by state) — red = global median"))
})

p_qc_state <- wrap_plots(plots_qc_state, ncol = 1)
print(p_qc_state)

ggsave(file.path(VAL_FIGURES_DIR, paste0(FILE_PREFIX, "_qc_violin_state.pdf")),
       p_qc_state, width = 10, height = 3 * length(QC_METRICS))
ggsave(file.path(VAL_FIGURES_DIR, paste0(FILE_PREFIX, "_qc_violin_state.svg")),
       p_qc_state, width = 10, height = 3 * length(QC_METRICS))
cat("  ✓ Saved violin plots (by state)\n")

# ─────────────────────────────────────────────────────────────────────────────────
# 1F: UMAP colored by QC metrics
# ─────────────────────────────────────────────────────────────────────────────────

cat("\n▸ Generating QC feature UMAPs...\n")

options(repr.plot.width = 12, repr.plot.height = 3.5)

p_qc_umap <- FeaturePlot(seurat_obj, features = QC_METRICS, reduction = UMAP_RED,
                          ncol = length(QC_METRICS), pt.size = 0.1, order = TRUE) &
    scale_color_viridis_c() &
    theme(plot.title = element_text(size = 9, face = "bold"),
          axis.text = element_blank(), axis.ticks = element_blank())

print(p_qc_umap)

ggsave(file.path(VAL_FIGURES_DIR, paste0(FILE_PREFIX, "_qc_umap.pdf")),
       p_qc_umap, width = 3.5 * length(QC_METRICS), height = 3.5)
ggsave(file.path(VAL_FIGURES_DIR, paste0(FILE_PREFIX, "_qc_umap.svg")),
       p_qc_umap, width = 3.5 * length(QC_METRICS), height = 3.5)
cat("  ✓ Saved QC UMAPs\n")

# ─────────────────────────────────────────────────────────────────────────────────
# Step 1 Summary
# ─────────────────────────────────────────────────────────────────────────────────

cat("\n", paste(rep("-", 80), collapse = ""), "\n")
cat("STEP 1 SUMMARY\n")
cat(paste(rep("-", 80), collapse = ""), "\n")
cat("  Clusters:", n_clusters, "\n")
cat("  States:", n_states, "\n")
cat("  Flagged clusters:", length(flags_list), "\n")
if (length(flags_list) > 0) {
    for (cl in names(flags_list)) {
        cat(sprintf("    Cluster %s: %s\n", cl, paste(flags_list[[cl]], collapse = ", ")))
    }
}
cat("\n  Files saved:\n")
cat("    •", paste0(FILE_PREFIX, "_qc_by_cluster.csv"), "\n")
cat("    •", paste0(FILE_PREFIX, "_qc_by_state.csv"), "\n")
cat("    •", paste0(FILE_PREFIX, "_qc_violin_cluster.{pdf,svg}"), "\n")
cat("    •", paste0(FILE_PREFIX, "_qc_violin_state.{pdf,svg}"), "\n")
cat("    •", paste0(FILE_PREFIX, "_qc_umap.{pdf,svg}"), "\n")
cat(paste(rep("=", 80), collapse = ""), "\n")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# STEP 2: CANONICAL MARKER EXPRESSION
# ════════════════════════════════════════════════════════════════════════════════

cat("\n", paste(rep("=", 80), collapse = ""), "\n")
cat("STEP 2: CANONICAL MARKER EXPRESSION\n")
cat(paste(rep("=", 80), collapse = ""), "\n")

library(dplyr)

# ─────────────────────────────────────────────────────────────────────────────────
# Define markers: positive (identity) and negative (contamination)
# ─────────────────────────────────────────────────────────────────────────────────

POSITIVE_MARKERS <- list(
    "astrocyte" = list(
        'Identity'      = c('GFAP', 'AQP4', 'SLC1A2', 'GLUL', 'ALDH1L1'),
        'Homeostatic'   = c('GLUL', 'SLC1A2', 'GRM3', 'GJA1', 'KCNJ10'),
        'Intermediate'  = c('ETNPPL', 'GLUD1', 'NTRK2', 'GABRB1'),
        'Metabolic'     = c('ENO1', 'LDHA', 'MT1X', 'HSP90AA1'),
        'Trophic'       = c('FGF1', 'FGF2', 'CHI3L1', 'SOCS3'),
        'Reactive'      = c('GFAP', 'C3', 'CD44', 'COL21A1', 'SPARC')
    ),
    "microglia" = list(
        'Identity'              = c('CSF1R', 'P2RY12', 'CX3CR1', 'TMEM119', 'AIF1'),
        'Homeostatic'           = c('TMEM119', 'P2RY12', 'CX3CR1'),
        'Activated'             = c('CD68', 'APOE', 'TREM2'),
        'IFN-I'                 = c('IFITM3', 'IFIT3', 'ISG15'),
        'IFN-II'                = c('GBP2', 'STAT1'),
        'IFN-III'               = c('IFI44L', 'IRF7'),
        'MCHII'                 = c('HLA-DRA', 'HLA-DRB1', 'CD74'),
        'Stress'                = c('HSP90AA1', 'HSPA1A', 'FOS'),
        'Neuronal_Surveillance' = c('MT-ND3', 'MT-CYB', 'MT-CO2'),
        'Lipid_Processing'      = c('APOC1', 'LPL', 'ABCA1'),
        'Cycling'               = c('MKI67', 'TOP2A', 'CDK1'),
        'Mic-reduced'           = c('LINGO1', 'GRID2', 'ELAVL2')
    ),
    "opc" = list(
        'Identity'    = c('PDGFRA', 'CSPG4', 'OLIG1', 'OLIG2', 'SOX10'),
        'Homeostatic' = c('PDGFRA', 'CSPG4', 'PTPRZ1', 'BCAN', 'PCDH15'),
        'COP'         = c('GPR17', 'BCAS1', 'NEU4', 'TNS3', 'FYN'),
        'Reactive'    = c('HLA-A', 'HLA-B', 'HLA-C', 'B2M', 'CD74', 'SERPINA3')
    ),
    "ol" = list(
        'Identity'    = c('MBP', 'MOG', 'PLP1', 'MAG', 'MOBP'),
        'hOligo1'     = c('RASGRF1', 'RASGRF2', 'KLK6'),
        'hOligo2'     = c('PLXDC2', 'PALM2', 'OPALIN', 'QDPR'),
        'DA1_Immune'  = c('SERPINA3', 'C4B', 'B2M', 'HLA-A', 'HLA-B', 'HLA-C', 'CD74'),
        'IFN'         = c('IFIT1', 'IFIT3', 'OAS1', 'OAS2')
    )
)

NEGATIVE_MARKERS_ALL <- list(
    'Neuron'          = c('RBFOX3', 'SYT1', 'SNAP25'),
    'Endothelial'     = c('CLDN5', 'FLT1', 'PECAM1'),
    'Astrocyte'       = c('GFAP', 'AQP4', 'SLC1A2'),
    'Microglia'       = c('CSF1R', 'P2RY12', 'CX3CR1'),
    'OPC'             = c('PDGFRA', 'CSPG4', 'GPR17'),
    'Oligodendrocyte' = c('MBP', 'MOG', 'PLP1')
)

# ── Remove self from negative markers ──
SELF_LABEL_MAP <- c(
    "astrocyte" = "Astrocyte",
    "microglia" = "Microglia",
    "opc"       = "OPC",
    "ol"        = "Oligodendrocyte"
)
self_label <- SELF_LABEL_MAP[[cell_type_label]]
neg_markers_list <- NEGATIVE_MARKERS_ALL[names(NEGATIVE_MARKERS_ALL) != self_label]

# Get positive markers for this cell type
pos_markers_list <- POSITIVE_MARKERS[[cell_type_label]]

# Filter to genes present in object
pos_markers_list <- lapply(pos_markers_list, function(genes) {
    genes[genes %in% rownames(seurat_obj)]
})
pos_markers_list <- pos_markers_list[sapply(pos_markers_list, length) > 0]

neg_markers_list <- lapply(neg_markers_list, function(genes) {
    genes[genes %in% rownames(seurat_obj)]
})
neg_markers_list <- neg_markers_list[sapply(neg_markers_list, length) > 0]

# Remove any genes that overlap between positive and negative
pos_genes_all <- unlist(pos_markers_list)
overlap_genes <- c()
neg_markers_list <- lapply(neg_markers_list, function(genes) {
    ov <- genes[genes %in% pos_genes_all]
    if (length(ov) > 0) overlap_genes <<- c(overlap_genes, ov)
    genes[!genes %in% pos_genes_all]
})
neg_markers_list <- neg_markers_list[sapply(neg_markers_list, length) > 0]

if (length(overlap_genes) > 0) {
    cat("\n  Note: Removed", length(unique(overlap_genes)), "overlapping gene(s) from negatives:",
        paste(unique(overlap_genes), collapse = ", "), "\n")
}

cat("\n▸ Positive markers:\n")
for (cat_name in names(pos_markers_list)) {
    cat(sprintf("  %s: %s\n", cat_name, paste(pos_markers_list[[cat_name]], collapse = ", ")))
}

cat("\n▸ Negative markers:\n")
for (cat_name in names(neg_markers_list)) {
    cat(sprintf("  %s: %s\n", cat_name, paste(neg_markers_list[[cat_name]], collapse = ", ")))
}

# ─────────────────────────────────────────────────────────────────────────────────
# 2A: AddModuleScore — identity + per-lineage contamination
# ─────────────────────────────────────────────────────────────────────────────────

cat("\n▸ Computing module scores (AddModuleScore)...\n")

# Identity: all positive markers as one module
all_pos_genes <- unique(unlist(pos_markers_list))
seurat_obj <- AddModuleScore(seurat_obj,
                              features = list(all_pos_genes),
                              name = "identity_score",
                              ctrl = 50, seed = 42)

cat("  ✓ Identity score: ", length(all_pos_genes), " genes\n", sep = "")

# Per-lineage contamination scores
contam_score_cols <- c()
for (lineage in names(neg_markers_list)) {
    genes <- neg_markers_list[[lineage]]
    col_name <- paste0("contam_", gsub(" ", "_", lineage))
    
    seurat_obj <- AddModuleScore(seurat_obj,
                                  features = list(genes),
                                  name = col_name,
                                  ctrl = 50, seed = 42)
    actual_col <- paste0(col_name, "1")
    contam_score_cols <- c(contam_score_cols, setNames(actual_col, lineage))
    cat("  ✓ ", lineage, " contamination: ", length(genes), " genes -> ", actual_col, "\n", sep = "")
}

# ─────────────────────────────────────────────────────────────────────────────────
# 2B: Compute dataset-wide baselines per lineage
# ─────────────────────────────────────────────────────────────────────────────────

cat("\n▸ Computing dataset-wide baselines per lineage...\n")

lineage_baselines <- data.frame(
    Lineage = names(contam_score_cols),
    Median = NA_real_,
    MAD = NA_real_,
    Threshold = NA_real_,
    stringsAsFactors = FALSE
)

for (i in seq_along(contam_score_cols)) {
    vals <- seurat_obj@meta.data[[contam_score_cols[i]]]
    med <- median(vals, na.rm = TRUE)
    mad_val <- mad(vals, na.rm = TRUE)
    lineage_baselines$Median[i] <- round(med, 4)
    lineage_baselines$MAD[i] <- round(mad_val, 4)
    lineage_baselines$Threshold[i] <- round(med + 3 * mad_val, 4)
}

cat("  Dataset-wide contamination baselines (outlier = median + 3×MAD):\n")
print(lineage_baselines, row.names = FALSE)

# ─────────────────────────────────────────────────────────────────────────────────
# 2C: Aggregate scores per cluster and per state
# ─────────────────────────────────────────────────────────────────────────────────

cat("\n▸ Aggregating scores per cluster...\n")

Idents(seurat_obj) <- "seurat_clusters"
cluster_order <- as.character(sort(as.numeric(levels(Idents(seurat_obj)))))

score_by_cluster <- do.call(rbind, lapply(cluster_order, function(cl) {
    cells <- which(as.character(seurat_obj$seurat_clusters) == cl)
    
    identity_med <- median(seurat_obj$identity_score1[cells])
    
    contam_meds <- sapply(contam_score_cols, function(col) {
        median(seurat_obj@meta.data[[col]][cells])
    })
    
    max_contam_lineage <- names(which.max(contam_meds))
    max_contam_val <- max(contam_meds)
    mean_contam <- mean(contam_meds)
    
    row <- data.frame(
        Cluster = cl,
        State = qc_by_cluster$State[qc_by_cluster$Cluster == cl],
        N_cells = length(cells),
        Identity = round(identity_med, 4),
        stringsAsFactors = FALSE
    )
    
    for (j in seq_along(contam_meds)) {
        row[[names(contam_score_cols)[j]]] <- round(contam_meds[j], 4)
    }
    
    row$Max_contam_lineage <- max_contam_lineage
    row$Max_contam_score <- round(max_contam_val, 4)
    row$Mean_contam <- round(mean_contam, 4)
    row
}))

cat("\n  Module scores per cluster:\n")
print_cols <- c("Cluster", "State", "N_cells", "Identity",
                names(neg_markers_list), "Max_contam_lineage", "Max_contam_score")
print_cols <- print_cols[print_cols %in% colnames(score_by_cluster)]
print(score_by_cluster[, print_cols], row.names = FALSE)

# Per-state
cat("\n▸ Aggregating scores per state...\n")

score_by_state <- do.call(rbind, lapply(state_order, function(st) {
    cells <- which(seurat_obj@meta.data[[SUBCLUSTER_COL]] == st)
    
    identity_med <- median(seurat_obj$identity_score1[cells])
    
    contam_meds <- sapply(contam_score_cols, function(col) {
        median(seurat_obj@meta.data[[col]][cells])
    })
    
    row <- data.frame(
        State = st,
        N_cells = length(cells),
        Identity = round(identity_med, 4),
        stringsAsFactors = FALSE
    )
    
    for (j in seq_along(contam_meds)) {
        row[[names(contam_score_cols)[j]]] <- round(contam_meds[j], 4)
    }
    
    row$Max_contam_lineage <- names(which.max(contam_meds))
    row$Max_contam_score <- round(max(contam_meds), 4)
    row
}))

cat("\n  Module scores per state:\n")
print_cols_state <- c("State", "N_cells", "Identity",
                       names(neg_markers_list), "Max_contam_lineage", "Max_contam_score")
print_cols_state <- print_cols_state[print_cols_state %in% colnames(score_by_state)]
print(score_by_state[, print_cols_state], row.names = FALSE)

# ─────────────────────────────────────────────────────────────────────────────────
# 2D: Flag clusters — absolute + outlier thresholds
# ─────────────────────────────────────────────────────────────────────────────────

cat("\n▸ Flagging clusters:\n")
cat("  Criteria:\n")
cat("    WEAK IDENTITY:  median identity score < 0 (below background)\n")
cat("    CONTAMINATION:  lineage score > dataset baseline + 3×MAD AND > identity × 0.5\n")
cat("    DOUBLET-LIKE:   lineage score > dataset baseline + 3×MAD AND > identity\n")

flagged_id <- FALSE
flagged_contam <- FALSE
any_flagged <- FALSE

for (i in 1:nrow(score_by_cluster)) {
    flags <- c()
    cl <- score_by_cluster$Cluster[i]
    id_score <- score_by_cluster$Identity[i]
    
    # Weak identity: below background
    if (id_score < 0) {
        flags <- c(flags, sprintf("WEAK IDENTITY (%.4f < 0)", id_score))
        flagged_id <- TRUE
    }
    
    # Per-lineage contamination: must exceed BOTH baseline threshold AND relative threshold
    for (lineage in names(neg_markers_list)) {
        if (!lineage %in% colnames(score_by_cluster)) next
        contam_val <- score_by_cluster[[lineage]][i]
        
        baseline_row <- which(lineage_baselines$Lineage == lineage)
        baseline_thresh <- lineage_baselines$Threshold[baseline_row]
        
        # Only flag if above dataset-wide outlier threshold
        if (contam_val > baseline_thresh) {
            if (id_score > 0 && contam_val > id_score) {
                flags <- c(flags, sprintf("DOUBLET-LIKE %s (%.3f > identity %.3f, baseline %.3f)",
                                           lineage, contam_val, id_score, baseline_thresh))
                flagged_contam <- TRUE
            } else if (id_score > 0 && contam_val > id_score * 0.5) {
                flags <- c(flags, sprintf("HIGH %s (%.3f > 50%% identity, baseline %.3f)",
                                           lineage, contam_val, baseline_thresh))
                flagged_contam <- TRUE
            }
        }
    }
    
    if (length(flags) > 0) {
        cat(sprintf("    ⚠ Cluster %s (%s): %s\n",
                    cl, score_by_cluster$State[i], paste(flags, collapse = "; ")))
        any_flagged <- TRUE
    }
}

if (!any_flagged) cat("    ✓ All clusters pass identity and contamination checks\n")

# Build flag_df for downstream compatibility (Summary cell)
flag_df <- data.frame(
    Cluster = score_by_cluster$Cluster,
    State = score_by_cluster$State,
    mean_identity = score_by_cluster$Identity,
    mean_contam = score_by_cluster$Mean_contam,
    max_neg_gene = score_by_cluster$Max_contam_lineage,
    max_neg_val = score_by_cluster$Max_contam_score,
    stringsAsFactors = FALSE
)

write.csv(score_by_cluster, file.path(VAL_RESULTS_DIR, paste0(FILE_PREFIX, "_identity_scores.csv")),
          row.names = FALSE)
write.csv(score_by_state, file.path(VAL_RESULTS_DIR, paste0(FILE_PREFIX, "_identity_scores_state.csv")),
          row.names = FALSE)

# ─────────────────────────────────────────────────────────────────────────────────
# 2E: Compute raw expression for DotPlots
# ─────────────────────────────────────────────────────────────────────────────────

cat("\n▸ Computing expression for DotPlots...\n")

expr_data <- GetAssayData(seurat_obj, layer = "data")

all_marker_cats <- c(pos_markers_list, neg_markers_list)
gene_order <- unique(unlist(all_marker_cats))

# By cluster
dot_list <- list()
idx <- 1

for (cluster in cluster_order) {
    cells <- WhichCells(seurat_obj, idents = cluster)
    if (length(cells) == 0) next
    
    for (category in names(all_marker_cats)) {
        for (gene in all_marker_cats[[category]]) {
            expr <- expr_data[gene, cells]
            
            dot_list[[idx]] <- data.frame(
                Cluster = cluster,
                Gene = gene,
                Category = category,
                Pct_Exp = sum(expr > 0) / length(expr) * 100,
                Avg_Exp = mean(expr),
                stringsAsFactors = FALSE
            )
            idx <- idx + 1
        }
    }
}

dot_data_cluster <- do.call(rbind, dot_list)

dot_data_cluster <- dot_data_cluster %>%
    group_by(Gene) %>%
    mutate(Scaled_Exp = as.numeric(scale(Avg_Exp))) %>%
    ungroup()

dot_data_cluster$Scaled_Exp <- pmax(pmin(dot_data_cluster$Scaled_Exp, 2.5), -2.5)
dot_data_cluster$Cluster <- factor(dot_data_cluster$Cluster, levels = rev(cluster_order))
dot_data_cluster$Gene <- factor(dot_data_cluster$Gene, levels = gene_order)
dot_data_cluster$Category <- factor(dot_data_cluster$Category, levels = names(all_marker_cats))

pos_cats <- names(pos_markers_list)
dot_data_cluster$Marker_Type <- ifelse(dot_data_cluster$Category %in% pos_cats,
                                        "Positive", "Negative")

cat("  ✓ Cluster dotplot:", nrow(dot_data_cluster), "data points,",
    length(unique(dot_data_cluster$Gene)), "genes,",
    length(cluster_order), "clusters\n")

# By state
dot_list_state <- list()
idx <- 1

for (state in states) {
    cells <- colnames(seurat_obj)[seurat_obj@meta.data[[SUBCLUSTER_COL]] == state]
    if (length(cells) == 0) next
    
    for (category in names(all_marker_cats)) {
        for (gene in all_marker_cats[[category]]) {
            expr <- expr_data[gene, cells]
            
            dot_list_state[[idx]] <- data.frame(
                State = state,
                Gene = gene,
                Category = category,
                Pct_Exp = sum(expr > 0) / length(expr) * 100,
                Avg_Exp = mean(expr),
                stringsAsFactors = FALSE
            )
            idx <- idx + 1
        }
    }
}

dot_data_state <- do.call(rbind, dot_list_state)

dot_data_state <- dot_data_state %>%
    group_by(Gene) %>%
    mutate(Scaled_Exp = as.numeric(scale(Avg_Exp))) %>%
    ungroup()

dot_data_state$Scaled_Exp <- pmax(pmin(dot_data_state$Scaled_Exp, 2.5), -2.5)
dot_data_state$State <- factor(dot_data_state$State, levels = rev(states))
dot_data_state$Gene <- factor(dot_data_state$Gene, levels = gene_order)
dot_data_state$Category <- factor(dot_data_state$Category, levels = names(all_marker_cats))

cat("  ✓ State dotplot:", nrow(dot_data_state), "data points\n")

# ─────────────────────────────────────────────────────────────────────────────────
# 2F: DotPlot — by cluster
# ─────────────────────────────────────────────────────────────────────────────────

cat("\n▸ Generating DotPlots...\n")

options(repr.plot.width = 16, repr.plot.height = 10)

p_dot_cluster <- ggplot(dot_data_cluster, aes(x = Gene, y = Cluster)) +
    geom_point(aes(size = Pct_Exp, fill = Scaled_Exp), shape = 21, color = "black", stroke = 0.3) +
    scale_size_continuous(
        range = c(1, 6), limits = c(0, 100),
        breaks = c(0, 25, 50, 75, 100), name = "% Expr"
    ) +
    scale_fill_gradientn(
        colors = c("#313695", "#4575B4", "#74ADD1", "#FFFFBF", "#FDAE61", "#F46D43", "#A50026"),
        limits = c(-2.5, 2.5), name = "Scaled\nExpr"
    ) +
    facet_grid(cols = vars(Category), scales = "free_x", space = "free_x") +
    labs(x = NULL, y = "Cluster",
         title = paste0(toupper(cell_type_label), ": Identity & Contamination Markers (by cluster)")) +
    theme_bw(base_size = 10) +
    theme(
        axis.text.x = element_text(angle = 45, hjust = 1, size = 7, face = "italic"),
        axis.text.y = element_text(size = 9),
        axis.title.y = element_text(size = 10, face = "bold"),
        strip.text = element_text(size = 7, face = "bold"),
        strip.background = element_rect(fill = "gray70", color = "black", linewidth = 0.5),
        panel.grid = element_blank(),
        panel.spacing = unit(0.2, "lines"),
        panel.border = element_rect(color = "black", linewidth = 0.5),
        legend.position = "right",
        legend.title = element_text(size = 8),
        legend.text = element_text(size = 7),
        legend.key.size = unit(0.4, "cm"),
        plot.title = element_text(size = 11, face = "bold"),
        plot.margin = margin(10, 10, 10, 10)
    )

print(p_dot_cluster)

ggsave(file.path(VAL_FIGURES_DIR, paste0(FILE_PREFIX, "_dotplot_cluster.pdf")),
       p_dot_cluster, width = 16, height = 10)
ggsave(file.path(VAL_FIGURES_DIR, paste0(FILE_PREFIX, "_dotplot_cluster.svg")),
       p_dot_cluster, width = 16, height = 10)

# ─────────────────────────────────────────────────────────────────────────────────
# 2G: DotPlot — by state
# ─────────────────────────────────────────────────────────────────────────────────

options(repr.plot.width = 16, repr.plot.height = 5)

p_dot_state <- ggplot(dot_data_state, aes(x = Gene, y = State)) +
    geom_point(aes(size = Pct_Exp, fill = Scaled_Exp), shape = 21, color = "black", stroke = 0.3) +
    scale_size_continuous(
        range = c(1, 6), limits = c(0, 100),
        breaks = c(0, 25, 50, 75, 100), name = "% Expr"
    ) +
    scale_fill_gradientn(
        colors = c("#313695", "#4575B4", "#74ADD1", "#FFFFBF", "#FDAE61", "#F46D43", "#A50026"),
        limits = c(-2.5, 2.5), name = "Scaled\nExpr"
    ) +
    facet_grid(cols = vars(Category), scales = "free_x", space = "free_x") +
    labs(x = NULL, y = "State",
         title = paste0(toupper(cell_type_label), ": Identity & Contamination Markers (by state)")) +
    theme_bw(base_size = 10) +
    theme(
        axis.text.x = element_text(angle = 45, hjust = 1, size = 7, face = "italic"),
        axis.text.y = element_text(size = 9),
        axis.title.y = element_text(size = 10, face = "bold"),
        strip.text = element_text(size = 7, face = "bold"),
        strip.background = element_rect(fill = "gray70", color = "black", linewidth = 0.5),
        panel.grid = element_blank(),
        panel.spacing = unit(0.2, "lines"),
        panel.border = element_rect(color = "black", linewidth = 0.5),
        legend.position = "right",
        legend.title = element_text(size = 8),
        legend.text = element_text(size = 7),
        legend.key.size = unit(0.4, "cm"),
        plot.title = element_text(size = 11, face = "bold"),
        plot.margin = margin(10, 10, 10, 10)
    )

print(p_dot_state)

ggsave(file.path(VAL_FIGURES_DIR, paste0(FILE_PREFIX, "_dotplot_state.pdf")),
       p_dot_state, width = 16, height = 5)
ggsave(file.path(VAL_FIGURES_DIR, paste0(FILE_PREFIX, "_dotplot_state.svg")),
       p_dot_state, width = 16, height = 5)

cat("  ✓ Saved DotPlots\n")

# ─────────────────────────────────────────────────────────────────────────────────
# 2H: Module Score Heatmap — clusters × lineages
# ─────────────────────────────────────────────────────────────────────────────────

cat("\n▸ Generating module score heatmap...\n")

score_mat <- data.frame(
    Cluster = score_by_cluster$Cluster,
    Identity = score_by_cluster$Identity
)
for (lineage in names(neg_markers_list)) {
    if (lineage %in% colnames(score_by_cluster)) {
        score_mat[[lineage]] <- score_by_cluster[[lineage]]
    }
}

score_long <- tidyr::pivot_longer(score_mat, cols = -Cluster,
                                   names_to = "Module", values_to = "Score")
score_long$Cluster <- factor(score_long$Cluster, levels = rev(cluster_order))
module_order <- c("Identity", names(neg_markers_list))
module_order <- module_order[module_order %in% unique(score_long$Module)]
score_long$Module <- factor(score_long$Module, levels = module_order)

score_long$State <- sapply(as.character(score_long$Cluster), function(cl) {
    row <- which(qc_by_cluster$Cluster == cl)
    if (length(row) > 0) qc_by_cluster$State[row] else ""
})
score_long$Y_label <- paste0(score_long$Cluster, " (", score_long$State, ")")
y_label_order <- rev(paste0(cluster_order, " (",
    sapply(cluster_order, function(cl) {
        row <- which(qc_by_cluster$Cluster == cl)
        if (length(row) > 0) qc_by_cluster$State[row] else ""
    }), ")"))
score_long$Y_label <- factor(score_long$Y_label, levels = y_label_order)

options(repr.plot.width = 8, repr.plot.height = max(6, length(cluster_order) * 0.4))

p_heatmap <- ggplot(score_long, aes(x = Module, y = Y_label, fill = Score)) +
    geom_tile(color = "white", linewidth = 0.5) +
    geom_text(aes(label = sprintf("%.3f", Score)), size = 2.5, color = "black") +
    scale_fill_gradient2(low = "#313695", mid = "#FFFFBF", high = "#A50026",
                          midpoint = 0, name = "Module\nScore") +
    labs(x = NULL, y = NULL,
         title = paste0(toupper(cell_type_label),
                        ": Identity vs Contamination Module Scores"),
         subtitle = "AddModuleScore (median per cluster, 0 = background)") +
    theme_minimal(base_size = 10) +
    theme(
        axis.text.x = element_text(angle = 45, hjust = 1, size = 9, face = "bold"),
        axis.text.y = element_text(size = 8),
        plot.title = element_text(size = 11, face = "bold"),
        plot.subtitle = element_text(size = 9, color = "#666666"),
        panel.grid = element_blank(),
        legend.position = "right"
    )

print(p_heatmap)

ggsave(file.path(VAL_FIGURES_DIR, paste0(FILE_PREFIX, "_module_score_heatmap.pdf")),
       p_heatmap, width = 8, height = max(6, length(cluster_order) * 0.4))
ggsave(file.path(VAL_FIGURES_DIR, paste0(FILE_PREFIX, "_module_score_heatmap.svg")),
       p_heatmap, width = 8, height = max(6, length(cluster_order) * 0.4))

cat("  ✓ Saved module score heatmap\n")

# ─────────────────────────────────────────────────────────────────────────────────
# Step 2 Summary
# ─────────────────────────────────────────────────────────────────────────────────

cat("\n", paste(rep("-", 80), collapse = ""), "\n")
cat("STEP 2 SUMMARY\n")
cat(paste(rep("-", 80), collapse = ""), "\n")
cat("  Method: AddModuleScore (expression-level controlled)\n")
cat("  Positive markers:", length(all_pos_genes), "\n")
cat("  Positive categories:", paste(names(pos_markers_list), collapse = ", "), "\n")
cat("  Negative lineages:", length(neg_markers_list),
    "(", paste(names(neg_markers_list), collapse = ", "), ")\n")
cat("  Negative markers:", length(unlist(neg_markers_list)), "\n")
cat("  Weak identity (score < 0):", if(flagged_id) "YES" else "NONE", "\n")
cat("  Contamination (outlier):", if(flagged_contam) "YES" else "NONE", "\n")
cat("  Global median identity:", round(median(seurat_obj$identity_score1), 4), "\n")
cat("\n  Contamination baselines:\n")
for (i in 1:nrow(lineage_baselines)) {
    cat(sprintf("    %s: median=%.4f, threshold=%.4f (median + 3×MAD)\n",
                lineage_baselines$Lineage[i],
                lineage_baselines$Median[i],
                lineage_baselines$Threshold[i]))
}
cat("\n  Files saved:\n")
cat("    •", paste0(FILE_PREFIX, "_identity_scores.csv"), "\n")
cat("    •", paste0(FILE_PREFIX, "_identity_scores_state.csv"), "\n")
cat("    •", paste0(FILE_PREFIX, "_dotplot_cluster.{pdf,svg}"), "\n")
cat("    •", paste0(FILE_PREFIX, "_dotplot_state.{pdf,svg}"), "\n")
cat("    •", paste0(FILE_PREFIX, "_module_score_heatmap.{pdf,svg}"), "\n")
cat(paste(rep("=", 80), collapse = ""), "\n")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# STEP 3: SAMPLE COMPOSITION
# ════════════════════════════════════════════════════════════════════════════════

cat("\n", paste(rep("=", 80), collapse = ""), "\n")
cat("STEP 3: SAMPLE COMPOSITION\n")
cat(paste(rep("=", 80), collapse = ""), "\n")

# ─────────────────────────────────────────────────────────────────────────────────
# Thresholds
# ─────────────────────────────────────────────────────────────────────────────────

DONOR_DOMINANCE_THRESHOLD <- 60
EVENNESS_FLAG_THRESHOLD <- 0.40
FEW_DONORS_THRESHOLD <- 3

# ─────────────────────────────────────────────────────────────────────────────────
# 3A: Per-cluster donor composition
# ─────────────────────────────────────────────────────────────────────────────────

cat("\n▸ Analyzing donor composition per cluster...\n")

comp_cluster <- table(seurat_obj$seurat_clusters, seurat_obj@meta.data[[DONOR_COL]])

cluster_comp_summary <- do.call(rbind, lapply(cluster_order, function(cl) {
    cl_counts <- comp_cluster[cl, ]
    cl_counts <- cl_counts[cl_counts > 0]
    n_donors_cl <- length(cl_counts)
    total <- sum(cl_counts)
    sorted_counts <- sort(cl_counts, decreasing = TRUE)
    top1_pct <- sorted_counts[1] / total * 100
    top3_pct <- sum(sorted_counts[1:min(3, length(sorted_counts))]) / total * 100
    top_donor <- names(sorted_counts)[1]
    
    props <- cl_counts / total
    shannon <- -sum(props * log(props))
    max_shannon <- log(n_donors_cl)
    evenness <- if (max_shannon > 0) shannon / max_shannon else 0
    
    data.frame(
        Cluster = cl,
        State = qc_by_cluster$State[qc_by_cluster$Cluster == cl],
        N_cells = total,
        N_donors = n_donors_cl,
        Top_donor = top_donor,
        Top1_pct = round(top1_pct, 1),
        Top3_pct = round(top3_pct, 1),
        Shannon = round(shannon, 2),
        Evenness = round(evenness, 2),
        stringsAsFactors = FALSE
    )
}))

cat("\n  Cluster composition summary:\n")
print(cluster_comp_summary, row.names = FALSE)

write.csv(cluster_comp_summary,
          file.path(VAL_RESULTS_DIR, paste0(FILE_PREFIX, "_sample_comp_cluster.csv")),
          row.names = FALSE)

# ─────────────────────────────────────────────────────────────────────────────────
# 3B: Per-state donor composition
# ─────────────────────────────────────────────────────────────────────────────────

cat("\n▸ Analyzing donor composition per state...\n")

comp_state <- table(seurat_obj@meta.data[[SUBCLUSTER_COL]], seurat_obj@meta.data[[DONOR_COL]])

state_comp_summary <- do.call(rbind, lapply(states, function(s) {
    s_counts <- comp_state[s, ]
    s_counts <- s_counts[s_counts > 0]
    n_donors_s <- length(s_counts)
    total <- sum(s_counts)
    sorted_counts <- sort(s_counts, decreasing = TRUE)
    top1_pct <- sorted_counts[1] / total * 100
    top3_pct <- sum(sorted_counts[1:min(3, length(sorted_counts))]) / total * 100
    
    props <- s_counts / total
    shannon <- -sum(props * log(props))
    max_shannon <- log(n_donors_s)
    evenness <- if (max_shannon > 0) shannon / max_shannon else 0
    
    data.frame(
        State = s,
        N_cells = total,
        N_donors = n_donors_s,
        Top1_pct = round(top1_pct, 1),
        Top3_pct = round(top3_pct, 1),
        Shannon = round(shannon, 2),
        Evenness = round(evenness, 2),
        stringsAsFactors = FALSE
    )
}))

cat("\n  State composition summary:\n")
print(state_comp_summary, row.names = FALSE)

write.csv(state_comp_summary,
          file.path(VAL_RESULTS_DIR, paste0(FILE_PREFIX, "_sample_comp_state.csv")),
          row.names = FALSE)

# ─────────────────────────────────────────────────────────────────────────────────
# 3C: Flag sample-dominated clusters
# ─────────────────────────────────────────────────────────────────────────────────

cat("\n▸ Flagging sample-dominated clusters:\n")
cat(sprintf("  Thresholds: top donor >%d%%, evenness <%.2f, donors <=%d\n",
            DONOR_DOMINANCE_THRESHOLD, EVENNESS_FLAG_THRESHOLD, FEW_DONORS_THRESHOLD))

flagged_sample <- FALSE
flagged_clusters <- c()

for (i in 1:nrow(cluster_comp_summary)) {
    flags_list <- c()
    
    if (cluster_comp_summary$Top1_pct[i] > DONOR_DOMINANCE_THRESHOLD) {
        flags_list <- c(flags_list, sprintf("TOP DONOR >%d%% (%s = %.1f%%)",
                                   DONOR_DOMINANCE_THRESHOLD,
                                   cluster_comp_summary$Top_donor[i],
                                   cluster_comp_summary$Top1_pct[i]))
    }
    if (cluster_comp_summary$N_donors[i] <= FEW_DONORS_THRESHOLD) {
        flags_list <- c(flags_list, sprintf("FEW DONORS (n=%d)", cluster_comp_summary$N_donors[i]))
    }
    if (cluster_comp_summary$Evenness[i] < EVENNESS_FLAG_THRESHOLD) {
        flags_list <- c(flags_list, sprintf("LOW EVENNESS (%.2f)", cluster_comp_summary$Evenness[i]))
    }
    
    if (length(flags_list) > 0) {
        cat(sprintf("  ⚠ Cluster %s (%s, n=%d): %s\n",
                    cluster_comp_summary$Cluster[i],
                    cluster_comp_summary$State[i],
                    cluster_comp_summary$N_cells[i],
                    paste(flags_list, collapse = "; ")))
        flagged_sample <- TRUE
        flagged_clusters <- c(flagged_clusters, as.character(cluster_comp_summary$Cluster[i]))
    }
}
if (!flagged_sample) cat("  ✓ No sample-dominated clusters\n")

# ─────────────────────────────────────────────────────────────────────────────────
# 3D: State robustness after excluding flagged clusters
# ─────────────────────────────────────────────────────────────────────────────────

cat("\n▸ State robustness check (excluding flagged clusters)...\n")
cat("  Goal: Verify each state retains multi-donor support after removing\n")
cat("  sample-dominated clusters. If a state is defined primarily by flagged\n")
cat("  clusters, it may be a donor artifact rather than real biology.\n")

if (length(flagged_clusters) > 0) {
    flagged_cells <- as.character(seurat_obj$seurat_clusters) %in% flagged_clusters
    clean_meta <- seurat_obj@meta.data[!flagged_cells, ]
    
    state_robustness <- do.call(rbind, lapply(state_order, function(st) {
        all_cells <- seurat_obj@meta.data[[SUBCLUSTER_COL]] == st
        n_total <- sum(all_cells)
        n_donors_total <- length(unique(seurat_obj@meta.data[[DONOR_COL]][all_cells]))
        n_clusters_total <- length(unique(as.character(seurat_obj$seurat_clusters[all_cells])))
        
        clean_cells <- clean_meta[[SUBCLUSTER_COL]] == st
        n_clean <- sum(clean_cells)
        n_donors_clean <- length(unique(clean_meta[[DONOR_COL]][clean_cells]))
        n_clusters_clean <- length(unique(as.character(clean_meta$seurat_clusters[clean_cells])))
        
        pct_retained <- round(n_clean / n_total * 100, 1)
        
        donor_table <- table(clean_meta[[DONOR_COL]][clean_cells])
        donor_props <- as.numeric(donor_table) / sum(donor_table)
        evenness <- if (length(donor_props) > 1) {
            -sum(donor_props * log(donor_props)) / log(length(donor_props))
        } else { 0 }
        
        top_donor_pct <- round(max(donor_table) / sum(donor_table) * 100, 1)
        
        data.frame(
            State = st,
            Cells_total = n_total,
            Cells_clean = n_clean,
            Pct_retained = pct_retained,
            Donors_total = n_donors_total,
            Donors_clean = n_donors_clean,
            Clusters_total = n_clusters_total,
            Clusters_clean = n_clusters_clean,
            Top_donor_pct = top_donor_pct,
            Evenness = round(evenness, 2),
            stringsAsFactors = FALSE
        )
    }))
    
    cat("\n")
    print(state_robustness, row.names = FALSE)
    
    at_risk <- state_robustness[state_robustness$Donors_clean < 5 | 
                                 state_robustness$Pct_retained < 50 |
                                 state_robustness$Evenness < 0.5, ]
    
    if (nrow(at_risk) > 0) {
        cat("\n  ⚠ States at risk after removing flagged clusters:\n")
        for (i in 1:nrow(at_risk)) {
            reasons <- c()
            if (at_risk$Donors_clean[i] < 5) reasons <- c(reasons, 
                sprintf("only %d donors remain", at_risk$Donors_clean[i]))
            if (at_risk$Pct_retained[i] < 50) reasons <- c(reasons, 
                sprintf("only %.1f%% cells retained", at_risk$Pct_retained[i]))
            if (at_risk$Evenness[i] < 0.5) reasons <- c(reasons, 
                sprintf("low evenness (%.2f)", at_risk$Evenness[i]))
            cat(sprintf("    %s: %s\n", at_risk$State[i], paste(reasons, collapse = ", ")))
        }
    } else {
        cat("\n  ✓ All states retain multi-donor support after removing flagged clusters\n")
    }
    
    write.csv(state_robustness,
              file.path(VAL_RESULTS_DIR, paste0(FILE_PREFIX, "_state_robustness.csv")),
              row.names = FALSE)
} else {
    cat("  No flagged clusters -- skipping robustness check\n")
}

# ─────────────────────────────────────────────────────────────────────────────────
# 3E: Multi-resolution clustering + State stability across resolutions
# ─────────────────────────────────────────────────────────────────────────────────

cat("\n▸ State stability across resolutions...\n")
cat("  Goal: Verify each state emerges as a distinct cluster (or set of clusters)\n")
cat("  at multiple resolutions. A real state should be recoverable across\n")
cat("  resolutions, not only at the chosen one.\n")

# ── Ensure multi-resolution clustering exists ──
res_seq <- c(0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 1.0, 1.2)
existing_res <- grep("RNA_snn_res\\.", colnames(seurat_obj@meta.data), value = TRUE)
existing_values <- as.numeric(gsub("RNA_snn_res\\.", "", existing_res))
missing_res <- setdiff(res_seq, existing_values)

if (length(missing_res) > 0) {
    cat("  Computing clustering at", length(missing_res), "additional resolutions...\n")
    
    # Preserve original clustering
    seurat_obj$original_clusters <- seurat_obj$seurat_clusters
    
    # Determine reduction and dims
    cluster_reduction <- ifelse("harmony" %in% Reductions(seurat_obj), "harmony", "pca")
    if (!exists("N_DIMS_USE")) {
        N_DIMS_USE <- min(ncol(Embeddings(seurat_obj, cluster_reduction)), 30)
        cat("  N_DIMS_USE auto-detected:", N_DIMS_USE, "\n")
    }
    cat("  Using reduction:", cluster_reduction, ", dims: 1:", N_DIMS_USE, "\n")
    
    # Rebuild neighbors if needed
    if (length(seurat_obj@graphs) == 0) {
        cat("  Building neighbor graph...\n")
        seurat_obj <- FindNeighbors(seurat_obj, reduction = cluster_reduction,
                                     dims = 1:N_DIMS_USE, verbose = FALSE)
    }
    
    for (res in sort(missing_res)) {
        seurat_obj <- FindClusters(seurat_obj, resolution = res, verbose = FALSE)
        col_name <- paste0("RNA_snn_res.", res)
        n_cl <- length(unique(seurat_obj@meta.data[[col_name]]))
        cat(sprintf("    res = %.1f -> %d clusters\n", res, n_cl))
    }
    
    # Restore original clustering
    seurat_obj$seurat_clusters <- seurat_obj$original_clusters
    Idents(seurat_obj) <- seurat_obj$seurat_clusters
    seurat_obj$original_clusters <- NULL
    cat("  ✓ Restored original seurat_clusters (",
        length(unique(seurat_obj$seurat_clusters)), " clusters)\n", sep = "")
} else {
    cat("  All", length(res_seq), "resolutions already computed\n")
}

# Print resolution summary
available_res <- grep("RNA_snn_res\\.", colnames(seurat_obj@meta.data), value = TRUE)
res_values <- sort(as.numeric(gsub("RNA_snn_res\\.", "", available_res)))
cat("  Available resolutions:", paste(res_values, collapse = ", "), "\n\n")

for (res in res_seq) {
    col_name <- paste0("RNA_snn_res.", res)
    if (col_name %in% colnames(seurat_obj@meta.data)) {
        n_cl <- length(unique(seurat_obj@meta.data[[col_name]]))
        marker <- ifelse(res == CLUSTERING_RESOLUTION, " <- CURRENT", "")
        cat(sprintf("    res = %.1f -> %d clusters%s\n", res, n_cl, marker))
    }
}

# ── Compute state recovery per resolution ──
cat("\n  Computing state recovery per resolution...\n")

state_by_res <- do.call(rbind, lapply(res_values, function(res) {
    col_name <- paste0("RNA_snn_res.", res)
    
    do.call(rbind, lapply(state_order, function(st) {
        state_cells <- seurat_obj@meta.data[[SUBCLUSTER_COL]] == st
        
        majority_clusters <- c()
        for (cl in unique(seurat_obj@meta.data[[col_name]])) {
            cl_cells <- seurat_obj@meta.data[[col_name]] == cl
            cl_states <- table(seurat_obj@meta.data[[SUBCLUSTER_COL]][cl_cells])
            if (names(which.max(cl_states)) == st) {
                majority_clusters <- c(majority_clusters, cl)
            }
        }
        
        n_majority <- length(majority_clusters)
        in_majority <- sum(seurat_obj@meta.data[[col_name]][state_cells] %in% majority_clusters)
        recovery_pct <- round(in_majority / sum(state_cells) * 100, 1)
        
        data.frame(
            Resolution = res,
            State = st,
            N_majority_clusters = n_majority,
            Recovery_pct = recovery_pct,
            stringsAsFactors = FALSE
        )
    }))
}))

recovery_wide <- reshape(state_by_res[, c("Resolution", "State", "Recovery_pct")],
                          idvar = "Resolution", timevar = "State",
                          direction = "wide")
colnames(recovery_wide) <- gsub("Recovery_pct\\.", "", colnames(recovery_wide))
cat("\n  Recovery % per state across resolutions:\n")
print(recovery_wide, row.names = FALSE)

# ── Interpret per state ──
cat("\n")
for (st in state_order) {
    st_data <- state_by_res[state_by_res$State == st, ]
    min_rec <- min(st_data$Recovery_pct)
    low_recovery <- st_data[st_data$Recovery_pct < 50, ]
    
    if (nrow(low_recovery) > 0) {
        low_res <- low_recovery$Resolution
        chosen_rec <- st_data$Recovery_pct[st_data$Resolution == CLUSTERING_RESOLUTION]
        if (length(chosen_rec) == 0) chosen_rec <- NA
        
        all_below_chosen <- all(low_res < CLUSTERING_RESOLUTION)
        at_and_above <- st_data[st_data$Resolution >= CLUSTERING_RESOLUTION, ]
        stable_above <- all(at_and_above$Recovery_pct >= 70)
        
        if (all_below_chosen && stable_above && !is.na(chosen_rec) && chosen_rec >= 70) {
            first_stable <- min(st_data$Resolution[st_data$Recovery_pct >= 50])
            n_state_cells <- sum(seurat_obj@meta.data[[SUBCLUSTER_COL]] == st)
            size_note <- ifelse(n_state_cells < 2000, "a small state", "this state")
            cat(sprintf("  ✓ %s: stable at working resolutions (min %.1f%% at coarse res %s)\n",
                        st, min_rec, paste(low_res, collapse = ", ")))
            cat(sprintf("    Emerges from res %.1f onward — expected for %s (%d cells)\n",
                        first_stable, size_note, n_state_cells))
        } else if (!is.na(chosen_rec) && chosen_rec >= 70) {
            cat(sprintf("  ~ %s: partially stable -- <50%% at res %s, but %.1f%% at chosen res\n",
                        st, paste(low_res, collapse = ", "), chosen_rec))
            if (all_below_chosen) {
                cat("    Absorbed into neighboring states at coarse resolutions\n")
            } else {
                cat("    Variable recovery suggests resolution-sensitive boundaries\n")
            }
        } else {
            cat(sprintf("  ⚠ %s: UNSTABLE -- %.1f%% at chosen res, <50%% at res %s\n",
                        st, ifelse(is.na(chosen_rec), NA, chosen_rec),
                        paste(low_res, collapse = ", ")))
            cat("    This state may not be well-supported at the current resolution\n")
        }
    } else {
        cat(sprintf("  ✓ %s: stable across all resolutions (min recovery: %.1f%%)\n",
                    st, min_rec))
    }
}

write.csv(state_by_res,
          file.path(VAL_RESULTS_DIR, paste0(FILE_PREFIX, "_state_resolution_stability.csv")),
          row.names = FALSE)

# ─────────────────────────────────────────────────────────────────────────────────
# 3F: Stacked bar plot — by cluster
# ─────────────────────────────────────────────────────────────────────────────────

cat("\n▸ Generating composition plots...\n")

comp_df <- as.data.frame(comp_cluster)
colnames(comp_df) <- c("Cluster", "Donor", "Count")
comp_df <- comp_df[comp_df$Count > 0, ]
comp_df$Cluster <- factor(comp_df$Cluster, levels = cluster_order)

n_unique_donors <- length(unique(comp_df$Donor))

options(repr.plot.width = 12, repr.plot.height = 5)

p_comp_cluster <- ggplot(comp_df, aes(x = Cluster, y = Count, fill = Donor)) +
    geom_bar(stat = "identity", position = "fill", width = 0.8) +
    scale_y_continuous(labels = scales::percent, expand = c(0, 0)) +
    labs(x = "Cluster", y = "Proportion",
         title = paste0(toupper(cell_type_label), ": Donor composition per cluster")) +
    theme_bw(base_size = 10) +
    theme(
        legend.position = "none",
        axis.text.x = element_text(size = 9),
        plot.title = element_text(size = 11, face = "bold"),
        panel.grid.major.x = element_blank()
    )

print(p_comp_cluster)

ggsave(file.path(VAL_FIGURES_DIR, paste0(FILE_PREFIX, "_donor_comp_cluster.pdf")),
       p_comp_cluster, width = 12, height = 5)
ggsave(file.path(VAL_FIGURES_DIR, paste0(FILE_PREFIX, "_donor_comp_cluster.svg")),
       p_comp_cluster, width = 12, height = 5)

# ─────────────────────────────────────────────────────────────────────────────────
# 3G: Stacked bar plot — by state
# ─────────────────────────────────────────────────────────────────────────────────

comp_state_df <- as.data.frame(comp_state)
colnames(comp_state_df) <- c("State", "Donor", "Count")
comp_state_df <- comp_state_df[comp_state_df$Count > 0, ]
comp_state_df$State <- factor(comp_state_df$State, levels = states)

p_comp_state <- ggplot(comp_state_df, aes(x = State, y = Count, fill = Donor)) +
    geom_bar(stat = "identity", position = "fill", width = 0.8) +
    scale_y_continuous(labels = scales::percent, expand = c(0, 0)) +
    labs(x = "State", y = "Proportion",
         title = paste0(toupper(cell_type_label), ": Donor composition per state")) +
    theme_bw(base_size = 10) +
    theme(
        legend.position = "none",
        axis.text.x = element_text(angle = 30, hjust = 1, size = 9),
        plot.title = element_text(size = 11, face = "bold"),
        panel.grid.major.x = element_blank()
    )

print(p_comp_state)

ggsave(file.path(VAL_FIGURES_DIR, paste0(FILE_PREFIX, "_donor_comp_state.pdf")),
       p_comp_state, width = 8, height = 5)
ggsave(file.path(VAL_FIGURES_DIR, paste0(FILE_PREFIX, "_donor_comp_state.svg")),
       p_comp_state, width = 8, height = 5)

# ─────────────────────────────────────────────────────────────────────────────────
# 3H: Evenness bar plot
# ─────────────────────────────────────────────────────────────────────────────────

options(repr.plot.width = 10, repr.plot.height = 4)

cluster_comp_summary$Cluster <- factor(cluster_comp_summary$Cluster, levels = cluster_order)

p_evenness <- ggplot(cluster_comp_summary, aes(x = Cluster, y = Evenness, fill = State)) +
    geom_col(width = 0.7, alpha = 0.8) +
    geom_hline(yintercept = EVENNESS_FLAG_THRESHOLD, linetype = "dashed", color = "red", linewidth = 0.5) +
    scale_y_continuous(limits = c(0, 1), expand = c(0, 0)) +
    labs(x = "Cluster", y = "Shannon Evenness",
         title = paste0(toupper(cell_type_label), 
                        ": Donor evenness per cluster (red = ", EVENNESS_FLAG_THRESHOLD, " threshold)")) +
    theme_bw(base_size = 10) +
    theme(
        axis.text.x = element_text(size = 9),
        plot.title = element_text(size = 10, face = "bold"),
        legend.position = "bottom",
        legend.title = element_blank(),
        legend.text = element_text(size = 8)
    )

print(p_evenness)

ggsave(file.path(VAL_FIGURES_DIR, paste0(FILE_PREFIX, "_donor_evenness.pdf")),
       p_evenness, width = 10, height = 4)
ggsave(file.path(VAL_FIGURES_DIR, paste0(FILE_PREFIX, "_donor_evenness.svg")),
       p_evenness, width = 10, height = 4)

cat("  ✓ Saved composition plots\n")

# ─────────────────────────────────────────────────────────────────────────────────
# Step 3 Summary
# ─────────────────────────────────────────────────────────────────────────────────

cat("\n", paste(rep("-", 80), collapse = ""), "\n")
cat("STEP 3 SUMMARY\n")
cat(paste(rep("-", 80), collapse = ""), "\n")
cat("  Total donors:", n_donors, "\n")
cat("  Sample-dominated clusters:", 
    if(flagged_sample) paste(flagged_clusters, collapse = ", ") else "NONE", "\n")
cat("  Min evenness:", min(cluster_comp_summary$Evenness),
    "(cluster", cluster_comp_summary$Cluster[which.min(cluster_comp_summary$Evenness)], ")\n")
cat("  Max top-1 donor %:", max(cluster_comp_summary$Top1_pct),
    "(cluster", cluster_comp_summary$Cluster[which.max(cluster_comp_summary$Top1_pct)], ")\n")
if (exists("state_robustness")) {
    at_risk_states <- state_robustness$State[state_robustness$Donors_clean < 5 | 
                                              state_robustness$Pct_retained < 50 |
                                              state_robustness$Evenness < 0.5]
    if (length(at_risk_states) > 0) {
        cat("  States at risk:", paste(at_risk_states, collapse = ", "), "\n")
    } else {
        cat("  State robustness: All states retain multi-donor support\n")
    }
}
cat("  Resolutions computed:", paste(res_values, collapse = ", "), "\n")
cat("\n  Files saved:\n")
cat("    •", paste0(FILE_PREFIX, "_sample_comp_cluster.csv"), "\n")
cat("    •", paste0(FILE_PREFIX, "_sample_comp_state.csv"), "\n")
if (exists("state_robustness")) cat("    •", paste0(FILE_PREFIX, "_state_robustness.csv"), "\n")
cat("    •", paste0(FILE_PREFIX, "_state_resolution_stability.csv"), "\n")
cat("    •", paste0(FILE_PREFIX, "_donor_comp_cluster.{pdf,svg}"), "\n")
cat("    •", paste0(FILE_PREFIX, "_donor_comp_state.{pdf,svg}"), "\n")
cat("    •", paste0(FILE_PREFIX, "_donor_evenness.{pdf,svg}"), "\n")
cat(paste(rep("=", 80), collapse = ""), "\n")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# STEP 4: CELL CYCLE SCORING
# ════════════════════════════════════════════════════════════════════════════════

cat("\n", paste(rep("=", 80), collapse = ""), "\n")
cat("STEP 4: CELL CYCLE SCORING\n")
cat(paste(rep("=", 80), collapse = ""), "\n")

# ─────────────────────────────────────────────────────────────────────────────────
# 4A: Score cell cycle
# ─────────────────────────────────────────────────────────────────────────────────

cat("\n▸ Scoring cell cycle phases...\n")

s_genes <- cc.genes.updated.2019$s.genes
g2m_genes <- cc.genes.updated.2019$g2m.genes

s_found <- s_genes[s_genes %in% rownames(seurat_obj)]
g2m_found <- g2m_genes[g2m_genes %in% rownames(seurat_obj)]

cat("  S genes found:", length(s_found), "/", length(s_genes), "\n")
cat("  G2M genes found:", length(g2m_found), "/", length(g2m_genes), "\n")

seurat_obj <- CellCycleScoring(seurat_obj, s.features = s_found, g2m.features = g2m_found)

cat("  ✓ Cell cycle scoring complete\n")

# ─────────────────────────────────────────────────────────────────────────────────
# 4B: Phase distribution per cluster
# ─────────────────────────────────────────────────────────────────────────────────

cat("\n▸ Phase distribution per cluster:\n")

cl_ids <- as.character(seurat_obj$seurat_clusters)
phase_table <- table(cl_ids, seurat_obj$Phase)
phase_pct <- prop.table(phase_table, margin = 1) * 100

phase_df <- as.data.frame(round(phase_pct, 1))
colnames(phase_df) <- c("Cluster", "Phase", "Pct")

# Wide format for printing
phase_wide <- as.data.frame.matrix(round(phase_pct, 1))
phase_wide$Cluster <- rownames(phase_wide)
phase_wide$Cycling <- round(phase_pct[, "S"] + phase_pct[, "G2M"], 1)
phase_wide$State <- sapply(phase_wide$Cluster, function(cl) {
    cluster_comp_summary$State[cluster_comp_summary$Cluster == cl]
})
phase_wide <- phase_wide[match(cluster_order, phase_wide$Cluster), 
                          c("Cluster", "State", "G1", "G2M", "S", "Cycling")]

print(phase_wide, row.names = FALSE)

write.csv(phase_wide, file.path(VAL_RESULTS_DIR, paste0(FILE_PREFIX, "_cellcycle_cluster.csv")),
          row.names = FALSE)

# ─────────────────────────────────────────────────────────────────────────────────
# 4C: Phase distribution per state
# ─────────────────────────────────────────────────────────────────────────────────

cat("\n▸ Phase distribution per state:\n")

state_ids <- as.character(seurat_obj@meta.data[[SUBCLUSTER_COL]])
phase_state <- table(state_ids, seurat_obj$Phase)
phase_state_pct <- prop.table(phase_state, margin = 1) * 100

phase_state_wide <- as.data.frame.matrix(round(phase_state_pct, 1))
phase_state_wide$State <- rownames(phase_state_wide)
phase_state_wide$Cycling <- round(phase_state_pct[, "S"] + phase_state_pct[, "G2M"], 1)
phase_state_wide <- phase_state_wide[match(state_order, phase_state_wide$State),
                                      c("State", "G1", "G2M", "S", "Cycling")]

print(phase_state_wide, row.names = FALSE)

write.csv(phase_state_wide, file.path(VAL_RESULTS_DIR, paste0(FILE_PREFIX, "_cellcycle_state.csv")),
          row.names = FALSE)

# ─────────────────────────────────────────────────────────────────────────────────
# 4D: Flag cycling clusters (>30% S/G2M)
# ─────────────────────────────────────────────────────────────────────────────────

cat("\n▸ Flagging cycling clusters (>30% S/G2M):\n")

flagged_cc <- FALSE
for (i in 1:nrow(phase_wide)) {
    if (phase_wide$Cycling[i] > 30) {
        cat(sprintf("  ⚠ Cluster %s (%s): %.1f%% cycling (S=%.1f%%, G2M=%.1f%%)\n",
                    phase_wide$Cluster[i], phase_wide$State[i], phase_wide$Cycling[i],
                    phase_wide$S[i], phase_wide$G2M[i]))
        flagged_cc <- TRUE
    }
}
if (!flagged_cc) cat("  ✓ No heavily cycling clusters\n")

# ─────────────────────────────────────────────────────────────────────────────────
# 4E: Stacked bar — cell cycle per cluster
# ─────────────────────────────────────────────────────────────────────────────────

cat("\n▸ Generating cell cycle plots...\n")

PHASE_COLORS <- c("G1" = "#B0B0B0", "S" = "#E15759", "G2M" = "#4E79A7")

phase_plot_df <- as.data.frame(phase_table)
colnames(phase_plot_df) <- c("Cluster", "Phase", "Count")
phase_plot_df$Cluster <- factor(phase_plot_df$Cluster, levels = cluster_order)
phase_plot_df$Phase <- factor(phase_plot_df$Phase, levels = c("G1", "S", "G2M"))

options(repr.plot.width = 12, repr.plot.height = 5)

p_cc_cluster <- ggplot(phase_plot_df, aes(x = Cluster, y = Count, fill = Phase)) +
    geom_col(position = "fill", width = 0.8, color = "white", linewidth = 0.2) +
    scale_fill_manual(values = PHASE_COLORS) +
    scale_y_continuous(labels = percent, expand = c(0, 0)) +
    labs(x = "Cluster", y = "Proportion", fill = "Cell Cycle Phase",
         title = paste0(toupper(cell_type_label), ": Cell cycle phase per cluster")) +
    theme_minimal(base_size = 14) +
    theme(
        axis.text.x = element_text(size = 12, color = "black"),
        axis.text.y = element_text(size = 12, color = "black"),
        axis.title.x = element_text(size = 14, face = "bold", margin = margin(t = 10)),
        axis.title.y = element_text(size = 14, face = "bold", margin = margin(r = 10)),
        axis.line = element_line(color = "black", linewidth = 0.5),
        axis.ticks = element_line(color = "black", linewidth = 0.3),
        axis.ticks.length = unit(0.15, "cm"),
        plot.title = element_text(size = 14, face = "bold"),
        legend.position = "right",
        legend.title = element_text(size = 11, face = "bold"),
        legend.text = element_text(size = 10),
        panel.grid = element_blank(),
        plot.margin = margin(15, 15, 15, 15)
    )

print(p_cc_cluster)

ggsave(file.path(VAL_FIGURES_DIR, paste0(FILE_PREFIX, "_cellcycle_cluster.pdf")),
       p_cc_cluster, width = 12, height = 5)
ggsave(file.path(VAL_FIGURES_DIR, paste0(FILE_PREFIX, "_cellcycle_cluster.svg")),
       p_cc_cluster, width = 12, height = 5)

# ─────────────────────────────────────────────────────────────────────────────────
# 4F: Stacked bar — cell cycle per state
# ─────────────────────────────────────────────────────────────────────────────────

phase_state_plot <- as.data.frame(phase_state)
colnames(phase_state_plot) <- c("State", "Phase", "Count")
phase_state_plot$State <- factor(phase_state_plot$State, levels = state_order)
phase_state_plot$Phase <- factor(phase_state_plot$Phase, levels = c("G1", "S", "G2M"))

options(repr.plot.width = 8, repr.plot.height = 5)

p_cc_state <- ggplot(phase_state_plot, aes(x = State, y = Count, fill = Phase)) +
    geom_col(position = "fill", width = 0.8, color = "white", linewidth = 0.2) +
    scale_fill_manual(values = PHASE_COLORS) +
    scale_y_continuous(labels = percent, expand = c(0, 0)) +
    labs(x = paste0(tools::toTitleCase(cell_type_label), " State"), y = "Proportion",
         fill = "Cell Cycle Phase",
         title = paste0(toupper(cell_type_label), ": Cell cycle phase per state")) +
    theme_minimal(base_size = 14) +
    theme(
        axis.text.x = element_text(size = 12, color = "black", angle = 30, hjust = 1),
        axis.text.y = element_text(size = 12, color = "black"),
        axis.title.x = element_text(size = 14, face = "bold", margin = margin(t = 10)),
        axis.title.y = element_text(size = 14, face = "bold", margin = margin(r = 10)),
        axis.line = element_line(color = "black", linewidth = 0.5),
        axis.ticks = element_line(color = "black", linewidth = 0.3),
        axis.ticks.length = unit(0.15, "cm"),
        plot.title = element_text(size = 14, face = "bold"),
        legend.position = "right",
        legend.title = element_text(size = 11, face = "bold"),
        legend.text = element_text(size = 10),
        panel.grid = element_blank(),
        plot.margin = margin(15, 15, 15, 15)
    )

print(p_cc_state)

ggsave(file.path(VAL_FIGURES_DIR, paste0(FILE_PREFIX, "_cellcycle_state.pdf")),
       p_cc_state, width = 8, height = 5)
ggsave(file.path(VAL_FIGURES_DIR, paste0(FILE_PREFIX, "_cellcycle_state.svg")),
       p_cc_state, width = 8, height = 5)

# ─────────────────────────────────────────────────────────────────────────────────
# 4G: UMAP — Phase + S.Score + G2M.Score (with corner arrows)
# ─────────────────────────────────────────────────────────────────────────────────

if (!exists("add_corner_arrows")) {
    add_corner_arrows <- function(p, label_size = 3) {
        build <- ggplot_build(p)
        x_range <- build$layout$panel_params[[1]]$x.range
        y_range <- build$layout$panel_params[[1]]$y.range
        
        x_start <- x_range[1] + diff(x_range) * 0.02
        y_start <- y_range[1] + diff(y_range) * 0.02
        x_arrow <- diff(x_range) * 0.12
        y_arrow <- diff(y_range) * 0.12
        
        p + 
            annotate("segment", x = x_start, xend = x_start + x_arrow,
                     y = y_start, yend = y_start,
                     arrow = arrow(length = unit(0.12, "cm"), type = "closed"),
                     linewidth = 0.4) +
            annotate("text", x = x_start + x_arrow/2,
                     y = y_start - diff(y_range) * 0.04,
                     label = "UMAP1", size = label_size,
                     hjust = 0.5, vjust = 1, fontface = "bold") +
            annotate("segment", x = x_start, xend = x_start,
                     y = y_start, yend = y_start + y_arrow,
                     arrow = arrow(length = unit(0.12, "cm"), type = "closed"),
                     linewidth = 0.4) +
            annotate("text", x = x_start - diff(x_range) * 0.04,
                     y = y_start + y_arrow/2,
                     label = "UMAP2", size = label_size,
                     hjust = 1, vjust = 0.5, angle = 90, fontface = "bold") +
            coord_cartesian(clip = "off")
    }
    
    umap_theme <- theme_void(base_size = 12) +
        theme(
            plot.title = element_text(size = 11, face = "bold", hjust = 0.5),
            legend.text = element_text(size = 9),
            legend.title = element_blank(),
            plot.margin = margin(15, 15, 20, 20)
        )
    
    umap_theme_noleg <- umap_theme +
        theme(legend.position = "none")
}

options(repr.plot.width = 14, repr.plot.height = 4.5)

p_phase_umap <- DimPlot(seurat_obj, group.by = "Phase", reduction = UMAP_RED,
                         cols = PHASE_COLORS, pt.size = 0.1) +
    ggtitle("Cell Cycle Phase") +
    umap_theme
p_phase_umap <- add_corner_arrows(p_phase_umap, label_size = 2.5)

p_s_score <- FeaturePlot(seurat_obj, features = "S.Score", reduction = UMAP_RED,
                          pt.size = 0.1, order = TRUE) +
    scale_color_viridis_c() +
    ggtitle("S.Score") +
    umap_theme_noleg
p_s_score <- add_corner_arrows(p_s_score, label_size = 2.5)

p_g2m_score <- FeaturePlot(seurat_obj, features = "G2M.Score", reduction = UMAP_RED,
                            pt.size = 0.1, order = TRUE) +
    scale_color_viridis_c() +
    ggtitle("G2M.Score") +
    umap_theme_noleg
p_g2m_score <- add_corner_arrows(p_g2m_score, label_size = 2.5)

p_cc_umap <- cowplot::plot_grid(p_phase_umap, p_s_score, p_g2m_score, ncol = 3)

print(p_cc_umap)

ggsave(file.path(VAL_FIGURES_DIR, paste0(FILE_PREFIX, "_cellcycle_umap.pdf")),
       p_cc_umap, width = 14, height = 4.5)
ggsave(file.path(VAL_FIGURES_DIR, paste0(FILE_PREFIX, "_cellcycle_umap.svg")),
       p_cc_umap, width = 14, height = 4.5)

cat("  ✓ Saved cell cycle plots\n")

# ─────────────────────────────────────────────────────────────────────────────────
# Step 4 Summary
# ─────────────────────────────────────────────────────────────────────────────────

cat("\n", paste(rep("-", 80), collapse = ""), "\n")
cat("STEP 4 SUMMARY\n")
cat(paste(rep("-", 80), collapse = ""), "\n")
cat("  Overall cycling (S+G2M):",
    round(sum(seurat_obj$Phase %in% c("S", "G2M")) / ncol(seurat_obj) * 100, 1), "%\n")
cat("  Cycling clusters (>30%):", if(flagged_cc) "YES" else "NONE", "\n")

# Top cycling clusters regardless of threshold
top_cycling <- phase_wide[order(-phase_wide$Cycling), ][1:3, ]
cat("  Top 3 cycling clusters:\n")
for (i in 1:3) {
    cat(sprintf("    Cluster %s (%s): %.1f%%\n",
                top_cycling$Cluster[i], top_cycling$State[i], top_cycling$Cycling[i]))
}

cat("\n  Files saved:\n")
cat("    •", paste0(FILE_PREFIX, "_cellcycle_cluster.csv"), "\n")
cat("    •", paste0(FILE_PREFIX, "_cellcycle_state.csv"), "\n")
cat("    •", paste0(FILE_PREFIX, "_cellcycle_cluster.{pdf,svg}"), "\n")
cat("    •", paste0(FILE_PREFIX, "_cellcycle_state.{pdf,svg}"), "\n")
cat("    •", paste0(FILE_PREFIX, "_cellcycle_umap.{pdf,svg}"), "\n")
cat(paste(rep("=", 80), collapse = ""), "\n")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# STEP 5: CLUSTER STABILITY (CLUSTREE)
# ════════════════════════════════════════════════════════════════════════════════

cat("\n", paste(rep("=", 80), collapse = ""), "\n")
cat("STEP 5: CLUSTER STABILITY\n")
cat(paste(rep("=", 80), collapse = ""), "\n")

library(clustree)

# Safety check
if (!exists("N_DIMS_USE")) {
    if ("harmony" %in% Reductions(seurat_obj)) {
        N_DIMS_USE <- min(ncol(Embeddings(seurat_obj, "harmony")), 30)
    } else if ("pca" %in% Reductions(seurat_obj)) {
        N_DIMS_USE <- min(ncol(Embeddings(seurat_obj, "pca")), 30)
    } else {
        N_DIMS_USE <- 15
    }
    cat("  N_DIMS_USE detected:", N_DIMS_USE, "\n")
} else {
    cat("  N_DIMS_USE:", N_DIMS_USE, "\n")
}

# ─────────────────────────────────────────────────────────────────────────────────
# 5A: Ensure multi-resolution clustering exists
# ─────────────────────────────────────────────────────────────────────────────────

cat("\n▸ Multi-resolution clustering...\n")

res_seq <- c(0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 1.0, 1.2)
existing_res <- grep("RNA_snn_res\\.", colnames(seurat_obj@meta.data), value = TRUE)
existing_values <- as.numeric(gsub("RNA_snn_res\\.", "", existing_res))
missing_res <- setdiff(res_seq, existing_values)

if (length(missing_res) > 0) {
    cat("  Computing", length(missing_res), "missing resolutions...\n")
    
    seurat_obj$original_clusters <- seurat_obj$seurat_clusters
    
    cluster_reduction <- ifelse("harmony" %in% Reductions(seurat_obj), "harmony", "pca")
    
    if (length(seurat_obj@graphs) == 0) {
        cat("  Building neighbor graph...\n")
        seurat_obj <- FindNeighbors(seurat_obj, reduction = cluster_reduction,
                                     dims = 1:N_DIMS_USE, verbose = FALSE)
    }
    
    for (res in sort(missing_res)) {
        seurat_obj <- FindClusters(seurat_obj, resolution = res, verbose = FALSE)
        col_name <- paste0("RNA_snn_res.", res)
        n_cl <- length(unique(seurat_obj@meta.data[[col_name]]))
        cat(sprintf("    res = %.1f -> %d clusters\n", res, n_cl))
    }
    
    seurat_obj$seurat_clusters <- seurat_obj$original_clusters
    Idents(seurat_obj) <- seurat_obj$seurat_clusters
    seurat_obj$original_clusters <- NULL
    cat("  ✓ Restored original seurat_clusters\n")
} else {
    cat("  All resolutions already computed (from Step 3)\n")
}

# Print summary
for (res in res_seq) {
    col_name <- paste0("RNA_snn_res.", res)
    n_cl <- length(unique(seurat_obj@meta.data[[col_name]]))
    marker <- ifelse(res == CLUSTERING_RESOLUTION, " <- CURRENT", "")
    cat(sprintf("    res = %.1f -> %d clusters%s\n", res, n_cl, marker))
}

# ─────────────────────────────────────────────────────────────────────────────────
# 5B: Resolution summary table
# ─────────────────────────────────────────────────────────────────────────────────

cat("\n▸ Resolution summary:\n")

res_summary <- do.call(rbind, lapply(res_seq, function(res) {
    col_name <- paste0("RNA_snn_res.", res)
    cl_sizes <- table(seurat_obj@meta.data[[col_name]])
    data.frame(
        Resolution = res,
        N_clusters = length(cl_sizes),
        Min_size = min(cl_sizes),
        Max_size = max(cl_sizes),
        Median_size = median(cl_sizes),
        stringsAsFactors = FALSE
    )
}))

print(res_summary, row.names = FALSE)

write.csv(res_summary, file.path(VAL_RESULTS_DIR, paste0(FILE_PREFIX, "_resolution_summary.csv")),
          row.names = FALSE)

# ─────────────────────────────────────────────────────────────────────────────────
# 5C: Clustree plot
# ─────────────────────────────────────────────────────────────────────────────────

cat("\n▸ Generating clustree...\n")

options(repr.plot.width = 12, repr.plot.height = 10)

p_tree <- clustree(seurat_obj, prefix = "RNA_snn_res.") +
    ggtitle(paste0(toupper(cell_type_label),
                   ": Cluster Stability Across Resolutions")) +
    theme(plot.title = element_text(size = 14, face = "bold"),
          legend.text = element_text(size = 9),
          legend.title = element_text(size = 10))

print(p_tree)

ggsave(file.path(VAL_FIGURES_DIR, paste0(FILE_PREFIX, "_clustree.pdf")),
       p_tree, width = 12, height = 10)
ggsave(file.path(VAL_FIGURES_DIR, paste0(FILE_PREFIX, "_clustree.svg")),
       p_tree, width = 12, height = 10)

# ─────────────────────────────────────────────────────────────────────────────────
# 5D: Clustree with state overlay
# ─────────────────────────────────────────────────────────────────────────────────

cat("\n▸ Generating clustree with state overlay...\n")

state_numeric <- as.numeric(factor(seurat_obj@meta.data[[SUBCLUSTER_COL]],
                                    levels = state_order))
seurat_obj$state_numeric <- state_numeric

options(repr.plot.width = 12, repr.plot.height = 10)

p_tree_state <- clustree(seurat_obj, prefix = "RNA_snn_res.",
                          node_colour = "state_numeric",
                          node_colour_aggr = "median") +
    scale_color_gradientn(colors = state_colors,
                          breaks = 1:length(state_order),
                          labels = state_order,
                          name = "Dominant\nState") +
    ggtitle(paste0(toupper(cell_type_label),
                   ": Cluster Stability (colored by dominant state)")) +
    theme(plot.title = element_text(size = 14, face = "bold"),
          legend.text = element_text(size = 9),
          legend.title = element_text(size = 10))

print(p_tree_state)

ggsave(file.path(VAL_FIGURES_DIR, paste0(FILE_PREFIX, "_clustree_state.pdf")),
       p_tree_state, width = 12, height = 10)
ggsave(file.path(VAL_FIGURES_DIR, paste0(FILE_PREFIX, "_clustree_state.svg")),
       p_tree_state, width = 12, height = 10)

# ─────────────────────────────────────────────────────────────────────────────────
# 5E: State stability across resolutions
# ─────────────────────────────────────────────────────────────────────────────────

cat("\n▸ Assessing state stability across resolutions...\n")

stability_results <- list()

for (res in res_seq) {
    col_name <- paste0("RNA_snn_res.", res)
    clusters_at_res <- seurat_obj@meta.data[[col_name]]
    states_at_res <- seurat_obj@meta.data[[SUBCLUSTER_COL]]
    
    cluster_purity <- tapply(states_at_res, clusters_at_res, function(x) {
        max(table(x)) / length(x) * 100
    })
    
    stability_results[[as.character(res)]] <- data.frame(
        Resolution = res,
        N_clusters = length(cluster_purity),
        Mean_purity = round(mean(cluster_purity), 1),
        Min_purity = round(min(cluster_purity), 1),
        N_mixed = sum(cluster_purity < 70),
        stringsAsFactors = FALSE
    )
}

stability_df <- do.call(rbind, stability_results)
cat("\n  State purity per resolution:\n")
print(stability_df, row.names = FALSE)

write.csv(stability_df, file.path(VAL_RESULTS_DIR, paste0(FILE_PREFIX, "_state_stability.csv")),
          row.names = FALSE)

# Save cluster_purity at current resolution for Summary cell
current_col <- paste0("RNA_snn_res.", CLUSTERING_RESOLUTION)
if (current_col %in% colnames(seurat_obj@meta.data)) {
    cluster_purity <- tapply(
        seurat_obj@meta.data[[SUBCLUSTER_COL]],
        seurat_obj@meta.data[[current_col]],
        function(x) max(table(x)) / length(x) * 100
    )
}

# ─────────────────────────────────────────────────────────────────────────────────
# 5F: Purity plot
# ─────────────────────────────────────────────────────────────────────────────────

options(repr.plot.width = 8, repr.plot.height = 4.5)

p_purity <- ggplot(stability_df, aes(x = Resolution)) +
    geom_line(aes(y = Mean_purity), color = "#4E79A7", linewidth = 1) +
    geom_point(aes(y = Mean_purity), color = "#4E79A7", size = 3) +
    geom_line(aes(y = Min_purity), color = "#E15759", linewidth = 0.8, linetype = "dashed") +
    geom_point(aes(y = Min_purity), color = "#E15759", size = 2) +
    geom_vline(xintercept = CLUSTERING_RESOLUTION, linetype = "dotted",
               color = "black", linewidth = 0.8) +
    annotate("text", x = CLUSTERING_RESOLUTION + 0.03, y = 50,
             label = paste0("Current (", CLUSTERING_RESOLUTION, ")"),
             hjust = 0, size = 3.5, fontface = "bold") +
    scale_y_continuous(limits = c(0, 100)) +
    labs(x = "Clustering Resolution", y = "State Purity (%)",
         title = paste0(toupper(cell_type_label),
                        ": State purity across resolutions"),
         subtitle = "Blue = mean purity, Red dashed = min purity") +
    theme_minimal(base_size = 14) +
    theme(
        axis.text.x = element_text(size = 12, color = "black"),
        axis.text.y = element_text(size = 12, color = "black"),
        axis.title.x = element_text(size = 14, face = "bold", margin = margin(t = 10)),
        axis.title.y = element_text(size = 14, face = "bold", margin = margin(r = 10)),
        axis.line = element_line(color = "black", linewidth = 0.5),
        axis.ticks = element_line(color = "black", linewidth = 0.3),
        axis.ticks.length = unit(0.15, "cm"),
        plot.title = element_text(size = 13, face = "bold"),
        plot.subtitle = element_text(size = 10, color = "#666666"),
        panel.grid = element_blank(),
        plot.margin = margin(15, 15, 15, 15)
    )

print(p_purity)

ggsave(file.path(VAL_FIGURES_DIR, paste0(FILE_PREFIX, "_state_purity.pdf")),
       p_purity, width = 8, height = 4.5)
ggsave(file.path(VAL_FIGURES_DIR, paste0(FILE_PREFIX, "_state_purity.svg")),
       p_purity, width = 8, height = 4.5)

cat("  ✓ Saved stability plots\n")

# ─────────────────────────────────────────────────────────────────────────────────
# 5G: Verify original clustering is intact
# ─────────────────────────────────────────────────────────────────────────────────

Idents(seurat_obj) <- seurat_obj$seurat_clusters
cat("\n  ✓ Active identity: seurat_clusters (",
    length(unique(Idents(seurat_obj))), " clusters)\n", sep = "")

# ─────────────────────────────────────────────────────────────────────────────────
# Step 5 Summary
# ─────────────────────────────────────────────────────────────────────────────────

cat("\n", paste(rep("-", 80), collapse = ""), "\n")
cat("STEP 5 SUMMARY\n")
cat(paste(rep("-", 80), collapse = ""), "\n")

current_res <- stability_df[stability_df$Resolution == CLUSTERING_RESOLUTION, ]
cat("  Current resolution:", CLUSTERING_RESOLUTION, "\n")
if (nrow(current_res) > 0) {
    cat("  Clusters at current res:", current_res$N_clusters, "\n")
    cat("  Mean state purity:", current_res$Mean_purity, "%\n")
    cat("  Min state purity:", current_res$Min_purity, "%\n")
    cat("  Mixed clusters (<70% purity):", current_res$N_mixed, "\n")
} else {
    cat("  Note: CLUSTERING_RESOLUTION", CLUSTERING_RESOLUTION,
        "not in res_seq, using original clusters\n")
    cat("  Original clusters:", length(unique(seurat_obj$seurat_clusters)), "\n")
}

cat("\n  Verification: seurat_clusters has",
    length(unique(seurat_obj$seurat_clusters)), "clusters (original preserved)\n")

cat("\n  Files saved:\n")
cat("    •", paste0(FILE_PREFIX, "_resolution_summary.csv"), "\n")
cat("    •", paste0(FILE_PREFIX, "_state_stability.csv"), "\n")
cat("    •", paste0(FILE_PREFIX, "_clustree.{pdf,svg}"), "\n")
cat("    •", paste0(FILE_PREFIX, "_clustree_state.{pdf,svg}"), "\n")
cat("    •", paste0(FILE_PREFIX, "_state_purity.{pdf,svg}"), "\n")
cat(paste(rep("=", 80), collapse = ""), "\n")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# VALIDATION UMAPs
# ════════════════════════════════════════════════════════════════════════════════

cat("\n", paste(rep("=", 80), collapse = ""), "\n")
cat("VALIDATION UMAPs\n")
cat(paste(rep("=", 80), collapse = ""), "\n")

# ─────────────────────────────────────────────────────────────────────────────────
# Panel 1: Clusters + States + Flagged
# ─────────────────────────────────────────────────────────────────────────────────

options(repr.plot.width = 16, repr.plot.height = 5)

# Cluster UMAP
p_clusters <- DimPlot(seurat_obj, group.by = "seurat_clusters", reduction = UMAP_RED,
                       label = TRUE, label.size = 3, pt.size = 0.1, repel = TRUE) +
    ggtitle("Clusters") +
    umap_theme_noleg
p_clusters <- add_corner_arrows(p_clusters, label_size = 2.5)

# State UMAP
p_states <- DimPlot(seurat_obj, group.by = SUBCLUSTER_COL, reduction = UMAP_RED,
                     cols = state_colors, pt.size = 0.1) +
    ggtitle("Annotated States") +
    umap_theme
p_states <- add_corner_arrows(p_states, label_size = 2.5)

# Flagged clusters highlighted
flagged_clusters <- c("16", "12", "13")
seurat_obj$flagged <- ifelse(
    as.character(seurat_obj$seurat_clusters) == "16", "Remove (Cl.16)",
    ifelse(as.character(seurat_obj$seurat_clusters) %in% c("12", "13"), "Monitor (Cl.12,13)",
           "Pass"))
seurat_obj$flagged <- factor(seurat_obj$flagged,
                              levels = c("Pass", "Monitor (Cl.12,13)", "Remove (Cl.16)"))

FLAG_COLORS <- c("Pass" = "#D3D3D3", "Monitor (Cl.12,13)" = "#F28E2B", "Remove (Cl.16)" = "#E15759")

p_flagged <- DimPlot(seurat_obj, group.by = "flagged", reduction = UMAP_RED,
                      cols = FLAG_COLORS, pt.size = 0.1, order = c("Remove (Cl.16)", "Monitor (Cl.12,13)")) +
    ggtitle("Flagged Clusters") +
    umap_theme
p_flagged <- add_corner_arrows(p_flagged, label_size = 2.5)

p_panel1 <- cowplot::plot_grid(p_clusters, p_states, p_flagged, ncol = 3)
print(p_panel1)

ggsave(file.path(VAL_FIGURES_DIR, paste0(FILE_PREFIX, "_umap_panel1.pdf")),
       p_panel1, width = 16, height = 5)
ggsave(file.path(VAL_FIGURES_DIR, paste0(FILE_PREFIX, "_umap_panel1.svg")),
       p_panel1, width = 16, height = 5)

# ─────────────────────────────────────────────────────────────────────────────────
# Panel 2: QC metrics on UMAP
# ─────────────────────────────────────────────────────────────────────────────────

options(repr.plot.width = 14, repr.plot.height = 4)

qc_umap_list <- lapply(QC_METRICS, function(feat) {
    p <- FeaturePlot(seurat_obj, features = feat, reduction = UMAP_RED,
                      pt.size = 0.1, order = TRUE) +
        scale_color_viridis_c() +
        ggtitle(feat) +
        umap_theme_noleg
    add_corner_arrows(p, label_size = 2)
})

p_qc_umap <- cowplot::plot_grid(plotlist = qc_umap_list, ncol = length(QC_METRICS))
print(p_qc_umap)

ggsave(file.path(VAL_FIGURES_DIR, paste0(FILE_PREFIX, "_umap_qc.pdf")),
       p_qc_umap, width = 3.5 * length(QC_METRICS), height = 4)
ggsave(file.path(VAL_FIGURES_DIR, paste0(FILE_PREFIX, "_umap_qc.svg")),
       p_qc_umap, width = 3.5 * length(QC_METRICS), height = 4)

# ─────────────────────────────────────────────────────────────────────────────────
# Panel 3: Donor + Study Group + Senescence
# ─────────────────────────────────────────────────────────────────────────────────

options(repr.plot.width = 16, repr.plot.height = 5)

p_donor <- DimPlot(seurat_obj, group.by = DONOR_COL, reduction = UMAP_RED,
                    pt.size = 0.1) +
    ggtitle("Donor") +
    umap_theme_noleg
p_donor <- add_corner_arrows(p_donor, label_size = 2.5)

study_colors <- STUDY_GROUP_COLORS[names(STUDY_GROUP_COLORS) %in% sg_order]
p_sg <- DimPlot(seurat_obj, group.by = STUDY_GROUP_COL, reduction = UMAP_RED,
                 cols = study_colors, pt.size = 0.1) +
    ggtitle("Study Group") +
    umap_theme +
    theme(legend.key.size = unit(0.3, "cm"),
          legend.text = element_text(size = 7))
p_sg <- add_corner_arrows(p_sg, label_size = 2.5)

snc_col <- ifelse("senescence_label_state" %in% colnames(seurat_obj@meta.data),
                   "senescence_label_state", "senescence_label")
p_snc <- DimPlot(seurat_obj, group.by = snc_col, reduction = UMAP_RED,
                  cols = SENESCENCE_COLORS, pt.size = 0.1,
                  order = "SnC") +
    ggtitle("Senescence") +
    umap_theme
p_snc <- add_corner_arrows(p_snc, label_size = 2.5)

p_panel3 <- cowplot::plot_grid(p_donor, p_sg, p_snc, ncol = 3)
print(p_panel3)

ggsave(file.path(VAL_FIGURES_DIR, paste0(FILE_PREFIX, "_umap_panel3.pdf")),
       p_panel3, width = 16, height = 5)
ggsave(file.path(VAL_FIGURES_DIR, paste0(FILE_PREFIX, "_umap_panel3.svg")),
       p_panel3, width = 16, height = 5)

cat("\n  ✓ Saved UMAP panels\n")

# ─────────────────────────────────────────────────────────────────────────────────
# Panel 4: Cell cycle on UMAP
# ─────────────────────────────────────────────────────────────────────────────────

options(repr.plot.width = 14, repr.plot.height = 4.5)

PHASE_COLORS <- c("G1" = "#B0B0B0", "S" = "#E15759", "G2M" = "#4E79A7")

p_phase <- DimPlot(seurat_obj, group.by = "Phase", reduction = UMAP_RED,
                    cols = PHASE_COLORS, pt.size = 0.1) +
    ggtitle("Cell Cycle Phase") +
    umap_theme
p_phase <- add_corner_arrows(p_phase, label_size = 2.5)

p_s <- FeaturePlot(seurat_obj, features = "S.Score", reduction = UMAP_RED,
                    pt.size = 0.1, order = TRUE) +
    scale_color_viridis_c() +
    ggtitle("S.Score") +
    umap_theme_noleg
p_s <- add_corner_arrows(p_s, label_size = 2.5)

p_g2m <- FeaturePlot(seurat_obj, features = "G2M.Score", reduction = UMAP_RED,
                      pt.size = 0.1, order = TRUE) +
    scale_color_viridis_c() +
    ggtitle("G2M.Score") +
    umap_theme_noleg
p_g2m <- add_corner_arrows(p_g2m, label_size = 2.5)

p_panel4 <- cowplot::plot_grid(p_phase, p_s, p_g2m, ncol = 3)
print(p_panel4)

ggsave(file.path(VAL_FIGURES_DIR, paste0(FILE_PREFIX, "_umap_cellcycle.pdf")),
       p_panel4, width = 14, height = 4.5)
ggsave(file.path(VAL_FIGURES_DIR, paste0(FILE_PREFIX, "_umap_cellcycle.svg")),
       p_panel4, width = 14, height = 4.5)

cat("  ✓ Saved all UMAP panels\n")
cat("\n  Files:\n")
cat("    •", paste0(FILE_PREFIX, "_umap_panel1.{pdf,svg} (clusters + states + flagged)"), "\n")
cat("    •", paste0(FILE_PREFIX, "_umap_qc.{pdf,svg} (QC metrics)"), "\n")
cat("    •", paste0(FILE_PREFIX, "_umap_panel3.{pdf,svg} (donor + study group + senescence)"), "\n")
cat("    •", paste0(FILE_PREFIX, "_umap_cellcycle.{pdf,svg} (cell cycle)"), "\n")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# MULTI-RESOLUTION UMAP PANEL
# ════════════════════════════════════════════════════════════════════════════════

cat("\n", paste(rep("=", 80), collapse = ""), "\n")
cat("MULTI-RESOLUTION UMAPs\n")
cat(paste(rep("=", 80), collapse = ""), "\n")

res_to_plot <- c(0.1, 0.2, 0.4, 0.6, 0.8, 1.0, 1.2)

cat("  Resolutions:", paste(res_to_plot, collapse = ", "), "\n")
cat("  Current resolution:", CLUSTERING_RESOLUTION, "\n")

# ─────────────────────────────────────────────────────────────────────────────────
# Generate one UMAP per resolution
# ─────────────────────────────────────────────────────────────────────────────────

umap_list <- lapply(res_to_plot, function(res) {
    col_name <- paste0("RNA_snn_res.", res)
    n_cl <- length(unique(seurat_obj@meta.data[[col_name]]))
    
    title_label <- sprintf("res = %.1f (%d clusters)", res, n_cl)
    if (res == CLUSTERING_RESOLUTION) title_label <- paste0(title_label, " *")
    
    p <- DimPlot(seurat_obj, group.by = col_name, reduction = UMAP_RED,
            label = TRUE, label.size = 2.5, pt.size = 0.05, repel = TRUE) +
        ggtitle(title_label) +
        umap_theme_noleg +
        theme(plot.title = element_text(size = 10, face = "bold",
                                         color = ifelse(res == CLUSTERING_RESOLUTION,
                                                        "#E15759", "black")))
    add_corner_arrows(p, label_size = 2)
})

# ─────────────────────────────────────────────────────────────────────────────────
# Add state reference UMAP
# ─────────────────────────────────────────────────────────────────────────────────

p_state_ref <- DimPlot(seurat_obj, group.by = SUBCLUSTER_COL, reduction = UMAP_RED,
                        cols = state_colors, pt.size = 0.05) +
    ggtitle("Annotated States (reference)") +
    umap_theme +
    theme(plot.title = element_text(size = 10, face = "bold", color = "#4E79A7"),
          legend.text = element_text(size = 8),
          legend.key.size = unit(0.3, "cm"))
p_state_ref <- add_corner_arrows(p_state_ref, label_size = 2)

umap_list <- c(umap_list, list(p_state_ref))

# ─────────────────────────────────────────────────────────────────────────────────
# Combine
# ─────────────────────────────────────────────────────────────────────────────────

n_plots <- length(umap_list)
n_cols <- 4
n_rows <- ceiling(n_plots / n_cols)

options(repr.plot.width = 16, repr.plot.height = 4.5 * n_rows)

p_multi_res <- cowplot::plot_grid(plotlist = umap_list, ncol = n_cols)

# Add title
title_grob <- cowplot::ggdraw() +
    cowplot::draw_label(
        paste0(toupper(cell_type_label), ": Clustering Across Resolutions (* = current)"),
        size = 14, fontface = "bold"
    )

p_multi_res <- cowplot::plot_grid(title_grob, p_multi_res, ncol = 1, rel_heights = c(0.05, 1))

print(p_multi_res)

ggsave(file.path(VAL_FIGURES_DIR, paste0(FILE_PREFIX, "_umap_multi_resolution.pdf")),
       p_multi_res, width = 16, height = 4.5 * n_rows)
ggsave(file.path(VAL_FIGURES_DIR, paste0(FILE_PREFIX, "_umap_multi_resolution.svg")),
       p_multi_res, width = 16, height = 4.5 * n_rows)

cat("\n  ✓ Saved:", paste0(FILE_PREFIX, "_umap_multi_resolution.{pdf,svg}"), "\n")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# MULTI-RESOLUTION UMAP PANEL (Clusters + Majority-State side by side)
# ════════════════════════════════════════════════════════════════════════════════

cat("\n", paste(rep("=", 80), collapse = ""), "\n")
cat("MULTI-RESOLUTION UMAPs\n")
cat(paste(rep("=", 80), collapse = ""), "\n")

# Auto-detect available resolutions
available_res <- grep("RNA_snn_res\\.", colnames(seurat_obj@meta.data), value = TRUE)
res_to_plot <- sort(as.numeric(gsub("RNA_snn_res\\.", "", available_res)))

cat("  Available resolutions:", paste(res_to_plot, collapse = ", "), "\n")
cat("  Current resolution:", CLUSTERING_RESOLUTION, "\n")

# ─────────────────────────────────────────────────────────────────────────────────
# Generate paired UMAPs per resolution
# ─────────────────────────────────────────────────────────────────────────────────

umap_list <- list()
state_summary_list <- list()

for (res in res_to_plot) {
    col_name <- paste0("RNA_snn_res.", res)
    n_cl <- length(unique(seurat_obj@meta.data[[col_name]]))
    is_current <- (res == CLUSTERING_RESOLUTION)
    title_color <- ifelse(is_current, "#E15759", "black")
    
    # --- Compute majority state per cluster at this resolution ---
    majority_col <- paste0("majority_state_res_", gsub("\\.", "_", as.character(res)))
    
    cluster_to_state <- seurat_obj@meta.data %>%
        group_by(across(all_of(col_name)), across(all_of(SUBCLUSTER_COL))) %>%
        summarise(n = n(), .groups = "drop") %>%
        group_by(across(all_of(col_name))) %>%
        slice_max(n, n = 1, with_ties = FALSE) %>%
        ungroup()
    
    # Build lookup: cluster -> majority state
    state_lookup <- setNames(cluster_to_state[[SUBCLUSTER_COL]], 
                              as.character(cluster_to_state[[col_name]]))
    
    # Assign majority state to each cell
    seurat_obj@meta.data[[majority_col]] <- state_lookup[as.character(seurat_obj@meta.data[[col_name]])]
    
    # Count unique states at this resolution
    n_states <- length(unique(seurat_obj@meta.data[[majority_col]]))
    
    # Track for summary table
    state_summary_list[[as.character(res)]] <- data.frame(
        Resolution = res,
        n_clusters = n_cl,
        n_states = n_states,
        states = paste(sort(unique(seurat_obj@meta.data[[majority_col]])), collapse = ", ")
    )
    
    # --- Cluster UMAP ---
    cluster_label <- sprintf("res = %.1f (%d clusters)", res, n_cl)
    if (is_current) cluster_label <- paste0(cluster_label, " *")
    
    p_cl <- DimPlot(seurat_obj, group.by = col_name, reduction = UMAP_RED,
                     label = TRUE, label.size = 2.5, pt.size = 0.05, repel = TRUE) +
        ggtitle(cluster_label) +
        umap_theme_noleg +
        theme(plot.title = element_text(size = 10, face = "bold", color = title_color))
    p_cl <- add_corner_arrows(p_cl, label_size = 2)
    
    # --- Majority State UMAP ---
    p_st <- DimPlot(seurat_obj, group.by = majority_col, reduction = UMAP_RED,
                     cols = state_colors, pt.size = 0.05) +
        ggtitle(sprintf("Majority states (%d)", n_states)) +
        umap_theme_noleg +
        theme(plot.title = element_text(size = 10, face = "bold", color = title_color))
    p_st <- add_corner_arrows(p_st, label_size = 2)
    
    umap_list <- c(umap_list, list(p_cl, p_st))
}

# ─────────────────────────────────────────────────────────────────────────────────
# Print summary table
# ─────────────────────────────────────────────────────────────────────────────────

state_summary <- do.call(rbind, state_summary_list)
cat("\n--- State counts per resolution ---\n")
print(state_summary[, c("Resolution", "n_clusters", "n_states")], row.names = FALSE)

write.csv(state_summary, 
          file.path(VAL_RESULTS_DIR, paste0(FILE_PREFIX, "_multires_state_summary.csv")),
          row.names = FALSE)

# ─────────────────────────────────────────────────────────────────────────────────
# Shared state legend
# ─────────────────────────────────────────────────────────────────────────────────

p_legend_src <- DimPlot(seurat_obj, group.by = SUBCLUSTER_COL, reduction = UMAP_RED,
                         cols = state_colors, pt.size = 0.5) +
    theme_void() +
    theme(
        legend.position = "right",
        legend.title = element_text(size = 10, face = "bold"),
        legend.text = element_text(size = 8),
        legend.key.size = unit(0.4, "cm")
    ) +
    labs(color = paste0(tools::toTitleCase(cell_type_label), " State")) +
    guides(color = guide_legend(ncol = 1, override.aes = list(size = 3)))

legend_grob <- cowplot::get_legend(p_legend_src)

# ─────────────────────────────────────────────────────────────────────────────────
# Combine
# ─────────────────────────────────────────────────────────────────────────────────

n_cols <- 4  # 2 pairs per row
n_rows <- ceiling(length(umap_list) / n_cols)

options(repr.plot.width = 18, repr.plot.height = 4.5 * n_rows)

umap_grid <- cowplot::plot_grid(plotlist = umap_list, ncol = n_cols)

umap_with_legend <- cowplot::plot_grid(umap_grid, legend_grob, 
                                        ncol = 2, rel_widths = c(1, 0.15))

title_grob <- cowplot::ggdraw() +
    cowplot::draw_label(
        paste0(toupper(cell_type_label), 
               ": Clustering & Majority States Across Resolutions (* = current)"),
        size = 14, fontface = "bold"
    )

p_multi_res <- cowplot::plot_grid(title_grob, umap_with_legend, 
                                   ncol = 1, rel_heights = c(0.05, 1))

print(p_multi_res)

ggsave(file.path(VAL_FIGURES_DIR, paste0(FILE_PREFIX, "_umap_multi_resolution.pdf")),
       p_multi_res, width = 18, height = 4.5 * n_rows)
ggsave(file.path(VAL_FIGURES_DIR, paste0(FILE_PREFIX, "_umap_multi_resolution.svg")),
       p_multi_res, width = 18, height = 4.5 * n_rows)

cat("\n  ✓ Saved:", paste0(FILE_PREFIX, "_umap_multi_resolution.{pdf,svg}"), "\n")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# OVERALL VALIDATION SUMMARY
# ════════════════════════════════════════════════════════════════════════════════

mt_threshold <- 0.15
MT_REMOVE_THRESHOLD <- 0.25

# ── Sanity checks: make sure all variables are from the same run ──
cat("\n▸ Pre-summary sanity checks...\n")

# Verify cluster_order matches seurat_clusters
actual_clusters <- as.character(sort(as.numeric(as.character(unique(seurat_obj$seurat_clusters)))))
if (!identical(cluster_order, actual_clusters)) {
    cat("  ⚠ cluster_order mismatch — rebuilding from seurat_obj\n")
    cluster_order <- actual_clusters
    n_clusters <- length(cluster_order)
}

# Verify qc_by_cluster matches
if (exists("qc_by_cluster") && !all(cluster_order %in% qc_by_cluster$Cluster)) {
    missing <- cluster_order[!cluster_order %in% qc_by_cluster$Cluster]
    cat("  ⚠ qc_by_cluster missing clusters:", paste(missing, collapse = ", "), "\n")
    cat("  → Rerun Step 1 before Summary\n")
}

# Verify cluster_comp_summary matches
if (exists("cluster_comp_summary") && !all(cluster_order %in% cluster_comp_summary$Cluster)) {
    missing <- cluster_order[!cluster_order %in% cluster_comp_summary$Cluster]
    cat("  ⚠ cluster_comp_summary missing clusters:", paste(missing, collapse = ", "), "\n")
    cat("  → Rerun Step 3 before Summary\n")
}

# Verify flagged_clusters are valid
if (exists("flagged_clusters") && length(flagged_clusters) > 0) {
    invalid <- flagged_clusters[!flagged_clusters %in% cluster_order]
    if (length(invalid) > 0) {
        cat("  ⚠ flagged_clusters contains invalid clusters:", paste(invalid, collapse = ", "), "\n")
        cat("  → Filtering to valid clusters only\n")
        flagged_clusters <- flagged_clusters[flagged_clusters %in% cluster_order]
    }
}

# Ensure cluster_purity exists
if (!exists("cluster_purity") || length(cluster_purity) == 0) {
    cluster_purity <- tapply(
        seurat_obj@meta.data[[SUBCLUSTER_COL]],
        seurat_obj$seurat_clusters,
        function(x) max(table(x)) / length(x) * 100
    )
}

cat("  ✓ Using", n_clusters, "clusters:", paste(cluster_order, collapse = ", "), "\n")

# ════════════════════════════════════════════════════════════════════════════════

cat("\n", paste(rep("=", 80), collapse = ""), "\n")
cat(sprintf("VALIDATION SUMMARY: %s (%s)\n", toupper(cell_type_label), DATASET))
cat(paste(rep("=", 80), collapse = ""), "\n")

# ── Object ──
cat("\n-- Object --\n")
cat("  Cells:", ncol(seurat_obj), "\n")
cat("  Clusters:", n_clusters, "\n")
cat("  States:", n_states, "(", paste(state_order, collapse = ", "), ")\n")
cat("  Donors:", n_donors, "\n")

# ── Step 1: QC Metrics ──
cat("\n-- Step 1: QC Metrics --\n")
cat("  Goal: Identify clusters driven by low-quality cells (high MT%, low genes/UMI)\n")
cat("  Rationale: Clusters enriched for damaged or dying cells produce spurious\n")
cat("  transcriptional signatures. For postmortem brain snRNA-seq, MT% is generally\n")
cat("  higher than other tissues due to nuclear isolation; moderate elevations are\n")
cat("  expected but extreme values (>0.3) still indicate damaged nuclei.\n")
high_mt <- qc_by_cluster[qc_by_cluster$median_mt > mt_threshold, ]
if (nrow(high_mt) > 0) {
    cat("  Result:", nrow(high_mt), "cluster(s) with elevated MT (>", mt_threshold, "):\n")
    for (i in 1:nrow(high_mt)) {
        severity <- ifelse(high_mt$median_mt[i] > 0.3, "SEVERE", 
                    ifelse(high_mt$median_mt[i] > 0.2, "MODERATE", "MILD"))
        cat(sprintf("    Cluster %s (%s): median MT = %.3f [%s], UMI = %s, genes = %s\n",
                    high_mt$Cluster[i], high_mt$State[i], high_mt$median_mt[i], severity,
                    format(high_mt$median_UMI[i], big.mark = ","),
                    format(high_mt$median_genes[i], big.mark = ",")))
    }
} else {
    cat("  Result: PASS -- No clusters with elevated MT (>", mt_threshold, ")\n")
}

# ── Step 2: Canonical Markers ──
cat("\n-- Step 2: Canonical Markers --\n")
cat("  Goal: Confirm cell type identity and detect contamination from other lineages\n")
cat("  Rationale: Every cluster should express known", cell_type_label, "markers.\n")
cat("  High expression of markers from other cell types indicates doublets or\n")
cat("  mis-classified cells.\n")
low_identity <- flag_df[flag_df$mean_identity < 0.5, ]
high_contam <- flag_df[flag_df$mean_contam > 0.5, ]
if (nrow(low_identity) > 0) {
    cat("  Identity: FAIL -- Low identity clusters:\n")
    for (i in 1:nrow(low_identity)) {
        cat(sprintf("    Cluster %s (%s): mean identity = %.2f\n",
                    low_identity$Cluster[i], low_identity$State[i], low_identity$mean_identity[i]))
    }
} else {
    cat("  Identity: PASS -- All clusters express", cell_type_label, "markers\n")
}
if (nrow(high_contam) > 0) {
    cat("  Contamination: FAIL -- High contamination clusters:\n")
    for (i in 1:nrow(high_contam)) {
        cat(sprintf("    Cluster %s (%s): mean contam = %.2f, top negative = %s (%.2f)\n",
                    high_contam$Cluster[i], high_contam$State[i], high_contam$mean_contam[i],
                    high_contam$max_neg_gene[i], high_contam$max_neg_val[i]))
    }
} else {
    cat("  Contamination: PASS -- No clusters with high off-target signal\n")
}

# ── Step 3: Sample Composition ──
cat("\n-- Step 3: Sample Composition --\n")
cat("  Goal: Detect clusters dominated by a single donor (batch/sample artifacts)\n")
cat("  Rationale: Clusters driven by one donor reflect individual-specific effects,\n")
cat("  not generalizable biology. If a state is defined primarily by such clusters,\n")
cat("  the state itself may be a donor artifact.\n")
if (length(flagged_clusters) > 0) {
    cat("  Flagged clusters:", paste(flagged_clusters, collapse = ", "), "\n")
    for (cl in flagged_clusters) {
        cl_row <- cluster_comp_summary[cluster_comp_summary$Cluster == cl, ]
        if (nrow(cl_row) > 0) {
            cat(sprintf("    Cluster %s (%s, n=%d): top donor = %.1f%%, evenness = %.2f\n",
                        cl, cl_row$State, cl_row$N_cells, cl_row$Top1_pct, cl_row$Evenness))
        }
    }
} else {
    cat("  Result: PASS -- No clusters with single-donor dominance\n")
}

# State robustness
if (exists("state_robustness") && length(flagged_clusters) > 0) {
    at_risk <- state_robustness[state_robustness$Donors_clean < 5 | 
                                 state_robustness$Pct_retained < 50 |
                                 state_robustness$Evenness < 0.5, ]
    if (nrow(at_risk) > 0) {
        cat("  State robustness: WARNING\n")
        for (i in 1:nrow(at_risk)) {
            reasons <- c()
            if (at_risk$Donors_clean[i] < 5) reasons <- c(reasons, 
                sprintf("%d donors remain", at_risk$Donors_clean[i]))
            if (at_risk$Pct_retained[i] < 50) reasons <- c(reasons, 
                sprintf("%.1f%% cells retained", at_risk$Pct_retained[i]))
            if (at_risk$Evenness[i] < 0.5) reasons <- c(reasons, 
                sprintf("evenness %.2f", at_risk$Evenness[i]))
            cat(sprintf("    %s: %s\n", at_risk$State[i], paste(reasons, collapse = ", ")))
        }
    } else {
        cat("  State robustness: PASS -- All states retain multi-donor support\n")
    }
}

# State resolution stability
if (exists("state_by_res") && nrow(state_by_res) > 0) {
    cat("  State resolution stability:\n")
    for (st in state_order) {
        st_data <- state_by_res[state_by_res$State == st, ]
        if (nrow(st_data) == 0) next
        min_rec <- min(st_data$Recovery_pct)
        low_recovery <- st_data[st_data$Recovery_pct < 50, ]
        
        if (nrow(low_recovery) > 0) {
            low_res <- low_recovery$Resolution
            chosen_rec <- st_data$Recovery_pct[st_data$Resolution == CLUSTERING_RESOLUTION]
            if (length(chosen_rec) == 0) chosen_rec <- NA
            
            all_below_chosen <- all(low_res < CLUSTERING_RESOLUTION)
            at_and_above <- st_data[st_data$Resolution >= CLUSTERING_RESOLUTION, ]
            stable_above <- all(at_and_above$Recovery_pct >= 70)
            
            if (all_below_chosen && stable_above && !is.na(chosen_rec) && chosen_rec >= 70) {
                first_stable <- min(st_data$Resolution[st_data$Recovery_pct >= 50])
                cat(sprintf("    %s: stable at working resolutions (emerges at res %.1f, min %.1f%% at coarse res)\n",
                            st, first_stable, min_rec))
            } else if (!is.na(chosen_rec) && chosen_rec >= 70) {
                cat(sprintf("    %s: transitional -- <50%% at res %s, but %.1f%% at chosen res\n",
                            st, paste(low_res, collapse = ", "), chosen_rec))
            } else {
                cat(sprintf("    %s: UNSTABLE -- %.1f%% at chosen res, <50%% at res %s\n",
                            st, ifelse(is.na(chosen_rec), NA, chosen_rec),
                            paste(low_res, collapse = ", ")))
            }
        } else {
            cat(sprintf("    %s: stable across all resolutions (min recovery: %.1f%%)\n",
                        st, min_rec))
        }
    }
}

# ── Step 4: Cell Cycle ──
cat("\n-- Step 4: Cell Cycle --\n")
cat("  Goal: Identify clusters driven by cell cycle state rather than biology\n")
cat("  Rationale: If a cluster is enriched for S/G2M phase relative to others, its\n")
cat("  transcriptional identity may reflect proliferation rather than a distinct\n")
cat("  cell state. Uniform cycling across clusters is not concerning. Note: snRNA-seq\n")
cat("  from brain tissue often shows high overall cycling scores because nuclear RNA\n")
cat("  captures cell cycle transcripts even in post-mitotic cells.\n")
overall_cycling <- round(sum(seurat_obj$Phase %in% c("S", "G2M")) / ncol(seurat_obj) * 100, 1)
cat("  Overall cycling (S+G2M):", overall_cycling, "%\n")
if (flagged_cc) {
    n_flagged_cc <- sum(phase_wide$Cycling > 30)
    cycling_range <- range(phase_wide$Cycling)
    cat("  Clusters >30% cycling:", n_flagged_cc, "/", nrow(phase_wide), "\n")
    cat("  Cycling range:", cycling_range[1], "-", cycling_range[2], "%\n")
    if ((cycling_range[2] - cycling_range[1]) < 20) {
        cat("  Result: PASS -- Uniformly distributed, not a cluster-specific artifact\n")
    } else {
        cat("  Result: WARNING -- Variable across clusters:\n")
        top3 <- phase_wide[order(-phase_wide$Cycling), ][1:min(3, nrow(phase_wide)), ]
        for (i in 1:nrow(top3)) {
            cat(sprintf("    Cluster %s (%s): %.1f%%\n",
                        top3$Cluster[i], top3$State[i], top3$Cycling[i]))
        }
    }
} else {
    cat("  Result: PASS -- No clusters with >30% cycling\n")
}

# ── Step 5: Cluster Stability ──
cat("\n-- Step 5: Cluster Stability --\n")
cat("  Goal: Assess whether clusters and states are robust across resolutions\n")
cat("  Rationale: Stable clusters appear as clean, single-path branches in the\n")
cat("  clustree (cell flow across resolutions). Unstable clusters show messy splits\n")
cat("  where cells scatter to multiple destinations, indicating over-clustering.\n")
cat("  State purity measures how well each cluster maps to a single annotated state;\n")
cat("  low purity suggests boundary clusters between two states.\n")
cat("  Resolution:", CLUSTERING_RESOLUTION, "->", n_clusters, "clusters\n")

current_res_row <- if (exists("stability_df")) {
    stability_df[stability_df$Resolution == CLUSTERING_RESOLUTION, ]
} else {
    data.frame()
}

if (nrow(current_res_row) > 0) {
    cat("  Mean state purity:", current_res_row$Mean_purity, "%\n")
    cat("  Min state purity:", current_res_row$Min_purity, "%\n")
    cat("  Mixed clusters (<70% purity):", current_res_row$N_mixed, "\n")
} else {
    # Compute purity against original seurat_clusters
    orig_purity <- tapply(
        seurat_obj@meta.data[[SUBCLUSTER_COL]],
        seurat_obj$seurat_clusters,
        function(x) max(table(x)) / length(x) * 100
    )
    cat("  Mean state purity:", round(mean(orig_purity), 1), "%\n")
    cat("  Min state purity:", round(min(orig_purity), 1), "%\n")
    cat("  Mixed clusters (<70% purity):", sum(orig_purity < 70), "\n")
}

purity_sorted <- sort(cluster_purity)
low_purity <- purity_sorted[purity_sorted < 70]
if (length(low_purity) > 0) {
    cat("  Low purity clusters:\n")
    for (cl in names(low_purity)) {
        cl_cells <- as.character(seurat_obj$seurat_clusters) == cl
        if (sum(cl_cells) == 0) next
        cl_state_table <- sort(table(seurat_obj@meta.data[[SUBCLUSTER_COL]][cl_cells]), decreasing = TRUE)
        cl_state_pcts <- round(cl_state_table / sum(cl_state_table) * 100, 1)
        top2 <- paste(sprintf("%s (%.1f%%)", names(cl_state_pcts)[1:min(2, length(cl_state_pcts))],
                              cl_state_pcts[1:min(2, length(cl_state_pcts))]), collapse = " / ")
        cat(sprintf("    Cluster %s: %.1f%% purity -- %s\n", cl, low_purity[cl], top2))
    }
    cat("  Note: Low purity clusters are typically transition zones between states\n")
    cat("  and do not require action unless they distort downstream results.\n")
} else {
    cat("  All clusters >70% purity\n")
}

if (exists("stability_df")) {
    stable_range <- stability_df[stability_df$Mean_purity > 85, ]
    if (nrow(stable_range) > 0) {
        cat("  Stable resolution range (>85% mean purity): ",
            min(stable_range$Resolution), " - ", max(stable_range$Resolution), "\n", sep = "")
    }
}

# ── Actionable Findings ──
cat("\n", paste(rep("-", 80), collapse = ""), "\n")
cat("ACTIONABLE FINDINGS\n")
cat(paste(rep("-", 80), collapse = ""), "\n")

remove_list <- c()
monitor_list <- c()

for (cl in flagged_clusters) {
    mt_row <- which(qc_by_cluster$Cluster == cl)
    mt_val <- if (length(mt_row) > 0) qc_by_cluster$median_mt[mt_row] else NA
    
    purity_val <- if (cl %in% names(cluster_purity)) cluster_purity[cl] else NA
    
    comp_row <- which(cluster_comp_summary$Cluster == cl)
    top_donor <- if (length(comp_row) > 0) cluster_comp_summary$Top1_pct[comp_row] else NA
    
    is_severe_mt <- !is.na(mt_val) && mt_val > MT_REMOVE_THRESHOLD
    is_low_purity <- !is.na(purity_val) && purity_val < 60
    is_donor_dominated <- !is.na(top_donor) && top_donor > 80
    is_elevated_mt <- !is.na(mt_val) && mt_val > mt_threshold
    
    if (is_severe_mt || (is_donor_dominated && (is_elevated_mt || is_low_purity))) {
        remove_list <- c(remove_list, cl)
    } else {
        monitor_list <- c(monitor_list, cl)
    }
}

if (length(remove_list) > 0) {
    cat("\n  REMOVE:\n")
    for (cl in remove_list) {
        mt_row <- which(qc_by_cluster$Cluster == cl)
        comp_row <- which(cluster_comp_summary$Cluster == cl)
        cl_state <- if (length(mt_row) > 0) qc_by_cluster$State[mt_row] else "unknown"
        cl_n <- if (length(mt_row) > 0) qc_by_cluster$N[mt_row] else NA
        cl_mt <- if (length(mt_row) > 0) qc_by_cluster$median_mt[mt_row] else NA
        cl_top <- if (length(comp_row) > 0) cluster_comp_summary$Top1_pct[comp_row] else NA
        
        reasons <- c()
        if (!is.na(cl_mt) && cl_mt > MT_REMOVE_THRESHOLD) {
            reasons <- c(reasons, sprintf("severe MT (%.3f)", cl_mt))
        }
        if (!is.na(cl_top) && cl_top > 80) {
            reasons <- c(reasons, sprintf("donor dominated (%.1f%%)", cl_top))
        }
        if (length(reasons) == 0) reasons <- "multiple flags"
        cat(sprintf("    Cluster %s (%s, n=%s): %s\n",
                    cl, cl_state, ifelse(is.na(cl_n), "?", cl_n),
                    paste(reasons, collapse = " + ")))
    }
} else {
    cat("\n  REMOVE: None\n")
}

if (length(monitor_list) > 0) {
    cat("\n  MONITOR:\n")
    for (cl in monitor_list) {
        mt_row <- which(qc_by_cluster$Cluster == cl)
        comp_row <- which(cluster_comp_summary$Cluster == cl)
        cl_state <- if (length(mt_row) > 0) qc_by_cluster$State[mt_row] else "unknown"
        cl_n <- if (length(mt_row) > 0) qc_by_cluster$N[mt_row] else NA
        cl_top <- if (length(comp_row) > 0) cluster_comp_summary$Top1_pct[comp_row] else NA
        cl_eve <- if (length(comp_row) > 0) cluster_comp_summary$Evenness[comp_row] else NA
        
        reasons <- c()
        if (!is.na(cl_top) && cl_top > DONOR_DOMINANCE_THRESHOLD) {
            reasons <- c(reasons, sprintf("top donor %.1f%%", cl_top))
        }
        if (!is.na(cl_eve) && cl_eve < EVENNESS_FLAG_THRESHOLD) {
            reasons <- c(reasons, sprintf("evenness %.2f", cl_eve))
        }
        if (length(reasons) == 0) reasons <- "flagged in composition"
        cat(sprintf("    Cluster %s (%s, n=%s): %s\n",
                    cl, cl_state, ifelse(is.na(cl_n), "?", cl_n),
                    paste(reasons, collapse = ", ")))
    }
    cat("  Action: verify DEG results from these clusters are not single-donor driven\n")
} else {
    cat("\n  MONITOR: None\n")
}

n_pass <- n_clusters - length(remove_list) - length(monitor_list)
cat("\n  PASS:", n_pass, "clusters\n")

# ── Recommendation ──
cat("\n", paste(rep("-", 80), collapse = ""), "\n")
cat("RECOMMENDATION\n")
cat(paste(rep("-", 80), collapse = ""), "\n")
rec_num <- 1
if (length(remove_list) > 0) {
    affected_states <- c()
    for (cl in remove_list) {
        mt_row <- which(qc_by_cluster$Cluster == cl)
        if (length(mt_row) > 0) affected_states <- c(affected_states, qc_by_cluster$State[mt_row])
    }
    affected_states <- unique(affected_states)
    cat(sprintf("  %d. Remove cluster(s) %s and re-evaluate %s state annotation\n",
                rec_num, paste(remove_list, collapse = ", "),
                paste(affected_states, collapse = ", ")))
    rec_num <- rec_num + 1
}
if (length(monitor_list) > 0) {
    cat(sprintf("  %d. Verify DEG results for cluster(s) %s are not single-donor driven\n",
                rec_num, paste(monitor_list, collapse = ", ")))
    rec_num <- rec_num + 1
}
if (nrow(current_res_row) > 0) {
    cat(sprintf("  %d. Resolution %s is appropriate (%d clusters, mean purity %.1f%%)\n",
                rec_num, CLUSTERING_RESOLUTION, n_clusters, current_res_row$Mean_purity))
} else {
    cat(sprintf("  %d. Original clustering (%d clusters) used for annotation\n",
                rec_num, n_clusters))
}
rec_num <- rec_num + 1
cat(sprintf("  %d. All %d annotated states are biologically supported\n", rec_num, n_states))

cat("\n", paste(rep("=", 80), collapse = ""), "\n")
cat("VALIDATION COMPLETE\n")
cat(paste(rep("=", 80), collapse = ""), "\n")

# ─────────────────────────────────────────────────────────────────────────────────
# List all validation outputs
# ─────────────────────────────────────────────────────────────────────────────────

cat("\n-- All Validation Files --\n")

cat("\n  Results (", VAL_RESULTS_DIR, "):\n", sep = "")
val_csv <- list.files(VAL_RESULTS_DIR, pattern = "\\.csv$", full.names = FALSE)
for (f in val_csv) cat("    -", f, "\n")

cat("\n  Figures (", VAL_FIGURES_DIR, "):\n", sep = "")
val_figs <- list.files(VAL_FIGURES_DIR, pattern = "\\.(pdf|svg)$", full.names = FALSE)
for (f in val_figs) cat("    -", f, "\n")

# Module 04G: Oligodendrocyte Subclustering & Annotation (R)

## Overview
Subcluster Oligodendrocytes into functional states using literature-derived gene signatures, then characterize senescence patterns across states and study groups.

## Module 04F: OL Subclustering & Annotation

### Annotation: Cluster-Level, Z-Scored
Same approach as astrocytes/microglia. Clusters scored via AddModuleScore, z-scored across clusters, assigned to highest z-scored state.

### 4 States (Human Consensus)

- **hOligo1** (MOL1/2-like): RASGRF1, RASGRF2, KLK6 — mature stable OLs, signaling/adhesion (Pandey 2022; Jäkel 2019)
- **hOligo2** (MOL5/6-like): PLXDC2, PALM2, OPALIN, QDPR — mature myelinating OLs, most abundant adult population (Pandey 2022; Jäkel 2019; Mathys 2019)
- **DA1 / Immune OL**: SERPINA3, C4B, B2M, HLA-A, HLA-B, HLA-C, CD74 — disease-associated, innate immune/complement activation (Kenigsbuch 2022; Pandey 2022; Falcão 2018; Jäkel 2019 ImOLG)
- **IFN OL**: IFIT1, IFIT3, OAS1, OAS2 — interferon-responsive OLs (Pandey 2022; Kaya 2022)

### DA2 Excluded
DA2 markers (CDKN1A, TP53, ATF3) overlap with senescence pathway genes. Including them as a substate would create circularity with downstream senescence scoring. Senescence analysis remains an independent overlay.

### Workflow
Subset OL → QC (doublet removal) → Harmony (Cohort) → UMAP → Cluster → AddModuleScore (4 signatures) → Z-score per cluster → Assign states → Validate (DotPlot) → Composition + %SnC per state → GLMM

In [ ]:
cat("\n================================================================================\n")
cat("04G: OL SUBCLUSTERING & ANNOTATION\n")
cat("================================================================================\n")

# ════════════════════════════════════════════════════════════════════════════════
# LOAD LIBRARIES
# ════════════════════════════════════════════════════════════════════════════════

suppressPackageStartupMessages({
    library(Seurat)
    library(Matrix)
    library(dplyr)
    library(tidyr)
    library(ggplot2)
    library(qs)
    library(pheatmap)
    library(patchwork)
    library(RColorBrewer)
})

cat("\n✓ Libraries loaded\n")

# ════════════════════════════════════════════════════════════════════════════════
# DATASET SELECTION
# ════════════════════════════════════════════════════════════════════════════════

DATASET <- "psychad_aging"  # Options: 'psychad_aging', 'psychad_ad', 'psychencode', 'mathys'

# ════════════════════════════════════════════════════════════════════════════════
# DATASET-SPECIFIC CONFIGURATION
# ════════════════════════════════════════════════════════════════════════════════

DATASET_CONFIG <- list(
    'psychad_aging' = list(
        cell_type_col = 'subclass',
        donor_col = 'Sample',
        study_group_col = 'Study_Group',
        sex_col = 'Sex',
        cohort_col = 'Cohort',
        study_type = 'aging',
        primary_var = 'Age',
        primary_var_type = 'continuous',
        covariates = c('Sex', 'Cohort'),
        group_order = c('Age_20_29', 'Age_30_39', 'Age_40_49', 'Age_50_59', 
                        'Age_60_69', 'Age_70_79', 'Age_80_100')
    ),
    'psychad_ad' = list(
        cell_type_col = 'subclass',
        donor_col = 'Sample',
        study_group_col = 'Study_Group',
        sex_col = 'Sex',
        cohort_col = 'Cohort',
        study_type = 'disease',
        primary_var = 'Study_Group',
        primary_var_type = 'categorical',
        reference_group = 'Control',
        covariates = c('Age', 'Sex', 'Cohort'),
        group_order = c('Control', 'MCI', 'AD')
    ),
    'psychencode' = list(
        cell_type_col = 'major_celltype',
        donor_col = 'sample_id',
        study_group_col = 'Study_Group',
        sex_col = 'Biological_Sex',
        cohort_col = 'Batch',
        study_type = 'aging',
        primary_var = 'Age_death',
        primary_var_type = 'continuous',
        covariates = c('Sex', 'Batch'),
        group_order = c('Age_20_29', 'Age_30_39', 'Age_40_49', 'Age_50_59', 
                        'Age_60_69', 'Age_70_79', 'Age_80_100')
    ),
    'mathys' = list(
        cell_type_col = 'broad.cell.type',
        donor_col = 'Subject',
        study_group_col = 'Study_Group',
        sex_col = 'sex',
        cohort_col = 'batch',
        study_type = 'disease',
        primary_var = 'Study_Group',
        primary_var_type = 'categorical',
        reference_group = 'NCI',
        covariates = c('Age', 'sex', 'batch'),
        group_order = c('NCI', 'MCI', 'AD')
    )
)

config <- DATASET_CONFIG[[DATASET]]

# ─────────────────────────────────────────────────────────────────────────────
# COLUMN NAMES
# ─────────────────────────────────────────────────────────────────────────────

CELL_TYPE_COL <- config$cell_type_col
DONOR_COL <- config$donor_col
STUDY_GROUP_COL <- config$study_group_col
SEX_COL <- config$sex_col
COHORT_COL <- config$cohort_col
SENESCENCE_LABEL_COL <- "senescence_label"

# ─────────────────────────────────────────────────────────────────────────────
# STUDY DESIGN PARAMETERS
# ─────────────────────────────────────────────────────────────────────────────

STUDY_TYPE <- config$study_type
PRIMARY_VAR <- config$primary_var
PRIMARY_VAR_TYPE <- config$primary_var_type
COVARIATES <- config$covariates
GROUP_ORDER <- config$group_order

# Processing parameters
N_VARIABLE_FEATURES <- 2000
N_PCS <- 50
N_DIMS_USE <- 30
CLUSTERING_RESOLUTION <- 0.7

# ════════════════════════════════════════════════════════════════════════════════
# OL SIGNATURES (Pandey 2022; Kenigsbuch 2022; Jäkel 2019; Kaya 2022)
# ════════════════════════════════════════════════════════════════════════════════

OL_SIGNATURES <- list(
    hOligo1     = c("RASGRF1", "RASGRF2", "KLK6"),
    hOligo2     = c("PLXDC2", "PALM2", "OPALIN", "QDPR"),
    DA1_Immune  = c("SERPINA3", "C4B", "B2M", "HLA-A", "HLA-B", "HLA-C", "CD74"),
    IFN         = c("IFIT1", "IFIT3", "OAS1", "OAS2")
)

# ════════════════════════════════════════════════════════════════════════════════
# COLOR PALETTES
# ════════════════════════════════════════════════════════════════════════════════

STUDY_GROUP_COLORS <- c(
    'Age_20_29' = '#2E86AB',
    'Age_30_39' = '#4A90E2',
    'Age_40_49' = '#50C878',
    'Age_50_59' = '#FFB347',
    'Age_60_69' = '#FF8C00',
    'Age_70_79' = '#E24A4A',
    'Age_80_100' = '#8B0000',
    'Control' = '#4E79A7',
    'MCI' = '#F28E2B',
    'AD' = '#E15759',
    'NCI' = '#4E79A7'
)

study_colors <- STUDY_GROUP_COLORS[names(STUDY_GROUP_COLORS) %in% GROUP_ORDER]

SENESCENCE_COLORS <- c('Non-SnC' = '#D3D3D3', 'SnC' = '#C44E52')

SEX_COLORS <- c('Male' = '#4878CF', 'Female' = '#E97B8A')

# Base state palette (same colors for all cell types, assigned by position)
BASE_STATE_PALETTE <- c("#4E79A7", "#59A14F", "#E15759", "#F28E2B", "#EDC948",
                        "#76B7B2", "#FFBE7D", "#BAB0AC", "#B07AA1", "#FF9DA7",
                        "#A0CBE8", "#D37295", "#9C755F", "#8B0000")

get_state_colors <- function(states) {
    setNames(BASE_STATE_PALETTE[1:length(states)], states)
}

# ════════════════════════════════════════════════════════════════════════════════
# PATHS
# ════════════════════════════════════════════════════════════════════════════════

BASE_DIR <- "/fs/scratch/PAS2598/senescence_analysis"

INPUT_FILE  <- file.path(BASE_DIR, "data", "04_subsetting", DATASET, paste0(DATASET, "_seurat.qs"))
OUTPUT_DIR  <- file.path(BASE_DIR, "data", "04_ol", DATASET)
RESULTS_DIR <- file.path(BASE_DIR, "results", "04_ol", DATASET)
FIGURES_DIR <- file.path(BASE_DIR, "figures", "04_ol", DATASET)

dir.create(OUTPUT_DIR, recursive = TRUE, showWarnings = FALSE)
dir.create(RESULTS_DIR, recursive = TRUE, showWarnings = FALSE)
dir.create(FIGURES_DIR, recursive = TRUE, showWarnings = FALSE)

# ════════════════════════════════════════════════════════════════════════════════
# PRINT CONFIGURATION
# ════════════════════════════════════════════════════════════════════════════════

cat("\nConfiguration:\n")
cat("  Dataset:", DATASET, "\n")
cat("  Cell type column:", CELL_TYPE_COL, "\n")
cat("  Donor column:", DONOR_COL, "\n")
cat("  Study group column:", STUDY_GROUP_COL, "\n")
cat("  Sex column:", SEX_COL, "\n")
cat("  Cohort column:", COHORT_COL, "\n")
cat("  Study type:", STUDY_TYPE, "\n")
cat("  Primary variable:", PRIMARY_VAR, "(", PRIMARY_VAR_TYPE, ")\n")
cat("  Covariates:", paste(COVARIATES, collapse = ", "), "\n")
cat("  Group order:", paste(GROUP_ORDER, collapse = ", "), "\n")
cat("  Clustering resolution:", CLUSTERING_RESOLUTION, "\n")

cat("\nOL Signatures:\n")
for (sig in names(OL_SIGNATURES)) {
    cat(sprintf("  %s: %s\n", sig, paste(OL_SIGNATURES[[sig]], collapse = ", ")))
}

cat("\nPaths:\n")
cat("  Input:", INPUT_FILE, "\n")
cat("  Output:", OUTPUT_DIR, "\n")
cat("  Results:", RESULTS_DIR, "\n")
cat("  Figures:", FIGURES_DIR, "\n")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# SUBSET OLs FROM SHARED SEURAT OBJECT
# ════════════════════════════════════════════════════════════════════════════════

cat("\n================================================================================\n")
cat("SUBSETTING OLIGODENDROCYTES\n")
cat("================================================================================\n")

data <- qread(INPUT_FILE)
cat("Loaded:", ncol(data), "cells\n")

cat("\nCell types available:\n")
print(table(data[[CELL_TYPE_COL]]))

ol <- subset(data, cells = colnames(data)[data[[CELL_TYPE_COL]] == "Oligodendrocyte"])
cat("\nOL subset:", ncol(ol), "cells\n")

rm(data); gc(verbose = FALSE)

# ════════════════════════════════════════════════════════════════════════════════
# STEP 1: QC FILTERING (POST-SUBSET)
# ════════════════════════════════════════════════════════════════════════════════

cat("\n================================================================================\n")
cat("STEP 1: QC FILTERING — OL SUBSET\n")
cat("================================================================================\n")

cat("\nBefore filtering:", ncol(ol), "cells\n")

# Remove low-quality cells
ol <- subset(ol,
             nFeature_RNA > 200 &
             nFeature_RNA < quantile(ol$nFeature_RNA, 0.99) &
             nCount_RNA > 500 &
             nCount_RNA < quantile(ol$nCount_RNA, 0.99))
cat("After quality filter:", ncol(ol), "cells\n")

# Remove high mito
if ("percent.mt" %in% colnames(ol@meta.data)) {
    ol <- subset(ol, percent.mt < 10)
    cat("After mito filter:", ncol(ol), "cells\n")
}

# Remove doublets — cells expressing non-OL markers
cat("\nScoring contamination from other cell types...\n")

contam_signatures <- list(
    neuron = c("RBFOX3", "SYT1", "SNAP25", "STMN2"),
    astro  = c("GFAP", "AQP4", "GLUL", "SLC1A2"),
    micro  = c("CSF1R", "P2RY12", "CX3CR1", "TMEM119"),
    opc    = c("PDGFRA", "CSPG4", "VCAN", "GPR17"),
    endo   = c("CLDN5", "FLT1", "PECAM1", "VWF")
)

for (ct in names(contam_signatures)) {
    genes <- contam_signatures[[ct]][contam_signatures[[ct]] %in% rownames(ol)]
    if (length(genes) >= 2) {
        ol <- AddModuleScore(ol, features = list(genes), name = paste0(ct, "_contam"))
        cat(sprintf("  %s: %d/%d markers found\n", ct, length(genes), length(contam_signatures[[ct]])))
    }
}

contam_cols <- grep("_contam1$", colnames(ol@meta.data), value = TRUE)
if (length(contam_cols) > 0) {
    keep <- rep(TRUE, ncol(ol))
    for (col in contam_cols) {
        threshold <- quantile(ol@meta.data[[col]], 0.95)
        keep <- keep & (ol@meta.data[[col]] < threshold)
        cat(sprintf("  Removing top 5%% %s (threshold = %.3f)\n", col, threshold))
    }
    ol <- ol[, keep]
}

cat("\nAfter all QC:", ncol(ol), "cells\n")

# ════════════════════════════════════════════════════════════════════════════════
# STEP 2: REPROCESS
# ════════════════════════════════════════════════════════════════════════════════

library(harmony)

cat("\n================================================================================\n")
cat("STEP 2: REPROCESSING OLIGODENDROCYTES\n")
cat("================================================================================\n")

cat("\n1. Normalizing...\n")
ol <- NormalizeData(ol, verbose = FALSE)

cat("2. Finding variable features...\n")
ol <- FindVariableFeatures(ol, nfeatures = N_VARIABLE_FEATURES, verbose = FALSE)

cat("3. Scaling (regressing nCount_RNA)...\n")
ol <- ScaleData(ol,
                features = VariableFeatures(ol),
                vars.to.regress = "nCount_RNA",
                verbose = FALSE)

cat("4. Running PCA...\n")
ol <- RunPCA(ol, npcs = N_PCS, verbose = FALSE)

cat("5. Running Harmony (", COHORT_COL, ", theta=2)...\n")
ol <- RunHarmony(
    ol,
    group.by.vars = COHORT_COL,
    theta = c(2),
    max_iter = 30,
    verbose = TRUE
)

cat("6. Running t-SNE...\n")
ol <- RunTSNE(ol, reduction = "harmony", dims = 1:N_DIMS_USE,
              perplexity = 20, verbose = FALSE)

cat("7. Running UMAP...\n")
ol <- RunUMAP(ol, reduction = "harmony", dims = 1:N_DIMS_USE,
              verbose = FALSE)

cat("8. Finding neighbors...\n")
ol <- FindNeighbors(ol, reduction = "harmony", dims = 1:N_DIMS_USE,
                    verbose = FALSE)

cat("9. Clustering (resolution =", CLUSTERING_RESOLUTION, ")...\n")
ol <- FindClusters(ol, resolution = CLUSTERING_RESOLUTION, verbose = FALSE)

n_clusters <- length(unique(Idents(ol)))
cat("\nReprocessing complete:", ncol(ol), "cells,", n_clusters, "clusters\n")

cat("\nCells per cluster:\n")
cluster_sizes <- sort(table(Idents(ol)), decreasing = TRUE)
for (cl in names(cluster_sizes)) {
    cat(sprintf("  Cluster %s: %d cells (%.1f%%)\n",
                cl, cluster_sizes[cl], cluster_sizes[cl] / ncol(ol) * 100))
}

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# SCORE OL SIGNATURES
# ════════════════════════════════════════════════════════════════════════════════

cat("\n================================================================================\n")
cat("STEP 3: SCORING OL SIGNATURES\n")
cat("================================================================================\n")

OL_SIGNATURES <- list(
    hOligo1    = c("RASGRF1", "RASGRF2", "KLK6"),
    hOligo2    = c("PLXDC2", "PALM2", "OPALIN", "QDPR"),
    DA1_Immune = c("SERPINA3", "C4B", "B2M", "HLA-A", "HLA-B", "HLA-C", "CD74"),
    IFN        = c("IFIT1", "IFIT3", "OAS1", "OAS2")
)

cat("\nFiltering signatures against available genes...\n")

signatures_filtered <- lapply(OL_SIGNATURES, function(genes) {
    genes[genes %in% rownames(ol)]
})

for (sig in names(OL_SIGNATURES)) {
    found <- length(signatures_filtered[[sig]])
    total <- length(OL_SIGNATURES[[sig]])
    missing <- setdiff(OL_SIGNATURES[[sig]], signatures_filtered[[sig]])
    cat(sprintf("  %s: %d/%d genes", sig, found, total))
    if (length(missing) > 0) cat(sprintf(" — missing: %s", paste(missing, collapse = ", ")))
    cat("\n")
}

# Remove signatures with < 2 genes
signatures_filtered <- signatures_filtered[sapply(signatures_filtered, length) >= 2]
cat("\nSignatures retained:", length(signatures_filtered), "\n")

if (length(signatures_filtered) == 0) {
    stop("No signatures have >= 2 genes in this dataset. Cannot proceed.")
}

cat("\nScoring with AddModuleScore...\n")

for (sig_name in names(signatures_filtered)) {
    ol <- AddModuleScore(ol,
                         features = list(signatures_filtered[[sig_name]]),
                         name = sig_name,
                         ctrl = 100,
                         seed = 42)
    cat(sprintf("  ✓ %s (%d genes)\n", sig_name, length(signatures_filtered[[sig_name]])))
}

score_cols_check <- paste0(names(signatures_filtered), "_1")
cat("\nScore columns added:", paste(score_cols_check, collapse = ", "), "\n")
cat("All present:", all(score_cols_check %in% colnames(ol@meta.data)), "\n")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# ANNOTATE OLs (CLUSTER-LEVEL, Z-SCORED)
# ════════════════════════════════════════════════════════════════════════════════

cat("\n================================================================================\n")
cat("ANNOTATING OLs (Cluster-Level, Z-Scored Assignment)\n")
cat("================================================================================\n")

# ════════════════════════════════════════════════════════════════════════════════
# STEP 1: GET SCORE COLUMNS
# ════════════════════════════════════════════════════════════════════════════════

score_cols <- paste0(names(signatures_filtered), "1")
score_cols <- score_cols[score_cols %in% colnames(ol@meta.data)]
sig_names <- gsub("1$", "", score_cols)

cat("Using", length(score_cols), "signatures:", paste(sig_names, collapse = ", "), "\n")

# ════════════════════════════════════════════════════════════════════════════════
# STEP 2: COMPUTE MEAN SCORES PER CLUSTER
# ════════════════════════════════════════════════════════════════════════════════

cat("\n--- Computing mean scores per cluster ---\n")

cluster_ids <- sort(unique(ol$seurat_clusters))

cluster_scores <- data.frame(cluster = cluster_ids)
rownames(cluster_scores) <- cluster_ids

for (col in score_cols) {
    means <- tapply(ol@meta.data[[col]], ol$seurat_clusters, mean)
    cluster_scores[[col]] <- means[as.character(cluster_ids)]
}

# ════════════════════════════════════════════════════════════════════════════════
# STEP 3: Z-SCORE ACROSS CLUSTERS → ASSIGN
# ════════════════════════════════════════════════════════════════════════════════

score_matrix <- as.matrix(cluster_scores[, score_cols])
z_cluster_scores <- scale(score_matrix)
colnames(z_cluster_scores) <- sig_names

cluster_scores$assigned_state <- sig_names[apply(z_cluster_scores, 1, which.max)]
cluster_scores$top_zscore <- apply(z_cluster_scores, 1, max)
cluster_scores$margin <- apply(z_cluster_scores, 1, function(x) {
    sorted <- sort(x, decreasing = TRUE)
    sorted[1] - sorted[2]
})

cat("\nCluster assignments (z-scored):\n")
print(cluster_scores[, c("cluster", "assigned_state", "top_zscore", "margin")])

# Flag low-confidence clusters
low_conf <- cluster_scores$margin < 0.3
if (any(low_conf)) {
    cat("\nLow-confidence clusters (margin < 0.3):\n")
    print(cluster_scores[low_conf, c("cluster", "assigned_state", "margin")])
}

# ════════════════════════════════════════════════════════════════════════════════
# STEP 4: MAP STATE TO CELLS
# ════════════════════════════════════════════════════════════════════════════════

state_map <- setNames(cluster_scores$assigned_state, as.character(cluster_scores$cluster))
ol$ol_state <- unname(state_map[as.character(ol$seurat_clusters)])

SUBCLUSTER_COL <- "ol_state"

# ════════════════════════════════════════════════════════════════════════════════
# STATE DISTRIBUTION
# ════════════════════════════════════════════════════════════════════════════════

cat("\nState distribution:\n")
print(table(ol@meta.data[[SUBCLUSTER_COL]]))

cat("\nPercentages:\n")
print(round(prop.table(table(ol@meta.data[[SUBCLUSTER_COL]])) * 100, 1))

# Check coverage
states_found <- unique(cluster_scores$assigned_state)
states_missing <- setdiff(sig_names, states_found)
cat("\nStates found:", paste(states_found, collapse = ", "), "\n")
if (length(states_missing) > 0) {
    cat("States not detected:", paste(states_missing, collapse = ", "), "\n")
    cat("  (No cluster scored highest for these — may not be present in data)\n")
}

# ════════════════════════════════════════════════════════════════════════════════
# SAVE CLUSTER SCORES
# ════════════════════════════════════════════════════════════════════════════════

cell_type_label <- gsub("_state$", "", SUBCLUSTER_COL)

scores_file <- file.path(RESULTS_DIR, paste0(DATASET, "_", cell_type_label, "_cluster_scores.csv"))
write.csv(cluster_scores, scores_file, row.names = FALSE)
cat("Saved:", scores_file, "\n")

cluster_summary <- data.frame(
    cluster = names(state_map),
    assigned_state = unname(state_map),
    n_cells = as.numeric(table(ol$seurat_clusters)[names(state_map)]),
    margin = cluster_scores$margin,
    row.names = NULL
)

cluster_summary_file <- file.path(RESULTS_DIR, paste0(DATASET, "_", cell_type_label, "_cluster_assignments.csv"))
write.csv(cluster_summary, cluster_summary_file, row.names = FALSE)
cat("Saved:", cluster_summary_file, "\n")

# ════════════════════════════════════════════════════════════════════════════════
# STATE COLORS & ORDER
# ════════════════════════════════════════════════════════════════════════════════

state_order <- names(sort(table(ol@meta.data[[SUBCLUSTER_COL]]), decreasing = TRUE))
state_colors <- get_state_colors(state_order)

cat("\nState order (by frequency):\n")
cat("  ", paste(state_order, collapse = " > "), "\n")

# ════════════════════════════════════════════════════════════════════════════════
# SET seurat_obj FOR DOWNSTREAM
# ════════════════════════════════════════════════════════════════════════════════

seurat_obj <- ol

save_file <- file.path(OUTPUT_DIR, paste0(DATASET, "_", cell_type_label, "_annotated.qs"))
qsave(seurat_obj, save_file)

cat("\nSaved:", save_file, "\n")
cat("  Cells:", ncol(seurat_obj), "\n")
cat("  Clusters:", length(cluster_ids), "\n")
cat("  States:", length(states_found), "\n")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# SENESCENCE RE-THRESHOLDING AT CELL STATE LEVEL
# ════════════════════════════════════════════════════════════════════════════════

cat("\n", paste(rep("=", 80), collapse = ""), "\n")
cat("SENESCENCE RE-THRESHOLDING AT CELL STATE LEVEL\n")
cat(paste(rep("=", 80), collapse = ""), "\n")

# ─────────────────────────────────────────────────────────────────────────────
# CONFIGURATION
# ─────────────────────────────────────────────────────────────────────────────

SD_THRESHOLD <- 2
SCORE_COL <- "senescence_score"

if (STUDY_TYPE == "aging") {
    REFERENCE_GROUP <- GROUP_ORDER[1]
} else {
    REFERENCE_GROUP <- config$reference_group
}

cat(sprintf("\n  Seurat object: %s cells\n", format(ncol(seurat_obj), big.mark = ",")))
cat(sprintf("  Score column: %s\n", SCORE_COL))
cat(sprintf("  State column: %s\n", SUBCLUSTER_COL))
cat(sprintf("  Study type: %s\n", STUDY_TYPE))
cat(sprintf("  Reference group: %s\n", REFERENCE_GROUP))
cat(sprintf("  Threshold: Mean + %s SD per cell state\n", SD_THRESHOLD))

# ─────────────────────────────────────────────────────────────────────────────
# VALIDATE
# ─────────────────────────────────────────────────────────────────────────────

required_cols <- c(SCORE_COL, SENESCENCE_LABEL_COL, STUDY_GROUP_COL, SUBCLUSTER_COL)
missing <- required_cols[!required_cols %in% colnames(seurat_obj@meta.data)]
if (length(missing) > 0) {
    stop(sprintf("Missing columns: %s", paste(missing, collapse = ", ")))
}

ref_mask <- seurat_obj@meta.data[[STUDY_GROUP_COL]] == REFERENCE_GROUP
n_ref <- sum(ref_mask)
if (n_ref == 0) {
    stop(sprintf("Reference group '%s' not found! Available: %s",
                 REFERENCE_GROUP,
                 paste(unique(seurat_obj@meta.data[[STUDY_GROUP_COL]]), collapse = ", ")))
}
cat(sprintf("  Reference cells: %s\n", format(n_ref, big.mark = ",")))

# ─────────────────────────────────────────────────────────────────────────────
# ORIGINAL (SUBCLASS-LEVEL) SUMMARY
# ─────────────────────────────────────────────────────────────────────────────

orig_snc <- sum(seurat_obj@meta.data[[SENESCENCE_LABEL_COL]] == "SnC")
n_total <- nrow(seurat_obj@meta.data)

cat(sprintf("\n  Original (subclass-level): %s SnC / %s (%.1f%%)\n",
            format(orig_snc, big.mark = ","),
            format(n_total, big.mark = ","),
            orig_snc / n_total * 100))

# ─────────────────────────────────────────────────────────────────────────────
# CALCULATE STATE-LEVEL THRESHOLDS
# ─────────────────────────────────────────────────────────────────────────────

cat("\n  Calculating thresholds per cell state...\n")

states <- unique(seurat_obj@meta.data[[SUBCLUSTER_COL]])
thresholds <- list()

for (state in states) {
    state_ref_mask <- ref_mask & (seurat_obj@meta.data[[SUBCLUSTER_COL]] == state)
    ref_scores <- seurat_obj@meta.data[state_ref_mask, SCORE_COL]
    
    if (length(ref_scores) >= 10) {
        state_mean <- mean(ref_scores)
        state_sd <- sd(ref_scores)
        thresholds[[state]] <- state_mean + (SD_THRESHOLD * state_sd)
        
        cat(sprintf("    %s:\n", state))
        cat(sprintf("      n_ref=%s, mean=%.4f, sd=%.4f, threshold=%.4f\n",
                    format(length(ref_scores), big.mark = ","),
                    state_mean, state_sd, thresholds[[state]]))
    } else {
        all_state_scores <- seurat_obj@meta.data[
            seurat_obj@meta.data[[SUBCLUSTER_COL]] == state, SCORE_COL
        ]
        state_mean <- mean(all_state_scores)
        state_sd <- sd(all_state_scores)
        thresholds[[state]] <- state_mean + (SD_THRESHOLD * state_sd)
        
        cat(sprintf("    %s: (fallback — only %d ref cells, using all %s cells)\n",
                    state, length(ref_scores),
                    format(length(all_state_scores), big.mark = ",")))
        cat(sprintf("      mean=%.4f, sd=%.4f, threshold=%.4f\n",
                    state_mean, state_sd, thresholds[[state]]))
    }
}

# ─────────────────────────────────────────────────────────────────────────────
# APPLY STATE-LEVEL THRESHOLDS
# ─────────────────────────────────────────────────────────────────────────────

cat("\n  Applying state-level thresholds...\n")

seurat_obj@meta.data$is_senescent_state <- FALSE

for (state in names(thresholds)) {
    mask <- (seurat_obj@meta.data[[SUBCLUSTER_COL]] == state) &
            (seurat_obj@meta.data[[SCORE_COL]] >= thresholds[[state]])
    seurat_obj@meta.data$is_senescent_state[mask] <- TRUE
}

seurat_obj@meta.data$senescence_label_state <- ifelse(
    seurat_obj@meta.data$is_senescent_state, "SnC", "Non-SnC"
)

# ─────────────────────────────────────────────────────────────────────────────
# COMPARISON: ORIGINAL VS STATE-LEVEL
# ─────────────────────────────────────────────────────────────────────────────

new_snc <- sum(seurat_obj@meta.data$is_senescent_state)

cat("\n  ✓ Re-thresholding complete\n")
cat(sprintf("\n  %-25s %12s %12s\n", "", "Subclass", "State-level"))
cat(paste(rep("-", 55), collapse = ""), "\n")
cat(sprintf("  %-25s %12s %12s\n",
            "Total SnC",
            format(orig_snc, big.mark = ","),
            format(new_snc, big.mark = ",")))
cat(sprintf("  %-25s %11.1f%% %11.1f%%\n",
            "%SnC overall",
            orig_snc / n_total * 100,
            new_snc / n_total * 100))

cat("\n  Per state:\n")
cat(sprintf("  %-25s %6s %8s %8s %8s %8s %8s\n",
            "State", "Total", "Orig_n", "Orig_%", "New_n", "New_%", "Change"))
cat(paste(rep("-", 80), collapse = ""), "\n")

for (state in sort(states)) {
    state_mask <- seurat_obj@meta.data[[SUBCLUSTER_COL]] == state
    state_total <- sum(state_mask)
    
    orig_n <- sum(state_mask & (seurat_obj@meta.data[[SENESCENCE_LABEL_COL]] == "SnC"))
    new_n <- sum(state_mask & seurat_obj@meta.data$is_senescent_state)
    
    orig_pct <- orig_n / state_total * 100
    new_pct <- new_n / state_total * 100
    change <- new_pct - orig_pct
    
    flag <- if (abs(change) > 2) sprintf("%+.1f ←", change) else sprintf("%+.1f", change)
    
    cat(sprintf("  %-25s %6s %8s %7.1f%% %8s %7.1f%% %8s\n",
                state,
                format(state_total, big.mark = ","),
                format(orig_n, big.mark = ","), orig_pct,
                format(new_n, big.mark = ","), new_pct,
                flag))
}

cat("\n", paste(rep("=", 80), collapse = ""), "\n")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# ACTIVATE STATE-LEVEL LABELS FOR DOWNSTREAM PLOTS
# ─────────────────────────────────────────────────────────────────────────────
SENESCENCE_LABEL_COL <- "senescence_label_state"
cat(sprintf("\n  Active senescence column: %s\n", SENESCENCE_LABEL_COL))
cat(sprintf("  SnC: %s\n", format(sum(seurat_obj@meta.data[[SENESCENCE_LABEL_COL]] == "SnC"), big.mark = ",")))
cat(sprintf("  Non-SnC: %s\n", format(sum(seurat_obj@meta.data[[SENESCENCE_LABEL_COL]] == "Non-SnC"), big.mark = ",")))
cat("\n  All downstream plots will use state-level labels.\n")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# VALIDATION: CANONICAL MARKERS DOTPLOT
# ════════════════════════════════════════════════════════════════════════════════

cat("\n================================================================================\n")
cat("VALIDATION: CANONICAL MARKERS DOTPLOT\n")
cat("================================================================================\n")

cell_type_label <- gsub("_state$", "", SUBCLUSTER_COL)

canonical_markers <- list(
    'hOligo1'    = c('RASGRF1', 'RASGRF2', 'KLK6'),
    'hOligo2'    = c('PLXDC2', 'PALM2', 'OPALIN', 'QDPR'),
    'DA1_Immune' = c('SERPINA3', 'C4B', 'B2M', 'HLA-A', 'HLA-B', 'CD74'),
    'IFN'        = c('IFIT1', 'IFIT3', 'OAS1', 'OAS2')
)

canonical_markers_filtered <- lapply(canonical_markers, function(genes) {
    genes[genes %in% rownames(seurat_obj)]
})
canonical_markers_filtered <- canonical_markers_filtered[sapply(canonical_markers_filtered, length) > 0]

cat("\nMarkers available:\n")
for (ct in names(canonical_markers_filtered)) {
    cat(sprintf("  %s: %s\n", ct, paste(canonical_markers_filtered[[ct]], collapse = ", ")))
}

all_markers <- unlist(canonical_markers_filtered)

Idents(seurat_obj) <- "seurat_clusters"
cluster_order <- as.character(sort(as.numeric(levels(Idents(seurat_obj)))))

cat("\nCalculating expression statistics...\n")

dot_list <- list()
idx <- 1

for (cluster in cluster_order) {
    cells <- WhichCells(seurat_obj, idents = cluster)
    if (length(cells) == 0) next

    for (category in names(canonical_markers_filtered)) {
        for (gene in canonical_markers_filtered[[category]]) {
            expr <- GetAssayData(seurat_obj, slot = "data")[gene, cells]

            dot_list[[idx]] <- data.frame(
                Cluster = cluster,
                Gene = gene,
                Category = category,
                Pct_Exp = sum(expr > 0) / length(expr) * 100,
                Avg_Exp = mean(expr),
                stringsAsFactors = FALSE
            )
            idx <- idx + 1
        }
    }
}

dot_data <- do.call(rbind, dot_list)
cat("Collected", nrow(dot_data), "data points\n")

dot_data <- dot_data %>%
    group_by(Gene) %>%
    mutate(Scaled_Exp = as.numeric(scale(Avg_Exp))) %>%
    ungroup()

dot_data$Scaled_Exp <- pmax(pmin(dot_data$Scaled_Exp, 2.5), -2.5)
dot_data$Cluster <- factor(dot_data$Cluster, levels = rev(cluster_order))
dot_data$Gene <- factor(dot_data$Gene, levels = all_markers)
dot_data$Category <- factor(dot_data$Category, levels = names(canonical_markers_filtered))

options(repr.plot.width = 12, repr.plot.height = 10)

p_dotplot <- ggplot(dot_data, aes(x = Gene, y = Cluster)) +
    geom_point(aes(size = Pct_Exp, fill = Scaled_Exp), shape = 21, color = "black", stroke = 0.3) +
    scale_size_continuous(
        range = c(1, 6),
        limits = c(0, 100),
        breaks = c(0, 25, 50, 75, 100),
        name = "% Expr"
    ) +
    scale_fill_gradientn(
        colors = c("#313695", "#4575B4", "#74ADD1", "#FFFFBF", "#FDAE61", "#F46D43", "#A50026"),
        limits = c(-2.5, 2.5),
        name = "Scaled\nExpr"
    ) +
    facet_grid(cols = vars(Category), scales = "free_x", space = "free_x") +
    labs(x = NULL, y = "Cluster") +
    theme_bw(base_size = 10) +
    theme(
        axis.text.x = element_text(angle = 45, hjust = 1, size = 8, face = "italic"),
        axis.text.y = element_text(size = 9),
        axis.title.y = element_text(size = 10, face = "bold"),
        strip.text = element_text(size = 8, face = "bold"),
        strip.background = element_rect(fill = "gray70", color = "black", linewidth = 0.5),
        panel.grid = element_blank(),
        panel.spacing = unit(0.2, "lines"),
        panel.border = element_rect(color = "black", linewidth = 0.5),
        legend.position = "right",
        legend.title = element_text(size = 8),
        legend.text = element_text(size = 7),
        legend.key.size = unit(0.4, "cm"),
        plot.margin = margin(10, 10, 10, 10)
    )

print(p_dotplot)

dotplot_file <- file.path(FIGURES_DIR, paste0(DATASET, "_", cell_type_label, "_dotplot_canonical.svg"))
ggsave(dotplot_file, p_dotplot, width = 12, height = 10)
cat("Saved:", dotplot_file, "\n")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# VALIDATION: UMAP PANEL
# ════════════════════════════════════════════════════════════════════════════════
cat("\n", paste(rep("=", 80), collapse = ""), "\n")
cat("VALIDATION: UMAP PANEL\n")
cat(paste(rep("=", 80), collapse = ""), "\n")
cell_type_label <- gsub("_state$", "", SUBCLUSTER_COL)
state_colors <- get_state_colors(unique(seurat_obj@meta.data[[SUBCLUSTER_COL]]))
add_corner_arrows <- function(p, label_size = 2.5) {
    build <- ggplot_build(p)
    x_range <- build$layout$panel_params[[1]]$x.range
    y_range <- build$layout$panel_params[[1]]$y.range
    
    x_start <- x_range[1] + diff(x_range) * 0.02
    y_start <- y_range[1] + diff(y_range) * 0.02
    x_arrow <- diff(x_range) * 0.12
    y_arrow <- diff(y_range) * 0.12
    
    p + 
        annotate("segment", x = x_start, xend = x_start + x_arrow, y = y_start, yend = y_start,
                 arrow = arrow(length = unit(0.1, "cm"), type = "closed"), linewidth = 0.3) +
        annotate("text", x = x_start + x_arrow/2, y = y_start - diff(y_range) * 0.04,
                 label = "UMAP1", size = label_size, hjust = 0.5, vjust = 1) +
        annotate("segment", x = x_start, xend = x_start, y = y_start, yend = y_start + y_arrow,
                 arrow = arrow(length = unit(0.1, "cm"), type = "closed"), linewidth = 0.3) +
        annotate("text", x = x_start - diff(x_range) * 0.04, y = y_start + y_arrow/2,
                 label = "UMAP2", size = label_size, hjust = 1, vjust = 0.5, angle = 90) +
        coord_cartesian(clip = "off")
}
options(repr.plot.width = 14, repr.plot.height = 12)
umap_theme <- theme_void(base_size = 10) +
    theme(
        plot.title = element_text(size = 11, face = "bold", hjust = 0.5),
        legend.position = "right",
        legend.title = element_blank(),
        legend.text = element_text(size = 8),
        legend.key.size = unit(0.3, "cm"),
        plot.margin = margin(15, 15, 20, 20)
    )
p1 <- DimPlot(seurat_obj, reduction = "umap", group.by = "seurat_clusters", 
              label = TRUE, label.size = 3, pt.size = 0.3) +
    ggtitle("Clusters") + umap_theme + theme(legend.position = "none")
p1 <- add_corner_arrows(p1)
p2 <- DimPlot(seurat_obj, reduction = "umap", group.by = SUBCLUSTER_COL, 
              pt.size = 0.3) +
    scale_color_manual(values = state_colors) +
    ggtitle("Cell States") + umap_theme
p2 <- add_corner_arrows(p2)
p3 <- DimPlot(seurat_obj, reduction = "umap", group.by = SENESCENCE_LABEL_COL, 
              pt.size = 0.3, order = c("SnC", "Non-SnC")) +
    scale_color_manual(values = SENESCENCE_COLORS) +
    ggtitle("Senescence") + umap_theme
p3 <- add_corner_arrows(p3)
p4 <- DimPlot(seurat_obj, reduction = "umap", group.by = STUDY_GROUP_COL, pt.size = 0.3) +
    scale_color_manual(values = study_colors) +
    ggtitle("Study Group") + umap_theme
p4 <- add_corner_arrows(p4)
umap_panel <- (p1 | p2) / (p3 | p4) +
    plot_annotation(
        title = paste0("Oligodendrocyte Subclustering — ", DATASET),
        theme = theme(plot.title = element_text(size = 14, face = "bold", hjust = 0.5))
    )
print(umap_panel)
umap_file <- file.path(FIGURES_DIR, paste0(DATASET, "_", cell_type_label, "_umap_panel.svg"))
ggsave(umap_file, umap_panel, width = 14, height = 12)
cat("✓ Saved:", umap_file, "\n")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# VALIDATION: SIGNATURE HEATMAP
# ════════════════════════════════════════════════════════════════════════════════
cat("\n================================================================================\n")
cat("VALIDATION: SIGNATURE HEATMAP\n")
cat("================================================================================\n")
cell_type_label <- gsub("_state$", "", SUBCLUSTER_COL)
# Extract only numeric score columns
score_cols_only <- score_cols[score_cols %in% colnames(cluster_scores)]
heatmap_data <- cluster_scores[, score_cols_only]
colnames(heatmap_data) <- gsub("_1$", "", colnames(heatmap_data))
rownames(heatmap_data) <- cluster_scores$cluster
# Z-scale for visualization
cluster_scores_scaled <- t(scale(t(as.matrix(heatmap_data))))
# Order clusters by dominant state
cluster_dominant <- colnames(heatmap_data)[apply(heatmap_data, 1, which.max)]
cluster_order_df <- data.frame(
    cluster = rownames(cluster_scores_scaled),
    dominant = cluster_dominant,
    max_score = apply(cluster_scores_scaled, 1, max)
)
state_priority <- names(sort(table(seurat_obj@meta.data[[SUBCLUSTER_COL]]), decreasing = TRUE))
state_priority <- state_priority[state_priority %in% unique(cluster_dominant)]
cluster_order_df$dominant <- factor(cluster_order_df$dominant, levels = state_priority)
cluster_order_df <- cluster_order_df[order(cluster_order_df$dominant, -cluster_order_df$max_score), ]
cluster_order <- cluster_order_df$cluster
cluster_scores_scaled <- cluster_scores_scaled[cluster_order, ]
sig_order <- state_priority[state_priority %in% colnames(cluster_scores_scaled)]
sig_remaining <- colnames(cluster_scores_scaled)[!colnames(cluster_scores_scaled) %in% sig_order]
sig_order <- c(sig_order, sig_remaining)
cluster_scores_scaled <- cluster_scores_scaled[, sig_order]
row_annotation <- data.frame(
    State = cluster_order_df$dominant[match(rownames(cluster_scores_scaled), cluster_order_df$cluster)]
)
rownames(row_annotation) <- rownames(cluster_scores_scaled)
ann_colors <- list(
    State = get_state_colors(state_priority)
)
options(repr.plot.width = 10, repr.plot.height = 10)
p_heatmap <- pheatmap(
    cluster_scores_scaled,
    cluster_rows = FALSE,
    cluster_cols = FALSE,
    color = colorRampPalette(c("#313695", "#4575B4", "#74ADD1", "white", "#FDAE61", "#F46D43", "#A50026"))(100),
    breaks = seq(-2.5, 2.5, length.out = 101),
    annotation_row = row_annotation,
    annotation_colors = ann_colors,
    fontsize = 10,
    fontsize_row = 8,
    fontsize_col = 10,
    angle_col = 45,
    border_color = "gray90",
    cellwidth = 35,
    cellheight = 12,
    gaps_row = cumsum(table(cluster_order_df$dominant)),
    main = paste0("Oligodendrocyte Signature Enrichment — ", DATASET),
    silent = TRUE
)
print(p_heatmap)
heatmap_file <- file.path(FIGURES_DIR, paste0(DATASET, "_", cell_type_label, "_signature_heatmap.svg"))
svg(heatmap_file, width = 10, height = 10)
print(p_heatmap)
dev.off()
cat("Saved:", heatmap_file, "\n")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# POST-SUBCLUSTERING ANALYSIS: COMPOSITION BY STUDY GROUP
# ════════════════════════════════════════════════════════════════════════════════

cat("\n================================================================================\n")
cat("POST-SUBCLUSTERING ANALYSIS\n")
cat("================================================================================\n")

# ─────────────────────────────────────────────────────────────────────────────
# SETUP
# ─────────────────────────────────────────────────────────────────────────────

cell_type_label <- gsub("_state$", "", SUBCLUSTER_COL)

state_order <- seurat_obj@meta.data %>%
    count(.data[[SUBCLUSTER_COL]]) %>%
    arrange(desc(n)) %>%
    pull(.data[[SUBCLUSTER_COL]])

n_states <- length(state_order)
state_colors <- get_state_colors(state_order)

cat("\nCell type:", cell_type_label, "\n")
cat("States detected:", n_states, "\n")
cat("  ", paste(state_order, collapse = ", "), "\n")
cat("Colors mapped:", length(state_colors), "\n")

# ════════════════════════════════════════════════════════════════════════════════
# PLOT 1: COMPOSITION BY STUDY GROUP
# ════════════════════════════════════════════════════════════════════════════════

cat("\n--- Plot 1: Composition by Study Group ---\n")

comp_studygroup <- seurat_obj@meta.data %>%
    group_by(.data[[STUDY_GROUP_COL]], .data[[SUBCLUSTER_COL]]) %>%
    summarise(n = n(), .groups = "drop") %>%
    group_by(.data[[STUDY_GROUP_COL]]) %>%
    mutate(pct = n / sum(n) * 100) %>%
    ungroup()

colnames(comp_studygroup)[1:2] <- c("Study_Group", "State")

# ─────────────────────────────────────────────────────────────────────────────
# TABLE OUTPUT
# ─────────────────────────────────────────────────────────────────────────────

comp_wide <- comp_studygroup %>%
    select(Study_Group, State, pct) %>%
    pivot_wider(names_from = Study_Group, values_from = pct, values_fill = 0) %>%
    mutate(across(where(is.numeric), ~ round(.x, 1)))

cat("\nState Composition by Study Group (%):\n")
print(as.data.frame(comp_wide))

write.csv(comp_wide, 
          file.path(RESULTS_DIR, paste0(DATASET, "_", cell_type_label, "_composition_by_studygroup.csv")),
          row.names = FALSE)

# ─────────────────────────────────────────────────────────────────────────────
# PREPARE FACTORS
# ─────────────────────────────────────────────────────────────────────────────

sg_detected <- unique(comp_studygroup$Study_Group)
is_aging <- STUDY_TYPE == "aging"

sg_order <- names(STUDY_GROUP_COLORS)[names(STUDY_GROUP_COLORS) %in% sg_detected]
comp_studygroup$Study_Group <- factor(comp_studygroup$Study_Group, levels = sg_order)
comp_studygroup$State <- factor(comp_studygroup$State, levels = rev(state_order))

if (is_aging) {
    x_labels <- setNames(gsub("Age_", "", gsub("_", "-", sg_order)), sg_order)
    x_title <- "Age Group (years)"
} else {
    x_labels <- setNames(gsub("_", " ", sg_order), sg_order)
    x_title <- "Disease Status"
}

# ─────────────────────────────────────────────────────────────────────────────
# PLOT
# ─────────────────────────────────────────────────────────────────────────────

options(repr.plot.width = 8, repr.plot.height = 6)

p_comp_sg <- ggplot(comp_studygroup, aes(x = Study_Group, y = pct, fill = State)) +
    geom_col(width = 0.75, color = "white", linewidth = 0.3) +
    scale_fill_manual(values = state_colors) +
    scale_x_discrete(labels = x_labels) +
    scale_y_continuous(expand = expansion(mult = c(0, 0.02))) +
    labs(
        x = x_title, 
        y = "Proportion (%)",
        fill = paste0(tools::toTitleCase(cell_type_label), " State")
    ) +
    theme_minimal(base_size = 14) +
    theme(
        axis.text.x = element_text(size = 13, color = "black", face = "bold",
                                   angle = 45, hjust = 1, vjust = 1),
        axis.text.y = element_text(size = 12, color = "black"),
        axis.title.x = element_text(size = 15, face = "bold", margin = margin(t = 12)),
        axis.title.y = element_text(size = 15, face = "bold", margin = margin(r = 12)),
        axis.line = element_line(color = "black", linewidth = 0.5),
        axis.ticks = element_line(color = "black", linewidth = 0.3),
        axis.ticks.length = unit(0.15, "cm"),
        legend.position = "right",
        legend.title = element_text(size = 12, face = "bold"),
        legend.text = element_text(size = 10),
        legend.key.size = unit(0.5, "cm"),
        panel.grid = element_blank(),
        panel.background = element_blank(),
        plot.margin = margin(15, 15, 15, 15)
    ) +
    guides(fill = guide_legend(ncol = 1, reverse = TRUE))

print(p_comp_sg)

ggsave(file.path(FIGURES_DIR, paste0(DATASET, "_", cell_type_label, "_composition_by_studygroup.svg")), 
       p_comp_sg, width = 8, height = 6, dpi = 300)

cat("\n✓ Saved:", paste0(DATASET, "_", cell_type_label, "_composition_by_studygroup.svg"), "\n")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# PLOT: SnC COMPOSITION BY STUDY GROUP
# ════════════════════════════════════════════════════════════════════════════════

cat(sprintf("\n--- Plot: SnC Composition by Study Group (%s) ---\n", tools::toTitleCase(cell_type_label)))

snc_comp_sg <- seurat_obj@meta.data %>%
    filter(.data[[SENESCENCE_LABEL_COL]] == "SnC") %>%
    group_by(.data[[STUDY_GROUP_COL]], .data[[SUBCLUSTER_COL]]) %>%
    summarise(n = n(), .groups = "drop") %>%
    complete(.data[[STUDY_GROUP_COL]], .data[[SUBCLUSTER_COL]], fill = list(n = 0)) %>%
    group_by(.data[[STUDY_GROUP_COL]]) %>%
    mutate(pct = n / sum(n) * 100) %>%
    ungroup()

colnames(snc_comp_sg)[1:2] <- c("Study_Group", "State")
snc_comp_sg$pct[is.nan(snc_comp_sg$pct)] <- 0

sg_detected <- unique(snc_comp_sg$Study_Group)
is_aging <- STUDY_TYPE == "aging"

sg_order <- names(STUDY_GROUP_COLORS)[names(STUDY_GROUP_COLORS) %in% sg_detected]
snc_comp_sg$Study_Group <- factor(snc_comp_sg$Study_Group, levels = sg_order)
snc_comp_sg$State <- factor(snc_comp_sg$State, levels = rev(state_order))

if (is_aging) {
    x_labels <- setNames(gsub("Age_", "", gsub("_", "-", sg_order)), sg_order)
    x_title <- "Age Group (years)"
} else {
    x_labels <- setNames(gsub("_", " ", sg_order), sg_order)
    x_title <- "Disease Status"
}

snc_comp_sg_wide <- snc_comp_sg %>%
    select(Study_Group, State, pct) %>%
    pivot_wider(names_from = Study_Group, values_from = pct, values_fill = 0) %>%
    mutate(across(where(is.numeric), ~ round(.x, 1)))

cat("\nSnC Composition by Study Group (%):\n")
print(as.data.frame(snc_comp_sg_wide))

write.csv(snc_comp_sg_wide, 
          file.path(RESULTS_DIR, paste0(DATASET, "_", cell_type_label, "_snc_composition_by_studygroup.csv")),
          row.names = FALSE)

snc_totals <- seurat_obj@meta.data %>%
    filter(.data[[SENESCENCE_LABEL_COL]] == "SnC") %>%
    group_by(.data[[STUDY_GROUP_COL]]) %>%
    summarise(n = n(), .groups = "drop")

colnames(snc_totals)[1] <- "Study_Group"

cat("\nTotal SnC cells per study group:\n")
print(as.data.frame(snc_totals))

options(repr.plot.width = 8, repr.plot.height = 6)

p_snc_comp_sg <- ggplot(snc_comp_sg, aes(x = Study_Group, y = pct, fill = State)) +
    geom_col(width = 0.75, color = "white", linewidth = 0.3) +
    scale_fill_manual(values = state_colors) +
    scale_x_discrete(labels = x_labels) +
    scale_y_continuous(limits = c(0, 100.1), expand = c(0, 0)) +
    labs(
        x = x_title, 
        y = paste0("Proportion of SnC ", tools::toTitleCase(cell_type_label), " (%)"),
        fill = paste0(tools::toTitleCase(cell_type_label), " State")
    ) +
    theme_minimal(base_size = 14) +
    theme(
        axis.text.x = element_text(size = 13, color = "black", face = "bold",
                                   angle = 45, hjust = 1, vjust = 1),
        axis.text.y = element_text(size = 12, color = "black"),
        axis.title.x = element_text(size = 15, face = "bold", margin = margin(t = 12)),
        axis.title.y = element_text(size = 15, face = "bold", margin = margin(r = 12)),
        axis.line = element_line(color = "black", linewidth = 0.5),
        axis.ticks = element_line(color = "black", linewidth = 0.3),
        axis.ticks.length = unit(0.15, "cm"),
        legend.position = "right",
        legend.title = element_text(size = 12, face = "bold"),
        legend.text = element_text(size = 10),
        legend.key.size = unit(0.5, "cm"),
        panel.grid = element_blank(),
        panel.background = element_blank(),
        plot.margin = margin(15, 15, 15, 15)
    ) +
    guides(fill = guide_legend(ncol = 1, reverse = TRUE))

print(p_snc_comp_sg)

ggsave(file.path(FIGURES_DIR, paste0(DATASET, "_", cell_type_label, "_snc_composition_by_studygroup.svg")), 
       p_snc_comp_sg, width = 8, height = 6, dpi = 300)

cat("\nSaved:", paste0(DATASET, "_", cell_type_label, "_snc_composition_by_studygroup.svg"), "\n")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# PLOT 2: COMPOSITION BY SAMPLE (SPLIT BY STUDY GROUP)
# ════════════════════════════════════════════════════════════════════════════════

cat("\n--- Plot 2: Composition by Sample (Split by Study Group) ---\n")

cell_type_label <- gsub("_state$", "", SUBCLUSTER_COL)
is_aging <- STUDY_TYPE == "aging"

# ─────────────────────────────────────────────────────────────────────────────
# STEP 1: CALCULATE COMPOSITION
# ─────────────────────────────────────────────────────────────────────────────

comp_sample <- seurat_obj@meta.data %>%
    group_by(.data[[DONOR_COL]], .data[[STUDY_GROUP_COL]], .data[[SUBCLUSTER_COL]]) %>%
    summarise(n = n(), .groups = "drop") %>%
    group_by(.data[[DONOR_COL]]) %>%
    mutate(pct = n / sum(n) * 100) %>%
    ungroup()

colnames(comp_sample)[1:3] <- c("Donor", "Study_Group", "State")

comp_sample$Donor <- as.character(comp_sample$Donor)
comp_sample$Study_Group <- as.character(comp_sample$Study_Group)
comp_sample$State <- as.character(comp_sample$State)

cat("\nStep 1 - Initial calculation:\n")
cat("  Rows:", nrow(comp_sample), "\n")
cat("  Unique donors:", n_distinct(comp_sample$Donor), "\n")
cat("  Unique states:", n_distinct(comp_sample$State), "\n")

# ─────────────────────────────────────────────────────────────────────────────
# STEP 2: COMPLETE MISSING COMBINATIONS
# ─────────────────────────────────────────────────────────────────────────────

donor_sg <- comp_sample %>% select(Donor, Study_Group) %>% distinct()
all_states <- unique(comp_sample$State)
complete_grid <- donor_sg %>% crossing(State = all_states)

comp_sample <- complete_grid %>%
    left_join(comp_sample %>% select(Donor, State, n, pct), 
              by = c("Donor", "State")) %>%
    mutate(n = replace_na(n, 0), pct = replace_na(pct, 0))

cat("\nStep 2 - After grid completion:\n")
cat("  Rows:", nrow(comp_sample), "\n")
cat("  Expected:", n_distinct(donor_sg$Donor), "x", length(all_states), "=", 
    n_distinct(donor_sg$Donor) * length(all_states), "\n")

sample_totals <- comp_sample %>%
    group_by(Donor) %>%
    summarise(total_pct = sum(pct), .groups = "drop")

cat("  Sample totals - Min:", round(min(sample_totals$total_pct), 1), 
    "Max:", round(max(sample_totals$total_pct), 1), "\n")

# ─────────────────────────────────────────────────────────────────────────────
# STEP 3: DETECT COHORT TYPE AND SET ORDERS
# ─────────────────────────────────────────────────────────────────────────────

sg_detected <- unique(comp_sample$Study_Group)
sg_order <- names(STUDY_GROUP_COLORS)[names(STUDY_GROUP_COLORS) %in% sg_detected]

state_order <- comp_sample %>%
    group_by(State) %>%
    summarise(total_n = sum(n), .groups = "drop") %>%
    arrange(desc(total_n)) %>%
    pull(State)

sample_order <- seurat_obj@meta.data %>%
    select(all_of(c(DONOR_COL, STUDY_GROUP_COL))) %>%
    distinct() %>%
    mutate(sg_factor = factor(.data[[STUDY_GROUP_COL]], levels = sg_order)) %>%
    arrange(sg_factor, .data[[DONOR_COL]]) %>%
    pull(.data[[DONOR_COL]])

cat("\nStep 3 - Orders defined:\n")
cat("  Study groups:", paste(sg_order, collapse = ", "), "\n")
cat("  States (by freq):", paste(state_order, collapse = ", "), "\n")
cat("  Samples:", length(sample_order), "\n")

# ─────────────────────────────────────────────────────────────────────────────
# STEP 4: TABLE
# ─────────────────────────────────────────────────────────────────────────────

comp_summary <- comp_sample %>%
    group_by(Study_Group, State) %>%
    summarise(mean_pct = mean(pct), sd_pct = sd(pct), .groups = "drop") %>%
    mutate(display = sprintf("%.1f ± %.1f", mean_pct, sd_pct)) %>%
    select(Study_Group, State, display) %>%
    pivot_wider(names_from = Study_Group, values_from = display)

cat("\nComposition Summary (Mean ± SD):\n")
print(as.data.frame(comp_summary))

# ─────────────────────────────────────────────────────────────────────────────
# STEP 5: STATISTICAL TEST
# ─────────────────────────────────────────────────────────────────────────────

cat("\n--- Statistical Test ---\n")

has_sex <- SEX_COL %in% colnames(seurat_obj@meta.data)
has_cohort <- COHORT_COL %in% colnames(seurat_obj@meta.data)

cat("\nCovariates:\n")
cat("  Sex column (", SEX_COL, "):", ifelse(has_sex, "FOUND", "NOT FOUND"), "\n")
cat("  Cohort column (", COHORT_COL, "):", ifelse(has_cohort, "FOUND", "NOT FOUND"), "\n")

rhs <- "Predictor"
if (has_sex) rhs <- paste0(rhs, " + Sex")
if (has_cohort) rhs <- paste0(rhs, " + Cohort")
formula_text <- paste0("CubeRoot(State_%) ~ ", rhs)

cat("\nModel:", formula_text, "\n")
cat("Transformation: (proportion / 100)^(1/3)\n")

covariate_cols <- c(DONOR_COL)
if (has_sex) covariate_cols <- c(covariate_cols, SEX_COL)
if (has_cohort) covariate_cols <- c(covariate_cols, COHORT_COL)

donor_meta <- seurat_obj@meta.data %>%
    select(all_of(covariate_cols)) %>%
    distinct()

colnames(donor_meta)[1] <- "Donor"
if (has_sex) colnames(donor_meta)[colnames(donor_meta) == SEX_COL] <- "Sex"
if (has_cohort) colnames(donor_meta)[colnames(donor_meta) == COHORT_COL] <- "Cohort"

comp_stats_data <- comp_sample %>%
    mutate(pct_cuberoot = (pct / 100)^(1/3)) %>%
    left_join(donor_meta, by = "Donor")

if (is_aging) {
    comp_stats_data <- comp_stats_data %>%
        mutate(Predictor = as.numeric(gsub("Age_([0-9]+)_.*", "\\1", Study_Group)))
    cat("Cohort type: AGING (testing linear trend with age)\n\n")
} else {
    disease_order <- sg_order
    comp_stats_data <- comp_stats_data %>%
        mutate(Predictor = as.numeric(factor(Study_Group, levels = disease_order)) - 1)
    cat("Cohort type: DISEASE (testing trend across disease stages)\n\n")
}

if (has_sex) comp_stats_data$Sex <- as.factor(comp_stats_data$Sex)
if (has_cohort) comp_stats_data$Cohort <- as.factor(comp_stats_data$Cohort)

model_formula_comp <- as.formula(paste0("pct_cuberoot ~ ", rhs))

results_list <- list()
for (state in state_order) {
    df_state <- comp_stats_data %>% filter(State == state)
    model <- lm(model_formula_comp, data = df_state)
    coef_summary <- summary(model)$coefficients
    results_list[[state]] <- data.frame(
        State = state,
        estimate = coef_summary["Predictor", "Estimate"],
        std_error = coef_summary["Predictor", "Std. Error"],
        p_value = coef_summary["Predictor", "Pr(>|t|)"]
    )
}

comp_stats <- do.call(rbind, results_list) %>%
    mutate(
        p_adj = p.adjust(p_value, method = "BH"),
        direction = case_when(estimate > 0 ~ "Up", estimate < 0 ~ "Down", TRUE ~ ""),
        sig = case_when(p_adj < 0.001 ~ "***", p_adj < 0.01 ~ "**", p_adj < 0.05 ~ "*", TRUE ~ "")
    ) %>%
    arrange(p_adj)

cat("Results (BH-adjusted):\n")
print(comp_stats %>% select(State, estimate, p_value, p_adj, direction, sig))

write.csv(comp_stats, 
          file.path(RESULTS_DIR, paste0(DATASET, "_", cell_type_label, "_composition_stats.csv")),
          row.names = FALSE)

# ─────────────────────────────────────────────────────────────────────────────
# STEP 6: CONVERT TO FACTORS
# ─────────────────────────────────────────────────────────────────────────────

comp_sample$Donor <- factor(comp_sample$Donor, levels = sample_order)
comp_sample$Study_Group <- factor(comp_sample$Study_Group, levels = sg_order)
comp_sample$State <- factor(comp_sample$State, levels = rev(state_order))

cat("\nStep 6 - Factor check (should all be 0):\n")
cat("  NA in Donor:", sum(is.na(comp_sample$Donor)), "\n")
cat("  NA in Study_Group:", sum(is.na(comp_sample$Study_Group)), "\n")
cat("  NA in State:", sum(is.na(comp_sample$State)), "\n")

samples_per_group <- comp_sample %>%
    select(Donor, Study_Group) %>%
    distinct() %>%
    count(Study_Group)

cat("\nSamples per study group:\n")
print(as.data.frame(samples_per_group))

# ─────────────────────────────────────────────────────────────────────────────
# STEP 7: COLORS AND LABELS
# ─────────────────────────────────────────────────────────────────────────────

state_colors <- get_state_colors(state_order)

if (is_aging) {
    facet_labels <- setNames(gsub("Age_", "", gsub("_", "-", sg_order)), sg_order)
} else {
    facet_labels <- setNames(gsub("_", " ", sg_order), sg_order)
}

sig_lookup <- setNames(comp_stats$sig, comp_stats$State)
dir_lookup <- setNames(comp_stats$direction, comp_stats$State)

state_labels <- sapply(state_order, function(s) {
    sig <- sig_lookup[s]
    dir <- dir_lookup[s]
    if (!is.na(sig) && sig != "") {
        dir_symbol <- ifelse(dir == "Up", "(+)", "(-)")
        paste0(s, " ", dir_symbol, sig)
    } else { s }
})

# ─────────────────────────────────────────────────────────────────────────────
# STEP 8: PLOT
# ─────────────────────────────────────────────────────────────────────────────

options(repr.plot.width = 14, repr.plot.height = 6)

p_comp_sample <- ggplot(comp_sample, aes(x = Donor, y = pct, fill = State)) +
    geom_col(width = 0.85, color = "white", linewidth = 0.2) +
    scale_fill_manual(values = state_colors, labels = state_labels) +
    scale_y_continuous(expand = c(0, 0)) +
    coord_cartesian(ylim = c(0, 100)) +
    facet_grid(cols = vars(Study_Group), scales = "free_x", space = "free_x",
               labeller = labeller(Study_Group = facet_labels)) +
    labs(
        x = "Samples", 
        y = "Proportion (%)",
        fill = paste0(tools::toTitleCase(cell_type_label), " State")
    ) +
    theme_minimal(base_size = 14) +
    theme(
        axis.text.x = element_blank(),
        axis.ticks.x = element_blank(),
        axis.text.y = element_text(size = 12, color = "black"),
        axis.title.x = element_text(size = 15, face = "bold", margin = margin(t = 10)),
        axis.title.y = element_text(size = 15, face = "bold", margin = margin(r = 10)),
        axis.line = element_line(color = "black", linewidth = 0.5),
        axis.ticks.y = element_line(color = "black", linewidth = 0.3),
        axis.ticks.length = unit(0.15, "cm"),
        strip.text = element_text(size = 13, face = "bold", color = "black"),
        strip.background = element_rect(fill = "gray80", color = "black", linewidth = 0.5),
        panel.spacing = unit(0.5, "lines"),
        panel.border = element_rect(color = "black", fill = NA, linewidth = 0.5),
        legend.position = "right",
        legend.title = element_text(size = 12, face = "bold"),
        legend.text = element_text(size = 10),
        legend.key.size = unit(0.45, "cm"),
        panel.grid = element_blank(),
        plot.margin = margin(15, 15, 15, 15)
    ) +
    guides(fill = guide_legend(ncol = 1, reverse = TRUE))

print(p_comp_sample)

ggsave(file.path(FIGURES_DIR, paste0(DATASET, "_", cell_type_label, "_composition_by_sample.svg")), 
       p_comp_sample, width = 14, height = 6, dpi = 300)

write.csv(comp_sample, 
          file.path(RESULTS_DIR, paste0(DATASET, "_", cell_type_label, "_composition_by_sample.csv")),
          row.names = FALSE)

cat("\n✓ Saved:", paste0(DATASET, "_", cell_type_label, "_composition_by_sample.svg"), "\n")
cat("✓ Saved:", paste0(DATASET, "_", cell_type_label, "_composition_stats.csv"), "\n")

# ─────────────────────────────────────────────────────────────────────────────
# SUMMARY
# ─────────────────────────────────────────────────────────────────────────────

sig_states <- comp_stats %>% filter(p_adj < 0.05)

cat("\n================================================================================\n")
cat("SIGNIFICANT CHANGES (p_adj < 0.05):\n")
cat("================================================================================\n")

if (nrow(sig_states) > 0) {
    direction_text <- if (is_aging) {
        c("Up" = "increases with age", "Down" = "decreases with age")
    } else {
        c("Up" = "increases with disease", "Down" = "decreases with disease")
    }
    for (i in 1:nrow(sig_states)) {
        dir_desc <- direction_text[sig_states$direction[i]]
        cat(sprintf("  - %s %s (p_adj = %.2e)\n", sig_states$State[i], dir_desc, sig_states$p_adj[i]))
    }
} else {
    cat("  No significant changes detected\n")
}

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# SECTION 1: DATA PREPARATION
# ════════════════════════════════════════════════════════════════════════════════

cat("\n", paste(rep("=", 80), collapse = ""), "\n")
cat("SECTION 1: DATA PREPARATION\n")
cat(paste(rep("=", 80), collapse = ""), "\n")

# ─────────────────────────────────────────────────────────────────────────────
# STEP 1.1: CALCULATE %SnC PER STATE PER DONOR
# ─────────────────────────────────────────────────────────────────────────────

cat("\n[Step 1.1] Calculating %SnC per state per donor...\n")

snc_by_state_donor <- seurat_obj@meta.data %>%
    group_by(.data[[DONOR_COL]], .data[[STUDY_GROUP_COL]], .data[[SUBCLUSTER_COL]]) %>%
    summarise(
        n_total = n(),
        n_snc = sum(.data[[SENESCENCE_LABEL_COL]] == "SnC"),
        pct_snc = n_snc / n_total * 100,
        mean_log_umi = mean(log10(total_counts + 1)),
        .groups = "drop"
    )

colnames(snc_by_state_donor)[1:3] <- c("Donor", "Study_Group", "State")

snc_by_state_donor$Donor <- as.character(snc_by_state_donor$Donor)
snc_by_state_donor$Study_Group <- as.character(snc_by_state_donor$Study_Group)
snc_by_state_donor$State <- as.character(snc_by_state_donor$State)

cat("  Rows:", nrow(snc_by_state_donor), "\n")
cat("  Unique donors:", n_distinct(snc_by_state_donor$Donor), "\n")
cat("  Unique states:", n_distinct(snc_by_state_donor$State), "\n")

# ─────────────────────────────────────────────────────────────────────────────
# STEP 1.2: COMPLETE MISSING COMBINATIONS (fill with 0% SnC)
# ─────────────────────────────────────────────────────────────────────────────

cat("\n[Step 1.2] Completing missing donor-state combinations...\n")

donor_sg <- snc_by_state_donor %>%
    select(Donor, Study_Group) %>%
    distinct()

all_states <- unique(snc_by_state_donor$State)

complete_grid <- donor_sg %>%
    crossing(State = all_states)

snc_by_state_donor <- complete_grid %>%
    left_join(snc_by_state_donor %>% select(Donor, State, n_total, n_snc, pct_snc, mean_log_umi), 
              by = c("Donor", "State")) %>%
    mutate(
        n_total = replace_na(n_total, 0),
        n_snc = replace_na(n_snc, 0),
        pct_snc = replace_na(pct_snc, 0)
    )

cat("  Rows after completion:", nrow(snc_by_state_donor), "\n")
cat("  Expected:", n_distinct(donor_sg$Donor), "x", length(all_states), "=", 
    n_distinct(donor_sg$Donor) * length(all_states), "\n")

# ─────────────────────────────────────────────────────────────────────────────
# STEP 1.3: MERGE DONOR-LEVEL METADATA (Age, Sex, Cohort)
# ─────────────────────────────────────────────────────────────────────────────

cat("\n[Step 1.3] Merging donor-level metadata...\n")

available_cols <- colnames(seurat_obj@meta.data)

cols_to_extract <- c(DONOR_COL)

has_primary_var <- PRIMARY_VAR %in% available_cols
if (has_primary_var) {
    cols_to_extract <- c(cols_to_extract, PRIMARY_VAR)
    cat("  Found PRIMARY_VAR:", PRIMARY_VAR, "\n")
} else {
    cat("  WARNING: PRIMARY_VAR", PRIMARY_VAR, "not found in metadata\n")
}

has_sex <- SEX_COL %in% available_cols
if (has_sex) {
    cols_to_extract <- c(cols_to_extract, SEX_COL)
    cat("  Found SEX_COL:", SEX_COL, "\n")
}

has_cohort <- COHORT_COL %in% available_cols
if (has_cohort) {
    cols_to_extract <- c(cols_to_extract, COHORT_COL)
    cat("  Found COHORT_COL:", COHORT_COL, "\n")
}

donor_meta <- seurat_obj@meta.data %>%
    select(all_of(cols_to_extract)) %>%
    distinct()

colnames(donor_meta)[1] <- "Donor"
if (has_primary_var) colnames(donor_meta)[colnames(donor_meta) == PRIMARY_VAR] <- "PrimaryVar"
if (has_sex) colnames(donor_meta)[colnames(donor_meta) == SEX_COL] <- "Sex"
if (has_cohort) colnames(donor_meta)[colnames(donor_meta) == COHORT_COL] <- "Cohort"

donor_meta$Donor <- as.character(donor_meta$Donor)

cat("  Donor metadata rows:", nrow(donor_meta), "\n")
cat("  Donor metadata columns:", paste(colnames(donor_meta), collapse = ", "), "\n")

snc_by_state_donor <- snc_by_state_donor %>%
    left_join(donor_meta, by = "Donor")

cat("  Final columns:", paste(colnames(snc_by_state_donor), collapse = ", "), "\n")

# ─────────────────────────────────────────────────────────────────────────────
# STEP 1.4: CREATE NUMERIC PREDICTOR
# ─────────────────────────────────────────────────────────────────────────────

cat("\n[Step 1.4] Creating numeric predictor...\n")

cat("  Study type:", STUDY_TYPE, "\n")
cat("  Primary variable:", PRIMARY_VAR, "(", PRIMARY_VAR_TYPE, ")\n")

is_aging <- STUDY_TYPE == "aging"

if (PRIMARY_VAR_TYPE == "continuous" && has_primary_var) {
    snc_by_state_donor$Predictor <- as.numeric(snc_by_state_donor$PrimaryVar)
    predictor_desc <- paste0(PRIMARY_VAR, " (continuous)")
    cat("  Using continuous predictor:", PRIMARY_VAR, "\n")
    cat("  Range:", min(snc_by_state_donor$Predictor, na.rm = TRUE), "-", 
        max(snc_by_state_donor$Predictor, na.rm = TRUE), "\n")
} else {
    sg_order <- names(STUDY_GROUP_COLORS)[names(STUDY_GROUP_COLORS) %in% unique(snc_by_state_donor$Study_Group)]
    snc_by_state_donor$Study_Group <- factor(snc_by_state_donor$Study_Group, levels = sg_order)
    snc_by_state_donor$Predictor <- as.numeric(snc_by_state_donor$Study_Group)
    predictor_desc <- paste0("Study_Group (ordinal)")
    cat("  Using ordinal predictor from Study_Group\n")
    cat("  Levels:", paste(sg_order, collapse = ", "), "\n")
}

# ─────────────────────────────────────────────────────────────────────────────
# STEP 1.5: SETUP ORDERS AND LABELS
# ─────────────────────────────────────────────────────────────────────────────

cat("\n[Step 1.5] Setting up orders and labels...\n")

sg_detected <- unique(snc_by_state_donor$Study_Group)
sg_order <- names(STUDY_GROUP_COLORS)[names(STUDY_GROUP_COLORS) %in% sg_detected]

state_order <- snc_by_state_donor %>%
    group_by(State) %>%
    summarise(median_snc = median(pct_snc), .groups = "drop") %>%
    arrange(desc(median_snc)) %>%
    pull(State)

if (is_aging) {
    x_labels <- setNames(gsub("Age_", "", gsub("_", "-", sg_order)), sg_order)
    x_title <- "Age Group (years)"
} else {
    x_labels <- setNames(gsub("_", " ", sg_order), sg_order)
    x_title <- "Disease Status"
}

snc_by_state_donor$Study_Group <- factor(snc_by_state_donor$Study_Group, levels = sg_order)
snc_by_state_donor$State <- factor(snc_by_state_donor$State, levels = state_order)

cat("  Study groups:", paste(sg_order, collapse = ", "), "\n")
cat("  States (by median %SnC):", paste(state_order, collapse = ", "), "\n")

cat("\n[Section 1 Complete]\n")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# SECTION 2: STATISTICAL MODELING
# ════════════════════════════════════════════════════════════════════════════════

cat("\n", paste(rep("=", 80), collapse = ""), "\n")
cat("SECTION 2: STATISTICAL MODELING\n")
cat(paste(rep("=", 80), collapse = ""), "\n")

cell_type_label <- gsub("_state$", "", SUBCLUSTER_COL)

# ─────────────────────────────────────────────────────────────────────────────
# STEP 2.1: CHECK COVARIATES
# ─────────────────────────────────────────────────────────────────────────────

cat("\n[Step 2.1] Checking covariates...\n")

has_sex <- "Sex" %in% colnames(snc_by_state_donor)
has_cohort <- "Cohort" %in% colnames(snc_by_state_donor)
has_umi <- "mean_log_umi" %in% colnames(snc_by_state_donor)

cat("  Sex:", ifelse(has_sex, "FOUND", "NOT FOUND"), "\n")
cat("  Cohort:", ifelse(has_cohort, "FOUND", "NOT FOUND"), "\n")
cat("  Log UMI:", ifelse(has_umi, "FOUND", "NOT FOUND"), "\n")

# ─────────────────────────────────────────────────────────────────────────────
# STEP 2.2: BUILD FORMULA AND DESCRIPTOR STRINGS
# ─────────────────────────────────────────────────────────────────────────────

cat("\n[Step 2.2] Building model formula...\n")

covariates_used <- c()
if (has_sex) covariates_used <- c(covariates_used, "Sex")
if (has_cohort) covariates_used <- c(covariates_used, "Cohort")
if (has_umi) covariates_used <- c(covariates_used, "mean_log_umi")

if (length(covariates_used) == 0) {
    cov_part <- ""
    covariates_display <- "none"
} else {
    cov_part <- paste0(" + ", paste(covariates_used, collapse = " + "))
    covariates_display <- paste(covariates_used, collapse = ", ")
}

formula_lm <- paste0("pct_cuberoot ~ Predictor", cov_part)
model_formula <- as.formula(formula_lm)

if (is_aging) {
    predictor_desc <- "Age (continuous)"
    PRIMARY_VAR <- "Age"
    x_title <- "Age Group"
} else {
    predictor_desc <- "Disease stage (ordinal)"
    PRIMARY_VAR <- "Disease"
    x_title <- "Disease Stage"
}

cat("  Formula:", formula_lm, "\n")
cat("  Predictor:", predictor_desc, "\n")
cat("  Covariates:", covariates_display, "\n")
cat("  PRIMARY_VAR:", PRIMARY_VAR, "\n")

# ─────────────────────────────────────────────────────────────────────────────
# STEP 2.3: SET UP COLORS AND LABELS FOR VISUALIZATION
# ─────────────────────────────────────────────────────────────────────────────

cat("\n[Step 2.3] Setting up colors and labels...\n")

if (is_aging) {
    x_labels <- setNames(gsub("Age_", "", gsub("_", "-", sg_order)), sg_order)
} else {
    x_labels <- setNames(gsub("_", " ", sg_order), sg_order)
}

cat("  X-axis labels:", paste(x_labels, collapse = ", "), "\n")
cat("  Study group colors:", length(STUDY_GROUP_COLORS), "defined\n")

# ─────────────────────────────────────────────────────────────────────────────
# STEP 2.4: PREPARE DATA FOR MODELING
# ─────────────────────────────────────────────────────────────────────────────

cat("\n[Step 2.4] Preparing data for modeling...\n")

snc_model_data <- snc_by_state_donor %>%
    mutate(pct_cuberoot = (pct_snc / 100)^(1/3))

if (!"Predictor" %in% colnames(snc_model_data)) {
    cat("  Adding Predictor column...\n")
    if (is_aging) {
        snc_model_data <- snc_model_data %>%
            mutate(Predictor = PrimaryVar)
    } else {
        disease_order <- sg_order
        snc_model_data <- snc_model_data %>%
            mutate(Predictor = as.numeric(factor(Study_Group, levels = disease_order)) - 1)
    }
}

if (has_sex) snc_model_data$Sex <- as.factor(snc_model_data$Sex)
if (has_cohort) snc_model_data$Cohort <- as.factor(snc_model_data$Cohort)

cat("  Rows:", nrow(snc_model_data), "\n")
cat("  Predictor range:", min(snc_model_data$Predictor, na.rm = TRUE), "-", 
    max(snc_model_data$Predictor, na.rm = TRUE), "\n")
if (has_umi) {
    cat("  Log UMI range:", 
        sprintf("%.2f - %.2f", 
                min(snc_model_data$mean_log_umi, na.rm = TRUE),
                max(snc_model_data$mean_log_umi, na.rm = TRUE)), "\n")
}

# ─────────────────────────────────────────────────────────────────────────────
# STEP 2.5: RUN LINEAR MODEL PER STATE
# ─────────────────────────────────────────────────────────────────────────────

cat("\n[Step 2.5] Running linear models per state...\n")
cat("  Formula:", formula_lm, "\n\n")

results_list <- list()

for (state in state_order) {
    df_state <- snc_model_data %>% filter(State == state)
    n_donors <- n_distinct(df_state$Donor)
    
    model <- tryCatch(
        lm(model_formula, data = df_state),
        error = function(e) {
            cat(sprintf("    %s: MODEL FAILED — %s\n", state, e$message))
            return(NULL)
        }
    )
    
    if (is.null(model)) next
    
    coef_summary <- summary(model)$coefficients
    
    results_list[[state]] <- data.frame(
        State = state,
        n_donors = n_donors,
        estimate = coef_summary["Predictor", "Estimate"],
        std_error = coef_summary["Predictor", "Std. Error"],
        p_value = coef_summary["Predictor", "Pr(>|t|)"]
    )
    
    cat(sprintf("    %s (n=%d): b=%.4f, p=%.2e\n", 
                state, n_donors, 
                coef_summary["Predictor", "Estimate"],
                coef_summary["Predictor", "Pr(>|t|)"]))
}

# ─────────────────────────────────────────────────────────────────────────────
# STEP 2.6: MULTIPLE TESTING CORRECTION
# ─────────────────────────────────────────────────────────────────────────────

cat("\n[Step 2.6] Applying BH (FDR) correction...\n")

snc_state_stats <- do.call(rbind, results_list) %>%
    mutate(
        p_adj = p.adjust(p_value, method = "BH"),
        direction = case_when(
            estimate > 0 ~ "Up",
            estimate < 0 ~ "Down",
            TRUE ~ ""
        ),
        sig = case_when(
            p_adj < 0.001 ~ "***",
            p_adj < 0.01 ~ "**",
            p_adj < 0.05 ~ "*",
            TRUE ~ ""
        ),
        Significant = p_adj < 0.05
    ) %>%
    arrange(p_adj)

rownames(snc_state_stats) <- NULL

cat("\n  Results (BH-adjusted):\n")
print(snc_state_stats %>% select(State, n_donors, estimate, std_error, p_value, p_adj, direction, sig))

sig_results <- snc_state_stats %>% filter(Significant)

cat("\n  Significant states (FDR < 0.05):", nrow(sig_results), "of", nrow(snc_state_stats), "\n")

if (nrow(sig_results) > 0) {
    for (i in 1:nrow(sig_results)) {
        dir_text <- ifelse(sig_results$direction[i] == "Up",
                           paste0("increases with ", PRIMARY_VAR),
                           paste0("decreases with ", PRIMARY_VAR))
        cat(sprintf("    • %s: SnC %s (b=%.4f, p_adj=%.3f)\n",
                    sig_results$State[i], dir_text,
                    sig_results$estimate[i], sig_results$p_adj[i]))
    }
}

# ─────────────────────────────────────────────────────────────────────────────
# STEP 2.7: SAVE RESULTS
# ─────────────────────────────────────────────────────────────────────────────

cat("\n[Step 2.7] Saving results...\n")

write.csv(snc_state_stats, 
          file.path(RESULTS_DIR, paste0(DATASET, "_", cell_type_label, "_snc_state_stats.csv")),
          row.names = FALSE)

cat("  Saved:", paste0(DATASET, "_", cell_type_label, "_snc_state_stats.csv"), "\n")

write.csv(snc_by_state_donor %>%
              group_by(State, Study_Group) %>%
              summarise(
                  mean_pct = mean(pct_snc),
                  sd_pct = sd(pct_snc),
                  n = n(),
                  .groups = "drop"
              ),
          file.path(RESULTS_DIR, paste0(DATASET, "_", cell_type_label, "_snc_by_state_studygroup.csv")),
          row.names = FALSE)

cat("  Saved:", paste0(DATASET, "_", cell_type_label, "_snc_by_state_studygroup.csv"), "\n")

cat("\n[Section 2 Complete]\n")
cat("  Formula:", formula_lm, "\n")
cat("  Covariates:", covariates_display, "\n")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# SECTION 3: VISUALIZATION
# ════════════════════════════════════════════════════════════════════════════════

cat("\n", paste(rep("=", 80), collapse = ""), "\n")
cat("SECTION 3: VISUALIZATION\n")
cat(paste(rep("=", 80), collapse = ""), "\n")

cell_type_label <- gsub("_state$", "", SUBCLUSTER_COL)

# ─────────────────────────────────────────────────────────────────────────────
# STEP 3.1: DISPLAY SUMMARY TABLE
# ─────────────────────────────────────────────────────────────────────────────

cat("\n[Step 3.1] Summary table: Mean %SnC by State x Study Group\n\n")

snc_summary_display <- snc_by_state_donor %>%
    group_by(State, Study_Group) %>%
    summarise(
        mean_pct = mean(pct_snc),
        sd_pct = sd(pct_snc),
        n = n(),
        .groups = "drop"
    ) %>%
    mutate(display = sprintf("%.1f ± %.1f", mean_pct, sd_pct)) %>%
    select(State, Study_Group, display) %>%
    pivot_wider(names_from = Study_Group, values_from = display)

print(as.data.frame(snc_summary_display))

# ─────────────────────────────────────────────────────────────────────────────
# STEP 3.2: DISPLAY MODEL FORMULA AND RESULTS
# ─────────────────────────────────────────────────────────────────────────────

cat("\n[Step 3.2] Model specification\n")

cat("\n", paste(rep("-", 60), collapse = ""), "\n")
cat("MODEL FORMULA\n")
cat(paste(rep("-", 60), collapse = ""), "\n")

cat("\n  Response: pct_cuberoot = (pct_snc / 100)^(1/3)\n")
cat("  Formula:", formula_lm, "\n")
cat("  Predictor:", predictor_desc, "\n")
cat("  Covariates:", paste(covariates_used, collapse = ", "), "\n")
cat("  Multiple testing: BH (FDR) correction\n")

cat("\n", paste(rep("-", 60), collapse = ""), "\n")
cat("RESULTS\n")
cat(paste(rep("-", 60), collapse = ""), "\n\n")

results_display <- snc_state_stats %>%
    select(State, n_donors, estimate, std_error, p_value, p_adj, direction, sig) %>%
    mutate(
        estimate = sprintf("%.4f", estimate),
        std_error = sprintf("%.4f", std_error),
        p_value = sprintf("%.2e", p_value),
        p_adj = sprintf("%.3f", p_adj)
    )

print(as.data.frame(results_display))

if (nrow(sig_results) > 0) {
    cat("\nSignificant associations (FDR < 0.05):\n")
    for (i in 1:nrow(sig_results)) {
        dir_text <- ifelse(sig_results$direction[i] == "Up", 
                           paste0("increases with ", PRIMARY_VAR),
                           paste0("decreases with ", PRIMARY_VAR))
        cat(sprintf("  • %s: SnC %s (b=%s, p_adj=%s)\n", 
                    sig_results$State[i], 
                    dir_text,
                    sprintf("%.4f", sig_results$estimate[i]),
                    sprintf("%.3f", sig_results$p_adj[i])))
    }
} else {
    cat("\nNo significant associations (FDR < 0.05)\n")
}

# ─────────────────────────────────────────────────────────────────────────────
# STEP 3.3: CHECK MAX %SNC PER STATE (DONOR-LEVEL)
# ─────────────────────────────────────────────────────────────────────────────

cat("\n[Step 3.3] Checking max %SnC per state (donor-level)...\n\n")

state_y_stats <- snc_by_state_donor %>%
    group_by(State) %>%
    summarise(
        n_donors = n(),
        min_pct = min(pct_snc, na.rm = TRUE),
        max_pct = max(pct_snc, na.rm = TRUE),
        mean_pct = mean(pct_snc, na.rm = TRUE),
        median_pct = median(pct_snc, na.rm = TRUE),
        .groups = "drop"
    ) %>%
    arrange(desc(max_pct))

cat("  Max %SnC per state (donor-level):\n")
cat("  ", paste(rep("-", 60), collapse = ""), "\n")
cat(sprintf("  %-25s %8s %8s %8s\n", "State", "Min", "Max", "Mean"))
cat("  ", paste(rep("-", 60), collapse = ""), "\n")

for (i in 1:nrow(state_y_stats)) {
    cat(sprintf("  %-25s %7.1f%% %7.1f%% %7.1f%%\n", 
                state_y_stats$State[i],
                state_y_stats$min_pct[i],
                state_y_stats$max_pct[i],
                state_y_stats$mean_pct[i]))
}
cat("  ", paste(rep("-", 60), collapse = ""), "\n")

y_max_lookup <- state_y_stats %>%
    mutate(y_limit = max_pct * 1.15) %>%
    select(State, max_pct, y_limit)

cat("\n  Y-axis limits to use:\n")
for (i in 1:nrow(y_max_lookup)) {
    cat(sprintf("    %s: 0 - %.1f%% (max=%.1f%%)\n", 
                y_max_lookup$State[i],
                y_max_lookup$y_limit[i],
                y_max_lookup$max_pct[i]))
}

# ─────────────────────────────────────────────────────────────────────────────
# STEP 3.4: PREPARE PLOT DATA
# ─────────────────────────────────────────────────────────────────────────────

cat("\n[Step 3.4] Preparing plot data...\n")

sig_lookup <- setNames(snc_state_stats$sig, snc_state_stats$State)
dir_lookup <- setNames(snc_state_stats$direction, snc_state_stats$State)

facet_labels <- sapply(state_order, function(s) {
    sig <- sig_lookup[s]
    dir <- dir_lookup[s]
    if (!is.na(sig) && sig != "") {
        dir_symbol <- ifelse(dir == "Up", "(+)", "(-)")
        paste0(s, " ", dir_symbol, sig)
    } else {
        s
    }
})

label_map <- setNames(facet_labels, state_order)

# ─────────────────────────────────────────────────────────────────────────────
# STEP 3.5: CREATE INDIVIDUAL PLOTS PER STATE
# ─────────────────────────────────────────────────────────────────────────────

cat("\n[Step 3.5] Creating individual plots per state...\n")

suppressPackageStartupMessages({
    if (!require(cowplot, quietly = TRUE)) {
        install.packages("cowplot", quiet = TRUE)
        library(cowplot)
    }
})

plot_list <- list()

for (s in state_order) {
    
    state_data <- snc_by_state_donor %>% filter(State == s)
    state_stats <- snc_state_stats %>% filter(State == s)
    y_limit <- y_max_lookup %>% filter(State == s) %>% pull(y_limit)
    
    p_fmt <- if (state_stats$p_value < 0.001) {
        sprintf("%.1e", state_stats$p_value)
    } else if (state_stats$p_value < 0.01) {
        sprintf("%.3f", state_stats$p_value)
    } else {
        sprintf("%.2f", state_stats$p_value)
    }
    
    label_text <- paste0("b = ", sprintf("%.4f", state_stats$estimate), "\np = ", p_fmt)
    label_color <- ifelse(state_stats$Significant, "#333333", "#888888")
    label_face <- ifelse(state_stats$Significant, "bold", "plain")
    facet_title <- label_map[s]
    
    cat(sprintf("    %s: y_limit = %.1f%%\n", s, y_limit))
    
    p <- ggplot(state_data, aes(x = Study_Group, y = pct_snc)) +
        geom_boxplot(aes(fill = Study_Group), 
                     alpha = 0.7, outlier.shape = NA, width = 0.6,
                     linewidth = 0.5, color = "black") +
        geom_jitter(aes(color = Study_Group), 
                    width = 0.15, size = 1.2, alpha = 0.6) +
        geom_smooth(aes(x = as.numeric(Study_Group), y = pct_snc),
                    method = "lm", se = FALSE,
                    color = "black", linetype = "dashed", linewidth = 0.8) +
        annotate("text", x = 1, y = y_limit * 0.92, 
                 label = label_text, hjust = 0, vjust = 1,
                 size = 2.5, color = label_color, fontface = label_face) +
        coord_cartesian(ylim = c(0, y_limit)) +
        scale_fill_manual(values = STUDY_GROUP_COLORS) +
        scale_color_manual(values = STUDY_GROUP_COLORS) +
        scale_x_discrete(labels = x_labels) +
        labs(title = facet_title, x = NULL, y = NULL) +
        theme_minimal(base_size = 10) +
        theme(
            plot.title = element_text(size = 10, face = "bold", hjust = 0.5),
            axis.text.x = element_text(angle = 45, hjust = 1, vjust = 1, size = 8, 
                                       color = "black", face = "bold"),
            axis.text.y = element_text(size = 8, color = "black"),
            axis.line = element_line(color = "black", linewidth = 0.5),
            axis.ticks = element_line(color = "black", linewidth = 0.3),
            panel.grid = element_blank(),
            panel.border = element_rect(color = "black", fill = NA, linewidth = 0.5),
            legend.position = "none",
            plot.margin = margin(5, 5, 5, 5)
        )
    
    plot_list[[s]] <- p
}

# ─────────────────────────────────────────────────────────────────────────────
# STEP 3.6: COMBINE PLOTS
# ─────────────────────────────────────────────────────────────────────────────

cat("\n[Step 3.6] Combining plots...\n")

n_states <- length(state_order)
n_cols <- min(4, n_states)
n_rows <- ceiling(n_states / n_cols)

fig_width <- 3 * n_cols
fig_height <- 3 * n_rows

options(repr.plot.width = fig_width, repr.plot.height = fig_height)

p_combined <- plot_grid(
    plotlist = plot_list,
    ncol = n_cols,
    align = "hv"
)

p_final <- ggdraw() +
    draw_plot(p_combined, x = 0.03, y = 0.03, width = 0.95, height = 0.88) +
    draw_label(x_title, x = 0.5, y = 0.01, size = 12, fontface = "bold") +
    draw_label("Senescent Cells (%)", x = 0.01, y = 0.5, size = 12, fontface = "bold", angle = 90) +
    draw_label(paste0("Senescent Cell Proportion by ", tools::toTitleCase(cell_type_label), " State — ", DATASET), 
               x = 0.5, y = 0.97, size = 14, fontface = "bold") +
    draw_label(paste0("Model: ", predictor_desc, " + ", paste(covariates_used, collapse = " + ")),
               x = 0.5, y = 0.93, size = 10, color = "gray40")

print(p_final)

# ─────────────────────────────────────────────────────────────────────────────
# STEP 3.7: SAVE FIGURE
# ─────────────────────────────────────────────────────────────────────────────

cat("\n[Step 3.7] Saving figure...\n")

ggsave(file.path(FIGURES_DIR, paste0(DATASET, "_", cell_type_label, "_snc_by_state_boxplot.svg")), 
       p_final, width = fig_width, height = fig_height, dpi = 300)

cat("  Saved:", paste0(DATASET, "_", cell_type_label, "_snc_by_state_boxplot.svg"), "\n")

# ════════════════════════════════════════════════════════════════════════════════
# SUMMARY
# ════════════════════════════════════════════════════════════════════════════════

cat("\n", paste(rep("=", 80), collapse = ""), "\n")
cat("PLOT 4 COMPLETE\n")
cat(paste(rep("=", 80), collapse = ""), "\n")

cat("\nOutputs:\n")
cat("  ", file.path(RESULTS_DIR, paste0(DATASET, "_", cell_type_label, "_snc_state_stats.csv")), "\n")
cat("  ", file.path(RESULTS_DIR, paste0(DATASET, "_", cell_type_label, "_snc_by_state_studygroup.csv")), "\n")
cat("  ", file.path(FIGURES_DIR, paste0(DATASET, "_", cell_type_label, "_snc_by_state_boxplot.svg")), "\n")

cat("\n", paste(rep("=", 80), collapse = ""), "\n")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# STEP 1: DATA PREPARATION — SUBTYPE-LEVEL LOGISTIC GLMM
# ════════════════════════════════════════════════════════════════════════════════

cat("════════════════════════════════════════════════════════════════════════\n")
cat("STEP 1: DATA PREPARATION FOR SUBTYPE GLMM\n")
cat("════════════════════════════════════════════════════════════════════════\n\n")

library(lme4)

COL_DONOR      <- DONOR_COL
COL_SUBTYPE    <- SUBCLUSTER_COL
COL_SEX        <- SEX_COL
COL_COHORT     <- COHORT_COL
COL_SENESCENT  <- "is_senescent"
COL_STUDYGROUP <- STUDY_GROUP_COL

cat("▸ Extracting metadata from seurat_obj...\n")
cat(sprintf("  Study type: %s | Primary variable: %s (%s)\n",
            STUDY_TYPE, PRIMARY_VAR, PRIMARY_VAR_TYPE))

df_glmm <- seurat_obj@meta.data

if (!COL_SENESCENT %in% colnames(df_glmm)) {
    if (SENESCENCE_LABEL_COL %in% colnames(df_glmm)) {
        df_glmm$is_senescent <- as.integer(df_glmm[[SENESCENCE_LABEL_COL]] == "SnC")
        cat("  ✓ Created is_senescent from", SENESCENCE_LABEL_COL, "\n")
    } else {
        stop("No senescence column found.")
    }
} else {
    sen_vals <- df_glmm[[COL_SENESCENT]]
    if (is.character(sen_vals)) {
        df_glmm[[COL_SENESCENT]] <- as.integer(sen_vals %in% c("True", "TRUE", "SnC"))
        cat("  ✓ Converted senescence from character to 0/1\n")
    } else if (is.logical(sen_vals)) {
        df_glmm[[COL_SENESCENT]] <- as.integer(sen_vals)
        cat("  ✓ Converted senescence from logical to 0/1\n")
    }
}

required_cols <- c(COL_DONOR, COL_SUBTYPE, COL_SEX, COL_COHORT, COL_SENESCENT)
if (PRIMARY_VAR_TYPE == "continuous") {
    required_cols <- c(required_cols, PRIMARY_VAR)
} else {
    required_cols <- c(required_cols, COL_STUDYGROUP)
    if (PRIMARY_VAR %in% colnames(df_glmm)) {
        required_cols <- c(required_cols, PRIMARY_VAR)
    }
}

missing <- required_cols[!required_cols %in% colnames(df_glmm)]
if (length(missing) > 0) {
    stop(paste("Missing columns:", paste(missing, collapse = ", ")))
}

cat("▸ Cleaning data...\n")

df_glmm[[COL_DONOR]]   <- as.factor(df_glmm[[COL_DONOR]])
df_glmm[[COL_SUBTYPE]] <- as.factor(df_glmm[[COL_SUBTYPE]])
df_glmm[[COL_SEX]]     <- as.factor(df_glmm[[COL_SEX]])
df_glmm[[COL_COHORT]]  <- as.factor(df_glmm[[COL_COHORT]])

if (PRIMARY_VAR %in% colnames(df_glmm)) {
    df_glmm[[PRIMARY_VAR]] <- as.numeric(df_glmm[[PRIMARY_VAR]])
    df_glmm$Age_scaled <- df_glmm[[PRIMARY_VAR]] / 10
    cat(sprintf("  ✓ Age_scaled computed from %s\n", PRIMARY_VAR))
}

if (PRIMARY_VAR_TYPE == "categorical") {
    df_glmm[[COL_STUDYGROUP]] <- as.factor(df_glmm[[COL_STUDYGROUP]])
    if (!is.null(config$reference_group)) {
        df_glmm[[COL_STUDYGROUP]] <- relevel(df_glmm[[COL_STUDYGROUP]], ref = config$reference_group)
        cat(sprintf("  ✓ Reference group set to: %s\n", config$reference_group))
    }
}

has_counts <- "nCount_RNA" %in% colnames(df_glmm)
if (has_counts) {
    df_glmm$log10_total_counts <- log10(df_glmm$nCount_RNA + 1)
    cat("  ✓ log10_total_counts computed from nCount_RNA\n")
}

check_cols <- c(COL_DONOR, COL_SUBTYPE, COL_SEX, COL_COHORT, COL_SENESCENT)
if ("Age_scaled" %in% colnames(df_glmm)) check_cols <- c(check_cols, "Age_scaled")
if (PRIMARY_VAR_TYPE == "categorical") check_cols <- c(check_cols, COL_STUDYGROUP)

n_before <- nrow(df_glmm)
df_glmm <- df_glmm[complete.cases(df_glmm[, check_cols]), ]
n_after <- nrow(df_glmm)
if (n_before != n_after) {
    cat(sprintf("  ⚠ Dropped %d rows with missing values\n", n_before - n_after))
} else {
    cat(sprintf("  ✓ No missing values — all %s cells retained\n", format(n_after, big.mark = ",")))
}

if (PRIMARY_VAR_TYPE == "continuous") {
    fixed_parts <- "Age_scaled"
} else {
    fixed_parts <- COL_STUDYGROUP
    if ("Age_scaled" %in% colnames(df_glmm)) {
        fixed_parts <- c(fixed_parts, "Age_scaled")
    }
}

if (length(levels(df_glmm[[COL_SEX]])) > 1) fixed_parts <- c(fixed_parts, COL_SEX)
if (length(levels(df_glmm[[COL_COHORT]])) > 1) fixed_parts <- c(fixed_parts, COL_COHORT)
if (has_counts) fixed_parts <- c(fixed_parts, "log10_total_counts")

formula_str <- paste0(COL_SENESCENT, " ~ ", paste(fixed_parts, collapse = " + "),
                      " + (1|", COL_DONOR, ")")
formula_glmm <- as.formula(formula_str)

cat(sprintf("\n▸ Data ready: %s cells, %d donors\n",
            format(nrow(df_glmm), big.mark = ","),
            length(unique(df_glmm[[COL_DONOR]]))))
cat(sprintf("  SnC rate: %.2f%%\n", mean(df_glmm[[COL_SENESCENT]]) * 100))
cat(sprintf("  SnC cells: %s / %s\n",
            format(sum(df_glmm[[COL_SENESCENT]]), big.mark = ","),
            format(nrow(df_glmm), big.mark = ",")))
cat(sprintf("  Formula: %s\n", formula_str))

if (PRIMARY_VAR_TYPE == "categorical") {
    cat(sprintf("  Study groups: %s\n",
                paste(levels(df_glmm[[COL_STUDYGROUP]]), collapse = " vs ")))
}

cat(sprintf("\n▸ Subtypes (%d):\n", length(levels(df_glmm[[COL_SUBTYPE]]))))

for (st in levels(df_glmm[[COL_SUBTYPE]])) {
    sub <- df_glmm[df_glmm[[COL_SUBTYPE]] == st, ]
    n <- nrow(sub)
    nd <- length(unique(sub[[COL_DONOR]]))
    sr <- mean(sub[[COL_SENESCENT]]) * 100
    flag <- if (n < 100 | nd < 15) " ⚠ (may be skipped)" else ""
    cat(sprintf("  %-25s n=%6s  donors=%3d  SnC=%.1f%%%s\n",
                st, format(n, big.mark = ","), nd, sr, flag))
}

cat("\n════════════════════════════════════════════════════════════════════════\n")
cat("✓ STEP 1 COMPLETE — df_glmm and formula_glmm ready\n")
cat("════════════════════════════════════════════════════════════════════════\n")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# STEP 2: SUBTYPE-LEVEL LOGISTIC GLMM — MODEL FITTING
# ════════════════════════════════════════════════════════════════════════════════

cat("════════════════════════════════════════════════════════════════════════\n")
cat("STEP 2: SUBTYPE-LEVEL LOGISTIC GLMM\n")
cat("════════════════════════════════════════════════════════════════════════\n\n")

MIN_CELLS  <- 100
MIN_DONORS <- 15

extract_primary_effect <- function(coefs) {
    if (PRIMARY_VAR_TYPE == "continuous") {
        if ("Age_scaled" %in% rownames(coefs)) {
            return(list(
                term = "Age_scaled",
                beta = coefs["Age_scaled", "Estimate"],
                se   = coefs["Age_scaled", "Std. Error"],
                z    = coefs["Age_scaled", "z value"],
                p    = coefs["Age_scaled", "Pr(>|z|)"]
            ))
        }
    } else {
        sg_rows <- grep(COL_STUDYGROUP, rownames(coefs))
        if (length(sg_rows) > 0) {
            p_vals <- coefs[sg_rows, "Pr(>|z|)"]
            best <- sg_rows[which.min(p_vals)]
            return(list(
                term = rownames(coefs)[best],
                beta = coefs[best, "Estimate"],
                se   = coefs[best, "Std. Error"],
                z    = coefs[best, "z value"],
                p    = coefs[best, "Pr(>|z|)"]
            ))
        }
    }
    return(NULL)
}

# ═══════════════════════════════════════════════════════════════════════════════
# OVERALL GLMM
# ═══════════════════════════════════════════════════════════════════════════════

cat("─────────────────────────────────────────────────────────────────────\n")
cat("1. OVERALL GLMM (ALL CELLS)\n")
cat("─────────────────────────────────────────────────────────────────────\n")

cat(sprintf("\n▸ Formula: %s\n", deparse(formula_glmm)))
cat("▸ Fitting overall model...\n\n")

overall_model <- tryCatch({
    glmer(
        formula_glmm,
        data = df_glmm,
        family = binomial(link = "logit"),
        control = glmerControl(optimizer = "bobyqa", optCtrl = list(maxfun = 100000)),
        nAGQ = 1
    )
}, error = function(e) {
    cat(sprintf("✗ Overall model failed: %s\n", e$message))
    NULL
})

if (!is.null(overall_model)) {
    overall_coefs <- summary(overall_model)$coefficients
    re_var <- as.numeric(VarCorr(overall_model)[[COL_DONOR]])

    cat("✓ Model converged\n")
    cat(sprintf("  Observations: %s  |  Donors: %d  |  AIC: %.1f\n",
                format(nobs(overall_model), big.mark = ","),
                ngrps(overall_model),
                AIC(overall_model)))

    cat(sprintf("\n  %-30s %10s %10s %10s %12s %8s\n",
                "Term", "B", "SE", "z", "p-value", "OR"))
    cat(paste0("  ", paste(rep("-", 85), collapse = ""), "\n"))

    for (i in seq_len(nrow(overall_coefs))) {
        term <- rownames(overall_coefs)[i]
        beta <- overall_coefs[i, "Estimate"]
        se   <- overall_coefs[i, "Std. Error"]
        z    <- overall_coefs[i, "z value"]
        p    <- overall_coefs[i, "Pr(>|z|)"]
        or   <- exp(beta)
        sig  <- ifelse(p < 0.05, "*", "")
        cat(sprintf("  %-30s %10.4f %10.4f %10.3f %12.2e %8.3f %s\n",
                    term, beta, se, z, p, or, sig))
    }

    cat(sprintf("\n  Random Effects: Donor variance = %.4f, SD = %.4f\n",
                re_var, sqrt(re_var)))

    primary_eff <- extract_primary_effect(overall_coefs)
    if (!is.null(primary_eff)) {
        or_val <- exp(primary_eff$beta)
        or_lo  <- exp(primary_eff$beta - 1.96 * primary_eff$se)
        or_hi  <- exp(primary_eff$beta + 1.96 * primary_eff$se)

        if (PRIMARY_VAR_TYPE == "continuous") {
            cat(sprintf("\n▸ Age Effect (per decade):\n"))
        } else {
            cat(sprintf("\n▸ Disease Effect (%s):\n", primary_eff$term))
        }
        cat(sprintf("  B = %.4f (SE = %.4f)\n", primary_eff$beta, primary_eff$se))
        cat(sprintf("  OR = %.3f (95%% CI: %.3f - %.3f)\n", or_val, or_lo, or_hi))
        cat(sprintf("  p = %.2e\n", primary_eff$p))

        if (primary_eff$p < 0.05) {
            direction <- ifelse(primary_eff$beta > 0, "INCREASES", "DECREASES")
            pct <- abs((or_val - 1) * 100)
            cat(sprintf("  ✓ Senescence probability %s (+%.1f%% odds)\n", direction, pct))
        }
    }

    if (PRIMARY_VAR_TYPE == "categorical") {
        sg_rows <- grep(COL_STUDYGROUP, rownames(overall_coefs))
        if (length(sg_rows) > 1) {
            cat(sprintf("\n▸ All disease group effects (vs %s):\n", config$reference_group))
            for (i in sg_rows) {
                term <- gsub(COL_STUDYGROUP, "", rownames(overall_coefs)[i])
                beta <- overall_coefs[i, "Estimate"]
                or   <- exp(beta)
                p    <- overall_coefs[i, "Pr(>|z|)"]
                sig  <- ifelse(p < 0.05, "*", "")
                cat(sprintf("  %s: B=%.4f, OR=%.3f, p=%.2e %s\n", term, beta, or, p, sig))
            }
        }
    }
}

# ═══════════════════════════════════════════════════════════════════════════════
# SUBTYPE-SPECIFIC GLMM
# ═══════════════════════════════════════════════════════════════════════════════

cat("\n─────────────────────────────────────────────────────────────────────\n")
cat("2. SUBTYPE-SPECIFIC GLMM\n")
cat("─────────────────────────────────────────────────────────────────────\n")

subtypes <- levels(df_glmm[[COL_SUBTYPE]])
cat(sprintf("\n▸ Running GLMM for %d subtypes...\n\n", length(subtypes)))

results_list <- list()

for (subtype in subtypes) {

    sub_data <- df_glmm[df_glmm[[COL_SUBTYPE]] == subtype, ]

    n_cells  <- nrow(sub_data)
    n_donors <- length(unique(sub_data[[COL_DONOR]]))
    snc_rate <- mean(sub_data[[COL_SENESCENT]])

    if (n_cells < MIN_CELLS) {
        cat(sprintf("  %-25s SKIPPED (n=%d < %d)\n", subtype, n_cells, MIN_CELLS))
        next
    }
    if (n_donors < MIN_DONORS) {
        cat(sprintf("  %-25s SKIPPED (donors=%d < %d)\n", subtype, n_donors, MIN_DONORS))
        next
    }
    if (snc_rate == 0 | snc_rate == 1) {
        cat(sprintf("  %-25s SKIPPED (no variance)\n", subtype))
        next
    }

    model <- tryCatch({
        glmer(
            formula_glmm,
            data = sub_data,
            family = binomial(link = "logit"),
            control = glmerControl(optimizer = "bobyqa", optCtrl = list(maxfun = 50000)),
            nAGQ = 1
        )
    }, error = function(e) {
        cat(sprintf("  %-25s FAILED - %s\n", subtype, substr(e$message, 1, 60)))
        NULL
    })

    if (!is.null(model)) {
        coefs <- summary(model)$coefficients
        re_var <- as.numeric(VarCorr(model)[[COL_DONOR]])

        if (PRIMARY_VAR_TYPE == "categorical") {
            sg_rows <- grep(COL_STUDYGROUP, rownames(coefs))
            for (idx in sg_rows) {
                term_name <- gsub(COL_STUDYGROUP, "", rownames(coefs)[idx])
                beta <- coefs[idx, "Estimate"]
                se   <- coefs[idx, "Std. Error"]
                z    <- coefs[idx, "z value"]
                p    <- coefs[idx, "Pr(>|z|)"]

                results_list[[paste0(subtype, "_", term_name)]] <- data.frame(
                    Subtype        = subtype,
                    Comparison     = paste0(term_name, " vs ", config$reference_group),
                    N_cells        = n_cells,
                    N_donors       = n_donors,
                    SnC_rate       = snc_rate,
                    Beta           = beta,
                    SE             = se,
                    Beta_CI_lower  = beta - 1.96 * se,
                    Beta_CI_upper  = beta + 1.96 * se,
                    OR             = exp(beta),
                    OR_CI_lower    = exp(beta - 1.96 * se),
                    OR_CI_upper    = exp(beta + 1.96 * se),
                    z_value        = z,
                    P_value        = p,
                    Donor_variance = re_var,
                    AIC            = AIC(model),
                    stringsAsFactors = FALSE
                )
            }
            best_idx <- sg_rows[which.min(coefs[sg_rows, "Pr(>|z|)"])]
            beta <- coefs[best_idx, "Estimate"]
            p    <- coefs[best_idx, "Pr(>|z|)"]
            term <- gsub(COL_STUDYGROUP, "", rownames(coefs)[best_idx])
            sig  <- ifelse(p < 0.05, "*", "")
            cat(sprintf("  %-25s %s: B=%.4f  OR=%.3f  p=%.2e %s\n",
                        subtype, term, beta, exp(beta), p, sig))

        } else {
            primary_eff <- extract_primary_effect(coefs)
            if (!is.null(primary_eff)) {
                beta <- primary_eff$beta
                se   <- primary_eff$se
                z    <- primary_eff$z
                p    <- primary_eff$p
            } else {
                beta <- NA; se <- NA; z <- NA; p <- NA
            }

            results_list[[subtype]] <- data.frame(
                Subtype        = subtype,
                Comparison     = "per decade",
                N_cells        = n_cells,
                N_donors       = n_donors,
                SnC_rate       = snc_rate,
                Beta           = beta,
                SE             = se,
                Beta_CI_lower  = beta - 1.96 * se,
                Beta_CI_upper  = beta + 1.96 * se,
                OR             = exp(beta),
                OR_CI_lower    = exp(beta - 1.96 * se),
                OR_CI_upper    = exp(beta + 1.96 * se),
                z_value        = z,
                P_value        = p,
                Donor_variance = re_var,
                AIC            = AIC(model),
                stringsAsFactors = FALSE
            )

            sig <- ifelse(p < 0.05, "*", "")
            cat(sprintf("  %-25s B=%.4f  OR=%.3f  p=%.2e %s\n",
                        subtype, beta, exp(beta), p, sig))
        }
    }
}

# ─────────────────────────────────────────────────────────────────────────────
# FDR correction
# ─────────────────────────────────────────────────────────────────────────────

df_glmm_results <- do.call(rbind, results_list)
rownames(df_glmm_results) <- NULL

if (nrow(df_glmm_results) > 0) {

    df_glmm_results$P_adj <- p.adjust(df_glmm_results$P_value, method = "BH")
    df_glmm_results$Significant <- df_glmm_results$P_adj < 0.05
    df_glmm_results <- df_glmm_results[order(df_glmm_results$P_value), ]

    n_sig <- sum(df_glmm_results$Significant)
    cat(sprintf("\n▸ Results: %d / %d significant (FDR < 0.05)\n",
                n_sig, nrow(df_glmm_results)))

    cat(sprintf("\n%-25s %-20s %7s %6s %8s %20s %6s %10s %10s %4s\n",
                "Subtype", "Comparison", "N", "%SnC", "B", "95% CI (B)", "OR",
                "P", "FDR", "Sig"))
    cat(paste(rep("-", 130), collapse = ""), "\n")

    for (i in seq_len(nrow(df_glmm_results))) {
        row <- df_glmm_results[i, ]
        sig_mark <- ifelse(row$Significant, "Y", "")
        ci_str <- sprintf("[%.3f, %.3f]", row$Beta_CI_lower, row$Beta_CI_upper)
        cat(sprintf("%-25s %-20s %7s %5.1f%% %8.4f %20s %6.2f %10.2e %10.2e %4s\n",
                    row$Subtype,
                    row$Comparison,
                    format(row$N_cells, big.mark = ","),
                    row$SnC_rate * 100,
                    row$Beta,
                    ci_str,
                    row$OR,
                    row$P_value,
                    row$P_adj,
                    sig_mark))
    }

    results_file <- file.path(RESULTS_DIR, paste0(DATASET, "_", cell_type_label, "_subtype_glmm_results.csv"))
    write.csv(df_glmm_results, results_file, row.names = FALSE)
    cat(sprintf("\n✓ Saved: %s\n", results_file))

    if (PRIMARY_VAR_TYPE == "continuous") {
        var_label <- "Age"; unit_label <- "per decade"
    } else {
        var_label <- "Disease"; unit_label <- paste0("vs ", config$reference_group)
    }

    sig_up   <- df_glmm_results[df_glmm_results$Significant & df_glmm_results$Beta > 0, ]
    sig_down <- df_glmm_results[df_glmm_results$Significant & df_glmm_results$Beta < 0, ]

    if (nrow(sig_up) > 0) {
        cat(sprintf("\n▸ INCREASING senescence (%s):\n", var_label))
        for (i in seq_len(nrow(sig_up))) {
            row <- sig_up[i, ]
            pct <- (row$OR - 1) * 100
            cat(sprintf("   %s [%s]: OR=%.2f (+%.1f%% %s, FDR=%.2e)\n",
                        row$Subtype, row$Comparison, row$OR, pct, unit_label, row$P_adj))
        }
    }

    if (nrow(sig_down) > 0) {
        cat(sprintf("\n▸ DECREASING senescence (%s):\n", var_label))
        for (i in seq_len(nrow(sig_down))) {
            row <- sig_down[i, ]
            pct <- (1 - row$OR) * 100
            cat(sprintf("   %s [%s]: OR=%.2f (-%.1f%% %s, FDR=%.2e)\n",
                        row$Subtype, row$Comparison, row$OR, pct, unit_label, row$P_adj))
        }
    }

    if (nrow(sig_up) == 0 & nrow(sig_down) == 0) {
        cat("\n▸ No significant associations at FDR < 0.05\n")
    }

} else {
    cat("\n⚠ No subtypes passed filters\n")
}

cat("\n════════════════════════════════════════════════════════════════════════\n")
cat("✓ STEP 2 COMPLETE — df_glmm_results ready for plotting\n")
cat("════════════════════════════════════════════════════════════════════════\n")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# STEP 3: FOREST PLOT (TABLE STYLE) — SUBTYPE GLMM
# ════════════════════════════════════════════════════════════════════════════════

cat("════════════════════════════════════════════════════════════════════════\n")
cat("STEP 3: FOREST PLOT — SUBTYPE GLMM\n")
cat("════════════════════════════════════════════════════════════════════════\n\n")

if (nrow(df_glmm_results) == 0) {
    cat("No results to plot.\n")
} else {

    df_plot <- df_glmm_results[order(df_glmm_results$OR, decreasing = TRUE), ]
    n_rows <- nrow(df_plot)

    fig_width  <- 7
    fig_height <- max(3, n_rows * 0.5 + 2)

    x_sub  <- 0.01
    x_fl   <- 0.19
    x_fr   <- 0.58
    x_or   <- 0.72
    x_fdr  <- 0.93

    betas <- df_plot$Beta
    iqr <- IQR(betas)
    center <- median(betas)
    display_min <- min(betas) - max(iqr * 1.5, 0.15)
    display_max <- max(betas) + max(iqr * 1.5, 0.15)

    beta_to_x <- function(val) {
        v <- pmax(pmin(val, display_max), display_min)
        x_fl + (v - display_min) / (display_max - display_min) * (x_fr - x_fl)
    }

    x_null <- beta_to_x(0)

    cands <- c(0.5, 0.6, 0.7, 0.75, 0.8, 0.85, 0.9, 0.95,
               1.0, 1.05, 1.1, 1.15, 1.2, 1.3, 1.5, 2.0)
    valid <- cands[cands >= exp(display_min) & cands <= exp(display_max)]

    min_sp <- (x_fr - x_fl) * 0.08
    ticks <- valid[1]
    for (t in valid[-1]) {
        if (abs(beta_to_x(log(t)) - beta_to_x(log(ticks[length(ticks)]))) >= min_sp) {
            ticks <- c(ticks, t)
        }
    }

    draw_forest <- function() {

        par(mar = c(1.8, 0.3, 1.8, 0.3), family = "sans")
        plot.new()
        plot.window(xlim = c(0, 1), ylim = c(-1, n_rows + 0.8))

        text(0.5, n_rows + 0.5,
             paste0(tools::toTitleCase(cell_type_label),
                    " Subtype — Age vs Senescence (", DATASET, ")"),
             cex = 0.8, font = 2, adj = 0.5)

        yh <- n_rows + 0.05
        text(x_sub, yh, "Subtype", cex = 0.65, font = 2, adj = 0)
        text((x_fl + x_fr) / 2, yh, "log-OR", cex = 0.65, font = 2, adj = 0.5)
        text(x_or, yh, "OR [95% CI]", cex = 0.65, font = 2, adj = 0.5)
        text(x_fdr, yh, "FDR", cex = 0.65, font = 2, adj = 0.5)
        segments(0, yh - 0.15, 0.99, yh - 0.15, lwd = 0.6)

        segments(x_null, -0.3, x_null, n_rows - 0.5,
                 lty = 2, col = "#aaaaaa", lwd = 0.5)

        for (i in seq_len(n_rows)) {
            row <- df_plot[i, ]
            y <- n_rows - i

            sig_fdr <- row$Significant
            sig_p   <- row$P_value < 0.05

            if (i %% 2 == 0) {
                rect(0, y - 0.28, 0.99, y + 0.28,
                     col = "#fafafa", border = NA)
            }

            lbl <- as.character(row$Subtype)
            if (sig_fdr) lbl <- paste0("* ", lbl)
            text(x_sub, y, lbl,
                 cex = 0.62, font = ifelse(sig_p, 2, 1), adj = 0)

            cc <- state_colors[row$Subtype]
            if (is.na(cc)) cc <- "#808080"

            ci_l <- row$Beta_CI_lower
            ci_h <- row$Beta_CI_upper
            xl <- beta_to_x(ci_l)
            xh <- beta_to_x(ci_h)
            xb <- beta_to_x(row$Beta)

            clip_l <- ci_l < display_min
            clip_r <- ci_h > display_max

            segments(xl, y, xh, y, col = cc, lwd = 1.8, lend = 1)

            if (clip_l) {
                polygon(x = c(xl, xl + 0.008, xl + 0.008),
                        y = c(y, y + 0.08, y - 0.08),
                        col = cc, border = NA)
            } else {
                segments(xl, y - 0.08, xl, y + 0.08, col = cc, lwd = 0.8)
            }

            if (clip_r) {
                polygon(x = c(xh, xh - 0.008, xh - 0.008),
                        y = c(y, y + 0.08, y - 0.08),
                        col = cc, border = NA)
            } else {
                segments(xh, y - 0.08, xh, y + 0.08, col = cc, lwd = 0.8)
            }

            dw <- 0.005; dh <- 0.13
            polygon(x = c(xb - dw, xb, xb + dw, xb),
                    y = c(y, y - dh, y, y + dh),
                    col = cc, border = "black", lwd = 0.4)

            text(x_or, y + 0.09,
                 sprintf("%.2f", row$OR),
                 cex = 0.58, adj = 0.5,
                 font = ifelse(sig_p, 2, 1))
            text(x_or, y - 0.09,
                 sprintf("[%.2f-%.2f]", row$OR_CI_lower, row$OR_CI_upper),
                 cex = 0.50, adj = 0.5, col = "#777777")

            pv <- row$P_adj
            if (pv < 0.001) {
                ptxt <- formatC(pv, format = "e", digits = 1)
            } else if (pv < 0.01) {
                ptxt <- formatC(pv, format = "f", digits = 3)
            } else {
                ptxt <- formatC(pv, format = "f", digits = 2)
            }
            text(x_fdr, y, ptxt,
                 cex = 0.58, adj = 0.5,
                 font = ifelse(sig_fdr, 2, 1),
                 col = ifelse(sig_fdr, "#C44E52", "#333333"))
        }

        ya <- -0.45
        segments(x_fl, ya, x_fr, ya, lwd = 0.6)

        for (t in ticks) {
            xt <- beta_to_x(log(t))
            segments(xt, ya, xt, ya - 0.06, lwd = 0.5)
            text(xt, ya - 0.15, t, cex = 0.5, adj = 0.5)
        }

        text((x_fl + x_fr) / 2, ya - 0.35,
             "Odds Ratio", cex = 0.55, font = 2, adj = 0.5)

        segments(0, ya - 0.5, 0.99, ya - 0.5, lwd = 0.3, col = "#dddddd")
        text(x_sub, ya - 0.7, "* FDR < 0.05",
             cex = 0.48, adj = 0, col = "#999999")

        if (any(df_plot$Beta_CI_lower < display_min) |
            any(df_plot$Beta_CI_upper > display_max)) {
            text(x_fr, ya - 0.7, "< > CI clipped",
                 cex = 0.48, adj = 1, col = "#999999")
        }
    }

    options(repr.plot.width = fig_width, repr.plot.height = fig_height)
    draw_forest()

    forest_file <- file.path(FIGURES_DIR,
                             paste0(DATASET, "_", cell_type_label, "_subtype_glmm_forest.svg"))
    svg(forest_file, width = fig_width, height = fig_height)
    draw_forest()
    dev.off()

    cat("Saved:", forest_file, "\n")
}

cat("\n════════════════════════════════════════════════════════════════════════\n")
cat("STEP 3 COMPLETE\n")
cat("════════════════════════════════════════════════════════════════════════\n")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# PLOT 3: UMAP PER STUDY GROUP
# ════════════════════════════════════════════════════════════════════════════════

cat("\n--- Plot 3: UMAP per Study Group ---\n")

cell_type_label <- gsub("_state$", "", SUBCLUSTER_COL)
is_aging <- STUDY_TYPE == "aging"

# ─────────────────────────────────────────────────────────────────────────────
# SETUP
# ─────────────────────────────────────────────────────────────────────────────

sg_detected <- unique(seurat_obj@meta.data[[STUDY_GROUP_COL]])
sg_order <- names(STUDY_GROUP_COLORS)[names(STUDY_GROUP_COLORS) %in% sg_detected]

if (is_aging) {
    sg_labels <- setNames(gsub("Age_", "", gsub("_", "-", sg_order)), sg_order)
} else {
    sg_labels <- setNames(gsub("_", " ", sg_order), sg_order)
}

cat("Study groups:", paste(sg_order, collapse = ", "), "\n")

state_order <- seurat_obj@meta.data %>%
    count(.data[[SUBCLUSTER_COL]]) %>%
    arrange(desc(n)) %>%
    pull(.data[[SUBCLUSTER_COL]])

state_colors <- get_state_colors(state_order)

cat("States:", paste(state_order, collapse = ", "), "\n")

# ─────────────────────────────────────────────────────────────────────────────
# HELPER FUNCTIONS
# ─────────────────────────────────────────────────────────────────────────────

add_corner_arrows <- function(p, label_size = 3) {
    build <- ggplot_build(p)
    x_range <- build$layout$panel_params[[1]]$x.range
    y_range <- build$layout$panel_params[[1]]$y.range
    
    x_start <- x_range[1] + diff(x_range) * 0.02
    y_start <- y_range[1] + diff(y_range) * 0.02
    x_arrow <- diff(x_range) * 0.12
    y_arrow <- diff(y_range) * 0.12
    
    p + 
        annotate("segment", x = x_start, xend = x_start + x_arrow, y = y_start, yend = y_start,
                 arrow = arrow(length = unit(0.12, "cm"), type = "closed"), linewidth = 0.4) +
        annotate("text", x = x_start + x_arrow/2, y = y_start - diff(y_range) * 0.04,
                 label = "UMAP1", size = label_size, hjust = 0.5, vjust = 1, fontface = "bold") +
        annotate("segment", x = x_start, xend = x_start, y = y_start, yend = y_start + y_arrow,
                 arrow = arrow(length = unit(0.12, "cm"), type = "closed"), linewidth = 0.4) +
        annotate("text", x = x_start - diff(x_range) * 0.04, y = y_start + y_arrow/2,
                 label = "UMAP2", size = label_size, hjust = 1, vjust = 0.5, angle = 90, fontface = "bold") +
        coord_cartesian(clip = "off")
}

umap_theme <- theme_void(base_size = 12) +
    theme(
        plot.title = element_text(size = 13, face = "bold", hjust = 0.5),
        legend.position = "none",
        plot.margin = margin(15, 15, 20, 20)
    )

# ─────────────────────────────────────────────────────────────────────────────
# GENERATE UMAPS
# ─────────────────────────────────────────────────────────────────────────────

umap_list <- list()

for (sg in sg_order) {
    cells_sg <- colnames(seurat_obj)[seurat_obj@meta.data[[STUDY_GROUP_COL]] == sg]
    n_cells <- length(cells_sg)
    sg_label <- sg_labels[sg]
    
    p <- DimPlot(seurat_obj, reduction = "umap", cells = cells_sg,
                 group.by = SUBCLUSTER_COL, pt.size = 0.3) +
        scale_color_manual(values = state_colors) +
        ggtitle(paste0(sg_label, "\n(n=", format(n_cells, big.mark = ","), ")")) +
        umap_theme
    
    p <- add_corner_arrows(p, label_size = 2.5)
    umap_list[[sg]] <- p
}

# ─────────────────────────────────────────────────────────────────────────────
# SHARED LEGEND
# ─────────────────────────────────────────────────────────────────────────────

p_legend <- DimPlot(seurat_obj, reduction = "umap", group.by = SUBCLUSTER_COL, pt.size = 0.5) +
    scale_color_manual(values = state_colors) +
    theme_void() +
    theme(
        legend.position = "right",
        legend.title = element_text(size = 12, face = "bold"),
        legend.text = element_text(size = 10),
        legend.key.size = unit(0.5, "cm")
    ) +
    labs(color = paste0(tools::toTitleCase(cell_type_label), " State")) +
    guides(color = guide_legend(ncol = 1, override.aes = list(size = 4)))

legend <- cowplot::get_legend(p_legend)

# ─────────────────────────────────────────────────────────────────────────────
# COMBINE
# ─────────────────────────────────────────────────────────────────────────────

n_groups <- length(sg_order)
n_cols <- 4
n_rows <- ceiling(n_groups / n_cols)

while (length(umap_list) < n_rows * n_cols) {
    umap_list[[length(umap_list) + 1]] <- plot_spacer()
}

options(repr.plot.width = 16, repr.plot.height = 4 * n_rows + 1)

umap_grid <- wrap_plots(umap_list, ncol = n_cols)

title_text <- if (is_aging) {
    paste0(tools::toTitleCase(cell_type_label), " States by Age Group")
} else {
    paste0(tools::toTitleCase(cell_type_label), " States by Disease Group")
}

umap_panel_sg <- (umap_grid | legend) + 
    plot_layout(widths = c(1, 0.15)) +
    plot_annotation(
        title = title_text,
        theme = theme(plot.title = element_text(size = 16, face = "bold", hjust = 0.5))
    )

print(umap_panel_sg)

ggsave(file.path(FIGURES_DIR, paste0(DATASET, "_", cell_type_label, "_umap_by_studygroup.svg")), 
       umap_panel_sg, width = 16, height = 4 * n_rows + 1, dpi = 300)

cat("\n✓ Saved:", paste0(DATASET, "_", cell_type_label, "_umap_by_studygroup.svg"), "\n")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# PLOT: UMAP PER STUDY GROUP (SnC HIGHLIGHT)
# ════════════════════════════════════════════════════════════════════════════════

cat("\n--- Plot: UMAP per Study Group (SnC Highlight) ---\n")

cell_type_label <- gsub("_state$", "", SUBCLUSTER_COL)
is_aging <- STUDY_TYPE == "aging"

sg_detected <- unique(seurat_obj@meta.data[[STUDY_GROUP_COL]])
sg_order <- names(STUDY_GROUP_COLORS)[names(STUDY_GROUP_COLORS) %in% sg_detected]

if (is_aging) {
    sg_labels <- setNames(gsub("Age_", "", gsub("_", "-", sg_order)), sg_order)
} else {
    sg_labels <- setNames(gsub("_", " ", sg_order), sg_order)
}

cat("Study groups:", paste(sg_order, collapse = ", "), "\n")
cat("Senescence column:", SENESCENCE_LABEL_COL, "\n")

add_corner_arrows <- function(p, label_size = 3) {
    build <- ggplot_build(p)
    x_range <- build$layout$panel_params[[1]]$x.range
    y_range <- build$layout$panel_params[[1]]$y.range
    
    x_start <- x_range[1] + diff(x_range) * 0.02
    y_start <- y_range[1] + diff(y_range) * 0.02
    x_arrow <- diff(x_range) * 0.12
    y_arrow <- diff(y_range) * 0.12
    
    p + 
        annotate("segment", x = x_start, xend = x_start + x_arrow, y = y_start, yend = y_start,
                 arrow = arrow(length = unit(0.12, "cm"), type = "closed"), linewidth = 0.4) +
        annotate("text", x = x_start + x_arrow/2, y = y_start - diff(y_range) * 0.04,
                 label = "UMAP1", size = label_size, hjust = 0.5, vjust = 1, fontface = "bold") +
        annotate("segment", x = x_start, xend = x_start, y = y_start, yend = y_start + y_arrow,
                 arrow = arrow(length = unit(0.12, "cm"), type = "closed"), linewidth = 0.4) +
        annotate("text", x = x_start - diff(x_range) * 0.04, y = y_start + y_arrow/2,
                 label = "UMAP2", size = label_size, hjust = 1, vjust = 0.5, angle = 90, fontface = "bold") +
        coord_cartesian(clip = "off")
}

umap_theme <- theme_void(base_size = 12) +
    theme(
        plot.title = element_text(size = 13, face = "bold", hjust = 0.5),
        legend.position = "none",
        plot.margin = margin(15, 15, 20, 20)
    )

umap_list <- list()

for (sg in sg_order) {
    cells_sg <- colnames(seurat_obj)[seurat_obj@meta.data[[STUDY_GROUP_COL]] == sg]
    n_cells <- length(cells_sg)
    n_snc <- sum(seurat_obj@meta.data[cells_sg, SENESCENCE_LABEL_COL] == "SnC")
    sg_label <- sg_labels[sg]
    
    p <- DimPlot(seurat_obj, reduction = "umap", cells = cells_sg,
                 group.by = SENESCENCE_LABEL_COL, pt.size = 0.3,
                 order = c("SnC", "Non-SnC")) +
        scale_color_manual(values = SENESCENCE_COLORS) +
        ggtitle(paste0(sg_label, "\n(n=", format(n_cells, big.mark = ","), 
                       ", SnC=", format(n_snc, big.mark = ","), ")")) +
        umap_theme
    
    p <- add_corner_arrows(p, label_size = 2.5)
    umap_list[[sg]] <- p
}

p_legend <- DimPlot(seurat_obj, reduction = "umap", group.by = SENESCENCE_LABEL_COL, 
                    pt.size = 0.5, order = c("SnC", "Non-SnC")) +
    scale_color_manual(values = SENESCENCE_COLORS) +
    theme_void() +
    theme(
        legend.position = "right",
        legend.title = element_text(size = 12, face = "bold"),
        legend.text = element_text(size = 10),
        legend.key.size = unit(0.5, "cm")
    ) +
    labs(color = "Senescence") +
    guides(color = guide_legend(ncol = 1, override.aes = list(size = 4)))

legend <- cowplot::get_legend(p_legend)

n_groups <- length(sg_order)
n_cols <- 4
n_rows <- ceiling(n_groups / n_cols)

while (length(umap_list) < n_rows * n_cols) {
    umap_list[[length(umap_list) + 1]] <- plot_spacer()
}

options(repr.plot.width = 16, repr.plot.height = 4 * n_rows + 1)

umap_grid <- wrap_plots(umap_list, ncol = n_cols)

title_text <- if (is_aging) {
    paste0(tools::toTitleCase(cell_type_label), " Senescence by Age Group")
} else {
    paste0(tools::toTitleCase(cell_type_label), " Senescence by Disease Group")
}

umap_panel_snc <- (umap_grid | legend) + 
    plot_layout(widths = c(1, 0.15)) +
    plot_annotation(
        title = title_text,
        theme = theme(plot.title = element_text(size = 16, face = "bold", hjust = 0.5))
    )

print(umap_panel_snc)

ggsave(file.path(FIGURES_DIR, paste0(DATASET, "_", cell_type_label, "_umap_snc_by_studygroup.svg")), 
       umap_panel_snc, width = 16, height = 4 * n_rows + 1, dpi = 300)

cat("\n✓ Saved:", paste0(DATASET, "_", cell_type_label, "_umap_snc_by_studygroup.svg"), "\n")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# SAVE ANNOTATED OBJECT
# ════════════════════════════════════════════════════════════════════════════════

cat("\n", paste(rep("=", 80), collapse = ""), "\n")
cat(sprintf("SAVING ANNOTATED %s OBJECT\n", toupper(cell_type_label)))
cat(paste(rep("=", 80), collapse = ""), "\n")

save_file <- file.path(OUTPUT_DIR, paste0(DATASET, "_", cell_type_label, "_annotated.qs"))
qsave(seurat_obj, save_file)

cat("\n✓ Saved:", save_file, "\n")
cat("  Cells:", ncol(seurat_obj), "\n")
cat("  States:", length(unique(seurat_obj@meta.data[[SUBCLUSTER_COL]])), "\n")
cat("  Clusters:", length(unique(seurat_obj$seurat_clusters)), "\n")

# ════════════════════════════════════════════════════════════════════════════════
# MODULE SUMMARY
# ════════════════════════════════════════════════════════════════════════════════

cat("\n", paste(rep("=", 80), collapse = ""), "\n")
cat(sprintf("SUMMARY: %s SUBCLUSTERING & ANNOTATION\n", toupper(cell_type_label)))
cat(paste(rep("=", 80), collapse = ""), "\n")

cat("\n--- Data Summary ---\n")
cat("  Total cells:", ncol(seurat_obj), "\n")
cat("  Donors:", length(unique(seurat_obj@meta.data[[DONOR_COL]])), "\n")
cat("  Study groups:", paste(sg_order, collapse = ", "), "\n")

cat(sprintf("\n--- %s States ---\n", tools::toTitleCase(cell_type_label)))
state_counts <- table(seurat_obj@meta.data[[SUBCLUSTER_COL]])
for (s in names(sort(state_counts, decreasing = TRUE))) {
    cat(sprintf("  %s: %d (%.1f%%)\n", s, state_counts[s], state_counts[s] / sum(state_counts) * 100))
}

cat("\n--- Senescence ---\n")
cat(sprintf("  Label column: %s\n", SENESCENCE_LABEL_COL))
snc_counts <- table(seurat_obj@meta.data[[SENESCENCE_LABEL_COL]])
cat(sprintf("  SnC: %d (%.1f%%)\n", snc_counts["SnC"], snc_counts["SnC"] / sum(snc_counts) * 100))
cat(sprintf("  Non-SnC: %d (%.1f%%)\n", snc_counts["Non-SnC"], snc_counts["Non-SnC"] / sum(snc_counts) * 100))

if ("senescence_label_state" %in% colnames(seurat_obj@meta.data) && 
    SENESCENCE_LABEL_COL == "senescence_label_state") {
    cat("\n  (Original labels for comparison:)\n")
    orig_counts <- table(seurat_obj@meta.data$senescence_label)
    cat(sprintf("  SnC: %d (%.1f%%)\n", orig_counts["SnC"], orig_counts["SnC"] / sum(orig_counts) * 100))
}

cat("\n--- Output Files ---\n")
cat("  Data:\n")
cat("    •", save_file, "\n")

cat("\n  Figures:\n")
fig_files <- list.files(FIGURES_DIR, pattern = paste0("^", DATASET, "_", cell_type_label), full.names = FALSE)
for (f in fig_files) cat("    •", f, "\n")

cat("\n  Results:\n")
res_files <- list.files(RESULTS_DIR, pattern = paste0("^", DATASET, "_", cell_type_label), full.names = FALSE)
for (f in res_files) cat("    •", f, "\n")

cat("\n", paste(rep("=", 80), collapse = ""), "\n")
cat(sprintf("%s COMPLETE\n", toupper(cell_type_label)))
cat(paste(rep("=", 80), collapse = ""), "\n")

# 04G: Transfer Subclustering Labels & Export to Python

**Purpose:** Collect subclustering annotations from completed R modules (04D Microglia, 04E Astrocyte, 04F OPC), combine into a single annotation table, and merge into the Python AnnData object for downstream analysis.

## Workflow
1. Auto-detect completed modules by checking file existence
2. Load each Seurat object, extract barcode → state mapping
3. Combine into single table, validate (no duplicate barcodes)
4. Export CSV (R), then merge into AnnData (Python)

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# 04G: TRANSFER SUBCLUSTERING LABELS & EXPORT TO PYTHON
# ════════════════════════════════════════════════════════════════════════════════

cat("\n================================================================================\n")
cat("04G: TRANSFER SUBCLUSTERING LABELS & EXPORT TO PYTHON\n")
cat("================================================================================\n")

suppressPackageStartupMessages({
    library(Seurat)
    library(dplyr)
    library(tidyr)
    library(qs)
})

cat("✓ Libraries loaded\n")

# ════════════════════════════════════════════════════════════════════════════════
# CONFIGURATION
# ════════════════════════════════════════════════════════════════════════════════

DATASET <- "psychad_aging"

BASE_DIR <- "/fs/scratch/PAS2598/senescence_analysis"
RESULTS_DIR <- file.path(BASE_DIR, "results", "04_transfer", DATASET)
dir.create(RESULTS_DIR, recursive = TRUE, showWarnings = FALSE)

# ════════════════════════════════════════════════════════════════════════════════
# REGISTRY: ACTUAL PATHS WHERE EACH MODULE SAVED ITS OUTPUT
# ════════════════════════════════════════════════════════════════════════════════

MODULE_REGISTRY <- list(
    microglia = list(
        file = file.path(BASE_DIR, "data", "04_microglia", DATASET,
                         paste0(DATASET, "_microglia_annotated.qs")),
        state_col = "microglia_state",
        cluster_col = "seurat_clusters"
    ),
    astrocyte = list(
        file = file.path(BASE_DIR, "data", "04_astrocyte", DATASET,
                         paste0(DATASET, "_astrocyte_annotated.qs")),
        state_col = "astrocyte_state",
        cluster_col = "seurat_clusters"
    ),
    opc = list(
        file = file.path(BASE_DIR, "data", "04_opc", DATASET,
                         paste0(DATASET, "_opc_annotated.qs")),
        state_col = "opc_state",
        cluster_col = "seurat_clusters"
    )
)

cat("\nConfiguration:\n")
cat("  Dataset:", DATASET, "\n")
cat("  Results:", RESULTS_DIR, "\n")

cat("\nRegistered modules:\n")
for (ct in names(MODULE_REGISTRY)) {
    cat(sprintf("  %-12s → %s\n", ct, MODULE_REGISTRY[[ct]]$file))
}

# ════════════════════════════════════════════════════════════════════════════════
# STEP 1: DETECT COMPLETED MODULES
# ════════════════════════════════════════════════════════════════════════════════

cat("\n================================================================================\n")
cat("STEP 1: Detecting completed subclustering modules\n")
cat("================================================================================\n")

completed_modules <- list()

for (ct in names(MODULE_REGISTRY)) {
    mod <- MODULE_REGISTRY[[ct]]
    if (file.exists(mod$file)) {
        completed_modules[[ct]] <- mod
        fsize <- round(file.size(mod$file) / 1e6, 1)
        cat(sprintf("  ✓ %-12s FOUND  (%s MB)\n", ct, fsize))
    } else {
        cat(sprintf("  ✗ %-12s NOT FOUND → %s\n", ct, mod$file))
    }
}

cat(sprintf("\n%d of %d modules completed\n",
            length(completed_modules), length(MODULE_REGISTRY)))

if (length(completed_modules) == 0) {
    stop("No completed subclustering modules found. Nothing to transfer.")
}

# ════════════════════════════════════════════════════════════════════════════════
# STEP 2: EXTRACT ANNOTATIONS FROM EACH MODULE
# ════════════════════════════════════════════════════════════════════════════════

cat("\n================================================================================\n")
cat("STEP 2: Extracting annotations\n")
cat("================================================================================\n")

all_annotations <- list()

for (ct in names(completed_modules)) {
    mod <- completed_modules[[ct]]
    cat(sprintf("\nLoading %s from %s...\n", ct, basename(mod$file)))

    obj <- qread(mod$file)
    cat(sprintf("  Cells: %d\n", ncol(obj)))

    # Verify state column exists
    if (!mod$state_col %in% colnames(obj@meta.data)) {
        cat(sprintf("  ⚠ WARNING: column '%s' not found. Available columns:\n", mod$state_col))
        cat(sprintf("    %s\n", paste(head(colnames(obj@meta.data), 20), collapse = ", ")))
        next
    }

    # Extract barcode and state
    ann <- data.frame(
        barcode = colnames(obj),
        cell_type = ct,
        subcluster_state = as.character(obj@meta.data[[mod$state_col]]),
        stringsAsFactors = FALSE
    )

    # Check for NAs
    n_na <- sum(is.na(ann$subcluster_state))
    if (n_na > 0) {
        cat(sprintf("  ⚠ WARNING: %d cells have NA state\n", n_na))
    }

    # Print state distribution
    states <- table(ann$subcluster_state)
    cat("  States:\n")
    for (s in names(sort(states, decreasing = TRUE))) {
        cat(sprintf("    %-30s %6d cells (%5.1f%%)\n",
                    s, states[s], states[s] / nrow(ann) * 100))
    }

    all_annotations[[ct]] <- ann
    rm(obj); gc(verbose = FALSE)
}

# ════════════════════════════════════════════════════════════════════════════════
# STEP 3: COMBINE & VALIDATE
# ════════════════════════════════════════════════════════════════════════════════

cat("\n================================================================================\n")
cat("STEP 3: Combining annotations\n")
cat("================================================================================\n")

combined <- do.call(rbind, all_annotations)
rownames(combined) <- NULL

# Check for duplicate barcodes
n_dup <- sum(duplicated(combined$barcode))
if (n_dup > 0) {
    cat(sprintf("⚠ WARNING: %d duplicate barcodes found!\n", n_dup))
    dup_barcodes <- combined$barcode[duplicated(combined$barcode)]
    cat("  Duplicates come from:\n")
    print(combined %>% filter(barcode %in% dup_barcodes) %>% count(cell_type))
} else {
    cat("✓ No duplicate barcodes\n")
}

cat(sprintf("\nTotal annotated cells: %d\n", nrow(combined)))

# Summary
cat("\nSummary by cell type:\n")
summary_table <- combined %>%
    group_by(cell_type) %>%
    summarise(
        n_cells = n(),
        n_states = n_distinct(subcluster_state),
        states = paste(sort(unique(subcluster_state)), collapse = ", "),
        .groups = "drop"
    )
print(as.data.frame(summary_table))

# ════════════════════════════════════════════════════════════════════════════════
# STEP 4: EXPORT CSV FILES
# ════════════════════════════════════════════════════════════════════════════════

cat("\n================================================================================\n")
cat("STEP 4: Exporting CSV files\n")
cat("================================================================================\n")

# Long format
long_csv <- file.path(RESULTS_DIR, paste0(DATASET, "_subcluster_annotations.csv"))
write.csv(combined, long_csv, row.names = FALSE)
cat("✓ Saved (long):", long_csv, "\n")

# Wide format: one column per cell type
wide <- combined %>%
    mutate(col_name = paste0(cell_type, "_state")) %>%
    select(barcode, col_name, subcluster_state) %>%
    pivot_wider(
        names_from = col_name,
        values_from = subcluster_state
    )

# Add unified state column
wide <- wide %>%
    left_join(combined %>% select(barcode, subcluster_state), by = "barcode")

wide_csv <- file.path(RESULTS_DIR, paste0(DATASET, "_subcluster_annotations_wide.csv"))
write.csv(wide, wide_csv, row.names = FALSE)
cat("✓ Saved (wide):", wide_csv, "\n")

cat(sprintf("\nWide format: %d cells × %d columns\n", nrow(wide), ncol(wide)))
cat("Columns:", paste(colnames(wide), collapse = ", "), "\n")

# ════════════════════════════════════════════════════════════════════════════════
# SUMMARY
# ════════════════════════════════════════════════════════════════════════════════

cat("\n")
cat("################################################################################\n")
cat("#              04G: R EXPORT COMPLETE                                          #\n")
cat("################################################################################\n")

cat("\nCompleted modules:\n")
for (ct in names(completed_modules)) {
    n <- sum(combined$cell_type == ct)
    ns <- length(unique(combined$subcluster_state[combined$cell_type == ct]))
    cat(sprintf("  %-12s %6d cells → %d states\n", ct, n, ns))
}

cat("\nOutput files:\n")
cat(sprintf("  %s\n", long_csv))
cat(sprintf("  %s\n", wide_csv))

cat("\nNext: Run the Python cell to merge into AnnData\n")
cat("================================================================================\n")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# STEP 1: DETECT COMPLETED SUBCLUSTERING MODULES
# ════════════════════════════════════════════════════════════════════════════════

cat("\n================================================================================\n")
cat("STEP 1: Detecting completed subclustering modules\n")
cat("================================================================================\n")

completed_modules <- list()

for (ct in names(MODULE_REGISTRY)) {
    mod <- MODULE_REGISTRY[[ct]]
    if (file.exists(mod$file)) {
        completed_modules[[ct]] <- mod
        cat(sprintf("  ✓ %-12s FOUND  → %s\n", ct, basename(mod$file)))
    } else {
        cat(sprintf("  ✗ %-12s not found (skipping)\n", ct))
    }
}

cat(sprintf("\n%d of %d modules completed\n", length(completed_modules), length(MODULE_REGISTRY)))

if (length(completed_modules) == 0) {
    stop("No completed subclustering modules found. Nothing to transfer.")
}

# ════════════════════════════════════════════════════════════════════════════════
# STEP 2: EXTRACT ANNOTATIONS FROM EACH MODULE
# ════════════════════════════════════════════════════════════════════════════════

cat("\n================================================================================\n")
cat("STEP 2: Extracting annotations\n")
cat("================================================================================\n")

all_annotations <- list()

for (ct in names(completed_modules)) {
    mod <- completed_modules[[ct]]
    cat(sprintf("\nLoading %s...\n", ct))

    obj <- qread(mod$file)
    cat(sprintf("  Cells: %d\n", ncol(obj)))

    # Extract barcode, state, and cluster
    ann <- data.frame(
        barcode = colnames(obj),
        cell_type = ct,
        subcluster_state = as.character(obj@meta.data[[mod$state_col]]),
        subcluster_id = as.character(obj@meta.data[[mod$cluster_col]]),
        stringsAsFactors = FALSE
    )

    # Verify state column exists
    if (all(is.na(ann$subcluster_state))) {
        cat(sprintf("  ⚠ WARNING: state column '%s' is all NA — check annotation\n", mod$state_col))
    } else {
        states <- table(ann$subcluster_state)
        cat("  States:\n")
        for (s in names(sort(states, decreasing = TRUE))) {
            cat(sprintf("    %-25s %6d cells (%5.1f%%)\n", s, states[s], states[s] / nrow(ann) * 100))
        }
    }

    all_annotations[[ct]] <- ann
    rm(obj); gc(verbose = FALSE)
}

# ════════════════════════════════════════════════════════════════════════════════
# STEP 3: COMBINE INTO SINGLE ANNOTATION TABLE
# ════════════════════════════════════════════════════════════════════════════════

cat("\n================================================================================\n")
cat("STEP 3: Combining annotations\n")
cat("================================================================================\n")

combined_annotations <- do.call(rbind, all_annotations)
rownames(combined_annotations) <- NULL

# Check for duplicate barcodes (should not happen)
n_dup <- sum(duplicated(combined_annotations$barcode))
if (n_dup > 0) {
    cat(sprintf("⚠ WARNING: %d duplicate barcodes found — check subclustering objects\n", n_dup))
} else {
    cat("✓ No duplicate barcodes\n")
}

cat(sprintf("\nTotal annotated cells: %d\n", nrow(combined_annotations)))
cat("\nSummary by cell type:\n")
summary_table <- combined_annotations %>%
    group_by(cell_type) %>%
    summarise(
        n_cells = n(),
        n_states = n_distinct(subcluster_state),
        states = paste(sort(unique(subcluster_state)), collapse = ", "),
        .groups = "drop"
    )
print(as.data.frame(summary_table))

# Save combined CSV
combined_csv <- file.path(RESULTS_DIR, paste0(DATASET, "_subcluster_annotations.csv"))
write.csv(combined_annotations, combined_csv, row.names = FALSE)
cat("\n✓ Saved:", combined_csv, "\n")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# 04G: MERGE SUBCLUSTERING ANNOTATIONS INTO ANNDATA
# ════════════════════════════════════════════════════════════════════════════════

print("=" * 80)
print("04G: MERGE SUBCLUSTERING ANNOTATIONS INTO ANNDATA")
print("=" * 80)

import scanpy as sc
import pandas as pd
import numpy as np
from pathlib import Path

# ── Configuration ─────────────────────────────────────────────────────────────

DATASET = "psychad_aging"

BASE_DIR = Path("/fs/scratch/PAS2598/senescence_analysis")
RESULTS_DIR = BASE_DIR / "results" / "04_transfer" / DATASET

# Input
ADATA_FILE = BASE_DIR / "data" / "04_subsetting" / DATASET / f"{DATASET}_celltypes_subset.h5ad"
ANNOTATIONS_WIDE = RESULTS_DIR / f"{DATASET}_subcluster_annotations_wide.csv"

# Output
OUTPUT_DIR = BASE_DIR / "data" / "04_transfer" / DATASET
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_FILE = OUTPUT_DIR / f"{DATASET}_subclustered.h5ad"

print(f"\nConfiguration:")
print(f"  Dataset:     {DATASET}")
print(f"  AnnData:     {ADATA_FILE}")
print(f"  Annotations: {ANNOTATIONS_WIDE}")
print(f"  Output:      {OUTPUT_FILE}")

# ── Load AnnData ──────────────────────────────────────────────────────────────

print(f"\n{'='*80}")
print("LOADING ANNDATA")
print(f"{'='*80}")

adata = sc.read_h5ad(ADATA_FILE)
print(f"\n✓ Loaded: {adata.n_obs:,} cells × {adata.n_vars:,} genes")
print(f"  Index name: {adata.obs.index.name}")
print(f"  First 3 barcodes: {list(adata.obs.index[:3])}")

# ── Load Annotations ──────────────────────────────────────────────────────────

print(f"\n{'='*80}")
print("LOADING ANNOTATIONS")
print(f"{'='*80}")

annotations = pd.read_csv(ANNOTATIONS_WIDE)
print(f"✓ Loaded: {len(annotations):,} annotated cells")
print(f"  Columns: {list(annotations.columns)}")
print(f"  First 3 barcodes: {list(annotations['barcode'][:3])}")

# ── Index Alignment Check ─────────────────────────────────────────────────────

print(f"\n{'='*80}")
print("INDEX ALIGNMENT CHECK")
print(f"{'='*80}")

adata_barcodes = set(adata.obs.index)
ann_barcodes = set(annotations['barcode'])

# Check overlap
overlap = adata_barcodes & ann_barcodes
missing_from_adata = ann_barcodes - adata_barcodes
not_annotated = adata_barcodes - ann_barcodes

print(f"\n  AnnData cells:        {len(adata_barcodes):,}")
print(f"  Annotation cells:     {len(ann_barcodes):,}")
print(f"  Overlap:              {len(overlap):,}")
print(f"  In annotations only:  {len(missing_from_adata):,}")
print(f"  In AnnData only:      {len(not_annotated):,} (non-subclustered cell types)")

# If zero overlap, check for barcode format mismatch
if len(overlap) == 0:
    print("\n⚠ ZERO OVERLAP — checking for format mismatch:")
    print(f"  AnnData sample:    '{list(adata_barcodes)[:3]}'")
    print(f"  Annotation sample: '{list(ann_barcodes)[:3]}'")
    raise ValueError("No matching barcodes between AnnData and annotations!")

if len(missing_from_adata) > 0:
    print(f"\n  ⚠ {len(missing_from_adata)} annotation barcodes not in AnnData")
    print(f"    Examples: {list(missing_from_adata)[:5]}")

match_pct = len(overlap) / len(ann_barcodes) * 100
print(f"\n  Match rate: {match_pct:.1f}%")
assert match_pct > 95, f"Match rate too low ({match_pct:.1f}%) — check barcode formats!"
print(f"  ✓ Match rate OK")

# ── Merge ─────────────────────────────────────────────────────────────────────

print(f"\n{'='*80}")
print("MERGING ANNOTATIONS")
print(f"{'='*80}")

# Set barcode as index for alignment
annotations_indexed = annotations.set_index('barcode')

# Store original obs for comparison
original_cols = list(adata.obs.columns)
original_n_obs = adata.n_obs
original_index = adata.obs.index.copy()

# Add each column from annotations (reindex aligns by barcode automatically)
new_cols = [c for c in annotations_indexed.columns if c != 'barcode']
for col in new_cols:
    adata.obs[col] = annotations_indexed[col].reindex(adata.obs.index)
    n_filled = adata.obs[col].notna().sum()
    n_na = adata.obs[col].isna().sum()
    print(f"  Added '{col}': {n_filled:,} filled, {n_na:,} NA")

# ── Verify index not corrupted ────────────────────────────────────────────────

print(f"\n{'─'*60}")
print("Index integrity check")
print(f"{'─'*60}")

assert adata.n_obs == original_n_obs, \
    f"Cell count changed! {original_n_obs:,} → {adata.n_obs:,}"
print(f"  ✓ Cell count preserved: {adata.n_obs:,}")

assert (adata.obs.index == original_index).all(), \
    "Index order changed during merge!"
print(f"  ✓ Index order preserved")

assert all(c in adata.obs.columns for c in original_cols), \
    "Original columns lost during merge!"
print(f"  ✓ Original columns preserved ({len(original_cols)} columns)")

# ── Create unified cell_state column ──────────────────────────────────────────

print(f"\n{'─'*60}")
print("Creating unified cell_state column")
print(f"{'─'*60}")

CELL_TYPE_COL = 'subclass'

# Start with subcluster_state where available
adata.obs['cell_state'] = adata.obs['subcluster_state'].copy()

# Fill non-annotated cells with their broad cell type
mask_na = adata.obs['cell_state'].isna()
adata.obs.loc[mask_na, 'cell_state'] = adata.obs.loc[mask_na, CELL_TYPE_COL]

print(f"\n  Subclustered cells: {(~mask_na).sum():,}")
print(f"  Broad label cells:  {mask_na.sum():,}")

# Final NA check
n_na_final = adata.obs['cell_state'].isna().sum()
assert n_na_final == 0, f"Found {n_na_final} NAs in cell_state!"
print(f"  ✓ No NAs in cell_state")

# ── Validation ────────────────────────────────────────────────────────────────

print(f"\n{'='*80}")
print("VALIDATION")
print(f"{'='*80}")

# Per-cell-type state columns
state_cols = [c for c in new_cols if c.endswith('_state') and c != 'subcluster_state']

for col in state_cols:
    n_annotated = adata.obs[col].notna().sum()
    print(f"\n  {col}: {n_annotated:,} cells")
    counts = adata.obs[col].value_counts(dropna=True)
    for val, n in counts.items():
        print(f"    {val:30s} {n:>7,}")

# Cross-check: subclustered cell types should match original cell type
print(f"\n{'─'*60}")
print("Cross-check: cell_type consistency")
print(f"{'─'*60}")

for col in state_cols:
    ct_name = col.replace('_state', '').capitalize()
    if ct_name == 'Opc':
        ct_name = 'OPC'
    
    annotated_mask = adata.obs[col].notna()
    if annotated_mask.sum() == 0:
        continue
    
    actual_types = adata.obs.loc[annotated_mask, CELL_TYPE_COL].unique()
    print(f"  {col}: annotated cells are type(s) = {list(actual_types)}")
    
    # All annotated cells should be the correct cell type
    expected = ct_name
    mismatch = adata.obs.loc[annotated_mask, CELL_TYPE_COL] != expected
    n_mismatch = mismatch.sum()
    if n_mismatch > 0:
        print(f"    ⚠ {n_mismatch} cells have unexpected cell type!")
    else:
        print(f"    ✓ All match expected type '{expected}'")

# Cell state distribution
print(f"\n{'─'*60}")
print("Cell state distribution")
print(f"{'─'*60}")
print(adata.obs['cell_state'].value_counts().to_string())

# ── Save ──────────────────────────────────────────────────────────────────────

print(f"\n{'='*80}")
print("SAVING")
print(f"{'='*80}")

adata.write_h5ad(OUTPUT_FILE)

print(f"\n✓ Saved: {OUTPUT_FILE}")
print(f"  {adata.n_obs:,} cells × {adata.n_vars:,} genes")
print(f"  Original columns: {len(original_cols)}")
print(f"  New columns: {new_cols + ['cell_state']}")
print(f"  Total columns: {len(adata.obs.columns)}")

print(f"\n{'='*80}")
print("04G COMPLETE")
print(f"{'='*80}")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# 04G-R: IMPORT ANNOTATIONS TO SEURAT OBJECT → SAVE _subclustered.qs
# ════════════════════════════════════════════════════════════════════════════════

cat("\n================================================================================\n")
cat("04G-R: IMPORT ANNOTATIONS TO SEURAT OBJECT\n")
cat("================================================================================\n")

suppressPackageStartupMessages({
    library(Seurat)
    library(qs)
    library(dplyr)
})

# ════════════════════════════════════════════════════════════════════════════════
# CONFIGURATION
# ════════════════════════════════════════════════════════════════════════════════

DATASET <- "psychad_aging"

BASE_DIR <- "/fs/scratch/PAS2598/senescence_analysis"

# Dataset-specific cell type column
DATASET_CONFIG <- list(
    "psychad_aging" = list(cell_type_col = "subclass"),
    "psychad_ad"    = list(cell_type_col = "subclass"),
    "psychencode"   = list(cell_type_col = "cell_type"),
    "mathys"        = list(cell_type_col = "broad.cell.type")
)

CELL_TYPE_COL <- DATASET_CONFIG[[DATASET]]$cell_type_col

# ── Input: Parent Seurat object (from 04C) ──
SEURAT_INPUT <- file.path(BASE_DIR, "data", "04_subsetting", DATASET, paste0(DATASET, "_seurat.qs"))

# ── Input: Annotations from Cell 75 ──
# ALIGNED to where Cell 75 actually saves:
ANNOTATIONS_DIR <- file.path(BASE_DIR, "results", "04_transfer", DATASET)

# ── Output: Updated Seurat object with subcluster annotations ──
OUTPUT_DIR <- file.path(BASE_DIR, "data", "04_transfer", DATASET)
dir.create(OUTPUT_DIR, recursive = TRUE, showWarnings = FALSE)
SEURAT_OUTPUT <- file.path(OUTPUT_DIR, paste0(DATASET, "_subclustered.qs"))

cat("\nConfiguration:\n")
cat("  Dataset:", DATASET, "\n")
cat("  Cell type column:", CELL_TYPE_COL, "\n")
cat("  Input Seurat:", SEURAT_INPUT, "\n")
cat("  Annotations:", ANNOTATIONS_DIR, "\n")
cat("  Output:", SEURAT_OUTPUT, "\n")

# ════════════════════════════════════════════════════════════════════════════════
# LOAD SEURAT OBJECT
# ════════════════════════════════════════════════════════════════════════════════

cat("\n================================================================================\n")
cat("LOADING SEURAT OBJECT\n")
cat("================================================================================\n")

seurat <- qread(SEURAT_INPUT)
cat("\n✓ Loaded:", ncol(seurat), "cells\n")

cat("\nCell types:\n")
print(table(seurat[[CELL_TYPE_COL, drop = TRUE]]))

# ════════════════════════════════════════════════════════════════════════════════
# LOAD ANNOTATIONS
# ════════════════════════════════════════════════════════════════════════════════

cat("\n================================================================================\n")
cat("LOADING ANNOTATIONS\n")
cat("================================================================================\n")

annotations_file <- file.path(ANNOTATIONS_DIR, paste0(DATASET, "_subcluster_annotations.csv"))
annotations <- read.csv(annotations_file)

cat("\n✓ Loaded:", nrow(annotations), "annotations\n")
cat("  Columns:", paste(colnames(annotations), collapse = ", "), "\n")

# ── VERIFY EXPECTED COLUMNS ──
# Cell 75 writes: barcode, cell_type, subcluster_state, subcluster_id
expected_cols <- c("barcode", "cell_type", "subcluster_state", "subcluster_id")
missing_cols <- setdiff(expected_cols, colnames(annotations))
if (length(missing_cols) > 0) {
    stop(sprintf("Missing columns in annotation CSV: %s\n  Found: %s\n  Expected: %s",
                 paste(missing_cols, collapse = ", "),
                 paste(colnames(annotations), collapse = ", "),
                 paste(expected_cols, collapse = ", ")))
}
cat("✓ All expected columns present\n")

cat("\nBy cell type:\n")
print(table(annotations$cell_type))

cat("\nBy subcluster state:\n")
print(table(annotations$subcluster_state))

# ════════════════════════════════════════════════════════════════════════════════
# BARCODE ALIGNMENT CHECK
# ════════════════════════════════════════════════════════════════════════════════

cat("\n================================================================================\n")
cat("BARCODE ALIGNMENT CHECK\n")
cat("================================================================================\n")

# Create lookup using 'barcode' column (matches Cell 75 output)
rownames(annotations) <- annotations$barcode

seurat_barcodes <- colnames(seurat)
ann_barcodes <- annotations$barcode

n_overlap <- sum(ann_barcodes %in% seurat_barcodes)
n_ann_only <- sum(!ann_barcodes %in% seurat_barcodes)

cat(sprintf("\n  Seurat barcodes:     %d\n", length(seurat_barcodes)))
cat(sprintf("  Annotation barcodes: %d\n", length(ann_barcodes)))
cat(sprintf("  Overlap:             %d\n", n_overlap))
cat(sprintf("  In annotations only: %d\n", n_ann_only))

if (n_ann_only > 0) {
    cat(sprintf("\n⚠ WARNING: %d annotation barcodes not in Seurat object\n", n_ann_only))
    missing_ann <- ann_barcodes[!ann_barcodes %in% seurat_barcodes]
    cat("  First 3 annotation-only:", head(missing_ann, 3), "\n")
    cat("  First 3 Seurat:", head(seurat_barcodes, 3), "\n")
} else {
    cat("\n✓ All annotation barcodes found in Seurat object\n")
}

# ════════════════════════════════════════════════════════════════════════════════
# MERGE ANNOTATIONS
# ════════════════════════════════════════════════════════════════════════════════

cat("\n================================================================================\n")
cat("MERGING ANNOTATIONS\n")
cat("================================================================================\n")

# Initialize columns
seurat$subcluster_state <- "NA"
seurat$subcluster_seurat_cluster <- "NA"

# Find matching cells (using 'barcode' column)
matching_cells <- intersect(colnames(seurat), annotations$barcode)
cat("\nMatching cells:", length(matching_cells), "/", nrow(annotations), "\n")

if (length(matching_cells) == 0) {
    stop("No matching barcodes! Check barcode format between Seurat and annotation CSV.")
}

# Merge (using 'subcluster_id' column from Cell 75, not 'seurat_cluster')
seurat$subcluster_state[matching_cells] <- annotations[matching_cells, "subcluster_state"]
seurat$subcluster_seurat_cluster[matching_cells] <- as.character(annotations[matching_cells, "subcluster_id"])

# Check merge
n_annotated <- sum(seurat$subcluster_state != "NA")
cat("Cells with subcluster annotations:", n_annotated, "\n")

# ════════════════════════════════════════════════════════════════════════════════
# CREATE UNIFIED CELL STATE COLUMN
# ════════════════════════════════════════════════════════════════════════════════

cat("\n================================================================================\n")
cat("CREATING UNIFIED CELL STATE COLUMN\n")
cat("================================================================================\n")

# For cells WITH subclustering: use subcluster_state
# For cells WITHOUT subclustering: use cell type
seurat$cell_state <- as.character(seurat[[CELL_TYPE_COL, drop = TRUE]])

# Update with subcluster states where available
mask <- seurat$subcluster_state != "NA"
seurat$cell_state[mask] <- seurat$subcluster_state[mask]

cat("\nCell state distribution:\n")
print(table(seurat$cell_state))

# ════════════════════════════════════════════════════════════════════════════════
# SAVE
# ════════════════════════════════════════════════════════════════════════════════

cat("\n================================================================================\n")
cat("SAVING\n")
cat("================================================================================\n")

qsave(seurat, SEURAT_OUTPUT)

cat("\n✓ Saved:", SEURAT_OUTPUT, "\n")
cat("  Cells:", ncol(seurat), "\n")
cat("  Genes:", nrow(seurat), "\n")

# ════════════════════════════════════════════════════════════════════════════════
# SUMMARY
# ════════════════════════════════════════════════════════════════════════════════

cat("\n================================================================================\n")
cat("04G-R SUMMARY\n")
cat("================================================================================\n")

cat("\nTotal cells:", ncol(seurat), "\n")

for (ct in unique(seurat[[CELL_TYPE_COL, drop = TRUE]])) {
    n_ct <- sum(seurat[[CELL_TYPE_COL, drop = TRUE]] == ct)
    n_sub <- sum(seurat[[CELL_TYPE_COL, drop = TRUE]] == ct & seurat$subcluster_state != "NA")
    cat(sprintf("\n%s:\n  Total: %d\n  Subclustered: %d (%.1f%%)\n", 
                ct, n_ct, n_sub, n_sub/n_ct*100))
}

cat("\nNew columns added:\n")
cat("  - subcluster_state: Functional state from subclustering, 'NA' for non-subclustered\n")
cat("  - subcluster_seurat_cluster: Seurat cluster ID, 'NA' for non-subclustered\n")
cat("  - cell_state: Unified state (subcluster for subclustered, cell type for others)\n")

cat("\n================================================================================\n")
cat("04G-R COMPLETE\n")
cat("================================================================================\n")